# Демографический советник · ML-05 V7.4.0 — Advanced Data Reconstruction

**Полный автономный Google Colab:** 52 закреплённых источника Stage I (7.2), семь исследовательских модулей, дополнительные строго наблюдаемые реконструкции, сохранение модели/прогнозов, независимая QA, полный evidence в приватном GitHub и автоскачивание компактной аналитики.

**Принцип:** если целевой показатель не подтверждён исходником, опыт фиксируется `SKIPPED_NOT_OBSERVABLE`. Никакой синтетической мобильности, дорасчёта отсутствующих OD-рёбер или автоматического присвоения кампусам адреса по ближайшему населённому пункту. Все результаты V7 сохраняются, а дополнительные эксперименты маркируются отдельно.

**Доступ:** секрет Colab `TRIAD-RU-Colab`. Для запуска: `Runtime → Run all` в свежем сеансе. Автоматическая замена действует только на `stage2/ml05_education/` после технической проверки; исходные Stage I и другие ML остаются неизменными.


**V7.4.0 fixes:** обязательное раннее исполнение DuckDB SQL (до GitHub), атомарная реконструкция E02, независимые индикаторы прогресса и перенос финальных отметок этапов после всех подмодулей. Остальные модели V7.1 сохранены.


**Статистика V7.4.0.** 9 999 знаковых перестановок для Monte Carlo (либо точное перечисление знаков при малом числе кластеров), 2 499 кластерных bootstrap-репликаций, поправка Холма по единому семейству сравнений. Значения `p` характеризуют разведочный ретроспективный тест, не подтверждают причинность или исторический источник данных.


## 1. Единственная конфигурационная ячейка
Первый запуск: `MODE="SMOKE"` и `AUTO_PUBLISH_AFTER_FULL_PASS=False`. Для последующей валидации используйте `PILOT10`, затем `FULL`. Точные имена показателей для прогноза приёма можно утвердить только после первичного анализа словаря. Меняется только эта ячейка.

In [ ]:
# Единственная ячейка, в которой пользователь меняет параметры эксперимента.
from dataclasses import dataclass, asdict
from pathlib import Path

@dataclass(frozen=True)
class ML5Config:
    VERSION: str = 'stage2-ml5-v7.4.0-full-research-high-resolution-inference'
    MODE: str = 'FULL'             # SMOKE | PILOT10 | FULL; FULL по всем строкам без загрузки всего E04/E05 в RAM
    REPO: str = 'AntonDozhdikov/Demograf_sovetnik'
    SOURCE_BRANCH: str = 'main'
    INPUT_INDEX: str = 'handoff/07_v2/dataset_index.json'
    EXPECTED_INDEX_BLOB_SHA: str = 'd6625ccddc24f24a1ee451e33c48b736d3a57043'
    TARGET_BRANCH: str = 'ml05-education-20261102T160726Z'
    OUTPUT_PREFIX: str = 'stage2/ml05_education'
    WORKDIR: str = '/content/ml5_v7_4_0'
    FEATURE_PUBLICATION_LAG_YEARS: int = 1
    LABEL_PUBLICATION_LAG_YEARS: int = 1
    HORIZON: int = 1
    ALLOW_RETROSPECTIVE_ONLY: bool = True  # Все исторические данные доступны в текущем, а не историческом vintage.
    REQUIRE_ORIGINAL_E01_FOR_PREDICTION: bool = True
    APPROVE_UNAMBIGUOUS_INDICATORS: bool = False  # Показатель для прогноза утверждается явно, не эвристикой
    APPROVED_VO_ADMISSION_NAMES: tuple = ()  # Точные названия из аудита E02, проверенные по методике
    APPROVED_SPO_ADMISSION_NAMES: tuple = ()  # Точные названия из аудита E01/СПО
    STRICT_OD_COVERAGE_BREAK: bool = True   # При разрыве допускается только описательная диагностика / разведочный тест
    ALLOW_RETROSPECTIVE_TESTS: bool = True  # Не выдавать за operational historical backtest
    MIN_YEARS_FOR_TEMPORAL: int = 3
    MIN_TRAIN_ROWS: int = 100   # For regional ML-5.3 overridden by REGION_MIN_TRAIN_ROWS
    MIN_DEV_ROWS: int = 20
    MIN_TEST_ROWS: int = 20
    MIN_MUNICIPALITIES_E0E1: int = 35
    MAX_REMOTE_INPUT_GIB: float = 40.0
    MAX_READ_ROWS_SMOKE: int = 120_000
    BATCH_ROWS: int = 20_000
    DUCKDB_MEMORY_LIMIT: str = '512MB'
    DUCKDB_THREADS: int = 1
    REGION_MIN_TRAIN_ROWS: int = 30
    REGION_MIN_TEST_ROWS: int = 25
    CATALOG_FULL_DOWNLOAD: bool = True   # FULL: strict 52/52 sha256 verified, no hidden skip
    CATALOG_SCAN_ALL_YEARS: bool = True  # all recorded year labels, without pandas materialization
    STAGE1_DISK_RESERVE_GIB: float = 5.0
    EXTRA_CATALOG_PROFILE_MAX_GROUPS: int = 1500
    MAX_PANDAS_ROWS: int = 150_000
    PANEL_IN_MEMORY_MAX_ROWS: int = 15_000
    PANEL_PREVIEW_ROWS: int = 2_000
    EXPORT_PARQUET_CHUNK_ROWS: int = 10_000
    EXCEL_PREVIEW_ROWS: int = 3_000
    MIN_FREE_RAM_GIB: float = 2.0
    BOOTSTRAP_REPS: int = 2499      # cluster bootstrap CI, separately from permutation test
    PERMUTATION_REPS: int = 9999    # 50-family Holm min corrected p = 0.005
    INFERENCE_ALPHA: float = 0.05
    RESAMPLING_BATCH_ROWS: int = 256
    ML54_FULL_NO_STRATA_CAP: bool = True
    ML54_FINAL_SEEDS: int = 50
    MAX_COUNT_PREDICTION_MULTIPLIER: float = 10.0
    REQUIRE_UNIQUE_MODEL_ARTIFACTS: bool = True
    REQUIRE_SPATIAL_BASELINES: bool = True
    RANDOM_SEED: int = 20261104
    DEV_SEEDS: tuple = (11,23,37,41,53,67,79,83,97,101)
    FINAL_SEEDS: tuple = tuple(range(101,151))
    AUTO_PUBLISH_AFTER_FULL_PASS: bool = True  # Публикует только финальный проверенный этап V7.
    PUBLISH_RESULTS_TO_PRIVATE_GITHUB: bool = True   # Только отдельный закрытый Release + свой сегмент ветки.
    DOWNLOAD_COMPACT_REPORTS: bool = True              # Скачать отчёты ДО долгой загрузки в GitHub.
    GH_RELEASE_PART_MIB: int = 160                    # Ограничить размер одного Release asset.
    GH_REQUIRE_ASSET_DIGEST: bool = True
    REQUIRE_COMPLETED_COMPUTATIONAL_CONTRACT: bool = True
    PUBLISH_PARTIAL_RUN_WITH_EXPLICIT_FAIL_STATUS: bool = False
    STORE_DERIVED_RESTRICTED_DATA: bool = True  # Только внутри приватного репозитория, при допустимости лицензии.
    PRINT_MAX_ROWS: int = 12

CFG=ML5Config()
assert CFG.MODE in ('SMOKE','PILOT10','FULL')
assert CFG.PERMUTATION_REPS >= 4999 and CFG.BOOTSTRAP_REPS >= 999
assert 50/(CFG.PERMUTATION_REPS+1) < CFG.INFERENCE_ALPHA, 'Insufficient Monte-Carlo resolution for 50-model Holm family'
assert CFG.HORIZON>=1 and CFG.FEATURE_PUBLICATION_LAG_YEARS>=0
assert CFG.LABEL_PUBLICATION_LAG_YEARS>=0
ROOT=Path(CFG.WORKDIR)
for sub in ('inputs','derived','figures','models','exports','logs'):(ROOT/sub).mkdir(parents=True,exist_ok=True)
print('ML-05',CFG.VERSION,'|',CFG.MODE,'| input: immutable Stage I | публикация:',CFG.AUTO_PUBLISH_AFTER_FULL_PASS)


In [ ]:
# ML-05 V7.4.0 · self-contained immutable notebook; no Colab get_ipynb calls.
EMBEDDED_SOURCE_B85 = "c-q9hdvhDtogn)Ee2TU_nFgc=2!L<U@GL|^60S*t0YKRvira1i4H6LuAZS1mLvc&Clh}Kci6^<4nyI^+%x30p?X9|Xt7KWSX+5jv6Hs4aKgs<)&gnkgeHx?`=dNcW0*%w>`8&V&<M^+POD(t8^DY&Qoxd`SWB4CGvByV47oJ`^bVj$^gS-7p7X8+3<3o48?X=s@l6ZLNj-9qMcHk%KA=V5Zo^V@lFd4O6F&=z>*EswA*^{&9XD`pbdHvw*;p?BDoxc9!?8)m-&z_#0o_%k?hnHuM;K65SFW|vb<KMqCHr6w_dE@3{b}pMU{^x_=8|C&Lr{8kh#u`k+sJpE}-y4r6t#NnIza{2wTnav2zI^sC(CxFc7q5Q>W1Kxadj$Qzgb`lCRQTcb!C=4V8mohzvsb))*_baFXWv1~r!dVYXRpqlz5Xq<hqkYOVZfX}#SUQRG|97vMq}*kyGGSWEoKX83zq?_eEn-~`Y8<k<m@r6>><4P1bX{5FByBo-FW=^b87UxWl+OcJYQ%an$t9&7&tJ@2)6I+r2)S`Jo_4c{SCbHkVd37IOC_dK5h+NKYjg6nCK}@?B(lU;MT*-PiXp2#0+qpv(uTgSE7I1o$qkYmO*WCHy+`7;GO5Nj<1X#m*J;Z%FeoXx@{Lu#yERqP(P<)#kAAL_3rq_WDoYlKLXl>hp>;-P)_G5oy#lQeSVh?4Ojb&b_DnQG@EqPH0}#p@hdz^0xx*cfJaYgmmhJLICU5vS0=Zamdd|8hGz(g56-@Y&v5#Dn7^RC;&X>-!VbV`0a$!yATYy>VSi}P0fwRZSH>re52{;Rl{I^_*0gI^8<qOa^40aqCt2g{4-7rj6Tv>t`!j<6;9AgB@r1sHp9vCh->?b83f-emUw`rXxA5#COozt(oc@Isz^C85{?7zywY3b)?6=SntjfUNl%t?C4UbDOM#l&baL))}U`aG-I3ei!Q~ZRTJVvm?ue5sx!X=FGE&THR>n{m5KPP~tt08U^j*5?=BlrS`{rYp*mEU0#hT`w=TEJcT4CnUxSAYaU5d8WpSR3s!Y{kQKZqDD+*}S3?MF@U^dkuZk!XMCOd_bGU$A<gJmx8xE3EY0qrvVcsV58*`(g27cY+`)UtXIoxnfi8yzT?6F9(w(X5e#o4Y-rK=q(147yN9mvKY#f<13nv0ulEVwCEETcu$i!pJOVBr{z6y(kZ^nt<p8u&-W|X%5-4bX^dheBslX=2CmtfQ!pvbWH*dS`Ny|Z`I`at~DkCz3AoLfs#UU7F#KBhvc5@|kU+4@ZmPD|7Oqb*vy73$9hJfjZ!Tux8IfWJQ{IP@Q371|y4?!k6-FvP_mmluU>#x)+PGCpy^^joj310Y90u6>7i5Ax;t=nr?jmD4HjTC~Gz+Y*EHGj(=5KMoSh;-qnO0J+op|m4-<DqXt1B4JCAVdlj%r}bf)Ttpr_X*#B#x{f!8N`v2Ajm!^^Z|$sARafAmi0(IKREtR@dO2$3)3Hfs6X;aD0fFC5J-sgF*@@4SLdG0*`N3t3+l;qlaDJ42eoW0FE1m+z7PoU5wA(CTFmJMK^U$M=7V==W6&Q1*>*;~0h|(|oUh@eeCqs?u{n3~1#SJafbJ=S2}DQO0Gc4V3fd#Wdq!CHGwKT30~8dDIh=9$Cq9Y2!C(w5i8IVd_CXvm5~c42)qF+U_lWl4?7tK6J@?V)6M_nfA(@|0?=Nx!{JI2KJi31(X#B!Q)SnDLG2jM1q^3{B6i*SY;yvR#@Ezfn$Gl-g#u<CzIvHWon@FL>&!6#eSV_c^Z?PmDU(h-P(IS?J>5gQg9z<BcD0gKLb0c^!gfaxE-M<n%=0%y<LO!?HDJ;6@FtL8KX-w?9MjrP1ANhp%Kp00c(q$}w*Yg|L=Lh1{U-}JzYT*fej|U%I5MWVp`yMfis!#@?3mdhyO6k_6#zyUf%B@SEAU61<yxpwXTiaLHtBo6W`C7A5x36uluiIPYMx%7i>3Oa~X#C5M7oGTlCpgL@=BVhzh(S~kAzXf>zTe|%8=q`d*K5su?i0)KQR#Ez6C4mn`Bxu!2&4HFC&pm$u#d&0LX74^2`|6pTYxC%Tl~#HUPb`dV<hVULt(<Dpf<khj|ik*NEXIt9cdV9%B=W<*I&Wx|0^M|uY86r)Hx&3(3xe=g}~OJEdkBF=e8zDL+#c8s5|`HpY(bvWG?Q_?Jdt077`(|g<OOG)ws0hc&<Geb@Anc@p$MJXJ&eXmeV^Jc;n(yZYeiIGZgTvAIG>fn2d*$ag~2tY_%3<XImL}aXFWnbC=qgWw+hVEVlMqd-M6dJ-3rruLQqKg5++Zq?dF-)V;JwRCr31>m|X1;L8dA!V;4qm^L~oY{h8pIU_5en|c-x4lKB3ozdXX!0m1I9M5w-qkA|UjK;wu%W%AQw>6HuGIYiVz3!fDz6D=(RPI0c{iNWvpSk@~b2M?&K?C{?_?`LHLBG@8F9yHhKQ}A&MzywCG=O#(CBtN9EOXeK&)gwGndt!VWJa#%I-}M>=AgTOfZql^0<lcD-*HDMfSKWb5r6od;-xhW^}lxw!YsypLtJzCgsoQSUqO%_F%1eN(5KA#e~o+%Bdl+T?fDMzxNML2uacXasel9zCtWXZglAT-Y}NdkmHXpCe{JxygLe1!;LgmNdpOu1IUU;@+;PYK?(N8`8nx~EYQ?@<FK@2i@H;tly8TGQ>gLvV)2?o=RX*~Y9XS2=pwpSjE!uYqGk9#CJGQ(1wtFx86K@dhqVmyJWwi-H9Q9&fU9VlW8#l^+ckP9R!hEaMZnp|^ox+@xcip-9ybD)qZfS3EcA-7%%rEBVW+UA;%k}G(W^4(-d}n;-J5$IN7V`OAp_yOEEfyC3F49MByNS!&s#mU6KZ<}(gpZHB`C+a8!CJNMzcbT9W~x7ix906Ti}swIi@b5IQf_Y7D@=D*%S`}Qd%b+!{<u=EH;P8LKZbtt;dj={S1app^u~=^z4~`}4Ya?O*Vk(w+Vx7aUTff?RBu-7+UEMlMdZ4A&>pWG0<{n%6Bvyr3SYrrd^rvuk=XSa@jjmuxqQwd=8L2rkz>3A6NYWiMq&KN+f~>!Sk?9FW_jJN<ns2lS{;vUt-9Kz8?VegvfQn$dJUGiy;<J4TD`tq+ioBb;$GHp2gQp-@oClnqSI<5kX(hde#<%}Ndvzk&hn+;@c^?vVm*N1+yPH(r@3a|tl8zYjcNlCnGJKUz#PYuVb6t0rqVRS5WDa~u%Hj6Zs8Qs$wN2?BK9yzmY-DV@=0)Pgqx(y!-CY4gqbzA{sL3W&z$`Uw)~F~)C3~`YSXT**{d*gy?h<fRHghu1V-t&NbF!v9H|&CQD`!xW<bxGkgfySHC3`D1{Am9A)m4$YRx3F`+}c8|C%)`jb<Z$&;Gzd<*x{Wh+regA$$#n?uamt(=!gb-gq$T0-s{+Ij!4c*BeKuRCSZl0$q+~Wn-&WFRx21FdJ{&te4^6dhNpik>ql;tJel2W907hz#HqC`RuF#L}WDTw%cys*gJwtc^#o3+)aFlwaU%V0EL`p8n7$pHrm+UtgcqK%Ino;wbB3-P|5o%n~yXve+2u!QEOIgrd-#nSBplcH*m1^TrQi_wJ)#H9Bd-x{(cm4b}kobf3@6Py%Cvkyw%$F>IZ9A?G2bn{bPH*x>0TVgknBl*tlwH+c$6ERFW{ecGvxV$R#(9ZE<AJ02`LqYu8x>TC07yxn3);#cvz_OsfD6&zLXF%ohyrz$wfx7yumIPS<T)Mt=aHN?_vM?hf_CG*-)-Hrx<N#v(Vu>Gc5oT7yyBZ5v0xs~Nz-yFFko?{>!rz&ry#*>5|Zap;U;T&LIlnG-f%FcJa0k+;{XjSt{_QEyxcgTj0i5-K0TuG(^vfcCCc*DE%{$Mt$`dn-7V`J4uZw#u7p<wp2E&1)b7+5tgFm}b}@_zS(1^bQcbo7KvP;ggIv24<^P2acv(|M50(N2@osH$RA6bKOg;2_wwL8zW3zt5+&EFr?u<EkyV9YOU4;bY9+qGYEFDfT|zCJhyri4~{WV`WUNK%kbQx1Gvxa9T_Y^xTC|#n2y<q<G!s*ePg@H?2$Uo@-mO}t}zeHzjN4y?rscvhsI&IZvcP+l#Kz#hR_u7LT*0F_+G2jE1SS1m)EzDTa~MXw)4@;)+ldmtyed%+Zy~ZzYy8K_4zrzYnwG2cc+P9w>5>0;Wvo=X;dl!f|%C;;%aSsv#D@LfIQcmRhV3*9xNdr!-JB~0!#~VW24ZsZkOxL>b3G}gly?wY;3^+zyz<B8x`1z%G7t@veY(cEddrKK7B5N7lDF^75ltKK5rFft=UCuE^p1xS__NT;<B|gYb`HY`CL97;kngKlU>l?)X3@YyD50Z%FpN1k@u!4h~hN-Ux2G&lg(3|aUiN~j_Ug*NA|t1bw<^01jfZ&0My_-8#Yd_UcFgvB3f(S*uJXaj{Y^o@TeA<Ku`7**(NmQz=OKmb3u|bt{B2(^}QkFP@xSlgmpP=K%|2bR0F_oV+$Y#?;)U}=w5{dXJG!=fGL<BG`^7~4kd`+pZ$?Jxo0Q`%jt%tWMNc@%;h@n(s~8p5V2qrXy8>J_bucM&3`bJKeA0-7-oc`j9I?tZ1H2uC2PgxPN5oPtQ(&11ON-5;I-;?puiWnLb!M96%>^5lfY;IzN@uOz!Iy|x{%<33&vYm-`=!81h}w)9DwLpZQ9q$Rk#Sy=XONyiPFs)%w(+sum$q~Xd`AEKCS_pibH+~#8AimIVm_!;}&e(Hlfc)@#)l;0l%_*^MTK}6W{$@a^MD8u764Q<U7=~M(;+w3Mel|Neen9)9SVBC0~P-Qf@sS<!Efc?^#5C(20>Ui61mAQ>b>}6Dff7uSLm_@qP)79Lol<E7TxN4e@9AHP+R9ZYHJc?n*vwyhr^+D3fKF)xJ0Bbh@ps3ri=fPKK>Ifqu*A09y}q@Nm#}drStIB%0dA2sOg{rMy4=srrleOF3;nQxq7XSL(1gCDaVk+J$mO*=L-+H`(K3>Gn~93a8Nq2JnvChEH&s6Id*Kq6Ojy_a3Sn@l$WG@0sah3I~~iUeeja+wJZs1xyBPm`8d>%e@COmOZ!~)jAGG08c5C9BE7oXD2j`mU*AQBfP*DgQf-GcZj^o9*`Povc1n%`ma?>*6Xi|{Pr|blfHn(=8g2_x!HDmX=!o!97~Sh><!h{A#bC9{|@-LUMGW!F{j%HPAk}%{$T6^F1fve5(8WM_fds7JnHWmAm#Qv<&IZ2u2$C8PzfQFmP_++zi(ZFdu-oaoXgqK23zap)yj?9I-oIop|IRuxTRo#w%hTCw~@QF-NQW>WR|wg3reLUnq{H4M$^H<oMrs1*WCk2XAeLUM38V(p#&^pn`|4Tn8P7JrI+G&vwKVP_^|D!G;7Mbefr6CyjHhsrqk)HXlzc#oy?LM;Zgc~B{_-g$aUIYN{ntfo(&?I2Y^tTi@XGvg1Fw2`9th#M2vM><B8Kt?%KSf`Q?fk>jzfUI&i(xPK&Ts3vYdYFH8P+Mz>?Mvi=Pyl}y|sldx1P%O{xi#tz(klowOxTR^3m-i<&&$LP8Jl$clg9l(k(FS(ca9!^pGyp!7%^N7F^{x@>Ez&8kAmC9%|7^O^{8B(n~Aj`H535|x=cZS{pbj|*GfZn1I5F4-*djgO#>^T6`a<Ud5l9i7^lK>@}1;}2h{$4t^3%F9Jf0X(&tSh)v@}Mq!yy+ikE5)^lZsf>??ZrvFt%W>aXy+H(^R096s^4sig&dtdNHlmZ95;MU==HD3Ddz<{%SewG(jWSq7h4KX!?xs4ZqC`C%zScl-u^`6eBvv(AF<B@NhYVH>twC4!iZhKBa_}IZVk=l52pI0S?n@`O!v=uwd|}DCzdS0uu}Fb>E-lPy1=~Mp>;ZQ%dJi)cg~^t&8E9G6kw%)qD4N2Ehn1B27krzGuTH%cx-)!?5@I|Zf*P46yelF4k>{VLT957xx&FTeoLM_xPLF%41?o+?t1~@jGhVR{5M+dxZuYl-jKw48)yGb<6Fo!P~p;|O2cx1AK32ib~?*@i%ZTAYWKH9+XEMn7miw3TGsdigvSp10yv9H^RAnBGfT5hCo|{Fbu!M<@_eS{?zNm;r!_yn+<7ymN-!S-&oaCt_y>`lWuTSS0<>ir2ab0@42fh<x&sRd!+~Y&w=7^thDY$y2uJ`L4lDzx!5A?;baw!I+_R9a9*zbr7wz{v8Zs~VTio>Tl8a?v_jh?Z`0vjKeK+y~974|}&vM^aU=<J49a#pjVa}vC*0>8yMm?MWFz22>)K4csnsxHrT4k%Uxdy^ewbCdZn?n!hZ5HJR%j{1Mhez;$|5_$1kFf=RfG0-|5Rbd~NPGgpV$!<Z-h)TtgQd_5)4L7Q|ELeY0;6`jJI?S|3%}A>{MRyD&KTGUZw!z8k06a8bnWjQ!c+gFg#>l%3E8C>`W70y?~d;dMz`<bisVPjyw~&Y-5qsdW8wKn@a04P0?QX<9ox%~mN{~NI&r-*j^uv~4Kw~}8|O!VEb}LWJvb5gE<WI%jSuj8Iz5mf`!4MrKFRn`KoE1r!`@)bZz^9-f`JaZp4aW~mv)A?_lXyPB~N;;g<pXwP!Bf?3p(^tX=D(>P@3&@`@p*cdz0eVp^ay30yyr*q__Y>7mY2Nv1!R0krj&)JVGjMnZPVYJaZrFQTh|9prUr<vkiE5CA6vZpQ&{*{|sb71OGq!lR+2wOx_k+tN3iVXreX|!@mB^v^u?scMz3SgVqslNiP21LXM!*7PG<zezuD~5nkxRP#Jag6>gI+NOPy5C4qz4lh?nE?-CIAg!SM*0dmR$W81ooAlFOn04mHfl|aVbGBXIK!|o9N?e^jF^ziCuGVQL1zh~f=j5yW|bz@qW#a8X6^?UVCTEeYR(U(L;iGxZy55`@Dc?`!h`a?_FuHgTNZ9syAUEnDr9(wHfUl9eu<s8%FpT_!?Hy5Y@T}zNj1xBF3?gcU`5d?Al0Lt+b_1TM}UcB#FXEGj0yo(PaZ3Onq!k)T4xaq!Im-wF097W+lfY5<InjH3<Bd71-9NkgEYtVc$_TOs2-=15gXGw2>z%3vKuZ?)I)^~3V#uXB<f%r7WcnqU<1E49fBV3F#vRPp!`l)x@`#|ru1^_&Re%;+i>3#q`R(g&%?zT4FaYApvqKvvNpAH_nPTzL+NIr6H7o``AA9_EXI3u^s&!PlXP!WF@o$*!t?oTArUeXuYe*OB@jik0jw6a8aWOKl&4!}=8)-<3_{6h8+NO>16bN>490h|>+^K^tergIOhARIeTl&BY_m~YA1CU7qyDdqwB!#@zn>~x0h17xs}F1Z--hE9^tgkMkKmhTO_Ag_Yh@$l?>c&xhMqa}W0t3=vD!2sCf!=bstKfH<Z5S2~X12D4vKCnVD-4Z-6@eo<r1Q}Um=1OMq&^;WCj*3xn6<P}H=cYQhE;TAmgIn6Y?qPRaGLKQQsBuC#F>hU()|ZK0jSonhR60gPsB|Q}6VfNsro*!VQa!X`lVCY8s}hWQ#T$=Oynkt5CxMEeOBVc#)4D?6^n!UyADf7~D;7Q+;A}}N5#8ebR#q^gsYvf**xR^XK|nBbknx`pxs>?F>=q-M)WFN`yW?TEor)R~NiBBA>GqHg$=B|5N2r6bdG4un)^8Di`CXwN6+YeMy&Li~mvZ>)UmLCc(O@zx8VSP~A}Hb;l2C~e!Rz-eJ4umK@Y!>cu78bz4?+md#^<EpHv)xyR^pp8K%r+k-V9I4OzWEi$C7I8&zL;J?4fhdOlRpXd8u^55HyT4EX7|$u(ikTJviqqx~U9P=_y@IMtFZq2sUb0;<>)NDgB||${D^>GI1`&;K+d)PWj&~_%W5Ua7<{N&WOR1`UQF#+MpRRN4n*9w{Pzq0e-#smPOW@*xj40JYgqg?-LL1<lFbo`}$tt^5uoxIok^DrtPSEujLNMMyl2zWubM`=}lN~m@W<-&(mOnerEteC1Dk_NPj?f@;rMVWf}nNemjNeAS;j%wuePQiy4^jY|bjAt)!KiBj?ceSD^OjKfFLEz~G`0!wq%7f_;{Lquoy-@<no%PG_;P-|>kKy%GOgFK?I@t|yr_7pG&w4iHCSyLa&7py0OO8|*nf1jz7@DU+!YFwEvL?dM4`;ynx^!!eG#bL&!Uhi+ZkJu$9#uc9lwWz-vupeY}UY+zbWzYlxSa;Hq3Uez4K?t5g^i+VZf&w!QC8+87xgYKB<(-~9Z;liM+qV%o!S~?V5V7a8k6@tfO;%mg_WyxdN*T`Z_ug--v6xSN`<44i5CoG2@Ju=W*nl)o~Bf30QqK&jr&4z?;QShnJL_dsXb)!;3O;Pr6&>xHk{ccM|(CD(zsIY%=b*&`yT3L26gk3$RSMeXa50G_o|5%~AC#JNX0f-;C_uAckU_4Xlonn4rH#CuE1#a}%Z5{N{o7OCv7+DTuv-$>LwA<;nLXU?Nkgh}wfchMLVo}g6n&{`TadlmUspGH8#agdjFYWBEkP}~{bbP{}Z1gUquT2bSw|`v5M%6}R8=k@^VIM1RuT`7$y|Pi>Y*tt4#~*K(>-cT60^fp>0;ex(g<r`bB>2{>HOuRz+{#7;-8$$wdwSI?tF`)?^1z?owd(clx|q|}dbR0KDHs>t*rwJUliY=myC-Im(B^pJi7g`#sH~Ch$?<3>f&c%O<L7)IZFDlRufrLk=E^MQPDm-#MU*1kwX7u-X@yxq81N8?Nl`G3QazdN5ALR<hB7-Dx6;|JH(<#)m5v*RnTxaz0c+u3UPj5RZeFX!>`-vHS!XzO0qY){&^1vBoP-sj8geuEuVsqf@xdr1z5ui?<Z`<wakJYS;aDCq+;4t9w;SNZ6wl%vxv0n{bN~m5UhW<oMVG8>5&UYI=vWl9r^QWk3c^2z_d-Gh4vP2P?sR)@3d6;sgC0cjL1aKQ^`gUkpWX%_+cVRs;^=6w6oM2vlAww-)pHK_+D@@UY)NYFaz0m>yL=gF?w$#!eUP0D$r!q)7fcRRWD^CY(V<N!$&0Q=yzFHi53L_)O$DmEZxTNwc_R8OC8d#h>sB`R*KjRNYvk^`_jH10e{#47EAspsFl=YX0~;Q@BfzzuMJfdkO{(sw<wj4yb2_e#4%ctcgTI1mf4nef#<rj@v}_3*aj&HAl6I5s!Sx;Y9gtrB20z14CVu=+_h^66=u9^6l$pB%(*XeLL1S`kQ=0V#cVT1j-UHjXWA+`?*2C;jP3`AOcqHttKsMp=q<<UmM&I0xtf}Sn2Shq-gvivu->L|JqJ<{go*WKAI<!0>n)Yq?NNNlE-elz3<mbWR#+ERLlwh`L-w{N(eWa6t%qAw!YaO_UPO9CB?yn4y!R!x882XdM>^Iov6-I+5BnpA&XqNs`4gTT)qKBb~Fx0y!!H!Xny-wHdwY?A#kvEgP0oolD*(Zm6Pji3UrFLhhwVQ1t<Y_rOfLdqR&EoGKoVkr&i#9b+wbMmoTpBuAdZrT)N$86906a0=x>2-~E<*i<J0b7uXczP_sT0o?tr$V*8u3<2B7_~{1LF8wX)7iO$$Jf5GSKk*=_Hy;RwcXLv0DdlBRqgHi!dyGwMT=yUI{HNF|}6kGYPx?SM-}f_Bt_4A*zmt!C0k0WMKvW&9dt4P4sYNN}vUoqE{wDNXT$j8vZo9#maZKq|tuT3@p(N|FNWp7y4S6Kl8?I)0%j&VO=tNI-?`qO!WGb2&<E`(B%jwsO)u;fDd~xhFg*C_6Dt;Zn1kMcx5+>Cg3DZ2#!e*faCI#rs)t$)Hd694&A8|JAjuD^#XMHO?)~{OCc8n_q!ziW)GdApwh;jA_5zZa7Wts5TNJ_+_ud!-I?j#<9`ZrzH<&Xj^OU}`JJO^>5&x*crEBZg}`9-05C0zJg__$t|c4v5UlP@Q-3fbablP#w4eTtg{ZZ7`UW)n+`$Q1MNjkXMyIacWNbv(uG7~KEkH%COtei}8(;D)<2JG&BI0b~hb(!7vXze35V`Mo+1=3{m!sVUxFF}m(2+18T9B}_L@j8S{1>}FQZOZOqUlm87t^3%NE_-;1b#A(2_@o{JTWa)i>bF?;*hkDOnf1Ld88;^?t?^oR1_Q3>J2<Ml|(`GSsajP{L~S<inFqSWYG1&+iF4ggS)nSI2<2|85CpEPP-#7cd^|;u)<V@*GAA+O7D@VTsNdo$_wct2{Wc5SmK<nm_>us=@Ic$zGlY9!6nckkqcz6m%7J{3ZYQ?j4d?df#5@jQ`Bdd`;2qmJfj>p6xQ+-*)oiZF^;jzn7O45NCfo}E_+82fd$;}KL+AI*#jI?q6uZa$zdv;_Lo3tXEWrhBey-=@6?q%mOftb>4JP0>FGqXoiJen0(DCzU=&HjOc?o<a9pIwd>JhA5E%9k%k|CZKvK67$mj%@W|08X#2aE3asT{lVhX-feJpv=bh0AW*T?!R$GtF%k(~(i04BQko6@luhJHUMTs$fpTg{Knb4Qx=yKw!*FtB{~9HAoyfZW+Vf36r&PAUjfa$y{Iu)f`pp`f7&Qcy}l_^Oz@(C%t97#Hb8#}f@p8V*!gW<~JF47zf`_%;fI=1ht&xOAQ{bvRj3!R=u4-BM?m&`LG{FgQRBL9hTdTsED^WhJ3+3DC!4XNuN|A~D5tjfxzDq__R_?CA=bwf}hbPiOxY_O}57b*Jmzovr~V$s)2mEtXdwhjBa^jYugfysj)(>{;lR(j8%hK}=gF6ootd!miY<NDUk639nVkjVfN+aE=dC^j_jWk>#-3G`Qs4zgZL$3u_`1NANY1Vj-qn2|nRZCy8=WyX)-t2j~J4v%(#^hb3Q~@B<j2u|HvZN-!sn2V<uvj68Y0BELd^;Pf=6FJbm#cQ~^+#Xqh!oS4OdmxaHK;P4^ZF~}MP^U)%qKhJ8F*JCzwALihjPfQL~l%3u^=%R|Kf2RxQl<^KRx<=oNSR4S$@yUrs@o82zZ<>}OKah<qE<&XBZe?uCcVQx<LH`gC0-)~W+r>M?MclUTkXjOYRJ|Q<zf%gPb)x6S@Es*!YV&4E1;_;dn|}#s$`sD@MZ-^3^FpTkl*t@lhTQj*tj^(dnWwP?uxdIWO0)7(c(M!k{%mMs|1WpO`DLG|8mNeG;Ufbb`)yJz1KeY}(HTnnF(U(#XPCyYBM_(T3=MsDYK4YIZ^ggo1f`f<7n!nCk;doKv#&+E;9p7Cd4neUy$BKZH-S@*2KgP;RTv_%S2#h=*^_L{zmNTJK-#j5gYjY18N15vt9W1Ff*$!k+5FWqZZtR671L04i=y&v%8St`f$)dh1V~RNyqGV*7vIW+F|R}(FyxivKJnvfz1C=m*e&?VrlPe<qk0|j=k{i`$!Tt;vM(^o&B~@NhylD;`KXC*zH7=OG}KiaVkS4MjqUO}w?wTGeE4zME^n@>ai#Fk4?-6~@+voo2|cdE^M<`%Tm7Kq?`^ieK4{%eX<Ig$wJoJ(g$TH=W$9S;oko=qI=%tj*x9uVb9mr*$Yo*pFZ%fUd41p++|+8lwz*N+q@UXE*dh0IN<{S-0fk^uIP1QoaQ)9Im$7i14f&<xHLxl64F`{!TGgSav<fXnIKz<I#WIPm$S>&dXv42<bUM#v#e?2ppS~IXS90}$B`Ks&L9B5K80`NIIO6q6Nq}?;Av$HgxA%TDo=@1>AH4qTy_voD&9o)nu)FOj^An?f=ZqZ|lp?x!pS8p586*2-k_v;JhSqx}K>w&4O5I7vG)MScD2hT9!tYg*iIXUM=L|jIz>wrb&o>ho<D(=B;neo9B0$^Izw7NpfGp}gtpM0L{lq~r+E>4XtT8kVF4S&%#ULPLl<!fC2yWA^YS<c=^10mQ{*3y1L}xAhm~cb^_q!|993L_4M+_E1*;P`HO}uLXXE3%{%avyBPSA!ri?^W1??*DTfS@;~`J-+J)*zgna5Z8mUK7jsVn#_~K6OU<luBrkCEdL6m_h+$#-K;sA3lKFbu1@P%;q~MKQg1EBx8Eb0F+Vu0ok5jN<Vrt0{7Yt6yXnn%YdJSIu=b}L@9h!EAVc#cmRu3q)Slqt<p~~MdFF4GO2}l!1V$iadyuKWI_AN=yb<3Ab<(&|4idX+0XGAu`m)3K>M84LZQ9qjPQuwYj^J$-uS5JmTq0T+ii~zipqf$;6PS5T&<YT5AUt)VcM)wW&{*#;uV+hiPO5hPaJmf-CSpRZ)x_{rT5>Pfo|V7qemBoAKmD)xi=qi#p!ns>9Q9wr{-{iofoomp20r$$i8-AQEo7`sLqQBV+j#$*7aJ>kfj5-mx5&OCK$Rpw=UuTg%;8o^v5C}RM9)~0EK2IU8{__#4OM0dl?V&nXSM*%kY|)mva6}i?C?J8(?_xyRCM6zTJszK_1%<oi_3e#e4z2;36|}&$4s#u*&GpzejU>|3BPzk2<J;^o(Gmj*oLcIzArDP>CXa0-4PHD3!_Qb3aO-oB%RNYBD!VEly6}o8d8HL#taOC}*+LbMLMEWCG$t=O`o63KYGe({eL=?)a|j_E+|uAs#m#S)3$Kt1^Q``aS5Y|NgNcR5LRRxni!+J}Js4KN1gM(%6uJV(QF65ZLebi_362Ib&ANOBE$zvCE7ua5{wN`h@>f`oQ?lryz^~FEAJN2Do$XvfI&I6A2IqA?Qj5nC@<oljq^wHTPbbNa*4hBA2ePv@p9j@86}1ZysLg(C+=KFSMO1*{O=gvF0K&n+x~w#NW{nM}g|C<f+hbe+&zS+F=6jSq_B?XF>ueKlITce&emlj0eNwqV}YY6^wn684&jIx<7OI;FSL7;cIKq8;pwYF3d0H7ZDvE2Nav2F)BD4$1#=*H#%#a{j*ANIN=u(-~SzR#It|K1FD4*O-d%o+lmqQjG<;EN;3g|9)#x$G8dtcS>1++XM~uOm}{T69=RR(dyoRW<{PbI9%*-{_?2Fq7Z=tf8cEnC?q>YAc0v-*?tA&{ym0-A_muQf33`mF#-l<98ozKJWB}6#UqoM#np6^zqH@U;U-%>~V8U4_{)KGK6J(h_OZJ(*Fx$))zmOjKFLZQ+UfcFBLP^<v1zl(loEC#JRYZzN@KNv+_EHkFM|2PWGe_TWN(xa61(RR=iV|RiQKw9;(04sd9CP6IV1BWLM-k(2WNKM5J&RbU_elg&7OV<O#i>pj)WJLIW(gF2j(<k{g%WkOqW%&Q6XSp%+JC%kVmhIdOqhxOe0dfZf8zqPWB5MjLPq+R3@JK~r1(pEr&yTH=>bq&n>QQ7mrqNofOYO`LWV5`G9J!=KB@9&lJMxeW$2Cyt&VTAUBObY{fcI``iUS@UFw)S$00Xw@|x}7*VL^`Ke|!;(MIt{ja!$}C%eXR(DkiL1h^PqF%x?1uLgfAhF{^u;SZ+vASIroolNOS2kORl-!B2-z|Hb8PwJxD%=G!Y8H|H4y`uSr)p5I}aHw|3|B6f%`1|zOpXrf+bVe{7Iq<Q;meTOSpY9#kr%_ZdqGu-eLHJ*GlqDB$(=jb_k;T*z(n&VS+dHI?V3!}&H&=M=Uo2L@I^X$AYJ1~I(oz+_dJJ}v{-ugogK;cX^*n_vTx1wu^Q*h6z9u++TTgVhuq)=RTZztC(XhNt=P)@cFM%zTcQ4UAtC)$jF~UA25~<vx+yMK+$kn3P_#d>qu}|mCD?H9y@}WL*&>vxO+RZhk&}*=%AZD{Gnij`zoO`{K_**}VU}T?mra;{k{_CBR>@c>`6ixS#ic5aUiLhQ#<o{PB#6JzWaXljz>{#}7A`7g-yI%+&3K4^%gGK$crBi|KH0dq45Tm2gu{oNc07j0$b{FnnIGoLz795m(9sm3bZE`*-8=4<>#~g0tM-E~5T*?%oW~;T0jVjvK&`Dyey17|d1CpbO5Y?b`R+rd!7UvT2IfMQMkhtATLgVmD9D(xzTo~(UW2HL5l@S0!iC#$3{*rxfB3CHcy(U_&%H9$qUUf#-oR-s$@<bF=LXMW0mr_UL!xqV`2ZxqkNx+1W*=%5)2;WQ@pGzuz0qd3vZ1SG##FJN04v(*8E*w!?EMlRjyGeNqWXZ$tsE#dPV#+)4jMU+$k?L6RV23iRQ+C1Kh<J4Opvwt6qq}q0>5fwcB`9v{{@6Dv<$CjKrQ8(hEfjlilRl+DNPufn((H%Tc3hBKIr0GAs7h3Q%5P~-8;En1rb3B^!=7Rx+pv<kiku4)@(R9+n2r4eMdyPn9Oi!k(Edi2zeU~7m%<6;VNm}_NrWW|E{kg9ew{{9<{LgznLr9ge!=q*^>`mrGK=R1MJQupm#?wnYsd!zK~QEQWoKecp@Mnr#_Uy0YI5rk9knUP2w>IQF8<Mw_K1%#zp8T+mYLQA2R$jwB)SH}%^jEUO8QetXYeZx;(Lz9{UI^!VA7z;uELsvynZPsN)UOB;R<d$h5e^>j<-lgXl`tBggv9bJYB_ym~7JUYdW39gs<solU2NdUx=KR5B%(5B1fg~CY+FaHJsC4<ENwmp74Pb<!rJ0P_PbV6p(4juy*A`KuWM}-vW3))7KJ+6%xQpyyvQ<E^s#$j&?SC$#YBJ-89w(?QGbK9k>OmuS9{5W)XgCK<BQP+A(wa9Q_A3K35PQcP)w;EBRrf5qfW8k|m1V4E0sW73M7XXWoBb1bjqaKXe@=o}rgPNVN)C|5Y}_MqdQUr8{Vc7Z6mcGDCw8_I?6fSz7g9i+c8|8)hj7paz?%?u@cYM+w)uh7rWdyYW>0{##j@H8k6w9CGia0$qln2}ga{oQ@5_9oo>%Q0J<-<>F@~Ab9kY@<c$$g=U8@?1Z3XH#;8C$mvok0ZgWM763`8s{l%-zb?)?JPq}*P3%OS1O~lmkF;YcJ!`LAcGB;4`?qBxL~T}neO?h0ZC0E}_?4i75>I@uX1h3*D1&nGQrIgtD;ngNA19YTX!hE<)^aO9SJ?XtXRn=I)MT#>j~&TgJLk4?a|?5swws&F%(dqiGoAUuLZ$#83$xDB++J?s&n*d%K*GKcYRaD~W%a0lyr?BlSY4zs`GV6I3x^L;XF)~ZPJ3j#xq=Nxj0wn0vr$>EtTv6+a-*5*wvBSbxLK~R-YD18#$dGX^hFghklie!S^wCkhE#qGnpQWPmFpF3x`)wU+!B9`OI|!czl~&cFtQP6ga!-$WJQ{zwJ)JL{**AB0~aaumS42g#-Df)Opw}jdgCL)C!yxypW=h~`w`6M4kk~+PS$GMSJx|YXWTab5~;#euWcA19Qe0s<HH-3dc~k*218rqtxs{F{%WnfUTLgWQs-=@WorGnO;LR2$=UP!L0xR|k^FE!yH5Zp+?wxmj#80<2ms4Wo2t8_U0$o(6#%Bj_QpmT_5y*jWCTDJIn(pb%%B=wsmtkN@hVm|ZQ`0n&Rve!;~&@1`*jogjsR!q9rZUaah#y52=Ipbx^)R3d6<tcalXd2>Uy(MH&T3d;UhPy4P$eAeLao6i46QQ8kAS|wjIA`<cwN9`t+TWac#X;tEa-x)1mPR15Y2_-yR?8BDcr<Ta!D*Z*S61<XZXtgr95jj%n`;RUtI4er)8e0&K?`Y!`fyU}LPokRhnmz%Ql$i9v4^{l#$H4*}kNAol2>PPss^(8CXK9{l}>IFY_Be*q>|)8ExvZ5>TCk*P?KN9Y9B{E+cgq>X~F(m;ryZIZ54hY8Q=&@%sHDySBIpUY~|N7BdNcY3|_Un}5iv*+oQ6-LN}c7@5L<Skb?rNcK_VbD#|GB>K5QG9_f9(Dz@W?>CQZ42ifx47VKWkePsUDZt^HFNo9dA+*EC6q7EXjIDTMYm?HxrAFyN;=8rVDur{H^%NEsWpiW!zBA2+&F@rs9J>GZ>~<dy|!eqR%=@yvz``ta1VK}3?<%guNdP4*O1h1jEYK1xiE4&lwvqDCVz%()Tamk;T}Y7r4l81rDU-DU}`d0ZAU^M7j&d9^5VVuCX6tv=yza7PL$&VgD2*d(88_&&k$F(_=7}7xnMOL#M^KPMMG4}p_v^FdTp1Ap~14pqPCj@VsAS^v(ho?U{ir%ByBWn#&KvrPmI*HT79G3G(<44WndA)I$JP|ziTwt(xzTN)5NsTm5<5*8X%Rm*~!zubpkRq_T0{3<QiCfr8gkOs!_-m<~Oi1xq*}`n=I79+A%`&+Isc+4b*|cD@BqBe8A(N*x3XRZVpXTw94=`B|O!H%mC+_v*r@jqjYSY?meu&6xRdckKRvvwm%)4O2FxKK}tV1h)oyii>0fi&f1g1!=tcTX(wV>=(ffn>Xgh}E|)hgB;_TL(F+lJUNX=AM-bkp@W{dLeNLf#LWgTc7=V&V*0pCKG{Xe}{BMP^IX;QV4W;~Ilx<fhTL>mQJ9V-O4<8)(dGq6PL`Xs6GgW_!&3Sp+oM(eM&&B8bZ={5H4${Rl@~HjHcR+q2OFR7_+geaSR}H7pL|OSTgh^X`)szCJmXhIQx)DuxPYuFbRQZ<b1SuY?XQxDIOe?QmJl!}%i^A#C$wG*;Y-zp2<WGJ+@+1lwT50rlrY{i3pS<QzMd2(`Rlh<wMO(;YQO%0Wo<2N#ZUS{bXZ`+)<lH<sFw7|xU#3PxwRVs?#@4o*&bB)#eM(%iF>3})zf&rp&yq<pqA9&3bWNuuZabIEwHyFwwD=g#K5X_rJC=Fwp##lZ9#%IY=F$9f&L?Ka0`&GmgaZj}6xpvNpNTb3LS)25rtAU-2iUw67rg=oBf<%q$gR8~DvssLk3kGE!kz|TV0J$QfN=k^B4Rgkv&7Gwmn^rNjY)y{;@d!qDzj!tOZR3Fb3tec7!9cyo}0of$m~fPOe^-|e7$z;0O3R~_7<tzeXOoy)s#HQ>T;>lY-Oz!w!jHrBLCVbtQ}T&z~CZ}@VI3NoI|D)f^!NHz+R)=U&th`=O$58OnTy|)lUsqkVZwMr!sFfQf{~<q0-!5P)D3|9VAo|=Y$8Gru&o)22oq|;vB#(b!}7EHf%vZMKqGq)(Rt4yE_t}=qC%nuh+AG)*VLWFH8sKm6WFJj6;vjbJJW=GZ7yQ_YQxhM{iUnlZ#0iRDfL;v2yQp+f=4ADoYFoW6s0quQSqOGNcnO?WCw#urnaSJHw$&p*@4XJFx<*7~DK^TZ0iLB1~EeaZ`x1xG5%bJ#=Gn!_sRWXMqjUWVm~$L?XD@Ec_g*5{p%DD>tiHY^;)n<3hH3Y^L7;H}RfM`j`wZ423jW(J-&jY~nqNos+I+DctKB&QPUp+yG>^aR!#}I)|~WM$tZK^#SmOdx9si4oKuL?F{KChL(-zAUf3-!$l{DA#E+c!UqSPqyee-L?&kM%?h^yb#T%9K(l{ImjY9<sTae-m9uZdkWv<+L0EoA{*QdbZ4AJAKNp?(8%<Wu5fK4q|Mg3?VQFB>ZH8le2~xS(TRjrIHtF?@r3|m0`tN!MknT|nCevoV7>MW*s-1c~jK#Y_wVAptgIxt+Whgb^p=<Jsz$EciVyO|o$p?yso}Bj7-?B{?-v&l+O_ujbA(8CfL+7V?Tg8)sC@VbssG}%(q=$@J_;Y$ie3&uWF~F4d+S8g>SxS*F_{FR1mGw%wQQ`de<dkTx1np(&M=U~O9>?5d#44zKe7x)OW_kT%Oif~=)(eq^=vurXI;V0csXcS+gAC?GNf@K0&w|WTRrB62YCQw7S-;fLqpxG`rI?#->tYt7yFA0ar9<=Ym}E%FP@9bCQLVO+FezMFkdTY;bdmc%(QO2o{!0l&HaIscGG*&n5X_=K=!vDCMPw}D*g#yYKWw^GS|a>_sATBZ(oqqi(pcMJ0us7N5Bdm-KVv6$<aV3c><q`wnkTzcw<W-HWm*rQ3xVkbIYaDdI4yM4kE<Dl1HXzQBcA(kZPefvtR%mALH?6bE%B0i2Z1HQ=~+bZt(Y{Y)LcG$j;_1XeG(({sCKas%O46qRFtdIZ(9d^)>)ieaF+7tl$7?HO)n`uo0a+7xzaW}O0ch5xNeRW1nY`iQzEDP_sALl0`(EN?#7LBrZB${?DE;al42Ac>q(RLnUW1P?8pmUi%Mh-w*@rJ_@r5{me(@%?F>EsWQBY)up+sXoDFi@SL#$h5}J{Rg7w5G<p}3ZUcg`Q$%G12(*%ffe@c=7u>PEXOmr^vxiE__DD2^sQ+A;;k9UNQ`UMw8|1O$W><2DSJvX0o7FtV%|0NDSp|QgbJ_~bmiwhlRDKpnv%xB=UlW8w4E@WD6eqk<`@8srZ=iki1Cn`wGv|@4b?K25!?7_@X97>PE<?3Y?7jeoMXL~BNW|;9R(P4cbc$LYXJs#W!>>eeS2^rThk=p)Xu-_veK7eh(8chHT;(=)pGx;8r{1z;=g|C8nPRd2(h2Yr*na#65$c$eM&jkK}`1mr0M<!=BM3di9qc4LRy$Y_tymn#7>xV{)FSVD>0;@*b0uww>2&KKiLNmS$N>5|~lZ&qJUqljL&cq|#h@XVZf)OfH@lVm1zUhL^4$VmG9MP<OP}wYnp(ptHDlm~HQ(j3d=fEwop6v`$Q*Xw+f;nO9m93hhV8L(;S)e!D;J6^QUu-}LW!;e{qxX-^@??B47zw==ri^o)kvjs(k!NX{Wun3Giz?3D>9@tKt`MVpZsy;QGJN|o<zY8-lT%AVpJ*J)<nx)rl9`<76t;kCFauIM7__a)sApZa#7a2gpG8!W8y!y?4PL3)oVAe4rLD`CZ{KxB`!Qh)15`D!zQ{v4V-!Ee0XsaGF1|@sN{r6sE{X~Gs>2!g9x<Yk$f>+l6)|M5h^paDITk%#$)C*3onL2;K(R!pEx<wDIJs@&uiQEQ<8Ph#8|+--Do0sPb<1uP7+XA7ShnVKIrv`z{x=K%o12yvfDR$l%Zi@{euV;Ly`Jk1Q-#Zy#Xh80FfOCppSW)_t6biigW1eYpN&WioG_o{_ZZD=xg(OuB{SPG#j)OKHn%9y3;H|}1_6wwd1!_3q>qd!Z3^4%5&|<;jP3fm!v9REcZz6olpWmui!D1+bt_@bFA1C}>pU~X<Qh%MTRvS05gJOJ#4kMJc&!(*5<GEpWS{snOqt^`q{b*Q2}$IfinNTx);?0L^M|BVj`6!+D$R-@md#4@!&?0VWK;e~F-~8zu};|o_$oORrGWjb07pnWQGgxLG?Wx3ue~`DL>Hc#e{#XjBRu-q=E6DZ<Rb9vF<#;mb1DW$IuXL)fLdZz%1T;Xk1gZOl{{rqwWJctN@!p>JT)Y!Hl<nL14Q47m4O?U{K=cwD}c9=JYO+qfcgP3?#$q;NRY|3MfAbdIm{U|sT9(V$+0Qs?vCjlIBE!kqBHAI*&>o+8#iQ51|Z8I$5wBan-%+dwRvMZS|(J{f#KS8)7PstBQFZ$z^Ho%7zOj9tksA0Doo55w2NAv8Ug4rRiI!d6Oi4ns06eyKn9}c^Oz26gcI~i`pNp&+B1nkJU-E`G$ncwZ%w=dZXSB<v=6&|&GSR2+aEjlQ}Uif=|Fs2qD*vXJ5IU48EnWZ=fU&1JPI~@CgURgZSp1G^Mn_j6;cXMzJwp?nmt7fnTe8UmM9DGN`~N(HUk|keiOEIQyi!;hao|eG4Yf|IEjl%l<xj790;+b@aecwDX*~^?H97HEsOjhk6<z|bbEO;WpD6jb$3EB>7@WI;2jjdU9FclS8piqG|P3kR2XQxW|y16`9@wB&0?>=YoIcO6cJP=_veu%M%EQuDl~mT^y5oTZK{b%VHD@IZNOLdUT?6+)hglgltK%(Q15O_t${x2SrRG~$r=MJNJIEI&>A`;=g`B11#Slm8+0_qI);3?Q1xM^-zk}>_%{XZ^g-Z{k-Q54w76-Q4*eNOERpcGaL;0rT0|y(E1^ncVJ^G3FxPfT*F4<u-`myIvxs-~uB74e=E^(!e%{!?y2{TfdxXN&#M62iJdYKFhzuSY^zsb_Hgj0>DH)Qf4+SVlUd7g$v|jjl3L&F0;P{4u9O=!Q)HiLW%%VU?#j}kaRzkKCihHAz!|rt4OX6wZ3&{r}XlqLRZVf5ZR0ym&%amxKBe{vl=YNo`niU(^6ROAAtb-fsqeoTyB$tR9P<5GXr6yAc`@2yM_fq(hBG#b6?ds-Q<)g^$jG#=xC{#Y$s;o9EYy66Rb-i}g#&p1x2-+V=obWrPDX$cDY@)vx|Lp6{PL+E|OO5EYeS}mr$dl;_DSxE0S7xmf)jX3z{y7E4r_rp+G~*Iw<shqOO4--&-cU=1TSUtUvr>D4#9jGWl)NCQUd$HE>HF{ps&I~TeJ()!RO-J&+aNbVojsZ#nvD{Ty<@UD!Q5Rj=(&CMV=PC0?1*15xyx?RVZ8(jv;6i+Xq9MSQr;E@ac-H8-c3dEKniKh82E$^#UD_0G768YPj(>Gl3a=w3$}DST{PU_7UN7pznc_B_fBbHPNyOb&-MQd?IrE_V}g!vk(RUGF2Z%l<f1CY1j-5|BDAG(;=fVetX{)v5+#*$RMOy!0}~l;4xN6tgOZ`SE7Rf^FUD?#5DMnKAhzh|0*)S1E~ioIc6zF^Cpzh!+q>$82xF8`$w&X*Q`C|QDe477^aI!+v=@;QM8e6+X7y2OWa%ntQ>jq)4v{5IxmXrZ;}!gKF>f=yvdxcoK-=BO*KevcKLhLA9}6Bor!z#lEf<~QOSu)1I~7&(rCc^4r;w=nJsMl)-M#5~;HdI<Z_v6;n#3$+ZxGW0xm;HMo}M+6-oId;Icl3;kt@1FXUh+#cYs+Lqcszh&3M01JU@k7tniq~Cn!-)kQNKUUO2O`%L!K7;wfA5M9=qk)u|HY;Z0BRdd_S^Q;VeZnmW@2l`Pd21vn2o17WF64Py333W^P)jzu!MT2Ex`TzdIx*kHYzD1pAOYQ5M+4B!8bJ8m6>y#`VeziLszq$Idb@s2mcv~PwHo_sf*s1{<@b>geBkM;J6kmK5kRB}_+B-J{woXbU3G~#lS5w?_oPe@8Z5+<v>$0(vnRuR1OYJa*c!h1g{Ne$LCg{z>l(442ThjC0;9b97I$*5<hH9R<1efpk#xG{<LV=lhs8@3hWdH6=^5M}frU_Lwgx$M5lrpI>eq8p#&Ea^jactf^Wn`Ml`_d6nkk!1D1HENrN!rQ$>yYVS;sSg4HT-?ccilGF`7?(~l^~kws;ZGt1#lt94n0jItD4WX{SX|s#rfl?mJ#f5(UUx6cj9V(iKm{&Ya9sD=-F>t*Vdml7u2KzCQ(c`y(I|4wGiTzpy^~F5{~VEb3A(=~3mWjtw=T)aaW>t!H1YKH-@pEn-2qT8n~J%`ouec2-8vJhsM}Nob1?C2bfx7SyHYVS!(NJZ-p1rP(Wg0hnwVyHIO+~Y!c#FcLVDWvMUV#fqAcND^BEpGH|BSNRd;qI@4kD^`^cvayettCew|?-l_MVd?y|~L)PhAP(x48ok-vV`zdKWtw~E}YMNw`MW339I;ga%|CO{VYDZ#or-O-`jmLi+ca#{z7E+(GaF2*6uycmzN2PoinG>_pmU{;`K@(4Ga4ki)9LhGZ^1U=r2#cV!%&lpYmMPtzK9g!cEl&TE!(*e>l#Bf(<#51H?;>i#!oeUEI7&B{61Ks?LfHzAA*Yd|CK!)BKisdlr>0k@ZBA(O%NWyRFQV1TCIYSi|;k{{)aUN`4;C0P8OBC<Bc!t+P9iF!sUlQ91Vc%W#wG}P0mk9S_0v6`RJ+C6F@gC`#DEDsO#>&EIn5xuoqBZL3WvZ8Yo=o3i<)E;8ivYzhxMn9uGJYM_e9Po}54CzEdhi{g_(#U^l-Zpa*Sl9&qzd*0NAqylBqs_yWm^Z=>qL9Z3o;j|DWl|Eq@^C_@raBKr=j-)FCwaD8gLQuT#hX~M!e8~4^2(n!=SapYVqAt$=t_;x;m{hh7qIsEF5R_Fyg)_eBr{He?vG&%Gf4$fkc(ia|4ZC**6iL$cD_yz-ibrK~5s=W>{}Q>qi*YYNZLdRp)W+A*QL1ZKL4-bXz%XY2gr&w^*M}UZR-=w_1l|$kq_PLboo_9b)+P^<fm@Exm9BOc}?@+D~HPn;~jXW506_I+})1BK)>+{`ITbA}jU^b$L@%aTmpS7v&iLC88UA89z9#92MupvHWu(M>V*Nv;PX)``hV9EGjon4pXTxhucX<dM1+FL0U`H$s1#=N$4H2Uc|fciDLX1l@i=k^PXzlS2k+Rip~28C@?Bm{1>t{pURZTPh%-m1-mTyA)&L!_+;|*d<&6dFzW83I>Ppb1N-h^bbD_wxE*6l?v-{Fg}0)kckhiyjx9R@c4TLlwo2%iM}wYg0!1|syZfT%%AqqJb?^DF*!OzgJyRcGc8`LC#UkR4fg0Jx#8voRyE$Omu=@+JVLDuJZzs2_5QFpZ0;2ukXoPP78@In<S(P%;il9p>4f!Z+zAAG0fS2NivVlLKjKU2Fam!<2y8n$qK0hx+Xu5=!@6n0=bATnlJ=r&g387EtP4L|PL%*sMo~Ll}<Tl+le-#m6_&Mf9!~^ccEGBfNME4lphA*Q|7_kaJMd~0vCh&PwY6WK%>L3h!_;=dt{!%o3_`hLC{+DUJ@lHh6qX_9r%qcfbr+1URp?*5)j@-5lJ*CK`VNoGCO@?c{FVuqWOoYKZ=JIi&S_zV0%g2JBt&oxmF?7)RNdk(YUw3j-R^qfdi+9SgM1ly0oVy3^NWfIDd)OUoVMr6LPt-)U+r;RI70NzRItI+vOVMy?_>BiUra$NK3WR9Fx|yh!vgef?sT`xl`PL|5bHj|ektD^0LNp16<Fq6>qlTdPlz&_%A<l>|o?fjdUJ)?TSiMo%C=-xq6gK)K<TAy$=qatOrr`l9B})(xw(L&^-G0ivb;-Qa3esP|R|~UpT{5v!!zGjOLyNVvx}@pEcLm*Ej-YkSwgHTo$WeAI8=h#>vqh7Ep>G00MiF|sQZhyMHuDNSj9)ec8gAM#jrA(<?OZpRen>}SEkQspg_GD^#FQq-@lgqT4vbSidb{h{PHQ}IdeWBWGa6gv&9!pFuGc<Hs$)mluBmpIk`u(wk?UvW#H~x2S8%B&rNOiJC(fvCa^?_%;9U?!@opfFN&^a*;+zg+zQGg&Nwr?yu&-~I>uX7Hs`%|9<$d`2w+N;<9@d-=GPkS*O`m{F#psQaoggi~w!OY?aVD`>f|z~>RP=j9GHQyv8J6lOv`o0#Y1Q+tH|dQ_>IVBcSLDv=^Q<JGJFQAJTHgk7HIJMQul;;^NV@RS+j^4~F6JB;Ud&-eIqm?{)8C6C^M8nJ>Q`?n(_H{hzs*I|oFlF4&p648thM;R_$;XoPF`d;^dgDojsupa7~*d%pLZ*&e4M3lQuKrWEmZT2dA6d;z@dG~qD1)K*hO9lZ?)1yL4uayGwVes#JID#9U4gpu0=(LwG2tU&7x1WlE^wrI_aI_d@ctPYlO2^ao|rK!S|Tq4HYH0{O?wR0PgRUM?^l(mqrJVxC%*~K)21h<TcDx2mdaZ8ZJ_4EHZor!|PO`jmCDR5k&bP4%!nuKa*8{CY8AwPjEYhk_$^qRX49!8qLa@d43r>%(#h0$Rb>s-$nh|o=8#ZIGFZ{C|b<8`zxtZl)=8F5PU?heJSb}pAx|kUP|zl(tI-8Z5EY|m2K^#S8c#u4`G{*LrOfD1%RH@8|))n;QRXBx&)AhS9`@c4*UF^h!}P=9gBabVU#pfre!D&LY4?!@K!MV)euF#tk7(BiunRuz!fy@ka8{|36oMHPb}`Oqu>|j^Gl0MIp>@xc)!^>QSfK~fN}6b2mFv@!C!EWPtsUFAlFujNu>ymIjH6fGxG)AVgZ~80e{Qx{Y3i-z(}}$Da%wbjy@Hb_f#maMBYYdBx^`SsuBp{$_kl0x!RR*w#0ySR>xTe8Br;shuq~wLd(~mD+u2*{DKM34L@7`17A}wOKyk9Hz=FI<DmS-OBjfP34(InfhLY(o=8>k4SP1zqE0dK+-GFgMYZ}bkx|;hN=F`?$v;w7LsUfk65o9&lpJ&#veGfDG+^3P^XnJFZQ9?9qH+0hJR_fP$(OotI%BHb!D#{5-}(vb#l91zeq`i0#WzqCG#^)BHQ;S{j%)O5R-InH3`_Yx<W5mJCAg@gwr8CE3Bc};q=~`AvQ!f5Ge0E5&rE<|^VDE%9p$BXz-5rAovhC!ib9i(MHc45*(w_U$7F25U+G-=MK-BFd;J?<>GMM;D`@3%t+tcP|7DXEEX@_<GeBlQ%7J@n+jqNdAZVU{-*Li))>3;e-=1wR=jZm8ouziG({`5gvvczch1^0;c5Pf*&do06=W`45OKoSa;N+ZEuCuhf+@4=r-dl29rDdVrayo^=@?zd;%{jT&-fXM0<mTs>3i<ZZQqH}^4R<x^6vDd`PN&eCZ!av(E$n6XIt#hX+`QY$v=%x}roFtk=X5&DbFIDno7PK_=nFB@e1WhSMz6QVh%l}QgmXn0eUZTw*`Lr#h&FStg>DHJv3`fhdH5E5wx3T)x4c?zme*_7?fHT(fy3GF;5z){*?$jiCl+)Q)3tDR0V=algUulk7h+p^vA`PKtZX)G^^Z$CyDQ?^$CYv&tBW=p%ERW3%0?yjv{uJ*1LgHbD5X*)W4fTscE>sF_KxCyHup*)hIR#bSgG5Q>@4!6abqpCE19{xDK*yS#`b1)wYpVax7R8gwd?ir){T$Nr1uKC_bRKk&DzGt_Imj$P;|Spwk;CqG|L|)_B3niwFyz3d{9+j(3Wa>tR_KuIfL0Wwrc*y+BINUD(mYFV&uweYt?4;X2rbFaN#NwGI4Qapz``U9e4wV6eoWJ<o@%9Q@f~Fu2&mau%@;~pKCDh#J!pehQCq0exsu9@y)GT;(PPKdvLC6+Zg{T4ym@?1iq`%NPKrechZe57!>|qUE8SH&3btgmQrt;T4k%bz4^gr?ZeIFssMs($GoVv*K$CLv4$3@1S6n<6%)llp;Mp@-!-6(H3Ee^fSnZatv;qsOSJ-}U5au8o5+!d%ZI?D{gKn2C{M^nBvpN0+9I8=)=C3I$=w2!X>P*iRN>}-Q298fWJ3Y1L`p_@DOt^U5s^P3?{q6M$ehw5hg1ZeNXe;Vp5RHi@RY1GL1k8AWmEde0@67ZVuJcGd6<XLRQReo^cYRn8B7!I38KiFz?FoLP&Oxl9EFuWd?PU8wVDVhEF2$MeLQX2Cg`TITE_CiHsWnln=2=UyWif+)+XaxXM@Tu`hotI_^n#=AEh~^&Kt?7EF+b-*Qn^RFR2oIVda>5_3Abu0zy7F%T>7G>y<Z=T0@(6#Wc(-$e*R$d+DrAnc;0)l{R5+BiHbT-Cob&3X`;VxT6N%V)JCmp~0>)zJ#r{w>N|N>W~C>+qFn(;8KMq%s#^+rInB32&dZ)lf>9XC7o_<t@2LEq!Z<2dVvz*7|3F=NVN)Qi^aJ@XKt|#3|JvQzr4I$fM0U*x25Iz+<Y4z&$<hpIj7CPooLQA$}t2dtN?Gft?Hw~L_8hVNyYRk`g4Zk@&Jfi2ZDbvi*bfRqCj6+Mb<4-#wSeEVClyk9vpdCW0C45+JoC>kyGoKoxuP@Sf~%>u?YJLp6I)!+eq}GkKmUR*Bc+${&hg$m)8|bNmtiv&4fE3cTcBUGkd^e_UwJkUQjft7rIvjFDD6cr*W^=X&h_E+&S(RbA|Q^$o#v;F`WKMk?ibOVpeh~xSLkjQq?Kb)T*L@O<2K<W`E%b>d{zx10Wq#aGGubvE1zyqYo1v86L3e{c-Qe@NRbjT-(u1M|hgiTBf2*j>h6hMn!%}XGUBJl%}QWcBAMtCVhTG^c^dc8Yk8C&O4=GL}6a!Tgt`@Iu0sTc1+DtrTtFHmv@epm#vsdQR-doVC7Q{t~<N&Nrw8M)A22(@5l<yL4$PC5~R)tj=b0Q)FlW^V8$FnMEJQe9!y#XMVXY>a142~abJw_!Dukq2e@IAA}92{dw|{zq(bI2#iV+<P?-!pyFzmiAaMcXpLm+ra~y2@%%=s5Az<|!lOTCV(InmxU<T0eelBstBN|<y_Bv%}Rk+LHC{%JN^)++Rvp5=AM<apVN;*DSRL&v@khD!=8KSMeh{EyfsU5T%&@aF=AjMtNaL`91hPTOKYP7=~(=O*cC$<P(`uR>IY=Ly=VGUiN)BA>Xe?qG#6-kP3pK$enM*_V>Cl`~l0U*XmNRLLb{6t9KN^yFp?ePCc!fYaPvg4+swBpDUG$CtJgog9q7~Q5Li4PYc<N@BY0{amX6_#yGa)kwUnN|?-JdAf9r}Y%F6(jYGrAzAI3fcEvHfi;+-C?)(_^=UP0H+TCpyLSq>jQJ@j{LggM<eo<Qv`%v0r+%Hhy{v`N+F{D`<eWC9ZcgeVx9YbfFON#gZT@92t7<k_!D2>0m6M%2ml~QYyc1BYavcF<u~|{Vz<Bhqd5#C+izi#(H`OF6;VQdI3NsY%1d?JJ+TKJ`&eO05;A!ye0NFZyHogT^b&>>5XdY5K9q!iPHQtQNStK)uFlVd5#;q3fg9FLjMI|ohh;T3MR{n3vhUR4c4nx?Rj1oB<Q)YG9fmI&$Uhy)I(V3D-+>)(^&EhFShmwU!ld_j!7V0-7{@f20M*Qp+PmA|&xYk6xbxQjh;#p|w(Z!~W%q$B5BH8zF}xAosJL=6Kz<6UvCO_L?0t06U>0b*7O+h>TwFwZ@5mNtn*?lVnGNBnEK(Fr31pLkmxdr5%1ol=^`>2~U_C!r7Ad$QN<0erByAhTe8N1b;1j9){3mo3&Dw{Xd^6&L0N%$S--JJDA2o>ypJ*-Qok_0;`zLXIA9;MZB&Id}fRaq-v5pD1v<5Ie;MerM26PaFfnINL*QQDiQF$QN^%&T1_nf{JxWG9Jh}aEXf|@{DMq{gHNPlc}0UJ!lUbpR19nSfJ&xA4@WsN%5PQeuIp4Z*$_PS%ZEiNW|8oRxt?6gvFfGA4|e&8`w2JMoCaYg?wbI?QkELSoSWDp-jXrLrqWJ2m4VMKFlg7t*}fRsS9QCBY?tIi>S(g8>La<CL)S8JP10G2?DV{cTNH)?A&;Oai6Yw&TUX=C+-X7ySXIJgr<$)V5QtiSr29Wb?9t7@Dg`Xyg|6jK(dvx-3=s_qp>d>sJ#vzrRr!tp>by#Q=Q(+X1y3MG+ZZ+peG@KTR=BJ9&H374$GLi`cGw=v`)p_-&br!^XQ9v3$;Qm{$Jq~C>~)iUNF<?`e-*O!q37Y8F09`buoy&V)S!dtsM<oBGtrMx?Pj?07JY`V+C9A?Jn%ud2Qf6Cqglx&m}H9ZMz$dSxrn3hzDgFAbn_&xld;u0#ke8e8_TeFEkKc&28Ur<h^r#v>rvJurs+2Sdugbf8ZD4{AJtv5b0WMM6JUI3*T)tvy=`3F_Hd&rrP*wU^}feqo`iJ}0Uo>IifNY998>>UxY?$e8u&sN<o4i}WQL(x4!UrbbEf5**udG>Y8%Yc46C5z{0ls)%zaxr}#Nw;`*`U9Kq3(JLt*3!b<U%1x5VnJgZ504!-jxRe7CKPXFmfLd+nYp&x$t-o|XEU=)d-Kl1ygQrkIB#Yg5An<<mzJ~rT7Tx0yp@g0MtKuU_8|ZI4^%{v!mKp6>Fj&F9W#|&Vdm_2=>9RoULav+6`noeOi|?KLH2RZuSW%5RUaHtw?c7IiDbiKzF4^wic1K+{Fw8%QEcp~C@D&}L4w`u-zY9K3ay8QltMuyq0k<R<_VjDJ3YkQX{>*kH)<hG2HSzJze#_J#n<mN@-W<gb#tvGf(x`kNNC*VAf!Zdayr>QgJ#Q~bGX-b45yeQeizH5iIS#3p@e7%V7BLu$jZ+QZ$2{3jdG(|sS6XR64xY_aG{=KMo(e8K6ihS+ZhOU0yh;IMfuE7H$nx2!jx`h^{(7pu?Q{spbAKADFffQ8+I7H3G*!E^B3%pT4TZj>=wRc@&=~z<+Q-8){J*P`-Yv*2AvB_v#aSF4fW?hu;2MSZUu}Gf-(x}<n5UY28s;<4CMOSg$IFuNRe#&)tz~Sc|$&k?El90=9JEZn?gIj0jJpN4goUSfIDH%!-J54-}f}Q$b1)y#lSNGO}!(h?@_rF>NtFv$(`YXy{_R_!#_6Pk6f-;^oFqKqy-bMVmRQwE0$m=kM#xg1fPR!;S5=u@;T{@$!Pi|xh_5zo|}Y3A1HMENl;KU@L%Gj_g~98RI*NRtaK=wn<%$FIaabS-*ZIdkD$%BTns#t7dWWpEp$OC$6^zM)pCZ~usE`X>?l4M!XzP`E2jCb$E~<#xN5l>a`_O^Vk#N^13P#HOGV-o?xT24Wcd^_-4S_~G$$iiHsk{z`~{*!-pHE5LBAWd4=Tt^d?P^PH1Gt*V;5zw$d^xz@Fo5gwPjzZKB}CVS6$<;HAx>l=Wqz*Ac#K;g3JL2VOW(-(>gJYu%?innnIJK@uebBDH^pG575&^7g7kGv?Co;M<<p+e5>l#0-M<nTbD#zy^ZOZmwKZ3x5)OsmrLJRt29<Mt7*S!{8d{8*N7`1{3i9Z?!5zN;*EhsLMz4pfp_=Kpj<g<vhHwg?_xq`1eA8$6c5C)Y$UNfIs1M()C)I}whmi@_=_dVOK}VYPpzFOUa*u0Kwi)t(t$GzU2g0W2(qS2!9M7^^%(<OWR?q9Nz%@To&YQ!#)fQYhYVP@3oRyi;Bn<tlXDsb-SFiJX;|VDA!bfL8TjR`XA;|E)WohOZuG2fpf}oAy!52O8)JF2zOj=sp*w8rfGp+$!)OTrN?W0y?hFUEf27<9=9Hf1KNWf(`CChDW~6J>q)DFFZ-d)<@!oNEfm@<KJ=)FmZqZk-$)!)zm<6A<Y7M)*woz>~sx@TOuu9kvwU;9^^|#%0>Tiqx@&2t_AL0j&Y1jcG`rF;J=l4a314Sv_S9KBh{csP=V00RP@aXlIp|_DO!2xdFx{G7o|Le5MByCo%pEE-m{3{uva9@N_J>+qoGNph0Ic5RBA6~=R3mEb;a|F+cFN(I{S`|bjzoy{w&+q#s+(jCF+=}OZ=4G)i61DHgQ(eG#R}2dK`T6Sy_le*V%kcWs(3<g7=$6EnOB@~La6Ef@U!lihTfd`d3=H1+k{`wPlIq>-I%YqGUviMreJ&~T`m6h-dk~cVK5a)nm%C3${{_|Y5jrMXEiel|$0P=%$>NSU2I%b7ebTx;CPmYy_eBaya(=+i_X!L>y>%-~H8{_{#!Y>8_9C6gFaKLSy9lycp#)zd+?caSgo0{B{R>4U;%n}tlY&lfX&LlHgb)RVrqrYZvKFMMX~^2P46)KL&%O_s?g%^G>H+2Kc8&tw0AFQ9t(<NGGaT*@Mn_uKn#u;2Ok$pfnEkHra4mVp{m4UbY57=XK~Ah2eiH{(-lVVcO?}o{m|Ua}IbEe6|51#W58<IKTP?X<Bj>zB);@#@rnDgl5C5)<k|yPZ-=Do&-dwA$m77s!i1L5Q3;Bkl@)*M5e)=GuD_Yl@H!AhW&$$1XBn!uypp}ABgTNgY!;MVR)IGb;e>e&T@q+@a_rtnkdF{S@{h4V^((%+l|B#^Ae#il2&qRdTDVLq`gBi({=Q}^AfslwEa}pP36e(ZJOVp$Pqd_%2gKPXymNt3Fb$Bs>N93L&E{}*8NETt|D(k?mqpj>(t&RePy?Udv`hm%r9D43Ma&PI_#Fh%N;>X+NI<mN%m3S0H{9QT*e!nQj-rn>BWVbh~32_j|rqulk=N~fv2M^hqgL+J03&=zVyu8yIn?<f>YSK)^qlsn8J0>5&_Y*x6EkH3Ge1zPSa4G8`qbNA2N`!3+)MNplloFvg*F#*8V00o}5BGpKBgV;SYRg;Xbw=}{9QbH%)21Q?=_!szN32zdaFuB*_MEs_|9OPF*Cm|}_Av09;LVfnQ&x1?V$y7su!PZ$iJbHKj{EGBL@crFH_9rbqXx0a@dotxqN7B1qnNNsFdFS*F==xzHd<Veh<7M|Negk#pAvq-s;!|Ic>kzuVLUA<i4Noij0q(XvvhFY>5H+~wKZD`&!yu~Hiy7r5UmUtg+!)s8(4uZ>=gUJu*iV^Retv7eYfw9Fjt&B#*j;k*>*w3@ALU4bk&-SMwn9m&VV~W&HQ-MJ_>gvVrBvpIn2`bS_eRQFu}lhz%SW^gu<@|`f}S(iR`;LV39qikK+d^_F}En-q0q2J{G8yAt2Ip${=TQ?FRCykmsnG9+!q3!0D3PVl?9e>SgNHEh5+T%8$3J^-2RvCT^7LA5$!$eX~-pUaNA@UASxPtLvd{&vx!O-5!FM?RbL@*D7AH#TYlMo0xFIu5GS=oUnHQn<D^c>EMSh7pt|+Yt{NjBoT!Kr6IcU6#c@_<3a17KLDO>Mr7>sW)26iO#o(ad?c!>oeW3aLzFUocMsSP;UpeOnK*M!YQ&&dh3xqLiH#eTN;6Dx;{H8TjiI5^H=GUN5wRyBQ$F3Psan+wdk_sZbmm?M^UI)zO^;KfS#BRX14aNq$~lBjx&%u^TMt~9wAA-HS^7-)-%HAR<=n-ANW+M3Jv|sHc&kJoMKKGO5&D!Tt1y3|`4HcZ!LudPxNivkDe~!e(mO?v^4R114p*_hn}s{ypBxU4Km@>W5ohS8GWqm{5V}lZDMW)i%sTT>Qm(+v|CpUd-(duT6ncw$0OXdoLU{O!h+z6-Q87T@16Rpbo7RQ#C>VPu@PxvWJud235t81})9Rca357-S*CvGkZmehKv!OIK5-|zHQm@>sRz9@1DnO?;ub&h8=lgA8QQ??l3~BY{U6Sg5tijARKUUNtz7MsLg69qHhY`w6p^<zaLUs)_^`4h5P$PdYMX`?vv5r8@REkaZH%z4>@>fVlB-M6f{QLhCcGY?P)xTqP<G82M6D2*y)4&M`WgCl~k6&t>fnNWb>@B1$`;4u3Q4b$Xr4e$d#!IEZ7c)Rn7>gC=7;yB=Mus_$=;_A~yC|}ni_#&d>?5QakCd`$N>Y`uGZGn>IH5_%+3WSMU=kme>zgW$(iF%%Z{7lG^{!um=>g@)U#-NXN;qJj`1+^gXExVmvwf)7P1VaElAJ@l0!8b_1T06%RF&x363cK9Jx6^g_%LECTtQ?PjnNxwFkgJ$+DMpie6@%>fj;9=cNgRJeS{ECWk%g<0tGEyGPEc%zYuPI>vf|HI?{Q(CSogjjzK=P5VnfX;({Uupsr}*6V#w83=yS=6eSSU;_yKjM1mJ3`4R-yJ~Y~`$bhX-KwD5$2vy{f0b}Y>wlT*XKpzKA3fXA=ci;$72BG7ZL3_nmz~uzJu}jCKtSF+5v%IxcuidPy**9x}=Fr|OZ&VtVudE2Sq@*6<rYG^bMqX(=8N$iRK?(5+i61zb7$~ApAhO=Z1ko%e*pELI;aBcY<$7bPgxzd=G#COqmCiaIK_YNc(XscOHVw-+CnKj4<8q;0+-IU=9`oQB(HGLxz5cC0_)kQ=aW3lpB2qC7HRO*%M_cr>@xq9S>6xWZoj-9c=*yYfe0t^eSkgyd88d3dH8Fk5^dSqdvSRxyQu2o=*o|6I^+imdCq>Z-XOVzcIMWEJ&wPkW$5mW?Yh+EZ?OT9Hd|vpU#*7-Ct(faBgOTkQQStzuP{~FRE(+r^muivFLDmO-*h(P1^4^g@6vsN4R_wGc;M3m_Gd5ZvFLc{DJK@|P+Yz`Q_YHQ~6x<|<1hRdM8;tq#-=`23^!G+I&2IeSWesvZHzQ$ijo#WtzuhS~&a%67j*GtEY`Tm7Ji6$U&i)yf&rvhY1#a`N=&llbM%Jy64Kpx$($yzr|0`LDM@<r^IPm{T6zO5Vv#^ssnq_GsFL0>eo&C`gDeNqE*8fKMQ2EIcWsY?CA8>q@Arl_rBx$<OVR|ow0rHjL<Ub1vrc+M5Cu|%IGU8=%)K@g3z-gMn**lJYuC5t^zH#LrCAHoo&id)sT0eWi<_-=mM4p=YUd7+~B{_D-jfg+6+x=Xw<u148I)7QWdywEYZujA_!*2IWOMC9#;?i<vVY!{l%q`9@X7*aMos7G;<T$NGcX44a_XD`y51qc-o9c7FRo<+ula47|CNO*{iil+N;CXgx8SH{@L8G~uN^aho0+As1ghkN*o81TyP-5KC>GYe6O@c{2A%}g=L%<#Zr&I!6+*-wR9|^RS<5A>HzR}r`RvR#4NvV;5Te9Ql(_&4w&^RNE#vU?3IzKUys?HAgr}-u3mBh+ll~jqD^PlX<#aCn<0$?09GcSf8>PHqW<G+{}mVsi^R#fjoiOu_4jOQymiLG!aGV9$^e=s_vd8QItX!_8sN7s?AnQlhWY44Uir^CrCQpt@Hd-691<}^V!>{Qb4nl8|IgsHYm$ds8Ykv&l?C(t6<_9B-%ee1%#N%ISvPZB#2>i^KYAx)x4IS_yIb|3f%p&X<0>dUOq%Bf{Tr=L#hMOj92SMl@L^oK7<TMP6w%yPYp)K2O@iEh++-w!`jMo4FYG3$ZQLkwe}lWtSRW%P?kzk%LCw;9M&C$CHn&jmZR<L|z~1a@R@M0>TihOxBO&1StU@<5uq7M#C>$^7wor7oYRLrTCTu+R3!sT|0L{i9SmknIBH8E1(8WyF>HNlZJ31P_zi03G|5#k`k9CcgGlTZ<gRT&ChP$*Wy-nq(4?^9*lY;Flq@yns;0figwZmZ6?ZP#s^HEkRUb&C%d4o7KDNlXDM-63w9;CLn;MpB)J%p_z+$?*a@^dM9Bs(P10+9?DkTvHdf5G~kP@=?~y8m;M|5Z~eg-2a9(tSL?R-U1?45(U~^^F$_2youR>Rqo^U-ha8SO<V7B7c?b5=p~0xG8eDDIdbGMaV-+2}D^gU;f|<mEe~sE?%7Bz86g-z!y;G4k=s8LWU%dV;zP#cqWI4=OsvbBEwjXHwC{X1Rc!YQt#qb}3AYNv|-QCe;2|b@Ph|(Uq#qkMn<zC-ExV8ommsK!kDRTdJsi<r!8PmP>bf?MWJpX!GJO7b$?)(+JCg(qWG=Uo9<4yWZ7nm2+4Hv?y7a0!I-A)cuht3f5(O5h3uq?T1E11x3TI8Rptx3eXR&JK<>*&~LnYFc|Ed4<F-U*gZWqd5DP8nPNO3I*WNrxwTs{%=N9$*eXyNmXWLuA%s{h%l=optv2QxR<&8jDkLMiaAKm}!gR2l=74)#T7jlPy@*159bPLU3T{Ydr9nkFj_u(+;DucD>Rl$s^c#TWGdov_?WgWGKMUfIbEG!v_(LKvAz=G4GpKxJ`USwV`6l!oE4-E*0H6N*7VFxt|q-rbWiteWW|wwV%N!ws4h($?PqdQJi8MX@k9(x#@__O{7u{CZkEM?3IF_caC;fhOi#x5ox)PGI<N9pN?gC<jDla6OkuNIR)%W<PgHq(w5nG@7i}f-@PgLErVmHxd65|_5)Zp%jYpyW^dq)<9%W;85;C$`XmG`9M1If{hPO*?@ydo%Nez&bvq-!-VcvL>9dsQG%E1ceW!o!uza~A7o^zf4V-a`fe;nX;x<H}uv9vV-z`n1B)^2YH`XO~3_w=l?tG|`SF-4e$X)5mwpsbmWc|9U0CoU0x{7XZJywmfP}S;K|0h0Z09ERs@+#QvPAP;2X_$nZOY(7ADpGcnkSX<|EGW{E^2>P?kCXoSlTE*$FX}Hf{3viFV5Ym{1Dia+09z^zvf*n!ll)WGJLq=CseF3MV8a0>AjOF4Jzy_yM}~?`CA$y(W#`kzePen%l!>wni@KVX!E2GhwW7)NT;0B$rKEJaS*fGb-_?4h`~k2M^-80*elwx0l;WEs(&~^hPXxJytNh5kF}1Y}+1_4>LcD%{kRpX8z5YsM8l~R84W;EIi>@r+n-oj)P^KXfW@OH%M|#YJCHNjbvha>jt|z1f5^fjlbHdgLzkm&^Dk+o5HzL04c}VdUl05z!^vb%_2V_>@yd;%ep(vxhzGklx5w+k4#XpD#0*T?QDMkU&wA-ddd;^0E?-+`Oiu4zRB*itW@8TS(7Bfwci$|&YTDkTAAA4`#+{TgS3I2bdqS_tn0<0nkfTT!*hA&_dlDXzv^F_9s=G7K~0)-I>paGB)Epx(c_e}R}&$P#LbI}{K-959j`|n1`w$zqw*@~F{0@PQyPjb&oW>#ia768h2@7%@p#8^aCR=($x&+G58VT8W)a^*sK!Q-eY4FVuv;q4-4L)z`h2#=`oYnwOrP-5bIk%2Ld-zaAqGW-0I1^%aGfccsSFII(vMR1b=Kl?YRkCs?6k3`n1Bv6FsnP3M#qs>6<aFTuqV?Ick{ggAHY4bC6sC%YvpY}j9Zr752DNt+@$jC1(oq3R%j>^3VI8X*c9`BTT)-$5NlZ*_i!|f&+VS{$J8^8#=0W+RK@9yZN2Ty?`9g!I*=trbxgb&1D^c~(T6F37`OuNd-p!*hKC`dXc-4(oFfq|X%3h1Yo%Lx0=r#N#Jyno=s%^dh@K*-nQa7ousTs};Me+Z2nG<5wm{njdk-{Y0gSRi%va@SFkQig)eQphpEKkcT}*SrkB>?Y+qm)+f-1ME?vs=mGDuh1Y2C?OPD@Tr;U0G~9Xlm6+@EP<h2|0=>!zanzPNzXp!%<UrW84!IoLyA~2c#OpDGs7@N$NBM!r@7kkVZow?nubru6IRZA!}-#b5w~D=S}Ze3K(gxvwf0i|it}pDf;DS+e~-?&0+=`u+R~r6#Wii_j^m!8v!?9IIa>{m72|#cafVsNwar`M)|-19TZGie;R2XBv9__jk2FZ8!*O@{B*G&`K@Lqqj5(~93Poqe(|!+*zix*U9wvz`&NPk5AP{+a@F<B_4QLwv=|mXE-yz{yfEwVbC9s1g5>)u>2pG)99T4P<w4c<M2?vY|74Yx;is-#~HtB%L`e{LvGcLY{<zJB{j6pie>hB#5Se79*uGE=RO{eQHHXl?2yHAytJJ{DaKu0%RJJ8?WTHD+U-&)&yV{3meOp-CtX7e-q;n-1XGbjBLWGR5n78DCbN1xJ1Q@)`+awvzUua)vm^U=+}d+Zu3xlZCbMV<D0u{O8(BK1AS3&<E``demB8u;(NJG=%P)bsILjPLUmOJ!0~&sh$XBIzpI?1{}1A>-0APl*!AZ^cBAOq@Y<A4|hcL~*);vk-od81BwYoWTi_7lO@`5>w1BIlmwR`W5ou!nfrkVvE1lurZ0_{~9}0(OM<nbkQoNyAzpZl@|-Ky6ga;TZK}dEtZ?+&rYKOdTVuK+G`!et^gXz9$<s>T(cGM;c_MQ(hrRzC082#38gy}Yh|{rX3N6GX;q=f<SUlrt4W`XAE;jFmji7fOz;u`hh>~xvC&w=3c{yR0~?4^Jr}KIN|kA&7u1TZh4}E08L5RL8u#EHI_?S=(JD{xH9sW(Rk(%LfVQBG5^+Qj4jeI;ht5z|o-S<VFI6+HsS<i<C@Xc8uzHrF8P&(Rv>z%d7&(Ip)^Wi9yjD77D_h>>lZxLsrU>4gM8mu2@40c>8MW82f<kN1I|(UD7-sAMJ_YL2hI=~aC?>oc$#N&!$p`G(3xB(2D05jvXT-9UieuIVL-^L$ZtU^e5c=;6ERbr@wLp&J0L<y6vw7AFkiAS1=ZPP3G4gHoE$02chQI@L2WLGI5AFy2RVvsU6Rku?;#`l9f-0MU)GCXjp9Q(C@^ZV$>X)XN{NVnD@@hBexe4j=af%o#f30zXuaUFc7bwv~&Y82eYQ{XtnGO@?NU$i*%!LLOwRn~wb%QJRFR_>~Z#Ygyl)PfX5vG}yAe;Bl_hlKG`;;4&=uE}#RYm-JVQ$lMRdIICiiEp&Qk%gcR*at_I>TN86op4=!-_&kfb&zvO<lI>Ieu4V-~!iSB^kHZp$JQBo!vQ6v9N$S#HuU!A0VWi!zvjYY*HXPE)Nlvwj0v9Y<Y@olj|Tk@uY2r^@RI0ry2-=*rAgpJBt(BHFgVWdwBe{KTd(I5~p&IRm%6DTc`{JMOQg{MLrNmj>gr-=f~Qu-Q|}l>pG{L!Ln(IUVY-L;Pj9WMJzBKjti+nf({?P3hEAb_x4xckwRi#T2s>dWypp?Ir_v_C<dJ2o3AaqP;^J9O+;-or*Z0_BMk;nVmy<d1}CXkslP9uQa&sKf+rQQJMvM_nl89eGx<E}rL{04oe##C%#V~JT-V4FCe;Vj${!7&Y2;Vg1I-F63bJv5?6XZ!nkKd<M@+d;>TMt#QX&!z&Sw(m^>gaBP&|rUSmYf*;h<7slNpC2L0zq&91AOFczRS##ZDaLvI0!1RS|vkvg0_rzS2WXs#W~&l|MzugK8$6j|~KyM1~M=3F9l6fiKP|se(MQj=b2Mvd0y-<DkhhNQfGD2dd>)PP<7lHXJ6so{_KePJ*9&dKHFfe^UjtId<~yL?$T-`y%+xq!alII*IO0DE-t;$Mwsn_}*lBwC^7Yy<o=n^&~oR`k+4<mtQtD&o5`pb5{9GU4itn=eS3O_9yI2q}@!SS{@6d6JaxFz|ql&lZL%(`dobJ&^Yj8p;f?Sp&HDWao~k&Lqr0w()L>g?JtxXuB;b`p);!hzGu~e(!D?v$smFM7u4{Ni49TNK%B(4eHMq}a?VAR!;odLe8rhuo1rqqidoik=J^~zzwxjkQS?U@%%}a*wUN>p(eD8M3;*P=(XVv0XNXc_f-npf=0^D>PYn0ixvMnQtQa0?-Ks4sIL}+#^l}eWTNioP3GK8`+M{ZQ{bUSivW45P^WRC#V>P`X0!-Joy$C^Lvf#i-OhA?7K9aE;$peYu6kG{hRydsKZCQ6Y5$=52*+Cz13PDkO=dvH@hNYxrq20`nt7n42Nvnvo;^V5aq8-d$n3FTMo}?`06vP3-bV+?gSc9Dv?rzs_tku@{-ogA3h;_7OV-gYzsBt<R)t%@VzhoP1baE2U=xCS(CBcz!pOQTM(ByB(`1Cg-_&hP7K4NK`?6os`JXRq!RMBpzaR$a*xdgEv6Uk%1b54;%&IznQ{f4OZw}LtSoO}Tu+87_of%8j&7r`H0eqQh{|BwN+ZVcQ!CIyF}$jE+_ewVlt`(U~FlQU!N2U;D6Zq2$x`P8)R>00l`ald*d6rA4s-Wka<KL1H7w*#N=JY>T$+(1kQ{ouS5flo=)i~gbW7XJ%6&*sXJ_6yr~tG>idfd^fFpRc6VcJiIl*V*biIPc*mFiZsL8T!|>Nt;qxXHeGE5ZoisuWgJ()_sCA!VN{Z9{Kisu0^4m2Z}{A+n#}OiDu&6$1*dI7E^oqaMsG)^YN_oh$$Fan6X!hfoSeDO^Scu?DpdUsh847f5zo9`NBZyUS1XU{h(O-D<P$e&1H<%K6J**4BM8rAAUd+m!mh6th3{<Bdaof0qtRF@5YiDvw=>pclWd}t%?VZEuv551pTKA*uY`SsB)v4Tm6+=J#^YXq&m_k6)5J!$#Rv~hAujD5)Y1Jwvz;KpkAm?u=VNot3n65=ZM(YOT)kcU)(-E?VS!q%LlRp+~*%vOKwstPJ;T}MQ7E^Ona5j2XF7j8`D@pj484@TE&i9J_K^!565j|Yv1z3bKNgI^H;(>^1M5}dsV*^w}56%wPw~awAey(Jc<U6ufxF-3JS?Q;r7nfc710oHz+AL3G6Pe>9K%zFapB9rMPn`k#hJJ2>SSyx}PjxBq_7H1MpK!s+3F(7SzaYNB9rFHw@^^onc|a1n$0s`GWmBM$3jyGZXy$H-!uh0wDy<loE@)^z!7KOb<D&<mciVPJl0dNs!wgk{>T;iS&re{5c5M-sOKKoe3rhTqu>j%impo!p;ZJ1S{|byYV{s1HoE<{rnSXPGAXqA^eCr(o@H(N2F%VQiXfs)=q6h2TZMIU7V?pwrhKD5Ryz%hB92R1=hZ;wWMsuWt@CMp%8X@aqDKL(v(eU8~$UsGKwTGHpK}M0;9UKwY4|vH{$_tG);enpB7H;Hrtq&2L<CHUBdx^D<Mw1!?hCuD1oR|>El@+yTYmJ=`q&Ie_gd+_snXoV76T)5;fyt1MXcURt4eRX&vNHQ<o6o5GoVG>Lfxkz3wC=WrtW(m&U`trj#n!Ztgy7RqmsBldNt2nxD6OIUF=Fv!Ldr>R+QOu%Bw7^Nvs90R9s}Jw9StjhI;?4~VeA?$HTeBkS%{I2xDVt}uDPQ-vTLjD+@UmR|p}RLJT2f8{W&e5HlAq<z*?A}vPJlK=HHj2RR?{?l(oJ`K&Z{HNd2d4Ei`I2@O>w6%{&uKrZ|WO78nBP3p*p#o50pCF}4GIiq5g1r)SR;!7xGMbWZ05hGVM8>sU4}<t_)pzpvw|!d%me?Cu3E3QXY3~TdH-<M4JQL1b|1Aob2<49YZfxyu?&;~iR5vj(H1i^k&o=6-Yqias8m*;)!CF?g_FrGG=V{<4akCv^oS=ZHX7K*3-Y9=>F<467p?9!JFmCWhrWkBD_^>#tyjfe{t>Z@|cMsv-Z?;J~OMtZf)cG1kgAwB7j^aVJ<>wygXZZp~w7EKFq1SdjOn<$JMA$;<b3yYcG8G*k1G|NY*ZC>k`Sr`xLP1OdTvTX@^5%D>!nk}g-ZwuZ^dNMn*$Z_){&!#6t?zjliI1BK_<#6UB%DRy<`xC!zvNIOh^Wy6P6|*wWb#PK5X=Ug3}-79JuvoYp0nrw1Ej4N-dk_fck1{<+X!0zx{@PxUA{n9YSiiE@(vW|FuvbsoemDi=)dZzJd2aN;)~WWPiG#d#?jzW)D$!LLM^_)rpFCGzGKL>sD51~q>PMw^SY4_um1f_H0Z5{^|x=V@2_SAlUWW>>c~7udU9bXTg6?)#`6y-Wz&a=O<kyeQogN}4v7`KjL7?xE~h7B`jiBSrB1H_?XD0)pMRJ#`oZ=12R_gUSZu4437X`uBua6DYbx(FBoBgj|I$<0!WhS7%KMp8l=M1HI+lWqJMQ$3a>ReVq8P1wg^CqUyPbA7g*5MQF|r(5W5Mu}JWe|^U@Ww}JJ098YDPr9t&k9!7ffC)_7}8Zyo@$H2kr?rb&|{Q<sX&XL7E}p>w^+&`u>fr?RW4TF_UoNJY+NyJy~WcC<k-Fe6SE)4;Clj%kOPLb&~HBvys2>ayPejHfnp`c5UY;`}Ms5M!m7UQztvP_tV|I)x7UYCCU}Z1(HD?ZpJ$sf%)pU#h6<|T$pV8^XG28?mKS^XAM49FuH?m@~~R+uk_jY+iL2%#2}*k;V3@ohalRJlhS#JI<l%iU%b+Qa9?GV6$V#c!E_ND4LV~HuDSs@M?+i3)p5`U!e7Xw_z9O~-?n4#o9ct`{KBz^Ll^@=ehX4G+e(B@{3Ju}8^>j}{It%3lm4>Q^h-|l1r)@?7Q_uPo}C(hd`!2+gzAZ@3FIJMwimI~c=(z}g{{-kR%?R}(WJ$nJbW5nC6S)K)-#cO>!l?!q^Ta;Y20$z$Cpnh&y(6n?I1~QgL^9Oo}R>m2v0^LEsvuwZEdgVyUY~2s!URow(_oxmQ+eQm5lAC<>Gk|w#V(h$-83W?r6*H(PrsCyP6l?B+31XnM87?^qS|u#;+*YQRM(v8n&-k@4#He=ZQ$a;+3u|^&W$F>fw`QM*T`ds~z7n+9^jcFwQeGh`Psdu5>+LnXHx$kJSF~VTZK!EkeV+u~l2I@7}2Ah$NYVN8)t5wsYf++D<<3!w{L_FhmR6e}M79p5i`TIB(u`Pp+`3Xg`Fqfl9GF%{H63nU3UUI&y6$;gQ5&e|J89Q`kY!H&4TU;FR?`YyK?T1@x)rHpf3~YJ{N&E}yw+)Uzs0rTYep*+=TYR_XOwX6@4zNRySn<{vuN<?0;M_akGcYD)N4AtJuw2#J57#(V)Ad~a(SlMho>=Uh+AvS#%^YK%^!PFl>BQts~lM$Xo0<=v6~SG8UdEP#7Zhn6UenM;cqLl|pTm_pe>0_B(m```24`r5|Yo>wjwivr9bv)3k~f#zBwCi+KJrOe3Kn~VyF4QR-4z)1Wv`>vzMhQ}fBDil+|BFclFa9k3b<d^Kif%`}_ei1k<Mev?`dEweFVZUW3gK<wo`cvYE!kcUB2$)7H_Ow;OKdd^d%c$*Y6=_={UEf{}eZ_xLOGvK73C(IjX0L%MnL2q*lBEXC-}t37KaKJ^2r;nGuTX9_Eo)b8cQ;cjB4ShcL!T~+-hh+)t?u7=bM<vL$?n2EyGoj$+Ks)icI(zoopzavzE8v0C%dW=T%<jMldTy*QP+&oXNs#?u|wtl*5gr6Vs@Kx2RErpIZ|kJ&`-ts=(tgUMIkWZyydB#SXUj#ge)DuA^;0KSAP^C6ecppx0Ep8TLuzB3(BWl0zTwKR&ejK8{=1m{{9sw??MP8$;NSNdbBBP@hn1wdYU@jXhYT!F(x=x3B#oZzOGqElApqRLHv^V53tkD;=vI8Wf<#0_ph?@#0II%vbrCyOfX`5`i^nZk|>f^23f;<LX&EA|K=YF`z||c<uoVj|8GwR`;JdAq_Vm8{x~abrja#ZH4|Qu`XMtd?G6t)hokj47rC=5IHjFUa!NZJ>kVUmKEX@ctVvqd&J&*)i3AvKz5XA8(FoV;x9jL2YPO>9IXA?NL8h`2tv|7K&mSs5(ufz+MNN@{T@c+TPInImA|nO949T;-LlZA%D^AZs2YcwPt<JI;S4NTOj0*XbeyI|gsDKytN3}8|deTK$bFTh|rov1}A+uPd5q7NzlhdwpK%+oUwLrN+bT0#m0@J-m_c#C@xm6VL9*hrBb)0J6gBJy8?(~SV`$hNi&af402i~t@>mpu_znuh(AzM3iBU>PF>2W-05APu4OMBSp^oDWnet}WN6?#3ve&z!xLV1@m-sb}tWv5z-XO@Bi{$(@s-Cmb{D(|zacGCJ-FGBg5%ZX6FV;{ou7;J^jtxcgDO=8}raVxwpdMgKg93u?-6kWe>0+5VbSc1N|8pMeJ+PJUdYz!TU&fRLxw{V-giGBp(om2Z$ge#8$ZzfQbAzrCf0oEZvI||)uwUmEZ{nA^47l(+tF|<Z<-0L-A9PC=jwW4b!jb|!{N0@7@qoBBIMte=AHqg`7(XP0tWJJDY5;$S!dj1KA^l8Zw39$@4eNd}w_mGB2eUzJ%+83KMNG0>AD%gD)G0OSPdo3bUqY<WUO(*g*CsDR_I*gkl=3Y)~;!^6U1^yo~OA7d%N@8I7&Yn`v0QYgl#C?vFidAMsR3s}BZtSn`t!;y-mIS8WuC4Facc(e4;;7+FA54oO5UAskItY#=D~D+m37L<FrRogOQM_S<1vmTR(IY_^ZzaesY`EwOfE7L6i}1~LKsE5N6PO0oHwR1uE-D@)!pPQib?Te32G}$&=s+o%usUK|r(2#s1FS?cpZaof{#F%HhG;h0>P%Igv7o8?%;lc0e@2A@&f#p@dJJCf50{^cxHluYn96^~3@013kO51prGNn>?Aj+vb(IN)AUqDTadHJ14=WIen?-(6!5shTw<@$y<N4D56R@>?XFQ;fI~-j!;2O<wE_x9inl76XUaHPw2NU_U7Yl>UOk6;HNuh%}#TFnHwavAk)=+WguNd}@v#W}3RZ~hjm};aMqYBcU%^qiJvIOJ}j#&@ZRS)%<32!5oW9G<(g3%(Sa*=IvI*ydmJC}$&^-25(IB|q?0^464tcr)~4)!hnfN+Q(>ORWKut><siK6cLN0^1sql3BNiL4`d@&wc~Mry#QIq2085oNIE6BA1yE8n`vb*NAS^=z2Cm@CLz^K!dy)C}t=c9x6fF+vvp6mA4CN(YRD%HPf)D+E`IT%6WJSi;qPL=+)UqHuSAW23h7jun%zqDtLdtFNzSlrk#~VSvb_tZ~47wYValE*MX(k8W(O?^X{cXrP4Z=@@ooJaB52#F<KzQyGe5pE*d_vCUKTpbh+f46GMCf<4#iol~$<B%ptWQ70Olhb_2!!|3cd;|+QSYvdHiJI$bEBwkuUMk()7r~t0;&SY4soQtr$=$NgMnb--g*pV@K;<WcGI!^i*GV!_NxLCf2P6Lh++H?7Myv`hr(_vhkuj=tEv;l$a=U2q*h4gFM|I;}+yKS)J=EMqaGWA&UVM5Qo6|S31nI;<vGLE`w5XS$TCMH2_9~F`?hQl;s5BSgqcEo`zo}epm<;KJlcv*o3<{qCES0Kw>sfmmulF^njBy;hrG-Myoj28Iy36>OJK<`}5IPJXbSUP$BUeif-Qm5^Q${UF-#`3Ub8;M=_1*+8CJGNkN3<3LgrNg%H=Q?cWjrr$2Y^hT=-T?{!wjzthwDf`@K>zPKviJUs=f@*=F>7hGh@k0XhhFe>E=)X~3z?_W*y06y4~Ww#6=CkLL>jK*bPPmG;~>l<`j8AI4;%!g5?ow+;1~*7;2#kp$Vb9)|EXoXAAj22MC!m-aao}VAww+JQM=W4B^Di5kw(~~VXuY4S-DuApDC4R=IoFn_w0U`&R7*)r89|SC=Wv7N^-^-7cnBQG4uoX*vH9#84V;y$naWm59bByr;xj0BT}j(6G1g~dLAT6VJnis)Dy)9QmM^uHS%0>F?=TclH6K6pG*PtEeO^6j8+^+An;G((Vbp1af%^Nq1_v`&5%hYy7xr)u%6?%gnp#uh3k61#goH}VDdm|rl%RIDEK`LcH#RPtmu*&?qPn?Rsl)>ABB55y7D|m1F;cOC33~s=nan*02bvjjPC`)jEhb(kH~mSuvct+`9jMg#wnjZC4=xsmU}a9S3SjvT){vI*}`j~WO*%!Zx4Kn<uKd}xI`-3Nb+%-NWdxCYoEk0*C+j)PBEO{i~D&Hx8ecfU&}7wcPptTI=x1uE4LgVV4M-@2NL!jQXciOp&Tm2Pxn~)OpqGRMIhW{dAI7IO}*1mqX&~5iml`a_-fM5Mc*+jx8n8*V-q#H#ctF$wh${`UcRJYQ!t&$b&1;SgqzkJEil$m=1g0`Kes8pmvvBnPRaEiB>=rz4l&fr;g%ieYNu~0hwt^&M<Xle)^|EBEC2ri`FC(YraKr|rj^<%bTtH0t#{?U(we+kTIOzPeb?>MI<R$sdV;5n68Z0alkisH=FF9ExtG4B);?M6s++aA0$XVAf!6fKK0%m1l`xg$_n^h~*}>QFND@ksC-e(}b0obZv_L?Nz}5&K5;TqQ70e6@;b|9L4H(aoMv;cGMEAp_m@?Z{h}z3v2cn2JfBGe`VggU}qi&2y7_Ct}NaSxnYr=lV4sv9Ge5NA!NUidS(4TcP=EoHq%UY-J_}~wh|8)7Um%pR5r0&$C6jCVS>{n87RXU;-H2w|2X=zp$ibed46wwb2Z{m~Fu1*9t_BVqAWrI*izEG;&X!n6bC8fJi=l6U4(@rGJlqc=uLF#YJu#1Q2s680n8B$922>;Lz1k^2C)K4xv7CH1*xWqV@P%LaYf@){_JM^=HZ71iMf0>=HV&ju?)zJ_fc373LZEf$J5LGYXYEc(7_4;)OK)FKcSxGzn!S^A%cTrUds^Pdw?1W8l;&Mqvua)x7Tvg*k+!oE`$4TC!ByHWB^>-X$5E@hkOplhSjBG1p?^FlqPrv0;Qt=+(xwbkjQfPP@NX3^c4%iB39~3kN>VCS7cBf5q=Ji_Moi>JK5Du??R=gvMH5;ZvFnV{ZNyiSf6{^bIR~AT)*awMg-uUH1@5SET!z^b$oN$eWP+bn2Kofb7`T;x*#<eo_<)SASg{zbFG9KqXNxOV}Ou-iqX?1_eR8KgEXf#sBMF^+mZ)9rfhm`y-LA4_@_*Voi4ZlC4|B~nN<>P{$oW0x9a<yJ9D~qvFCVG6x*-Yh4Jj^)aj`K|2#s3`GX-Dy)Aex7c@+t7sN$#hvMyH}5K{OO@5Sm-6?aV5(;+(-ew-J&%XLKiVNj3LvSYO?*;vWhUwcBD`&N2OQ0&4cUvBSMowsD}Dt5t#=N`nW4TNWy$hat0O>PKLMSIKJ+Wvah`RNC*wgPtd|0u@|+Na{y#FlvAq<Xt#Ou!JP?1U%QOQ}i=vVwgeIU+a#p&-+eqF`ZO}`e?hO_&!XPtnhmv1DYR=o&_w)7-8ue6KB$~%Bem%r~BK)yj8rdRX;#A{MKq7U5rNPK*7I5v?gE_`t7Z22&J9muEO^+;Za&K=O^BiRFQD8%o8p(`U3HD;VuM^6V@@%4FbE4s@3#!Q90ZBEIRr>biV;%!UJ;ll#q!C=IHN;uYQ(*B|R1a6C!bnmbH%b8t6vzILSf(i1=2#T`Bh{G>&Z_!hbHX7Z3h7q6BfLLy<@?Re-mM26vmVow=7UwVU{`Ip2zx7NdrED8mBb)dRsyn8onn1(#)!2;LruQUBOmxcvKqS8JX{-3DA6xI?@hynkW!PRUEe`?oJLodcQoO~FQ`+`JiL;YsQ`x_pwXfSDxw8x~-m|AzFKk7Wpoa!NuUtvWI*-*8wBW-w6s5Wd1_K(fYQgrLX-lS&6=dY)tMjcgA|&a48jz7;?K>Z-#jV3ghGKlfJI`%6z~pnSF{W8p(oYN<d+ajAY|rT+d>`p`e4dFN_A(Sh>{6L$_?SYm^)$(Sh*NZ;_bd)n#fJ8HSqY!#P^%b7b0Snuda{q$Z!u+#8#(8iDNj7I%oWp)-7MV&jn;i$3%b<K)ph4*p$_zt#OzP>D<Dcw^uyw`3X$D^Uzlkm1wT#V-y8%y!x^?3fMRbDR5wW8>#xlo#0o|{`<kX3sx%`dm2#i$vV7naKNOLNzcnseoaXsKAdetr3RDV|G8HsW}3em;s@&BbWGcztPkp|n&iEiJVcVdUj9pAw!@d0O1F<GsX?MQijfT51#*7F%;O@qD8-GaomWW?GF_V`ib%isqumQfawZ(hneiOQSu^IFcSP^C0%$UM$QPig@_O9nOkJmLzz|uG_=F1L>TLZ>^G_?XVBm2{6tGurxH>3$1eww`Y9DIU{G#+Se?2n>`TmV5xvF_lAu-y<R6l_Y8o@54m_C*-a1%6Lpg(RfkwJ+jF6sXs8VRx+`&(vjt8=#Ue6{-aM8hk*VaFlw+veXDYeFQ<J=?ym{HC+S)22a@XM+x>4WV73mLk$7d{%gjZ0MK$d@_Su{xzHui&C^{o)@)rn<p*LT)#uGLq^7KH2i_1b!E^Ty<I+qJcwx7K#+W6Q}vppX)vZDPe4sCpPB7&QkW@;|whqqm4P$qmy93}g!TF_{|t=bs=v>0>iPwA)|8O7U>;f4uwdy?5Uo&K$ZEJzOCfM1I_`S<Nxb963xf2LJ-|*>FvRG2Q?pkj>Esa8xlc4ES2qtNLSm)9D@O$bK1!M2sws1zoa|R*S*l6caOI3#cA|zwB<ijzyxKfHMtb=yVYK6>dgoWZ@^br?eHKVTaXL`o0kz5q4khUp}B|X^g@F=|J$mflnHB)<0FyL4~`XKd8(Vi-#8(oF8dVO5QC0F(?1H4t!CkGeiC|il~wv(*v6J3R{~y=Zc)YLb0C%*6IPlMfN0y%grRZdv|~uwh=xZF~DM+E2NcD1e(F5#BEY;kn?Up5wXMb?&wZDYB!)9(AQY%Mc^Tld0_Sj?fY6YA+B$)S(S{rJ8GaeXdkz`p;ALOV}v*s8atI`OnT&~xCoWq(UzP=t4?L2xwXfX9J7VxsKmddP>_G8GsLo(=)LTjMdV00>h$nRgUD=598_3;yCxBbI>l5UT3<4buE>&@*|H<6CZXnVaV>t+HK1#%i*9`*0WEqvz%;f`;ySKN&WAIPJG<hky43-|t%w_*j*Op<b4iAkf2~@!a|Pf_5rQkbFLfbWH?uPVJE}u0vsmmG5ml=<Q?)wfR1Mk}rtDjy_|tl)b|klKDF{7%9=L>XU6YkkG-uHrSWabt4pprlCFzLMM|`rLh%EFlAtxvB5mBpZ&OY-h%$eN3!rm0+PfvW87moC2o?HW$^b^-VYt;Gid84%8SvhYmI6nFAzz0gsyFe@qoOWw>jQe^B3l>x0*Xz6W9ZUi%`x(|(Z`GLy@78Z%JXpB7wHIzuXjpBXGoEF-zVWsQ<?X}M6&{_7<&_<nWQ@+3LdS$mXxa~~I)Vvg#j%C^QF~xU4Jr&p)F1}o0OdtAExiW}Rk8pFm6_6Ep#3gY4)gAu?S#rk)#QJ~5N%j<{#vPmy0>=s)Zv;N-k<JXiGL<)3)CpD9XM{uR8X^3qyqd=yEky6M}_Cp_l1Y~ojcPw9iYnG-JPQzB16hp`0yO{!K%TD8eikM(CX_vBFq7An@X}!f$Wa{Mhw&)ipBM)QSoMF#IXBXb<P6kltexuN8&LKI-eQ&B^1`ZR>;e!eKRYXRRrk1AXeRep&NCls8TLQs?zhM((`F5-Kf<uo@3Y^j>yx8hv`=cT_gveMn^+#pdi|u;a@bUJS=$B)!eVSD^s)!^q=R&YEi3P*b3>ckOOv>7Vl*F3A1+u9RJ-g8inOzdCu55RdjHg%lf_(wNFAO<7hnH*Z@MUz=x2A?}l5O>+ghbt?j+BwZ9jV{JwcB{Aqn>D_q;y-rwV3U;F&na&(Ho;(@paY!;5a9}NgW&(zf?=VqcKsTAvc;LyTd2IxbjJ^VDCX%Xsjy}noX$6v4h=9YPrIYIPX-wXSIh%`i5MduQcZg^2+&>O-(osJisMePm_-6PWOYQ*2{d9amFK>mVVs#H(j39LM6o8SV$k7s(V8KtB0XkAlIeN#`<?wf1d+c0#PhP}0odbqp4jaz~Z3W6-dP@@sd0muN}=w45ZV4SE~u6XhNJJIP7J=r^mRgNA;qeQhf19K5~V5%^_AB9~n!ttR90di$;a2$2p@6kPQO$B;iBRQ2k9}IbOx7|I4d4*a)Og#6FVo&0ofD{Xj-@6lcp*zUSw?WV($=rjLJ85@&Fh?MO61VJOiciufGQZu8#nRou1Phkv5*N$pDrM#In)T(o1VSMR=XU@ItGY%f{j1g0?g}|Mo>1*%;Q1e`fNEL{V9-6nj34s!)ox-A)7_t}qYHMcL#)g2w9$y;=Fl}Q1hTA0hC;3RDfFP6)IN#H{#(t&M7AY%czXV^$mBf6$JLB>%~pD5;RCY-)pg5nTIv8fWE5t|)KBL$XG}$)FGk+QE|Sc_Oq|_<JEKerUY8=3jCKykXN<|z(5A<TMKP7PhzSW{Be>g3M@JdkP%8EG<1@v!z*wfTnTY8z2QejcM=?s4@hi0_Rx-e=y$6!GyCx)mk-dAgd&T*wYx@w{t11O?*0%){-NH4mNUg3^5q>b`HaPGlxfB>uH~~^SL_;7wEs7dIjEA{_@5+AlJ{cvxmNrH-0KTuveQW5KuiPz3;eq$-0TmqEH<4xr;bNdqUy#?M>g)@3BY{^!_7N4=W823j39uC#d|xF5`sn!w&wu{D4QY!do{=4uihhRn*!RT{&@B|6KBAHzzb}lee@z}|PzZ@$672W<qZb+*Zj-P*v?dmxI@bL0<=5}?KYTio*uFrU!}tV!I2y?X2kmBVX~*95UiKyzxAqKt^Ou<)rB--c3+9wfJ<-4{hHx%94ojf6VQ^P<N~%z3!-bwh-}>4%*`-#)+gsX)o3#y~LQ}5+n;L{mIc?-{c-rT9;&GiyFW$JhYIiwIP3XzV(1B8ZF+a^$yyD3>Qy<vVQST(ePWD=@HoCg&^G&Klj8WTV-jD|G{iy083f-VfCXmGKuy+Th#N$;X{;}Y4QX<Y-V`H&J#m6Ba9X><vp{!1i6NEJGzMOo#i|l>bV(+2CP1qQOKN2KYrX_xG)WQsI<ba1CB-}U$mE##8-aGJy%y4Tka|XxG_9RV}V+%X}4aLV2ItXwA`h0qFX}o9<dmN{F6gSyUgoDE!iF}%x&`e=un>7O-hNXH5_Z)af&?IrUQ6d_2WXlv3AXDNHj~EG=K${DiMy%6|Q4EA4-T0pHQLx!!S#9<qLM50vNx2t3s$&k)7^-9I9c+Lt%_^qh4=d|TVbsIS6*)tj?PQBs?zS-1jA^P&mnGB9r8_xwSKhPI!rb+Zj7(vCnYUqG;oOQ`!!HJgt&c0Kp&_t@*^D>o&ynp;?BcVpa3C*H_*Z7d!8ZDv1!`!r?#P2@v+k;2@P}sjfgk#D2TCJNkuN6&GK`fkXHP*v!dH+ta@^?<jW21HBFYn!FX=78MxUfa6365!Hm2>9w(uQG4<&iUD85tGf_iK9G`i1Eot2*SnlMPd&JMH9V#Zpe-LS0%ar@ccjpM%4X+&0RqTczpx8C=!B>^>AtDjk!j8?Cvhn@9_1D><$?{4ky+^C1^wb$$GiONzVE+`$7(P``E&9xhAa9nO|y@6V+tY);^MrQ<6KfgPpm9#Z%v>8jH5h^1ZC)gmdS|1w_$#8#j?I-*7Q2Gfw&1Vi@ER*v{SPf&XG@BZ9%}F5+z5~jsBb?aaRB%k=xs^m2DLt-g>+A2poYXd6U%R!xwZ9wg)pl;x_xzxkc89igd=d>O(FfAr3s}>~+<G8y_B01WGmmOTM=Z}~&+$Q`l;v#<7XT_VpxxG>cj5_6@hp-t0mVI5iMm*U5p<BBenpK0J<l7x-k^!D74h&DU_;0y7v^x(VJ)$OE*Suy!`w^PV@?fyZd#q_IPI>}je)@h#tv?7<b&WMg5M)xv6OQq9FqH}$b|+Y)OIr)ZE(iyAt`+#wMNqE_jp3?V(yi0p^N^rN4>$&YoFjaz1N}Qy+Itij#R%ogfj8*WN=jWI*|y14URm7`NaSBGI2pgD+gE-r4%@>x?`LE8bgq^^4Qjx@&q;P9+CK0yJO*RZoY67kM705Pt6yDr3HC|V9~Dey0jdWmdD>9&<QamcjI%>O;R4oI<*PQssLv(>vMc}g#jjgfenCuk1y*LY0t-u>3Gm?8oK7)?b;sD|NQsDK{nG|neQu=EP;hF3r?{u22_~-A`60xYS8A~s6cxja7iQ9CRIN5c5JH&xk%S!z00acp>1Jc?C7+qqj-7pS&)?Kb&u1<<ioL@)#zPFY5=3Vska$bO#@5j8p`YSy|?Q1O>e$PAY)4lnSg2r2AOlbvaZrH6@~)~`$^NL<8~gyk2l92;O3<B+Z=Oz$=gd`_KXTJ(A52XY6E7}pq~sQUR(A<d+)5GDheVkvEvtAb-ZlOj!su}-!<f^U0O@}nqtk|9z9TPKpH6}?g0f;4r|(a1?cD~*l!obA=z%Ik1pMDa)*hJGf1ND62`UVUoDSw2pYj1Ta}kBI!g>9^hq1a&kyLDqixKWCU0rHI#X#}fTbR^8&&^KM3}-Dj|aDVi<5HCyHYfU&Gv8<gzfuP>PP_&<>1`t1y1WhauWy19qZ`8-D){MW?K2*sg_)=?O(vk+DVZ~`G}JC2;?u2y>#!xtbDj(7dVo5I8wc6<MNWluYLajnkgRU4>LY(ma4*61gDD&zjA$Uu~aA)mzIzLX_oC0CA~yedQv#2RiZqDfx2<W6&8me$6xZNVqD4h=%e5&Am~O``sDBqaDdgLUaynW+9G2-Ou(zva-k@GlxN_<Yt>?5p_rMtR36yG2`k|gD>z*B)4?}V0sH9~6ir`YZW4!Ck<%F@FNFIuz0&XXF_kJ^WtfE4Gs&!|Ih}^bpa=slA0i6Ppx17OcTb>}W+^~8`(C#hW)vuo?Vu#{$R4rx)yS{dD~QfdEI)M4Lrq^SOX9MROTUqz8)5XaUjgwa6Cn`FD0~boNsORTa!flcG>y)~B%kKa*4voe^&L*4Y^fJ0c0XL-y1_nCgeQZ85l%(>n>+R0t@YdWRj0d@GuA2Dx7og};zJjm`LVawO_oB(li;*Dc(V_QtYJvS5V^i~cZkGY6JQ6imjqNd3oe>{+OKLoG5zl+s17pqPu9Hf<p(uq?H(s2XLcp_;vQBF@Q^pwcK5b+P}PtuXm-^q{>*Mm|LyGd;nm?2oY%am>cOaf*!NQEHuAq{U78-t(^V;M?U0ZJ$R#bQS<c4PZhLrw;lmqSt93tc9a8P!?bKnJcK3Gn$*Y`Q((HzvNSL&{nbHOxQlnW-KSD<=@Lnz7JqbKP`atMG&qQ+Mqss*-psV6raS!>-risylu}&2JDMC%-rl~by%ax`wQJ4%;XF_ZN_RetRyWfcWBQN)6d`^)m!EN+p=0Eq&`}`LRvIsRoQAL#(o_48wTiZm>Nhp9WgQ6rOitVD(jZsE*scTlvg~u5b=;BvB$^s+6uvd)zy?p%qZ=Qcb7{lJ>Qzt0aw*)Zo{O>5u2gk{Zr1XeZQ1Jv_)#v1G@|fVsDbXc+R{VnFV-e%$nMh*t^UEhQ2^vhn>C$6tGTW8sY_Uuu#D#>zJgTNXNko9;h{JKty4F>Yk4VNxM<_q3Rn&N-o*ty0ye1)m(icmrtV=S%GN;`k(T8ekpgQeS57nSqcb;lyj0BU;0%vp}GasCP8%phO$}Z?+ee2D=jjiy;8@0__^=VSVjivX-!2k059J%YXlJo8wqI+eW+xpu#*7q@Kw|nSf&>6fBiHiE{BEyEnGx6mxC>^p7=n|{+(V5Q7L@wXZK&^UbsAKoqI+Jot<P@#x4Ro>_q3^CzCzFa8=_i_N&!Jf>=pxJOq7R+JxLj5Xkma=-(WPauTdv672}m(n9*P{n&z}9>)@t>EGZZPfSV{Z~64!;Q!~c)~V(0_Z_romlJj7aIlRO1u+!Qq&w`g#e@tjrf^Dhd~5-luWZ!KOwngG$Fij9Y8xn95|RUeRpn}mE8DdL_mQX%I;$2)+$=P<SL6HbZq#7jU%2%qMI=O1dAEC0aQC{|S--yn!@n1z<8AS<7sk1=7pe5(o~I-<a}B*+uGzZAT6f?|TO*H1*A;crB;IT|CS*78K6VZvm@5joObX?K>yzJ4Nr_$AhZK+5F4eN4CCV}48s!(zfd`<@|HfDAVlipxtsFr>=TvIDThY}*7ZY0NLoEk_I2XX1sU=FEJf*qUk1$K{!Y<yK=Znr|G<wU(|1SYqOQ9>pz8($<PWmKc}-pEilOS+DKw?_fxYfBApfFm?8BHn*Pe3PqPnT-fobp4R5}4d?#L|C!y&c%UBM-&Rj579RC47&FiA(j$dM@;E@;AadcCKvR^B@gw2V{f(Vo=C7^9+h|?~A$+@<q|38>rF0MgD-7NEEj*cA(SpWj6{)usDZw0-R1EXtg)N3?J#u0Wq#tJY%~65f!K~`+UM8J9bR-*0$*1}BA{^*7hN!&<aaz9AwOejzkFl^O2F!?w9EhCLFloouuk$pRiLC8@I$3NJpH#J!1y7phP1U^EbAVl97EslCpw%xq@<YMTW~wtaNEeIWMa((!egQ6#T9-<VX;@r!sMG-d@`XoW^RubCQyWWRRO;(@Xs(8{JDR%5Ow|qi(5;=?_8SPm4}zT`_(`lgjQjY2z@&)=J)-arg!R1q^!a}#`i7M4GmCuwo{-KMTj>d7Bq!JELr#wKHL?RJz+(84NK3&G*pDgVB!jFz0{-B?NE~JpMD`(H;Xfy;`3+^yG@#b-g{WJ?JLz7hLNiMs5SgbYIl}IvLo@C`p9qyoizQI4u><{wOYS6JPx<+apRVLovy`Q(R0ql-ASO;k;^<(l(;SveRAPPCDb2CFnLxuJe!#(%s?Qe;*OlzDFP%_)oZi$Eh>TyUTlfj$eSIalNNPzT4I^An8-*}b&tNp_y{L27xCv-#*SEIb+}DUf_IxL(Gk<xNMX_mwB3^JyzKuv@E2|*160lYl;>9?BqCs>|M<L&XeYA^yf`KA2C{89$K_i4BMROTShG_2y{LvulqHEEJ*x|9}0+gJ`B{uNP<e&7+XVbtYNU`};-wzxthP+(ibwL}$_jj+VwllZ~M+Zi|AEJ4=45Tc@#ZYkR*tE7vdB>td4Q4S}%WK8_%gz^a3tJY~CJj^rC(c-rV8VP&2}zUcRyF6Ci3BDlP%lq_j)3Gr2AzzL7|q6p46kZO{ouf2A=3kKu%h}W6`R*~=*v@;Rx+p6wv&y%@7}uj@aeidcD?O>D>Bv;U?M%l63O083AV@y@G}ZrM0}j?;E1Xl+oh~J52ixta!rbBk2&`vtV70h$UmIU>yx3woGF(Krte?UN)SpzPU;q9lpb<`WJ+0-VEO`ksH<dv7hFO9@BcDXF35N}RW24QIb;5GSa$04589lfp%>q;OdR6By1MSv7JNfvHWIE?JpUSzZs5lXA#ay3@ZYNKY`Vd{#lAAv-VfwLj?<L_+x$;Bo77vHH<3hAB!u}m0oW(qMK&z)w~`BfCRJlHkXB$HYbXHB<v*i^CT5wl$&vC417@{GlPog|@mgb_Svh);i8Ov#&4L=w0N(x=U1U&fykp^LsGm|+u!pL<hQb9EJ&7vlmnxG1N-{|rJ;aJ-c<Mw;XbL*~&_>f(EiPaz!q!ez&n=WwE*4cEO88cmln7m<)R*Z@J7#KkGKRtjQQ;VB_650yN$$2=aPa-$;E>#i>;SZBzRPfSOY@5|%emn^963brD`hXo{|$fEsU}{+S{D1jOg>DelfW|vssE;6%t<-{-4QqWlpwEk$5Z&zdOJ<c3UvYooDl5sAG+LZ$A7^-xD++s=c5Lf0yC5r3PRxdoXR5!a_4w{?I#jFA_wvf#nQtY7eWD02=y&xa>4WW1diIT{Ry2`DT8b4>#*AH!e>T4YuFe`O5yBu!e_3uUQu>7FY(RT>_%sHZllC$2(CBQD|~^3=I1m^A4=uLw+X$ltqD&ps`QL=6C$O3Mmi8yO8mx>Y%x0v-N28bYAwymBeHRPq^$Z{T0}uAmA)7Y{H?<^>8yHoa8}UIm=UOIgBNJSdEX?B?@LCc5e;Kz;x$l%K&eS4I^mw>E2fw$H3(<cH&gKglhY%WbS(t>&TMV0-H;Jxw`zO!)l5sR)J+&0?V6v#GidsSMsF}cWG4llq`-kWMU~DTW0%66+FP0;-<%(ZP8#RZEHoY#Q%e&9P4rIMjWD4ix^I;1%vF*Q(RKT+RKK}6lE*1c&Wba6kV@yq$q<P3eEw^%>TTAfm5&X7PpGbMl_!j}_K0+NqSb;m$;7~9DpJJulrvx}x#OyQ<J{n}(*zI~w5#s>RT9KdIZH!Vln6;uN=rwS%BKyN`M}S-1v4b5bjR$ZOV%|zDJzng(Bvx)u88djRE?5d^WXQc#TKxSULciu*+@GgmBvrhW;Rm+UOuicfBaWu<bEO&ix7{>q7m7!pXfDF>e+AjTzo|a@kc<jKTFKm7P;=C=``w8?KN}+@mVutCpGJg+8pzF4s#}TH==q|shwV$!vd-NfP+2B35`jbbxrslp8^3KMA_(t{-bd3PCNkGd7#&FD1*{V`2vusQa<R7pjaw+VG!eSikmPlbTl&R?c#Ut>qnez`E~w#x<o(*UE53yOc@xHiH7^~znBT}5U|VW*M-ag^N4n#+=3}2H~pTLnfc(~6qHwXm(*3IJd4y71*ZiDdP4VBg7~2YpI=PnpC$*zmWQ&#@<0St_c<0<8bx@oZLY4}sO@d-Ok;|OUHgG>62|TaxJYr3h5=0+6Vr~yEAFzQ(anbH1Y((W&&<SVHT`m>yNTjmT%dc{v1v!WzIMVbvXEi1DqzGFANFA`8a809;5L$KSz4^QybBRsO!R?HWikAU%o@-WSFnzD@5TpLCTxM@;z)@BhuK=1s1q{3b1pa=>dDl+%|FvY{G1F<$YYaZ73Xq#IIG#3zEJ3k#4<lc!;?s^sB4?h!0Hh$8S+KQr#uzTPU!ViFcWnokqQI>8&dv>oWxQMgboG{Zl^g61U>`^|ABD82|Lko6|E|8uMgVq!Ie)Y7CYX&L$(^>kP~i$SiHur{^5F!krsAw6Ka*BTQ^q4{HFH8toaUA{KnSK+E2GOu?o=*^K<bqwdqx&Al~;$mXu7)_g+<flX2eDyG^Q2dg72H(kq_u-&5+-RB^k-3DRXR6)u(TQ%@&6#*$t;a0Wi;G)Z>%A7!?qF{J1--HR|qpHf3R9aVeId4x9#jYNzX9%bgmNGC~Dl?d&&R*6>a%pJ!U?Jn{pBAnxN)N8ejE+q9izST~X`<t0oXYyxZS$t?cpD8)!4~p5Qm*kjMBnpgum151+-oeDZpdERKjmKfDNJsRgCMD4O$0*)RbX}b+|1at#tU<b;<&2KXg<v@eYeM12?sjT(xYL@B5qZ+>l&5G2x6_L@b%4!jYoq|0@Hy`-Wzfw&$&!347;gP^SiQaGJRxE!oE2yX`WoH_jOOFKWL#&D?*^#LMGOQNaMtdk<2DomcCr5Uf!_dT8-q&`hQ#riC?DR71_Gee->L8IZ0&Aij*i=P#(LKR1#Yix?$r<k9s?Z_Twx8P7hN3!Ixf_b{{CidcX#d9W~!v%=W;kZ?zw-JW-f!&0sZfEaxUS;yb(Tu8v!^9<Vl=P-p@!)^N{b1&-9>3;xhcaGf5eA%*7{dt%m5(v0im^EZHup-*;ZBB*U{+LdFYPve0xE)>MP0#t4^F&VZ9kmoH5aG4+Yjg_J%+6-)OlTeXq2h|o&C0bzx0Z}}+&$nVG!@VU%>d--HG!6BB)H?)X-A|Roa{Q0@?>9gGX*>=aC)4xJ<0sbWJwqWO|CJ#Kh8FcTGNt@y@%W*^$I{~>MJ28<cq|GL|inz8}k|sMV**m$bz^Bs=#A(0Ja)Z&zjvta<l7~lyt(`sHXyAmORC#f-GRKT%Anfb6-94$&41F)PDMF)8h43VSUUQ=I6hmWgo{TG2yM(7~6GoQRq?3qTwOyf~kWK~pN>)c?i7A~}TwY~|(-=Kd|KrT?(-6?BjvXk|f8~71nzf6J0~vuKD0j<v3khGZL*wf%4vd0N)`hIlq~EhfM%3Io#ar4t0bT}<R>8SUiE6=hHw(i7y)X3JXT6a?N13P&ISk~1vGpQTtr)9#swo}HSQ>!;<J^UJDX#G@>30jFBRJx_Qf|&o>c2}NgmpIC1CbtvYT;dEZXZ7jG^SOJC%Ck={0FLm3!85kF91z{5MFuZ!Ck8*)HH<;+7~{Ju^rqsA#x}!+QmhN1wHG0w}{MS3@I(RT<s)ZRrW|Y3&Gk;UPi)FA}G-jFpxGPR0nRkxLH-nnwl-r*6P3eJ&b)t2y)8ggvwh|%>_OWjZ`>RA+kMal_(qNZaiqm!(0<Nq^g;m(;SDDKFZW=sHM;zv$9hRNMSxBY|FH5q*W2B9qZLpZ6zvCjK6C@Axti%n{;LMQ>#7y07MaK<X08UaR1+dPyF5G-+RL5$|);^Zy<$invoyUjSF9+qr?Yf_x+YbKayY#{u(*VUttPwR<H^i?@9Z3Any>7?Mb7%Tl;%A*SFqEn6Am}|5f<QDg2aG14i245E>bc%ZjNUUyc1tjojwzSJZR;R8oqou{|Zf`H-`_%Z6S13JSMy<H*WjM{$F}bn}_L>qI?c<wrTVsRYKc<Gm`UbC4R!g!<FI1jYdE8_jrWiHG{DY#DKEeeO--BH-WOR!K;Z*fRzl(t9u+F+yl$-W(I0HY6@|%D@3PF;7HrjF*Khmzd1NG`ZM~lZ-7n1<0<%_&yAbg;g)Iy`xUD7@reuN%@b=a>n9H`)+pL4U+gWob{krY7vBfBUzLa_csK1Z7iLPPgW0u6IZ0Ilet#kxF%+BTt%Dl?%vKj2WozTNvqS3DCs2a0WKtuxKdKPM>bY|)tLtuo0rT;GNa7o4USouboSriNv@<WcAd+?v3Vxdf=q1_CH2lP$RG4?rI<wy>TLH!k4NRdK=w_`Df{_HD>7EkbQXI8#edCyHD7ZW0et!k-8?S+X|%ZbC&)dZX-oH0-@)L3jS&95xwc-PL_0)rb_ICgC&?0oSR+gez>P7Bp+2zoW4~V&Zw_RU0<UNllLq+tm(^3FiSc@RsI}iu@i<d^QT6)=_K0+JS*Y54>uUV~O0kiFQav1|#P#31e4LaU@fQ-eB<1da>-m|sTi8qR8GWUNGt5d8dBxLx3sJ0R2Y#WI#6VE?LHwL|kmf6^UVSE9YM&+E&Y5ml6Uazvr`Wk+#<*qm`{Vtx3g~&2%S&>UIKGV4-`zDGtx_H_q89dy2}Hw(DX-hkX%nP1g!RDl(eLMH@Za9ue40Yb(v|V<Q}gafV0lu>kIIoxK#L8Q4P96XEfY8GwAU1pB7;;&IfN(|DY;GZb@_Nj#KY?$I>MF1?Rok+dw73MZh<1?+&L2Ea7Az^(aA|XOYYdSXzH15bkH{n(-U9_B*+|aBSlVUD<%j+^?4UB4!MDJ56)?FbHmB~Z|ttZg!Q3u6rZpwJN)%WKD>{V6S_f^ohMr&06c?($01_W24+3{6;3|s*Oa#mywZK!e$MO4Fhh7xJf@0&L-wqvbf4gAlkI|agI_Q{3Q&Zn21W}0=XlW9Pusx@o#ZK_S53(eWINmZkZ@oq`RVk<CGcq`Ll{M-6BxYH*?qWbR`?OyNXg?oJm&O)zMV(~EuU+(^}TlzhdZ)$Zf|Yxud|hwbFHk{*DAE(;x)wfMg?=LjXABY?<NGu{ln!yU;bgHgxUc*Q8N1L5$%RQCKt<NrE3th9rw#$Q$XX_+Lri@ils~Yn?LbL^YEFd`qA@GgnP`F&h3Tpi0ozl3F<B?f*JXiz<Y!%jf@bXWW7>zt?kI$+gpn2t=4yM?5u6$98cm=&icKCVYeBN_o~M0428(}4R4uBtp>*O5bup=A;bvYbFuGPv+>?{JFbBlTIZDd{kVB5U|d@qzn(l0pZST1B(M+kD|$+jpb-Ue`|4zY+^Yi)W)BrIwoN@$u)C{zwCJ?EFjxum+-`{6Y&l;GJ{Ut{#;gR$l9(}U2)nD<#;oO`0%=Xd4Z})fHJDUU)wNyQdqevCRkE;kar;WL!HEs~oFFlyS69rol(oxJoz-5tf{vj)dRGVv#qj_bx61*x4n~8hNjmx#pK)yN6KR(E%!|)h(>b%v0fH;uHpnB<(4>A(kR0j}X8i$t7~gXvKxPB4wGeqtxV7TNoQ<I-JAV13;%%+6ySmgUBf?_9gq8Hg;;pIl8VADI9}srjO?HJ*s{9Ir8mndkA6nMDR4~~<Aj%kx1yWfVqz}ERO9+hIa4Vh+i%J~chunZ>+e>qG(wgd_c1V1nY_@K&Ndq)5YK`K76EjZ-rBuuI!myO3xmL6kmFFkGQmSGTU@0&E9ub;ILjRg0>UH(LaEunnP0yr{v|-9ka@c=P0{pYfXU^zt*A!m<Pb&4HM_#Wa27E>`r#MkYOnnL>%8o5ppqf|{$A<G4$TBeJ4_!rwzF$f9egZlmnxhqpABN9XvLOv0Gk|8cog`Ck{^ar>5v4iEK!+x#$T-&UQ(-#?*~kKn*r0^)WHk^~fL6_`7w2c|#Rcj*KM}ouME(A?VB%np{F7K*F%Y%ZibRq-+82a0fPFQhqofpo6ODxRkv`4j)a(gbSzuqL56Q*(p^3Uk3l8mmAOrj-MJnQ}=YJ&@mOKXxGzIc%`cGQ$Byb@ANvtBvksL&(_6nhE7y=@1H-~To%@VKy%_jBuiys1^bbcv1T5K%-un48|iwpAEAa2D2yxW@Ly>|0BW>isr>Alo!F3iszEtHDIqvm|Hczx-5aq0S8F<!X7G#51%=4I7Ta=8&Tmx_&6+*+D1w^~Qd<;A$wgmQ~>rMOj0N|xfKIG%^M*W=>+LbJRyHy6c6OAAMhqeii{c$Ab}ikpkk!ot$>La98z7|kygmlvDYTSrkm-&!gbTS-ayta#KaE-uX#m*<Pc#i-Ppn_G&PmdoYVVjP!~l8wcMW+PsjUv8F;=I2XM)R>3DrRBI>T3DQ$UwDZN9=Z@*jkPfmT$dK&>p-6uW{&1t^E30Mg@u`=<)zY0qg-B?UszsRI+|a+8iMPOJ!F3t${uGtfm>`itY9JxN(%11(u@bd@-<&!x6q*%jVMcoN`Z|U9ppY6GOpRkpjeL@cf4N9gZgoQfh;og?qGiE4gw*ZQ)`I%?Dg-Q52=pbH)`HByxnIcC*Vb=O^l#Eg5QO#TU0>L5{{cl|Lo^iZ06_2&era(2>T`{Sw%pJS_P7<1!(k!2PtgXk2M0C&C-Z)Y+xC7K+fI^QRLkvAh%*!iO%p7na>*Yp7Igmo0Cj_<Bj@_H}T2M+Q!=YJ0bK8I>tpf^8gLT$h`IX>c&=BU){$F-+srECs)k3THfhQ!IQoa&DpiB7U#2GvFr3Z;l}>v+KsjCbOEZDdqKQ6rEIn(3E;+8X?AM6yY)SDpk|m<IQ&^hrg%nV0pY96*{Bg#;w(QdC3RM{@(fV*z~$Y|QM_3|$+!YIS-_jcCxPI;tEGiv5u~`2s$ahKddimgV^6@I(AVBcM5!(ia={D3waxAQy)fil7k7FcICCo^@960;mS0jv(>@6VmU^piVzuCZ^D7u6u(P$bH|vWd?9cj`?0un(@*C!-^JjB%6xaNM03~t68Wh-8RyeuaY$Ha;fZ){#@gct7MvKy2m+cPWUTCyQNV^)U9~g_)x5d%XNuQkH5o0DCp0--;`#I_&I{qYGU4e3i({88Ty_=KS?%bVGU^UI7f`qF<iiG!C)vquysrX|MY;?eqmMd%e1NeF2PMU7y$;2;5caZ)Kt0fpi>SZ6*`-3wyi^9Lcs}~lr+QJj!PgOt2#UZ<kK@Y3`HDkJ0Uk~>-w)2xJy0NwWj+gVV!Kc??X7;u`s~(vB{59Xp-Q3#QsO@=z+yx${NdZ{}0sFn5?(VJTU46#cuUZ`!``nq3avd|@Ko9UhBgq%%E}v)U80?D^IR>|g+<=(WeWl;3_J39oY~#&#haa0CIVrd_Y@~}P(GdX-E1>?h3X-iVbWcwxJ6?uCwN~)zXGj*L1UicpQe;Z45TSr0H$y5Z0!Nxvyyq-it(Dm9<&I8AUbpAbUh$;L#v7iV4D-g-yfCx5dUJhtk^b>u1-yuEPidoyy-{=@ohMF)E9|X5s~Y*g|K?wmO3PzFO1$NTtJX-TPwWR7nSPKaf>wNt0C`ovSoZT4J}6!f0v^=(1{NPk0)A4d0Hcb|8$rpYxly}m!UMrk>o72HbdW~LI-JBWw0yX~y@Lnsi`i1?5O`x~;KC2uL8+9<3a|@&v)aXcpt%M)qcqs%)s!F78cdpt2D>+QhWWsOk9t0+vTG!HMvA|<e1fTJnL^^8t21UsXx2i8=0jbPI7Q!t(E<jAkY)Ji`5*&?Y#4W9R$f+p!t1i=jK9B0CJM^nvR=PkU-yyjXt{2EWh04uBGDXG1Q+}@cytZOvQgY@pAJ??kQCK-U&8#ePRuQ;QP(Q#Rpx_ffRpmj59le5v}#N!IUZPOaK=j7NLqBWR$Erq<#%6ta6Y&&2b(CLo%tFcB(r~kpD7KrN*Nf)s$nhdx(%Q7RNWFLtx|b??bh1no>9M0g=c((DzvptZCuztvvC4%>xMEFTMB+HeT^voL*_S52DP^9ZEo#(n?P_-g{d~gg<Y${H!B6@U@nDl#;dE;-MmZ|8{{4#omF#yaO&(Sw6D?(06)o=26*jM9jvTiWBn$GB;r!d7v4hvmRvcnnA#<Sr%kt`2{eh>UlH6&gn2}<2f)fmsSJU%=#lh~WAAPai%M-g_Rbpm6;k$Q;ByHZqg-RZ(k~ERVB%{)D^_P=LO^~IAob=}$S8mzO>)Z0{@t<1V0v_?2hq5gu<&MWZGC@7$3KVp0=nh^7Gu0ITqSsVP$|t7r|@Id9S&gcMQsFdX)@A)xK5djtavCGkm%j+;>`y<BMA1e7Z~ly>B{zJT6)5-n9gO_T5)5Zt#qS_-3j-<eiJi?V`aqB#tQ7F#!+N2bgvh@8cbfP<kgGiSrGF*YHe6yM#?)n_xh(t1p0vciZhHN2+e(%84YiTfiGiJqO`kxV9`YSu9!f1Vw`kXt<L!i@>}y)=eOo7l{)fiX>KdwtujzZ3A*&)(1FU40cVOeW^fElR+Gz5A$`(R0tv;ifLnES`DyuzEkICgG2=#aW+F~qX^CkUcJ>QgK~`6^duZIvle%}l>pg(>*X(;>VZuFNX|kLI!A#Ca`<dp<K^CTW(N7T&{l@UK?}Y8z?(T~d{Qq^s)srx5#4U8(87`uoc~ilNW=dDPt4ushCmiM#uPE5VKnA-oDH;u=74?RVsGC8-$?>Jr35OgT2lI;hVvB-XD>fAszfqthgHSi+fF4RPG)Z@fZg=3Oyb}Q*yW%z5(Q&s2q!TZZPShWg)~N?%8>H_O%;kvZPoveCOPG2wtY%Q?IUf#g2R-;SOLeH(r?sy0%$d4Y*6j-~uf5vdu3=stqU9NcWK=qD09mZ>=(B>PIeFufCxu%NH_X1%s!}@(N!+S(Ir2yqf6B(t8C89mKP)8Pt$e;YqA<_rCKXvIaki7`bgA!72ciS6#*NLK6|zd#$ME!s?xxd05<jB%F(Y!LORlu=u^dAI_Czk9UpcG7)X<Gd)x+v-)-vKlP20c}M2CYCs9wd2VG7WA2)m0tODo;%L#n>vrSU6goB~w@i*Ta)Znb(wTm<x-o!H^0+ACu!A0bK$C3dd(1gx^Ux<>FTZ>{aUAvRsciKm68MGVh|vUlcrH|HupZD_+PGj1(7jmpF=4ObOp+n8nxgmUFA22TWZ`7x4)0dhJKjL*xl79ZByqnIk9QJB;fHElDJV&z|ue_NQEe>1)T`Qh+;<KO5IR1bXNmS;QA9bP;586BRVv7U+Nhc24a=6Q&d#Dn;m<oK+^M{(#-zVH{lb~opvLgy~3bNnPSow{HrZP!&J53rJpAjP_gDaukr;Qu!2t82B*oZ>HO>!Uku^AuA6CqVLf&-UPxg{G^xA46B5YS^Nk-Z?P%j;iImP!yi3;z&du52H>rI1gK}gToVd{r8}ptIDxltl~;}mZxS+n?cvjj|RS)IUzVI##Hg6<>CZ})UcZVApDunp=10>HxNI?sR5cMw=C~Yi}eh3ikAh@%FB)C7EW=m1)r9S<@uRXd1ek-wM16#wZeP7!QJp~w|5UnIvG-3++RXL<qX4iIB9pzlL>B)qm;59a)o1?$U$aRht>WIa@SW~J2kW46E95BDhs|*G4;q&D0~mRu*v^6?!=9|LJ6SzRS+0&lyMox#rZHUE`+GUk!#@MfQ?z5ivO(_JUvah%y#Z6U!IE48~tS6!>t~m-D!TVbeH@$hDe&XZF4md?f5+4##ya^1rKumY7rxZiq|WZRa7&eg8`iIqj*p(Rw~1OM3@h%?F>(7>wh_)d<}N$w?x(WQ~sk>ucvY+vj<H1c@ph@mA26P;R=nhORL`2p>4*h%1wd+JUN$MMxV`m0P~x7-0c(Yb?b{eAjov92V?>C?(^j>n(o~lo%Bpxd2~?MM<t#?uSOk8f0Y=KY)}4_Lv5o6w=set0R3n6W8in-3t$+qhEA`0EdPc}aCkaYNB~k3!-+QUMBU?9e~f2RJx(=y1UGvVZ%8f?0y7)RxM^ll?q{$lXbPk0a)I*U+0*kcMgf-%*lPeUvOKzn#P>Ns59CRbBsy?Ki`!19XNAUA=)0Tj5ao;^hYe7z(s`=rZaUN#2R2bheWDIa#c=(}{BRgo(#}2^rFHh5&oXdw@%c4Bl>X}heK{SguFfcJ{mCcfI~{(S7tFO`cxTJ6!c?{-n2Y?1r)o;QH#@IWb*E9US)dbV+Vt>Akmx;|g~^&vb4@nZHgn=r;X`1U*7#YV7t`m-EUdrDYUZerq7d1YxOA^BtJ%a#yUBu5K#W;snz4Og7@o~F&R^ucrj`D7bF;qV{l~4f&2iNjI{|AsvX95sX68~?#Xxo9tG~CZLhJRIT~Sr<j#7xtdO5S!OwDtexpz5QUD?0N<z9u1{<|+Z+@&;Ms&uW$mk{by@(~>}_7p<GdkXyDz$5Bk@#adw{QQEqS}b{Kbw1a1kiO}LA&?)G6soQw{$VQH()9UWP&t(vRc1%NQ&V~OXeG6l!K3xJYd7}7tvB~JgoZ^VSl`&L?Fj5#6mD+RHn#V7Em_cXUF8<C0>Wgw#14l^g64ofL*OC7hd40v#s!B>BZgcptu+njt=jsVX~|%sC3oIs$<@Vzb{9h(M+SMEQ5HijB8rwoOVKe***B(`T+>r`TD95Df>+mcVeKy5<S@k328DF3qL-FZ450_PK9)cI76<dE-<s}F0n!7ptY@*uQH$MpsLK*irBDz|bN}moP<MK4^xCCW$3}>KPFi#Mj_I?pOAa+{PIM4?uiQEBV<MX&{8G3puLNY}j9@xWM?LnnZ$u~k)1lxS;(<0&SS@(at@=3;cJvFvVEefVxb{BAK*8f*9b7(!H=iHA&)F$NfYu-1$LN^ne<dMUE}y>7Q9}YD^XD8Vg{cBFgOLiaq=TY$Z>pcAn3pWp<q8&CeG^;OTihxc)?3;|-@({xeM(u}+@Ge>QKl4)z=cHj!ju;a@<vnSs3bq9lXjw7G%KmSJ$)t7{ftU1rQxJ{OjV~|(>PtBUT#co%L$8?4?L5-5~m|?k~?ICc=PG5UHFuCT!x+9^zI7ggM|tFj;4SAk<h<a22pz$d$)1s>VrXVkn@#v6so*Y6#1}wqqa$Rrp>#w63e|yO_~rR)R!JLPkLdZc~{icPl6N1=}Uj?y^h=uh?F2$!c0;gsNrd6<n_9p^MbdAnP`sh%rqjB%ZbV7z=WTyV@v{nCG=K6fM;N%vr>Kt?*_wB;omijNFniLo^yGlt4yy>r*z)M$?>k1Xou;CwW=RFpPkiP<f~31AzoPdgW{nZ4UYQuD&^6EbB-~{tF>M7UdymEx<hO5>K;?x85CqphCkjpKTA7y)#>54CUJSKT7>5Hd2?rnbp-uS9;7%QndwR=>xmb2Mv&{@-l^YStG~tPdYpMRK{ldwz7TAVrfN;NCeo-P?+UZDQ`-P$q3;{%M&U5d;Cc1x#o2l(#rtK>fx?K<lDi!WRggxuR`*4|4ih5A$>t|~rbU@_*tyIO{cy~CdckBT#Ms2b?Y_CTjSizQM#<2-SoA45`~~3fO+7zuIa(0S%>|i=A)`T!W8B9~=!>!LZ=3=XJA>yL8}Lw$MfDEGg0d75paYChKEe$#=$!~7IBd1V9T8QMI&YrzIMJb)V!?Oc3<5cVq%V*Cl>};p;jc+i8?5acuJHSmH;^y<G*{f15ti-57p?)KHbbg~w|o&MhZs3gme>TJlhI1%B;+;ix-DF3vzQ+bF=58W8h}gxL+1Y=K(mJot@R-x9(+g{@1Fl1#mH0sIuW)m@Ead9Xc)!OqoAR|Iy~kiA<rmi{wc%vku37%i(*(Tmlva>xKW;fVWEmm!m#+HfPuaSu#}W^1ZGQ;L4f9Ap**`#o`6z;DFZ1N!Iu>9|5*Yypqh~g01{PB%n)L${o2SL%ZS@g)r?b?aE3x;*cPlsg(|}}JHzM)6s)Y}o@I<RM#@aE6`oNY9NYZO<+tB6#z3(Y%{Q(uF8{C?19NlB(_suOm70y^rFf|r&Bya|*XN?fQM6ntFE1Atig2qolGy#YaWpsIYF?jjw3g<ZP^xvb+$hD@<8otud9kRW5zH?|jbhwrwB}po`MJ{kQM5F7y}8_IEx`XRFD#9T@;7J4jPkz@ym(wLEzLBS7LR7;qovkNbD^|2Q(BCnr}Oc0vv_pXD1W&7iVNQDdGFu8Fnk!%v@|0k3_V!Tj&L$mY6ve!o8Al_+yg2Wql*&|)@TTH2yJ{ri~&S&0H(zrTxfBpB@`V}%R%=>CqBkRQv-}fApMSnzbRw`7g@3Ux92M!fe?P7A$<_y3@R@SP#SsK%w*1AYLp6T-=`{X$Sy+ecx&15{kP{s(><1g`;({jot>>6Gh!XBs9`$@Eypg7ZBIL6p;u;T#x90tly?~AS6DICfn~~B5v^fAUl=w}Yz!liK<|<J^U&fd!cqGq7Qujqo!79Z8mMc+*(1sBq3ohzW*}fFwuq~xteoH)aK7nrg~-jl{%MERctHY>Ci4YM*CjNvDP_mIFFl~C5aDG~5k)bD^g=}7p{}4(5w9(lH$1az6r&cE3hfEkDP9@Xasj(m(~T`Wzwi=Y(`<h>SF{uteh{8rXlg0khV6#P1DyI~cYgzQF?AH<Ha-0G)uOkDf0w-aIy^6W_03hSI)W=bi>I%zI<~&|;z4ivx=5pgDxPsYL?0PcDWTf}4?a~hD&=UJgx0Nna(WV)yE6Sv?QQLw=>9a{)NbEOzB%pQqYDj6!{+FsN7n@B#NEBs)%tCO=uB#9wugOSatScNu3&Oxim6ls3UZ-2_mxt9eW43r;%r_v%j@RPwxoFZAZ%BP<>rM#Z&(Rm99tqpfQWD!gotozh=@tS-*9&xz#uN*GCcH>EO~+C5((#4fMk6`obD6gZ9<9>2hxumK9l==a<SyXcC}ja=yLcE3spD|^k=oImCjEU@i~Q3Eo>fTC`jPVSjAdtVea|{>(=z5lXYu$<u8bD;jHFGF)mV-Nt}!DXKt8_@H^pLP;-N-)qSB`tO4(Eqn<-*rV;OaP|OEo6^}w677EmhM3d;$BBBr&4p7PXBhqt*wuft8sH|&4u^_CrZuvI^W5{8P52%t0??+YmGdg(pC9RToUpl;S4L{WJH$np77z?furqMMh;wr^*xg0}73_2CQ?(i!*P(Kb;(s(K-eBqKu`7HlTZJH{{JO)1U1)S9vs>2_Mt;*M_d@8Q_aqh#z@aSCRju4fXu=3%ZDDgeRQ6W3SR+Qa0YRD7OvV@6)s?hcBqb2@!eW!K{CJIB`G8#VtL9P?Vo%V72s7+1^6@RDxll?W=&f#8d=N2rBOz5>6);3oqJiDazNw_spVPr3-iKYvk4!_gWXmcwtpPji8V%)?+K^GdnxG;)Q)b7B{(J3N5U3m$%_P^JiN3k?l*>kDwl^u=q0=uoYwjR<c-rOS-mTO=5h88mf&fs(yH_h<0^xV25{N^#hZ=r1P7ShRC)|}<hY*ZsEz-Yn{^Cc7HYLL+<-d<xhsjlc7bjzTm39@HEbr;Hbv6Jf2u0EBnh|)OMC~i;8F--4P7cuudxRhflpGuu3eSs=ibP^T&L1G3`+@w@T+fBosKg?&Sg?HK6bmld5D2cnnHY$hl-i`*1J81kXFt#7T)1sdPvzc~aI=O^5I#K&Xpa2r#k&=IJpbPvsT^Q^*ObgB=H8VS83(9cJs>X3!!OK>hGOP|Jk6oNo4D6`aZ#OWJPqBo?0i|JW0`J3-6||PhYEUdgvATq4N=!2XPAw^1nw8(Ayo`%;XsX%X^0rn3&&<Z85Eh}DZal(>4q07!c5e2%tG0Oz-Q9v&?Y7YksCyiE3b34Ok~rz@c*wPtXDJm&x~>IuZ<2vc9D5S+<v<kynQ!z?WGxG>8m>bPGGc(;kpjlF<E|v&<X+T3yMefS<S22*EVvcH-&wj4+zk)v#gY*QQZJT`->Bm+m&`I*wm8BX+}gQi6sCta*Xrx5#^a6sP2>08PHhtwbZ1Y0Y7awhZRgcthqI_#1pW$W4`UFf07pm4Py3d3I2LhSAPLEa<KIQ9P7;+rY_u`oL<=qqSpo?ZEZaMEDe8m*YUzfg$u8s9Nklzj{T)Rl_fKU+s8};sx6r4aJ{93s!i1LdSNZmsjnI<-UH=745XMTH5vzET+g;d+=)s0W>27>JOTeQm{uScrQ774l-V(tY^*qk7MG>CE;pquu&r47#F(L`X<Qf?ol5skly;JBE>%^h+yognb<mQVZ#7t0G(cC2cVS&WL*K4A8%%FXrjOeJ1A38fAZ?-#;Ur~2A?-6T%`T2|Sx2z=dxHqiJQElYR03u<O17UeqMl3)z!Y=2y?~%O#S`)*P)3rWB*Sa#)r7mM%UE9+jp<l^hu`stVw{*Q6|F8y&a=GX*2bgV}=79Odqxr?CF*kEmS}xAaFE7o_G#ZWNnYsCerFd~}er};$zN$Ik_F`eaP())u%9!<}lC48G6A_S6k8FHTQpvVuBDaTZSR~&<3t{{Na!!0=7#1+|Hm4w>v`bGezd)|RQqYT>D+sR!O^GyM(x~;6d#J$EpNix!to|n>!BaDT0vRtq<WzNm_xzXmH-RA$4EV1^Cc3}9f^A~4qA=!plrrVRN1T$1C(Sv@lnFxEB{EEnV+<3E^0?IYSJ(CoMk+j8l=*n74+i?_tqe#dk95@`?Od2uD1fLjl?c>{waweWyl(BhV^zUS-UE$>F$Z?}I~R&Q8)6?3uDcFa|8WH;>H~T%Q1{9D%((ripJQw46JACE@-Dn%0;6%{>n{HhA>YYJ{p*17=)ZaXF-<)sh=XUJKK~86(F!>D6l6aik%|d~p8)u$*O6Y(co=K{F@NLCAjXU%n|cFc9_fR}oPYqkJO<8Qw+U0vH3Kr5zdoaEXHP`FGumNB)AO`A<ZApxLSe(-pCfGcAB6ss5+d<zQ;$AE*h9*J@X$)k#7Wbrn~yKQw3<RD^zt8IW`s7?wG^I67yxxYqr>^(^IuokW5JsA>=a^;8D(FjZ@?O?jvy~f`7K<~-;Ue~9|zJ2LR&)4pe7SU(EAUbf2i>(7t;$mPK%&+sQIpMH-iTk`IWF;H4dx((yAB5#%kD?!J&qzpBF-{Q?+E(!yQ8fi>l4m2_Wxbi*L}-kR#+K+Y+=ktn5A}-BVn4N<^_0HZnU}@3f!N+9c>F!pXnl)}5vWAz;GF3)e-3{5RC6lJLsc7Ms>?!gL#-q$+Opdg#eR?I#)==DOQF=X#OeT|e*W9)czDl*+xMD#6X*_(E!#B-SD&b*pI0G5Hg+<_u*!$+8#PL#i(4>Zzgd5}QhUmQU-yL?QFBYl#HAjM6cRLt3~!vAq>_Q7Il;+Zy|uoN>6mQ(tw}uJjvJg#fr&twZ^Zwaxl!xWBo#w$53IK*HQ}$kL{yh5!I<fs_;K%!b5}SlfGtaC5d`Ly@MIbhi7OYkTYgOzw@=iM<%ZIG7qS^QBvOSd=Qdq~kIUhxK$;Em`uzSdE(96RB7OAuTQY*`*8C0rXx@(GA!6@?Lg`l_`RgK2nbA-De_?U*}SKK(@&2pd5K3!w8VEL?#99l#h`g`L)g}n+%3vk~v)HS8zFht#Xx7`kKUmNgirn&YH{=J%XklKmXN?&C2=?aSu6OEkh0q64G5x_o5>ow;+|z(Y&{xB?+eS22C+kD;AqiJfgVDIcKy?E%Xql=_6_hlSo3Hc$+^p^0S%li^$b|lqBzQXI0Zva~aJ43#aFbi*xZ(>-yXT*G5%ryldlPfwOdD#w3}LPVzc`Ex465^DHL%Q{<0$^no_fmrpbw$Ny$Co4}w+ze4gs7>>{r4;NZ+w#bV;Mn?WKm|o>`^)>O-!Xt~BJz^XGN^xpm^1@<~M}fD4j(vf5BO1Nl10j3G+uq(-x5T+e*c;9OCbD^{q+HaCZzV5mZamD{^>dK^KH-F5%ouCJM8Vsu@9v3nh0Ob>w2L0HJb-8GQ+C}Hm45s^`%^B1(0?>9(-Hk(>WSi<Lp@=(ZK)>~qVmylu{5*TTxibB&y^QvmKWl=nQ|k(UYd`~i_7zKS5r^e;x%8CtM%QrTWD&!{1@joeay=$JTN|$f*Dg3c4ZdQ9c`jnxra<&X!eu*H}cBGROa7M;x-|{d_X!6PJ#T8nEPLl$odthh7A-s7K#SZxY&@&-SCl+WJFptd}O#sYnq1o+uQY>wGA}1ZLilhO{w{eoqhZY-k89eqa#8<wq1UN^A`AbdV}_RveX+}J8M6MQhGJwo^JF`PT-=hh%4*H*2V^0)5=nbaE#$mr+0*WF#g@{wtB>@nw+XHd^y8Tbj)pv+=b!x{_E>&BGO{L#<_=gjZew1E%|W0_ByQl<j<53d`RXc;4%+<jR^UqMDL(gnVhQ-le-z6^brzf0CEDp#~p$2b)5*Az%B_Zjc9a48zk^Mcsh=cFhy*?f6@utt;D?xf&GrrzaG!+5Ob`?$44MHwA%OKt>%*jA6aG1iAz$=g8VY*oe|uT3>#qu{%MymzQP!~I}8b2Fhn;w@m{a3UMz*cTcgUofleL7`6R|6#}^`BIPEBZwSJqz|Hyx2G-Oa33TQry+8v4&IXY*G$hp8)JPdLk+(WK~hK^P~=|l{_NM%204||LgHIzZprrbS@<K{4I#%H_((8X=HUSHh}#G4idAc}3Zwpm*jZ}lohAgf_E0${~@opu9V8xY~3fnF>pC%vvWJnf@xVTHPJ-i2Oc^tEt2zyzl>yT0p;oWTMjJUGa1tM#`J{04$p;=GQmwbbCflWx#_B1`LC^N0|8g!natpA!DZ>-9%7?JhhaM!FGugu4(pRR%~A!Z&KWZ&cv~^tvRx+=+&Fau(Il2;4SEEa98=cXq1>GA=6c+4#XZ$VCqEz(`EaD=eLWB|%Xix`CX8AOqtPRny`0c_k^`2_}~jaZIP9UaO^YJDZc8<v^oh#e;NYIV@|@Y=JLi^*{<{n7P<(_m*6f)QM>UA>Xnp>XBq8B%_*WmY=M6r2ly;G-*$m0ku_s4<&C-JQUYRJj7?-+a9qjU%{D*8XY~|9YzC%oBJEEZoqSE4%76jocK@kQk{TsMP3=07+Qn{(<5gdZ1Is++WweJ&{L7`7NwZ`=i%Lxj5<}4=<uu;4%-ywMP{<laNwpV%6mH;Si3#lNk;gvCI+qtOmfa8M6MNc0P3OoL+y0-I%O>oiKAH=;pEMRT7w*cRoOxtNO;5QftFcEE}P1iL!T7{fCSt6DSr~zchES@3m9Dx3%lnz8;cl7ilYG-{+I@2&0{N86&l}Hoc@H30&Ic7Xox<ixtyGKOkNdQv(C@*`6&ik!MT%~yM`l6fWGZziU9nk!X7ikBqDeCdpb;7>53=wZeM=Ep>2OH7)Mqnr}JKm%()ce_5le@zwzBEzpw)VE=ty(xsPUY++jTKKz(3uo%;-&c7O%0VSz(uRI*wp8v1>h$Euju`!uHl|EJaIMWd`wv3C9x|5HG)e^>`U3^&$xch@#=g|K8gkdSla_p?SEw^}hp^#H|*bLYW%-yZ6DwcCgKqro6LhXPG(sLDNF=^O9*#!Isnpl5fnaC=CU9-7L(tpDu06H?)4V#kS*Jn%Pab-&WZBmv#P-`UvJe!W%O)_wtJRJ43M&VzpS`~Yh`4Ep@9bG}3?G?E<DhpZ_68yyXE=QI60W{~2hRgvM(21!vW^vWycd_D_}QuI}QE^B6}@mcvq{H8b=>)C5X_HodLi|QRzy`AUt7h|qWEl5(CP&M6@C)0KR5iUOAYay+;Qbzxp?YFEy`qoxQ;q6FT&{<#(XOYJ%Sm`?B6X5CystveZq(0)rA=&_aStTDx`U#)YuY4NKgXtf6`^gQ$-k0@_?Y(!xn>)1)J4~m=N4GG<q-m@M%o)5<-g%TMsX%o|C<m0%@P0mtjcOIN3JT3ZuTP|s5du<RT>^=pYRtb{Haotyxx0V!=Gu)l<OhXwF9}gLQZq8nSVfiUIhZLOuFx|k@KyDGMw2A%*o129AZPUbRrOikrAbW+E5U+4t7=W>-L#OFv2|cHS53O<{wUcFIOQZJuWg2Vq%=oh4V+6=M&|M2W1$|qs>Hq+HG*hlRtO~=^}$p3VgliPHR*_OAEn~2$`2jav%c~ze|_t%aCd(j-P0<3e5wyb-4_9M`GKnS!na{zPzEH;#2=;l^rPp0!!~D98WS`WT%o4=?J;RmpO|Wn$4Cag;@X4;<@rY(z?lS|vtI;RYkwtGI~I0KvPpmp=x67g&(1lXozcKJJQ-&OD@f-@lsPT*cJi-P%Z{mTo>h%vbYS3Ld0v`l*$0Ij<S~0lbu>80nH_$$d6s`!duAVuq=}4kLD5V-G~~9U4mM!cUxl7!e<&t^ItK$!b|A8F_(s(XZ{=o3IeiQ&4NkTCYz2q})t;Jo=w|b@-{A~pxdTH4)ye6hWEq+B+udGA8yazb;u;cOBVVy`U-}!sR;8i%=>ta>$`9UnJRCXN&Rpu(bQ0)nl~;y!Go*4D-a%ww#CxFDgl@*!D2`Z4G>&!)8S=TiBdt?HqY(AcA#@KGwgM7C<iv;&l5GzjzUT~KeLVh79NmS%qcf+YX^d8U!1Lu-Mg#iLufTr-QW(KA_%Hu0A|}+I;;ER*iuS|Bh+#s)Q4i!NR-%O*>s4R%U884@(wl2<*H<YWVg0sK_@&%PJ;H&c_*$ujLKWVH4Y3LHV!1&m1RPE@qZB{=W??rTw1HbWz!a<XJ=om&LK_qGI8iB+k-=;pBtH+uMij{q#05iBEFKlL!6yU9f=FH+P0=K4)!_y8-B*Vd%tKC;KuHxsUVPymZRnlCXe8cpewv*dZ>-(6@(5bVpUvyU%`PKw0qI9)64kJ``MP&&eS@JAPTE82CLUxI;60S5Pbg{Q=IdoG$d`jDbZ~tH=NaCGgD~h8(E8)GktZH`ufyMUuBA5!yq%cLE4{%+`+mEdeBbCqLs%aX#{GI510AByP3-KdYBgtm9hjBd4E4J&;s0U3b5W;%C*q;!0}pm))M>vLW5`6EqgRLIOW<{S$1@lTMx>z`L#GC;Ocu`OGeUpMn7swpEK|Tcg)A~v!KBte43A<hYc<Bm@<2VIl^+(oo2crJpbPh4bxttu1tT*88CF98TQ(OlY#(=d0PAHQ*64i2<HSErtfGan7l?~tqv)X^9v=(%Rs_zwL-Hwlds}SJ-kq3cK+c8+T8u}kRQ%I#Pon!dz7EK^kgs@+PP^Ynl{SUv2!u7*QX)^;C>8uZ>}k_XV5PVphwTo<@{o=i@ixg}z7$_y^6uS%cOp~hAa2C%Gi(mM!wRl#V9#933qzFE!!XB4A+$xRr9v@~6Im@576PfU4@GM7c08z-iiM&p*%nq*;CfgProZzm^jucTZ^SS0&d~-JOTWmc2hwVfcnog`=Xqxy06dzFcK^K4>%&=roi7h3VGO|Uu^)*r9{CL{Mcd)>66wm5w}tKm9??Lq&*xX%su!(90gm>as2}HMN^agTDs<swRI8;*RE@OKL1uX>0T;@Ts;ozl>A;g;vP|jlwd?Z^^h_ilM$88vIzIz55$ZP8LhN9JE#=N%EebwDb_3_5(`j^iLsVA5uK}Ja_$mN;)~S}_ng55qcW-jzxbg+RO4*Ka0b~IL`oRa<r5UgrBzxU#w7Z+K<Tkbfj{+JZ8h}t}k}bM#=;3+X_009P#~ZV8cgC}`dt?9G2wJjiN}{NUoi9Lsh500V9+_F0RapgeOPa^Vb%e!6Ju<U0Po6x_-?{0{@UKc%Z8!UXh3hVgS&m>c2I~=yz+%Q);ubEjH71x(5VNAt_TUF>i;6d$L?ENlpU|H8v*?v>mc5(h%ofS8i?J)+hlL}!0DLEs79OwN!omf(1Fyv!Dc#cdFb5GA0?sXF*AK(d@OU!m)nJOR&WcyyXq`_-r%@rCw@dIyVOA`b&aUV`J>X&}tx<ei8)l(w+0DQiLxhMdRLh=eI>T;?#f>Kw{3`T;MunJ;@H&tj-ooCnfpVo%O*T<1+5iG*Kc9{Z9T;rZnumV-&+vXp^f1fF{0Fj%d}YGpx#3sZy&ml1LZ$45I5T0fc#XdHDsa3jI=8~-dy_d0=crn1dGe2pE(bnL#NnTb&WKYKunXuwtn(G>DC>1>`l@e?K=znH|11FU+~Nlpy+Qg(I;<l9s8Cv2DSPn$V(TjWxL)?I!=G2Q#j~;=;F`YVlm%EF?+F6h&W|TO-`y$QzH|4WbldgbX?GbsMD)i0coWL!!c<QLXQ5h#tLqieTIMLs!WOT>kAeJg<2;+OKud5^Q1+t6KoqaR+c3C%PGQFMF|Qb$!og1BP`sDmpzcu*utPwY7=$vB7kfE@5rB|tqhDIO?j-C7K0f(`&o?-s0@+Y6pjnD<Ogirf-9j9C@<TFOXK^2)Tm5)T!6pMD+t44ta0@@R=`tF{kcoL9PKgci6$t(`pPi0LEf4$(UnkJm_G3K-(PuW9X%WNs(&x5lK%CvSbz5|%e*IH&5vPwCeaz=`*gWjjd`919M*n9qo!B@$ik$4RN9i$|A{@UHP1{{c_f|p4K(UxK>0Hso(<^#ef-GrW?+F?dTCqS1!9Z)w7?(jAnr9!R^h2fxRDt?<%!cFc8HJ{h7KU}wf3#`o35f<tP@X3VN-kc>kOM71)F@epJ+*Ey<z8tO*GHW<)KkvxOwcu&Flo{7ItU1OpcO<n-|_Jv%HE2yk^ZbLwt^5`Z9QG1kM{%}WHl=H7ktdgX=0DlV3eJ-sC$eoX2=;&6CQQqbC#3Z@I@iG35oCy4g!c_4?#^e=6O06ild}bn=~bu0JfUtx10{Az<9?YaUsbUXa{&iBxEo5FnGnURs|>o27~!jZ_(4FI!?>8ga(k~*EhllZ2kn84utqqp5Ws5@#j$-#$v?XxK_(q8cMHa)l0~jw$rHpuznX)7hnC_2(6BTRKCDGE<`o`+(_i4!0W#dIZq!8&{PrVEq!-%kwdvv6eaZFD16!H$rXERKJ#P8InRFVINx|{9_qJoS>n4XS2KIZsgW|zV6n^`c6w&@ljrCVmRJJ{B8P87w<QGyQBTOU*w|#Tff7;qzKA-stO_(Sj?nrV&QM~)be72V_5^_z;0pTeUGHFssngph!1CY|sZ3-zIP+bGP<M;P3zU|R%NUY$2#wFNc;l>hfInx2VzE`EX||Qe#-I=kD$oZm&5s2Pq=ex&$Kk!nyfMb0=UL$x8V6Dp0gb*3`rKrORUK&&Z31BD9bcaTTz=Qvq*LxN?zW&0!GrT^7KH&~ltssrQO_^0FEJ?v8LCEs!fZkCk-bNg5y(%zJBr?&xi}%#VWLD;021~<iaDG5ctx@kpj?{(fZ>@ynjoJ(9FBW#jk%61Ig(q0Haw;R3X}18N^AJvg#DMQ`AGuP!dd?ZmHlf{*^TT;QN-2Y#n}yojf(7EqiR5raDI7p-NREDwH1F2%XdJJy~zn>s)c@mtmp#U4bP<F4@B&Sx={2~jIHl>&*1d!{zsr$6>9x$QtbaPi0t|jM7DGH_I+irFN}jw8tNl|Wu=0X5RAh1qcHNTOmDq#IzRT;J#r#o$UM%SU#_e#-sGUg2BY>#r`PsDH$lg7_jow!1yLJUgL(k&*#Tu{e&&NNG!EVY5yh`woU;D|kTFVzZY@*Pjm1YYBTUIAiN;Bb`IAzdlsQr5B`#-1{2xsI--jpxS_^E!al99WR8fw({7R+rAAqnqyX7*ZZ8MlD!Fp-ax+M9Ia8IJ+)M9MccL(>Cvq`>(p4FP+<r6-J4X6ERGVDcZ-<|Z=ZSU_KuxkvRg;oPcT?6JL<g19F^VL|VZHx?y!!6hk+&!IPT)J>!mM>4nM>$0LZ8L-=SU`GbRmz#ItNpadOD#nzPj@6PGh&M$P?p~!NM0=Ow6(oS>1g*K>>O_I?QA#pzqyUe5&o@>ueE`5ztoLG2oTJZf=}u@d-v)=HP~zH)_1mlg5fB)SKRz&!Y4o!)5%CASa6LcF+0KiMtwJ^!@tdg!=PGuE5HeC-acsTzX^?VxFyZ!f=)CG@8A^ZhjT(B4ot?`MED7iG3MbkIugXuN?E;%7wklu(quGsW=9t5TQQG&s+>X#!-MAir(Mw_kPOD=^KKK-+Txm#Ho>A%rEDe!oLekCM&f{dI=BO0^ds^DU<jC5fL0g28K!UAnUl=0jSoFS)Mj7I^@?L+JsjL`{;=Wl_pDs6<gQq@lsZ~LfwqtNJ}8BvJ1XeI8nGHg0rt5qOIZ}nd`jRHa%(6|eGGZdNt__QB7`Cg;6#-n9CTb^AACF;qREmFP9lDL5t~4%C6M({d^UJ}`zO99_<Dqx@x2rUN1@H~g_&<3i+EsR=F@G5?o}E=(jsTi@W}M2$k6s_kpl=Q6wE40Oc&aVX*^e5*aej9sztjvG#uAv35?|kid8(Dst|%2e~T#ifZtNF$A-ZjxjI?a=UE8D?J2vne4LHWi)_npCC#Do^&?*HmUe;Gyp{gY_z9w()&gcl?8JM73?+V3(om}m3vv>{4827xLt_|^l2sS{_;p38t(ep^(yroJ$4O5v6!DO0)>LC;92xrK<>y($4_`RfZDmo$AXzQVY_U}t+eg(<OHHqJ{I=z|z$dvvPJZUPO>U+%?DQa4mKr5_>XE1)LnC3q@NHweMX%7l%70y3(Y`4$OkRQZ)$;24l|s8DUTqgM`O5A3L1Slow_#;11Xv?i4cH3yxsWb8S#;6KbO6P%ZviFr$OIw^m-U#P!d9lUp;Yx`{3{$Q&7B?0n>*aT-$<<qMjjh@N^v}{Xjx0Zu5l(2BVEO5*=D5fR^txBNV%;m$PhUZwWuv@O4^aRu6pY-{nc#xhr7)m?_QwcI7(i$tuE*Zgn2FKX}zKi?UMec9m&FQhKUCB?2o)zk-6zLxs)<VU7nqks$PrbfMg3<IqYH_8MpJsWRJ*~4chPKwP$w@n)9@LYZrt;u;%I4Tv)lglr(a1xV_8JWWp^5y8)O9<ADn%Qg9J#&HBiycmx`}Ti@OO2{&aSF*`265_>J_LS1@a$ifYJ&U2RcQHLd{WoN&hXQlHNmP>q3rn#$B_+mzux5@^maDO<*RU*0A+Jvhy9=Q5UWW~m3%RN{(KtGDDiS;VY^b^Vx`y)c1Njx0{{9-s$hI;*T0k|0>Qc?c*3&M5yLMD`MFf1y;g&|On%tvKNtN7KAvD|XP`Z5?zI_;4|U>b+b!}?B|<0sK9H5C&RR3mo->!w_=>=of;!h6^HxSI2TlJ{s7d-eUpgJP|+t`*oyR9y#EBn*1}pnH5eei(_f4O>if61OMKt{8Jh4;DA#PrmBQ)L^k6lEGCYCLXIauZ{Unsuka_SV(Zt#TX{^U=@lRMnT)Ys~^=-8#icjD$eb|Z6a1X08w64Mk^S{dot1k-4qFdFdh1_q#(d2u8*@xVj2II`8Fi1$WvJ+P?5@^KRlMnj94~|ncsah;JEf617;5w!9ubz0}A5Fl$v?sv{@>~wySi0Wk#8HY*Z;J{slsD@IU&B#D8AGVWA6)zWzlP28jhu{In=1B`yBf({iSijFDXtUXuewYAdb1x@^c)k`jRVQVCGD5ov8GA|g6DYUOmCEA?26NfC%*b04vWM#D~-@iYtXM3Zs32gg8EpxN;#A0CmI1ZNvZB)df2;ZWE@KwLy04vf8g96sp{2O)mQQLTF2;u#`r;lBHnr&<*i`{{Q}KmD$U`6Wf|TA8i|aPd5NIt$CuX{Rvz>35I*=X&WU?b3Ud(ygF%RX*eKYEc!a)-un_wS+fTSMUv*IrtYR4FAFTIWNC6>A?J0>QZQc#{#nKS{l_@S|H#CE~SKv-Dw=4`Gdf789}sgk$B^+2YUpk))4r!z%94$)He??xEQ;#&^mtxEp2R=!2UhV$w1SdziN364H~dR#oPF+U^#ARoKrtx|KNAi){h07vY$%nm`y$6Ei;kKaYLnb+%TfKg8jx$v%ckeI0si<Otb5vMP=MQvw4iP#W=z@TE03YvmV}WY&Q3|4$@hf7hoMahoC)nq+My*wHM-$E)ET(OB+_R1eZogT8T}KTbps2>6<*Imw=e()>7srb-N2HKrLx(%IZ-@${Bg17!K)W<bN`eC^)=~W;0y|a$GF(XFZ0+7nf~0_7f~<TRrGi$|*@+2t^d~su?+DrZeL=pk?Im^~g+78J<AJC^}Oi`XV3$n@$wO|L2IbN~pG<rIC!15_sAWA`kjlP@+vmq@{Ra5)~ty`z96;9cHWqrF0<rc(Z}~jl=!z%?oT8llpDjN81sWoMYXPfTy2+_>yLf;m?TEa|t!rajsl}8A&d^)N2@$3K4R?Ojy4VIio<oJ~WFu{E>inpA+2ihYrK*{65CIEp9MM?Z*O!hl`#d2p!`;3#ePdw*9=ETMAcziHL*uyyl4G&T}N<GSS|L)cH$?L~*fr#hQ0+fU|mrjz(W2--ubNeCwsEi1Asz*G4+aeZ6aMKAE>ijQXQ0rr3Ni3xdOSuxA5`_R?U_m7A+K*4NiN3&5V$WBFjuH_OheUkiBWmwGLf=Y&G|si;!%7Zi4FKo`cSP0t-dFaD!*cR0Uy+9_fE4TNWAz(R(%jc24<#lk)xbI}|ExzI}cphm9oABPdPUr?0vOR5nfAl5z*Kr#|={DofU=hH+*Aymi(i(@RQ*B_Y>)`-6QF{$q#NJ>l=LD9&GYKVMJ7>^RG@k^M-{|yH(`KUINgzy-Hf(u*0bFS$jkmWv#Z3c|t7+3s}=-Qv+WB53k7(RUci*Fbrx^wG#_eOuc`fpLe%Iw>Oh+bXmc6+OpmD0-UjecouWxZOuxl-+xdexQc>dn<#VRyCv^&q0D)B!>+9pE{cKqO17J+PpUh(d8aKj-3AbW$=3pu!>gOXj{w6nJ3S|GR)K(;(|VwroJT4bZssLV%4Yx64aGi`ergqKMER_1(%(d_jYH&LE>ieX{bEZsdx|?FF`(f>?)lUKj<p4jMamc-o9wQD`MO#l(D=;?vQ*Kw@KeGVYP#G^uUGjl@Sa182~CE;oX)Ay_sBr5!2)61OK|Q0m(|?zue~Bk_h1jd0ajV$UdJ3Vbn9)W9L5zw<Be|3=F{5w=8_;ol>*B=ETrxQgl(ykvGo^Zg;=*ds(emOA?=iEJg1VYzM*a*Rw=E5wOL(b@D79X&N%D=wjlJq>G>6)Y!jShQJQIa|Wq;i?QN5o8ixUAa;?E3H(j70kz00u@i}`N9^_1Wmf>tUZR)adlk-;o?is^oqAqS-Eb+M+H>s)Q__S;lgmnR||4Nj!X=u)F*rK=d@a${%}jXmFo3Ker3z&&)ksLg?O2|F3jZOdLY~p>2@_u%y&tx-rn4<?=TWvb9d)${T(7hPn5v)E#QURsy2LmL%L{S4EE}~AdEEL+9tL~C5QQ?H>#CN)|<P{;LZcWV;8vW@G9*t@p6Wb=?A{Joec{`RZOzF6}Uui!XB^R#-gG;13DT~=C@L?7=ghb1mvI$>IXrSd5#)9HFqQso?2PhY3eZzGpMy_?6iOgnvE58e)IqaPgOy+rl7+Mfrps|Z)8gdN!8M4B*AEnK_Rx(l5;}se%%={*y`UWiY$I6r8VK6;p!V-AaEc8|7pi7GDMJn=)hu$lO9b}T`?-MWJ+~Z!A*E<Uc%XyVy%pLm>`V}#%eT|ZzDJhC>_t^e*)Ntb~CiB6~K*qNdQM<jrO;<?ly#FDQNuYL4C*dbmNrjG-3UU#~|8IVv*oq`r-uLPH^Vof3wFOmow0Z&Q^2-Yag8v86UJ+KfK+<h|ZjEqM;-|h)0(Enj5iME_3s-#?^dqh{e+GYy$^wP=_imMNXoGc_FB2{XPyz60|mD;TWe3{+m5zdO3*nq!~CB#+VyB^@GFh&E3W!%;@gnHnQp6#@+hi_L~iw<KUovC;1sT9_<@44qs!c=@XJu&0i#@+W6veM+qDJH39^31Ml#rAd^QD8j?yAzDvF&tqo7URjdW?s^sLB9ubFVI#e|8UFrKaBIJVU1eXB*o4o{-N8{PLRSr7M`;+jQ8jFHyG;sv-_U>WhZet(IuTiZuW&YKd1jl#5C81ABE(w2OJYmUIIip+IlOS>~zL9AFI=l_XDTjS;Z6EA$-5C1>j5`;efU#}+k2~2DV2n=2fdvS;$8w6vUTkhio(IFtS!DV*;`6c&vwz~koZ=uG_Wt?rxIikG6#bpBas7f4y=9r<*B|7SFnf`*EueDX>4l5AR1_)*ap~f(sFbp_XXppnyQgiCr|0<HW3q7p-`%ehXVFQi)9wXaa%Fawr%$+VS#=1I)7Lx~)hVvR`VpMYRrh>JN9tvJSh-{mZ!giqe<LY`ODzAvJvVfa?g+6!w3)?<6(b}m^hk(VEEsd@xv?ytMZxv^x3}*;Xg)XyxJKmu+d;foTS82yzR8D=rxpLyLgz$CfbY9TKj_ep+d49+qSkW0m$3UwBk@;kB+f7!kb)eWGG1`O<akHC$gobZEy1>>;;NA0Nhj>}!d^f#gc~wS6qJu`2Ut;_WijoGu|(vIQpyUmBWXy$d0|t6syMARld4U~6}{w8$Ri$!dh~kl)TC2L7CV&o9j)w(Fu*Kf-YX_oFXrB)OnIgnXv!G65GchbdA76cB99e5Ml~LLW<M!n8JERHoqjx%(3vC`S;IA}+of&cz3TPK^;<W?1-5YYSiUX%RvGy@nq>akQAM@c&Lb+V{f=yDRGrP?K*N7||8MD!8(MwCe=ycoz4V-YDzcEKX?(Ncqa>_w&uC~gj#O2k*MBaGHqw6kkkHnJr3tO?s%(NnQo)LG9||kyAK>p_2^$w0FDcYg%>dQXejy4Ul1BDYa0T<+`piPbszZ7oW95wEnL_(}Nrtp%sLqii%FqU1J^P03+&6mN8@-kF8{e*-`_}53#m;T^ZQ8lp>+O~G)mzs~{ab6RrL~*uVW}Ukua$0IuiUD3!uD#vTD_Q^+lYH}J0Wm9QROa>-gC1Yt7#TLXuR3j#Xye-JB<TTI#GK5WUZ>_iRyzJ(yRAZw9uM4`vbw(&~@_*H}<I|#Le$5*n$}a6@j8)1N;M{>5yB}aM~o!uUHcMNR_e0dXO)S9U9vcECf&HVCd)>eV8t9@q~eNT)(^9L`@FO;~SgJ-8<X+_v?qv{kQS=y|?$8hxZx>4Z17dIe2jUuzv7E)1IElKLN^n7^=X-$HyoybDclmO$FdlQW;4&wil8v^VGnzz2{ntLLZJF1Xgux9MG5+$K0_hc<9zDXkWzx%dNp!&NBlo=Z`1B<MtW){)5hBGRGw1DM|1mh=5{|qo6JS=)iYPBY*YQE!Jw#a9pic*3!|yhmR*EXxf<$2jOIVJi&5=vIjEmD|gK2l39L144oo-C4Ulrc2$A)(-dky`wEecG<UmX;o4D>EK}jfQRMO?i&~*2k!X=Dw3Hb~-AT9C`8w@!>|j7Zfg+BuUEB~?JKEKbJyb=$U;M#MS(!w>442}<O0dZDc3_lO>J9_YQ;px4TOVJTieUTriP0(6kc27ZU?A1GcB^CD>Lg<pAL%)VOnXMvsZ;$v5omVZsh<C+bh<%bCV=sJjBHsfZp_95Y>AUd%diL9Ag~5-l_Mz=iwXp5unFl>31a^V?gv|RU$^ZOz<D?;K<nl1@nqNy)y#P2G>NE4=F_&6^G;4<9nYB9g^fL?sicWK>b8%bw61hJM`tarY~+z^=)K{z(DpjTR?aL7VtdSmscJY^l9j~;t{#OmltX*oV>+`V|8X&+G7`RD{6V!+vC<~}`ENv;FjWTe;Xsoqk|)D40#>eE!9S=n7{Y_jW<T&N#j90;>kGf}ho{|t!*ah^%qUNmJs)bqL16%O0W>>k%PUt_><B3Fyk)$2>ju6U&!&K5x3~56W;j_REK*jr;iFI<sI&Itbd*0s2&gKSB5){OFAQEbtA!<wcUh27#|dP~BZf^R7ynlxAWBO2!u?@Z6xHGeP|QzgtG~2FNyRo3FGd^@x&+nv@*Kj=(J@<P`gt-`&{~)q<Vrm@j1?0c9m{kafRU)?#sL=6TJg*&Py`rEB}IP4XLm7M%4C3i3<<{-5Qc2dP>Mad{6``UgH-Np$7`}`9QsDs=Nnimg;^8QDC8_I3zlrbs0^0bfCUs(dQ8bTXeT#eNf4ERt~yw&8@VPS$zU?TRi_=5CjAU_N;W`XSsWU<={fUsf1+pj&yrP2$36dBtHIq$O-y31*0BGpH~?}UNb`C6y%hRHZMl*2P;=Yc+ums&Rx7S5o5hjM=$Wrn?9g^tVvoEs4476J`}D90k4Cgq-`#57Cr46hgAPyHvokZtoNAIq-3j7JJU!=M3K@{t+M+5=H5;lCR)=FEa!7#wOxh~Un#q=68Z&N>iinq#sp11w5YkrvO^Qe=y9pO8Bl}@KJF8_ImcQ8)DgA#_YpxXTsdTDTc1szFQbtE!usBr;`?dObFOwfd0!!>?O6<D}WNK)Y;RI<O(V$z2ka4UG#;!weT0Ei>su@)X!YAEuI(HiM4<UL)a4lyBfw)5oqi{uJ`&Cy$mRxl)8-~=V@JX>81n5r)0;~qRznf8ZB>s|&KFJk@(1%Y?YC|I8p?8AR7ZNrYZ0-UQbMZX$b=uk}0>*uL55qH`K^L8qT7heq=#$#=ZenmfcMU&_#mo|Y8dt!x*8bf0993~G;?Rw+$R6||S#o&4gJR6OGSyf9*fFC!I51Kx7)haW;#4B1+Q*_SIzRrroVC4?V}f1`^asa8;YiVY%VP+@01v+>Ho+ldb1_LAHtz2=_c5+P(nXC+VG8e^9vUsopv2XF&&H*c@pB_##*#KMRZaf&FFX|tgT^b4fPTgfK@RA#QXX#ipmWCdKw#}@O>z;NTt=W1)<Cuyipc+bv5b>lL#*~us@fqQm%+ozJfub2Nrsli7)CiKvqAQBMeZ|wY}_GbSsI-dm+SG)gPomJBX_oU>pP;EBSN&ezq7s;XqQ`Q`EX6hht^hbbFNo-=4thTQ#;tVs*MbCfAlcQ8Z>MPZDzHW?Q@IN7p3eMlJbTGp+zT;$>XRCuUgFHl8Y%aPCuc{f9EN8c7Bbt2<o5+Br1jwUyF&h!1vGrtH$~#>%L!!fBJs>ViwcPmoTROMKaCpNlD9OCwti^a?Z(1%O_1L)8aRDsBRw|JZR*lY=IDa8o4zQ)`cP8=oDWdiYRk)-sud-HyVDqL=N0Is#R|;mU3KaNry3QkEz5aGbcpwCsl9*=H2PHJ2Q^%bfVJ^=qK~jdFY-ObLGn<3n2cY37s;mN9rk74M?dyi^NVoym(%JRJRE=RP6z%Or-pAHNdDlL<>QG*mWoX-{Uy^;$q3!_XV$Hz_rYDT%xoZ(BynLwuu$SF($=8SuTiP_S#Q|kzXk;ca+4Hit$}gxw5jOFPQkKdI-XC6rOabgsNriUa6YJFqThgya-4Hr%y=mCY1+i8?RoS)&Q%>+b1GzvQ$lgE$NZN9NwJQ%9Sgb*`U^Gw57=!dKwYeaBR|#1*ec*oiEv@Mp48cv#`o0H0g+cKV@B$YMh%YDw31Qny~Ss<2e*2-cD3FE192ad{E1-&F8#N*~a2AFSliD6%qq($p*pF<}Wqyg*FBqzM4iEvICzQ_N@!|qUYaguC1lydD(IT`bav?=$<l<^4?0KB$=Dce8bY=MFuUEMw|Hgodisv#c4D2v;C_`GM+|0-k|LbI@!CE+^)B7<sh=+<=f5X;lbg4eJ|K=>>b2bg=~9N#=5}RzZu_PMxK7sX`nez+r@HkHklHDu0&8NE^MchYNumsC!v9Lk0(($PFOS&`}EQ~n0HHil;3<lgV7@zzU4DXE8T55Hu%R_2Yayjg2*u$0D%u9QC+=$(7fZOcM|u$+YIzCgdg|!>igRV&E1@0>n^tfb105cxj3$}>cKp_H97ynDCKGI;cZ@fdDGGjlYjczQnKMISGvPgZSu#Adn`)8U&2izoi7qc0F$i$3PZ>|${VNO6y@o_DEy~tP(S@H{Ql{8Teb4);M^1T3X#lD7{LBh@j1qm`33_KpmfBeYn4x@JyI1(S@0?7wy_nk3~iIEdAY@7q~F_CR=9i$7zM?61TQ%ZiDUL)5=0D~oSN4kaOg)ysGSUswFwZ<`4-RX2QpgL67{#?xY;rjl6WbRsh`YG6++E(mT~lSUuc^G`96C6mooE|!&c!xpXlM~S~%ZJ*S7P<F(`?rwykg3$8rxy&ET63Ps)IOF`|>CC((1+{bnG@<g+2`HhrX}%mD%Q(LJv58ZVP!SX!f#T1u5^7J|2BYYDcHkr!<YrM9&8jd6kdjX+vDgPraB+qONU6mKa=6G3t?-%AAZu$x0=tSu0#!#Sd4#@s;9?)$`Uw8*??BG&X%%DsQaThTNT(@Qc`OJbOOY9FuC<|m_GICI*w?lD3zI)Zaj+1}A)jD@)Ms-h`fUY3nTDzPVyu-sgeoBDZN>`>%(z>U9v{o1Hh*(*+)m$qMuqwt1R$Pr``2QKqsgClZ3q4aUWssw)W5yv?`kNxS`M__`LmW>os+yYI`dfHDK5|-JOQc1|zm*ntXVqdBuC{0qwV}-j1jr*A0uzBQ?i^0ApMWvqrV}>Kuu#h#N>E5tCn83aY!Y3fX^~}!q>vq{<!ofsA2T1;*s4t-|_WI@NWR8MAv`SA%9YWyPQIt|jV@<}Js9A)e(4o9I?1VCT<>ON29E62RD&3LuqY{sLG6gSj-BR5?o${-^>&wmpqvEG$H;XUDRHZo6gWdhcL30OF^xkgN_d#X@PF26#2yX8;>OZubDKswiPhwAKa9Gcr=2i(kO0VBXXVgk%Wfji9ve=2g7f>GTaE^CijIO1eR$8JAN9??y1NXVS#9Uy*D1wla9KvC$arE++7h!Y>UKfQY7+}?nuF1u6j@z@7$$0o4*GX=7XOk$xk6-zh_x~2>z+iZ!r^!f?cvWQRlFr~>W@a?tbp3(Zl>jeMX+>}wI8+IE*RRmg_4<9cn8Q<7YMfB;dp(Ze*ROti4VL#GU;S3*-)K%XDZ!A(kpvzb#vfwb6F8|prJ0st<R8)4zo1p4SM>by=b{}cq>0b?==Fagkxm7u!zBxw-M>quo7@QK6%j~fWL-d237_R{NtZfj3YT3~SXPe|C61>Zt{I=DMl!)5DSHYOZ?)Gj$MWE?zJG`&_;?esszKsKh(`%py)lZ$tyRu1;|H;G!4;RMVKEEL@GvI6OX?7^^;Rm4`D-~g(z7Rh7i!`_z2e0~7gfqVffr-XM5WR?&!1C;-)X<yN5OHFH|>dpge>lh;VlUuy(?GvYqQ)N%L@#a#!)7i4cd}3!AuevXxWbn<9LDZCM8`k`CBCA6m1;IWyD-UeH&QjQ%k^zKkC*0l?J51;HIyLap4aHa5500kbVAMq<S$F#xE!%_$4cuzgN)XQa{YFO(VdXJkEZPL7xUET;NM{+r5)vgr&ctfBrjt9sc?6PzQ1z59i0uDNt-@I5?f0Mua(HL<w3<diELm$7Jdc9+~{{3<dWIoCzz}-rQ?mJJ@TAsEHUzP|nRu5ilzu*zj*oY&C$|o-=s6B~3L>=gtwmU*^o4V2C9(br@8M&OmgrgY>0Oqntn_VCP|YW<Tc?NvhqMn9(?eR&PAuEfhqj9dhI+a%mGy7KLA!f|G4bl(#?_`A4&OKg2n;iRRMLjwtOtDZ`7id4!)`5EdAhQJ|h+BeD<Mp_I2PFJ<#tstdubp^I0))nyEk;wG#?6o7+nX4iAv@;j5^xX}OUcc0E;^r&<HsWxrrKmBgz%;kwXtF=ThRE`kEGS4$I)V)v1p}!zG)iEE}vz@f`x8D&|*(bcEekTM?sjr)XmQwO~aCm2@`C}n{)a=LjDkwAhbIWQp`a8?0HJ0aN-NtfrDFw$O_O8kbL{@&BLexW^_KekYTmd~XR{9q+S5!Ozf^jDE5tTihsCx{vh9L>;9rGN``39Tpw0EZNASQQvhTMbW`*TI@wVJ8dkoiKyuMv&BK#~*46|aAe$CtE~G9MO+bTCza4*&i^Po=}?HIe739fnd*O4Hlke)H~r{eA%dA_{DhWlX1j;t8vSH=SaDIu->Tw;ASbI#ts3kqt&padVQ?5d11(Nl|rxK{`J@aV;Ir!VytC;%Co7OspSak!Av!+idP`ZDVDc`cBZ?4H|FNHxGktxVDL22?C!{s5J5e1y;D5?uk24a!)I+)dioaefJv`4=$J=a?O0iV<(#BbD%wtFMI9dQGnDm8G~@f6KQsT<;qIJ<24y~;aRTnc;Sin+K~v^l=g(dpkTA(PpLRUKz@$)N#w)2OvCOF>%5Xt2qkMigI!8k<Hu<Z;Mqh`ACDK+vIXzm+lI%jlObLzW*ysNXn`>yu4nvQbUvqB>mfM9uye|dO{a@mFps}AH|1H`Y8-6tZ|@N<)tmL5ZBS=~mqdc7DlT9#SV$o0_c?Txa*-&3R)bqeNxh5-{G1=Bsy!P{bL*e~PJH7@m_k25d*|y9U5~vG(`m7IS^MRhd$F|Vl!gJFd!|iXfdBq78TD(LB;v5%6n7O`1I&Sp0XAkMVWttR5`<>OE7Te`ckPzli>-gA-;CU5rZj>_*;m4%A7k>p`ZZc<N>u@9^ts5n|3v69za-nWLKBe2CEay~RQA50v0oB!l7b4PWVtwn49J9zDE;K6F|eH{ZK@J4nsc1#^2(a$e=Q135n$GUbgKMEnMosJos@F=LO3R#@lSbZ|6bxUkW9$CnfIlJ82_mw!=ga;B+1i4$81@%X~t=5V@hensd@DqGh2v7@-M_rdS6D*Yuzd{E+tp86Z8|b`ie{fLlm=RAxh5-DU6)C^vgUY8byATt<;I*kQ>5t9)&YJ^JQ|S2OITKI3huo&bvG@Y+d9{ej!A#MrG|<qp}{0>c3-~v4KG*j12;TfWFV!I_*B(x-B81V{hAY#fpIX9u33CG*c4y_%nx6HgMHHGv%^W6njbw1XA1zV7$yfD6jF23XAWPIo*AL#b)mAHy`YQN_78zegAEdFuO=!TfZd3GMLss3alQaB{8kFJm!PWaq_W!MTjBqiyYSHalJXa77))peB5Yqc7kDK=r2a2`&y~GQd+HUIQxx51camrltUr+F&?h{GseDUev4TXw+shbzu*~|<U>JNS#xWaQF-P=>*}F({h^HZ6LSWGjb}}_O~3jlDQ%FZ`!dd^MyccHWYa;0^MUlyh@_kiM*Ajfnj?19OXg#_R-yCd0@_l{3}^Y(e^rVh;wZsvi4&1G-3uL~<QZ}@kx7*Rzm-|zq6&kZVcCQPx%Uo|V!>eMt9vqv4Wrt_xqsxcx()|p4-NmQ4cW9TXksus8qCV(<E+BBlFtSpEK^bqybx;y;w4QXm~`F=yYqk;C;S{>-ZEYfd#7D~kZ-j-9(AssFb+h_6vug^mH;}zBMgU|pP>Lhd=J$Fu2~%@EkS+?KA)HiStQ-2EQn^*28h}t{NREsKq|zA0BZyVC*)9B*qF8{rr`_~2WZHrWL5kj?i+1k3A6}7$We_-FPdD4BNwJ@hGv>DpeiY#c=b39fRsv0R&cD1$oUB1*z3-QQ&xPD{4r}GLcfmeXOV(}k%OwaScjXO&Oy}*BMK)Zl=A~y338ur9pyIBN>O;4?h@K#t7z-2f7DM>JDhMB0P1|-=`XVp$mq&6$sj&{|03HVxeL<cUS?Zh=%^&pcxR}VzVVF_lk;Py-o+atmxxo7cnMn0pyzlx6=kEIZS;0AmHToMjFep{{`wl(wrh3-vu6ZFYsxYlX7m*sdOs1Nna_nWSfJC!XqGQFFwzO9YcjSLpOSj_V0L+}XMxONapa{5X?YRhw<HuilV|@k4sqpD9C47wmmJ#g34EiNCNgh6H>86l)E4|UC0EK;lUOg=@N?<Kd!DN)T(3DIxa=9Jy;5*xN+8|fuRVv1t{-x^1C?7+<=<e<H^hSoJtJL{MzD*(Hzo%H*LY&ec&U088i%Gpp*xdHooEWgZsYLB&HW!DJf8S7Xl&hW2sFL>_YpsIn}0G^`r(0*#!SUTfERkeyD3~kgpZ|Ida_P;F^;E5*&S1M<wUf!7%7HJzxrEp96rVOl9|v})52$RYM<u^JP-Wwlk(Q^F4jdaSn7I{8wtLyJe-HK`RQ~NMg`q6Y7YUK_QZ)aOBjCHoszuzMkOP9U{G%NK!ooNC~E~<(=&659$!qS{}crdmw-h}jlvSb`lQSi(t}AK<5=Ov$uueqWPu;G7PzZ_M+^^Dvp>jiNw#Ku5_?q$ZXnO&^1@M@{ZrQ-Q#;Vcgp+{vRqfe}Q#i+FxRmrqd;G94kW=9DzitY6uQ!}=i1Yw6$Kv0G6cW!F^m=j<Vf*+8dg9N@qsi%`_7Gzq`3feZ#Em3-Un)l@5eg%Er4)0cYuF`WkMc@R;p85Y)UnJU(uE;$cBA9r)PFh#xkaI2sIC&I7&spGDdXb=XCR+U1kXq=a)fR{DofMyfkPAsZjN)}J>n#TvRGNm=98TO9iU~$8qqi$chQY9fK`OW=!G~p1$;d9@mS}5FyTkx&)jFz{lLbFba=QUfjD?1-)Zq@1zzCwc+l7he;oCJz_`qUT3!~=UUckZN7wK_9LkD&A^!D7h28D?K(brI0URF>1s5np$5*0b86hhp)!>S1a@-!<8$nMA=$@jX61D`7k0t|r{lYB@UJ}bFaiLG56i*a-Vp+OIljp+XMu_?|!0Nej(LpkY(Z$#03fsdp;cf?ES%Q1_wlz6N%#s)17oVGWMqIVGE%}aHqrWJbj0P@Td!DOy3W4UwTQ7dJRJBX(Ae9Y4o=Orp<yFAI@J@oBi(Y&X69wWkDPDSR*>bQeCHeAGG1b-#$suR;MD3zl+jF&}$*tMa9$k?}sO_WkRBoJ-PjWidI3SCz%&hFO@JwIgLB(BYLbIo9aj<%v3?{1N#o^kRp{M_*yd7T~e<Ge6)D=rax3Odzy1&v#$Km|(WcE<#twvCiSiRg}q`W41Vyta4MO?aNed)~hwWZH(U2q<G=oH`ZS8AyctIlwQpiefOS8*009BM>5cHGoatyFD!sjJmm%XS()@y>j8^tD@_`nzMZK*%wfWF7QplM~uK^lLuBo#W#42k*4I-S$lAW>3C<R=h@k{OpXFFZ`=B-lbs0B*lwPskwL`Q0I8<thiijUvocnF0&6z8-gS%HMt3rQrDLxDB3f*-rVaK^XC7Psqv+3$ks{)q|s3N68m3Kfu*ZoiPk`>-bg5b*!mX?y1+4J94xQgR??mzKXRdheciQUsB(D0$HG|gSJJw%I1zzvP!a-#Ugas<H)>8&=@nskoEtzE`L6_G{fG30zmP$>jmot~bwm0q(X{`cl?CQARVsq3h!~+$vVial_6LZZF%@SJ7u)8@f-fnn<X5j=XnHqlE8&y5k6aA6<bxymM>%GLY8y~4RXSQ8Xt5yLDNi2GPbP6(_8X*8Ae1S#yScpww1&(wXjB?idz~oNQbHTIHHyg-2f>}@zRFL`Z&QU929G8|1R8UswVDE&dg6=sj*eXVE_-d!li^?{bX&G?PAAjTkqF~)(_Ol)RvF|$-0R7ab|<6LlX2uf75&wW@{oy+YO;`;(PItGXO9ByilvbJKWL7u*S{p%S1B<vRQjdNYKb$GKNLouk3}%44ogh-9g3dDXxLGE;4w~L0+R2fl2Oa`$>JievTj_egsZrb_)ZvuTtGSB`SY7D5#?%Q<v}cuglSv(n(g96+Pu(0!(5+K%Agt#!U9Z2VuDkjW*u9qla?*d63&hM##<hU(L+3feD?%SNI18H$rv`v@#L}Zj>3LQr9G)6%H*O1)*Viv#l$|?z(?asy^7HwqQhb1tz@uwd=kekhLx}?YNZgElk1&S7MbtA=g7D>E$Ki8chT1PJ;f$HWB$%{q+d#d^K*yo&3`b`(~wKcf<3BW9Hst?vN>#<eI~PdzkL03hJXARIw@Zay!xb)0UK?F98TanZ@@yDRZ%4ncm*+$<5GGS3X=MlpbU!xO*B>YRijsrtc=tSC`gXm$=H)`qvU*4a2pj@U8)p4xKNERREw<)6DNa)?FsPQ0yH!{nV!zk+k7xY^yO{-i3U(0$HAE&d#9G%+66a)okA_n;rs#t`hb3;##caFgX`@h;9H?RnjW|Pl}fo-Y!!2c3nJ;fFrOXCuWNbpnT&SN0#j6YW<EC|5rXGo+)VN7z@vFAq}^~mxj@=e7luJojm4$N(>q@QoXH+_P&3rsn!CZlgFTQO4r2i&n;~BS-7F1ZrURJCkIqV)>regtX7lhG4?l9R_2F}a_25SERD8*K`JG8;H0-z={BOB;dNSqd!S7|@NLQir*W_R4UVoaoidc!$7wzHSh0JF~wLQJuCcl0pM$od;kMZ@HDT4_2xQ+TtAN9nOu;rs4!8#aI)4Ab(TZk~avz7UI99CpHx?bcJTze6hp(ZFD=hbgt{kK>D=c~Ve^;@a!%v0VEyRvh`RS5nklUe8}{^+Q44UXedDzRgkth^iK(#p(jv4=o?8CRN!Rgy0=Tz@s^X@<&huFc6t+Ruyw_>S*X^0M$$e5F1UPPYw=I&x&;Le9U_N?DMEg8Fz?_19mc$4*q)z?0gjg_f=yMfJzDlIO_`N4-cIedu#VP9i9ikWS(PW@Xde`7p%dUYLCwXObPD8}KCUh`{P2^ynqiS+Oq+xmA@drJTbU{9o!()p4@wS8>%gE`_D4zpG%>FJ$~F$6Mp1{G6h{UnEoYQ@9AO)p{49Z<akz9BOo$wAmfShsZ<V(~r(cXn5#7vP@$OYmLepA=dsxPY5J3o`|R>Tdt)R`;cqck)=p^95vT_`mpvWk#uPvtG&9KSr)L}2T3K+=g8m<dc{t&AmF8oc6O>aJN@cPZ$Z&c_1J=<ov;3D*}>3n6$Aa;C`<V%$EtrI3i;}B{+!ooso<&A^86<eWy!gZD(6!x-l-#O#WtE4g-50nU`~n!1?*Et?3dq*K+O+nx8w!vkQ@7RViSz?o{VaL9;ePAS}Ag*XIh=kIHAN#g}2Cj6Xw;Q3lIyDb|cu&`!vOrkfAxpMZsW7;eo-c%xYMj|063S^|{3w7U$F2OFbG5Bv=Pp(5Py$82z)jQn8UJ`(-?B+QP|b8_ss+KRFwKFw)YY+rWUGrHrFmSE;(MABm#NpAjm^$8s5<<=+zR5e`&>q^*!JBEwR^JjK0;^p`&pb#OL^Uh6rAqftc$xVXTVoK2&wKd)YV!<ATjx7w@y>%HE$TZwh8YOBO*_H9;T?S|_&d#l%PmDX3=*Gp@?m9W&QuG}c~s+Al4)z$8;>(%bXDzTDD7-PH${~OVE9)*-^I|?awZqg^bed59b^)_N@BzyvzM0bd2Dj1m94IPYCjLO+%TS3cd>~1x-^vbQ@1bgJXD@p7s08N66l_X7N2QR_M<e)MtoW>YFn5{S*4^D^CG1Z?SOOc)^qW~2=kuNeXwR<3};mk$?$&<9O$V^IcH|x7w+gq4cvROZ@?=<gHpzvTc>9j|YNFGP2JM~;rf}(Oysi(+Il)frKmV()26qdtx3vPYu{`SEEW^<^g-J%_MMnE^rCXXW@d({Ach|pY1?5>#~i*dJ!j9ER+bqS*cb{-66Ts$;~7gQ>%8FVYdDySq4t5b#mWG}O*eWlG$sog$uLHVCeC{%z=jO=zK4IBko14?N@Q$ws>2}<Z(STlY1p!r~bvk@=LLD1aYd7F}M-fZk|-`R%ErB?4C+n6|7UkMbM1~C<5WR)7g5>f8S$x#VXd5BR~1dBYUAAKW;{&yRP!GqoU-rj!mO@t^KlgnT0zig6%2&J*oknTaW)FFIsm40$7g5eqb8K1vOtFTE{vOaSR@)Coc#<Cgd>kDyTOvOTeo5s!Z5R4?>Cg4oQC)6nmiE6X?VE6FA%?hN^2&a}@io`CLYoP~B8;D(Z8uf#1Oc^_00POG>%OaQXIEH<#<tqy97+En=U|XdG!Vxw<4xRpR7R{X%uTrTvDzu)Mhh8EZi`CrQKA>XQS)2g~9}`y5B%`uc_v(lDi2C>1^W%c`cCj2jfib2P-l?DSQeqGLt_!QR)7U(8k}e=`vwm<`Fq0>o`hoLieSh;FY(vMqm0DDK-CQ=~b@Nv0^@9iZ3&|9WVy$+2`|kGcVS-7g`$j@5T#2983OtU^;%p7FTenlM-*14*x?4!*c8DlpnZ(qWsZ-R-=}EvhQk8@5>^JWtgXp0L0*;}1;ZIY$?8o;S`;EkmI@<>hkg~Ij#fe}eIfsbOjkB6uDx6pN%A)>8vIj~nb#A}yRJ|2%6_(w6fMl@VX%{1Y#$(ucC?G^=S+i9D^BMPZTxQvqRAwR~Ir03_1^4Q^SKXrP9ib6(T5J`|{czqro{V7wASk-M8p;pY$z6X7NOpIten2^q87>NAhv_4FSL?Lbx8=v=)D8qeS6HEYAckR-;Wa$v)`C7r_5rhov$9yu3}|X&dN!MTZ_{*?MP+F8Z5R~6$r<q#_i7NmI|@b^+uk0*ci==8uV&iki_T%wNxTG~0k1iQJ52<4clPT0KYGwO^bp~Fzd?t(^OJ+atzuRk8_FCL_Gte8m_o%QfEWS2TySk9wOq084To@+ofY7$f^zUXtjlP~AhywY`WTjwK*kG&iv{=)#BqgOMajzwt_I7;jQw%G`0#5uxqrp^`O*i<MOZ$?Gon<*Tj|8>XUaCM;t*cFxF)be=ub;FEWsx07I|UY(C`mpa|ff!8R+OLYyM@X_>qL4qtK!xu`MB9Am{<@ym*004vvoyPCIjJ_UU!|6F5QI(L5kSH;D)~30Fvk&>Mz4Dv(ck{7{uL!Q=xF3IeHzWeOP$P--VL9O0P(g+OT%XqcA+W8725OqdT(P_skX!zr-rkyZxPo5ix;;a==xa@w=d;cSD^86KEi060PcMu3-!??SZ@<eSsZ2$WE)-6gYB+n}k_3F!=G!(LLZLBCO+P=i9dzWrF7cg>@lbW;5(ekc`>l6)+&Pv7(Po3haLd;WVFNl<x-tro1SL}!*(S~wG2<OtTPTX-)O#N~yv*<=RnqC3IN<m5KMv0@}EOBqq&XyLFdQ9G?v2Gx67?(D+vq)=L+Tc<vG)E-nx7Kd*gq*SBhVSiqzX1pb4XpjzUv%iOx-AKep&tmzc{Y1-Wcu#2El|&GzqR;n^Y>kBP7O;<E@%wOB4N+dk%4FXc*GZrk8O(N@g-+?%rBy$bH$kEAL;+DkVAP!P;|3CQTZ(U}5f!P-RWY}t6F5^&XQ7TA4=O)u1^W&3`eRWutW$JYkKw~nHFoo|fa#bOhpYYpniEKzxj2YL2kr&fzY4_EJLM1rTrxY`r>F!p+lp+0c#S-)7ongJV2rYTx5jxi@uXW9jLYt27z5n6T3|5Xa77dav~#Ap4^#hOtFa4;>Q3{=`AX_~jclfC3h#!84T|YWdlX*lhxm_V)DJOBEea9aa2}%S>R{9pj085mhPmRPH5q9_W9kG%vbBAH)qg1AG>f<Q+b2WNqA0k<tZhmeYRIZ*?eQQiRK4}&Za_xk$I0ZPDK1AjOoHrz8tZE^C}-n91L+-zkH-2k{>oI3FRbk_u)&4p*VRfaz|w#+6_Gx0g(#BIeu60JN+AjdxLm$lS$B&^!&0@SKWk}9UmCfe`sTT8Jd^ld8x4rc(Z*}(+R@o(N3zW$JxP5Q_QRQAfqJU?q9z1#1)?hqiLtTH9yIR+dk=2!Y#-c1$?opm{l;CiPTQd(4ZgN6(J&A|aEhs$l=G78dJ#`sf}deByjW$LROkmp%l7CjlBxa1Tw9Zy*cBrT82nOlq*y)#L`iLmO)vmUs|w^mu&fw*DfsdB;l1XA!+>&%cAA@Y#Aiyy-ChXTp$@9j9Z;2mJN5h9JKGJLWiSpr5?EgnUy9T2L7{m6^{?>fhvZh30ooGRSK<bmFmCAW(_!46T?RdGBf|}BjoXnCPN_9ofMNg=`fxg%bU|0Zg0v?SkPbnX@0>>SaTvi7aIm>~?Z=y&#QTK)AhK*!MG{kPzmA7KDE}$GeWm~ea?;3RhYX_xIA$`Acq*d?!?%;qKP!toeVM*b5w_p)jokZMMz+2sO+S^NpP81Q$by%7&>30qg9Rz@8zVSIFasJT-08r37@qml0iJoOnoCR*?U9dG@G8ySDaXaAjBOm?pnTkp_?sVjr^zpPTp^0Y-e6#l!}i0`nLG}|UX0?3Km#-wcskieWV8WOIT;Tw-pVm(20~*yE)$W5G?@i4BguhRmmT=A)+J8JaeB3^>I3K3bwbp!eM`3K20uU%MgDR6eo7gUo6edeJENSwF_&qw5R{hrst?Rs*GqAWK;jJ0tBknp)?(7RrZ<IiWBks|R0N%FHbBW5r(Q$*tv#-I>K@aAemAP1h4X}*ne)jskXu0{{Nfo#t8MAf?1<+*5Zn;Dt#Srtl)Rx`w0#BDe5OYPRG5nuN3Im~nJ8&(6u-`#C7iD{7YO3P=HUaZE621!=EBDPz3u(&O}Kb}8_o*CK!oFB|G{4BxHW{A>p`4bi7^t%K)jd)mr1SEAOAY(m1GQ+M+3K)Ol-`Ab$az13x3aY5rm4Ty$7=XAlwHE@|+#U`|t$3GNqt=Ng*4;&FSeUDFt|BL?gvDj}*v{)|dfF4^Lv==Ch5A!r3y)jPpD#FjLPiVG6}OPc~x2I&HkPu1Gav5>tTbbSkrQf4(4-^Y+&Lrlay7OT#h9pg>ZmU5sK!6rd+(oYdnv5qk8RKRmqOq)2P>$&tp{Z+9@P7?_U}V+ksaGvOb-A+rz?>gERBgU@=fb7;UF0gd7y(X9T~+e~7V#;f94=x$@{fp{YJLB}C4nDCIK?9k7AF>OT5PKk_dYQo5t@FHemMTWA~mJgcM3B6qiIYJx`9IMGOYWLX)f)6HcC!6=fqp*x=V~^RU6~0@j7Qc^%3}bi#k}uEFWfjCuh675Zy?nlCi!lg;-YEuxVH7|ir>%@x6hm1sTkyo9`QkLU$#`_ebRGxG!xib5HiGOupzz7VPP_ZihhvHmxt#V~RG==~XAB2o8hvv<IO9tx$4q>p?ZZmd^ovt8u;SZaXSFgCd7V@~Z#uU$9<wF2chnvPd0q1pImw^;>~abyey>NFn(}kCS>r^^A;^y~9DA;2(VlJBMUlIZo+s-UBnDlq;rw@Dx0AeA;9r)Glrh}4xCwJOJR&4otTk$*=uH`8<7jojL0_4LNY`U$Z@*DOv!|XNLZ{6J=#`JK9IPl^PL)R^3tdwNBypS6U`KN{ySHe(==K^$q(<RrdE6lvOs`e!m>l!Sr}kq#G|6w%0fOTIw<(si5Np3698esue!;dU;&;a!rdUWZc!dp`(K<(bcIVbtRr`zKJ!v-Rzegl}MP>fdT;PG?_wz#_RZBDnd2j>8@2h9!7rjxnfw{`SK72T<?4tHD35Lkuq)9#amk7yBtUZ=S0Xd$O=pzQr)c=dJZdMwnD#?xJ03HU>VWm|zJIQ$e2@$;cXc-6`>l;FoMLce0AH~)Jwwn|f@{2S+@t;_7{1@=tL>J6bW7rFl^W3Zy&F>lKo*QYdW7ULK{SEf`X@tJ(xic?S=}hIAx_LP>ZX}ipV<!IO8|L7tuJj)%k#8v`zcidd@-*<Q3(#6()JLgZ;7dYV=4D}rHl2hlUn?^@h{SB}6@sx5AG}$=zxUuEBL>RQQEw2K6b5RZPLb}&QK*KpjIU7S;~QUKEL!%7le>k0ST4|}bROroiO4XN(>g1@K_<QRiboKWjk`FL$WssZ>+oNyRba<%Q6d26#x6vZfyucL3QaMmFzWZ5ZNi5}0f4i9ABJZU^}-p}PWKp{d=9FWN?OnuKzH(}JsTp*cXlUebv+@p?5F1&nDl};1CbB!q<5Eyu7DZOf>f0MlMMw?7Y%r+Y;?0-PxYJAq{LRR0V^CWCGWM07XUkv*a!&Iuw3(545U@d@AY2+;=x3E$N+jkjE5A|LnRyQAs_QW9=%K{hyamKOA9)|AoCn(2(vd()+JRRA<48B5+Js@U<4^tgn~pxpoI?HtJu|5S3)B!XpUk6+Fh*FJQYiy1Drt65ztBEpsC4o<~+YM9i>5b?DeMzWIIJ$rN3I-m`#)53rF$DTFcCO{uqS2#sF=Y^27@eWreA6AE$xh??8|c^=d)D?Sin>9(T_cP3^@n9MW$JPxS>+KVplR3H-5ei)y$Y7^DD;&SOSfJuU^_B=<hU-+*Y%!go!aZ5{{NXxGBUd_mF}?m?Z*<Ygc-w|3Nm3&RolE>_mByS3wXbR6InoXYbBlZg@BKrI)vZ^R;H{^73o0{_1>#+DAHvJ5xP!$DJHuNfdiOH34skXX1=K;E6sC;h&(DB;&>%`MTET*USx*jFh0UA`$XJ=T`Ask@3I&QAR{>>d(Yg|NC)=Y+9?<$Gedt*uHmmX<U}%syr=F433`kW1U33(lIppGFXZW=#koVqJ_|4qI0u`}ppD<8Z%uu-Dim|BDIpvxsv|`$z5|#A<)j<1;O=CYBrDb~h||NBu>7?NT^Km*!}W4s?_|A6^!yQD#J;t>Ss`Xh<c8+}if;R^zQB7fuTFeFVV72n@-T2J-f!;qq`*KAZUwtb?KW?s~2^UUwWYLYZRCd0&7}BYK(Q*}~8wuFk`xB9XGTsN+AsA2`6_%x?$x>kYSNO6Erxgoz-AVTEtUQ-y+cOWx0IKdh$Tt;X-VKsa;iJ!n)xiXNXvFn3vxm+T|R5pWiQFqw-N(>E_8np!pVqXk4~M`1z~hv$gss<4BwFbTt{60U;)wur0QkOQq`L2&r=)r*>{6{&7Ae1pJZO;l4!mPt>bcu-)}OBf*HEHIiM=cN86d3Qx({>N0a5fOzHtOX|9|7TGZ({xay9H(uRS}NGWwuSVRPzq&>4H$PDn4YtD@9p#noG4<O285zdv;upm$&t7kghHFf)xc;qf}%*3^L_naa9z1rO3J$|pjT2^J%6o1BPJ14uuzW`qacSfU%8@9sLr~C$GGYY!ZGHWNlCeT9CjZ@HK#L~A9E%P*(Ji+Y)Dx!qK-WIiILTmkR|k-W02%$7_@v6!n3h&LbDwl?7=CF=dlozaYSYmn=mvtxA>W)b%TylU*5IKcl1b%dgSvwB%@K;Js@DmDTIg1OHf4)YC@Mzq$ROfmaNq7R;5-g=;&miI`irFQK{P6U>VAvpOsMVx!2s^{z-G!0!&Jesd4hSj2boZ$aG6}#whWBT9jO(-E^6oOElgW5sbm{WYhzZr$BkdQF}DcGE~h%5H-=(JTIVjfchhA3xxu{cGdkEN0yW2zE}>&<W+~ij>9LtAqFZHibu7ZEw6gL_$TYWU$IQ%#D6-A!xP1<5@M)1*cnP_>j5ms-R<4rUUO%w`2b`t%h}QE8BOZ5n%(XeoD{jON~tIEj|%Q#V}E}eQ=vHO!?UfvzAnRPI4(>kiw>awv}`*I8Y;M(a3~%i>cX-uem|4YQcw4K-`ID2|36)*+hPVRt6Upz7?FTQ|Dv!);kI(&YMB;q=1XbTGMX3WXL4$++EN+RWfAK7ec9<kV|Z3~;k(qG_=x;w5ah^Do+wpUhapw7!nTF4X;{F5E4(>bd^Ofq6JxPXOT6LeS<C9iFXZ-#qFx|ts_?&_28LRjpP`Z=4zmExx*<v<d;5*8#^z3aUlTvJH-mfigM03IraM7Q%nzhNKXxAgSCT%0_%8B`oWyy#Mj9%jDJ4lQX~TN(N|^zhC~nw8o?}<S9%oJwaE8<YFp^-3QF#%D9^f4cw4vYvb~Qv!*zO)X(X@><Cxodt1H%_v_4~s)<><1KTF_`tOSSXnGjAR$-T4AG>@mg|587zS<&+ntcgsIA;blbzhM$Ryi{sDPNk^_yo!~`SE7hb>Q8tomDNtQa^`r6D-cEDBjt7IAwN?ki@nD4bCQmRG7-ne$Wi^Tk=^jsp-Ec!hM1`YX2@WV;EUMHp5{ndNtP(y%4{+R!F#5Y8>aWA@IGm#!1Eb*wof9|DIfZ54@)z!Gnn?|8=hmnoQTgqnewq|nuqSgmJsEKr%F=YAR(zgGC})eT$V3P9iC10Hj9dQ6+#nKpla*mfSNe0MBz5I@@VG8s;Z2O!XkeP8w|_cD!JmFmT3z^L!D4M696ZQYLvRBtMb2bKck0_a5B3|&*iMx$C)ePr>$q3NT-4xCI?cl;#d1K|&_RGE@OMv#Gm4LZV`T~pqL`8NVP~0h7BkI^NbYJNx(Jk9kO=0sHoZ((xBbQ1g=3KQQ(PR*8DmN*)X3d&J~46SNzAUw@W*T(VOnNEa4wg1swc=QmTf&9WG`O4+t_XF*LSvmA`H2vRj~eMqkedh5gMI>yIo%i!Y6Q)MC5igRz{n<B@1p@AbW#8h$5qJ0k~Jd>o#!ky|5?Y7i+F}RDhP^)hOmRZs0=ycM+to+dN)>D6pQf#PaLEA*01tTy0q*kbX)?(qD=a%#05br}hzus>Iqhp{6ej#oN1evv;e#a%%w;uX=0|6z{Jx!Cp;7+^;BskiPscbP!px&HPe8*nUiz1TX3PKPQuqNMrbrX7>}$KhUsx|IrSzE<tu*ifU4y3S>E1==#_2;%`CZ;WgrC#7i$JL&0i-l8dnbO77M3Zx}Rny?ecPvmf60c0p5bRBl;7Q_a3j(A4WUyX_lcxK^rGI@e2U*E{Q_etUJLw0diOy%$#7?XVMG6g2hCwVEUF8hRki587jCPZK%G9C#yuR6~cA6g-9$sEk5}i9AkkmmmO6j7H!H;K+-Cs^Z{i-(H@`1?6pIfDnoTF$#2A@sf`js8?8vpJgMizPSea%54yD8~fY$8@q>DxT|PndbAdh6_TI?WFY~0)cdaMQCx2&c`&BBh>yqUv=cr79b<SxS!C=#x7(g6f>xjm*%J6zMv>JGVgW?{1^O$e4H67UQu-KA5S;)<x`nVFnECE{I2x94WUD`}@9$<I{U0zg5W<A41l6GG9FqVC(x-zJ3qS;QTq^$aT!LwPf^j38aLdl&@i4-WPsE{Xi^2ofI<xi|G&#)!u908>3-V|>2|!{;a5#Z{KqIJD%{7jpWD(3T1<?i#$(^A=Ma4VAg4l+gM}9wsK~x}n8n9mDZezcg21s-S7~eY}7zQKQKqCX-P`{B_YyHI(Frs!{ba7b_MtV7p2}=(#anDP<fI$Hn&*LpyZ!rukGJWG^^A`6N=ZB@APuxrPmV|heAYEAY8x9@s{`PJ`EoSVyq`-3QXRG-DHe|8rfF3kFIZb}e{rX!=zTSR<uj_B8Q9+&>xO|G@6C0ROAP<4R6c(Uo7X7?kXnqMoN5ZxM4h6a-Y_KpYfMgoZ3c2K4^jy6-<j)8O;dih97r69!8YdR}Sb$zh`zZsU(gdgU*1UD^x_867X~0zCtj$lReu@v2@w^M7(|&*Wq~MmR&R2V+HhXbHyjeaSgQokiKx8-`1e1r!V?^P?^g3mU3`6yxC$qDfOUi7}>m&{axT26@e)`>m#-Sq$Pf!sNL^WBvdH=Ti)9=DSv`1Zha(;|t68Y8Sgr=VaO+qAfL)(4=>2d3JaCm>OXz2k%>@Zmk_6620!tDp!B%JwuG)?4oD?uxXM_#oeOp2Mk6v`yN%aC-w(XE>V6ZK+L(e0hX#=cYFjgr{$ewFA<Y-iuHFD0dlN8a!&^s|2Zpdc=lfDm`L<ga4!2i352Qx0Hj``~bU*8-JWTn6$LzKg+t<eV93s3|g6K%pseFK&n$UxAovfsQ1^k@WL6#q-gc?Sy2ws+8&}0$mU}Q5?%U3X#b;1yG|x))H9G>OBHCljSUd$&hTp_;a;9o|fpva?VRE9PBQpErgjBdC@^crp-B)w~lRg0a^_PSQ^Iuel;1n?TYN{#B(dykN9p5p2El-3if|1Kx4ypedq9P_&ZLg*lE1k*num!bKswN*$&^{YrusEyW4Lz_7Aq<HN29X169G9v3mCJ)_0|aL&F*a3YSQvx8IHj;-0sPrf?;3qVK?e8hOBu=?QwHQp<o7z6c{-hM88^SFc}~nM!V>!(C@!cYO`c;#``+<g}E6dQKjz=BfJ~_(CTI?kvBbahntKX2)%&>58$N(@bSKEaz0oIaFimBI4^YK15T&5Z3o<Szd`ce6B)46<Bu!j_QskV|3&@)pf*MD{eR|t2Z9TDcZv^Xl~_9#}uX_L=J|0>G9q}td$163^BgVnNQ-bCEvNG!MP&9pqJGHCAuAl&W-Xqki}?3<vIlXD(2xkyOdTtpPeGg8shk!_RgF*vm;~Zsr{<2DlJ~l!r5WylFce)1#%-2>7c1M*>lkftqp5W6n7CU?|<*#lnECO+&&5&-S2_kCQ0Cey7wDDda%9U*s7sBjS~uRecLla7?_Sihb+dR$fo0iB*z^Wzmqd&%jhg4>+{eQ=I2iH;Gl*pHW-3r;6J5)&OL2UggyVMwa0Te6Oa4RdJqGq^L~vnHj@H|1&K~>I8B7$3GiHfVA^f_oY+wA`m=VI9R$^K#dAAwjP<Y>0IfoBN+^8fxHFGP;YCUHkyHoze)17}tl4Dd1cV@fXcbE3x8V^d)O6ztD&klm+)%C=!CKdzU0Whp&_4>gEua4>_klZ&+XXiWXm9P_4FcDQ-b#MwFdV^Y`>!+lWZy;^bov4wPK#3h-JfA5)Nt&<Ne%KqG5t1IXr?z^aTvRal%izK>$<}+Ar7ZagXHCi$`8)TDI7q^={upLoKTqZ9C2U+JTie@Jdd0hovfT9o1kw&K*`M@X?YLBaB4-}r1lmUmY0>PFSv%S<bsgR)L!0x`C+8X+w*mMz+`3tB~kAvPQ!Q;F^^FaN3bm=0B62vB!ehRsDMotfboa;N8_#f<{|U<2lunvvm{lhwuFX|H>wP!&8XSFRD~su?uzCrd*e}9PzcN1LMoPlWc)xC*v)DdmLydsVS#(MmRSQi*^ID?!?ZQX5DiO0#>6>pF*eAh9U|9=P_H3_xt1_Wie0-wJGQx{s!R>X5s)5vrAkcZnF7U<o&fO?7V6XMYUPykRZawUeT%sxCqTRfq}plK&d;+%8RZ69a**!s#6&?-frbu+?_HzFEj-oOUBI~8>`aNrYOC*j;gK9pCQd(mtYg$@JSB4QJfATtg-AvpwFml8`o&~b=*!H8Iu1<Q8|vOTp{t0_)24}{?wcB=MBt4<7len=h#;yL6cU_pp{mpQq$H}wIc<m0g{bbL+%c$O-~r_HXOj~G<SwyYxaTw~D@HkqdhA713Y9d{d`uY;S|M1A&zcJ-ek976Q58-Xm8-S`dxU1FLvk;im}6o(!b9olgXo;X)|@+=^<7kaJM#SH<JEAM#-1LGh@^v9Ee?F~tZg2<xDsJyCE<92IY?OJpbt@U&;Pm~|1F!EtCf|ti<_F^`?4&s<b~*&b-Q}r1$<;_=96;Vj03aGO48cOekB;Z;huok@S3HAr5!(bSkhH*X=l&zaB$3*)BWbylT1gB|EFZRfqA`%k0f2j%djXDAs0Q9UQ>FpAxr3cC*Tm?0phFb1sw6AJh_622mW(}|0yTeQosz-JvH63o<{OAJWE&NNlrbIfleqTLU=f(j^Wr0WdX#&pSf3KkVWcZ*h}pS!SML&7D}*cj2bjvoDUw~tqGa-Jf}bL*EgvRkSLxdpvw*~Bz@DnR`HwenDV!~A+H+3HSpS4td<oq(7OTJC;n-c&2Y2(f*psHOQ>){B@}nG4^m^W2;{o;+}00O*!6<P$7eoz;wCeiLeK&!2bvr9K#}AO3qKvB&o@Ct@>3D|c6pk+)DiMBBxdN+=ZB9tkpjPT9>&)&qqmhEksW9u*+!teI|IoB9i>?8B)}S<z4plz;hSNykpeFtz-^7qh8}n<9`peJ&j`;>=_q-@zj10jh;TY-88M()sqss(OH&bJN>~!i>Y|#&N>E1!=x$x4v&*m1BcydrA-RDr8rrV(49XDat?SmzGbSqndSOjo{Zm9aUNLITdu7ye%^iN69K9k%Tqj;7k#`UxOT+B^4Ux$QlA#chHTifr8i8;h>o>Sd3_<gs|IT5y&zajeJxi)t*(*o~Iq;XFrl&BET|#HdA&l)YMMh8%pezxgf6O%!I<%RhPIYYwPt^tbQcnM|qLl7cI6~;0*}yvE$m-BqM&ce<HP0DIY7iMdP##4$q)JGliXJX?iS#gBP21{H<T)ZcINKXGiV~bTRywTo$qsH!A*4P%b23DH+_1-inh0|t0EEZA*!QHQ$ZJ@hFp8^hsM6TJU$Mcz8f=V<7P9YL=P}7NgA{Md4QXF8j+DYiUy8ux&+V`Rgk@kveu@CnYw*u+86bcfc>PxbMm-%CdZ{4swHv*))wTA@f*^7A*rFiuzby;p-1&e1h-mSjy!wI>8C1CTXBgq|K9vmrG%n`-H!vX|h-mred<(;~DTdn`c}-E|*+2}`=m!E0A1>SM2(SKc0RZu_HS(4sboC%$8cQa^M215A>VK`Ql$}@q4=TgXl`EJCaOFx|J@jQp#n25qa)S6;dPMs^QJqw)+Czct7}Wktny4=&<OG&sS2zBc71#bU{-)=o#eeb*2fE*=bSt+ymGy5q(EZlBCD7gM+YEHS(YtxGdTT8#-MHTFm)5%78>L&<Z`~+$s%tCVezkwSd-LLf?(4W02wdbV%2W7KuX|6kELQR*MIL7E25I%O)0sWW9QgVIxDsubJTnYoi2tEiyp6-(Uw`D;D2|F#eZUAPFXPmg&k=@$I%af_7yQhZ0`ml$SCuEIC>0T_OoYL&`z2MA5PT&*Rah3gU||1xJdH`dT&-MpsJzw(<wWr;<j$$>D9}i9=9jB(Y4iK)MNT+B;wan=>idm4sD<^z2M3v8E~3`*T~xH(*b)v>pkW2qEqd#i!-YU{xpqH#DtdS1e|zSz&1M}|VzW`Onz%eS2zKk}gmo;IF7w^&2yv~I{8;gf_96iizo<#^TyCqe84LxOj3&*87Dau_q<@fRr1>xr$9;h?aA6I+d?2{ZG?-%+yjZ07*S9&xVP#9$9<4#yMrheGChT-c%|NfDxhTXhQFSxVip?8#)_&|8Z^au~+Dg^-PlpYOn<^M4ZnTVO@L4^U(^M!8pYbz_zLfbhv$^<DG>K~0C52@)oM9bJb5p5`V>K9dP_cfc8ZCOPvyILaNWA4iIZw$b8)dSDX$jgVo#EhgavI47WU!9bTgld_CH5xyNG%zoad-^cI>=WTXVDRm9v-li?&uWD0<d05wGt)c@=%YXAPueRIU5bj)&-KCr;N%tzdyssN%OI^PI%;!{T9b!i?vAcRHM=XRi<yX4EPxgd9f7^IqCG;7-d*XfFr#VAwJXT$uz<=vhsW~8X^*s7mlf7bUW$}hq8i324>RAS}P;C#=eE*?{Ctk%`L+#$xaqq>(!pITI=JxnbvylFxRTChPRYiJ-tbfT$%4FohqUg{03QxUE3AV(-LjUISNKVH8OzCXornX`GI{arZp5vSq@S<yc27($Ruj+@NDOTc0Uo~qZxPci`@<L*_7i!d)~Ag;dCwbJQ$5z{14jwvReK{?JV$2h^5R6VG_q3yDJ2TWy`3|Z;`#?(hH%rw}dZZ39TbmIamQ?rH^%nj|Ix&UtZEWVv&emf0V6en6Ov2z^D~P$86VM7$0S0RGC3iE0+zDy0Cp+u>p<fFXbYTR`#Y02gxwEcg{=;Cvu25W`2XygQdm2J2?gsBl40_Sa1^0C-e3w54kv7xUtQHiTrV?84zB)gcM`=+KNR_gH01rKgq{rS+|RYnlJubLp#2=eQ?<1f}#rbSS+C?4D*$gjeqfB5=Z7KEC1*$j?d94poA1q(yjiKrHjGv!iqtWWb9(Hny$scLuU>9+>~uC8Bv}dYioh^nxIjnRV9u`s=1!+yXBts+@I<23Q2Q}a3x_<oe|ZKo~T&*O`epR)Qb#z?hLxgFtAIRw{LcD-nbcd))ttz)nkjy+rKW0!Z6Hx?J&5=vKN@9vk`02_Vt`?wO?VuR^abnlCAl90t`hSBXD>;r^jB>OP@IhKiUz)`_-%8AT;tu0%w9>`-FP^iVMi|G4mnAr?9(0BqS0p%5zmIUcnY~+4-0Dg7DuoF3QcYck6oiZR4V>UB6*57n^;X=Hk_xx5Dmf|9a_Wc;jYiExff_>QuTnOZ}U@{(3j;cf0Goi{qjQ7s?IHn0Z*j>Y8-s0NHKuopm%Ah3GVgT?oua!O|&FijMY+L!QL69g~-p#$!UW(<-#ImWx~xgMv$2=lKR)zPn%FdQd-X#8=I*#7^&I!e-1Q4hCw#z)<iw4xh~70zDeX@T>A|7C?-wq&=g@@-X5`9I0rlli}Ad18d8cfrf1ba^bQlk(-rVQ^@VDSi^>cxD2QGneN@kt}ajqIMHifi_4b0%~6GMcJvYkEowYfx=a`i;IOk!uU0mM8_=4}2AigLWz`S8_qaKF8HAF<2m6g>YIFpsQez|EM1fkR*IQD-EHzn-FZnjg)5@jis)qEbW@STjE;9-m>NO9Sq}JuilqK34wUxM=pom26S-2}|&sP53=Yn4jT*KZFMwmxfA<aF(__d+!F+K78gc+81qR%D#(zXly(rEyGn#<-0FwhLVSWEFdk1xgxEuel9B}+}umT@bivs5!FI3x2`y5FQLDBl63RHS6#at)hE+T*eUwUt-q#QC3^+q)^}4AwN{4-;#p56>4Lkh+nnH4-3zv7mmsHnRO~b`$A!+XWOsNLGLO?a&^&2GmrDC(H?Y!-j4jk1$0Fnk&c-C?JwXY$#lMeOOKIV5|=ToobOitou`;(7dZ4QQRc7if3N2fYd0_rHG&g#jE)`z8&jEeDI~fZbV5<ImHorB2?93w3f*7YL3j3_8i(7z%fDF9Cyz+7L3Xwv!R3_nM2I!w&5)YTvFMd*OA!Ti5-V7Q*+lAk<elFC_y^0p)>e`9kNGbc|Y*~6q7g@`aMoL$Zao1;31iBVdEMd4nu){zg!1eTZ_9PK$wfPp2e#ccO^@0FVS1x<cqmSGnAsYZ)D=_^B!i_(@Hd-B)f5f+M-2Y48jB>42?!Pj6ccolUX&kSC51!R9b0WS-UEKwXUpOWxulI4w+aKaz|!YIZu)*)jBs8=>kR1v&x+t!Jt%@Ic`x^X68XI6UMH2`uM}pQC<jSY53&!e#2j{6u%$ew|(}d3fRX@YP<0be?{-i+&phA(Uygf7mO}`H$l0+U$vD7k<xRPPunuj%9ol~e?#+AgR(xIP5L33CflR<Kt@GPHD+h;hMyV!gA)84CNC;vpP)0B!^vL%oNQ<xm>9`FLmLTP{;Qt}sNyfk>GOx5k;(1P*=mBJbU$Nbi-`68fCB|zf9QH$to6V|PPm5bi5a(ZoiDceGxuQjG<jcZe`Zfq#?H;(znm;K*1uzjd#=*j2&6#4#(j7wA=H7QgeSc1HeS!pQi)Ae+q+w+yf^oQgU0>(?&0>P@9yn4@38O#55(|h0;bMI8t+UdBhh8nGz;y~6=3bDvh!5QC}TcIG`7n+SC&i6#Od+>G*2u<DOts{B}69I1;s$#`&0ZGCuhoo!BflQ(My>J1vy{%lS~vgZGp`Ie?9_pKmpYtQkceR<D;v%b<7I{%s=xs#^Ax`OhG@MlQ=?wpn0b6miKt(@1W684W=xO*X8l)iNG>QgCZNHImNL{MiO!vrN<aYCAo6CM8(Q@0pdGRc+$o)_wX{>#G^v7JU|s<0nx)of3RDBv%bAk$IPT;L9V4BN__fnf(uC_kWm^q#Z-#<O=epa4I^Q;K~d!u5`qagY80M=Rg~gTv6Zqksj9@wm9Ebchv7Rj&p66(%orU;&GG2|ZN?^7AI?macSv)Sd1ATZLni#)y%_02mdhp$vOU*@Z>Qu`;r|KX3&uI=p3X2tr6e+&`ZGA#XQy3kDg$B;!=h(%2S3BGUi<(r==u@XR4)|iEAbP}hy5Tf#-%;IxXTL^cusYK))5R@%jqgD2D5ddJ2^Rl#f~N6Ws!9_9l5fsopmFaR(*K>>QN4fWu10mbUFf6;wUexchv4+CaoeD1R5(~Ra*~wEqVp4dkcSpP?{5Df{u27Z#bnAK?QgpLBd?`V2GmP_Q~`V4WAV86%NjPmrYJ?abXGgf*S!X!@L~K5rJ{mJHVeaSTj%6h1$n|3hmMKxb3f0%9UcVRm>l0xj&pE?HsAEZk6W~tTGS#ltH~PwOG_rO5VzA{A}t|2O>ZS$vmf%yq5aHUdDy8+-rSBe@}VI&dcviVE=Y<>V)$H6v4^Gd9XM(j!*dx;4D<bOIL1I=ioWehlf#y=nwT*^TrJy)&sdibdA3dMki0UrL8=k;wJ2lCJ~+()W(q*ackMx^n%=~SE3XYiaXFs8MT%cOSt`oZ|yfWoBLbK)h4Hk<O!iPu1zlXTU=5+w{GCHJwL`JDs!y}G)x8@oa<7sE_+wu17dFwOo_3m@;WT6TRU1%C0kk-wR(zyt;ixLde1^$DYu5aF0W#FFf~Qgr&^?IEl^d&btO`*?t;>ax;L_x<nLM|3HfktmaJU1H#iX|vlEfqe>$J^`-^+dZBQ<cS&sAa?OCgP=6&y?Lo>g-MN}V0)7IRrN>!7U?{DAT7iG4n{zz^&LKvgZNOI+3KH*`p78$QO8AlAjPza`;+Pd-xJe$T9_!9NlS+hjZ%U#r&H8@w3;of#PIQJunM^&J|b5E^gslRw87O_3}knPQHkaaC^iNtkVv*%D_e|EKPK_4Q7{1GUPXiTYr-qm>P$nApij=CCfx|Vn43QSrqX$bKso|#AO8HgqGm`StMMrV1#QmL~>i|tS6X>o<v+!3YaOfkGf%%Z$pya0AcQg=^=f;K8aqbj3Zd6_Ue^v%Zp_ML6Gv{}Eu_aHyuRRY?mX%XSCYUb|O8?g4i9R5+IC2B_UooaliD(@@>7;+;W&}@=j{PFx{_?gQggy3#)IQ_wKuP#vZuevhajH;eCe|UJmDZ*G;Mev-hW`ljDgw${*iXI^ZUAKK2VLB1*@MX|KmRM9HzW1<kf3LZ(NB1U{%UCQMkcg$n%#y$Y@AMuOE|xtef`;uTC427fZZjaxL@?PX!0FYGZ3<dexNo}8{CM9Ke&Tu!E3K6#{SpHPIwI(XO0e}EXk;`wLzE1M!4St3)EC7dGBGAFStIlgWJN6CwZl$`?~i~+W|Jr?i3G3qh&{S()ZD@@NK4o$LI{s}1T$wkiRN(X^iFw9u%{3#q#ItMlpBZGVw%NAD%buqQ7Z5yewUX{2dA|6PaVA?kO+Ex^*d<hFJ;bpcAR4{K$+LbxW)iSjII8*3-DYjPAC}Uf+A)W?_%<?d&VzNE-X<TE2A!!y@x?a@TdAKe960TJ+Nf^*_rM!ZQ5}k)tj6!BOccis2%f8b_>%GWCVDUH)Yt<+mP3SX?MO%Q65$A<J&Ed#m<AJzTkG_@W+kDuEUi>s+CG^OPRCzhF^(25VZ9y+0Fwj-R&Q_8qpfq>+pjq$BN>fSaRK36%1nIvkyC|tbX<i+^cSSE@L%dV_XA%DPhI7yqUDX1_PQAdR_afSXeUxuTv4$qD~OnOCAm3<3@qv>$;LW7{Kz!dClfC0S<eLQZU6#YBRCODh5KRR56C3EPB8ngoTpS85F4U+;E_-R94nX)s@mJO5ejMz;Y<VIc7VfUd3e$(s|VI1#kF3TpZfXeZ1k{6h8-LeILb>_eRn{O$fr#!x1e1xXd{@Y+9lsDxAG?7jq&Hj!*mjQFwsqPYFI7P5sm(g)wJJ0beEwCWwabg?@R(o527lC*+0-lfZyU2Hy69jz12cz$8Y~GPQ>v?opCvlq8X2ivOeg1D+x0YS5Np=^Jn()NXX_@#@0-g~uGWWb{-XM!jK#^$F-^XxvPKC17B%xyx-^ZzJe-{Uh5Z<^7w4>wk+r4lSkK@l|*?{<^p5>(C%)-m%2jKSY}{JYw^2y@hQ*Z29sJRbAr2?hkjHKi)MfOHjA+`Ec>`$)zG=z!A*C;EV=u(r?0^n0y)YWBl>7j9C+$(l<cGh92g{Gg?D1%tEnf9fMRk3zDUFM!-u%5DdQPdfZ>pI-l??rz}WdO@*cDxv;LC2?&AKoKXC}OPT4j?eG}|ooJB09}g!hTm_S1pFXUKSwt#;Mdgk$Xq+mHGyROuGn%Lq^(e}`O~jEK^rDmHIM1z7XJV`Cq=rMr-`Ex3zHPmo^}&;#zuBJO##z}9NfMvT3KTm^1}eXLv*Pu_DG0%99&t=MF_4j!E7v_Zayt_h*OC)lBRHT98I^@u&Kv{I$qLZp*^tgAv?xdUAu`H^QuoqypLC=E1t@E@4_qLt(LvI6ay#qE_H6Da+!;(T=*>7muRR{41NMXBwQkpNcre2{zfe9c8tA6}CZ&iR!rVq^28Im)dk8lxe6a`Z;zsGZUUMAYo6H-8h%qZ1!%ASipz*`;<nh>dhl4R_Z*I}U$!KNgfXgk!o4gjFhOYa%Rx4#Mnv8%Te0LPRJIk_6a+nCdy>s{W{g~v&cx`w>ArXG{M#aO}a7lO=XaU6c2-d8+k}r>pgLe>Hqbf)f<Uzkwy~qii=|sdiN8cm-L;k*S0KdpvzmPKDLS*1Yw26mwlInnUT!8MDp(^YHj*aeUI7JcLt9T$#6|K+dM*}TCe8be5P;xTO@DHww($)bcI8ILI!Bc7|$6Kg-3Go{}XguH97r;aq4!)D=*E|K-B6d;6EnZ_L9RIC2@oaviI72IgC-d&m`cFS%aZ#%Eq%M+fN^Y$vR+`q(Qhq>pN1jY|f8<H_^9vl2>$)|c>?V!MTClm_JUD=zw@Ikw8M|2#Hqo6h&?iq0>5vrT&GtybCuH?w^Dz#rPR4=7%b0X@cE<@9CvL)&&~?I6-;|s`EVt&XvkAdjs#VNKIBAeD13_wjQb0|P?N~WNFzx2EDbX(N^qQ3g&<Kx^_4NYxD!m58eKxf{e3C~7=5iWd87d`pud%ZgG#?zA)DfVbm`NgrKV%>rB0P=C@sUte4F5;Mg*7-M-_?0Gh>&1uWCayQP|a<IbtlA(E%JseX!9h*BAzK<%iv_8n<Z@?NV_9Mg`j)Qoh_X3#{IpW=G*rrwhH-L_8W)ro3^L!IiWc=d@Hn5Qp<z70-l;UuRz`>ntKHs!wAf9emv<3d~Bk><hf${Ro2sfm956X=Kl5`c~!7lut7NWoh;9abf?ruBiIV3Gc+AvYgE>qUKn*}!zt3k240LSBsH4{x-Fe5M)f8k!wR9Yau!>?uSvWjWfWHql!58XkbZ~6dS$vXw2gwxR=;!D*w<^h)q}g}xz2V)tmnES62rKlC*pcdFKac%OuR1DE~2_ph*d}H?hukVhN_9#jo`}YY(u4wIUQJ^hg587?1WFy28G#VAo;cDi5f`baBfC%>^2U6+}!_Rq72g57Ivm!yQrC>f(n2iWeP~hHXh+)l&j%Kl%QBJ-jlD{crH-+BaqD}kCExVhsU*S>Zn$#w!HXfwbrtiDDg#SM@FX@IEsxITk$a>Q2C}Xry%=2v;xMdn>Ap_!wEvbi3>Kja4^p99X{%7bk2bT3wlSx6PB5o$_(@|rk|TV#yRH`U_K({+$u31Pe71=<RfBidv?+ugP6k|l+Bs&<mJQ|<NR^HUwp$~<wO7!#>VZj$Nv;FYI=`T-hLx+h-_@Mq=|?Gx`DIpxgz9K>?3gsiqT}9m~!}mRxt&cv@0wgC{%W$jJYUf&4n~BCFg`EM9jyc5z&9-a>()ELQ%8O$S~@IpoZ9}ba9Ec$CDERH=S}|DxGYb&D}fOm;~G0e_Qo3eU$D5uPxDsWLyC;E+FOyyZeoU=FXdqEfii>gXVtFJiG_28pD@V{bGa0lZ^U$fDAN<4-iuf(8$Nmq%VcD6!zA_TkU>j0nU<oY!S}VZxjX&LZ^R;*h2&q{+u$aUc7pVFiD7Z_Y!f8UOjvLbFG*H-NYxqdi@JR-NuZnPxB#}KFNb~`b5XsLuVH{j-Ode3j8Jy<z3<{=@3$!H6}2^;X(FE3+z7zZu$b+`-1{OeqkA>%o9U|@v`&kw>g01p9vgytRjHOO@!?9`u*3xQq=+w=1Cw&Ij=r;2n6aw2S;Fnpg%*DLV+cPh){$D1mE=fZwQ1L2C3CM_$8veGmyLJhQ9PALCUKt82E8t2_Sr&_&>@oHinFitC9-T7k=F5I9<Por=Ic43|yt!rV0<wC^hdTO`Jf;_Z)QflcWD{UjH1^_t2T?*z@r4(dRhb4Cntj&9Rza6;FVG_<}C~;#)w&s`l5yUiW7I+eO1#S*cjiu*|+qG_330Ze?}tR;P5UztSnK^=|b`x2mh{Qs?^m`da7Kt@X8b<#NRnm>1M6oPg->O5QHQC$4y^1V$2DllNk8*dB~0h=go}vD^*E2!jyE2q4(!lQZ@KVi?Z`Vfl4Ir9T;VOCksgr<OsO;iigO5fM?Chk_+KdzY(QaHHld`dG<czGT8qY*^ep`K<6on%8}rTecvMLlRn)H3DiI3*Bws*~Z+l0AXEaB3Z(ubG{gtOT~7IFt|<7;I_9w{Wx$@Iv$KB9m;A`Nl2N`r?9*pfu7@MK5mEU!gs7dU3f!4b#2I}QV3Um)b4X+vwR`(h^;%HA0=<>|Hs_BHn)+S>4JZyY{$BQRuKX~fEUQ<6A%eW7?T7Ikh0qxZk+;9pfDl<8UQ7-)tu;dPy4d=jE{Y0b|+@N_RNX1A2wn)LVb~2-B!yV_76yZVSdTJm&~lpsw@CxtLLmIrUe33dCxcBd@s*KpCYsr4TarAOp(CqCUA-^A6z_TVyQ$7meyZWaM)#WC*Kf68Wx>4_9yz-pDZx;C-cTWGTF|BSfeV@m@7fn<QLY#;o7ecwPudhoGRV`_8OFhYPu*?+^KL~n5bTsGBpUV<c$l#zTA|$C|O-N0tD7Her?INI0;TO0to(`?mQOlJBALQaSb^ptXuE7H_lVh;MWel!b<jkDp#xd^cKfbSff^(zs&ISTgqZMd|O%g%j~iQ9*2At1<2n<1*ri(%u&Qh=OUBnL=u?a<8>?JlkCX+5*7m^%&Qr-LWHTrRhB_c>9uSx42N-TcT@Qy+f_2Q-4fNpcR31<#|N+?Q;_LTXAR5v%+a(1oJ3uARk%I3^J{{d#36!L;qnU|!s{w3C?X$xa7-Qm)xLT5Uwp8G_S&dNjBKL*)Vp+L$P%rb*fhMIfC<r{@v%i|u=s>vAW@z1L5T~I1c87DA{>bHr&BN?@QsJ&zDe85;+Umszniv6#Y0rvVS2*1s`cR;lH9UXw!(ACN(ekOB!G-T!KdY-c&K`Adp12QT-K;Z;E8hMsU_D*7RRzwT*ebj_^lkcMoPP7j27XA#f0IxMR^Gsa(J2EP8hG*Nz5X1$AcAgOexHI?{b4_crwY%0?DT{`Trj&%**prn7OeW=pa|g0d?Jy$1ER+VMAUe7a5A6$T?YY<ml$*m{RwgBI6m(kz6)hbI6p#vwOpZKVnMA?7>$;<%TO5cSv}osV9Z};SE;?O3Y}z;JfJO!t=poikBu@gJqGYi|bGG?pI*+C!y0i8%_r&VI6_^GJH2JuL`<hB--dB{VRgE4<@7zUp1|p=f(_4fmyL+b&D~Xy>NApvFRbki>b^faxZCtEt(t-kJ|FXv4q*fjeb8PMO-XNFDlX&hcSD3&JqY5LYF>oGAJRQpd%*qWH6y50e8`SbhUU6)nWS$WTht|lO*=ioHX=kEwp#x#mH=-t%*BLXip7Yz|oo%VmT$6WRcW&H8AP1CNx%wxDZ<VUpx6-9m2`z6tTLlmIsgPwm&Nu-aO&IenPw|FOKn@o*KW%E7wet|Jp^TrptfrlG76BzjncIW(zKwNYBNQbl#~q_uGE0-EM9)YO_MW;z{<h%d<6x%LP;hflii8zo&hmrQj<CP73x#G)?tCYm{@8%fEokTC^|M%kI}pn5Wo<iA;a{bf}?vlez!bU7ZPmf5j7H&IYoEA|~DRsaus1=rIAGM|j@Sh5LpL6+rdQL;=vw2qQpF9T0QZvMYmiO)P=LabREx%<f9MwS%+$Q_uyt^NWKEEaJ&9Ctj0BI<c10C})h8hQ`ML5K6Vxv|5)8Y!#{O&nk|sN^0#qOfDQX;Mw#^pHYE*Ik=34z-@%l)`Io;r$Y3T0z;TpjbOv7L{Jn-Trtp~efG79U`L^>EJ#6zFbjltC}tI*D-9+(J#y{}d~_E01}QJ}g@Tx#v`o;TvCvx4BM-Obutcms?C(2D_$mb#4!I#!U|}#>8ugv-aWFU`GYtGjA>ISf|L-?%ws)HX%K^4*Dq=(0PSM%X8pFw1zds<Ru&Av%cqV5*lhH%$XA1OlZHNtv)VMQZ2m`^&7S$C&o;m^Dahz~CIy-t00no=}RXP`2T=dMMW3#t-aaOifg4;L_F>W+CW?HtiQPA1h)Liqz7>6eaUhD**TAT)i<$&yL?zMw`me;$>g|))+oM0dI*z91R`yAu*S!&?SGxm5RVDbwJ!qC2q@EV?1oaUzIP)x>G6bJGd`n6yGN(7I*{w=)y)Jx~Hdd9=`W<xdBSGvpVtKF4vI8>utl;5c|>7M@>4RIV^!9PN6<#MTjg{-cul=3V2PBB;sR#(IIa-p!imJemo&bJHeon9|rE_7GI<x**RxwBeaE|fckVyCpWoX-c*ljUG#H4I9th2C1Jv|3tT3&LWt)6K7kz1427$Pphrr14-9b9BsLl5(+E=&h{vm%?tLvs5artuJ-D#m>@dx7^A1O8w=aSGaO8$!e~Y%P+Rvw3S6s6omMp$onMM{BsIC!$2a$7oqS9NtpNCVXP5C8u>ZJ5{cig82PbUaWH6!LaQk745MFUO|O4V%br51elFS%?RiEB<DZD`#6y){Jx|*Q947S_BKZ4^`u`_<H9h=8kCrLt3OCES<(m}HZ3)E^LzFTu&=WJ3i{e-Q@ang({;ya6{ObR7V5E^~P><<)9*=MN<n=G$=Z`4Dthu>_{h}zWkBq3ESAU=&v=8aOKZTb*eEkove@QWd7zoFqF^-_J!%(!(WQdp$M4wQhzvO@#jFE$Z|B%DNo{I;GY*9TRNfRmXhNEwO2|qD;BlY10RR#@+!g_vXgsiXnqh7VS=_C3`>)^oVg%HLPj_HNJdzceB6q=L#awJG>BPwtabzlLOph~TA6L8k?4H{I3ZiXLBXq+AswN%~i2*E$xMA21IL(2Oy;^gouElVrQN&`p6Z+cz>>$vXN9%ggF@FL-wv07eNVR?#hHr#3=))5w!VuAh=7xn9(i`dHl_kW{E&@TzZ=qdaqKE=m0mRn#S3QL;7{Z${<7Bc$OS{RdMc&M0I@Pb`Yt5KFgJa+1IJa0nL>)E|nN_czy?t-r-UPSWwD!eXZP)^4O2iM&QBFiAD^hS9Hv6nFQA$&AIKotJsGAjaL7FaH`Lt!k*2WT%w*Oj7S#?E8n?+^-7Uv}xM(XsyAIPSlS&v#@Y&`c08+}bdOpiNAC*<ZNTK^2Lg1>!dR3T?9B+H(*1VZ`L6KfP!a6{m#!?Iu=3PAdF&W=Op&Dz_Cvc{5N5{yV!6ytL~14W!&ru7qebNB#5Q=gjr6#9x|gB1=gTMw)|Y)|{41&Q9cbrnSP>2u2plrWB~!HnT*t2k9lkG6~1<@Kn-3(&;3vBh5R0^LD%5!fb~6=G}Tby87-l_uvY1XRG-`*EW$xT7-Xw2@41g&~qz-Qv=ol#Sq|pG6p+*qP6mNAy+0Bru^c2xs}yyHYH;MWlaqJ+@FNQ30wmoQS6Lysf+KLg|pnsxnf1p>P9-2;%ydR#Q^9~LlJ-eDT=p${gI*I!Jni;d`1eyQxQ{-ICaQUpCP<D${~18%GNW7qxofoFTVPzP-dTIO*QF{2=9e&Qo$50&kEP)*kT61^Bi?>2{=5`e1A@T`*S=3nc1PgaQ^-Mf1DFRAG^lU&5aTw{*PDx<PZkg%lP<k;m>mBsfIM>lIPxU*AdDr#xN2+KOJ8r;(=HFFkdtv{9s(ODQus<#P6HM@CilEgxOo1S2D$76f5O=WRr?stWvq%xC{H*8A?kfu>-_N%(%{;0T%~vjK)X7aPTf4e1qO(c4ewG8|GeCvChqFag3==6&CfI#F~ykrx8CwHLTWhKi_y=-^)V{;ZZ<7k@;0bBP^+QsC+m-0)4PO*Y~mYzVFgN;qM|6(N_GmZ~@a7W@^fyNZcVV`S&72jQEHG?LFH6QeHemt}JoUC3bqa9=b`cI)?b;y=f<(qg|%Qu7ZLy9^nzfSA?S&eQRzo84NR1xCMk6^v=|FcUw(7xb8PK`LR>muD7%HsvZOrbGZ{C5LF82dpA@SCwE2eXqw4mtRVkPiyQ;$iz%G~Jt@Q}n*riO_f7^xD+Nm9G&qJ@DC`djRA`oH9?`^!$0V7lCb0bAe08{t3kOYyND_y+*OD1%Vyzg>hJIvkZPkCU-)O<HPFU;hduX<(@7;qV5;N-ecN#y~ucH92?KFPOrf-qOc+bSpl?tg%R8!4b+WUDoJe@jq`Uh>?C}zX+n3#nk_Uoyh!XkPKr9TJijp~HGMk$u>_;+fJt^L*<BzH&Jo-1^<gmz?eKCLF-zu{IK_XaIf`1+tySjivGE$k;}gE1;O@DbH8I>95;^<aoLqR|*;tQ#aLwk^pr6~26?X+HWYjFwSvFd_X5l%JGISzPNO)0KeRr6fvaiXw{$X*3=u4gQ?8n-7J}`t=8nfY|+*Qvf-)@Xzodnq5AlLg-7httdkt8e2qu-dAWUj#*I9%I^!;j7k!+P+zf8-SwVa#;(`G_Mdn;%`R^#LinwQbUQ;-hBllTIq~gLI1=r{`>rd}jS+ZyZ-4V0iCr$hLsiDKEG0=r4il*LiovT*?dKrI5O49SM~=8&cp~EkBBC>F0alVY1YF`D=6H1T@$_`8L>baIr7<dtvEtAK@+(K98h>57Lg1hX3%=a-^7PQwL(CzlL&hiLVqNJ2vX&}-yjqD8R9p6hO2))N6!W-c<<&>)?Ashjop3@%-XO!P$qs5u-HBVA2{Vte*HHrB27;qB{d<kOn3k@^R?ymne|Bpd7&;QKL`b(2vJSX0IDVfX=Kb66@CiYHEb5k}E#c<=9(=FElw2OBUF@P|D~D06A98+qwyohE$mM^SbjhI|URZAJMslaQ>3X=llh;I-Ezu5YIIn+tD|x*U_qFJKi`Y#nS?(>-4E-Ji`V}RD;yqClp?FW!1t`d23{{F&)3x1!rEs}lC>7V|xC7N=^K$?HDd)^cqc`0%m{Cqiz?i;`$*S<yGg2*I@TK<kuh8pL&@%M@h(GhSR9=dorw*eyx1tMo;viF(*WV->CR5!BHlD4z$^2t8-~E>GR?;n9_fb$Qxf!p2?Fh>OS%~ozl{I~g>Cxg>^0FAI!MFsB%lfM?V5t8Ms`^N{9DV}@ilRP&G6?+sDOstXnJ2AGHh$(%MlpLkp5suzS^sG_-zoJwrQn<OpOy+Gi+9qj+w@MZch|b>`IYWcp}!U`m3oD6sTbx;OT8fI<x9(Joo>E-RqrH-?|fPFQ6k1vFYKc=3S=56?T;xuV&WlQl!#lHJ_r#K32rTjk92<(6nIb)mspB%{PyIk?#5iH60O<P?Iw0i+NPtksPOOa|5vF-q&X04n}|cT3+DlXh_wyJL-Yy8!3la}ap;3eeVXAudK4$c=bZu<%S+s`yeGqETu+8s^(OQe#iih(;8QFNd>41z#9EfTPKXw`&>GNE_+;a-tw<%Hb|v+GH_CNnwP<Jy%@#j!pjFg<C|7;9l^yRJJRkeUQ#Th>TKF?uQp)-K;pHr#uENdbn=2K(lY}@H7}*0|<UB(SLt)~afgXx$llWtJ(qMgZv*DmC5O7y60jY&U4oBy1t<m~nqg_vA=i?F}n{YMuB&DUrzl)iXgDGL^pk_^L@X%8D=%H|-*~-tPRxs-Ogm(0HHMf>Vu%7<EndJ%(0F`Xdvx~GI>Os$c0AkI@B94PownCtY2*!g6!wl*jofjM#+jOjBBJ4i27j-}%@5@`YUA*wzZcxy$XMpRW8njhM#nZA>wK>0C+p6ts)Hk!F0vT<zS(lIlU^x!_7A-Up=!D>K!;wQ*TacE`P3PTkjF=p!$3d5MQ4dz*flHQM3`n<K(%mL3R$9)aDU19Vq^sd5b3*Xr%)bcsZ_<4L`a0CQG?||DE*$o<EHb*wIJ*0L=p0d_f+M8pj^6zE!fe7-Vt9r*Kf##UeaM`|D22RyaE+0v8{_x{4xgfoU1Su6n+jQn`mpF2MTPcFG+JT+nHvbi>Pav}DHA1N;@c`9bfNKF!P^(3`$gk<w^iG{_l*weo{Yky8?!SWb<QT!AsoI434=kaxLl?qeQ?z(;>B{ML4_F{hSSHR@k1!|baXsAy5K<+05lX{sRk~_hU~_XCEtJ$Xn()4yGsGl{&sC|1FeI^!RV?pd1pi+s>~?j(755I8fjfZGuXs9yJKULa$41Ty@5z#!{d`i7w-X%=G?0|{lR(ITN0-dujnNldMAVTYQB=I4Uu&)fI|UuntWIFp!>j)c>)M7p+{S)jL60&UD)yL!xM56brsKh5><FfUqU6eC$jHgV+;(+E2#=~g5!YV{IFSmq=QJsNQ<EGL{e6N5FUfdz#-NiXG?f08Am0Vc!(Oqnp7d5=f6;!D(2${vdCt{<*!v|i<q`6Z>mOpwZQh<#!GK=zqL_k$MJ$IAc~cSMLmoi@BM0oF-l%;_|Gsr&deL9q?r|yW%0cHdfmL&Xzw*!WKY?u-F+iHJkYZ54y*9D(Vs&j0}oL;-1jg;crf&W^TDKA$R^7Z88wGh{%{Mwi?X*4tLk%t;09DePxuO|$_jUiTY=Mv`r0Z}pQ5!UKUhYe%0`gaozXD-8gsA*dVO>PkO=Q1&ber{cg<NqmKkc|0W0}PRD=97@;i)hkE@=1QL7%fTI30c&@pCE508jN#YeYAFE}~H9S<5I0+mS0l<+t}wHxHW&Yu#i<U7vp?)KKro!cOfKqeid_Yc$(^brPh=A1tW;Dj1HLaIKHd#LZg;SA#M!l8IICEsqv3&uyKIMO97Ohs>^-xfRvle|K>gJAf}5X3EA{Ff%DVHa;FAnn5LLtV7e@N`0yx`cx)vcPP(Oe`-kNR5ZnvNY!3fBWte+0Dy}^Ms4KOuxP=<c7i22iYzbo?LYNI`Xf>|HCyEQ~Q<*Q8u4ilyk2PLkuinr!h4_=<TPcC&yNtk9nJbldd&QN3mxrmQzb7-6#4SEQyJ^G-g$RbJJ-S5o3|A$DrC*(Q3re4U2Db)`^rrNgc@%b(E?8h}3zJQi2*KiBo1RRn9HfsxFw9Po>q_E}!ziuErjIqXD^AZmn4-zEa4HHhK$&AzH>i)+UgdyG{Kr0^h(aYTp=PUhpdpkA8PSS?eN$gjY<UWf@%)cY~_Y6Ag@N?kFRS5(75P#-oaAlzo*|XOi8YoSnGYtIx=?n2~tz5;J3+gwz%$=A(rHI*B=<Z+&r-Ow5V6VDiKyYo`oV#*9ZOp1oEL{@ZH)u<gZvLDYZzVfGb*B{cHi2pt*Z^nrGhur3%xl~GsX^yX&|=W79a-G&-g+uCBMlYYa_!~fYtGG19g7SSC6&4GAhfsSA}#f7wP;i@FzH$a>n0UET}t%buAD#EM~iLD78*>skK;`#tGYed;~Ms^!feNW*2@dys&FpNO(;SCu8qgRNx5j?#0P=Mf=g(j<ZbeBO%jwf_*JiWL{kw-FI9-g;b@gnCH4z5*}$_8;7eKq;kfv(S5ahF7vzYQT?0sust8HaO<=uDS(Y7-VY<0H`?`)$F}a?en^JxhgIJGPlopB3TEctX?0-TTdm(OeChJHG^7cXv1HtvxORii^T^gb!Ah+Fqbo(Cb5DYXmVZE+57dIW<7@%hJYOv=$?)WMGmQdkS<yJt(C-asN;sXWc}M`n6k$gdx*ppSqr?-z?HkM^;h|!21g7EGc&u)$s6qzP2m>N1g%+&G0D9sPLv(mlC=ob8B`-!~S55oP{%z;;E=%_*h<1O#RPjvAqDR?FjEO)(E?{ZxG(Q7DY8(RaTqIynbosIpYqNMP)zMze{qnaZ))MWl(e%=cuddbZE2ADaC^>z&jU)kR{10PYQIRoZ<Co308jhA!@Hsgp{x8Y1rS)fZMOf3ygXs%J32$-+J+J%v!R%=bqx%?u=<~K}SnFtxqW@81xBG|42IFkt*?Z#I@r%IC5w7b5YCcHTu`CZ`U|DXX_mh<tos~(nr*7+}*Rz@nG^G^2woB-RTW_Z0{G5p(2kyeHkzD628C6hTP#1Ucn+Y7=<4Dsu5Rp{zh^)U|Uccl+<Lpc<-KqE`t?-mW76j!ElN$98*D(ehGzT&<lx+BlXfJw5`(3a<1$U_8od3peN*GSN0KfWWroWVYK7mu^cmHwB^W5+VGLtna-|UEQ=238<FjV<UHjMG$^`8T!N{^2u>fTx>4KOdq=V~kvMd{`=I|93#LvPAJ9z1pN2~Ao#uWk;%T~5+iq+%@UBM8dT{I;WgXfSw#iwyi?DbUh=!?-V~%kX{_t$ID*H$3{y4%86yR=%EjZ88k&$IotFrwH#(lT8Q{QTHrIH!i*O<3_70MjPg2`sH+1VD7<kVnnaCGiO$2pGz$6BflxW5zO+3(d_ck5P}DLo%`ZM|t~t?*Vvn}$}GV!HpEQD;eV3owlU+X*$xakQLfE&A1pGddL&>b`uAy?qPbA9V+!MQ<sO2eua~J_<(|<j`{vDg>iD2chW;A#Z6U8kLU&pC1gNoavd89elHaP1egfJ0a}h5Y9e$fowRDoI%}b>`VqQI`jg<nL>$_cY_|?oEcAosD}RCdeeg6qa6qmn}vc<37>g#I`ZWJ`=V<Mc!Q;4fl{z$FRiy8;}Po?!Y)+fkIuj7bfsX_!WsqW{bBlLhHA5-<EvkDsU^=Ua3+v`RQ5b3gJ?vwmlg)W$GndiCURN@9A9QsqnVV_VU!m$C8-Oa+#0lyXhvZHRWruqM&RPJqByVsYB+^&s0z3nLc!NM=FRTK9Mo@;d<*b17%0dwP^MYS@FaB!N`{7%m*y_BlZBQT6eSG>ij9HaHan5>uFk9wdq4WgKBnTaEATqVhGyRsQ-kP)4ls9EO&L7tAaonU=}O5>n-&d|Qx&C^zl4|Ld!TCnxVd95K>}aKS`(E^Mck=_@NU&PyGTTtZ`Ig5(Y6!}+PB!@CkKm+ahzWyZbC}6i))2d%)WS$*$G<X&m!sIC$ujvXD%TDZy@_=(rlRHnSv{T+!|@X0!l1m;nlcwNgCJYjl;ryQI&oJ9ZKDP)~SijhGxdIO7$ekidfsa2R=%qK*VlM;LYIZ5XD5MJ~Z{4s>;kA&J`o7s&dMt!K<bel!ka~+o&^}5`&+$$%-4e(Lrm5JHn#dT$3MZ73j<7#&tvqN_}Fl6h1jwaB}6O%pPmxkKn3`I24D~#?EH_M+a^fE&vGm7$iaf%;R4G&B^R36QDpG|0fHb>!f6tR}~~oVg@4k9}OWVf`~DU)h8mu?BFonpe-uQzFjS*S+wzvgO61a%G7u%)jEs|u4=rTe#%7kjMcmmi%FvI-!n=~4cv{rX|B5ORkbG0qVnLX^cGnr15s{_HqEEj;GV*GujSQ6S(GuGOM?+K`1RauX*Yw>*yx{)EjgXv;#?A39AuEP!&^);A7;HP23Tj1#lw4Ja{q7^_eOs(HHbF`Ot$g0xi_xLyiuIbya7#R+6`U;2Rt7Z)(x>lShcwdF$NQDU_FY)fEQ+3U9$?C1Q>T=(roliX(?vRe<Ih|r6j<Z&3{2fFHCm<w|C}EP;s5~=GUORfDtmdF3uV7U(4E<fD{M`f4WJQ15TXP;m|KV?#5#56&4x_gT;c1%35eB=FFeRDBjl~qfn#DVO&f@aCPp9XetHy1oz}kvQH!xvN-}t&ND66hyun6EY1@vL4je0n|GLby0+h*4-zck#g_`6q@0MxBb{@ChbT@u!HaNCeQ{0-EANCjp^~f<;*`1)>BMZt2`({mH!n(TkmQ;WC23#IV3`mP>nHgYDo&@E5ap&kaFsezxh6yv_TR5kLmKCVIX2fXf7bp>FiOn7`^>FRF-V9G#E-w(oDnDn)A%BEvWU1$V8Vzb-NPKYut-y`NB1D#o>hZ|=fM~>?@e+$*rq>4z|Ij~*ATy&fVD{gR@#x!U>-4prY&EAok2-b1B_pa(b?5nZ<rd41qVGONEEb&F3hAb#|I;r5_C*_8e@faB5+RTun{agE?4vH%&Jw99cZIsWjxS-SF{J0R|wd^C6CQQ#bl?w*V^B}A;Yy#xbtpW1h*UsV*qCC1Gf#cyd*3Of*gQ+khjQRhxR!cQ%!C?_V}G(a6j25X39})Zp^cM!@cF+5W6k$JqV)(Lv0*)SH(R!n!ghB$5a*_HIPHayTRE6xe=Ix9d4m2_(`xGVAR5B5=(_!Alha@cVlCxy?^HpNZR_&9!J(RTS|(}4YIk)YQIdFW-LvdO@AR8?PeDZ$eFuXEltoU3#{GNk_35GMv*r%>y*q1B1Q#so|kA+d*~E5z0~URZm&x7pMT92dyQb9`5WDmNU2s%13B-Lz0w9RJ}{DL<9IN9IFWfiBH)+@f`ShEL1)Z(*m8qN@o5{y>y4l_M8R@QNwo#;!Kri`mK8ReXv3@r9z=>&LfX*c_~WNx3=(1yCJ)Lw#aSP6wk--Nt%3^6?L@)udmM>KmthcO-LQwofl6}l5ew`*+D#ZX6FV7bpPb`dC#rFv9vN&xaZdN?0uH*@<of&|!P;_fwbSj-0R&Qy%>xAbUvq?tLc9%<3FIZC^uPL;anTVi_X9%Kc~0;*0!;EV0j{gyWsx)hVQ>)r=Py*iCyY9YU^uqrNN_<07la0bqvPfLMNk%KYy=FI@*;td`|3pwacu3-@6VX*;ibSLQt&?vB!n<L5%7<O)?*=kc&1|w;g@e1o2RrM7CP(Q{5Q;Pvb<`+<}vFwv3b_QVmDtd<d@1}etD_XFO-(LE32KQ(&}nweYvofFBdys7n=v>=sC>{4)9Zhl!Cy)c)buihz_2dwtAme;($3O67#>RX9?`ddG+63{o&RB{_6jJ^)Ijfz*vc|{-mIK2u$^-gxjP+Ct`a(p=N4dCNQk<hg$+XQGo!x{^%z5moTEAnFJhiFU0_!uLC)>Dxjo@Ndvhtc$jH`1yuEy1OrWk2(N!dTt9z?rhh<_@pn4@HU0W2?XXXYtK;t&t9llXN_vjx{nes=Ji~p(aByHk{v&uKIO`2m_I)JcB{&gT1M(tR4<U#^FQtFOM|?C}Ac&GRFNQ*-)xH20k@9#%s6%w)M*}70{|ew|MhoK4Xtbm1@hjPbte3zN#GkRF+1a>46~yiQTnH-#Bu)zyXOkA)ZydD--{+kqS!E(Qh+E_{i57<vnZVAIMeBgW5kpPRoO1&4gL`4^Zaoz;d_mx%(`pjq2W?0>^vepI=8SYY%a}ib{V#&u;w2uqn7SXa9r!0?(SaWh+ypQY7MBDS!l)N`D`QXphZ@LLmY@natPW3ej~|3%v}NU(RbW7r-1rcK8C%wwcs62bz^89oO7NHx!O|Yphwg)52>+n>g`y<%260g)F~zJ#DUJo>jhiVYEzr2hgF%1l_c0sbu!=28PE_uAn#rwY-?KN!+Dm$6wVZt~{R!w~A-BAowKPuNo(KZV9}FdE9Icp~zSPhMjSo>p(SlT8C(9P_N`eA}J)S)u<8mb$b@Clr2?S1tiD-=Y(W>J$nDaj)z6dud67`L#V}2G9{fc3?F=J1SQZK~rjwYNA;Q6QX3|Wylk1|v-xmMQdZ``Zxz(4IeLVPkEvAbJqw}lnvG`m2SyM##p1F3FbkrEr@gL-bG#1fhvo{b8~@wr2%`1{0+@LXUnKNqU&(+K|2s`t(^Jm_hxw<Jc6ynn+a-cl*15f~S){K*_hp=*t>p0>OBSG5f^rU*<Sajae~&z6Sq7m_%_AzV+wG5l1j=8DWM!Ladkrpiz13bLxbU?Mm{CMH?dM7f@!<SC9ZI=95t03oqSYx#tV6B^d*e{ksB`#=zgd@5U_F>}NYGY#duzR|!Bi=1iETN+4nBCx%h_Gn5>ut?#o`oV#F2Ex?`8N;v<T6$*=gZzg<MuuK<bQ0#CBxsRDZYY(<&{~qJg3^d&;1q)YFJ5Y(!nhPDzAv+jP-@C=ldo}C2!hL?9guxXzQN*%F8jWbKFrxodUfBOQ4GjAjzFLTk-;}I2Y!&7Q@*E4KU^@j=D-#3ObFG?xoVVGhZ4}0$!O6+Ap5@L1`wHT?xJ;pejS`0Pmyonuj(6ner@}9<L*Av!SC-h_I!>WY_@!8!G7JfOyh9m<8ec}5L%K>PGC;YIt*}U%_C0PFtM{{e5AqVM|2Wu7Cb+=>OoT^F{oD#gVnqvP8a`&jlFw*{YM*H`<$6A%>3C!a_e6GlQ>+E^6{xsis{U-Qu(}k=A}^Vpm>s3*=O<`5zIevQTLS9^7lbTkzk^83OOfVvgW-cgso&jdi@)!{|ly_wDbcF!eR$F2#iGV1Zo~c-kpb-3`3?C7E7BlHcyX(GltXU1j8$yQt8tZH_|qC?()ji&!*y{XVWsDO?Z~mI7KYAUobK>Y6V~Nc>zbj3p&Q0GoymI3cdb-X-hwdW{Als1wR7eJ412eaCu3Lf`5@`)9YWu5o6aD;+LctQssg-rBsxi&M1zp^G%??Xc;x?Vlx*2#*4qD!RcU{j}_i0E#C(^iUM0DsbZveZMz;R!K?u{cKkXqk_dTA*bFF5cbmKWTY^x0e@_KB*tF&N48_G54PcA}O+n2-JFgUPx!5FKO)4V%3q#H@2LN@4tRisED9j;E|L%+yQNZ0-{|0OR1?t+S6#;SKe}ORT7Is-wyz!Z5XmN`aMeAmH{TtVF+|3cWASq#!QB^e@PQt~4rX~c@^~OL?Z@gD)MTksS*N+he{3)@FK7ajFA!Sem%`m$ur5KNgh&xWZgP;KVlEw^GW1@i*nDuCK{4kBjfUJ)s3CUph>XVodiqfQSw;G%C%{ALw<&ppdcqbu&@Ty$ZpRj{_DMTE~XLCK*B$P_ch{NI!Tsx00{x+XRcTSoXS1=A9bHhM)q+-qXsxnYt9v+bgc#Qu)IL+{90g{|?9rO@;L<?q5G2;p+A~1RIE(u7gS@cM@#VB{|9{kVwrS;m<kAtOm54AdG1eVzTM}>ma!eHpA(d4+>sb%eLr6^&Jsq-nJC>o`BFR1wb+3_*Op<=7K(T7h;UZHsDdc$Dms0h!fTpOyQ82wm$I%0*<5h&h_r!&d*m>mDCIy#fXX`3+5y;9yQ;S&1s9#VhsI81b1I}AO8Ob{Mc%N05T4RX_hRxob=u@CZ8ox0kaQSmgXVus<F<a~R0=Sm*RG+wYijzLC7j4>Swu=kaVHL^C+^EtgXtIFDJZP&IMKSpu<qZ(XcKwfQap^B&oQ59>{&StHJ_<9kD2T1wNx_<}D*mF}FEC!Q~brCEMyH8+9h@0BoyKsJB=pzP&xF^RT;3Ap8_Q4BR990fMEpSlvRZf2fMTnpfF!Q9o4KfrI<@)_b{fBJQ%O<syga|ja6jOR>8PVg>Wb!yTeh8Whjm1@UixQ#f$+|bTK#9OnZ$)!d+uv;Lnfe9Ql(nI7Sk;W*)5#e*ixWmYBszpc?y0;-N$bz2OodA4<t-*-`CRg@e93MwVKhRI;{WpcH>i8)UTIkn>G^0l#G994g+fi=de3T0_-ZZi2FG5r^P%oSMqEPbBnTr7lVz?-v-r>(=+Lkk#KEl88XFwS4ArL0v~<F8`+@W=#^&V{80in$kjgOyAMtRKN>b04Zr1Z<>7j(1lyIBswmNjs|B^1u@HXD(K9}>w^gZ|x6XD4ji|I8cT+2K2)Ea_v*=)3TYb{FiN~ktR)%YWJ9pzy)8cC8C+FwH`Mle^0=hIB4OSkMU{t(H06tMl^WN>`pQVL<0-^+T4e`djSerU@Tgxftv<V~svS{+b1yZ{3coDOn$W=?X0iPWI{9y2i>9dr+~nBO*2b`gf`@FGK@G2I+9oo1tGnT$MI83@d!KnMm3VSi{gYsquNtCKYWj*{PCP&@*k<G!c^+JETrxz0zfhkikE`lcn^4W{7<#v1aFi_vBlI`~7UZinN091?ja<(&m0Vi21#Gu#z!y(j3{p|S_z1dq7Rpa=V$6E!>+`36Yi@}t;tS*=^4_Fv9O1r$J_YNih=D>Jj}=%9@&&Df0tEb1_+Da+*68D-)y((5-|pOf*>QmDpGjM!mS@S{@`a{Qp4A4}h7bxQY?X@lh<o%mF@rERvYZytvV{fMXC*HC<PK%}&YN5_ouh9+gbgV;(sTqN2%F4D_nAS?MG8bMg^)HXqWBc%or_?nk=OD9toqkD^i>N9~Z;%N;9xPU~<nV)+S9%1A|)y<b@q*d~%{23jGbgO+zp(5-*i~G%__{Oh2OO(N9%>RZ~&p(kZ^NU!>jXZ+UKlJNY&r>~aXx&uEP8iIrHkcqr!PsR52sI9;7}}alIkX<PbLU1k9<a{H(*kN>eQR^>aekHZ)x@{=x3_DpcV=R=tswED<|MU8@rVItf1QZ?A$^L?*-s^T<8uyvvYB>X&@ulTG-7{Fr!$8WqOa|@B4X=BB1%M3%OLm3)?CYje45NMiSv0Fn(v!W#~x;)Dw)_n40C2Wze?ecy+#+V`1!=(be`YA-SqC@G{dDIRF(>dp7>m-9NNwz=om%}`JL>yh||^&vNsA9#sjqDv!b!JiVdbDc-e4Dea|XrY!j^<{+bDoE@%Q$3+{ZKf(iKz%g!k{o?a%UIJk48N`XjsqCr=9Ddp(1rOU#~bHwVXaG=yD!Ul)3=9%4g|174|CVbcO%E4yMr>+&pU<j80Sa%^#HFc2tq+^<7-_w59dTKmln?bGY&g`-~S1o(;0O<*Qu0izTyP`(R<HuotTIC)br1q0p<Bg)T-<{oU+bcFLa*Xk>Rqp0N>R{L#(0LOL!1ET<_i-R3WDhk;K=oJ`vNt*FBdZb`W-#gR5j;Q<U(zN+$USfeNCECq5-N&w)MDZ)LcYXe<0?Y_kY=uROQ^X<Pg4(G4qh6WC5YfTE(Zq!>aag3BkK538mIKSkQFf&RwiELK~H~^2V;T+Fh|n|!fE4Hgl@`j=SZN@=nJCo)Z*kYE>s{sJg>k6l{ERZcHqw5rci+>tP8b({T+ii!vAr-)kK@qRsulEa?qLzvEV-QX=rgu*rnII7=kvN%xdKL)+?HdbRs#`FK#)x<<en=S?yJK1*$y;AZm$eik@90n#_Im&t{mbiY3CoqYzj@J8YBvU$6y)vm0KL_gO@P8_d^cjM`JBcFolg7aTx+Os?&|Lbh%+uTZ4iO|q8Z=_uMx9JRz}i`x1N@|KfE8S|Ethw9nuU%vir6r5$J+r}|f_3>5U1$x2Itar<Z^GfjPvO@bQd)1<whqTW*!bbIyLi&u*doBtYcu7wc2_n!Kv_1Wny!}l2{4|*Y08jqm)xW&@Z*UL#L&BZL>1C9WyIJ4aSSbFPGOu=<NuyIZs%pR6El%*zfvXTm5xjw+uQ2P>8luB*s5DMVV=lAQQX)_3C7JddWySfOVAwOGrYz?Yyqpv7!j=0BBuj_63(cT1FXh!P_6nUNSTVDFP0cqCK}p37^u~5$?;U1VB;SDKX@IS<fxnRa)KI={@<2sR?eDbe?dBFnT*U{S%_%e}i*_!>soAXx!-%wJX{a38bd1>uSITNffkoF2$AfT^A-f)4FA^2Q9D;T|Wt|bU{_fj_!yGxUQkK~9bRbYwu?4CCbj;xxM9?9uC&}L}rSUZbKctrG%~))1V+Z~BjrJbOe2jZ#IvZ=VgA0V9!ca-U`XU!R$2-}Vd&*vpdsar;v`D}?*;sU|grj*UCEk!^mo1l=AMDp!7!kNr&xPkxvV5sEmpXKZxd&m;%M|i?ba+n&N5d2sq$5*mDF0LTcYA+puYsA46fC}TXREQX2TKvuP69G=IT3e~Z~?~8EX>c{%$NKG?1Vvf8b#+teU-kVaAXq+SICpK`jqs%#Aj@Uk|vRtaIAbK@&;cThs07O@^8&J%ka$553rY*U-<WEY>_EuXpH#`etTa=Ez?t<^SSf-r}n@-3}kYq2!ZGfVZPzs;6N9r$IRRoOd{}prcCA)K3A()Y=(5>Q(b0KN1DZ=>B%Z1%ip{c+eny0Ji)L--%R)QWqMRh2~8$<%}2I7kZE5=zSp5O?YI<W(4i7xB-d38HG~&XWR{g3gxd};l4MFP#{zoxbY%2FXer4ix&W!dnwP~k46e|}QdF%MWRoL*+J}_a%dR}qNbQzH%u`1HJzQdw8Hm?TbY2UBZV;^Z<|I0+$7Uxw7xE%4_;WUDQq<E6*3@25T<80o6yhN*MlinctX@nlclr1FTUz1>XNa1fM{!r+Rq?4zOXg$-PAB1HQOsTbBb$$2(wiSfIm#jpg_P~DlmmOSexD8K*efl&kHv<_R7LyMZ?u+}Gp{_UXnqB~RiwB4EK2Q`xzmYNfqFj3rq>^_JvefXzu|1?{Mvdq?5}jc;cV#jb@|;m6mEi^{}{tlID?sggwpec^<J+Zu9w47sbA=J`s>SUowZ_TWhGx;Da*uZcrqUbLD(-B`o)z}sZ%Nk-LTN>bykXDx!4Q3YtfVaau61Selb`tbW4S`(ptY)Dz5cA%d3U;aAmm^JsA}9`SNP8)?E(zrQ&M7vsNxIFE5vZ{&Hb;rQFeKUoWq9x}DBiw_7f)t`s``aHU&X?)Lg?-Su!)tGyIJ>!G0haxq_8UGA(yOV(k;!(yoe{}jK?4<1^w#m%`fv&FlGPHB0ix4N|6EtZ!`Yiq@&APAP1mX{05%f(W#-s$(QmMtF7ct$h|Q1TiZc(q+{gVR>~N@R3>A-?-d9E#vmN-}y$d*E|j@O>vukkHlf6ScRWB?_%Xx6Pl6NdD(H;p+b(9rwS4LZ98VNu8Ud0DLAkY<yeNo%9n9Skdm98?~KUt5I(o?g%rnnji!FXM|06BJ$8O<KVIPFua&}!Z3xD<#a4s_{rf}X6r_ad@_zx#ycUc4?Lg>)V{Bz5kEW0By0uzgZH-D?k=KD-`+x|CWd5zUvOqHDH#t@jE|K8Ixt2WUW4=@J|&3>&$%%_`nU>BJHS!6VGB=H9XUKq(hpdeG;Fsz9KC4l#!0#Wb$k0>vGTQs<o7fdFER2O#>Uds1RFwH#}8BcP<aEB7SC>xsxsqoz)p=f{j`w6BRP&O;8?Ry&dHZ%8FR$U&Sr0oi!dm*NgEgGS_=OpUJZYeIXTa2OGC0*T;-jtv<Kv~HwyW;-dc-wxR}=A;_M!O$I-7eXc$w8EJFMElpR}1tpmP{1QZ#((07ZEe{w74tS37iK?r<+)A{#8Re4FSU&FNaT$(kXF$9K#ws5uw;u84Nt7kbY2>NM(0*@T{kOHdtHvE!;=$Y;Z>_Z6|mjou>W?ubs6pzM6StTk!*EeJ_M(^w6BZdBR&h}!V@7@psG%+dvh}-f7h30eK)^o==4#;2lob1M*GF%8$@FNLz@+VUEK62PwG;!$q@MruQLoPl>WCt>Ea&DQx7Wv9JB=A4%mN4SLgaFV{7(D~DU;I^6e`@QJMTH7XJ7p|%{#=YX!{E{5i$e9ZZ~*hBgU6qoA0BYS4h<LwOUn*1GM(;~Xqd{NFteFasH$T1w0Lk@{D!L%0MOhy*yU%z@g&3bXDji=_WNl1aj7MzUpW^rxl9=eegTW|S65kX^Ch&%Fdh1!gpba;4=~CMcOCZwE{>!lggxOHJUPF#DHc-ikWNOclF>xezP);DQ5yAR(wInW*DXwo)y&yF7MAd+4sx{d_pqMFpp{~<mrNDPCN1=m*f(2QtPSFEPf?M0Cd;uR(GiJ}mMV8!&AYAIHX36vkUTEdP)}wy-PnT`>Z5tdHr{+qV4-fSHhJ^wOr7$QZ7^iSQzGam-mW<KZQ0e<|7N|t(Q52cTopyqC+shp&LR?i#W~4prEuUqWn*3<2bk~gcuW4N1Sz;BOh@8uk%_;Ib;XaPDM&b<KBusg55=#XC-!T@ohwtY_IDZ!u0?!QF`OMHsK*qT@~n-fjVjujn%kJ3-h}=Aqdk3m>8`ec+2jQME8@?!G1i9npDCoC1I9mBR<T$Vd5VgWII`o*w;DUGZaF+EL^?{Mn|<@Pp>1w5O=x_jeTOV3Y06L{#GR1M-lt0@Wgu&ILrs`#!af!~WFGdh!1$k%ni8v&<L#nAOl8!IsRNU7*FOm+ld6=_UoS=2+{GN?U{*LiIjzc3JxHDham_+iiH&rl8`xaJ!tngGDm%9j>z1WvQzTW$PTa*Kks1>ZtUzF#Ds)do_#p2oh4-X(NsS_ra$!u&f}C7pabkt6A;$?Q@Ryu6aKvPhGn(&_q5|S~$U4meit@i&)99ZiSNC(xk-_Wq&vXe$alf;3-B`Sif--pFW)~LnxUOogyA6bMh$%Yem~p&F{EnI!0atoae2u#RMsJJ?ILAt<G7VI=$eR%pz&@Fq2|w3xV8~9*45TrPQokcv^J_;yCw>{XH*<Kkj&S)Kr<~{f-_K<5HD)@A(Cx3Zdm>!X{|jDM{`u9vz516||D=hG-BtpY;8v@?QQO+U40`zO;P4hc<6}}0Vh^DbQpHRaL$uN(yS3I{+oH&%8T|Sdi=UAGhT-k0^MuOZYwp#yD&D1Lbf;t5dBPf%Y(f^E_&X@{3lh7Zq=OgYf0CVJ8l4`j6in2(>4v>C1&*%x3Ua5x_$O!K)Quk_NR91_A2Y`wDYA|RY57#T7NV(PBjrmv3SS^v>E>DY;pS~mY`LEc736d06jM?sx$bBP&522UV*7=5K13Js5Q8+b=rn{sY{^4NZ8E942ks5nR-8QW*WY&k`r9|qIdvLk3e%S!{1^Vo-k`tel6gv5kkNSGa1SjkLjF&x{lETpyS`Q5*mK@;?zEcQ=IA`(hFp4QL&_xO2$1TOCh6O@cApY&O=8MYTAVBa#v{k>iZBCh+i32-<3KZha+!7ZnvM$%I=ytA4Cj+N(77M%*Y`XJ;euOr0yS}d+}_*FS_{Xlo2p)eEBe0C+~3*Dyp^>K!GFrS-hsVOXiW;I-3PGMGTDRtp*{8*>ikw!+Z`<8TXncD&<6WsT!!EZ+vVDwC;0j$3Y8bE=zxZzZM92l1nOYOp)p2J-G%yU_u&5UUcFU!#NG;fPNVJY>~C#3&6dR8*1iD|?+hk{3=(G4JK4lf{ANmx@NkOxDUEK%{hHGx1}!l?DE~BLH|v%OxnF^q5+&H@B+{A|nwFGc5%H`rEb!IVQx{uLv!r+gm*pYhhc|($F(|-Ob=2BUN`jYdQ8<(X<J^A7slgQAtKDzx+*R#;yO90bV;>EU7=Fbmfcrzw4q)YnLm(-BL)VGqx%)h2f}(1ybMzeqCVr{n3%6_ao2~73{pJsAyLI7T{vDjo|McqjF`u%4Bf?B8?NG}lYcP&&`!X=6R+~h9Zq<aQusYnoo2lJyiygKU?LnG&=vT^n?H{!EGWhat^N00T;&rc><q~jI(u&yMPO17P%7@0CjQK=1>*c9T9ORTT<w~nmZZ|;D+`IPbEqFz5*(-TvZ)Nt9?e#OZSgT~6q?_3el(+0n*}~XJr<l*@=T#q5fg(JE1G~249##`&#ge23!{FFdB=z-}O{P?hQHh|}av-qBQ`kOm>C-Qt7S)B-cBtv#gbFz!QbJ}(!p(l7f`AUI)p3qL(~asx@OrSpE{nhN=UHJz^rB+=6D5Yn@_QmYI^?2|BY?j(>BQkj?F+prieiZyBX*r#h~a%u_T4;0l>03b8qZ@cbD4wb-jE}5!<B!8wlh+Vtg6yJND))wV8Zv*6G`_o&Y>g-rlaF(AzWJZ#`04()_iy@dAY32Z)<cT6XZ-3Stjsx5q-f}zh3&vT=;_aM`7GEfn=rS|K8~Mq!L3(NRQQdW{LXov|QPj6005cfoC$--Ov<Xec_zy@dC^cWo|+7WsR_$j0+pioA_0@fMcPaq~pmaPKEgg-a=E&cWK6jrWJc8e!1sg58PA#0gMpGktY(sp+OuXo#x_`m7pt@p7f#uDN2?An<vvr#`>g4V@%MuUXd(U`yb<&R$XdC_MmW>((PE%kA{yU8w$&M8Xz$of>`IXp1XzTqmOi1%^ja5J^|ej4)k0;q_qFdsF(4-U5u$;pklZFgMEx4hHairn{3TOCV0MmX$R}s!?D$?gqh+WhmT0&y7eFJZo!#@3)pXgoNpp13Yqvt8g?;XTybZU<%mQG^iB|8I2@US2}uP}>%t=79}}z)%2bR}fj8>=NY`M5A61fR@yTF3M{1S=In?z0YGfIP$rnZSfP-;{M`5Pmt;M$wOVMhdJpoS!m$9olo7Ji!;PZeVk1b~r9wR47E+KxF;vm>%c_^)N;)y6%!I~hWnLfD&+twdf`M2Bvw{(h<KhD7|U_ki;lfmR$Rd=&~pWGB~%F%^Zl;bnR;;Hf_z%Hj5Qk3{b-sfa74iH1H4`LjCRlyL>B|dNYto@1ufoUlaYAEB{JY`oA0|oxnL7u-`*3eJoP+|zu_XTn_7b>m~DV)uG62eqt{3Nn$<1Vc}8RyjBu-z-U^3mlj$Ji7k3#u{m>6N(NwDHWi6zTA>02vT|H5>tqb~PPlUr84@4(#W`Rq~WE#WB?7XN2d?uA}#bPV<uCnqGZL@pYfD|48|2U~qUrDEanwH9ucivH=-m`4_Fm5t_?$rkBxLOnyPTVf>k(zae8MDP{%$y)B83ktNeuOs_KKoW2F>k$6P?9X-Me;fXeSl^ghrYr)XarkrKSfph>K-V)7H0exSRHuJt1&aWa^_PF_1{16MZ7GJBue;RAgy?1t-d-v*X-HGtv;&e295P~>xD}DDKi#{(kd_s0%2J@cKG!c=V#0(TdUl?rU(WtoE2-!*#z%0UZ)fr%BpIgZg)c*v;+2|PYg+OJYj1S6o8$)+EU4oz`2#Bcwx6YL`5FcLF6004Z_SnX=q4I-7Fey}GwOY2nV+9>_nL5(PHcb^Ipa>SMQ!a9&eXqXh+FdD}5F+O(;X;Wf6#*)=={&TTMi(ebeu)*-29dVM{^|v3?-&i<&iX2Dn^!6c9fNQozlay5^Q$j@5&1U}y^kCRzpC)A_(;OJ=q_|3Vq<D7=UY{_Y<(yochW?SxSUA7I75%o^bTQ?=wW7075*P@esb%;@rw%p=D(#V_V}XCIz~=JXzA)BUJ6kXF~45*;32*B&@<jcNT><tL!KC{es^k(EzI?d%ELIXVBCG+BRpxmf$)al*+me@Sq-H7dUMm4u>FiEdVImD4zQ61Q(xuT>h+_atA5~4!|sD2>cObXQX(Aue|`Hlwidgg52JGYnqwjMBX-eElphHmtbiabTd-qlt&MxWa+4ZkLA7k$xtqhqSLJtHt|Mj^=Hlu<s&DKQls2XCk&TJaw7Kj=Nm%^iz1nST!Afbi$dS}A$+|@s2~qJF2hh@I+YS*~UVC!rotsYdicoinEiySro~4Rp{@6yuj~iVk1X-n2e37IQX>$eHuCheHW08X;ZG-=J>3G*!0)DEL%%>vZ5N#z9nIQu#K9f8+FF7YIvL4HTL{3^WGTyG6<F&H9URWuGD|2$Z)MN8<yk7kmOrWCJ14;Lo)Q_Jh-NQnMK)vc0oCp`O(+EvM093tr{VOd8>|aT@{!GU;pmYqyfc8AP<;kQ2B~wwYe{RauFA{e)QefYQaegMr^hi03#c311x(GkEgq@Lf3~m1(vqdk0V}HYWS}TQquU9Ute6x94r9#<~hh)|rW360YUI_|o%S+|bYPeMDF0U_j!`}K*DZgA?S?(9sgL3hztd;jE1(ehU!6YkazYs`HgA6>rK&Uw+0!FI7&S><|oj6KlI4ERiFghCfP4`Qc7`a~RBG)n5^fZd6uRroQJV#~uC6=JEMrSZLmIzaXEj$Q$+6I|!WE;u24&CZc;(RxlKCpS=2NUw>CoFg>I-%Fe$@m0Ro=@QJS#c>+!S8i!kqYpFbbeqc!Jfk}`Nv4JUUduQ<(2K*=<C0wKkDL>(+B7unN$l_k11#8RE@Wkn`{%OY$xF_Q;soSFIFE6Xma(I<_<zv&_brVt2k@~y2D9lG^vW79Q>{Lz=m=LXgF|B!!e_2V1~6j9v#!20;yqq<ZI~#C#P^GBf=4)(t`dD3w(k})9HwB1cx^c8E}>xaRmnsbKH#)Jj)z<)GnM3dfzf)7VMs9_Uw64xPZLq&KqVUM$b!zse3Ssi1$R{l2GdvA{g{I24QadPfl%T)8U09{}9VZ{7LgvHD*Y>1B0_vm}#7Tbd7Psa3F~_@*mT;*+q)hcXDrek8-h=b)3IyHg=$!`|ZZg-Hf$|3$zzK!JVA&7C#anvfo2D)XaUEvKKvE_QL4NpM_xH&$Hd;Y-;@)#mc9`<5ZcT`Q#YH7$!eE18!0<)M21$XTpz*L5h1ymiR}x*Wy2?w8ejPg|gKWSg>#m?T7{rsogoHgifd@i{zzTf|O+2vK*=zP{>DxD!0=pSSI|rzQ4coecz7=NDxkNR&hm9T(Vu)F0(tRP9Kl_$H4^}?tBpCQ$%P!%^-T|BM-U&|6laR;pwD`_qN;3=3aZRRog{kr8WhE(DyCa!p&{C^CtZE3mAgsV2LS@302uQ{!#5~qlYlFb40ltE0vMgu}HpF#!8=qei=pRG>53|4MWPTb~IUY%YlJif`N4;l=q;!ME-m=$ea`{3}I0FjM3qPnp+PC_)#HB7EXT)m%H)e+tszo>~ZqP!z0{)W7rX+lbk@k!f!K+O>;D#QB&3C1eFWL0qyWfFdhdN8UN7|EH&P>b4nx!-vnP4T^Ds=sW7BgkoW{ddgS*?F6V4zzS`xZ>$tjsx1n&H!GRHOq>N~vC%`T7bJi0<rmRRWz{2A{rYjHi5+f2&OUh)Z(t}5g(EW&^?z!t8BYDyId2~msMvNMhMQX%FA}Zxnjkog^8c%5!XY5LxGUlns&>VWQq4MSIyvop>J29pnYnk^F2&gA6dlt}rTv4q-g_N6;MF7)*kMwHM&p$asmO{8#P_<qzUo4yG@{{b_#k`I$zlhF_^*n&?&Zua8^^H^{0}v`mC|=0rz2aN=32S6K;&3!XN@%X)dsR#y&)&>LmEI^U8mo{}=G)dX-^d^ypMmx|p7?=VF$I(t{nl|Sm@;sgULc()^A-q-7kz<<W=0dac)RskQN?`yu2?VC$;1|2<C7t26<zh4MLZ!CLivK!4@Hz(CDIcy7g$giVluR3wiwR<(WP;TZY26F_yNxXO@IO|)E{`H9(e05n95moC>(z>RAJ(w7Uv;G&<yt=utzPNY1Xw`k3SK9EE$FpOtrW$32V{XxYxKJb5r>+m3X3<0ae!dO=cO&mlJrOC&A=lRsMRQMJCxIt1QpOkruTlFcQf}!tv3VocigZR@4d+Y9&ixyhi-NvzKK?Sy}86)`HRkuV#c#X~1zMy$uAI!Th380FSJ7G2a<HEQpqsSHrY;eaG2&5OyCTLGJhh?#Dkl!zzV+a3V;bjxRh?_D*00gT6R@5KNsw+>J5csdv_ecTkOF2co75`p_l3j*a0a#M;x7E1GDCZ4L9yk3p#&bcZ35?i}NxebhP8OgMfxoP;MZj5_Ly&m&4s|LT)VGm6MRVO}3aIj^WSW#V>%+nla*)hF+AJS)o|Mz4D}7MB}6g7Pn0twL>hPkT2R3hTC-OmIbjpO?wi%hlSfg)m{&#1N+%vcBW&?rv}0+_~)_kS-QIeGm@DivM<rMF%oCghydSmOODz5eTtIMqxT@9pn$n*%FpeXsGO1-h0|jmCjnpS891=Sv+E<+i4a+Jm=v;euS*eRU~T9aFx!0^PurX8fOwp^wTWrCjoiG4@z?h%uohlMvsp(GvAGZfUU)Nco9MOnf2GK3Kd@CP+?hP3gv_vqEcFbSA2A2e4BrA7&(Fs@I`hByD2rA#o;OYlib-Uo`HzNe|my&55w@NGZ-fFh*@EdwjQ>jiUk<w2Fl~aHjxMtshg#?MaSYEY#2n!HO@>ou@w_rSZ|fA>+lT=rOQRTxtiWhQx{+a(|W#MxK<Zesf+bI#<CXNgcxK&4kaGMWMR=F6uk(DLczGpO5!aUr)DiArK2qh(`OfY_24{02yL|6`*kYhJf#pCQ}Lr%_n`Ktq_}|ir<fEGP~o|P<w&59MBj_(TmAd}R-H%-1;(%{2fvTAXhg@5K0nk*UpZZ-NrE007snUrhmJqKjO$#;xi~yhY(TQMB~_XQv01!S5n~>y@CxJN4tzjSz}mrtoLr-`sW|i}n8Kj4Yh`g)h&YeZ-G$0RF;%3t#57gaWWx|z=FJ2uPsAs+SKHffn=66#mSE1l*WB8K)5x!HgD(EgHp&x}iBvOeK=>O;)a^7!l!1+TM526{)sy3LnK?-DQiJwQdq5RjL%&x_lj5nvU(c5uDtt+l9vmxI(9ISKq?=nPf$rYkzPS-hZ=+HK>q9ts!v#ig#Z^bAAZCcL8SLF%VRj8zi(wZeVX~jrWbCva?hv79h^h>R20EHdW&@>IREe(meX%r!N-!qIrRV@;*<!2%&vi{CVhv0u!ANV-qAgckdA;R-B|&gyW;3iohP=OgY?oR=bd^O3ah?<d=PU-Y*%=|6#9Ar%Xy86O2_A81Qh4mc7!3L#889aF>?D&Wn~k6`J&Mm#X>{YHinUyb3e6A^ZO*EPx@2T6S?iG(7VZhk4k>`*vJc*evEcRNr@uB!Zn>Oo74Y7pWsEeHVpMrmTi+8;`f8y&?>VxW2#(DIA%4Y$nLVf!iuprra=G+_%JQML24T4$O#CR2i>MnS`@9%=?X=Itj`S5fLE`M(7N+&IM*Hd^!ZOeVI&W$=&6cmjSW(Cw@=~``LTFL$&{cZkN8vrkNbfU_BdS|=EymT#bV)NZ1g#tU9Op;;2yi060l@ME<sUpEG=O@m)oituV5FsdkhV6oIkdLNY(#sbfoIMgf}_$~klMhxw6-d-6Ydr&<K=Mi$k6P<HE2Pjyr5?Vj(Qv~v2fuAXx-cS?DgV8?L#~x7E~K_!RWBr+}KA&0S0WdwpZiuP*r;RBqicTW8D@puUM&w3^gl<ePSq>?du1Q8k<Smr9rnFo=$@#H^IRZgc0Hhf9%HndJB<3JXdF06><=xefJWF0@Er&Wx;UbvoV1W8`?R;DbtQc(aksv_iJ1Gbr=Mw4I>A567}6=YUIf}jn;vpf+(22uo+|BxO>l$O=X~`e#1ksk@by>3Oif~lYDU^o?is`f_UfXR~`ttf4tOVL-qe!k|W+040ZOkvRHor{amXtb%lCv#8)xn1hMa*#Hd6|O6owC_2{_|&Q9QV4SLjg^aydh!mx*#%%icd(Z*MdonNiy<F&!46!XPpUlP&#9XLoIg1mtqkB%RygHQe5>*r2Klmh~l(!(aH5!;Cq<C{DMc}-!QIpHP=5^R-U0!@&aE}#v{$sk@j5spD`TpngEgdHVE>_*O#iDI|SA$-q)_H@sVdB!>HP(a8q`!&2MPA~h!rvzlT*#5D^p&t&b*VGa;v<ZUlp2GnF7xZt=*$TQ3F?3Xph@j$e<ZwAm7(Q_VBNxx11lX9(7!5Fnp_(lfp5G@sMP#&9@P=MSnJNqeWA-o!1COOl*j5O|s_3D9TIi6;u8T8t!J794D`z_3!CcN=>IuK7sB~@DB+zZD{>2gs+<VPd<HyY%%R_Kg8Vsv`{~*?g<7vjf$bL8a)`b1Zp~n|0uyTD83Tv`RWX`>?zJSJHcHC2QvTs*8+=8yz{&r)>-)q(2zpdsE+je^|9gfCxeDnP4zfRK*xH@R5tv0;FFHo6$EX8_>YtVMh^9;a?ihqG_N6_8-+<kE$;Gsz6#vD@P74N5f2|MJu$KeslUU{wcj|M22PtQ8XBIV3K4vwTokHbHMUNq(o#C8h0)|fLey@;ehk8AOWK1-vGjnTjf@6>DU#_g@T-`=jlnSQT<fq)IT*6~!QEBVp_D}v872^44V6c5|8$*z~>(O+3#@AbO#vOMHtv$8xe$>X;<=heR;<|N~1U`D8Ozg&qYteJ_>UMzB+vSNt5MzDJ6nHq{4l3`X88T*`4^D%#I6Zx<HA1V~V;R$Z(%LFqeBCgo@bZjZD4j0AY{Lf@gxt21H>C&?ES_;5Rf)+<aap(aI+V9nX5CH#6{Ljb4X!e3QP|~tWUL-NABz?QW_7r48ALCa7)ZhhE7U5|9TsBMc!!w$(oh^x(nyjd~e~Hfq4F?hj{Ab)A$iauo6sDj5XXwyVjkWUyl<*5JZ{iEf_`l(FnDwyG>xRqUa5_wJdDW5*W7ch^!<55ru)bREFRiTgf~8Wo94xKp*Skw=>%<VWR_rWyubK`6M?$^~+YpmtP(xlKSDtiz%luUjU4$X2eNJx>91TYka)?sWpI~xxK#v&@&KFLK(RMyF#eNx+4RJvrK$RtC=ARkW(2^FWB*s$>C3QwZOj+D1e^5izQ~nb!Eq-ldqrSUWLwZPE2UC_pxWY{zOsWSb2QGdRca@}V@Gu!CViuA8R#n<H<G;jB$5!5Z7|PzDPmwh;sVQQ@n>Y`GN1>x&>KtAc4wdg(K*;%iO?Sz!-!5|<(Si}0IsZaPprZA9W1`3eV<_XUwf?DqZ51ZWo4%Ti*ps7&pv;{{sY=uoUs!qZE95ROmRHaO`yiM+;K$$<^T}~|JPP1~mN1S-i^!%dMfSERUMm6^^CG8b#gC(edG5wOs0-V6BRN5lQCSR1=5g;U(KvW6bj+8D|595TJ+u~gbQX;9`l(}RX-!hr0eX)2^~Ck^>1Cn2U&LIc7NZ{V^0{mbyImXC$kup#OjL?`w{$x}^va^El>7kFq1SzYp~XGoccR%ADa@{2Nh=Hg0j*@;$RLcG7qiJ~EnGDFzlHpnU8o}-o`l%BEQvXcMQefV7duZldMTjzB?k{gr!x+Q-3K|_u6xry@RO73!8pSI)12Vrqyr?rcQIW9!2z@aBLDX|CB$ev*Y%xndfR>2Go0RbFL}c09fMke4#u2t`#ay?Y5s60K}L;R1uooYCv@8u9unLqS>5GzauR<Jmv{Baxe_qO!&=b5!tqziSM(ncxaP&fYT})vYv&{~qYHN@kES)W$L8~$7xl(sKT(;@MiGxRkb|{E6Jfhoqp+S-pU|vI(Vb13CE?_RDN>1=dT__Wlc={<v_P~NPk3z0yIMd6+6V45&`O4xBJJ!Fzmm10KhRY-p1Z-Z-Ewe1wTe|9f8G~Lrh=M2n6ZKt0Hp|;6@!6s$sWhQX3_nf9D$S<5q{Qq)8Xi5y<FIK-^<&AbI`6VRw2tOhXg72R{zXBr*C4Njc?(|+EC57&D-sI>psFq)LM7z9G(%6l$NH4#EPYPnBiVB33OT_%8JZFkwF6n9l0X)@Bo?&-gNXEy=Qzouj2UKk9|wE`yfW?Pq;ALt@{nTu4QWIb_<oX(})Int`SFof-^PnfX;KiA}Sw61TpKML<=$&2jEJ;6ZX%Fs(4x0s}oB@y<l;}tA_|UmsgSY;7M?bK6KAqW_CaJ5Z48+{3xg*sxZDN=llQBjpbHwQ>mR`;9^a>BMkJ{*0Zfl2{<cOFCo@g-S$!ZXNr8D=ev&?0^=_HOM3u$*@uJuZoO5t=yB#kOj8ndCz;`6RAt+p^12S#W>12=aaWvjQ9<=OXre$DpA&$3hX6cqF5nvOIUTs|;RKxx2Z(ZyG0$RPv2~arF79HCjX%j59hd(8`?yo#Djbh4i2ENC7s_NA;$A77q|*6-YQ0d}V$RU)TxaMw#TlA(dc=>eiv3O?y*AQcCL^oM#hc|~>P-N;P%g?P7E#<MM_`uL4^Nw3D(vQ*5Dc}5a>?D92&jgKn=J%Ar?dTjeMfg^MwMCH!8iD0`c>38Z|}ycj<$*jOsu8_8ECE2;|r|KVp}_lVmQO(I4fO!cBF)@SxIkX88@7f9WW+Ce99vC+?)KHJs1GQ(tET|jTv*~Dd1WAiM~$LGp*$wJF7QW>Al&rdu@k6d8NA?mR5r~4gvMpY}lv!t7vny*v-kp&k)-m;8()BM}SXp9&Tv@G)#pR-lfhB4I+oR96lA;q|aafjOo_|+MR)l^=6I9z0PW(uu?Ank1!^ib(_ZIV5Psjw!FN)v=SEkOQq%2^3qzd)L&XJ6@yN(-z_W`)~<}Lw3;jB@-(e6w~3tApTWn!(EQ<aw($sd8qUA}>~}o1?B@B5i36qI#!e21?~otpBc4-%|Mk>>(tZ8wT$})&*2JsNB^#53b~SMnpR?D-XpX{df|Zq6hfnESt=`C!VsZn<srzQz$X6?=BWd3twzi2qvKm|F=>~B<24Pj6X2e)MF|x;yiyogq?UTZb5Fr(*uRMNOWcy9LaVSDF9t=*aa8GD#*IMrgd6`HrBO6pew{xRV!YKvic0zhntilYT5P8JACBUZ-j3*AW_V{(IiDMvU;H4nIjUODiTq1J4#v0=X9-1^o%^1z<2Ud4_Y&dgJo`7e_U@q^$VK%zR@FQ@4faYiTkE{%<mS2kz&SJn#G{&IRXnz^mtzx?B`GQBBAO5H($qZ%qI)0@RebG7vTfdLmit^HX8e7OPoKi3k(_VN7X%>|gL#vJ2QIzc%A8Bx@F8mJP1jE6*T#br_@_T~`DP{Cj$$W~Z@Ca2$tp}=AMQa<{yG=MT@7mh2z?<8(HWMOGPDffR$HC((G~qrJR4m|>Jc3e4CIlybFPwD8gHtw*Q<S1&-xXKXG4uQ`GFYkwrO2dfYYJwKh~a=n^v*EZTQ%en$MX-!`^Rlc&-PF?meVV`)2MH4>QzNu)O$atC_Th6GYOB-?NxR2WfudT!+gmP^JP*?;fi`P0Ogd?g7s28RzihASD87QhD}!1x4plEmo>OgbqA-Q`1ZmRMu)*QslG-KnrNTUmF8Tlbk*9Gy;g0fjf4vxqG}(BIV35A>*D-E-(%2f!PiShn?{|pg|-^7B5P=ppjaS8wL`tO?JhoaMH;$ZzUW5;&+;l{+2yO)ly-f$*1~+Fy?ga-q70T~!i(2iqIz)?a3m(XpyKmYjLfupy||0oXbeM;+QsVx5ruKBRZXak5E~u6*}RWI`J0HfR8ZdtNj`~9h`0r8w3_Yq#=ZK+_i0LMo12YzPR*QVtx%1V(|{s0!*le#(ms0}96v-_RPoCutnbq?TsFhU+PAkix0@7aq<%}AHR1D+3yE%CFnt^M(2R^Z2yTisLcrm27WG>)r_MQ>guS;E^zEZi+@vQSrD<V;yg*oT=uBf~B;Ed?yu~?ltR%W{^~pH0lxQv`-3)hWzB&#uU)}QM0UU(-xsC~<@dNDA`L3haZz)B{Z)-`VMGAm((E4(5k(Z^&ww6#{qy!LX)|Xf3ybSw98jY%yw;6LKj^{b>3Czu&6<n}o9L6e@nb@dt_1Skpk+B(BsHA%wo}IX4XgWBI@zA@bc=aEiunF6&NM!^{VVX=g7DK57c*?R$W<{UtE~|_LutRsRAH%QX?+o9c+03kJClb2*l&NUduLnJRJ9;Rm3n7F~Q{lTBhMEfHO_><#KPkV1Ue?}aU4Co}N;QxT7#c!Ww2Gt5D|p+a#xf`eyRs>qRJy*=q3s-<h3sWPq%HY2`NVZI=Bn`NZZ&r5wY&9XO!eHAiYiw%wpKAoE$+4%ca<9u23ZWYhi^~hgyA^pnD+Q1d$v%OPizpftwXsY4uzl1T*E)V>e^<GJ?ZvDPK|sUhENQizYVXz;egKFC}-7?u?}($jt9esc)ce9O=O2EWWgoEVL6x-rjn&A47*K<pdHQ=EFwgOBcA9QpU$+#D`X_Yk7`8QjN?Akq?JyD*vNPS{448eP9m)o>WTbK#1pAsjUc0BHFE3WVBeDd7?}1f##KvQgPlqAZK41v>!2HFCqum$+LY&-*qA9F4`-DP()MUXyysUb9$uSSJd=M3SCLz56Hsb>QwCxni^t9;ODfRKTaEi<6J>0u+I|}?ove1%_V!wh+mst*teS)XFufSDU+w1oRdMT*tI~_VwjdbMs%&hkFe2**TaDvVC}D}X<qJR0q?$A@5Uv8-6_T#@YSa?qNFfBL_;>0+z`O?(>vlRWX&w@y@5!llM9KC=6yoOx(ztP$$YGaEAhM*Bw31FlNzx#aEUE{>q&rpOSV~?wC=4`_c!~Mn6KwE!{T8l6A_;MC;uz+n!&Eel1VTbsne!b<D>aK10&)|&Kl~LOs6P%Z0!a#%;6Q#p#6X<K6M<~uR?uOAKkHvwH=b~`$>z(VwFpQt1bkA7KeRAPcD%TjR`O`6gfn&&vl|yj4lj5-l)my*(g}w@DxM;4Q1nB8G@6EEkxnlj%BEJXn2&;CaC|}3O}Zlyl{qOcD6`IQ+=;>uEPbM4rsPMK*11SlOA}q<EycqkQzTE}7KaHf`rG0++rdE7=j&Jn$_N;}&x_siI{zSE<g89H3-T3lUY$*FniSd<Qc~@<n&?1c!Mkw}Q&o5w(3@iJ)pftwQnyG79I-TJI2uZ3B-R{2k7=~&x=HZZ!uSmW{VwG|L6PX}-Ky>G;&oJPB=}Yj9f3FD$UuPY#s-`l?gFXQQmryNVs9|!%c5-_7bCAK9B@mxKvJ;B4;l04v{OdZB90lqT%?o6eAm;D0P{j)W#%FL&twbYet5I{<ZVt!FWbHrC}AlDn7W!$W)NVkAmP3i3DjM6L>Uz$#e~U@J>3bYQd|srPT@+CuPWjz#kKMQWy<^+{VLMuV>0lWlZ|dw7#PB7uw-doUr}F4*Vs|=RU_X!9og!yh?f&<Pd%&0!Q<+a#7dW!ZsMU6ro-kP*j5eLZtabG%_apF-f#OG`>hs&1flu6y|=&ljz9`(v*X($H$~6k76Hl|Z`e(`MWC<$_B&|2sO>a<T#E!T6N&BP4%cwb(u!V&qE(C>wqNGHw)eNO6RkSl$niHAr?%c)j`wYVv>}}_rTh}5`U$us47h{0_THgR>@ipF{tn|MVU9L*D0-&3nO0~S$2x*sYVXu`+xMC=*d>&8Tg?p~8R!YZ5v3If=eE1~&Q5I`Ww9I$M37<>BR1`+;F;U4dM&l6Rei4NoBJCoW4gAzyWfT}B!wKt6&`{rHfp=IjRtfVC-eJz+fDb9kbV+b7&2Cbq821h(daQ>7v@HR3A~K#y!dLUI1ynj*MOV6Od2~m?J_Ca)C(zkjx!oyC)sqWWPDuI(d4g|Q%4Pi7Dy+-ZM2xU*l=3Zpk&G_-@KtA@fbZzGeopn6yy+(r?kC6hdW;PC#;4h4uF`HV!(1s?H0ft#7GE-X=pp3Xd#LP#y^_Ol)lV}=n~Op6duDT+h+YvW2Yf^n`RJa&|&<(RsX?0!vDHTb1;W~rUv0pE{4+wAt+TohJFaZgY1mMB%wIyzozO9G29ADLY}blpucaAGn7G2t1@~<=pU0?&J>#K2u(>q;tF$wCnh+iA|zGL)JcY#Zs|yRn)Qe;*+OwK@8>GC%8b3ec4Sk2b-7zw=`PQSY*LTSCA9k;C&OAIn>Zfi^>5+NF9o}?$b2Lv1(SY>OZW>dEBW6<y7N;b;W44dNKWM_+60ss54k*Fau5sCjuEr8NTU6fCuTbt(c&@p9MK?~Ovb7Oe|h~OWi~5Dv%iPOKM>4>&n4&e>tEeU$;Xy7Xd104W?n<XBD%t7ibz{=oqcNMQOChdM)D9d+NUCI^phw_{PjP)>7?PpTCvdYcD~^_uVUVoG;Guz<IwA_uXo|crRBA7WvSFJb(T8oL4RqjTVCn+ORLM}uz$^@VHNGQ8;+Nvgi|MqZdTx!e1$?YeK5R`0EUX7R5%0|4~xo`Di$_K<`e&nLbu+hr1qDbY46R7i;9Rc^>qB_lz9G8%95U)P+Z$=kx1HU@3r<fP(PFx!o*n^$EX$|)2St-ICIWDc3`BwM@(&KG_gBxgghr7o-oZTb99!e1~Tp~mw6-LaAMSY<0%pPWI6#2&ZZb+lR*6y)zJ}<<qK2L8~cRPOCJhuDxaLZND41t+>Pmp$e!4Dbi5nhxdmHXbmuiB==Cx&S8jI3a6e&7COd}2mss;e*?&==ZfsiexYB1jm>$rI&>RYfRb?lV#!yk44QG3Sq9cdIAtT<%;>^+_^#o@UwZb7x_Zf4~o3`}SN*B$KtvUU{dDx3&iUfnNIN>=4EP=eaziI+SiCA_ArVRrV6ik&fIqPT%^m-fubR;}C^)qaG{YQ8a!h4*qp5fE}PlEq2+d%lJ$_->P2L2c~3MlFq`dRE-6R<W`Y$xbdp=>S`{#A*VV<Xv8rqK=qK?5086+T*|o=4?{EgK0`X`W%eP%^PiS!8fNDoZnn?e3K@jSRIiu-Q~b)M>%5$x>Xxhi%fKWc;b;)J5MPN?5h@WWU)xn44=6$IYe%n0BRwOM~sQs)iGvDEPEHd~H7nFC>hPE!PTPk&&X>TN3@0eW^&hgAWeF$8iK%SZ&#vDPqYxaWY$MPP$n#Kg~spwZFT6d#lk#Q*<ojlI<N^^}Dq#cud->Iq0tK&dTaTC6EyE`ax$*02~S<&v--|Uw`s&029^|Zk+TT*ZzKEch}f*;+~b<c=^kig2JgGB7KeWe5<Co#_35-S~Ibyc$KA~%3@_{`Aq34M)oG=;%f#af`u}I-j1?fR-=rV$Y;PUu(xhj6Rxlh?d{UstJOHbXyg$zoV=d!^C=u>`d#_{Y%83f?;@=YKHk`YF8b|zHL{Btg<E$UVkiqk{~|?HtMl3;;)@~&8pd3@m89{&^g=7uL-Rw&TA$*+O0j-x3f@&}VYw~LJo`J_HQspXJ>|&!(edbrdV9o}cmbOw=`UY1PD)#=9poj|h4&^#&r-M=y6hVtJNBB9+N(u599nH!C|1O1mW%5A7{axX^+kOrr8&fO6|qi1MJi!dgqUHv?BL|&OvIlt%~a^fou-nM`j}WSCNmj{hu(+`f#VP}0pa^_Ds5MI&%&%)5L8mBBz-ftpr0<@x9qp7l;}(K#ed8tcIx6DH7LswWuaxTl1O?LVm5{nHA7}z>GCQ}sp<C(tE<p*2TSW4YiXiu5yX#k9CRN}YyoR@7qMT*mBWf5coi6t<;NVXA6UerX>c6J)J-Q-vJ?~LLd-17_Ucf;Fr#`w;p@07ui|W_r6F+4jm5Df@tqx&W#V8BGC?{RI%mV+q%$}=8=XxYYI(|#-mN!%DfprRjsKG4^AtuwXhF~u!kWO8V4;h#>+KGl^$q2tQf@QXzHm}|E$si?X$lI4eE>s#eBq#o8B0=ig|E>KK!={#{lwTCMp0q^li_%sjVBJeVgSd7k4zwgFa|8HB3vE>3P~Kb#RbGe(iE@3UfWT;?Wz!*`m1z_n1<a4LptUpDFH}B3e<%I-kQd23<BRV2I3B4Tp0*`L$*M_WP9<~`20$~jJ=IbyaD?kzsW@j1&J!-jkevL?6T2#l?OUqjhq-anMG7tqxj|~<<4L}9^9plQD299Snntl{x~_B@Z9a%7RIv2SVi3XO}S(kl^ov3tT-xB$0XNFi-||GK(xb&#1znvO*_y8kw~a$+-ie(D-9W8mKx<>9fc#b)iW~~jo9{9d6nMfTz>OCQ!^XWh1eBA%`8de<*(89#T|>j|8c$5L}bJ*3}D#d@9xxYBa$yJov4z_G<BvQpK=XNGeI|CF58Ed%->8#!(8v|<aCmG<b^{-ANK<&Y{0iaPo$Do1)I8_q58<nU_OqKA^XSz$^FoKB-g$!wq&^a<>5@gI^{$}1AR=HV4Y1b^uQv-;Hrb}sP~*`YwK8+ag94!<UDD@aPbD@0{Th0b+dOaLpDT{$9M~iY|v{vCWGZpzt}Cz!(djA&2>!v*Btq6W<W2*9F%kN@}En`$tTjuXvJ0jW87Es3HfSX2vBG8(tRd;b05C?i}cf>>n54x&zQmn<U{1hL4aT#W#G9$VP+4`C*&HGPDD!@Keoc^T+BJIeh1w|pVIr}opcZ(O8^yrCY+&!RDc-BgopfQ9gIQPE3K3}<^K^5Mze0y!I)nw<%{8JZ)v?7tS^=Nt1C-kxm#EY`kmF)wPJpyyIlUd4#u>l8hM4`&zrGDMPI?J5SBO1E;-g;MmQ0l!JkhV8=F@f0d8N-3u)A8K^gIQdx&)thcu2Yz<>YQ?@SLqe34v?>7K{I32JQ)|8qR(<m4(rh1GwwTJ+@?_L#;ZiqRIRcpHt|jV(~G7g1MFFW@|dFHetyDdJ1z_zPb=oe>rqehbwtQyY@uq$ei>P}gMoY;zm5`-CdoIUaTJv_Ku97oPirVShw+C)4QU%IAkxQXT_2))!0#%DR<}DQoDfBHIlfZA7sEM}KphP!X{#L|Wr>U0^cAffJ4LD1nX8-bi27G~76hT!OJq5}g|5#VF>ul^6=`?=|;8MfYON2OI#_*x9WANI-4DT^02(%+N^`{gX365Ho{VT@k+|g0fr{c13F4lhcPsm5i|pb8?NqPn=+2{CW!abYhsn9L@2dLjrQ5%y4JJheJ3p-Rz8jZ^}7&ctqZ|?4?yhBafV@ioJ3~S&lfQa3>AWK-W9#KJ0byn~MP5=q^FrJseDzjzM!B@}KmILg<{u_^IQc;#FiK(5n&hl+ZNJ@nZ)!Q-E-(=f{)t$76KiyPPG5T<i(=-*kLYv25W3a~cj0WM2=f_PLP1$A&dLNArJdhIaYUoze6TtmGbDqp#QqMuFS`*@}9#n`XL0^zy+J<k-@v@5ppL&JPW`51eQp5J%=Mfpv@tKhkia2zeyXNzN2~wRZE4%`9)G>=$V-SX{aqta02g3yQJ?zt%ZZvbW%BaHM{UZB#F9lO_tMWG-m(bR_W`Gw`j))JIjfI|`xPM5Zd%Bi_R33ABQ6><84I_0FbUI$Y+Jf&)x!D(p^)VneuDY}Sds#Y&}D6F%l}9KKg2H<x%pXZlGrS&V+tpGggmQFdlk0%+`4Oae{e>y&w+`WsPhs1WzUFL$CPgnCPb>?}%3=E%|eX-3k>FBR?k49|ig$$7S1xwfTr?u59={Rq-WP~AlHWp(;UOLoBV%^7#M+1SC|NqEB2uyh~3tKF@0Dv&C09@=Xp{gR|PQA&YXL)MIvHTI|;`&#r-2nggECiju&6JPFPTsdML&AiK6rP$Cv9YMiJp@$nLf)9;QZ>k&<tG6|`*!0Alw`LjAk2|-4Fn!|@nvMys2f;aFZOi=awOW_GV)JzIy&Di<y`l;kOq}2tR!;9i1>~Sy#ti=HgV344Ea?)R?Y~RJv-U8&n0Op2d~bAo;sm|FIh#!Jf^R^*Z9E80PVwDIc+4m=&h|ds*Wj>mPJ;8n$>7}(nU(2`#U{Z6bfJ6vARIcI_4^K7rYI;G(elrRia0EGXt37Su5U5!j7bziN2Pg~NF^>w2|^EBU?qx3<$&<hhjAHCzvbYkxuTv6(PdFeVcH`L6)IE+4h>T65mGJtT^(mT`Z;@;h<lW(8in_G1pG4^d%hIC9<?azp=DE`C}3Gf<I&k^=OS|u0W$ELuBY>782$Db*+RriSdlC!DYc4qsT#zwVZKaV4UUem{xCo-P?(sqIgCP^5*fsKN<5^Z+vDo8v*72(!SG>KJeb1>KJhuKSQ3dmget43Q(|_hE%dNMtaeynx_jJ5%Ymwor<OlcHROQNNdU5`WiW&~eU{aUCLcyID)_4q9}K^4#mhe?Z*0}5n8s0GVp>t4R#Hl6ePgSJ*P+`r*rbh}x@+SYd@vfoW!lxB-^J@~dymp4P(x)r^j$ay8$UMGRBcYNJ0fi)HD*<~jjG}~8eaKQJj^ZYe0L#QU_oagOW{30rSQk=7R2L%qdH)>8^}eH`Jf7=hgHy|if*P20}qA7$E%ViBVJLlZgS4~syyS8hiZr_3&jHw78y%cL$R6FBy+7rh2Bz2M2-NiEtdM$#o%|(#<O>js>Zr|qP_=e7eSHfn+SfKvxh9FRVwHU3(<;{!#ZG!63O;abcxSVQ+G*lo60VK6{)(eIn#<HmZ>36r+8QkjZ_UxTJ`F<=T!@D@XTqIP_ZDoO-)HF8wWkDd`<jV%RmX_xhQ2>V&F5#<*WI5NGEeEQZ~Y)YC6+ujmqN@!gxa2T5%5$<?uO>T9Dw|WP5|=<drTrc^ah_usD)_S_w#FbbQ8kM7RE<-K}P;w%2UEW3pL*%;gTF+}`vz>oQN)jb91k8d&UJaP9{~<T_wT9<@3cQ83oVSlV5p?1jmi8X5=F5Fv1a<0VjBW%vmPnjn1DV?kCtL?;(pICxlZPT`E6pmutbVe$}82^Cf@!>WZcT0@er4#y!%IY6u)6|9x5oNL5ciI^@lIT5n%BtyE4P<$?J;8}^>mXw#1h&B<}i`JNU8VXU2xOy@QG33NEC2TCiR0gSyP>wbIrK*q70l#oKON2&4ZIPieZzKT{75!=597)_L{R>Zri=mHt5D2}Y4TS8(%&1I<?9sf;5Ga{e8c2n{kSUW~0`SBx7+ggUsi5d&rg1I?OO+)XDS=4Meq13<rBo^UE|HEwn9Rb5x>K+1?YB&TD0{hPcmm%YW{EUqmN6k~;_<RPAK7Yk<|NsLo3^n{=@!+g!9GQ)H4o!41-2ry-lD4EfxezSJv@0qvDijGn(jcS3}tbfmpKDuzZKM?N+xnZd=$GWB4wP(0BP(l=-;GFoI&Eratq|Ikb4FKz=D$NB$##|fJ|jW3}Pv6-Z{|!(Divtx#vF<*(<R8&yJ^)YD&O4s(dLwEml4`f=3UlGeHRA_+TN-CmR6R*vWmJllzm%KM=1atwWkPolE2tbuGD6UF;klsF<%{56P1Kx4ewKq<y5!c$2fvWD2^0j6<5&aiXA^*X<cKi<RvoN&xNQ#j2H=AJ?qe9HuA%N1JxboAKnMBs~+()I!)57NKLqd(d>ssDe~n2y3Bm!2pFfneb!9!Jxsm6{F?8RNQ2gkRAYZmtw)_QYsdn3T@SfR~RYGqHq_vwkaD$#9kT(9lF1vG?`FjIXhwk&^$cMfKP20^hBS<%?7rECPs}lkKqd6s_)zd{SnS^fj-?fLlAU^r3r@uXs-P#2FlAg^vI(xzfN`r1A`aNYl&wqBx7h-jnKZZJD^BOzc=choe+c=x7I#7)#|Ffk#;#LWdBft@?tP<%zZDC@RVev^gD6+vjrWzT`evz&nm>A&y%KXc&l78gC<}J7_7L^EtTqPskd6SqI}p#LmNO-k9pf(iY?k?I`Wlj?F%5nffO>S70jF+^HN~f2}St5(IQRV#Cz$3V1j3XOuR5#B=w4vmN49!rhd#jtT`9eXb#jJoJ~h?jKB)UsNaZ70^KwI*-*s<lY>OGP+GHXnG!Y<PI?JFy|rB$wN3wiQ+sizwq0*0TXqyO%>v_Xo?rMK7`zi$U!y*3IE+&fq4;K_a7+wTALi#uWe2rCpHl(<q81;6>2T6g#xXKqsql0)v=j&(S#T1~jT;`UZM9kSV)ieQubg6vxq`jhvs*q7v$i#jJBm9zCG_$mbx_mcJMpDrABjWFlgQG*676(p2R<tL;Se;A5cV1-(+a68(T~iSYI7i-xa1wKJj75R&Y_^0c$B@=gQ2a*l8lHx#8ZumSp<hs@*LWW6keT#i5e5|4<0VuB<WsybhrqpVy<XVhX0@@v%V^hd&Bh{0UUv8+4H>PI2SzZhgE9}=Q*##h-Ao*2}a6Zk$4k^FkGZiQ*)hYyz$CipFCmH^<ANLrv$!(vLQ1883@gjFjltPH(#okG}Lr_KE7iu`^0l`9+$+{V!Vsdajdeq1|}3RM=2-nwPVw}>!p=^uQZPiP(3y?HvKzJq44VCoQ(PX_xJzNp)hU6tRl3YKTBAr*FVKb@+jUm5uwhYo)sdDd;PwA?B(lUyn6oXV~z=D@E{8Pef7D($@-F^KM6&O-}(X%swphO;FNxU`TBhZ?7~Rm*T3Ni1T5||f#UUo0E9lr&|k(Cm6g7H^`!xA^$CUjz8{66BWMwZGD}2K(8OZl!f!U5yii!{Vc7OJ8%|y-uE_7kAzW5M%0_yOobVh1&ObWeUS004l*3}L9IULCI;GW45#z>}*So#t-pa~aD2qncxz*l!m|rgz*4Ij_z2e&P+HyBsE3S8ga&NV_96h;O?60pcm-?j={Qp|NyB_q{i(xrmEUgy9m6hJN`N2a=z`Hp*X25&7Tv&rPt}f;CowcP>r@XQhuH;vi){0#ywBKJV_j_MA;5~{o$G=*&?WK0@PTj$S#HnxZHZbJ~K1JDS2F>9g1&EcxI&&y^{)Jdv@RdVZ7EfUr{sLCrKRB;{Lrd=qSlJ)uoL9f)g{lJAMTEM5UPEx)FNuEeml1&OXObcD^+zV>;_o9=UeCbo<z4xL(k0%PAim}N|3}@sH@9(Q`GWuNrzm&FOaW361SwJv(C{ooLK1KDWs*|c!{%0iK#_z60x$}cDAU~VW7^#l-P4}-^z6*eqkDSp-PnJ2B52uaOR{7~-1`NjuP~ov&nvSstFk~)w&TWiM^hw_?>u?(oRjDLjz<puU)u0ooVD@u7hk{pje**0g@%0k@;4H9;ZiNdGk&ibc=0}*jE-RnM7JYM-Oy?8i0iz!27hKMlUpzacK0*iv8ynC*>7dPWUt{;zcBz+Z79z8GhzST)F=#RU=rTr(!pV8*hVQzjf_4ZO)ehBu9=PsTJSV>njqpmm5`oKDQba;;#)i8Dn_~(i7h{U@hq_UDzQtXvtI=!w?x(m9iE+I@(wJ-T5XYB)<Ndn2t53RavO&mo9ngBnpee><C*4LqY_<)P$U&Dmjo+#X0$@8P|AH?fh%}vFZxt91&|XOPWs|DCzVYRc0_f4ht1ZsVJ*}3$?iiQFcr+5aTpJy!NdS4u%}+iy|}&Co!ITP`|%u^(R%~T;r76@->cINC)z{52Lk#7Y#)~>YIYjvz-PREqldLZ0~I|64e_{znP!0;L1XIRj0YfS4k<V<miglDTwh(?#Jyfa@hy`n*~iI*IGLR4zy+7Ziso)$ABa)q{_DWG=-|ZAmn`alh=+(@efCfkB+`4aIn~U38l65Go;-fS#}O>9+^DaU?o?Y^OU01T3N-GEK4{=UgbXgPW$&;-`Q5aE#v=#pR7>joE7b~Tf@f{wdXKDuy>8p%b_m++-NQzQQ4VB=jBnA7aRsxdq1WVWO4w=*?iwWQId9mA?@sbwlS3R7nDc7L((su!f_23YfF9H~yF+0u^@vUvrg=duYE0Iy@~|qP3uM8COsQg$dSWcdc;a^7(8cgE{RG(&*xVxy38ZV1<h(Uu%~`?F&&!Of5`qBl$r)>^I7n2TMjvzFRXt2E;!S!w!3h3NSAu3pz`(n4bO6K|?mRH*MqML*Uh`N0{b3L8w%UUN{}Hn{KV#wHV^E48QwyLz_F-(P*3f7a#)rGR?W2NU8XolZ09^G;hh5ZR3d&%e?aGo4eh(XBrc8d~QLx@K6jQu~!{*(E>mJ=ySl}t%MPK4A<h}eG`t*q*xmbCXq3Qn$U!brfnVwlVaDv*VV^8ru1eps%_fRPS!QvNuM}A|(_9dcvlMaPb?2r`6GvV$aio!wTAVaoFM2KyngpAY5=NR?`Ssj-t-D?jhA;>{=0301tw5f8%Y(P-f^3&W?1nW=658}z)c#<dJyB$>vP+=0I=CQZZLF33TmWYbnSfbQL1Ifr$5q-Wx%!3NU#*nUlAs=E0TY0St4M5EiK6}l>0TQ^Zk=?0$XLmfplh9moU}^>ZcYsdgg>pcBz`%;$gzDJU9$YJ_g`1QSb=n7Q3Ln^cfOe8GmohW6+~&@nr>yMj-t>hFm&>oVu&R5;+&WkH<1I6eFn5I|60NYhL2MobMzovJ@2+J;(_ZSTh=zxQZZeEq25$~Db&PYawN$m&y=&#NcLCRWy6n8LmT4vVzS>n6Rw@*pcp_+OzxIyp_9waBQhE2pTW()RrZwWaVq!Vi!BvN4o}HQwX!)nhGgq0i3#s8x^1zEQgL3K8B`aH&HEq_ZO09N$mx0(Qlx*R8xVgMmES7NDkm7hHE|tf=x=A&Bgr*VAz&YZcb=gPo>$G<hW)6%T5ePn)lrUmc`I5rLL632-issJEUbhP(Y~~cg=df59{h1WWlqgAV)WYu1ezZ~F^!RM>YFGVZYw1sxulqmy(TFyDY+<;M=P0gD`Dc|>?{jbl5t)c`7Y{(Ds1jmPy1LLkv@&uvFx%+8s6f_e2WA(2Zc1x?%m>zOxGNa_VtMC)s3KXf3mM!J{3<}F=DnzkE@le51V7J_DgIx)_VLPp@ywPT2*~>}+b_@pBA^dS_&~Vt56RT=i0b3lPuO(wCEWiY19<=AA53NV*S5g%8CxoB2~x})*a-JcfE0{*|C@xQPZMH&<dh`HN6IEvv?H{3vJpcw;X5EMBH`|6YWdlZ=zp25;g=i_GodZ6z%Sxw<B8|DNA8xGozBxnaQ!Pq39-tQZTQCe>as`Ifk=ymzi`|Lpx@Tlz3Xp(KmGC~1CNu?CK7sayI5-Nj^^38WF_)o`wn^q?x;l|1|$R<SMu^nsWGs<7WPhJ!gyfCe9MP&$|Dgj#+Pj+x%=C4b=z$h-9{rUsP7*=T!Z>2ZEJKz?jqgp$sArkBD1S+HQ&SQ$qztdK8*KADg>XAMUYT2Je!%b@QpTh_0i4s)!S=GJ@j1gjj2koynaem9zo$#Wst}7{wFdoKCv=}*+==0^umzj)v(Nx70g+|sRsB8Qke$?|K%&;9gmc<Vswhb=5R*jB}mD3*div9D(KUQvBkxdnUi%_P8sqMEeK3FLyKaRRp=*T(K}5Hr!tpPE-?~8h4H%`pduId5YBP_gDYr$_&KL>b_$Biiz@}rd|0eh7FKUxU#fex4Ua#{sZx6@ze)%qkGop!&dq|T-LTqb%|sb@*W}&00Clb-<kKeBA;6HqVK*LC#ol1EdW$TBO$_gdT5<l;%&*p#>KpU*!hCIGvmiOc2hkqd{-BvVwe|U1we@j)d#M3763CvnTd&fD-d3}6v50?(i82IXGk@p)mz+^M<%V<it@qPg_4T@(8gFrf0+Ka3Q1!n^53jbeAjj)G733VAql!MT=dVMd+2hw@TjDt7HH-i5e8%8(DR!3Pd<?Y`c>0fFglYiAs<|WFvzC@|#WfYA@=oBvO^U7^5yet8IAU7l61PKoKzZ(~IId#gP34&oU`<dM%nC)`MQ_U#_{|jF&}~kHCsKDbu%%1;!U(8K(E6fbb6=9dg*)h&T%d=Nel*}et5cURU7c}C0_ph9WM7o#F)FguC|y-gZ6z`B{**p1=+!`@B`hCLs}wi)qk~33W^#N*1MXAx-;0N>Jd`0a^)L{$(LgCe!+TsXMv^p*o-7(dCqEE<6uPm<Y0t@mi8s3vniMn|J+mS%*(qN3*|)vZ93Vaa%-ui|q1{6=;Yp9T6(#yd2!VbU$Eb&A?;^{wR0lI|(|g;Q14S0Y;~Rs(-tnqcZwj^9cf3FIrpowdng6U-JzLTpG2PVT>*$NddYT<wbk~3Jch2|O<9ALbvx0c`wH&0~V{=U_!O2;Dz7wS8oqEm9uU#B#!T-#g=4BqSo_oy_sxv9WOU5LTVh?y6b6A8)kIhamj$8(4n}675i~PRnmT*sJ^bwYqDCPcx>FyH6C3Rpk28KL^5`X-7WD>#@9;YNh%k3GzbdXi))$hGf9=yi0R2Vu(TJ9cA$f~q=V{vJ-zU~!R1Y@-Tg%Dc+Nz{e!*1lE*yw~4IiS^&DVMChEjy@4gbtmpV#<S04Eh=<fZp-BkC!;X(uv3-RVgHr4n8!JSk$!J#Xnq&tEtUFDORTk}rTkL2YIhb_ZYI-w!<%v!e#K3(fL(I-g-u51-kMjo6@C{&v{a`EZ4wp|jJtxiE=~pBbl_bIyqUn8J<o#XSq)9z4;&0u8=oqzs{mt#x(^Dv$W&z;-T3v981I_cP40<P6aZfSk{ItPgB*V?rey8vr|fRfA&LRo{!=|x-JXoRF8-G6<<Ho?i?52rbMTZO&!3>1Z0b=S{mIA$*CP~U1Y7>4x-`tY=9?E!OAdQAISve>ey7nCWkS0jI7cflfY%#wRHt7-Ot1WqQo%Xg8}vq=;p`^XTMplna0$~p$Tr^!RgE+R8=hLB4vXbOt{c;4W&XT4(k9MjzOz-FEtZc-TlpZtU}pbn8K>oMm1i4Peyv28h9{}gco?diF_jbd4k}G<rc|1qd@31lZ*GH~JHDhd@R*Vnli4`1hGAq^7J-bKcqBQRpNnw*cnp{FFZIcIoJxy~>m}75(i7FMave<QyvV(8c>xxHriQ3-@|URO_TwtsG4v=x<3p5As#90Wm(uzubI7{kNT}S&xM`64IxTWK^vLOuf?d>ix~ozi=UGWy$GqOIl{9J2OS}iY3udH>G#_8^5Kt$@Ryfzw@P2Osp#fxQ%Cv$eMZ)>J>b+j|ZsL!rOLN|GuRDyq`Np8r^LoSmXmB5BJk@Tk(=i7GY=yMa2yVJA=zIfox}&f#Om$4_jdp&@2zAgH4%$a0*TSY>Md?<~^KL}eMn%*UEIo+gCn+vuZ?Vy$?QV(frHWcD+us#4Y?f-g?p?cf4X?LUOZa*2O_%s>6#T_5xtLM-tS1hx2tDnYE}yB+?=?C&Ei1(9*l+n2d@pXqjlrPtK%IdWMoCoRb8_0eJTn&Tg^!Mq(ELV(KqLjBxT1m+^U;N=yJy3c@=dxdDX!HSSX=-CfM31+#ywclC_>vw-0R$9w^>BJ1XfGVlpbLdAs!~s(oUm!x8!<)k4t;}>MmiNV``n+QMID8a2rGPi_Q#PCdM9ht)L?j#PNJ-xs3k17*C09dK3~5I?}FkYfcS=?jdrU<cTX##yvkk<REr!RDolBAHxtCnK3>*Fn}3T?FL=rT9jUj#Xx<6y``OxdWXa6vENk;H})%gm=Br3*Si5qr3V77fB;*B)s2CLcp6+J=UWFRac;KH(Me`s_TXq;C0|)|teMUCx;VaY0mcjiW6%H?Bmu<1nzX}C@4of-etU0Ue~^DZd{8~Z#t>6Hp_@?HPa3hNAHjwSFj{$3bK0L>r9`X%oow8ud|q|JA6}@h)mJc`N;toCdjpi=4L`Lcu+PW~WD|Mcw-Yio4u0CD1}MX-DYP<1T@6`N6<2T^1jhzWa=V#8LE4xux?n=9d|8(yZBGcF7m)PmQ84542E5@Qs+O~a@OpLXl4JXS?A6sv0(pW@U%0Tx*Wf+U*J5<}0xE7H4?zB-X9J}t`n|Y~J4ZF*QnvZK9@<M^f8E=2KWOYkd+jcnh#TF#s89~>2Gg^bbKM+4{Ed-2H@G;}4$wL6Za}|Z#L9tY;)k$Km!^wbl>qOloR<n3dbKcpp-`aqy*J+Qk}mc#trfjL_og&+N2b>uUM)^oAGrxZn}3P=fd^67|2vX{MF=@h2)wr!DHIj@pc)~t$M>Q^4?k$^#0AxK@r^3}sWuCS7fnn}ySy#?0vsvLPu_KvfhCq=(@+42#uk|-+TDANLAwEWG9FE|Vm09k1NjbP)nQq@|Ngm2F4-P9+g8=Dzg3&pr#IYKytxwAS8gxY*EvU@!-rNekLumjQ2?RDN$(`TO|bmNH_HSGY8V?k7zhe<VamJc?YD<<agu%|nWi)*4HFaBc|htLwdJ)X3~s(oNeI_hf4Y$iFS=7jY93ve+Ml3ai=#*y7zNU1v<LSOjA}ZF+wVoy!cMSf91$X32jyOvE=^38{uiR(!qi3S0k`ZTZP;j1o*Gq75#;jf%4R*BudOew+GKfSX>n~NJz~pgd2c?g>)={8QG^BZn_moosm48}E3lP(2Qy}1%n|!iJ7jm<MpC{DwlolAgxM_WVTN{7O`M;K$D&}nmt0!b7`rJ_-peCpV9!nzkzu?=eN_0C*x9Zk(Qc{sw~1~WNA0*erS4?+mv-x&Ua;TBuHQXu&>r#*1Y4!@^lVVNc4c<E2m%7-`*F+6h4SmwK3vTV`(OyrThz7|SGRMLvW_7@BX91sTc$qBP;xnbzZprep9q0`a5}xBAqVnj`|VsX*sK_chcI6V)W7C6sKFl)50|iWNqh4v>WjRJSqYMl`AME7DfRwEhM&*LboHxzOOiMuCblFqSr|gn?BH8gzBy)Vx)gZ8e7C&|-qdX1ZSDGd$L$l&IoeJR>!osWTZ}cJRcCB9*RC1+%PDC+?tM#XBHQ>9d8y2qR`gd|XEDDngZ`k3D9YF-lZbom{>$p(3WtWP__<FL-+YP|{{4&Jz4%x7`G1dau>6dIh+jN=`6-2?la-mh6#ireYgtC-M=yWN!RMcw7!dNHX_~B)xOF?98@-3J%eUIhmmhO}wZt{@BjtXI&7A${nerV%v)I{AlMP08g<GLyJVP}~$?Kh)&>JQ3fAn_bH*`kRw%$?Bk&Tj(Py;>K6t+3GWq+mG%WEicafCnYm(8=i;k<@jhR7*hW+&bA>I$#fA%)`X$ginV=vmef^m{7Bzq(5)_c$sie>h7fUU67d|23*CdBYi{9gkmJKJhTb()ur#$KXzBtcNddR~7w>+^^5)eW+LUTvg8RfoQJwE7Tgx3-Ju+KRUqT?!w-3f<0UH$~oSD<N7=hmk-#`;PCX5*OpiI^~20K=;xQ`(nsx&d=-}H_wVu`ocE6Y3a`B5lc@IOKCh#!Rq=e}YF0fj(K_m$!-3YU%?JJUH_r7Mp0iEaSMcN!Ht;SWc_|kIl2}Jk66NyYJFbYd$CQ<$8p~6XO?IO_1JyM<!m8+=hVw(YjCqPVZWtq70Sv!3o{aO&G&2juS}s=eygwx9FO?W5$Iq7qf~pz<|DBjINu19)bSV)cX(aW<c&#PLSadu3qw|d?JCOYPxm!|5zhOv_NGV0e>4g^{3bUr3#;a6TQ#eOmdcq3HnYf@uKYL4x=ky)k9qByGkUoNufS!ntQFhPx99FzzJS-=HcT9)p<ZQoTeBNQRH`;%M&L$FC!I1j&mwLr==uGq(gUqx5UD|DTI^9M=okt~G70p~g#xvGg!zVbVk=kAUF5uMlD|{@Po$G0L1UpOVfZ1o<5q0TV-<7`DyC}edHUv}gOjk5_frCb*pF;BYxrEAnE<S*!?oo{;Cn?jGyVdjzHOl;`M=rsiC&6l~3>vvkjKm*JZI61?N^&M~>1k|0OA6}KOC#Gbk|}ZV8Eizdn$lgyxv@sI!GJ#v9=AUN)+m)Ih!2kKl9V7+AxRfU_Dd!lz3XVqf$I60m6AG|0=SAv5Toj9Ymw3E0o<FhZcrmO;oueJj9fb1&`vbO1K*SUNv$Yu4%&U4VH1amz_CcPuUjK7j$UMymPiz8SzzH=`eOu7Ib*2lWK0Tp8goyn<gebSyiEAtg$u;x+yzd?V37k%HBSy1AjFMei0Ufanjq^gFvrqt;LYNXY5F-;F5^G^?WiNdSvsp*ei_m2rY2_bpUV@|^zZcL31|p^djtR7enqwh$7h&vGa0oPxDJO0d_tkdcqB$n5aPPxW}Kti(KMIW%o$hd$yjkeaCc#X49=#6%aZtFs7VK&?T9f#I7#FDjlq<^2gdw;G3W1#L1#&(5Q9$QcMWdKi(WjlnKZjqUI|#clp?h;7@#ZTUAk49@WySF8MWD5&M1L$ogEd&?Fq5SEzmtoZw<|~1Ya+|`q@V!WiT3{$Dx+aTs@$3GpgmWV#vmb9ozL^{K1LS=^eLECXdsez$bGIYk;vr&r?BT$sBW|#@d0GZt=``F$XU?E8t7KfWMW2QtZ<IL+=nA>PVk|_3~qRilDHj7thcUm%M0SJO<r{!I|x_s%M$WBh+bW9Kh1_)i6Sub|9N(s+!;Dd1q(D*>k#RKUV;VLLP*CliB;1AD-=%-Dk{to`{i8Ix|9fS_-`Uw9@>V2`x#kRLLUk{kf$@%eVNkDhSfq{RoJ!jF!=HwhUjo5EGQ#;?oIJt(B%RIXx^hGsHYyR2-PM(YyuN>p0Ca39h=GHg5#lUx6C5s~6#>8*jS!#6IXF>!VzGGVS~O(I9PVb8H;;T<Q!9rSfbhk#MH?*<v_0Rn8!!nwdar3wLTOn+T&h56bG|LT$62W;;2fe}x&m>MZsgvluZGBfo}=g<`tuh59CC)i7!SCrEih%Ip6eCF!WYd3*6rZHXF9kw)@oahyPu(qSJbnU=kL(E?4>*hDPJaJnqgvEb{5a{N_oK341;nW=*eB4p7y)tSW`GZhcJYP5RKB>JV21k#AjHB(ujm>aurP{v1MmV`*zYI)GlFe_kV-MtWmfc7p(=Wz%Z<UT(8x0Dv?&Us{y1RwCT5kNV>(hkK(CCO3)J*&Me&h>UK!JXt?Tbx`c-O0!$**?#urUG&dQHSHJza$CaPaIo~aXl2!rhE?!Cj~;Bw;<`=*Pfm1SA0*{qo2|)(E1v|DiaJo&941k;Q-O%DU61+sw5%io3&zTREU>8PXcC*-cpjo#5_`?ni%d!uzj4<^`IFtG<K&4+S9hU&!L;wRFFBRlg?<Ai&MqX{Xrkex`d4u8nfYKqG5vXNX|<+c5kn{@U};;8HNI(my@>Jn%s;RFDtL4J&LMly08|<SbSKFSL}SWyW%)~w~Yv!@46IcyXTKHBJEO)TG`;Rt24hNN<A|L7?iC$<g$@7>1+&6VmQ}PY8P~a#dC>Jl!VkA$AlK&7<!ZqZ0`^sF3q^HWC$kX__4P!9Q3+-9vu8bWN}mvTc-f0JwckOaEQ=@cGPKk{g8(6x>p9Nh4{p5*+g$Ir5qQMW~%vb6lib5jF^QPkX_LKOcisvq0OKh+V}||EsUVN2ab#@Tfi}g)0Z4RZT7Sb_p}SM=(Xlx1R#1FdkAM9H^l2tP`x?c)MOw1(Zq>6Chm*OCJp*Zh80Tcur#5%la<XVyE<EbS=J%dzz@7^DhxBRG6MzWHET#shX<Git#`mq`9O);lp5!0*~pPWb^yA=piMioT6zJuiz9$0Q+_dYgPS>TFsEw8J9f&LCn>$v@YkF3{B+C5PSNp%d^{<I%|}8NLL)JBo4$%p9PMVnFtZ{W2H+c_geoHfSH_T%Kx3XEx??gf<ivZCmE7f(^LRBbH!feBx;A}A9xq*NWF9YXYN~`@mZyYfqeGd$BCpL42nJKEE6Idj{0>T-3bg(+e1qoq=Eoq;eafzx_|a24=KC8c$Yve|x+xLVzab&+5&P<LXf{TifBEv`K&9Y9BM>J6AcF^Cfqntsd}C}F$%0HIR8)ex-Keja2<ZudP}7JXGvp_X`1#9^i9`^b`bp{gP2qKEdS|9pp1%6Srtq4+bj6V%%kJAwkacyYdF|S4bY<ew?4{<!O!-Q4;_B?x#>7rEHGO5e8C|+`?dtgwWTkT&-C5kYU0d4l1OLTmNL{eVWZX+4_MLeifL+wfAlVYu(MPmpw8v?WUn6OoeT%>NZ}9mS=rfFN;+AvrbM_*Cq~N6GGwnG{aMawql>-Yuc*x2J=BpqV=AX|cx#Gxx`B1Q9AE^u!WW_Y+MK}i^y!^<J3i_R4Y0zA20Dq3PdhhRouu@{}!LRrW(L0Shhu%g5v%&Z*(3IdG(km*3sd6w|4E)AXgi*I`N1|yPiGUjj<HkMouZRxPHZ#n!RmIdNlhRl0Pg22)eyNXv)T!bygCUY(*l9cfDk&7_@FQ#<su!@epw(|zr!JR+ot@rM*zRKT^Qupkd@k+8^SH5i^Y(gOy8+;%3$|8t2Tt^@Y+x2a{F5u`-{%B$@mu8-{}?wC&8mjxN!Wbsg1%N;sV~7@kEZPB!Y&c<|C<DxTuMr;Y60;xJi9g3*F*0{y|#&SGZo1mTX3q}73I11<uffKSl4;6SZelcw-}6NM#-gm>>KYX{P=|d)-4@-5%|O2u+b4=?Ia_+tCmXbnBR5^G8Q!bA0BjL2k875o$BBb={IS9bs?{1^0t!2fl}%<bL*_$wu9E7*Y65?W6+30!&kRRq36aLR14|B1r(p03G&6rM$EoyakxJavNclFrwkPTt)PNDrlfo+v`1iY-lS)awV2rewcM^(1Nm;t+QaRtZQWN57-fQdyS}-|swOX=Jo;6)&PP(|IDQaTb;ILtVbL5ui+ZI)%sbG$TR`VA*}Jl%Gx&dL5Yd(wgtA}`iz}#<uh(y)H6Xx0*K5h2w^x3$vij4Na2|w2pdYJ;62cfT2|2>aP;7)AU9<~qFlZcAi7OuN^gEzj6@-0PnR^%KaIpAI;VyJg^>=V3XqNIsj}0`A9B*-rqY|E;VSCt#sy+}_g`vn9=H?*<?P#n(mZz;}H5D(F712Hg9;Sc~`RXOwey*^39?LC9{Iuu?N95*F^)X%4%U>X8Rmq~hz<bmW9;gEUn!a-3`$q!Vqf$1(2QV4mJ>yQahd08O|KeYu^q<uH2(suj!RKMo<2i|eJnh_K|4P@kK*ZDZf16-AJ|)TYS4a?WG1I<Xbq{RVZr(-bdAQ9vX3;<LgF%m9bTU*b#G!9vzdRN8g!@>0-{uZTYR%rgXn;;ct=&vMlzUa<Te%|Un)kMRI$&uw(Y#k8ti3dApc<MZMy)-{7@etWmpFsL;Lz@6*ovI{bYF7dB;Pida-X~Ay;nJQGjQx+%jndpMaWLb?@hRfaJK{32b@i)Ko@ntH)JTBaAy!T?$S*yfHZvj@YMsw=@(45!>+IDGGc#Mw6Yf6p85l{`9OVq!Ia*&%{Y`?Wa*<4zVHuGvh-j4qmoDTDdUrm!ZOL>pZY;zz>6_Rtwk|7dVp!8s)ceeHC+tw_ht`sXIK5+?k;*9^?RU~!%eW?01}AsJ<~t)%w*WtF9-(pYvf92FfWCToXZ1LXrOXM_*%W28|>=u&f-oAKkSq>3bUhB>Op%};SsWfxPW9FwG@r1t%ilds|<nV3dQ1?oMD>wAXF#Y!4g*Ln?GG$|4Et-x^TuUu~C(q!_hDtKs4l`u?wSJ|ELOIZ>b{HD*oIqbvx*X6%Pwjvw`34qD?F6?hW_-)Ty<;avt8J^Pk<R;4+_GxF+`tM<q@`5Tf8<I*tcNC8PVZh;NsvkB$PT^a9nBbj(NAo}-l`l?OcVj&W5^45IoYwDF8&y_7u!FI&9W5_&`J<V%~ZPW?0K;S&0{4%=N8o$#!oqL!tK`PG#hi|fnjMSiy`Uu<nDSG~YDePU7il@NUTb9+0ZY*0H}cCGXc@ovd-4bEILP~Yw*Q{VjhjO*py>bvT4ftIW9ZrQ!FH^86UO_mUMp_n^40-}Ra*_9^khN&;p{pH;f)hlj0DgND86%rdm1AT{ED$d=nmM+aWX&X*Qp)s7eKRF$!7X_&l+;6u)$-f^c8vMru_Ys7<sy7vsbNCuJdJ!9yS!Nph3?sV*rBS5CFENT+nQ%_6>`OWQG>-Ohc?(kk$nL#C)!!R59{2%%#{EqNl;O8px^gB1ZVZWA=ug-m8Fq;&KNW1kcoF&RLLJ=YS{;-){As~vKK28-%_wgwp}v~Hk&>iTtXEIjpm54O<*&(1l(lsw`AvU5S|Tpa*<4}<$upz`bc%X8Z*ehg?e}kD>b3?!+OgHA0;>(=B378<nPzUK4ZpID`_k;yolA`?XW6)Qu`_Jk(<Sf4e*ioGkQA5?UVc=uvk6D{+AZ=304?cedwA<`2hBSyJ<x_3#ycpE7`7cwc28T%!B%MBc_@&j&sf`eD3Z!I41K^KA{l@hLP-*ze!mvx*=x<#<yN!t!&;ax&(1h3%y!?lg}FIZ-fc}^ZA`Q-U%NCh)0n+HarN@l&O~D-nr>a2nz_<wG|p#XmZXg&WEOmOaS5X%%*|^Pv~7s4zqPiyz6lo^ZDF7{enOIs1eSM6klHNqHeu+Azv7c=-Lsq<_`P-?GhQd3?+$tg9?9n*)5j`>aC5)WHryENZ_U@2#MsIciTz1`d1*GRFWjEzC)wr2o9pUvcH{Qa=0@qL6CW9mUG+PQ3;34mZb!Z>!2pdqE32FJ>#M6j8Mp6JyZ@lOV>GZ_TUoqO-`LDFfGz?Djc$7v#FNrHF=($jR$1$8div0O!B@p&%%MT{S}pIBW#bv4R-fvV1*9;^2%nzz^EVudz9p#ZrwLq%VQ}5pTt{T2TZ=238`Wc9M;`*Eq!Lr1c53Fjg1=4aOMhqxqWOVaDhvo;q=(Y@-QhuxI@yDjV>bZ)2DHU?6TOf^gcd*(8&0whvQe)U(nU0)qu$Ot_$;nkkNZ&*G*p-)57386HqhYKIOre7+@H*Vs~A>cSR=hqMl^h~FqRT;+4J_{K7~T)%H;keoX(j8A_nya2O{Dg4nfIOP3%Su^v_7r;HocVaOuz8?I)2TfwDsE!@%C7%|xFE_y^>U)}b|_xB<$~1K4%taYLV<<h{4R?Th3@PA&>p0XH+)MRhDK3+`6wW2Zd2Thew+d<<zaZbTCe1Ntks$cb``E>bMMR>cpK55%6W8Rd)LDNK4+pTAOR3F-Zm-_(FReu5{{^Dp9TvdW#Yx->#^x7*z^KHe@7dKi9N5MssU6}(>8D*(}2qf<>aBQ`Ek8tBt1{IL@a3aW}jY%&4_tE!kdEbBiF_jaRk0rkyd3_FvvJI*Xw_;FDxY9`j#+f`$1%2MR}&Lrme8sTok{c&c@s(qu&;>bSj#jjcmqZ(iA4lmF64#+#3S}V>4qcWP~<k+>|Q#-u7)!Z)53C>y-|5Cg5=kLiz<)RKqEr}iAQ?OQBfAe;IGu)WJRbQ_8!LEOd1b$LFZk|Y3hPF^CaQQFTLOjViJMDBoR=5w0uKM2SRniH|Xc^2)szTLihG!EVH37<Wqu(xJ*%;O$lpBGL9WPwY>M|xL_oY;7cS(G(h^xoc;*dEKkv6wb+K(Eo!gRS@E}rD>0v-lx20<0*m)EPuFtd{paf#<y3cVbvZ4ep)a-$6xxbX5!kl-|)*{ex+BRWHxGLmEug93VJ1*l#IWU~z@TyVsS6~~(YL8D*jG!AxJ4ewqB1+G%-@Sq<T?gdddraLljw%hUuDlo=XSS*6&4ly9ol^x@(|56#i^Cm~&^aN9naXL0;J>FsW#$uw$nsz#=HN}bSJOF+aB%6v)C!OdB_mubQl~<Xv45vj-O1Aurl>WzwiyFHraRGWT(kUmDf|xx2Yhm~4%a2noD(sS^l(=Bj&<v58jFcI!#Qs?-6UO67+i9MqWZeXrx=Z=}Q4)bt3``zVMEacc{e2EyeqX2NeSf)De}8>>qy9ciN$;ac;r)KEf7l@zs^|yJ0NPm`wdzFQWAI{!I>4HPwHD?J=hrVk^^4ypQ;DPVpHZfJkJ&R{RridOa@K(mf0x}Hwd&u{jx3o@KD5%fD7%s;R*Od}EfIWI8w`5)7x^|co?uRu?-lB5$|6M`S@I(MLCQ*E`tq61&zx72AuOvdiP&ddOT~Q0DPNa#$deJ%y)}(+uZB`VRaa`;fvb)s3{|##DIEBIrhK9hLb)>9?BVA${k-IFpA>ClJAqc2(=6#RvX~1}w_Ny?VJchH99pT3?|f6+P{){9T32wM5kJGS9vP}J>=fs|Ipq`akT}$zD1H2IkTw`N6&aIrlfTKhY}&6C7({K&%|)ilC=trotH&oPor$ResH1szgtC-p>Dl$gwBs?m5Vu}wBRDY_aNvmv;uDT0M*FQ55uM2zLjHwcnRM?_zRzEB5Hp4}PnD6c{_5q2C8H<G_?ZBkK7mP=Hc>?tXEL!($_i`JPqE}?Cu(%7XXuo<4t*^kv3sD_PEu)f=;O+19>ZB+Tf1R%|FC-(ld3?k0eRo0+TN`uS$l6F2D76OayrCv;X6$yhh;64n$BpT@I8Cdc@kuXS9hMK35^TIHIR*!uoSkAhJ0#1SJCBaXZ9h|F!V#{h|mX{h6?Bv)M+;nx|;&B%H<%LQ<0#n`WQwQ=XeZp!@wD^c2YBdh8#<B*lod$(5G_`qje57eYCc)%y#wI=j10wAHFP#@u6NPSMV#Zl~31KYQ>?kc)u|{d)cq-H{$(pR}%u7(uVs|2|q|_0mv?R5dxvH-8nbBz!<Kc$T=|UTt^}!Bu6;g+k$vMia<7O^ckSD;zR^DgiEPrUPF`{kUtl{JQ)!Q*J(!aRwq8<MAjIHgsl!kHeq`gjck09U+gsfM1(%|-x3f0wZTQa0uvOH7R?nA0En?X%R!4vCr-c>Km$7bi_^gX9Wq0QYwJKf^`A0GeA)>>W=&!<)J*e>xGI_Xc6z<LrADh2-tP_W(vlbG=T_y?6sYO<<I?URir$OxTtsnUDo8s;?Nc`BOml1qEFAkgy^gpz2ma1pNUwu_#sBg2wVA7z@peXkljP$Y^dA{ckA~s>L8GrKqpY*ng);ZVQ{^NxDanQn6b=Wx9AvC+N80KZ@UXKjppBR7Un)DyA@2lD8)fz=;Z;(b|1P`}yhA)@=2C!e8dFomX?R4%i_;kw5{eUR9R<iqQ$6IM*BaopPV8Ft>8NgPyB(D7QBYd%ekE7WPxL=79(yEvRRkeI{Jd!dBXqzq)Xr3k=TM4^=ja(06T=dbLu|so@XbL#)3+$OW4Fo%2~4#RWZI&br}MQ%3Na`V*~U1GnDr1VcbKV)jNMhDe4`6nKr5N*@ScOm-s4KdSI<2N(7nc7YwINAp(qCeP?9;X4SZ&3ci@E3k!KGj{zsq8i7EHJlowlg=Ss*Nw*J=Udab%;RL1DNu+izvPKQZ^d8l59IEc@`bA?mOs%R}VLA?B!O$y%Zv~OM4)FL}tAN0CmeAvgAxiAL0$Lv3#(Qu2a3)<(vaBp+9Wj;zZT9lCvervU3az>NpNUnWO@8GCPe3Zn8k;}WI-Gjq!yV>qHIw3yC(!{xsJvdm>=rfq+bB@`dR0RAmk46mFNM+g3M+BW;XQ!nVd8!vbtUu`#0gv2^$5EfsiW-OoKOlP1FGQe!>>3NIB<7yttYU7B@s99X|3n2R`lhcC6U%1p`ci$vq89k?toT%Df5r&D#H}^P?x!WYG2&9G#l+4<{%>aNm9xd1cvKl3X_UW(fsy-*1zXE5m6<jhtpT<P3STqgq`vSgtZ;)cGNIY2QQJ&9?o;yKqj})FKGD>ZoUVE;8aCz@mFRmED5WDfIV5F$o4DC)R9^QvGTYs0ba2q(QycC)P|u3ev~eggK8Zc4@|rV_CY&SlaMId0<aAE4c8NBfr{?lwB+7J7+eu><126`gG{nt+$@xCuLs|Qrmg9ae+>3ggg0*MhkH)IubP(OAXomg6i{FpgAZEr1DfYh=8T8?375%F|<j4oK47wMh(+PK@1_h&|w+X1w2T^yZ2TSl}-pjuccLuhLaLm_7xrA`gnZ1Hz-UESjAGm{d^O08TWv}iU@~Gxw(2LLmJt{|20tORX&2bhL40e6_YtEjNJHC5Yri6vDk6hSjVhV5LxGAP6v9oz6Nq92cZ)}wxMj-a<waakQ3`f=t4I)ya28W3oo5iC?G5|XgJg0|UeDO6|cB|dk>-GryLUr-n4x0W#+#KPCN-d>*AIzvbUJzg@!b9`K8IN4em9cJink_<(%Sp9^4t;2d4-ex=T6oOPG^9a(rCsE6F1Z;sk{AwOi1q}E^W>Hi-$ch<mtoJ`nszNo*oL~sXAD8WlOys;$(r)W7I~}5uw^u-FATHE&pCj}(VWG@7_5JfAXY5iGvmoE@e#f5e8Gp<d+}MC^^9!utO2|Fa1I*u-nOvz_;4r1dS=AINvwvM)Z|~$GZn^DJ$5ur3?p$a5-Cv=z|Ywr;l23ni{HKY7w7D$1%~ahMNMu*IF1pDme&TSLyOI7X>_N&&JCWK^eCL|<Ch=0E6#?^K`)MD&NXQ=${A093_c~2|B#TU)T>EeW2sSa(k=EnJEQL9u#2IN=*8G<#IkN2_CC7bC`s`iI<Vh_=1f@83~Z)rM1)}&Cld-gD54C^c?q!@Y*l4hbMu8kuY)P?1l?M$(RN3a5Sr~%#j{}5MHdlo3Kv&Mn6rH$%^0j%l8tHQ<7xJA?H-!!xZTC)gB}V?9=fl8sF-ky_hp)~Zb%MU<y(Xr`lCqs_C4v*Q3Y<0ZVV`5O5fqeI~cS?nedbd&O)rptY*LLtu8oPoeB8~=|O111Sgjpb@A^c96ZN$x0y7dey|%mu`I_j9N=0ejNjy`9>%wvM=_M0A-_N!?38SaqK&aGCTVN9A00$?W53rNj@afidBA^~)54%}KU6MZu{3})_=2$@47U5$(#BiqMs`{UJ&^Vf9pWyuBplY$;49YUomuKcY3{-e!}1W?vr?`8BaG*-NNh|Lo{uPx-YMq(`==NG?qcg&f@HRiZ1R4ibC+8DyR7t_+fGylCRLsdarqPqo<bzjJNb+T_aPz3{L12+BSa05^iduc<{f8Rl@;XJl3(G?{Rh`Hp_Oj;5aY_@@SqV>oFbjrKj2P$p*Wf_Po0A@a}~A9)5a}ok=TFmHBZnJbZWe*sj;Y5*a?tbbTmY(4S_@XKqhe8o6O{~^Q<R=lbm{{2jeOCQ+>eca0_?^703w3)aY#3I~>9dgG?jFYi8IMGW#sEX^+(w2v$n=s^Ob28sY;1Md!ub)_)ZbgC_*B@-Vr883@KT+bdboL!T#+BrF`4)OoUr0;5=_<VwIxRIgd0oNQnjZxI?n-hY5uXPf5!H-?Q)Z;zkgA92ut;+Y&I^d<ixTjALJgo`%%XyE`7MVD-*P+X&wip>{hUeg7iD-RxM+SFQy$pew~KhkeZnL$~CE~EIPKss#1cVmvR|GlTqj$CdWG?A+wB8!;PEH&iE6Ko5SF>LINM4J7Aqz{33qS;|1$eSs6CfPY`1U{Y+xm#zD&CDx>u7cZ^AScjWN`>smTSogxLZ)GO*=twT8V(v=Za!w10qvY6y45czu_+=@J(OTm=@%{x)G$J|gIN9Fl6~@Mo4=5ILLikK%Pg&k*@h$uG1(-$M*R_MO{h;%iB$p0PE4h{ASaB*0g5d6?yD+&V`7C-52_%jN))f!P^<n=qz`=}CKHV*q4g%!j_Wp1jcV^@Zt^px#&6RDA!INWdWdW<-KMzvM|^jYdaTx>UPdKj*7VeP<VkeNb1N{NFK(kTGZ#$)sEGsphm69JdlMS%B}_g&Z$tYaSCl+fkG?L{rj{ts*U6Al5HItHNZ?=cjquE6GeifKdj~1jXz1CFyU}S=FGx5FN+i^zJ_HUFj|5EY8`ndI1h!x%T+|tmZjPwAe^)ZK4n>kX422VodcyY@ohMl!b%Uvs^+$m#Q@)pJJeBMPmTe1fGuRHkUWprtHTaQ;{HHXC<nnB2Ws30mo)ek#isoIL4xL3`SQ%x7DTK?~*02|UO!RC#RwJ1`_qqR|-ve1Zj_eM<AhII(q!(YQ4x^5tm?O$dzBzylz33d!qifF8gNaewX6aJ`0(@)%Q1g9GsYNy4`;0EdXIjb9qUGnLxj2>R-9r+^Ra$i<`o=)^63#h=_fq)tH+boOOanmv|CDDfFy#Y!emO)dC&myOPbslNs`-lF97uSUs8jJZ#v28pcQUlbzKpO+&HeD*LyRFGVzw>X2qJuKDj+WQEm@Bq8I-SikH~^?irr`-X{5-duB28-*$&kCrgN<xJ4PaUV0+Gd3g)93CV8<RUGvMOUt_0naHM`pLKOw2lX(@Jrso=4S;&7!ou+#@xnUFD2rakcq|thL+I(8!$3jz-bhk|gmeiv!Y}Ib3cV8MNP|T0WiD-;RCT`b`84NWbjG-HH+k%mFF4Y1%NbXx!6Ae6e^VW>@;>aC*#x;&jYc|;c&4o;DB$BlrbfK%JYEc{9%r(&%q3$*58IRX=k)&Unrk$@%1plaa^0}*RR&J478t)xQE-GkQHLswml-1yKT9F*;h#GL}B|ShR?(OOnbo!}%+7nM;ljG65CK;!Jbvl<KIs!!+({4zLyp9-Eljq)PmS=EyAB=p4;Rbb1JTc9omslrx?YXOVEiEskASE1|W?Xx5z8z5r!hIEHr{9G`7e(l$uTQb@&_AtB{UjTHjctCL6^)Z-os12ktMN43C_a{3z_H3Ab*e1qenm;)=(_(A-#`|;8kYE-Lb%Mrl`>K;D^DNE8vpmNyyc|{Jbk~nZB=R4#S}Q4e-fNM+Rq3<<3om^a*P)J6z^bumerMUn)^5MYKQAFw=o2Q9;c;03WC%x>DpE%09UIUtGCzZ>*4(B^77*5I8FZ(E!`<;wO@>En*8+8=GWUoS6hF3oaX;k_`(NdU-QH~Al)(UU8W=SbgFG^)Dgsc>@G|wKO}(Fg$d0<ktMmQ&k*Dl^A$j7hba_V{m)N6vtW`$d!3U&rcV*y%Ya!;^Z|BI;8?6jWF0&mha|}s@fAOziL37OT=FErEXNOmOx`J*#xu{x8&oX;!Hz|giS%v0d`NVLehW4pe4GTkS=+1wL1}I{9$o$3c=t}=8tn@tL2?k#pPW_uM`s|>Bk{8K<f4ibNCy(x383rcujYszn7y*#VE~4oCKiOm0yNI_{-A>2>A-w4V$1*Ch}^qzdud7Jwk$C46Wi&D%s)i11VQ~!0sJuNBoVCS$oqf-dAQ5R*66o$Q}6K*IJk$GmYZaca*@^_86N2NlfQxn3Fm8T2vY1R6i`tx$V&?qsoydYe?d$9EQP95vfc8v_eV=hG6_aQA;%dXqdn0nl;a7C1R&5aq0U+;!atN)2b0E<`*ELO1W39MR1ie5IKn}HIC$W2RmSWl3e2VSm+d~qwCMJ{M(n+p<KW!f-K&zPai`a4#fA4MR)o|#3VWI4WF!oUlIO%rkt@~rD1wD?sNCgbRt4yj@h9bG^o!N1aZ%EWB_p^uoAu`5@!;A~-=l*vsX0R*olHy^#-}={=p3zLurzJ$U>M^Th!G&3^oz;vQd`AWX^g~3S7T{#=ObJf!g?Z;S`PUbfsO0++Z*+egLJBAP`MQ^Xw@7uu8qJdbr|L&#-y6mKvtNcqg1BrlXw+A`~8V8ph}t{5F+)*q~|{soBF8{5&Ve2kK`&nGa~LYV#{nee=b0qkLl)q#H=haTWicEU?iUt-~OeEll9OOE_+Ih>E++>lMpaV;=E6gL#5(bUhTvp9UMd$9FD|bp18_LoC2;>0u5D2D+!?^C-$U>L4OGW2G;QbK2TxDp}ZWoccTZ@wFWf7jUwBN4stSJbHO#t!m_csx{j#{Z!9j=$-^Jpsrt_2>eHFYRAUExZW0mc=|hI4hsJ_*JRy8$h7%TxBUjPj>LcQ(`_(ggnd_Grsi|~ux7Ef7GyH?`3<AoF7xwO+g>Jy~Q|k3W@|9IaqEX>r{Dpsi$ES}OU#JrA9}e3c0-nKX^mg7+@nT#cn7J@jo}RgIVLGR0N8n^m$xK8aEJ6~{=L)P$h4wkaBa0*Osm>edXGi&^L;UiCbW?oe8wJp7a|{aIO)o<6L~)WN3x{n1MTs+yZDqjrUyT|6k+mykozIlaOYb8REUy4xkx!52qC|zD;ZsqL8WdrMjSN*EQQtT!&X>=^yZ3?EVodhX=c8OX_EVvTD%y>)1mbYmqfolTP6JAF_>=kF;h>{FKRF?S=Pn(m)DdZ^Ocepk)%r=r&%v|~+W|2X0?aJhVQQwG)*X*Fnz$IR@i<gzw9WR47!eK~pZEa`A@gmrQR$>xRsd3V{zmE!XuT-&8gJ=L;|`4@e6v>6kE4kNi4F^?CoezoU!m>Ajg8wlN2eO+hQ~Oo$)(ks*xvs|{zYV~cRP;d<mp_lZ{Av6ScO*JuKK?JV=*$ZG?PkHC5jXaLF78<4!ELZqoe#|vo4VuhxJQ&NqedRShI+%X4W97Px-k$dJki*{&}L)V_HDQhQcNc0Y)kE!ZK_fn54Xi%Hi!>hKKo@eY%L*i6J53d<i2(?eiI`nUpxq!k7`$%0spvD<6wMCAP^$_As58ok`c3pR3p=(+x<aFDJ_mdrdrH`zk)Cna>0QOCvi>1{8Vw{2G?{HyIE|^7RnV!d(<=?zMY|akxilww>fb+xs8?V3cbf4hY~e9v-$HAcg&X)_z2LB~Q3I0loFy^UT!hzp>DB(75HcOJriQaIT*SsXP=E4jU6lk_d^1Cg!vF4W*E=b5gzhc+N;=#@)EnpxMtpBK0SFeRh#yHmRpik8*!bF4XT#V#YsZ_~PVPV)C+#HOclWpW&yPq$w^->|>`!!Z*qt_cdjm`xyAZ-;=fCu?L+BkpL|ruQB0lP<()<F&a+b0u4cjM)^6>{bQZwgkmjBaO&sk=zUI39kd{Gp5Z&|DKDWUeU!4~iPycDah^~~Ig#>#@lNsbH|83<EMa8BdYD+o9X`q4bb0!+fp|(3OMHVcPaQedh<;qVgYOe|5*E_Ppo6LZ{NV2pX2;@O>|a23Od8g&a>)0&GogZ!hcBH{K`$p1{1QfchFzG)U?K9G<8-}I*UK}L_>B0UAN(%$l*mRb(BgX!Fw`PJ`NU<InV6cMxP(~(6uyFRQ+j>VnXFUrN&%W{(7*~@h)##NyBh}|*WMiSAGERU5mAAhTBfz!6mu^gO>#U5aZMfn$ypn52~WjmhbM211=$2;EYN;TI3_{IVJ)1FqinXy_xOwtlln8AHP67UU~eRC)ufYRam>%u)$a6jCktl1Jc&M?DXx~JF|jP~+L-zzLpa8J729UfU<*r(XE#AcQ}V$#jvny@rD~aTu^gQ2J6-3iG+d6<PNe|b49jrBmQ#mI49=;$idJh6S+vLIXcWaBnIrtYWK;RXTn**tl8F-JK%Sq=xb;08s0)z!zSRGlmw$yo+2nDq<Q6-|SC;)DJuIhh{>SoY@)6-}bH2V0=;lgaOCT)%pgNrf3jfQ+HIMU!re1SXtOF=sw9+w@y*!gu;NSNN!cb+3)&&>W!iD;crP^kFK^N%23_8`z0i_CtrY_HLhk16d_ZSHQHFaM{5lZM!>7eBYhBRIrUEOKf4h*o3MvOKl%n@Ha3yZQ%9v{`|;<$bGJB@C(N7%>u`r<MsIbU0<t$gpjDyMHCobpt*`_+YTqrO~Q*<74=b$#yc-fd+2P`cswKR5QcK)X&9XZu}TS-ZU%ZrrMUFEhimkIr65JN@40X1P0NdtF{#xV`iPbW1*kj{Y{*mKHZT1^4&frvxEj{7Agt_@VbP|4AvhGaD||7jD+q5vmG5LR4*R-AC7^K`X}8U<`!+od*zgw4c0lcax0YKjeC=pgrQF68ZKgl475zh;DU1R6N*$iRszPftl~jge7FNYhY-cN}-UjjgseIl<Z~tK?rsn7FGviuws`8cF8SDsA0<If+V0*B@TjC;=TACJst_L?8hP!K4(e8NWJzvky=tZ0E?=l<SrB^R~JeygOeonVDGP(f&9d_2J3$iJJa)+2F2EH%9?1m_)7Lf^~#o*6zxW{M#a?k<Czt+AW5y=EC^j|R{-cnKOgbS4=#oAhBNZTCH>$-uMNey@t-UHXFOT_Fb@|N(p0}{VUh}w<v`OjpD6SL{qA=$^>XFJz0bbEwPPX%gu$wu^VM(eJG?OEJQV)0Tlhn5pSc@jyDm6Z>c`#^KJCfF-~<7HMlDYQWh!vUnw`T|1X@Ehc;Ll{O++ffD53}6&SASVOrE|ey>k~n=Qj#6T1NLvZk9&Q3iGQgoAtLg)dr0`$HG8}Ec(R~J%h%DY>toLr)`5|E=qatKS|Ik6b3YZtX|*zi@f+}9|c(`I16fM4){_%Ybb&fm+WH|-43RcS9$4mQwS_Nsx^YfRZX%uz^~bIo1~got|u3Bnsf^y2`pE;D>kF7EOG}*t<@IswEi&SAG#$IP8N{2KrOgNF%9z2+B%56^NZIPK^T2|lr(r2Kw|Ddau9@+fCwd6@(yeB^YyjO8s@3`0fa>m&*DkUOGTHuLmmfZ_Z6k#0(&T65B?;_5XT{q0`-AmE!21B^Bmv3JC{B7A@;=gt|Hmya(!#8jCLyTz+*<mIc48&Ho+Ph&0d_&dGnYP&cjSel-P@(Kx_=5#Q&!>Ms76<ofQL-=@M$?!b^tpNIhzxIZ1h;!&l<M1Pm-f=+}HRO5APqtEw0WN}=yYtEFQHOn)nq4KYfXGC)Ap68;fq-43ZxNhjm0Rw*65?_jlDQj*p=K~cFxDuEvXtR`i<I}B`FBIJFPRB?31$O#oWq_oSrEK_#6WH{{Ijk?vnPHzXOw@_rqz;JW*C-s$7VpubW23I4*PI3ey!IeHMzA-gRAMu}*e_D@*q);m%#9vhQ_WR-cd+p)=;SR9EVOZR%P5fnJ;=S_3wQ&1*x*ULLY_yPC)Ip;iN8UQDQTrgOBO+nJ7joFB&);4LCRNddLDy~rCLyUCJsRSi#}c=<)9E$uMlIi3o`Zg<s#lehYY9zpVSKo|+deAzB`71A?uw2*;uDX{9$2mA^-=;x+-blXtKQ6ln4z;3c6X9(C4W&>G-$*a3_}LM8GHA(nkDg$)<DF%HKyzB*Q$;}3Egjxz4XS4C-~zEE?q_3kCA(E*H4{nk47A@cTY01;)zH9j^n%YV{cjV44SW($&*~--fX#Jv*vnAcK1g{kibK-&40+xbM%DpnB}3Lvq{{HH3sE<&XZRudDTv1@<5T)Yfoa#e=LFCT!2{OpDgn{vlX$$Rge25+*qP-A{%bmKF@)oplyQ2*Ye+qnYERdt!ie~@P*o}NE?=gl6V=A;($ugzx8m~M7g_h&@Wg*y~=I~q~GIuAZV-x>SGL0Y!6eRGWuJNasdt!!U5hv`;{Z|#nl@Z(lc0GS*X9YrDwFQJKzPy0?G(xHYO5kDE@}D)aX-X%BvNv0;P2;mqz4LT29Uo>rd=4CFUiKK!IZrP!Qm8X2=fpsDtSED?L|xpLC=h=_{O#GF_osHQP38bAT)(K1e?p00E<XyIxxlF|Z^RK;$R*bs-w>L$x+&s1I+|Hf{y}y;+XTYeo`92Q#&R6KRK_fn<iL>fD(Lg`Vo`>#OS<fp1}UayWH^0fR^{i&|Lq&MeX&%#|{n)|Ox!Hl%JEy%#=vpka!o?&|vD%|)!XQNL4P2{+~!>nod!Hx}om9f1L4Y(>8Ok(!5v!vC3=MvbvyEGg_lu@oO36pFBTzC`D9n2sbl417CuzoKl{=2Gi0T|ziHM8rT|N#3c|g(7AaMNdNcJ6r@pT<1>*4fE4=bV57q-tG48clGDDSAMdx`qP!v1_Nj4n83e$PRKH`xw=_fqWzUsX?OIJV4H?cmZDOJeVJB2qd26iX>b|dOUVnW_K@Iw$J#}XnoY82EY~;J7w0#M%C-Y5n-=}<Y9?`Si+|ZRj3e-w`h7d$5cjs!uiMNZ?gj*?pSH~?mDOH)2JF4Zz{1FbQsKjXrM9pR>x&abe!RYEFIQG+gvD_viFT!|!YhHlm5iF3Q_+NLpw%@hqF+&(8f>iUMpB0kKCqqj+Gd*1!88<tg@owFJUOTCNiUXK>C86Ojqck()M}xCmxl{C&T0ZT%3y?out>X;yj`XaGs^;sP%P0Pz;w$fv>_~0uM6E(NK7l>XQhK4aK~P^-7FL*fQPJahMUlSFl7&^(}o+lA^yYi{^s`&cRC7>U2z&8B#UtE_VuO3ja&5vKXrBDy^ay!yNSyTRGFvGNf>H2y53IYp<pn;H1myJ%!~r!yT_I1umrMmD5O+5JORV~2vfvF9!<H0vb~vz`eYpC^k3}!-T3+;?=LRlFzW$92t2tWsmg^dNd{1cFmp~QacgW_4-^Eg6KJdI9}agXuG#`NB|l%neC-~ZqCp_2_yNg)B%lOL0Q`X+D>pe6T@IjmfBE`cHYKvo6mGVnKFe=NOY|@btyL+Z(<!&xY21kn99i5C8jNtW7wL_}Z}H3R|4ov{Q%3tAX>A>|&SSL}+*RC+q^XZCUESTCxqL-FRDv3Ob!=v{*Tsyi^ywV*KmM^db*bc`->yoWh_byM@R<ORYLt6{@kqm>8P9)@^s}0q{yaj*c0vfAxhiPHkjba)5%6vIu+uSDYx;6jzBb*Q&Rwl08jtj!-j8+~aTFd7+E{siIPAxj$w>rb@9g*DVdZN1YI#z26~2$!d;8eyrRgj5(Me!#f!()Kgcj+$-JPkc%}bYC6Xi=gt%;dyJLQQMd^QnXZe5)LnQmrwH!`<QY%{ob+wRRwGmJNDVQpoBdE7ing~&Yq`HZ~{#7L*`$I1JT-N>IMj+KukP=!3`PRT1&n#x8S7p;Ka|CT}-ed4`Yvx9k0EvMxq*qm)@Y$2JD8tjsMD&Fo!k|{&|Tw0w6L1F$@ef}pKmT3z^d7Jx@O3H^<4&mVh_xB%E{Y=j8-rXvugDL28DMPB+k4oZl>{ACI*<%k{2KG@L^HHVk-R00U|3+<biBi@#(4Jd?A)FMQam2uzkV)bG`T-}tt9brJs1R@??sP7_yDsj<)AvCr`pLreaN|wNn{{Jp@#ZbK97$M0x(Pv6TVJei_>>Q)kTS}pYN3L);1{!xWBwY6WaK$Izq<Bz;AtE+Z{zmza&7%>GSKw~dlQu1k>qUe;4mIy3Po5X{FYCA1otXJO)HoQ54j|UDx0&H$%zKYI@Q|Zip;`N+NtS`a?at$D9X%l)=_}~MLfK|w0fQN$!t$7b&C!oqAAK2q7>k#poL!Hu-;CmfMPOJ1g6GBfrJDbsFLKMl6oah@lz-|y43@OQ(jwIy~!PlqFb%bPG_r#dJEHl1-MhkhgLsOb`cm0eovV_3nO%BJ&(qgPZ~F77v1J`=Td>@(ZG+SPBC{Z>#IK<XDm{BaRruob9MdgoFNUEHg|hHypU0C343=XVw!?vGwtM$jEn#ai<rFp`fcW;uHjL)TWm}qs71B66wB{#%-2>{V3VkP0J}}{YxtA;+oOk-H>!p5G#*h@Ge<0!Q3_USYARWvg!Zac|K|GY?KRYImX_2>!NhqXjB;W1Hu~tzuijqS^pk;_4d)N?&f4l2gTzwU^B4nV#;He<u9-iQdKvVp^6VI+xlvnQTzVTTKzSI;%!r|ccxY=2pcF3NsT<?6+sGf4vY9F2-Zdy3lhd|Ve4{#jIoot%K)O>~0{w?lN9C`*E-fp=EQ99kGDj7i9g4{Vt{;5h4*#;gzUl|?xowidzn1Eob-1WF=_lDwk~EZ@ln=UYbRNX*7#&6$9vWgi%d+H&8Op)y#R`U<Wco()&-^C2^0_Q!wGK2aRyH13Op8D*$9xSbL<H0my!|#PbzM*A8*z^2_R1n`n5+-*_Qe}8h7ElsXgd*eG0V9RG?JB`hh|dL#D>t7Aezgvu|e0_D;^1IYa;}Px_Dy|Zy~z%*w_S}yje?_<<XXHY27A5KB5C=ly`Pz-g#mN<k@MnpuX}FLm`_mO5MY5``yE+fF<(gfwz><pQvy7rv!&%m(98Ua8^sRGsxaHYDg|44u}y3O9niIC1*HYvf3C>%2pUD9C~15?$g4iD{Haf2$8EgaDw_gT{0Uvt1@Gn^)}aUZ{C9Of+Q$sHTg~*1U-}0^_z5;72hb+ei#nkJUzU+IuBTt3K6<B6gX0#iGbbwT;fN03V6pN%5UN7T73m+&9$0Df0;hDWfPGJ@R8FgS8<K7wgyZJxLSC#MuHVi+M1cVhi%ytrDS?DjL3h##Ld8UcL#2dh2_N!d>sl`YRhmC-_I1<Sj(2mW+zpBsG*b+NKE{y)Ee~q-3G|ajTrS<Oz&7MDf)M<DiFs8^83w27CFS$&5S}Dxi4DA?Taaswo4d>a)>3mbbA3;P&HH%%uS9eEp;Q^MX43VL)am<g45S930PxC>u{cK+(uWA@J79c{{p8YL6`ShE>FxIPnt?1<$`QfLk+{xrus@w1`Zk_aimG+B#0E`FE35ZmZlg0B-NoMIM^Lx@wBEgqWw$fZQr>_XbpyQu*70JG1!Ed;ikH=xW0_EoF16^uGHM`wVP461J_ktLjQ-tmcLNH6V^6XZ?JGFC2J@|5-Wh=Y#{xDL|4D#Z_6IKrthg<I=$wWv&ZP&!Wey=;o|Bu!luf9=5JH8Mr5k5=*+Q(b>ILXav>{SSX{Zefg0#Wef^G7w{E??wz_!>KFVRE7=ARrKJN#)SUV;3w}NkBjU>^p(TY3f39XdQSS78fAua>*NH`boWjkO<s-w;%BE6E&#nC~>Peb}-C!@rMrPbO3AGiai&q0JbF>+Og2ueH#m5kvcPa839!C?s%<Yx2~_<3R+68|j<T@|N*Z|K**HnaPPE0=;Jx8n7lh%1-&n6iCEGUKR>1A=yLuau->+MTJp<0vN2xQQ1IcEWb^5vFfdJ3fs2?Iwo#R|ipP7j6yGH2k0aY-cPdBq)agb~FUn;8L54%!hoah}|TevuhN6#UA@z?_7<p-rfX>8ps;od+`7^<$7ncr5|i<7mZxw%C`rNTFtD78l2hk82thVy+ML~ZE;131GX2}LGl0i+0Wt&u#$f^IPAXvXMaYI8hC1p*20a^*?RrW+w_#TQD>tVYs55R#}u-21$Av|daRCsi?2{%+{WTx){Wz5zgxUM9hS>w{QnuaZ^)6Nqe{9TkfRnCK#mg8E+DZaIx3bzl32oU6egp<P_nxtd;X+x+#e;NM%oQqBNY4<S(cM%Vm9sqR}PHmf9WC}ChYW4dgcidtR0AbemphpVvUBtQIFORR>SY|)q;(SMwVMlN1uyX#wK8m>vxgOy4ru=8DGJDslz%3=9}!lG9j=-xa>XgbA_+)1VgORp-aPb7w7ULbKFc<u=Ef{R<I+B6;E&kew`4;LvQE75CKu25k3m}>_3ypY8@-DbISnTeaoTz-g<4hG%|*S&A1d%aju9T22HEJhwKa6DDXNL&$FiZ6zB>n0k+1&VmAt>>}C9yA9$kL(XJO~HH9M*BtP<2LH+@)r13}T7BJnrR7c92BL`0@pW8U8ZCp-s=Sl@%*rUeKGm!MfSTxC25XqXW%rww{fB!*@PEO)J8?`(Fdx}e0UHXc$7m!~En;V?UYeo=fR}HIrVmC{DQA%*&X&FaI-szg?zQvY*?Vpy^&XlGA!FE#>4OvXe6{4Z~5?q`ZWHmc2mMSqv=!T_0*2<n9dZtQAJqeg*o^v;%UF9Tk1v-`GX`fKrFnmD^KT+BMe8Xa|Jsn#Vb`2;~v(#0oxqv|wrj!;1{MgwKb9VBu$i9cwDYlATYqT-68$~TNdkzPUW`uOBY@_Wi2vWXt*4pjs3K>SyPfxh2XcmHHrLUDi&b9hxIhUDkta@2Lz!6)vx*)1!jh`8omqE7LxV^>(aWWdoB&?D}x8|;`nFE!S-rs(`dH@PCe5^eQIa|HL*YdZ6?3V(n<z#uDVIR#5%Q43(<!HpO!yrLoX;l4htgkMI{BVrVx#_NG_SzxG#*Sv6nAB&=!^Qg8c`+zI9x-f+bFq!M?4a58=m2yx&+w>1<&|a%BusID><V?*N7WN_9~u}+0Qft)KS_+kI%xlm*lEOv1*<LGK_|9_P(RbEmqB773+p+oH~kH+gyJDQ$b3?qvW?l~1-G%eE^pwbxt4k{%aTP`8@k<Jk7@OWVWfJN$Agu`bP9SF_ThNO$ZQxsYc!g}Lv;F8kLNAVcBoX(9e7fs)|^LKePFkQ^RNU2Olc6|oeCS@9ztXB-FCkpwQ@EPbV7c_#8*XSJ>5vGTsZ#3s;Z5*MDb_YXwU9(z6L9^QBU=Ev%ad|LtN&ybeR%6*bW~ktPM0!!LZ(1TwdIKJ6x`9&flUJz||W!7Uvi7m5KVL6HQjztfkbJZ%Q+mhol?bw*(w=7xxN@$F@gqjl2Ump<&eA?~;t8UZ0AkRD2kTTOsc#GpgyZz5>@o9VWMOQ#iF-mC=hj4RRZsuMc6X2Zf?)G4L0Y@d7QVBwq?>SS~X>48|-_iGXX(eT$*n=?0JiEUEIKpfjZIFo%n4-U*scouCc5<C6<9@o@#MJf{uyh`PA4i6Rp2Gmzm@?RtGF<uIzHb*zRaSk0A8A8QvYrdJ!DT75|@!Z6?xXS>B6R@_S^Ia0PL;A7nf1t*6d4tkjB3NuZ4l8>IBFU22q5Oo?m5x&2IsE*I86X;;;C4G7-i@WoA@yZOE8tb)<I>8bpzCUjFAaa6Ee9%&L41Xcdzv_|97<0h1%#`>O`vAqUa{D1+Xq3|rkCM1z;(x7nKjoDz&td7$LDUn+ZWayLtJoC#+}$dva^v>WrZP*5`yD^QrrgPSnFxXsQ!h0Ja9H4w#NlCAV$=L$63uAKtZ!7OFI`H$6H0?mXJ_1lla4dDs(VnW_=iwr&}ub_gGeu3s&>@ukSK;;m8hzPgftNQK^d0aJ;cS88})UPCYNCa7uSGtfZJvlk2k3w?)N$e$@AJyG`t_dT9kvS;t0(LTB)xu%jJoiwYo&FuHmiKrRD75c2Dr28N&Gj9h}pIwW4=9`CD&5gYt8999L6t7x!TEqupINxK!J0dJydiewy%Kchh)p+S6Lw@ZP9;GuN(N8-FOUPkR(<BuVVhFaZ5kt7JvU+W_D#wEi6L<Ca5Uc{-#Bdq$`P!Tdyw?8`mQdCO>)_2wIcPS5L4BnzRO>VdKto?`aTytuvBrKlz{R5`bLe(CncW_=yE8DIPH`m4=v-UemyZR1tZDST(9n?#3*Ou(;iE*aaezXjds-ZvI+uD}(1dl`<Lc;hakTMq~g>r1P4F<_2szVrFaUtHO^P2vXP!l6szS{;6@=zccz+unJ|4;C@p?X>$CITH;f$1`oZlc>%?*D60Sw2Y_)Xprq4A{43Oc)m+DJ^~I%TN{(E6(2Mx+&z(zVE-{>FrM?S1xVf&$wUK{wlwxIOY7i%+txG?y@37kNl&Vd20n1s!=%scD6Vc9X>yK=iDsDUV$`cpiJ;YxJ;JR?k*?{=i9SaL1=EX}iwbw(p;xz#ws;n%D^GUp*wc}`6drbUBkdNl`1RYMhSyo_UR>CA9;}7ithH~Dg!EtFQUOK84_reM+^fa4I@(6K#zt|gGIeR&wf%j18TO_wX`d@pTKHvWQsUydWsa%=B^0LMNvBFoc(X>=6KkC2{wtNcnu2~$Jh(@Mfx8^*cYX6&ge=$Ja5%W?s#pn5QlOg{GclLstNP}}(~5VDOjgw?C3?rteuOR)Kl_o6sLYSx!6zCSMdA_vs{~~u6(jWGi8cn9aprH<=qX0>Jxn5MC}-f|$D5~w;jFWSD{Rz_TeVU)Hhasb2#WAsbgDL2*;Q1zSZ?`R&4F5p%e-T8_)BGsGaTwiw{KW6um|$x+}x}$ui<?#x+>WtJ2<d{Bkvpz^Q;NUj7Fn?*|=SvPyRsYJElxV|8oPk3?&fgrQ!!U#H;p4gai81n`CiZpO1iJC1OAS-7ynfqQHy^joSb_S3oiv06$=?zdvb+KM(1{mD}!3lYr?7i}47bFx}g+D+DGDN6d<RQQ%W4R0aYb_V~q9kf)w`0=-VT4+-b^&DxyukudQIaoo>Up6?Hw$pcI!3-p8f;@W2Q*a$hPlc$V09}@rdjc`Mu;?$9FRdS|vWqL4kT>v#WPDbFrpI9%T$Qh?3do{c-+&pRC8giCKSIGl`RLWSz7H07!0(E-aL4HQ~o`d^ay!RH|QNta4vocZ{p9&lq1y||hl7w8TA(|gg61)C_z$(8|*~dSUJUkDRC-OOtbb5-ZP%Z2vg%F~DmAoC<jAO49c({A5yfZs>Y3Gc<LtSii;321W<ZKg3K9uBsq%$M5<l2)7@4|)6`uwex#d$G)Ubs*pkS16H6<TP*w+nRtH=HKoTl=;)?ytbRl8S5e$iL|kWdnHerzA0qfW)V>lLibG`YEMbwc*GaKm1E02L}P)OO_{WkG&}HjQ^6y6<Apo6fC*C(=(X4YZhKMb0<dVVX&Q~m@>qUlD({k6Y&z7I_nFxv^sN#fLu}5y-#>tqv6hzh`xuJ<gvIj1`KP-d+{IS8TOII8x6iG9v}&TyVW}02b?mC;OSMmPh3C^(4~q$PO#gJ)Fb#&YC%BfS*p#~X%!wx;*N|M#aB-{zb*~}7M=L}x0L^s_cU`~@Q|MC+XmR&1RV#4yZtC&+K@>|5``xz&zMgn`!2)lGCQVV-7lo#M<tb0m2Q%st9(h~g!4?U#RP~g2SQ&mZGVg>f|B51lAI51)HE!H3-r_z=Z)tY_*Wi(M8cGf{(vb;am`;i5{A$f_%YFok=@3bzb8e~W{@K8qtOk5yDc0x?=D=o!b&N*1Z7Z9k~p4P2R^xmB>?pqXL^Khel5#sFr7!dgTxV@Nd?6@sAxJC`q7-3%l4FWE^!8XmcBn!jO?+B(Z%g}sx!Li>`9L#lr~B0hG58F(9nR7urz`P@f$gvBwoQWc(se7QI4)oO|{O7P}jxIh)|yan*B$<2rHf(iEjnzO5%=BIT>i`0zEa^!6~KOPV;@ackDMwF3FMV&xkOk8+`z2H-u>UUZ%75{f4eLu9RmQ4QTZT4PC!BHRA|fxBIq3*Jo#<tI^Dr*@@}sT61D%H`<wKwysW3T)I3vUB1*RPnFB%^M$V4S<>)HOb4$hi-9-ksb@Lw=Hlk9+t)qe$_O$7-_jfW{HC|C`qLG3gf?X|N@qRs28soEH7b7`{D=T`Jbs#u@dIGi_%Rur7~TW_ZVei{Lrov>L1WlEXg8ty9%7$(0*+^?4hy}A5szCD8ZcC>a^M|y@d*Y^8vQ}<ov1mC@#)ief7#PGB5}!MHxZ1&py%3qkwDhr!+U=a?Y565;%2WOwT#Z&2Q+Y0{{pWU2cV4(_XBe6>a=$P4=LsHOrSg?1CLXHVsanj9dXb*@bImPo`<AK^>^Um&-X0+N2-8=HuRy&qcdKr#NnU=1xg&#EydnF><uHgW?>s&)b`2)%Z4BLk|58Jl?7KI$mJF!zEKTMJCzG)V4irQ(R{ND=={oa%zLfsvF}(V7mkop`hD#LO%S=o_OB-*PC3LQaglvi+)HFl!cC02UAzP8QOIXy<S@Q<>D3RDo8?sFBnz9Q1y=BXv^F{8LJaSExB%xCU-Dse_&$a2^9!}j+Q>2KoJD5V_;2j7>cmc72CO1qa&e5b(Qh+KuYa??=?B+q8}*`k!D`<FdiRT?#!9_>OP9}MmGcaRN+X?sv4*Ms4Treb(1%4L8Iy>6^HWEBBYr(5{$<4pxN>TW$Y#bVk>Hdn!9vkrQG&xyQ6hCrBnkO^t$GCqogeWD1;!qBHt`>8)3niacv&ESuEkh8p_F7ae3}Pa<2E?KttHO_Y;KPIIS)QNPjcoiVe?hD1adKfN-Ugdh1j;;IXKBXHCJ(>f*3idapBXBz13>Da{d)oTBZvZok3cw#^VRgQcqyidq*mFJhhbaLwn8iL&F)FrEK_3C~IMPj89k3Qb1Zs?#OTCB%~<rG|7CaQpg#|H7`H*i;iu}+w7tPu-TZTjg%G~pD<6d;O1Yen$Fq|H~f(t_LOp{{8DbWbUrY-0d>ZeQx_GO;?4$pi!~el;o%_S;QbcJJ%fXGm!1F<AT&gO^m@DeZli*|IF&#HpLIG>8q?d5ZP0h^9)g9AkC+Whn=F)Ge_lNlMR<`5X-sO}Q?W!e92GPLm-eHhR(mgU#9RDW2abVsIA}LI?f1lsr2%4Gdv`Y?h=_JKdAS~v8@smyXXY+QeCv_&@)6g)#0NMY!m&z4KCoxwD+Ia0vYZ*c#EnY?(`(f;%j-3hD(50f%dYcVt)1D3@5V8GNRr*xl>aA%&dwo0S$MH$=^#9wU>FaYE|_U^`r8hGbyXK7K1cP`YHXkHh&r=Ze3=2j=gfU<zcRqH$b`o46JJyQ##8r2htLly(#PN@TEs9;-eEVq8$C!Ln}WfrFZ%EM7cF*7pO}5<!(^YLt;%$H+dSo@YK5RPRsDlbd3NF+DTfn?y)fZll(CqK0;)oEZE!Fm*?Sh>Dh)6re81phU={qZxB^#tt;(<E)05ayl4f~wr1-AvP&z??OJ4o=7r%e;&oBPJ7ypDAtR8wf-56{9ZX;AcW7g{!J!GHt^qG}yU(InT@EGl3?5nihRnPREc98LsF6Q9(-9z#z78SyLv@xA_*pE~%ffK7m`&ehcGIJ^j_Oj8`Xn0(67!CBfB8{Fa)2=Waj2>aGQ?ENf32r)p;`T@jj>yG@9T_?DkKJX5y#W@98gz?xD1ygqf2HT`;^uhGPNRJQS18;wXsZC>ULsSx2+@!|<-B(<^4^+LMfO3!@bCzE5-il{Xuk?i;)cxrMATwv*J?ICCyNl9)J1q$t{67sdsPr<CjI55*$`;5Y<lR{7fiU=5|lL#spX)NLY6IJ_fWiG_e4+FNAk3t5<Vy3(oK&M?zDr(-~oFA<&F}cRSk4qO1qNZPR=8I2Do_KL^Y!x<7xIxWoIt59n54eH4e@$ZS1mM=A@(Ztg{l?Gs8=HqK326sZlv;4o8zTd7gpfFVv`|$%KK)7+!HT$id&W7-_yF@l=@wvBIrnEV|KG*cZG{;{4swp7|qohp!iVZ*;Tg8Is^z=dF$cM8t8-S!p@3<D~Mrx+Gk5*8+%3HTP!mBIRtYdrsOo=WK!X=aHkUV-u3gn6|WXp1$E$%l4bRse_zBy=$pMI^$~R>4+2Ly0$d+(oA=`Ytjtmt>>FGc%ES-5jQD(cn$xloSF2Mm;m*AtzPr+fKqNHEh30IU76M66K4QUYZujk6sW!<L?uBtvk=$=M@0qKM`<k?JE)SxkWkdrN;-#tu4_i6E!5{1IV#JIAn$5`qFW>+hN~s<F0S&az!r0)`a@Se3aXUggi7iI=eTr;SZY?;FI?3z>Wk2fj6K*`tC6o=^0}*KCZ@`-ScJQE>?33LD|a!+0{QM&?pa9@Mn-0t4oxRH;aZ3cvrh-BKqpILwl4Z~j?(IGYgZj2s&}6AgFiIIZ&4@PRcgg@N0#&p8F?y`fgvqKrRwI$P-Eu%JGW0`GBe-Ng%2~ehd0{kz|0GgVS83YLHa^X1sup`+3lTY*Vyfwu?jOIF;h)AMq+(!j3jCAyd$BU8uZ-aUK180QiNew%%zK0%tkqiC=5x7rTc1{m%B4fiFv}W$S;A9iJ(U2jY_qvawTy{+$$zVqJ|{RP?F#Xjd<esuf8;p?C_nDp<@%NJVCePsP`;rY`WJLkZN-%8FEx^wF*ZNr3HGcENR5<Fe1r9q=asza#ozs3p!1f(rQC>spAsZ1IY*5aF4@G1sdmM@bT|mF>)@c6i_bVJ)L632LTJO3LfRz?$oKlL2Zl*&acfV@!A<Xi`L**3<F5Id1or3cvO0+#1fNkYh-?<5y^^}4#r5igq<k6qfR<w8Y6?`YeFLOD>+$0G9-{}ZL@JNYQR7k6z?Iu1T&tIm~F}FQ@N1-HNj&^dV|CdRGzRI`sk}O{TA7GRJ;306XKDm{Z8?@vy_e_qqIHe$*4qJ4z9ka3mYE+*Xiyh<m`}LvUTC=GE(rM(Y1r#N0M6Utl(hRSd8U&rPO+bgh;Aor5eyscVVv4fIP|1%ps+x%C|J4JI2}MQO;aHBP>w7Dfxz{F^o=9<a@C-nab2Uc}FoSfeOXOltvtTHDII=#sQhrVO$zijIpMo#=Qp$eqsfblXWp((#zmqF0Pr`dON*gT#X6Rsc@WmQ<vo0lzLv?NcEHWNr2B5oyNgVtKs!4{d9V5_VbG6rJiv-1NvP*76$h7c3@baOe0WYgvppd4AX>ZD=<`drC2A*t+~5F%F3BGIR+{7-1bEZxzl28l0xF*R!W*oiFCHOwSAFR-XJ;jjSJsv_YIwcXtf5U`ZmIog!+93ln^NBp9P(&n+*K>h{|}-mw4x@F1fe{8r+Q~gw4?fI&cMbs<Q}r+Pe>JU7leVj67VwKEKJ)W>@mlVE2PUl9injk77Mq1&PCwHt<QQ@5<F~ChS&aIlAfaz>%_hbWT)h;KZ52d-QTTR{|!K0-wc~A_Slv?IuQ^b?EH$rx#!1>#XTr%QqctQpznzk0f=SzA;rUUzomdVY*y7!!5;ZpIjDxjt&?eHsCF{uLqV(FZp>rN+rIf%lOf##J*Xl!d*3lFX(ag8#}ZhGn(VPc~0^iOH;ci-iv>MvcJuDAe4Zm7k>+K#NWR7=NJF@;@{!_|McQ-D}G=ckoWz-S3-*I{tpAu!BKh1&S*T8M2^O?+c}K)b37UKs)%c?hz1*Dq3+Tf*Iuu;2V_Zp%F5G|coC8%eafuK1*)sREUCK%G1CpwcB7syQZGbCd81lfdobL`v)7J6!hN6yh9_sK-Mt&g=h@owMM0HJ4=CrJTtj(-UJVJ3<$zG#n)`=c#w&zY@6NGLGNZvA>;MyQk>{P<rSfRXcd7Y02RjOUd@!sc<=~#(yp@2nV2;G)#p^{rXPKwrsmrgAtoC{pq~#H(6Z`cDHIn?H9zlbp5+)&pQ)vS+z)V5%8pw5_UiNtflB=RPr%Rs4$&M?r72-HY8<)FS<C#|Yp4U*{WUKvQ)CM1y3BKRjK1ng{)EF@T0W8jrPdqywG}FhdH*9pO=*L}ZM(s{+HbYGljwA(bx=zq}yg<#lymJVtc9$-7-39VBZ1T8$ajH_j)H-p5E%&Q>`zEmsN6Uo6r^lDaq9)oG^5!77SCR*ugvhM3V>btp7FM+sN73CvH<)rknZn(kQ*(IGul7s0Zy<B+?zWE#1kXDx!54lGI4tcld@oyjU$z!K`78Nh-yk<LE=IMRU+Dfmu%M)9Wpotz&f#tqr%sMxxCb-L5(3>ou{gQ~VHwbnFoea`v=6xm2BeKrbgL0w!@h-ZAAe$C3Xkp)S4qS%=(RYfhdM!^TKHM+PL;%T;RN|L_{EkS&g~>!A5yN$)+Z;so0Iw6(e^lUg(Cwag}QNOMXep-Q9GGDreit+>w&KgN}!u|%S!vABD+RdcgOrw^h1PONRBg>+Tx|zPBW@q*DZ@ceuwn(1^YKBu;NwOWdao9D+W85W6_vE>99{I`}W=f>6v8=kFnRPCQGi2^w3A6$mf)m0O?V$GI_VB+zG#wXl-;Vv_0@F-$Bk6==2#$F4B#(x}Ma^QPz#^8Ktm~t)$<II+#*}_JqZVl2_|`#6St16V1!00}P@=gbG|;S$Z3fyI`ioI<+u4C*f@mS5R}mWT(+&Wj|C`JKL-a>d$s7<4MLzDUTb9tSXHpRiGmt<KnJADXqnm$4R><lVTnZfJPe!F-QO~Tev#*;1q*~a8f*REej5rg(-W;#)yM{Z~%0a9=M>3_lt8tdzSaLU9QnVT7OVIb79B~@6;h-6QmVM)J4B1uu7bjhNls={A?jUCsc}ALWLYL<*|EMuv<iwuV7-T=zs;Uyb7oDMQFt%rhsEIhYT~9^5k<lTyifG$;hDdmdfQ|8^!%9A{mu94zEB2WzZ9E$|SqDOFLUW=CUjX$V)I1q;RPmOaaq!W$+-&dKJ47D|?9Ut#kB2v(t+$S9Ie><-)I3r^_>yvqGU%%rmLo8BJWSSqV>^;4|&JO%p=*hl37Cc`8bjKa5M9Y6JnCC(-HvXAR|O)ca0GpTE>88RH%lP1O_I#|gS+9Df(}IbRp0`>a34#buKV%zs!&mMdW~EF{RdgMFYE;IQ!dJU^yQY(D72_sBj@0*u(L?pkzo>2mozgfxP7sz5tamEO)fT#1-wo?MHiR;+MMbH;8$O=J5zmdxEOW;Wh=N98rjsrKD?BjrMdKH%1Xe=N56n#mT;?-fUUEyWR0H3B+K+R@DLwQ-o?NDEdHzb_2YKAji+IhFn5xk{n*P=ygPtb&E`U}F${{PH&jwu>ErWF~*!dB=4P?tm)T+$Wd$&DuI#w&C^l+RFSb(=bk3gd|=?hCsg+xSlPe86{t<N^7_3aO7*7$%9TNQl{n+DxhG7BRY6|*lYsl*gfn>1T+N_p?x=M1t3)pjp%8PgSNqKjeMD<ad?9ggv>}b3SGnZ`khX1r@%1+7fqZYd_IfX{h?TLp+MD`zw2zvwGy`AI8+O&(*%1DeFfm=m0u;(wOJ@_j)1~eP{p|qt~DYOE}cirEH>(vqG&mZrDmKMii-G}G@D<W@I?Cguf+Lb1acw+R)?hh3z8fjGqj9qRghImCb1KX8#N_UCa!VpzG#UQzmKu>7^yZK8Zt2Y%-fk~CCdl_?jo=Pq6M&5m*u0ssxT)x&LQ5%P`GwuuiK+II15Z*&VzsM0pWI98104b`U+G)0|1vY#yjJIJ}sfL6`8l?&Dx|Jltw=<6Z!}%pRnL!OMjxme_@M$udLpTHS|=lQ25I~G6n`?p_zy(Pg1ZmxYd7eK)1a7O)1CtY}muiH{(tt<1mzZH%_ZsSfv}aQnwSDk5cU>?}TGnw%;*XMW(GJ?9~zNYFyy&j6D1DEdhF9Dth=jm#3<EzH<f(Ksr5ZnT4;tvXhe$wFRFhf2578>MIbd{7NE*8seu*@PvjE;S>Hi@y}_rC38opFz3)iMr-3P6YxlPt>jzsLYbXDw<nF^!=Suq@F`(D*BRS^BXo70lOn?B&O*M|K#rw<v+Zl<81{QWAh6B8fSyQh6aj4+4x)%Mf6O6krz9!_V8VDNOwR&+;%9rITa#Vcga%oNtN1|SYc=$nKwQqZf%J7MYo`x|kZ$4(n}1=uE9Dqu9Id>vVFub##K%L#gQGq%ePlAc3@VNlYwut>Sl2>H40RHFjqk`PA+o}PYDl>m6Y7<h?}hc+jm=QukfuOZxvM;Y)%<~s8Z1yJ#}h&e$N3Cq^wB?Y?m_;JAAHZwGKqnL1*w!zwj2XZ8*TyAB$gb>)7`G(>fcqa%?OAFMW*Lav@#@X_yUV0P!AL68aU<_4Z<lum!h?cE{O@S6L+f<*<e;W5ceiv1=AW*mK@0JpPf%yvC1$GyPg!tvC94wqc-_Tg8;0R;O7bt_Ta$P*G<KU?c|0~C>&iPQPT1tZ!<mUE!-A?X?gJ`qtKTKUO`?&6K|(;wV}*9rK3)Kq#jq<iOSthmXa&RKIBx&rPGSJ8Scudi{7~S-Q_K>I;vrLTDj|Jk1(l`c!)X68?IVh!So|@*3iynTt4wN8#Ny7fM$&<aZ7X7O?=h3mf^eRE}uBGNd?2h%~^we-MFy$y*bO-PH$X3<p`HsFT0BwpSk=#)4P)nezj{}xNt0$#g(g{_-|!B@J>dYFL!`QCFQYqM*U1!M<wOrH(vinFh<383Jjbwn6?>o3U|?L2GTa#UgfG`m$thE*l#ebV&z;gHIE6T4OK`+Dzo*>Jdg|W!a;u$xClU~X)(IWO^vhXU0_$63tkg$Nq!mg21-K*w(RwLpjXG<u*X4!e2z*MbiqNJ@OsQU(@Wj_EJbBhQd7=Cc{yWw$y~jn^*Js{J-w>yA<ai6Yf``kC;f}k&X%vMk;1v{Omwbim{-fG>gqD|-i#hSARCsqDNUol{hGO4qby8M$y@$Q!Wcf)XicBKcxXbwvZ0BAI~jo$73TD%pX2D!>wpvu+RTWB!>@7Kj4>zg0&ou<f}VU2-yTogYQO42C4D;VHqn&n7teSBGLT{k%#|%>2qq3R9AvRbG?4A1mw(MzBrb;w91Q`ZSC5ZWvg9PnCq33EqBQPC))0gMqW47<DES3azM-}PAHz&CY~aN891!>LZBDs&>w!(+tE|N>oGu+?2d-{yyL}O`BI);(R}5w*y_oH)*vh=@**hR$4j#zsFZ>Xti-o?T_v8P+y=&c#<H*kc{S<8i!HDY-*_Z0u5DX$)v>`yU98poS5EhN<u4-v)lEa-DF2xA^u}FaIzXU-pK#*jEeFE=djWsse%`4DvlymCRU45J3NHljD6b##(uIjq=sZ;0j)j7_gh!`r-jZzz~PzKXQ<(k8ifyC7+h@R|d`%g)@79T4kQ3GpKv4(#s9R={<_cB<cgWts&eWVv)05XHM44VJq>0y)Tv3wx`%RuGo%GhWkgNW|r5m5ns)X;Yrm+^f-DO1W3sPnS+=yEWVw)loM5=T(br22H?&JQ%>xo$)Vkd~mmiKIM%(18KL--@gBbz_o(pl?NVlkaat$mlNzHlAds*J9GcN|-cLHdt2Y^9i0+eLi7R6q|WckK(1=-kK+m<<E&)g1#B_oid%C)eLms{p&nB2DL^2Hp)f<2^qDwQt1qX)WmtKERL5BvWm)JE_dIYFHug~?oXfv9m>h!f2;ZGgK6^8SnFOYGEPC`U>i97gxUISkD;`(>`u&+q;;H5>b}N9Xt|N5V#zPzqpIKI(Fs47ylph;VR(IPxMR&8FpxmeJKMlE!pi+y>&!O<wi=l~u3~YEAm%!1r3CH&){No%jZDd;muOWIB~x_#1VuuW92<_C9q0LLE=0p8e{uL+%7-W(q8Ia23Y$#IigLsT+eb66wrUSB09`zcZ0f{H+dskYbni`lf}8!A$2NT>F7FGkLv2Ib9wWPIyLDy6>wZtZ?s$L&)N>fx|MgIs#ymx+a^a?n*XCA0Z@sx`dRmn&_HdI5Y(5@|Z#kL>Y&l*;RsE|cJ&oowc7Z%f*|MF`-wpj8=aYtm5H0M$o(y^Za>qHWQ{cwCu~QrPp=NF2hl<15pvuvKfmsxQytfSl=Z^E~nTS{}^S$Q$XomT=XYwYU3S=y<5-p}^mfsomP*65ynjj<qs+DF=!G!kl?*?o`aEp6}Riy%tj!i&geOICtECp5M>G+<)+#%_55uXg<pk~iwuLH=tHq=M~FW$<|E_VehFdN^#gzsU-SG^Re?4a^}Z``T85{Gn6aN*`t>B};=VnU9FN2(RR{w(B+>oGd*XveC`{!3*58E3~+DQ=nl4u>~Z@3+1(-ItHoB7y{WpexZpbQlZNxPPZWLG*?pK_-;17q?dj4*&gkKZ9ETZHBS_?RS6E2kaX%CQR02I(pUA4%TM_Egho>|E<bwBcgQwQsp73;Rkq(SBQ~s&GWklIPkj$`U97(Pn<B0AWX~N5JN%jT<qe&QI-~&2Gq313tIAhj0~!!<F?o}0`kNQLqfnA(CK>?t8QT(va=g_U0^(Mh0t=*$m?Zf18Um2B$(<G1gAK7iTa01<UlYBlCzUvICI$M>^_D<#S;V%5OEn!^2RYrEy#3HrV~&UbxjWqX88ix(N6a^QAe%r)2Dj@xkyPZZ&cM7I-Tm}>tFHKO$XWFcJpq4v6oR@A3}Q$ejRmah-K5Oh7KhGe;?MVg(K>kMkIsIu4O}lnODtkVn9M-5?=bMI({wbG82fVHZ7gZ&U-CFRqt9~6hh@osrmk>X$R78uQvb*u5|w4e#g|a`fBh`RcPaH1;7kLu&abTcmwHYG9quCCQZtMI#qy#IsLDEzs|b#&8WNL>a0_Th7cP2iMk^SH~3HmVv#U&;{?^LM+u}-mFh^OmtDUl0?RmCqd#=}>rSz1t1(HDN1$0!D-Iu;io>RX=XHh_old8Z5&l|an)vB=f8WFBW+&xzek>HNrez+VfYz(HpA0CE&y+>LdEs|;F=0WWuaNBw?^q~B^ih3&nA(awL^?n(tpA$s%eGZa{~WXZ7~XJ!?dp6F`R0fF_Y9?{=jfKcgeFW>E1p@?lCZs59Coes;p#G-#i*g+3br94`7`e5L=5c@B^z5HQi(Y_eU&hQY9(GpVliVcLSbd|F%dEC#!>kOGFU9@>Z(zRQo=Aq+Gr##vSQ#hW@%5VMrwGtp3<+rX!j%(`g$Y+T)-7%`JRLwqw8o}RFucYt|o2c)D;eV&K5iGB9pB~)~rBL3^g@?2YRn|45*Ij8LFumx~P9_++xkcpq={1#_gDBGn_O`HGBT-;6UVc)wlOVsI~0jqtDO<n+>FDes?U<#)0ViOmrFrq3%t-T@9y$DX6bHXc!&f9u0xm_zDw4qaA2b*N}1<UT6`#8?lYj@**HFURFV!`%k+8$j$eO)zjT(!c@C~v3{lSnyvFyNf3(=fJE|`Rp>8+{I<u}OC)|R)K|UEgYK|Q+-nlX5;4}MJ8a;cjr>f*^I74662p2o2UIrT?~xzmI?kSKjsGdz$Wb$}p$zwpNtpu<oNR>$M}PJ%ZNIM{r<n)bbm7QD5%q&^9tgd`4I}MJ|4laFz^V5ie1$>J%v!yDQ+*MLQsbg!a~RgFV#7L;i&VPQOus)n#=yc!GHrhMcQtXen%Q{0<Vy5?e<jA9Qz>_1(jyS&P{+N~EVIvUVOl`sG*Ay|E$#N8Wka&wRl!8wVfit5C~nqTavgmnXSUKA+8T79!_T;wB)eb11o|}s5gSg6BE8p#R=|61>8c-m=hpqcL8(jtpbfcf<4q)QPnP(p%p5#g*V3!G=K#^4Nyal1>15~p$7iS#k5+$FW2()1htD58Kj^9<^;{4~IHW!3KxhAY^$%D7e)Z27?o8Wfetz|zSO0YNi>rTx$NzQpFYvc0o?P$4>xO?}21Tpw4Rtc^mXr8bz!>B$n9-hTjBZ$vOpFz#HvjhOHzJ8T+T;bu3e!Lv(5$ZqF1YnjXa8!-s}|1=UOYPZ%M9-%W=|hK%nlDe!{j%30qpSL*^9x&{STENRm7QX#f7aR&xrirzd=~#ui8o4|EKnIH?R3M&GYS)ZDL(S9(9pQe*G=K>dM&<e$Xerj|}Z`b3J4v_)ONL>`waDy!|%rx9n_Lj3zqm@y^NY{GQHyyaPoTWppMJ@7*&EoA-t<D~f~y&z?y5y<jWb1NzK+-#3!9(9CW=5vO2#p<g!c<o&4stJSpiO$%(cA~<3rdf}R8zlN>!cbYa0uh-Sr(267DUW@PtmDgNdOA^^b4jyJ7!zTF{_r(Jhwtq0fq3I>VEcThuM0>oh?X7!_&?i?v?@6hwqdbW`eZNy_<AwEF-pu{A+_2I`w&#GR@h76Vf_`={@Af3@9Y(a|biV%zPB|c-d&>Oy<iR5hxBT=o82Vu=awk-8h6aTPMwEyu)BOhyCUtLTr*Gz_$AFu#SQwmHWP$%c<%Zv4d{KaefrN;e<tx3xSAY$XvseQzqzicL#5!I3*ztU)zD2of1gOS@bMUvw*D`icJk|k;l}UR?Om*ukKF8E+r@W}fs-ip!LpKgeMvHU;xi(uzpirH`fUxLP6sZDTUp@@Bs!8fMvgt`VtGa1cuWKK}Fy|NQ&Lvi0XT?`6&qB|C^6{Rt0Idh!WEjuu$7dXlc8nkC)REK|drpq>093)dzejSOc%`WR8q-sbR~fK%T;&kcq&!p8Q0tS4obp(s_fnFoY+PZj2P`+B%JMA^u07}Tr`e|u9v>cLarX4nEZ%bt{wjO$`SXKzE*RaedGi+A3=<LuWtxVBxQx=mPs4%+ei*<XY3aFMUQ&;6s@_-kmGMnMOvdu!G?dZl{d$p2ot?@rl6AVQ>!@t5s#FFJz+-4y0BKH-&tx=<!#{s4QwHu<=65ui?a6d~un#2ttIW7)&^Eg;dk>nugX%V$9fPj$28(^ykM4WkeV6PxFTQx1`I(n_DjOkv1)L496*+Lk&P6cq3+I9ZClDcYO5ox-E${{h!{cK%-^HCXrANyPq1%j|;Iof0kb(oGIj_c7Qyy9@sxd;Ryg0+D*C~K?3d1_pqDEE$gLyq;0^Lb5KC1fC1+e!epJ(v;67RR13Z{;u;|u7?CuyokYWnJ=Jer<n80SGXi#5_z*J!LEX%bDC<!93)hGmjH!H#)>@z@oWw-W_^{?N>A_2R+fM-LxV^j)`?;monoLqQI^i;I)xYmPA@>hKbLj^YJpm_!pE2^<?xgOKk<_)CTD>Q$zMl&?sp03Lz66%K%%dK|{AUl@dYdNx0uEh-k_G*EBOM>P|MznAB;`^zbxn|;ex&Zi9TwO8Mto}TIoyFWQ$J71mw`{VN+S?|8kiatUCNWUT2k60r4)7f&<Fv*KNPl_TWaRghEM+FHARwhNt<D{;tlB=l%0XUOCQ~ljn<JuMUIPR-%u8UVKi&0JU@5l54M7<~Ytgg};G~(wr8%yhj;n7qTSB(eZdsBWcV--N96W}QAvJFON)L%I;QLj0!7x2ia0*vt$1}_ly+|IycsQ{7;lr8vC*(u6rz`Eu=kC)5Yap$o*?mN%RBmSoMVOAc^&O0Tq=2MnY*r5X@XP7-{c64@pvRF}LMblV}xlsCxWz+p-ujpxi!F;GQSn*Rll*EJ<YwE0=RaL8Z=&w1(0qQ*d5?DG0{2ZbSUEWU_N4)I50unyRhCPqpWCMf1BG>@S*WHro$>2*+)c=xn{lg)=94ZpXT)CB6<-O#LcAQ>v(FNI?;i4|NpHhZOWas(3cb>~xm!y6&U8r=5og%#?TS&!Q>hY!g|K(USc1|fTL$CBBLUIy<V&*ZHM?MdIMhnZ>En~Ng-7<E|*zX!+_uQBVagp#i;v_Hq)C)>Va~66*8U$&$5o1Tvna_?E*J1AAFmwO(XhxS`C`;aY9?w93E{;$EUkC%%cmwv|8S%;^AY63>tQhrso&<e^96Sxe6{mq7UF$r^={#F3FJ$k=^Ogayza(4EW=G3umoksU(St8O`r?Pmltj{!#*}eCOnj0AQO*KCq>1ZBQ5tzX*lKVBc-M2?#y*EfuDM6i!c`e>Uc>LdhP9&R%2yMsM^SJd69V!Kiz#uVBn`d9^^&BFOE>ZT!gyN*1(D+z+CXW}m>YUU!~;TtAdmgP3v<uUVS=tptz={+BP$tM$;e7ZcTF<l3Fjn@dBj~3GCv6d%JU@5i@b2lIR8UPM*09h#6#@{c>E2DKP!gz$H4BzuzKVk;1)9^=B@mQ7GDjBI2FAN(TdLsTKaw(d$b_bE%Kl!6YhbO<FO=hxuq3LE1FhFLJOA{MM+AZ1RlvL2*NBa$`JlZ!^F~xrIm_S*6B-Dxv<KGRW7V@VU>%!rCbo2_&F#OZs5Bl4)PSl$CQ^5_el=_^Q{nIg#b+}A&p{ANerq4Au$OO%6%Vn(3G>dV7_Z<#nOtVmDn%SG$17*@NZHUz*kbAhptazpGOh1v|?$cqLrYCLhdu<FC-^%?ngxs25G?p7DWlSu0QMgyE)e%<1tIQoBCdwkQmLWNl@^_PYW8dnB9`=Z!Lf9y8hC1VE-f@E7F7~E+w1<Q9z5Frk<asZs-=gzO-QXkKI4-zkhg<1KqNS6lFpfP+FO%g~vf-C`cN+<aYMYTI1y=uJZ&0SQ>`te~0}?N#dq4i^`mGQYNlj+6`nk(9Lb2GS1z+bmJs&Q{uWY^~xXs_MQ5E8FTL6680nx860~gWnNB*N6}y8m0rqyFN^~cZnS|?H%PqDjlzUc;!_v)W|^cZgDsVVmgrjZur&``^RP7!Tl4T8GY?bWb;Fn@ML<j9$8Mg4VGsm_mVp;XVQwWOD;eFKWVH6y=Iy8!>s%t6NJhEqrHqw44LNA~UXhn+kmS%o6uDs(x^@HE4RnJWXx;6&TiHNmNVyN2hNfOYyo4krBYslmK@93e6p)QJP!WdEhMTx?6cIOabDu^uj$uQ2UhoG(wAl^x|8)bcy+m;<8)&Vgb)yZmPSLlXIO~biw6g9(lBE?(D~eWFT2Pj_Maj#A_+^>1GzP+nk{}K|URs-<wfR|_pSAf}o8Mit`7sufAP+&J%o(9^ltgY41uhSPwbLRbR^zf7*A2pP)`=0fqH(RgW4VdOwH9GyB_k^tS;@#sMpiPqTapp?i-Zx61uXSQp3;OBC8MbqkTCMy$U1YZWORd)(OOW|tw=^|BMNLH8Lh>RTFJ;tMpiPil982+?v`Y<*5z*<N7iwqX=NS$-3kF#2(Utc6#}dfa5sd24-EFM^W$X4^h<G8`0eM@6Y*TAh?g=q7{0r8lFI~nP)5w()f?dlm;VP|CZtF"
def notebook_from_embedded_source():
    import base64, zlib, nbformat
    content=zlib.decompress(base64.b85decode(EMBEDDED_SOURCE_B85.encode('ascii'))).decode('utf-8')
    nb=nbformat.reads(content,as_version=4)
    expected='EMBEDDED_SOURCE_B85 = "ML5_V740_EMBEDDED_SOURCE_PLACEHOLDER_629d6"'
    actual='EMBEDDED_SOURCE_B85 = "'+EMBEDDED_SOURCE_B85+'"'
    matches=[c for c in nb.cells if c.cell_type=='code' and c.source.startswith('# ML-05 V7.4.0 · self-contained immutable notebook')]
    if len(matches)!=1 or expected not in matches[0].source:
        raise RuntimeError('V7.4: invalid embedded snapshot structure')
    matches[0].source=matches[0].source.replace(expected,actual,1)
    nbformat.validate(nb)
    if not any('stage2-ml5-v7.4.0-full-research-high-resolution-inference' in c.source for c in nb.cells if c.cell_type=='code'):
        raise RuntimeError('V7.4: invalid embedded snapshot version')
    return nb


**Совместимость:** некоторые внутренние имена `V4_`/`V5_` сохранены для воспроизводимости исследовательских процедур. Все результаты, манифест и публикация относятся к V7.4.0.

## 2. Среда Colab, зависимости и статус
CPU достаточен для первичного запуска. Библиотеки устанавливаются только при отсутствии. Результаты SMOKE не публикуются.

In [ ]:
import os, sys, time, json, re, hashlib, base64, io, math, gc, copy, traceback, platform, shutil, subprocess, importlib.util
from datetime import datetime, timezone
from collections import Counter, defaultdict
from urllib.parse import quote

DEPENDENCIES={'psutil':'psutil','numpy':'numpy','pandas':'pandas','pyarrow':'pyarrow','duckdb':'duckdb',
 'sklearn':'scikit-learn','scipy':'scipy','catboost':'catboost','lightgbm':'lightgbm','statsmodels':'statsmodels',
 'networkx':'networkx','xlsxwriter':'XlsxWriter','nbformat':'nbformat','requests':'requests',
 'tqdm':'tqdm','joblib':'joblib','python_calamine':'python-calamine','matplotlib':'matplotlib'}
missing=[pkg for module,pkg in DEPENDENCIES.items() if importlib.util.find_spec(module) is None]
print('Среда: Python',platform.python_version(),'· проверка зависимостей завершена;',
      'отсутствуют:', ', '.join(missing) if missing else 'нет',flush=True)
if missing:
    print('Установка зависимостей; дальнейшее выполнение начнётся только после её завершения',flush=True)
    subprocess.check_call([sys.executable,'-m','pip','install','--disable-pip-version-check',*missing])
    print('Зависимости установлены',flush=True)
import numpy as np, pandas as pd, duckdb, pyarrow as pa, pyarrow.parquet as pq
import requests, joblib, networkx as nx, nbformat, matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, PoissonRegressor, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

# Ограниченный DuckDB: промежуточные сортировки/агрегации — в локальные spill-файлы.
DB_TMP=ROOT/'duckdb_tmp';DB_TMP.mkdir(parents=True,exist_ok=True)
def duck_connect():
    con=duckdb.connect(database=':memory:')
    con.execute(f"SET memory_limit='{CFG.DUCKDB_MEMORY_LIMIT}'")
    con.execute(f'SET threads={int(CFG.DUCKDB_THREADS)}')
    con.execute("SET temp_directory='"+str(DB_TMP).replace("'","''")+"'")
    return con

def memory_point(stage):
    gc.collect()
    try:
        import psutil
        proc=psutil.Process(os.getpid())
        available=psutil.virtual_memory().available
        # Colab может использовать cgroup: psutil.virtual_memory() иногда отражает память хоста.
        try:
            cg=Path('/sys/fs/cgroup')
            if (cg/'memory.max').exists():
                cap=(cg/'memory.max').read_text().strip()
                current=int((cg/'memory.current').read_text().strip())
                if cap!='max':available=min(available,max(0,int(cap)-current))
            elif (cg/'memory/memory.limit_in_bytes').exists():
                cap=int((cg/'memory/memory.limit_in_bytes').read_text().strip())
                current=int((cg/'memory/memory.usage_in_bytes').read_text().strip())
                if cap<2**60:available=min(available,max(0,cap-current))
        except (OSError,ValueError):pass
        info={'stage':stage,'rss_gib':round(proc.memory_info().rss/2**30,2),
              'ram_available_gib':round(available/2**30,2),
              'disk_free_gib':round(shutil.disk_usage(str(ROOT)).free/2**30,2)}
        print('RAM',info,flush=True)
        if 'progress_event' in globals():progress_event('MEMORY',f'{stage}: доступно {info["ram_available_gib"]} GiB RAM, RSS {info["rss_gib"]} GiB',announce=True)
        if info['ram_available_gib']<CFG.MIN_FREE_RAM_GIB:
            raise MemoryError(f'Недостаточно RAM: свободно {info["ram_available_gib"]} GiB на стадии {stage}. Последняя диагностическая стадия записана в logs.')
    except ImportError:
        pass

START_TIME=time.monotonic()
RUN_SESSION_ID=hashlib.sha256(f'{time.time_ns()}|{os.getpid()}'.encode()).hexdigest()[:16]
STATE={'technical':'NOT_RUN','scientific':'NOT_RUN','publication':'NOT_RUN','overall':'ASSEMBLED_NOT_RUN'}
LOG=[];PANELS={};PANEL_FILES={};TABLE_QA=[];ISSUES=[];SOURCE_AUDIT=[];SEMANTICS=[];QUARANTINE=[]
PREDICTIONS=[];PRED_PARTS=[];PRED_TOTAL=0;METRICS=[];MODEL_RECORDS=[];MODEL_FILES={};FIGURES=[];SPLIT_AUDIT=[]
MODULES={f'ML-5.{i}':{'status':'NOT_STARTED','reason':'Ожидается DATA_GATE','n':0} for i in range(1,8)}

def stamp():return datetime.now(timezone.utc).isoformat()
def note(stage,message,status='INFO'):
    LOG.append({'utc':stamp(),'stage':stage,'status':status,'message':str(message)[:600]})
    if 'progress_event' in globals():progress_event(stage,str(message)[:150],announce=(status!='INFO'))
def fail(stage,why):
    note(stage,why,'FAIL')
    raise RuntimeError(f'{stage}: {why}')
def hash_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024**2),b''):h.update(b)
    return h.hexdigest()
def norm_code(s):
    return s.astype('string').str.strip().str.replace(r'\.0$','',regex=True)
def number(s):return pd.to_numeric(s,errors='coerce')
def safe_year(s):return pd.to_numeric(s,errors='coerce').astype('Int64')
def nonmissing_code(s):
    x=norm_code(s)
    return x.notna() & x.ne('') & ~x.str.fullmatch('0+').fillna(False) & ~x.str.lower().isin(['nan','none','неизвестно','unknown'])
def canonical_hash(o):return hashlib.sha256(json.dumps(o,sort_keys=True,ensure_ascii=False,default=str).encode('utf-8')).hexdigest()
def _disk_schema(df):
    # Примитивные наблюдаемые типы сохраняются, смешанные метаданные — строки.
    fields=[]
    for c in df.columns:
        d=df[c].dtype
        if pd.api.types.is_bool_dtype(d): t=pa.bool_()
        elif pd.api.types.is_integer_dtype(d): t=pa.int64()
        elif pd.api.types.is_float_dtype(d): t=pa.float64()
        else: t=pa.string()
        fields.append(pa.field(str(c),t))
    return pa.schema(fields)

def write_df_chunks(df,path,chunk_rows=None):
    chunk_rows=int(chunk_rows or CFG.EXPORT_PARQUET_CHUNK_ROWS)
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    schema=_disk_schema(df)
    writer=pq.ParquetWriter(str(path),schema=schema,compression='zstd',use_dictionary=True)
    try:
        for i in range(0,len(df),chunk_rows):
            part=df.iloc[i:i+chunk_rows].copy()
            for field in schema:
                c=field.name
                if pa.types.is_string(field.type):
                    part[c]=part[c].map(lambda v:None if v is None or v is pd.NA else
                        json.dumps(v,ensure_ascii=False,default=str,sort_keys=True)
                        if isinstance(v,(dict,list,tuple,set)) else str(v))
                elif pa.types.is_integer(field.type):part[c]=pd.to_numeric(part[c],errors='coerce').astype('Int64')
                elif pa.types.is_floating(field.type):part[c]=pd.to_numeric(part[c],errors='coerce').astype(float)
                elif pa.types.is_boolean(field.type):part[c]=part[c].astype('boolean')
            writer.write_table(pa.Table.from_pandas(part,schema=schema,preserve_index=False,safe=True))
            del part
            if i and i%(chunk_rows*10)==0:progress_event('DISK',f'Parquet {path.name}',min(i+chunk_rows,len(df)),len(df),announce=False)
    finally:writer.close()
    return path

def qa_table(name,df,keys=None,allow_empty=False):
    if df is None:df=pd.DataFrame()
    if not isinstance(df,pd.DataFrame):fail('DATA_GATE',name+': ожидается DataFrame')
    if keys and not all(x in df for x in keys):fail('DATA_GATE',f'{name}: отсутствуют колонки ключа {keys}')
    duplicate_count=int(df.duplicated(subset=keys).sum()) if keys else None
    if duplicate_count:fail('DATA_GATE',f'{name}: дубли ключа {keys}: {duplicate_count}')
    if df.empty and not allow_empty:note('DATA_GATE',f'{name}: панель пуста','WARN')
    TABLE_QA.append({'table':name,'rows':int(len(df)),'columns':int(len(df.columns)),
                     'duplicate_keys':duplicate_count,'status':'PASS' if len(df)>0 else 'EMPTY',
                     'unique_keys':','.join(keys or []),
                     'storage':'PARQUET_FULL_PLUS_PREVIEW' if len(df)>CFG.PANEL_IN_MEMORY_MAX_ROWS else 'MEMORY'})
    if len(df)>CFG.PANEL_IN_MEMORY_MAX_ROWS:
        fp=write_df_chunks(df,ROOT/'derived'/'panels_full'/(name+'.parquet'))
        PANEL_FILES[name]=fp
        PANELS[name]=df.head(CFG.PANEL_PREVIEW_ROWS).copy()
        progress_event('DISK',f'{name}: {len(df):,} строк сохранено на диск; в ОЗУ только preview',announce=True)
    else:PANELS[name]=df
    return df

def limitation(module,reason,required=''):
    MODULES[module].update(status='NOT_FEASIBLE_FOR_PREDICTION',reason=reason)
    ISSUES.append({'module':module,'status':'NOT_FEASIBLE_FOR_PREDICTION','reason':reason,'required':required})

def diagnostic():
    mem='unknown'
    try:
        import psutil
        mem=round(psutil.virtual_memory().total/2**30,1)
    except Exception:pass
    info={'python':platform.python_version(),'cpu':os.cpu_count(),'ram_gib':mem,
          'disk_free_gib':round(shutil.disk_usage(str(ROOT)).free/2**30,1),
          'gpu_available': bool(shutil.which('nvidia-smi')),
          'versions':{}}
    note('ENV',json.dumps(info,ensure_ascii=False))
    print('Colab environment:',{k:v for k,v in info.items() if k!='versions'})
    return info
ENV=diagnostic()

# Прогресс V7.4.0: истинное число завершённых этапов и отдельно текущая операция.
# Этапы меняются только в mark_stage(). SQL/INPUT/DOWNLOAD/MEMORY — подоперации,
# поэтому они не сбрасывают таймер этапа и не подменяют его название.
import threading, html
from IPython.display import display, HTML
PIPELINE_STAGES=('INPUT','CATALOG_52','SEMANTICS','PANELS','FEATURES','CROSS_SOURCE','MODEL_DESIGN','UNIT_TESTS',
                 'SEVEN_MODULES','EXTENDED_MODULES','STATISTICS','VISUALS','EXPORT','QA_AND_PUBLICATION','DOWNLOAD_RESULTS')
_PROGRESS_LOCK=threading.RLock()
_PROGRESS_STOP=threading.Event()
_PROGRESS={
 'completed':[], 'phase':'BOOT', 'operation':'ENVIRONMENT', 'detail':'Подготовка окружения',
 'stage_started':time.monotonic(), 'operation_started':time.monotonic(),
 'current':None,'total':None,'last_render':0.,'last_log':0., 'last_log_text':''
}
try:
    _PROGRESS_HANDLE=display(HTML('<b>ML-05 V7.4.0 · старт</b>'),display_id=True)
except Exception:
    _PROGRESS_HANDLE=None

def _progress_html():
    with _PROGRESS_LOCK:
        s={k:list(v) if isinstance(v,list) else v for k,v in _PROGRESS.items()}
    elapsed=int(time.monotonic()-START_TIME)
    phase_seconds=int(time.monotonic()-s['stage_started'])
    operation_seconds=int(time.monotonic()-s['operation_started'])
    n=len(s['completed']); total_stages=len(PIPELINE_STAGES)
    phase_pct=100*n/total_stages
    current=s['current'];total=s['total']
    numeric= isinstance(current,(int,float)) and isinstance(total,(int,float)) and total>0
    unit_pct=min(100,max(0,100*current/total)) if numeric else None
    unit_label=(f'{int(current):,} / {int(total):,} · {unit_pct:.1f}%' if numeric else
                f'{int(current):,} обработано' if isinstance(current,(int,float)) else 'Ожидание завершения операции')
    # Никакой искусственной оценки процента во время fit/prediction/SHA без известного total.
    current_bar=(f'<div style="width:{unit_pct:.2f}%;height:11px;border-radius:8px;background:#0f9b83"></div>'
      if numeric else '<div style="width:30%;height:11px;border-radius:8px;background:#0f9b83;animation:ml05pulse 2.4s linear infinite"></div>')
    detail=html.escape(str(s['detail']))
    return f"""<div style="font-family:system-ui,Arial,sans-serif;max-width:980px;border:1px solid #cdd5df;border-radius:10px;padding:12px;line-height:1.45">
      <style>@keyframes ml05pulse {{0%{{transform:translateX(-110%)}}100%{{transform:translateX(430%)}}}}</style>
      <div style="display:flex;justify-content:space-between;gap:8px"><b>ML-05 V7.4.0 · Advanced Data Reconstruction</b><span>{elapsed//60:02d}:{elapsed%60:02d}</span></div>
      <div style="margin:9px 0 3px"><b>Этапы исследования</b> · {n} / {total_stages} · {phase_pct:.1f}%</div>
      <div style="background:#e4e9ef;border-radius:8px;height:12px;overflow:hidden"><div style="width:{phase_pct:.2f}%;height:12px;background:#2863b5"></div></div>
      <div style="margin:11px 0 3px"><b>Текущий этап: {html.escape(str(s['phase']))}</b> · {phase_seconds} с</div>
      <div><b>Операция {html.escape(str(s['operation']))}</b>: {detail}</div>
      <div style="background:#e4e9ef;border-radius:8px;height:11px;margin-top:7px;overflow:hidden">{current_bar}</div>
      <div style="font-size:12px;margin-top:5px;color:#65717e">{unit_label} · операция {operation_seconds} с. Известный объём — реальный процент; неизвестный — анимация без процента.</div>
    </div>"""

def progress_refresh(force=False):
    now=time.monotonic()
    with _PROGRESS_LOCK:
        if not force and now-_PROGRESS['last_render']<1.5:return
        _PROGRESS['last_render']=now
    if _PROGRESS_HANDLE is not None:
        try:_PROGRESS_HANDLE.update(HTML(_progress_html()))
        except Exception:pass

def progress_event(stage=None,detail=None,current=None,total=None,announce=True):
    now=time.monotonic()
    with _PROGRESS_LOCK:
        old_operation=_PROGRESS['operation']
        if stage in PIPELINE_STAGES:
            # Не возвращаться в предыдущий этап при nested helper calls.
            if PIPELINE_STAGES.index(stage)>=len(_PROGRESS['completed']):
                if stage!=_PROGRESS['phase']:
                    _PROGRESS['phase']=stage; _PROGRESS['stage_started']=now
        if stage and stage!=old_operation:
            _PROGRESS['operation']=stage
            _PROGRESS['operation_started']=now
        if detail is not None:
            _PROGRESS['detail']=str(detail)[:230]
        if current is not None:
            _PROGRESS['current']=int(current)
            _PROGRESS['total']=int(total) if total and total>0 else None
        elif stage!=old_operation or detail is not None:
            _PROGRESS['current']=None;_PROGRESS['total']=None
        line=f'[{datetime.now().strftime("%H:%M:%S")}] {_PROGRESS["phase"]}/{_PROGRESS["operation"]}: {_PROGRESS["detail"]}'
        if announce and (now-_PROGRESS['last_log']>=1.5 or stage in ('QA_AND_PUBLICATION','DOWNLOAD_RESULTS')):
            _PROGRESS['last_log']=now;_PROGRESS['last_log_text']=line
            print(line,flush=True)
            try:
                log_path=ROOT/'logs'/'live_progress.jsonl'
                with log_path.open('a',encoding='utf-8') as f:
                    f.write(json.dumps({'utc':datetime.now(timezone.utc).isoformat(),
                        'phase':_PROGRESS['phase'],'operation':_PROGRESS['operation'],
                        'detail':_PROGRESS['detail'],'completed':len(_PROGRESS['completed']),
                        'current':_PROGRESS['current'],'total':_PROGRESS['total']},ensure_ascii=False)+'\n')
            except (OSError,NameError):pass
    progress_refresh(force=announce)

def mark_stage(stage):
    if stage not in PIPELINE_STAGES:fail('PROGRESS','Неизвестный этап '+stage)
    with _PROGRESS_LOCK:
        if stage in _PROGRESS['completed']:
            print('Этап уже завершён:',stage,flush=True)
            return
        expected=PIPELINE_STAGES[len(_PROGRESS['completed'])]
        if stage!=expected:fail('PROGRESS',f'Нарушен порядок: ожидался {expected}, получено {stage}')
        _PROGRESS['completed'].append(stage)
        _entry={'run_session_id':RUN_SESSION_ID,'utc':stamp(),'stage':stage,
                'completed':len(_PROGRESS['completed']),
                'source_commit':str(globals().get('SOURCE_COMMIT','NOT_PINNED'))}
        _path=ROOT/'logs'/'stages_v74.jsonl'
        with _path.open('a',encoding='utf-8') as _lf:
            _lf.write(json.dumps(_entry,ensure_ascii=False)+'\n')
    progress_event(stage,f'Этап {stage} успешно завершён',announce=True)

class _ProgressBarFacade:
    def close(self):
        _PROGRESS_STOP.set();progress_refresh(force=True)
MAIN_PROGRESS=_ProgressBarFacade()

def _progress_heartbeat():
    # Обновляет только виджет, не засоряет логи строками [Пульс].
    while not _PROGRESS_STOP.wait(2):progress_refresh(force=True)
_HEARTBEAT_THREAD=threading.Thread(target=_progress_heartbeat,daemon=True,name='ml05_progress')
_HEARTBEAT_THREAD.start()
progress_event('ENVIRONMENT','Среда и зависимости подготовлены',announce=True)

# Лёгкая машинная проверка интеграционных инвариантов в самом ноутбуке.
def validate_table_contract(name,frame,mandatory=(),key=()):
    if not isinstance(frame,pd.DataFrame):fail('CONTRACT',f'{name} is not DataFrame')
    lost=set(mandatory)-set(frame.columns)
    if lost:fail('CONTRACT',f'{name}: отсутствуют обязательные поля {sorted(lost)}')
    if len(frame) and key and frame.duplicated(list(key)).any():fail('CONTRACT',f'{name}: дубли натурального ключа')
    return True

# Проверка дискового ParquetWriter до начала тяжёлой загрузки GitHub.
_probe_path=ROOT/'derived'/'_memory_writer_probe.parquet'
_probe_frame=pd.DataFrame({'region_code':pd.Series(['01000000','02000000'],dtype='string'),
    'test_year':pd.Series([2024,2025],dtype='Int64'),
    'measure':pd.Series([1.5,2.5],dtype=float),
    'period':pd.Series(['historical',2025],dtype=object)})
write_df_chunks(_probe_frame,_probe_path,chunk_rows=1)
_probe_pf=pq.ParquetFile(_probe_path)
assert _probe_pf.metadata.num_rows==2 and _probe_pf.schema_arrow.names==list(_probe_frame.columns)
_probe_read=pd.read_parquet(_probe_path)
assert _probe_read['region_code'].tolist()==['01000000','02000000']
assert _probe_read['test_year'].tolist()==[2024,2025]
assert _probe_read['period'].tolist()==['historical','2025']
del _probe_frame,_probe_pf,_probe_read
_probe_path.unlink(missing_ok=True)
progress_event('MEMORY','Тест потоковой записи Parquet и типов: PASS',announce=True)


In [ ]:
# ML-05 V7.4.0 · обязательный ранний тест SQL DuckDB до скачивания Stage I.
_adr_e02_query='''SELECT CAST(id AS VARCHAR) organization_id, TRY_CAST(year AS INTEGER) observation_year,
       CAST(indicator_name AS VARCHAR) indicator_name,CAST(indicator_unit AS VARCHAR) indicator_unit,
       CAST(area AS VARCHAR) education_area,CAST(specialty_code AS VARCHAR) specialty_code,
       TRY_CAST(indicator_value AS DOUBLE) observed_value
    FROM read_parquet(?) WHERE regexp_matches(lower(COALESCE(CAST(indicator_name AS VARCHAR),'')),
       'прием|приём|принят|зачисл|поступив')'''
_ADR_E02_SUMMARY_SQL = '''SELECT indicator_name,indicator_unit,
        COUNT(*)::BIGINT AS raw_rows,COUNT(DISTINCT organization_id)::BIGINT AS organizations,
        COUNT(DISTINCT observation_year)::BIGINT AS "years",
        COUNT(*) FILTER (WHERE observed_value IS NULL) AS invalid_numeric_rows,
        COUNT(*) FILTER (WHERE observed_value < 0 OR observed_value != FLOOR(observed_value)) AS invalid_count_rows,
        COUNT(DISTINCT COALESCE(education_area,'?')) AS areas,
        COUNT(DISTINCT COALESCE(specialty_code,'?')) AS specialty_codes
        FROM read_parquet(?) GROUP BY 1,2 ORDER BY raw_rows DESC'''

_ADR_E02_TOTAL_SQL = '''SELECT organization_id,observation_year,indicator_name,indicator_unit,
        COUNT(*)::BIGINT AS n_reporting_cells,
        COUNT(DISTINCT COALESCE(education_area,'?')) AS n_areas,
        COUNT(DISTINCT COALESCE(specialty_code,'?')) AS n_specialties,
        BOOL_AND(education_area IS NULL OR trim(CAST(education_area AS VARCHAR))='' OR
            regexp_matches(lower(CAST(education_area AS VARCHAR)),'^(всего|итого|total|all)$')) AS area_total,
        BOOL_AND(specialty_code IS NULL OR trim(CAST(specialty_code AS VARCHAR))='' OR
            regexp_matches(lower(CAST(specialty_code AS VARCHAR)),'^(всего|итого|total|all)$')) AS specialty_total,
        MIN(observed_value) AS min_count, MAX(observed_value) AS max_count
        FROM read_parquet(?) WHERE indicator_name IN (/*VALID_NAMES*/)
        GROUP BY 1,2,3,4'''

def _adr_e02_copy_statement(source, destination):
    """Build DuckDB COPY with paths SQL-quoted; the SELECT itself is preflight-tested."""
    quoted_source="'"+str(source).replace("'","''")+"'"
    quoted_target="'"+str(destination).replace("'","''")+"'"
    expected=_adr_e02_query.replace('read_parquet(?)','read_parquet('+quoted_source+')')
    if expected==_adr_e02_query:
        raise RuntimeError('ADR_SQL: source placeholder substitution failed')
    return f'COPY ({expected}) TO {quoted_target} (FORMAT PARQUET, COMPRESSION ZSTD)'

# Test EXACT production queries before downloading 2.25M E02 rows.
progress_event('SQL_PREFLIGHT','E02: SQL preflight на 3 искусственных строках (без Stage I)',0,4,announce=True)
_adr_fixture=ROOT/'derived'/'_adr_sql_preflight_input.parquet'
_adr_fixture_out=ROOT/'derived'/'_adr_sql_preflight_output.parquet'
_adr_dummy=pd.DataFrame([
    dict(id='0001',year=2022,indicator_name='Число принятых студентов',indicator_unit='чел',area='',specialty_code='',indicator_value=17),
    dict(id='0002',year=2023,indicator_name='Число зачисленных',indicator_unit='чел',area='Всего',specialty_code='Всего',indicator_value=19),
    dict(id='0003',year=2024,indicator_name='Укомплектованность',indicator_unit='чел',area='',specialty_code='',indicator_value=12)
])
try:
    _adr_dummy.to_parquet(_adr_fixture,index=False)
    progress_event('SQL_PREFLIGHT','E02 SQL: фильтрация и COPY',1,4,announce=False)
    with duck_connect() as _adr_con:
        _adr_con.execute(_adr_e02_copy_statement(_adr_fixture,_adr_fixture_out))
        progress_event('SQL_PREFLIGHT','E02 SQL: сводка по индикаторам',2,4,announce=False)
        _adr_fixture_summary=_adr_con.execute(_ADR_E02_SUMMARY_SQL,[str(_adr_fixture_out)]).df()
        if len(_adr_fixture_summary)!=2 or 'years' not in _adr_fixture_summary.columns:
            raise RuntimeError('ADR_SQL: candidate aggregation semantics mismatch')
        progress_event('SQL_PREFLIGHT','E02 SQL: проверка уникальных итоговых строк',3,4,announce=False)
        _sql=_ADR_E02_TOTAL_SQL.replace('/*VALID_NAMES*/',"'Число принятых студентов'")
        _adr_fixture_totals=_adr_con.execute(_sql,[str(_adr_fixture_out)]).df()
        if len(_adr_fixture_totals)!=1 or int(_adr_fixture_totals.iloc[0].n_reporting_cells)!=1:
            raise RuntimeError('ADR_SQL: organization-year aggregation semantics mismatch')
    progress_event('SQL_PREFLIGHT','E02 SQL preflight: PASS',4,4,announce=True)
finally:
    _adr_fixture.unlink(missing_ok=True)
    _adr_fixture_out.unlink(missing_ok=True)
    del _adr_dummy


SQL_PREFLIGHT_PASSED=True
print('E02 SQL: production templates COPY, SUMMARY and TOTAL — PASS before downloads')


# V7.4 · Проверка реального плана всех Release assets ДО дорогой загрузки источников.
def v74_release_asset_plan(index_file,chunk_dir,chunk_index,small_zip):
    parts=chunk_index.get('parts')
    if not isinstance(parts,list) or not parts:
        raise RuntimeError('V7.4: нет частей evidence')
    root=Path(chunk_dir)
    uploads=[(Path(index_file),'application/json')]
    for record in parts:
        name=str(record['name'])
        if Path(name).name!=name or not name.startswith('ML5_EVIDENCE.zip.part'):
            raise RuntimeError('V7.4: unsafe Release asset name: '+name)
        uploads.append((root/name,'application/octet-stream'))
    uploads.append((Path(small_zip),'application/zip'))
    paths=[p for p,_ in uploads]
    if len(set(p.name for p in paths))!=len(paths):
        raise RuntimeError('V7.4: repeated Release asset names')
    if not all(isinstance(p,Path) and p.is_file() and p.stat().st_size>0 for p in paths):
        raise RuntimeError('V7.4: Release plan has missing or empty files')
    return uploads

# Full 8-asset plan is exercised with tiny real files, not a static string inspection.
import tempfile as _v74_tempfile
with _v74_tempfile.TemporaryDirectory(prefix='ml5_v74_release_preflight_') as _v74_t:
    _v74_t=Path(_v74_t);(_v74_t/'chunks').mkdir()
    _v74_index=_v74_t/'ML5_V740_RELEASE_PARTS.json';_v74_index.write_text('{}')
    _v74_reports=_v74_t/'ML5_V740_ANALYTICS_test.zip';_v74_reports.write_bytes(b'PK-TEST')
    _v74_recs=[]
    for _v74_i in range(1,7):
        _v74_n=f'ML5_EVIDENCE.zip.part{_v74_i:03d}'
        (_v74_t/'chunks'/_v74_n).write_bytes(f'valid-chunk-{_v74_i}'.encode())
        _v74_recs.append({'name':_v74_n})
    _v74_probe=v74_release_asset_plan(_v74_index,_v74_t/'chunks',{'parts':_v74_recs},_v74_reports)
    assert len(_v74_probe)==8 and all(p.exists() for p,_ in _v74_probe)
    assert all(isinstance(p,Path) for p,_ in _v74_probe)
    try:
        v74_release_asset_plan(_v74_index,_v74_t/'chunks',{'parts':[{'name':'../escape'}]},_v74_reports)
        raise AssertionError('unsafe path accepted')
    except RuntimeError:pass
V74_PUBLICATION_PLAN_PREFLIGHT_PASSED=True
print('V7.4: PASS ранний тест 8 assets, Path, уникальности имён и защиты каталогов')


## 3. Приватный GitHub: доступ, неизменяемый вход и SHA-256
Токен читается только из Colab Secret `TRIAD-RU-Colab`; скрытого ручного ввода нет. Источники считываются из закреплённого Stage I; чужие сегменты не изменяются.

In [ ]:
progress_event('INPUT','Проверка токена и GitHub без изменения репозитория')

def github_token():
    try:
        from google.colab import userdata
        value = userdata.get('TRIAD-RU-Colab')
    except Exception as exc:
        raise RuntimeError('Нужен секрет Colab TRIAD-RU-Colab и разрешение доступу ноутбука (Secrets).') from exc
    if not isinstance(value,str) or not value.strip():
        raise RuntimeError('Пустой или недоступный Colab Secret: TRIAD-RU-Colab')
    return value.strip()

_TOKEN=github_token()
BASE='https://api.github.com/repos/'+CFG.REPO
SESS=requests.Session()
SESS.headers.update({'Authorization':'Bearer '+_TOKEN,'Accept':'application/vnd.github+json',
                     'X-GitHub-Api-Version':'2022-11-28'})

def api_call(method,url,*,headers=None,stream=False,timeout=(30,600),**kwargs):
    last=None
    for attempt in range(4):
        try:
            if attempt:progress_event('INPUT',f'Повтор GitHub API, попытка {attempt+1}/4',announce=True)
            r=SESS.request(method,url,headers=headers,stream=stream,timeout=timeout,**kwargs)
            if r.status_code in (429,500,502,503,504):
                last=r.status_code;r.close();time.sleep(2**attempt);continue
            if r.status_code in (401,403,404):
                status=r.status_code;r.close()
                raise PermissionError(f'GitHub HTTP {status}: права, доступ или объект отсутствуют; URL скрыт')
            if not r.ok:
                status=r.status_code;r.close()
                raise RuntimeError(f'GitHub HTTP {status}: отклонён запрос; секреты и ответ не выводятся')
            return r
        except PermissionError:raise
        except requests.RequestException:
            if attempt==3:fail('NETWORK','Ошибка запроса GitHub после повторных попыток')
            time.sleep(2**attempt)
    fail('NETWORK',f'GitHub ответил временной ошибкой {last}')

def api_json(method,url,**kw):
    r=api_call(method,url,**kw)
    try:return r.json()
    finally:r.close()

def repo_head(branch):
    p=api_json('GET',BASE+'/git/ref/heads/'+quote(branch,safe='/'))
    return p['object']['sha']

if CFG.PUBLISH_RESULTS_TO_PRIVATE_GITHUB:
    _preflight_repo=api_json('GET',BASE)
    if not _preflight_repo.get('private'):
        raise RuntimeError('WRITE_PREFLIGHT: репозиторий должен быть приватным')
    _rights=_preflight_repo.get('permissions') or {}
    if not (_rights.get('push') or _rights.get('admin') or _rights.get('maintain')):
        raise PermissionError('WRITE_PREFLIGHT: TRIAD-RU-Colab должен иметь Contents:write для приватного репозитория')
    progress_event('INPUT','GitHub: приватность и права на публикацию проверены ДО обучения',announce=True)
progress_event('INPUT','Фиксация HEAD исходного и целевого репозиториев')
SOURCE_COMMIT=repo_head(CFG.SOURCE_BRANCH)
TARGET_HEAD_AT_PREFLIGHT=repo_head(CFG.TARGET_BRANCH)
# Точная фиксация одного исходного снапшота Stage I.
def read_git_blob(path,ref=SOURCE_COMMIT,with_metadata=False):
    p=api_json('GET',BASE+'/contents/'+quote(path,safe='/'),params={'ref':ref})
    if p.get('type')!='file':fail('INPUT','Ожидался Git файл '+path)
    if p.get('content'):
        content=base64.b64decode(p['content'])
    else:
        # GitHub Contents для >1 MiB может не возвращать base64; /git/blobs гарантирует исходный blob.
        blob=api_json('GET',BASE+'/git/blobs/'+p['sha'])
        if blob.get('encoding')!='base64':fail('INPUT','Неизвестный тип Git blob')
        content=base64.b64decode(blob['content'])
    return (content,p) if with_metadata else content

progress_event('INPUT','Чтение dataset_index.json и проверка Git blob SHA')
index_bytes,index_info=read_git_blob(CFG.INPUT_INDEX,with_metadata=True)
if CFG.EXPECTED_INDEX_BLOB_SHA and index_info['sha']!=CFG.EXPECTED_INDEX_BLOB_SHA:
    fail('INPUT','Изменился Git blob каталога Stage I: требуется предметная проверка версии')
INDEX=json.loads(index_bytes)
if INDEX.get('repo')!=CFG.REPO or INDEX.get('version')!='7.2':
    fail('INPUT','Несовместимый каталог Stage I')
ASSETS=INDEX['assets']; ASSET_BY_ID={a['asset_id']:a for a in ASSETS}
if len(ASSET_BY_ID)!=len(ASSETS):fail('INPUT','Повторяющиеся asset_id')
for a in ASSETS:
    if not a.get('sha256_verified') or len(a.get('sha256',''))!=64:
        fail('INPUT','Каталог содержит непроверенный исходник '+a['asset_id'])
SOURCE_MANIFESTS={}
for k,v in INDEX['source_manifests'].items():
    progress_event('INPUT','Чтение манифеста '+str(k),announce=True)
    SOURCE_MANIFESTS[k]=json.loads(read_git_blob(v))
EXPECTED_COUNT=len(ASSETS)
INPUT_SHA={}

def download_stream(r,path):
    part=Path(str(path)+'.part');part.unlink(missing_ok=True)
    try:
        expected=int(r.headers.get('content-length') or 0)
        received=0;last_report=0.
        with open(part,'wb') as f:
            for block in r.iter_content(2*1024**2):
                if block:
                    f.write(block);received+=len(block)
                    if time.monotonic()-last_report>2:
                        progress_event('DOWNLOAD','Получение Release-asset',received,expected,announce=False)
                        last_report=time.monotonic()
        progress_event('DOWNLOAD','Release-asset получен',received,expected,announce=True)
        if part.stat().st_size==0:fail('INPUT','Получен пустой файл')
        part.replace(path)
    finally:
        r.close();part.unlink(missing_ok=True)

def fetch_release_asset(asset_id,path):
    u=BASE+'/releases/assets/'+str(asset_id)
    r=api_call('GET',u,headers={'Accept':'application/octet-stream'},stream=True,timeout=(30,900))
    content_type=r.headers.get('content-type','').lower()
    if 'json' in content_type:
        obj=r.json();r.close()
        if not obj.get('browser_download_url'):fail('INPUT','Не найден адрес закрытого Release-asset')
        r=api_call('GET',obj['browser_download_url'],stream=True,timeout=(30,900))
        if 'json' in r.headers.get('content-type','').lower():
            r.close();fail('INPUT','Private release-asset возвращает JSON вместо бинарного файла')
    download_stream(r,path)
    if path.stat().st_size==0:fail('INPUT','Скачан пустой Release-asset')
    return path

def fetch_asset(a):
    name=hashlib.sha256(a['asset_id'].encode()).hexdigest()[:20]+'.parquet'
    path=ROOT/'inputs'/name
    progress_event('INPUT',f'Исходный набор {a["asset_id"]}: проверка кэша и SHA',announce=True)
    if not path.is_file():
        for _previous_root in ('/content/ml5_v7_1_0','/content/ml5_v7_0_0'):
            _prior=Path(_previous_root)/'inputs'/name
            if not _prior.is_file():continue
            if hash_file(_prior)!=a['sha256']:continue
            try:os.link(_prior,path)  # same Colab storage, no duplicate 1GB download
            except OSError:shutil.copy2(_prior,path)
            progress_event('INPUT',f'SHA-confirmed Stage I cache reused: {a["asset_id"]}',announce=False)
            break
    # Fast restart after interrupted 7.1.x run: only SHA-verified local copies are reusable.
    if not path.exists():
        for old_root in ('/content/ml5_v7_1_1/inputs','/content/ml5_v7_1_0/inputs'):
            old=Path(old_root)/name
            if old.is_file() and hash_file(old)==a['sha256']:
                try:os.link(old,path)
                except OSError:shutil.copy2(old,path)
                break
    if path.exists() and hash_file(path)==a['sha256']:
        INPUT_SHA[a['asset_id']]=a['sha256'];return path
    path.unlink(missing_ok=True)
    if shutil.disk_usage(str(ROOT)).free < CFG.STAGE1_DISK_RESERVE_GIB*2**30:
        fail('INPUT',f'Недостаточно свободного диска: требуется резерв {CFG.STAGE1_DISK_RESERVE_GIB} GiB; можно подключить дополнительный диск или увеличить доступное место')
    loc=a['locator']
    if loc['type']=='git':
        # Вся загрузка закреплена SOURCE_COMMIT, а не плавающим main.
        progress_event('DOWNLOAD',f'Git blob {a["asset_id"]}',announce=True)
        data=read_git_blob(loc['path'])
        path.write_bytes(data)
    elif loc['type']=='release':
        progress_event('DOWNLOAD',f'Release asset {a["asset_id"]}',announce=True)
        fetch_release_asset(loc['asset_id'],path)
    else:fail('INPUT',f"Неподдерживаемый locator {loc['type']}")
    if hash_file(path)!=a['sha256']:
        path.unlink(missing_ok=True);fail('INPUT','SHA-256 не совпал '+a['asset_id'])
    INPUT_SHA[a['asset_id']]=a['sha256']
    progress_event('INPUT',f'Проверено {a["asset_id"]} · {path.stat().st_size/2**20:.1f} МиБ',announce=True)
    total=sum((ROOT/'inputs'/f).stat().st_size for f in os.listdir(ROOT/'inputs') if (ROOT/'inputs'/f).is_file())
    if total/2**30>CFG.MAX_REMOTE_INPUT_GIB:fail('INPUT','Лимит локального кэша Stage I превышен')
    return path

def original_spo_workbook():
    x=[a for a in INDEX.get('extra_originals',[]) if a.get('role')=='migration_matrix_original_xlsx']
    if len(x)!=1:return None
    path=ROOT/'inputs'/'spo_original_e01.xlsx'
    if not path.exists():fetch_release_asset(x[0]['locator']['asset_id'],path)
    # В Stage I для оригинала нет SHA в dataset_index: это не равно подтверждению контрольной суммы.
    return path

def source_asset(segment=None,role=None,source_id=None,prefix=None):
    return [a for a in ASSETS if (segment is None or a['segment']==segment)
            and (role is None or a.get('role')==role)
            and (source_id is None or a.get('source_id')==source_id)
            and (prefix is None or a['asset_id'].startswith(prefix))]

def required_one(items,label):
    if len(items)!=1:fail('INPUT',f'{label}: ожидается 1 источник, найдено {len(items)}')
    return items[0]

def read_table(a,columns=None,where=None,limit=None):
    progress_event('SQL',f'Чтение {a["asset_id"]}, фильтр={bool(where)}, limit={limit}',announce=True)
    path=fetch_asset(a)
    cols=a.get('columns_manifest') or list(pq.ParquetFile(path).schema_arrow.names)
    if columns:
        missing=set(columns)-set(cols)
        if missing:fail('SCHEMA',f'{a["asset_id"]}: отсутствуют {sorted(missing)}')
        selected=','.join('"'+c.replace('"','""')+'"' for c in columns)
    else:selected='*'
    sql=f"SELECT {selected} FROM read_parquet(?)"
    if where:sql+=' WHERE '+where
    if limit:sql+=f' LIMIT {int(limit)}'
    con=duck_connect()
    try:
        if limit is None and int(a.get('rows_actual') or 0)>CFG.MAX_PANDAS_ROWS:
            rows=int(con.execute('SELECT COUNT(*) FROM ('+sql+') AS _guard',[str(path)]).fetchone()[0])
            if rows>CFG.MAX_PANDAS_ROWS:
                fail('RAM_GUARD',f'{a["asset_id"]}: запрос даёт {rows:,} строк; запрещено .df() в FULL, используйте DuckDB COPY/агрегацию')
        result=con.execute(sql,[str(path)]).df()
        progress_event('SQL',f'{a["asset_id"]}: получено {len(result):,} строк',announce=True)
        return result
    finally:con.close()

def sql_query(a,query):
    progress_event('SQL',f'Агрегация {a["asset_id"]}',announce=True)
    path=fetch_asset(a)
    con=duck_connect()
    try:
        result=con.execute(query,[str(path)]).df()
        progress_event('SQL',f'Агрегация {a["asset_id"]}: {len(result):,} строк',announce=True)
        return result
    finally:con.close()

stage1_sources=pd.DataFrame([{'asset_id':a['asset_id'],'segment':a['segment'],'source_id':a['source_id'],
 'role':a.get('role',''),'registered_rows':a.get('rows_actual'), 'locator':a['locator']['type'],
 'sha256':a['sha256'], 'columns':','.join(a.get('columns_manifest',[]))[:500]} for a in ASSETS])
qa_table('stage1_sources',stage1_sources,['asset_id'])
source_ids=set(a.get('source_id') for a in ASSETS)
for missing_id in ('D03','D06'):
    if missing_id not in source_ids:
        ISSUES.append({'module':'STAGE1','status':'SOURCE_NOT_INGESTED',
                       'reason':f'В каталоге нет {missing_id}; нельзя подменять внешними данными','required':missing_id})
note('INPUT',f'pin {SOURCE_COMMIT}; catalog {index_info["sha"]}; {EXPECTED_COUNT} assets')
print('INPUT preflight:', len(ASSETS),'objects; git commit',SOURCE_COMMIT[:12],'; no writes')

mark_stage('INPUT')


## 3А. Полная инвентаризация Stage I — 52/52

В режиме FULL каждое зарегистрированное Parquet скачивается, проверяется по SHA-256, схеме и числу строк, затем построчно оценивается годовой охват без переноса гигантских таблиц в Pandas. Единица анализа при этом сохраняется: **проверка файла не приписывает его значения образовательной миграции**. Годовые shards СПО рассматриваются как альтернативные представления, а не добавочные наблюдения.

In [ ]:
# Stage I FULL52 — strict SHA+Parquet+content profiles, disk-conscious, resumable download cache.
progress_event('CATALOG_52','Проверка ВСЕХ файлов Stage I, SHA, схем, лет и строк',announce=True)
FULL52_INVENTORY=[];FULL52_YEAR_STATS=[];FULL52_THEME_STATS=[];FULL52_ORIGINALS=[]

def _catalog_family(a):
    x=a['asset_id'];seg=a.get('segment','')
    if seg=='01':return 'MUNICIPAL_DEMOGRAPHY'
    if seg=='02':return 'MUNICIPAL_ECONOMY_LABOUR_EDUCATION_TAX'
    if seg=='03':
        if a.get('role')=='education_od_matrix_raw_cells':return 'SPO_ORIGINAL_OD_CELLS_NOT_ADDITIVE'
        if a.get('role')=='organization_indicators':return 'SPO_ALL_YEARS_OR_YEAR_SHARD_NOT_ADDITIVE'
        return 'SPO_REGISTRY_OD_REGIONAL'
    if seg=='04':return 'HIGHER_EDUCATION_VPO'
    if seg=='05':return 'GRADUATE_LABOUR_OUTCOMES'
    if seg=='06':return 'MUNICIPAL_SPATIAL_BDMO_TRANSPORT'
    return 'UNKNOWN'

def _source_year_scan(a,p,cols):
    key=a.get('time_key')
    if key not in cols:key=next((c for c in ('year','observation_year','graduation_year') if c in cols),None)
    if not key:return [],'NO_TIME_KEY'
    column='"'+key.replace('"','""')+'"'
    try:
        with duck_connect() as con:
            years=con.execute(f'''SELECT CAST({column} AS VARCHAR) observed_year,COUNT(*)::BIGINT n
                FROM read_parquet(?) GROUP BY 1 ORDER BY 1''',[str(p)]).fetchall()
        return [{'asset_id':a['asset_id'],'year_label':str(y),'rows':int(n),'time_column':key} for y,n in years],'SCANNED_FULL'
    except (duckdb.OutOfMemoryException,duckdb.IOException) as ex:
        ISSUES.append({'module':'SOURCE_AUDIT','status':'TIME_DISTRIBUTION_NOT_AVAILABLE',
                       'reason':a['asset_id']+' '+type(ex).__name__,'required':'more spill memory for time scan'})
        return [],'YEAR_SCAN_UNAVAILABLE'

for i,a in enumerate(ASSETS,1):
    aid=a['asset_id']
    if CFG.MODE!='FULL' and aid not in {'01::0','03::42f47d3cf2159992::000','03::89505d5993e6f4ad::000'}:
        FULL52_INVENTORY.append({'asset_id':aid,'segment':a.get('segment'),'source_id':a.get('source_id'),
           'research_family':_catalog_family(a),'observed_rows':None,'expected_rows':a.get('rows_actual'),
           'physical_read_ok':False,'footer_rows_ok':False,'sha256_ok':False,
           'years_scanned':0,'year_scan_status':'NOT_SCANNED_IN_PILOT',
           'source_sha256':a['sha256'],'level_gate':'PILOT_NOT_FULL'})
        continue
    progress_event('CATALOG_52',f'[{i:02d}/52] {aid}: SHA + контроль Parquet',i,len(ASSETS),announce=True)
    source=fetch_asset(a)  # verifies SHA both fresh and cached; never silently skipped
    footer=pq.ParquetFile(source)
    cols=list(footer.schema_arrow.names);registered=int(a.get('rows_actual') or 0)
    found=int(footer.metadata.num_rows)
    if found!=registered:fail('FULL52',f'{aid}: rows {found:,} != Stage I {registered:,}')
    manifest_cols=list(a.get('columns_manifest') or [])
    if manifest_cols and set(cols)!=set(manifest_cols):fail('FULL52',f'{aid}: Parquet fields != Stage I catalog')
    # Content touch: inspect a physical row through a LIMIT, not whole table in pandas.
    with duck_connect() as con:
        touched=int(con.execute('SELECT COUNT(*) FROM (SELECT 1 FROM read_parquet(?) LIMIT 1)',[str(source)]).fetchone()[0])
    if touched!=int(found>0):fail('FULL52',f'{aid}: нельзя прочитать содержимое Parquet')
    yr,yr_status=_source_year_scan(a,source,cols) if CFG.CATALOG_SCAN_ALL_YEARS else ([], 'SCAN_DISABLED')
    if yr and sum(r['rows'] for r in yr)!=found:fail('FULL52',f'{aid}: сумма годового охвата не совпадает с footer')
    FULL52_YEAR_STATS.extend(yr)
    family=_catalog_family(a)
    FULL52_INVENTORY.append({'asset_id':aid,'segment':a.get('segment'),'source_id':a.get('source_id'),
      'data_role':a.get('role') or a.get('data_role'),'research_family':family,'observed_rows':found,
      'expected_rows':registered,'columns':len(cols),'bytes':source.stat().st_size,
      'physical_read_ok':True,'footer_rows_ok':found==registered,'sha256_ok':INPUT_SHA.get(aid)==a['sha256'],
      'years_scanned':len(yr),'year_scan_status':yr_status,'first_year':yr[0]['year_label'] if yr else None,
      'last_year':yr[-1]['year_label'] if yr else None,'source_sha256':a['sha256'],
      'original_grain_status':a.get('grain_status'),'level_gate':'ORIGINAL_GRAIN_NOT_ASSUMED_ADDITIVE'})
    del footer
    if i%4==0:gc.collect();memory_point(f'catalog_{i}_of_{len(ASSETS)}')

if CFG.MODE=='FULL' and (len(FULL52_INVENTORY)!=len(ASSETS) or len(INPUT_SHA)!=len(ASSETS)):
    fail('FULL52','Неполное чтение/контроль входных файлов')
# Role-specific research trail: every source has a declared analytic role, with no double-counting.
for fam,grp in pd.DataFrame(FULL52_INVENTORY).groupby('research_family'):
    FULL52_THEME_STATS.append({'family':fam,'n_assets':len(grp), 'source_rows_scanned_by_footer':int(grp.observed_rows.sum()),
      'interpretation':'ALT_REPRESENTATIONS_NOT_ADDITIVE' if 'SHARD' in fam or 'CELLS' in fam else 'OWN_GRAIN',
      'sha_verified':int(grp.sha256_ok.sum())})
qa_table('full52_source_inventory',pd.DataFrame(FULL52_INVENTORY),['asset_id'])
qa_table('full52_year_coverage',pd.DataFrame(FULL52_YEAR_STATS),allow_empty=True)
qa_table('full52_theme_coverage',pd.DataFrame(FULL52_THEME_STATS),['family'])
# Original SPO workbook is outside the 52 Parquet assets. Record accessibility separately.
try:
    ori=original_spo_workbook()
    if ori:
        FULL52_ORIGINALS=[{'name':ori.name,'bytes':ori.stat().st_size,'sha256_locally_calculated':hash_file(ori),
          'registered_in_52':False,'status':'CONTENT_AVAILABLE_METHODOLOGY_NOT_YET_IDENTIFIED'}]
except Exception as ex:
    FULL52_ORIGINALS=[{'name':'original_spo_workbook','status':'NOT_AVAILABLE','reason':type(ex).__name__}]
qa_table('full52_raw_originals',pd.DataFrame(FULL52_ORIGINALS),allow_empty=True)
print('CATALOG 52:',len(INPUT_SHA),'SHA verified;',sum(t['observed_rows'] or 0 for t in FULL52_INVENTORY),
      'Parquet rows across files (NOT unique observations);',len(FULL52_YEAR_STATS),'asset-year groups')
mark_stage('CATALOG_52')


## 4. Измерение и определения исходных величин
Для E01 сохраняются фактические заголовки оригинального XLSX и исходных raw-таблиц. Нельзя автоматически называть показателем миграции населения или числом зачисленных наблюдение об образовании и текущем обучении.

In [ ]:
memory_point('before_semantics')
progress_event('SEMANTICS','Аудит определений E01/E02/ВПО/СПО')
# Семантика извлекается из значений исходных таблиц, а не из догадки по названию файла.
E01=required_one(source_asset(role='education_od_flows'),'E01 OD')
SPO_ORG=required_one(source_asset(role='education_organizations'),'E01 organizations')
SPO_IND=sorted(source_asset(segment='03',role='organization_indicators'),
               key=lambda a:0 if 'all_years' in a.get('member','') else 1)
SPO_MASTER=next((a for a in SPO_IND if 'all_years' in a.get('member','')),None)
SPO_REG=required_one(source_asset(role='regional_spo'),'SPO regional indicators')
E02=required_one(source_asset(prefix='04::E02'),'E02 universities')
VPO_211=required_one(source_asset(prefix='04::VPO_2_1_1'),'VPO-1 admissions table')
VPO_213=required_one(source_asset(prefix='04::VPO_2_1_3'),'VPO-1 graduates table')
GRAD_ORG=required_one(source_asset(prefix='05::E04::organization_total'),'E04 graduates')
GRAD_FIELD=required_one(source_asset(prefix='05::E05::field_specialty'),'E05 specialties')
MUN=required_one(source_asset(segment='01'),'Municipal demography')
TRANSPORT=required_one(source_asset(prefix='06::TRANSPORT'),'transport graph')
SPATIAL=required_one(source_asset(prefix='06::D04::D04_member00::spatial_reference'),'geography reference')

# Для FULL/PILOT10 производим сканирование всех строк выбранных источников колонночным движком.
def indicator_catalog(a,fields,cap=None):
    cols=[c for c in fields if c in a.get('columns_manifest',[])]
    if 'indicator_name' not in cols:fail('SEMANTICS','Нет колонки indicator_name: '+a['asset_id'])
    progress_event('SEMANTICS',f'Сканирование словаря {a["asset_id"]}',announce=True)
    path=fetch_asset(a)
    with duck_connect() as con:
        # В SMOKE ограничиваем ввод — это не окончательный словарь.
        sample=f' LIMIT {CFG.MAX_READ_ROWS_SMOKE}' if CFG.MODE=='SMOKE' else ''
        group=','.join('"'+c+'"' for c in cols)
        sql=f'SELECT {group}, count(*)::BIGINT as n FROM (SELECT {group} FROM read_parquet(?) {sample}) GROUP BY {group} ORDER BY n DESC'
        d=con.execute(sql,[str(path)]).df()
    d['asset_id']=a['asset_id'];d['catalog_exhaustive']=CFG.MODE!='SMOKE'
    progress_event('SEMANTICS',f'Словарь готов: {a["asset_id"]} · {len(d):,} записей',announce=True)
    return d

CAT_E02=indicator_catalog(E02,['indicator_name','indicator_unit','area'])
CAT_VPO=indicator_catalog(VPO_211,['indicator_name','admission_type','education_level','education_form'])
CAT_SPO=indicator_catalog(SPO_MASTER,['indicator_name','indicator_unit','indicator_section']) if SPO_MASTER else pd.DataFrame()
CAT_SPO_REG=indicator_catalog(SPO_REG,['indicator_name','indicator_unit','indicator_section'])
CAT_VPO_GRAD=indicator_catalog(VPO_213,['indicator_name','education_level','education_form'])
for name,tab in [('dictionary_e02',CAT_E02),('dictionary_vpo_admission',CAT_VPO),
                 ('dictionary_spo',CAT_SPO),('dictionary_spo_regional',CAT_SPO_REG),
                 ('dictionary_vpo_graduation',CAT_VPO_GRAD)]:qa_table(name,tab,allow_empty=True)

POS_ADMISSION=re.compile(r'(?i)(?:^|\W)(?:при[её]м|принят|зачисл|поступил|поступивш|принятых)(?:\w|\W|$)')
NEG_ADMISSION=re.compile(r'(?i)(?:заявлен|подавш|конкурс|при[её]мн\w* комисс|при[её]м на работу|планируем|места приема|контрольн\w+ цифр|ваканс)')
POS_GRAD=re.compile(r'(?i)(выпуск|окончивш|завершивш)')
NEG_TOTAL=re.compile(r'(?i)(процент|доля|удельн|на 100|отношение|коэффициент|темп|индекс|стоимост|тыс\. руб|млн)')

# Утверждение требует положительного семантического определения, единицы и отсутствия разрезов-двойников.
def classify_indicator(name,unit=None,category=None):
    value='' if pd.isna(name) else str(name).strip()
    u='' if pd.isna(unit) else str(unit).strip().lower()
    cat='' if pd.isna(category) else str(category)
    if POS_ADMISSION.search(value) and not NEG_ADMISSION.search(value) and not NEG_TOTAL.search(value):
        role='ADMISSION_CANDIDATE'
    elif POS_GRAD.search(value) and not NEG_TOTAL.search(value):role='GRADUATION_CANDIDATE'
    else:role='OTHER'
    unit_ok=not u or bool(re.search(r'чел|человек|единиц|численност|шт',u))
    # Автоматическая классификация создаёт кандидатов, но НЕ утверждает научную семантику.
    grade='ELIGIBLE_FOR_ROW_CHECK' if role!='OTHER' and unit_ok else 'QUARANTINE'
    if role!='OTHER' and not unit_ok:grade='UNEXPECTED_UNIT'
    return {'semantic_role':role,'semantic_gate':grade,
            'name_exact':value,'unit_exact':u,'category_exact':cat}

CANDIDATES=[]
for source,frame in [('E02',CAT_E02),('VPO_2_1_1',CAT_VPO),('SPO_ORG',CAT_SPO),('SPO_REG',CAT_SPO_REG),('VPO_GRAD',CAT_VPO_GRAD)]:
    for _,r in frame.iterrows():
        d=classify_indicator(r.get('indicator_name'),r.get('indicator_unit'),r.get('admission_type'))
        CANDIDATES.append({'source':source,'indicator_name':r.get('indicator_name'),'rows':int(r['n']),
                           'level':r.get('education_level'), 'form':r.get('education_form'),
                           'admission_type':r.get('admission_type'), 'unit':r.get('indicator_unit'),
                           'indicator_section':r.get('indicator_section'),'area':r.get('area'),**d})
CANDIDATE_CATALOG=qa_table('semantic_indicator_catalog',pd.DataFrame(CANDIDATES),allow_empty=True)

OD_SEMANTIC={
 'source_id':'E01','origin_definition':'region_of_previous_general_education',
 'destination_definition':'region_of_current_vocational_study',
 'label':'observed_E01_schooling_to_study_value',
 'admissions_confirmed':False,'permanent_migration_confirmed':False,
 'yearly_count_unit_confirmed':False,'unit':'reported_indicator_value',
 'methodological_status':'SOURCE_DESCRIPTION_REQUIRES_PRIMARY_CELL_VERIFICATION',
 'statistics_available_asof':'2026_SOURCE_VINTAGE_ONLY',
 'admission_target_gate':'NOT_CONFIRMED',
 'source_page':'https://tochno.st/datasets/monitoring_spo',
 'primary_workbook':'Stage I extra_originals/migration_matrix_original_xlsx'
}
ORIGINAL_SHEETS=[]
try:
    progress_event('SEMANTICS','Чтение первичного XLSX E01',announce=True)
    p=original_spo_workbook()
    if p:
        xf=pd.ExcelFile(p,engine='calamine')
        for sheet in xf.sheet_names:
            raw=pd.read_excel(p,sheet_name=sheet,header=None,nrows=25,engine='calamine')
            cells=' | '.join(str(v)[:100] for v in raw.to_numpy().reshape(-1)
                           if not pd.isna(v) and str(v).strip())[:2500]
            ORIGINAL_SHEETS.append({'sheet':sheet,'first_cells':cells,'file_sha256':hash_file(p)})
        OD_SEMANTIC['original_opened']=True
except Exception as ex:
    OD_SEMANTIC['original_opened']=False
    ISSUES.append({'module':'ML-5.1','status':'PRIMARY_SOURCE_REVIEW_PENDING',
                   'reason':'Оригинальный XLSX E01 не открыт; ошибка '+type(ex).__name__,
                   'required':'original E01 workbook access'})
qa_table('e01_original_headers',pd.DataFrame(ORIGINAL_SHEETS),allow_empty=True)
# Даже наличие слова «миграция» в заголовке не доказывает, что показатель — поток зачисления.
if not OD_SEMANTIC.get('original_opened'):
    note('SEM-E01','Оригинал не открыт: допустим ограниченный анализ наблюдаемой величины','WARN')
else:
    note('SEM-E01','Оригинальные заголовки записаны; семантика admission пока не подтверждена')

RAW_SPO_CELL_AUDIT=[]
for a in source_asset(role='education_od_matrix_raw_cells'):
    try:
        p=fetch_asset(a);file=pq.ParquetFile(p)
        RAW_SPO_CELL_AUDIT.append({'asset_id':a['asset_id'],'rows':file.metadata.num_rows,
                                   'columns':','.join(file.schema_arrow.names),'sha256':a['sha256']})
    except Exception as ex:
        RAW_SPO_CELL_AUDIT.append({'asset_id':a['asset_id'],'error':type(ex).__name__})
qa_table('e01_raw_cell_sources',pd.DataFrame(RAW_SPO_CELL_AUDIT),allow_empty=True)

# Проверка ручного списка после чтения фактических словарей; не считать пустоту списка отсутствием исходных данных.
def approved_names(source):
    manually={'E02':CFG.APPROVED_VO_ADMISSION_NAMES,'SPO_ORG':CFG.APPROVED_SPO_ADMISSION_NAMES}
    names=tuple(manually.get(source,()))
    existing=set(CANDIDATE_CATALOG.loc[CANDIDATE_CATALOG.source.eq(source),'indicator_name'].dropna().astype(str))
    bad=set(names)-existing
    if bad:fail('SEMANTICS',f'{source}: утверждённые имена не найдены в данных: {sorted(bad)}')
    return sorted(set(names))

SEMANTIC_APPROVAL=pd.DataFrame([{'source':src,'indicator_name':n,'approved_for_admission':n in approved_names(src),
    'semantic_role':r,'unit':u} for src in ('E02','SPO_ORG')
    for n,r,u in CANDIDATE_CATALOG.loc[CANDIDATE_CATALOG.source.eq(src),
        ['indicator_name','semantic_role','unit']].drop_duplicates().itertuples(index=False,name=None)])
qa_table('admission_approval_register',SEMANTIC_APPROVAL,allow_empty=True)
print('Semantic candidates:',len(CANDIDATE_CATALOG),'| E01 original sheets:',len(ORIGINAL_SHEETS),
      '| SPO raw-cell files:',len(RAW_SPO_CELL_AUDIT))

mark_stage('SEMANTICS')


## 5. Исследовательские панели и контролируемые сопоставления
Статистические уровни ВО, СПО, организация, филиал, регион и муниципалитет не смешиваются. Спорные организационные ID исключаются в карантин, ноль отличается от отсутствия.

In [ ]:
memory_point('before_panels')
progress_event('PANELS','Исходные панели: E01, организации, ВПО, E04/E05, муниципалитеты')
# Шесть исходных исследовательских панелей и независимые таблицы ошибок сопоставления.
OD_RAW=read_table(E01,['year','origin_region_code','origin_region_name',
                       'destination_region_code','destination_region_name','indicator_value'])
OD_RAW=OD_RAW.rename(columns={'origin_region_code':'origin_region','destination_region_code':'destination_region',
                              'indicator_value':'observed_value','year':'observation_year'})
OD_RAW['origin_region']=norm_code(OD_RAW['origin_region'])
OD_RAW['destination_region']=norm_code(OD_RAW['destination_region'])
OD_RAW['observation_year']=safe_year(OD_RAW['observation_year'])
OD_RAW['observed_value']=number(OD_RAW['observed_value'])
OD_RAW['sector']='SPO';OD_RAW['origin_definition']='previous_general_education_region'
OD_RAW['destination_definition']='current_study_region'
OD_RAW['target_definition']=OD_SEMANTIC['label']
OD_RAW['known_origin']=nonmissing_code(OD_RAW.origin_region)
OD_RAW['known_destination']=nonmissing_code(OD_RAW.destination_region)
OD_RAW['code_status']=np.select([(~OD_RAW.known_origin|~OD_RAW.known_destination).fillna(True).to_numpy(dtype=bool),
                                OD_RAW.origin_region.eq(OD_RAW.destination_region).fillna(False).to_numpy(dtype=bool)],
                                ['UNKNOWN_CODE','INTRA_REGION'],default='INTER_REGION')
if OD_RAW['observed_value'].lt(0).any():fail('SEM-E01','Отрицательные величины E01: нужна особая интерпретация')
if OD_RAW['observed_value'].isna().any():
    QUARANTINE.extend([{'table':'edu_od_year','reason':'missing_value','row':int(i)}
                       for i in OD_RAW.index[OD_RAW.observed_value.isna()].tolist()[:100]])
OD_RAW['explicit_zero']=OD_RAW.observed_value.eq(0)&OD_RAW.observed_value.notna()
OD_RAW['missing_edge']=False;OD_RAW['suppressed']=False
OD_RAW['source_asset_id']=E01['asset_id'];OD_RAW['source_vintage']=INDEX['created_at']
# Год может быть статистическим периодом учёбы; не переименовывать в admission_year.
OD=qa_table('edu_od_year',OD_RAW,
            ['observation_year','origin_region','destination_region'])
OD_INTER=qa_table('edu_od_interregional',OD[OD.code_status=='INTER_REGION'].copy(),
              ['observation_year','origin_region','destination_region'],allow_empty=True)
OD_INTRA=qa_table('edu_od_intraregional',OD[OD.code_status=='INTRA_REGION'].copy(),allow_empty=True)
OD_UNKNOWN=qa_table('edu_od_unknown_code',OD[OD.code_status=='UNKNOWN_CODE'].copy(),allow_empty=True)
if sum(map(len,[OD_INTER,OD_INTRA,OD_UNKNOWN]))!=len(OD):fail('DATA_GATE','OD: категории не исчерпывают данные')

counts=OD.groupby(['observation_year','code_status'],dropna=False).agg(
    rows=('observed_value','size'),total_value=('observed_value','sum')).reset_index()
qa_table('od_composition',counts,allow_empty=True)
EDGES=OD_INTER[['observation_year','origin_region','destination_region','observed_value']].copy()
EDGES['edge_id']=EDGES.origin_region.astype(str)+'|'+EDGES.destination_region.astype(str)
edge_years={int(y):set(g.edge_id) for y,g in EDGES.groupby('observation_year')}
coverage=[];transitions=[]
for y in sorted(edge_years):
    keys=edge_years[y];past=edge_years.get(y-1,set())
    coverage.append({'year':y,'observed_edges':len(keys),'new_vs_previous':len(keys-past) if past else pd.NA,
                     'lost_vs_previous':len(past-keys) if past else pd.NA,
                     'intersection_vs_previous':len(keys&past) if past else pd.NA,
                     'jaccard_vs_previous':len(keys&past)/len(keys|past) if past and (keys|past) else np.nan,
                     'total_observed_value':float(EDGES.loc[EDGES.observation_year==y,'observed_value'].sum())})
    if past:
        transitions.extend([{'year':y,'edge_id':e,'transition':'NEW'} for e in sorted(keys-past)])
        transitions.extend([{'year':y,'edge_id':e,'transition':'NOT_REPORTED'} for e in sorted(past-keys)])
coverage_df=qa_table('od_edge_coverage',pd.DataFrame(coverage),['year'])
qa_table('od_edge_transitions',pd.DataFrame(transitions),allow_empty=True)
if len(coverage_df)>1:
    coverage_df['edge_growth_ratio']=coverage_df.observed_edges/coverage_df.observed_edges.shift(1)
    coverage_df['potential_break']=coverage_df.edge_growth_ratio.gt(1.5) | coverage_df.edge_growth_ratio.lt(0.67)
    if coverage_df.potential_break.any():
        ISSUES.append({'module':'SEM-E01','status':'COVERAGE_BREAK_UNRESOLVED',
         'reason':'Изменение регистрации OD-направлений: прямое межгодовое сравнение требует аудита полноты',
         'required':'Единые определения и покрытие матрицы по годам'})

# Реестр ВО из E02: комбинация исторического года, организационного id и географического положения.
# E02 длинная таблица показателей (2.25 млн строк): реестр читаем DISTINCT в DuckDB,
# не загружая миллионы повторяющихся идентификаторов в оперативную память.
with duck_connect() as con:
    progress_event('PANELS','E02: DISTINCT организаций из полной таблицы, без переноса 2.25 млн строк в pandas',announce=True)
    vo=con.execute('''SELECT DISTINCT id,name,inn,ogrn,region,oktmo,institution,parent_id,parent_name,year
                     FROM read_parquet(?)''',[str(fetch_asset(E02))]).df()
if CFG.MODE=='SMOKE':vo=vo.head(CFG.MAX_READ_ROWS_SMOKE).copy()
for c in ['id','parent_id','oktmo','inn','ogrn']:vo[c]=norm_code(vo[c])
vo['observation_year']=safe_year(vo['year'])
vo=vo.drop(columns=['year'])
vo=vo.loc[vo.id.notna()&vo.observation_year.notna()].drop_duplicates().copy()
vo['sector']='VO';vo['organization_id']=vo['id']
vo['campus_id']=vo['id'] # Только при явном собственном id в исходнике; головной id не переносить на филиал.
vo['parent_group_id']=vo.parent_id.fillna(vo.organization_id)
vo['entity_id']='VO|'+vo.organization_id.astype(str)
vo['campus_geo_status']=np.where(nonmissing_code(vo.oktmo),'REPORTING_OKTMO_NOT_VERIFIED_CAMPUS','UNKNOWN')
vo['entity_match_confidence']=1.0

sporeg=read_table(SPO_ORG,['year','region_name','region_oktmo','id','name','city','lat','lon','type','head','address','branches'])
sporeg['observation_year']=safe_year(sporeg.year);sporeg=sporeg.drop(columns=['year'])
sporeg['id']=norm_code(sporeg.id);sporeg['region_oktmo']=norm_code(sporeg.region_oktmo)
sporeg['sector']='SPO';sporeg['organization_id']=sporeg.id
sporeg['entity_id']='SPO|'+sporeg.id.fillna('UNRESOLVED')
sporeg['campus_geo_status']='POINT_WITHOUT_MUNICIPAL_OKTMO'
sporeg['entity_match_confidence']=np.where(sporeg.id.notna(),1.0,0.0)
sporeg['parent_group_id']=sporeg['head'].fillna(sporeg['id']).astype('string')
sporeg['name_normalized']=sporeg.name.astype('string').str.lower().str.replace(r'\s+',' ',regex=True).str.strip()

# Проверять уникальность натурального ключа после исключения полных дублей.
# Исторические конфликты не удаляются молча: полные дубли снимаются, остальные в карантин.
def quarantine_conflicting_registry(df,sector):
    key=['entity_id','observation_year']
    if df.empty:return df
    df=df.drop_duplicates().copy()
    unknown=df.entity_id.astype(str).str.contains('UNRESOLVED',regex=False) | df.entity_id.isna()
    if unknown.any():
        QUARANTINE.extend(df.loc[unknown].head(500).assign(source=sector+'_MISSING_ID').to_dict('records'))
        df=df.loc[~unknown].copy()
    counts=df.groupby(key,dropna=False).size()
    badkeys=set(counts[counts>1].index)
    if badkeys:
        badmask=pd.MultiIndex.from_frame(df[key]).isin(pd.MultiIndex.from_tuples(list(badkeys),names=key))
        QUARANTINE.extend(df.loc[badmask].head(1000).assign(source=sector+'_CONFLICTING_ID_YEAR').to_dict('records'))
        df=df.loc[~badmask].copy()
    return df
vo=quarantine_conflicting_registry(vo,'VO')
sporeg=quarantine_conflicting_registry(sporeg,'SPO')
for x,tab in [('VO',vo),('SPO',sporeg)]:
    if tab.empty:ISSUES.append({'module':'ML-5.2','status':'NO_REGISTRY',
          'reason':f'Исторический реестр {x} пуст после контроля конфликтных идентификаторов','required':'unique stable IDs'})
registry_columns=['sector','observation_year','entity_id','organization_id','parent_group_id','name','region','oktmo',
                  'city','address','campus_geo_status','entity_match_confidence']
vo2=vo.rename(columns={'region':'region','oktmo':'oktmo'})
spo2=sporeg.rename(columns={'region_name':'region','region_oktmo':'region_oktmo_source'})
reg_parts=[]
for frame in (vo2,spo2):
    part=frame.copy()
    for col in registry_columns:
        if col not in part:part[col]=pd.NA
    reg_parts.append(part[registry_columns])
REGISTRY=qa_table('edu_org_registry',pd.concat(reg_parts,ignore_index=True),
                  ['sector','entity_id','observation_year'],allow_empty=True)
REGISTRY_AUDIT=pd.DataFrame([{'sector':s,'records':len(g),'unique_org_ids':g.entity_id.nunique(),
    'years':g.observation_year.nunique(),'confirmed_campuses':int(g.campus_geo_status.eq('CONFIRMED_STUDY_CAMPUS').sum())}
    for s,g in REGISTRY.groupby('sector')])
qa_table('registry_coverage_audit',REGISTRY_AUDIT,allow_empty=True)

# Словари по учреждениям: ни один показатель не принимается по непустоте конфигурационного кортежа.
def selected_names(source,role='ADMISSION_CANDIDATE'):
    # В словарях сохраняются кандидаты, но в прогноз попадают только утверждённые точные имена.
    sub=CANDIDATE_CATALOG[(CANDIDATE_CATALOG.source==source)&
         (CANDIDATE_CATALOG.semantic_role==role)&
         (CANDIDATE_CATALOG.semantic_gate=='ELIGIBLE_FOR_ROW_CHECK')]
    candidates=set(sub.indicator_name.dropna().astype(str).unique())
    if source in ('E02','SPO_ORG'):
        approved=set(approved_names(source));unapproved=sorted(candidates-approved)
        if unapproved:ISSUES.append({'module':'ML-5.2','status':'UNAPPROVED_ADMISSION_CANDIDATES',
          'reason':f'{source}: {len(unapproved)} неутверждённых исходных имён','required':'утвердить точные имена в конфигурации'})
        return sorted(candidates&approved)
    return sorted(candidates)

# Панель приема ВО на уровне E02: все положительные candidates + точный натуральный ключ + единица.
# При неоднозначности исходных area/specialty/indicator нужна quarantine, а не сложение строк.
def selective_records(a,cols,names):
    if not names:return pd.DataFrame(columns=cols)
    esc=lambda t:"'"+t.replace("'","''")+"'"
    name_list=','.join(esc(t) for t in names)
    return read_table(a,cols,where=f'indicator_name IN ({name_list})',
        limit=CFG.MAX_READ_ROWS_SMOKE if CFG.MODE=='SMOKE' else None)

vo_admission=selective_records(E02,['id','parent_id','year','name','region','oktmo',
 'area','specialty_code','specialty_section','indicator_name','indicator_unit','indicator_value'],
 selected_names('E02'))
if len(vo_admission):
    vo_admission['entity_id']='VO|'+norm_code(vo_admission.id)
    vo_admission['observation_year']=safe_year(vo_admission.year)
    vo_admission['value']=number(vo_admission.indicator_value)
    vo_admission['indicator_grain']='E02_ID_YEAR_AREA_SPECIALTY_INDICATOR'
    # Сохраняем полный programme-разрез до отбора только общих строк для орг-панели.
    vo_program_candidates=vo_admission.copy()
    # area — раздел мониторинга, а не доказанный подитог. Ограничиваем только specialty_code.
    for categorical in ('specialty_code',):
        v=vo_admission[categorical].astype('string').str.strip().str.lower()
        keep=v.isna() | v.eq('') | v.isin(['всего','итого','total','0','00'])
        rejected=vo_admission.loc[~keep].copy()
        if len(rejected):QUARANTINE.extend(rejected.head(100).assign(source='VO_NON_TOTAL_'+categorical).to_dict('records'))
        vo_admission=vo_admission.loc[keep].copy()

# SPO данные организации сопрягаются только по уникальному совпадению региона/названия/года.
spo_admission=selective_records(SPO_MASTER,['year','region_name','name','property','indicator_section',
 'indicator_name','indicator_unit','org_value'],selected_names('SPO_ORG')) if SPO_MASTER else pd.DataFrame()
if len(spo_admission):
    spo_admission['name_normalized']=spo_admission.name.astype('string').str.lower().str.replace(r'\s+',' ',regex=True).str.strip()
    spo_admission['observation_year']=safe_year(spo_admission.year)
    spo_admission['region_name']=spo_admission.region_name.astype('string').str.strip()
    lookup=sporeg[['region_name','name_normalized','observation_year','entity_id']].drop_duplicates()
    count=lookup.groupby(['region_name','name_normalized','observation_year'],dropna=False).size().rename('matches').reset_index()
    lookup=lookup.merge(count,on=['region_name','name_normalized','observation_year'],validate='many_to_one')
    ambiguous=lookup[lookup.matches!=1].copy()
    if len(ambiguous):QUARANTINE.extend(ambiguous.head(500).assign(source='SPO_MATCH').to_dict('records'))
    lookup=lookup[lookup.matches==1].drop(columns=['matches'])
    spo_admission=spo_admission.merge(lookup,on=['region_name','name_normalized','observation_year'],
                                     how='left',validate='many_to_one')
    spo_admission['value']=number(spo_admission.org_value)
    spo_admission['indicator_grain']='SPO_ORG_YEAR_PROPERTY_INDICATOR'
    # property в мониторинге СПО может обозначать форму собственности организации,
    # а не разрез значения. Не удаляем реальные организации по этому полю.

# ВПО-1: полная точная агрегация, частями на диск. Лимит 512 МБ используется только для лёгких запросов.
VPO_FULL_PROGRAM_DESCRIPTION=pd.DataFrame()
VPO_FULL_PROGRAM_PATH=None
_vpo_names=selected_names('VPO_2_1_1')
if _vpo_names and CFG.MODE=='FULL':
    _vpo_path=Path(fetch_asset(VPO_211))
    _base=ROOT/'derived'; _base.mkdir(parents=True,exist_ok=True)
    _shards=_base/'vpo_year_shards'; _shards.mkdir(parents=True,exist_ok=True)
    VPO_FULL_PROGRAM_PATH=_base/'vpo_full_program_description.parquet'
    _esc=lambda value:"'"+str(value).replace("'","''")+"'"
    _where_names='indicator_name IN ('+','.join(_esc(x) for x in _vpo_names)+')'
    _src='read_parquet('+_esc(str(_vpo_path))+')'
    # Каждый год рассчитывается независимо; результаты объединяются без повторной агрегации.
    # Группы год×индикатор×специальность нельзя дробить между выходными файлами.
    _select='''SELECT TRY_CAST(year AS INTEGER) AS observation_year, specialty_code, indicator_name,
        object_level, education_form, education_level, admission_type,
        COUNT(*)::BIGINT AS observed_records,
        MEDIAN(TRY_CAST(indicator_value AS DOUBLE)) AS median_reported_value,
        COUNT(DISTINCT object_name)::BIGINT AS distinct_geographic_names,
        FALSE AS totals_additive_confirmed'''
    _start_filter=f'({_where_names}) AND TRY_CAST(year AS INTEGER) IS NOT NULL'
    memory_point('before_vpo_full_aggregate')
    progress_event('PANELS','ВПО-1: проверка годов и полной численности',announce=True)
    with duck_connect() as _con:
        _con.execute("SET preserve_insertion_order=false")
        _years=[int(r[0]) for r in _con.execute(
            f'SELECT DISTINCT TRY_CAST(year AS INTEGER) AS y FROM {_src} WHERE {_start_filter} ORDER BY 1'
        ).fetchall()]
        _expected=int(_con.execute(f'SELECT COUNT(*) FROM {_src} WHERE {_start_filter}').fetchone()[0])
        _bad_years=int(_con.execute(f'SELECT COUNT(*) FROM {_src} WHERE ({_where_names}) AND TRY_CAST(year AS INTEGER) IS NULL').fetchone()[0])
    if _bad_years:
        ISSUES.append({'module':'ML-5.4','status':'VPO_UNDATED_EXCLUDED',
            'reason':f'ВПО-1: {_bad_years} строк без корректного года исключены из динамики',
            'required':'проверка первичных дат'})
    # Очистка только технических файлов этой конкретной агрегации.
    for _old in _shards.glob('vpo_*.parquet'): _old.unlink()
    _parts=[]

    def _vpo_copy(where,output):
        # Точный MEDIAN и COUNT(DISTINCT) без замены статистик приближёнными.
        _sql=f'''COPY ({_select} FROM {_src} WHERE ({where}) GROUP BY 1,2,3,4,5,6,7)
                    TO {_esc(str(output))} (FORMAT PARQUET, COMPRESSION ZSTD)'''
        output.unlink(missing_ok=True)
        try:
            with duck_connect() as _con:
                _con.execute("SET memory_limit='3GB'")
                _con.execute('SET threads=1')
                _con.execute('SET preserve_insertion_order=false')
                _con.execute("SET max_temp_directory_size='40GB'")
                _con.execute(_sql)
            return output
        except Exception:
            output.unlink(missing_ok=True)
            raise

    for _y in _years:
        _filter=f'{_start_filter} AND TRY_CAST(year AS INTEGER)={_y}'
        _outfile=_shards/f'vpo_year_{_y}.parquet'
        progress_event('PANELS',f'ВПО-1: полный расчёт за {_y} год',announce=True)
        try:
            _parts.append(_vpo_copy(_filter,_outfile))
        except duckdb.OutOfMemoryException:
            progress_event('PANELS',f'ВПО-1 {_y}: дробление по индикаторам и специальностям',announce=True)
            for _i,_indicator in enumerate(_vpo_names):
                _ifilter=f'{_filter} AND indicator_name={_esc(_indicator)}'
                _ipath=_shards/f'vpo_year_{_y}_indicator_{_i}.parquet'
                try:
                    _parts.append(_vpo_copy(_ifilter,_ipath))
                except duckdb.OutOfMemoryException:
                    # hash(specialty_code) фиксирован для каждой группы; 16 частей не пересекаются.
                    for _bucket in range(16):
                        _bfilter=f'''{_ifilter} AND hash(COALESCE(CAST(specialty_code AS VARCHAR),'__NULL__')) % 16={_bucket}'''
                        _bpath=_shards/f'vpo_year_{_y}_indicator_{_i}_part_{_bucket:02}.parquet'
                        _parts.append(_vpo_copy(_bfilter,_bpath))
    if not _parts:
        ISSUES.append({'module':'ML-5.4','status':'VPO_NO_MATCHES','reason':'Нет строк по утверждённым именам индикаторов'})
        VPO_FULL_PROGRAM_PATH=None
    else:
        _filelist='['+','.join(_esc(p) for p in _parts)+']'
        with duck_connect() as _con:
            _con.execute("SET preserve_insertion_order=false")
            _con.execute(f'''COPY (SELECT * FROM read_parquet({_filelist})) TO
                          {_esc(VPO_FULL_PROGRAM_PATH)} (FORMAT PARQUET, COMPRESSION ZSTD)''')
            _actual=int(_con.execute('SELECT SUM(observed_records) FROM read_parquet(?)',
                                     [str(VPO_FULL_PROGRAM_PATH)]).fetchone()[0] or 0)
            VPO_FULL_PROGRAM_DESCRIPTION=_con.execute(
                'SELECT * FROM read_parquet(?) LIMIT 2000',[str(VPO_FULL_PROGRAM_PATH)]).df()
        if _actual!=_expected:
            fail('VPO_DATA_GATE',f'Потеря или дубль строк ВПО-1: источник {_expected}, агрегация {_actual}')
        _pf=pq.ParquetFile(VPO_FULL_PROGRAM_PATH)
        PANEL_FILES['vpo_program_description']=VPO_FULL_PROGRAM_PATH
        PANELS['vpo_program_description']=VPO_FULL_PROGRAM_DESCRIPTION
        TABLE_QA.append({'table':'vpo_program_description','rows':_pf.metadata.num_rows,
            'columns':len(_pf.schema_arrow.names),'duplicate_keys':None,'status':'PASS',
            'unique_keys':'','storage':'DUCKDB_YEAR_SHARDED_EXACT_AGGREGATE'})
        progress_event('PANELS',f'ВПО-1: {_expected:,} исходных строк, {_pf.metadata.num_rows:,} групп; контроль PASS',announce=True)
        del _pf
    _parts.clear()
    gc.collect()
# Полная агрегация выше; ниже только просмотр небольшой выборки, который не является полным источником.
vpo_admission=selective_records(VPO_211,['year','object_level','object_name','object_oktmo','university_type',
 'education_form','education_level','specialty_name','specialty_code','indicator_name','indicator_value','admission_type'],
 _vpo_names) if CFG.MODE!='FULL' else (read_table(VPO_211,['year','object_level','object_name','object_oktmo','university_type',
 'education_form','education_level','specialty_name','specialty_code','indicator_name','indicator_value','admission_type'],
 where='indicator_name IN ('+','.join("'"+x.replace("'","''")+"'" for x in _vpo_names)+')',limit=2000)
 if _vpo_names else pd.DataFrame())
if len(vpo_admission):
    vpo_admission['observation_year']=safe_year(vpo_admission.year)
    vpo_admission['value']=number(vpo_admission.indicator_value)
    vpo_admission['geography_grain']='REPORTED_OBJECT_LEVEL_'+vpo_admission.object_level.astype(str)

# Автоматический агрегат для прогнозирования строится только когда один уникальный показатель
# на ID/year, и источник действительно указывает итог, а не несопоставимые категории.
def unambiguous_org_admission(raw,sector):
    if raw.empty or not {'entity_id','observation_year','indicator_name','value'}.issubset(raw):return pd.DataFrame()
    x=raw.dropna(subset=['entity_id','observation_year','value']).copy()
    integerish=np.isclose(x.value.to_numpy(float),np.rint(x.value.to_numpy(float)),atol=1e-8,rtol=0)
    noncount=x.loc[~integerish].copy()
    if len(noncount):QUARANTINE.extend(noncount.head(200).assign(source=sector+'_NONINTEGER_ADMISSION').to_dict('records'))
    x=x.loc[x.value.ge(0)&integerish].copy()
    if x.empty:return pd.DataFrame()
    keys=['entity_id','observation_year']
    counts=x.groupby(keys,dropna=False).agg(n=('value','size'),n_indicators=('indicator_name','nunique')).reset_index()
    usable=counts[(counts.n==1)&(counts.n_indicators==1)][keys]
    good=x.merge(usable,on=keys,validate='many_to_one')
    if good.duplicated(keys).any():fail('SEMANTICS',f'{sector}: неоднозначный показатель приёма после фильтра')
    good=good.assign(sector=sector,target_definition='source_approved_count_of_admitted')
    bad=x.merge(usable.assign(used=True),on=keys,how='left')
    bad=bad[bad.used.isna()].drop(columns=['used'])
    if len(bad):QUARANTINE.extend(bad.head(100).assign(source=f'{sector}_MULTIPLE_ADMISSION_VALUES').to_dict('records'))
    return good[['sector','entity_id','observation_year','value','indicator_name','target_definition']].rename(columns={'value':'actual'})

org_parts=[unambiguous_org_admission(vo_admission,'VO'),unambiguous_org_admission(spo_admission,'SPO')]
ORG_YEAR=qa_table('edu_org_year',pd.concat(org_parts,ignore_index=True) if any(len(p) for p in org_parts)
                  else pd.DataFrame(columns=['sector','entity_id','observation_year','actual','indicator_name','target_definition']),
                  ['sector','entity_id','observation_year'],allow_empty=True)
qa_table('vo_admission_raw_candidates',vo_admission,allow_empty=True)
qa_table('spo_admission_raw_candidates',spo_admission,allow_empty=True)
qa_table('vpo_admission_audit',vpo_admission,allow_empty=True)

# Образовательные программы: источники E02 и ВПО-1 сохраняются без географической реконструкции OD×programme.
parts=[]
if 'vo_program_candidates' in globals() and len(vo_program_candidates):
    v=vo_program_candidates[['entity_id','observation_year','specialty_code','specialty_section','indicator_name','value']].copy()
    v['sector']='VO';v['geography_grain']='ORGANIZATION';parts.append(v)
if len(vpo_admission):
    v=vpo_admission[['observation_year','specialty_code','indicator_name','value','geography_grain']].copy()
    v['specialty_section']=pd.NA;v['entity_id']=pd.NA;v['sector']='VO_VPO';parts.append(v)
PROGRAM=qa_table('edu_program_year',pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(),allow_empty=True)

# Выпускники: FULL обрабатывает каждую строку E04 и E05 через DuckDB, в RAM — только итоги.
# Оригинальные E04/E05 остаются в Stage I (SHA фиксируются). Отчёт об исходных записях явный.
GRADUATE_GROUP_SUMMARY=pd.DataFrame();GRADUATE_FIELD_SUMMARY=pd.DataFrame()
GRAD_E04_VALID_ROWS=0;GRAD_E05_VALID_ROWS=0
GRAD_COLS=['object_level','object_name','oktmo','gender','education_level','year','university',
           'count_graduate','percent_employed','average_salary_fact_avg','average_salary_fact_med']
E05_COLS=['object_level','object_name','year','education_level','study_area','specialty_section',
          'specialty','specialty_code','count_graduate','percent_employed','average_salary_fact_med']
if CFG.MODE=='FULL':
    progress_event('PANELS','E04/E05: загрузка оригинальных файлов, агрегация на диске',announce=True)
    path_e04=fetch_asset(GRAD_ORG);path_e05=fetch_asset(GRAD_FIELD)
    with duck_connect() as con:
        progress_event('PANELS','E04: полная агрегатная обработка DuckDB',announce=True)
        GRADUATE_GROUP_SUMMARY=con.execute("""SELECT TRY_CAST(year AS INTEGER) AS graduation_year,
                 object_level,gender,education_level,COUNT(*)::BIGINT AS n_source_records,
                 MEDIAN(TRY_CAST(count_graduate AS DOUBLE)) AS median_reported_graduates,
                 MEDIAN(TRY_CAST(average_salary_fact_med AS DOUBLE)) AS median_observed_salary,
                 MEDIAN(TRY_CAST(percent_employed AS DOUBLE)) AS median_employment_percentage
          FROM read_parquet(?) WHERE TRY_CAST(count_graduate AS DOUBLE)>=0
          GROUP BY 1,2,3,4""",[str(path_e04)]).df()
        GRAD_E04_VALID_ROWS=int(GRADUATE_GROUP_SUMMARY.n_source_records.sum())
        progress_event('PANELS','E05: полная агрегатная обработка DuckDB',announce=True)
        GRADUATE_FIELD_SUMMARY=con.execute("""SELECT TRY_CAST(year AS INTEGER) AS graduation_year,
                 object_level,education_level,CAST(specialty_code AS VARCHAR) AS specialty_code,
                 COUNT(*)::BIGINT AS observed_records,
                 MEDIAN(TRY_CAST(count_graduate AS DOUBLE)) AS median_reported_graduates,
                 MEDIAN(TRY_CAST(average_salary_fact_med AS DOUBLE)) AS median_wage
          FROM read_parquet(?) WHERE TRY_CAST(count_graduate AS DOUBLE)>=0
            AND specialty_code IS NOT NULL AND TRY_CAST(year AS INTEGER) IS NOT NULL
          GROUP BY 1,2,3,4""",[str(path_e05)]).df()
        GRAD_E05_VALID_ROWS=int(GRADUATE_FIELD_SUMMARY.observed_records.sum()) if len(GRADUATE_FIELD_SUMMARY) else 0
    progress_event('PANELS',f'E04/E05: обработано {GRAD_E04_VALID_ROWS:,}+{GRAD_E05_VALID_ROWS:,} допустимых строк',announce=True)
    GRADUATE_GROUP_SUMMARY['aggregate_totals_identified']=False
    GRADUATE_GROUP_SUMMARY['employment_asof']='2024-12-31'
    GRADUATE_FIELD_SUMMARY['aggregate_totals_identified']=False
    GRADUATE_FIELD_SUMMARY['employment_asof']='2024-12-31'
    qa_table('graduate_descriptive_statistics',GRADUATE_GROUP_SUMMARY,allow_empty=True)
    qa_table('graduate_specialty_description',GRADUATE_FIELD_SUMMARY,allow_empty=True)
    qa_table('graduate_full_scan_audit',pd.DataFrame([
       {'asset_id':GRAD_ORG['asset_id'],'source_rows':int(GRAD_ORG['rows_actual']),
        'valid_rows_in_statistics':GRAD_E04_VALID_ROWS,'method':'DUCKDB_FULL_SCAN_MEDIAN','raw_stage1_sha256':GRAD_ORG['sha256']},
       {'asset_id':GRAD_FIELD['asset_id'],'source_rows':int(GRAD_FIELD['rows_actual']),
        'valid_rows_in_statistics':GRAD_E05_VALID_ROWS,'method':'DUCKDB_FULL_SCAN_MEDIAN','raw_stage1_sha256':GRAD_FIELD['sha256']}]),allow_empty=True)
    # Доступен только явно обозначенный preview, не выдаваемый за полную базу.
    grad=read_table(GRAD_ORG,GRAD_COLS,limit=1000)
    grad['graduation_year']=safe_year(grad['year'])
    grad['employment_asof']=pd.Timestamp('2024-12-31')
    grad['salary_reference_year']=2024;grad['employment_location']=pd.NA
    for c in ['count_graduate','percent_employed','average_salary_fact_avg','average_salary_fact_med']:
        grad[c]=number(grad[c])
    grad['source_id']=GRAD_ORG['asset_id']
    GRADUATES=qa_table('graduate_outcomes_preview',grad,allow_empty=True)
    del grad;gc.collect()
    note('MEMORY','E04/E05 FULL: все строки агрегированы в DuckDB; в Pandas только сводные таблицы и 1000 исходных строк')
else:
    grad=read_table(GRAD_ORG,GRAD_COLS,limit=CFG.MAX_READ_ROWS_SMOKE)
    grad['graduation_year']=safe_year(grad['year']);grad['employment_asof']=pd.Timestamp('2024-12-31')
    grad['salary_reference_year']=2024;grad['employment_location']=pd.NA
    for c in ['count_graduate','percent_employed','average_salary_fact_avg','average_salary_fact_med']:
        grad[c]=number(grad[c])
    grad['source_id']=GRAD_ORG['asset_id']
    GRADUATES=qa_table('graduate_outcomes',grad,allow_empty=True)
    gradf=read_table(GRAD_FIELD,E05_COLS,limit=CFG.MAX_READ_ROWS_SMOKE)
    gradf['graduation_year']=safe_year(gradf['year']);gradf['employment_asof']=pd.Timestamp('2024-12-31')
    gradf['specialty_code']=norm_code(gradf['specialty_code'])
    for c in ['count_graduate','percent_employed','average_salary_fact_med']:gradf[c]=number(gradf[c])
    gradf['source_asset_id']=GRAD_FIELD['asset_id']
    qa_table('graduate_specialty_outcomes',gradf,allow_empty=True)

# Освободить полный первичный E04/E05 сразу после расчёта сводных таблиц.
for _unused in ('gradf',):
    if _unused in globals():del globals()[_unused]
gc.collect();memory_point('panels_after_graduates')

# Муниципальная демография: точные исходные ОКТМО и исторические года; значения не размножаются по вузам.
mun=read_table(MUN,['oktmo','region','municipality','year','population','migration','migration_rate','births','deaths'])
mun['oktmo']=norm_code(mun.oktmo);mun['observation_year']=safe_year(mun.year)
for c in ['population','migration','migration_rate','births','deaths']:mun[c]=number(mun[c])
mun=mun.drop(columns=['year'])
mun['geography_grain']='MUNICIPALITY_FROM_D01'
mun=mun.dropna(subset=['oktmo','observation_year']).copy()
mun_unique=mun.groupby(['oktmo','observation_year']).size()
if (mun_unique>1).any():
    ISSUES.append({'module':'ML-5.7','status':'CONFLICTING_MUNICIPAL_KEYS',
      'reason':f'D01 дублирует ОКТМО×год: {int((mun_unique>1).sum())} ключей; исключены',
      'required':'reconciliation of historical municipality borders'})
    ok=mun_unique[mun_unique==1].index
    mun=mun.set_index(['oktmo','observation_year']).loc[ok].reset_index()
MUNI=qa_table('municipal_demography',mun,['oktmo','observation_year'],allow_empty=True)

# Только verified campus география; наличие без ОКТМО никогда не трактуется как отсутствие.
conf=vo.loc[vo.campus_geo_status.eq('CONFIRMED_STUDY_CAMPUS'),
  ['entity_id','observation_year','oktmo','parent_group_id']].dropna(subset=['oktmo']).copy()
conf=conf[nonmissing_code(conf.oktmo)]
conf=conf.drop_duplicates(['entity_id','observation_year'])
conf=conf.groupby(['oktmo','observation_year'],as_index=False).agg(
    confirmed_vo_sites=('entity_id','nunique'))
MUNI_EDU=MUNI.merge(conf,on=['oktmo','observation_year'],how='left',validate='one_to_one')
MUNI_EDU['education_coverage_known']=MUNI_EDU.confirmed_vo_sites.notna()
# Missing is unknown, NOT zero campuses.
MUNI_EDU['confirmed_spo_sites']=pd.Series(pd.NA,index=MUNI_EDU.index,dtype='Int64')
qa_table('municipality_education_context',MUNI_EDU,['oktmo','observation_year'],allow_empty=True)

qa_table('entity_match_quarantine',pd.DataFrame(QUARANTINE),allow_empty=True)
print('Panels: OD',len(OD),'inter',len(OD_INTER),'VO+SPO registry',len(REGISTRY),
      'org_year',len(ORG_YEAR),'program',len(PROGRAM),'graduates',len(GRADUATES),
      'municipal',len(MUNI_EDU))

# Финальная отметка этапа перенесена после всех его подмодулей


## 5А. Advanced Data Reconstruction · реконструкция первичных показателей и географии
Только наблюдаемые записи: E01 на общем покрытии, все кандидаты приёма E02, географические соответствия СПО с D04 и раздельные флаги пространственной неопределённости.

In [ ]:
# ML-05 V7.4.0 · Advanced Data Reconstruction: audited targets, balanced OD and spatial links.
# Reconstruction is evidence-gated. No inferred school admission, residence, campus or employment location is recorded as observed.
progress_event('PANELS','V7.4: первичная реконструкция по опубликованным наблюдениям',announce=True)
ADR_PREPROCESS_AUDIT=[]
ADR_ADMISSION_YEAR=pd.DataFrame()
ADR_ADMISSION_CANDIDATES=pd.DataFrame()
ADR_GEO_SPO=pd.DataFrame()
ADR_GEO_VERIFIED=pd.DataFrame()
ADR_OD_BALANCED=pd.DataFrame()
ADR_OD_PAIRWISE=pd.DataFrame()
ADR_SEMANTIC_PROVENANCE=[]

def adr_text(s):
    return (s.astype('string').fillna('').str.lower().str.replace('ё','е',regex=False)
            .str.replace(r'[^\w\s-]',' ',regex=True).str.replace(r'\s+',' ',regex=True).str.strip())

def adr_numbers(s):
    return pd.to_numeric(s.astype('string').str.replace(',','.',regex=False),errors='coerce')

def adr_log(stage,source,status,n_rows=0,rule='',reason=''):
    ADR_PREPROCESS_AUDIT.append({'procedure':stage,'source':source,'status':status,
        'n_observations':int(n_rows),'strict_rule':rule,'limitation':str(reason)[:300]})

# ML-5.1 / ML-5.3 / ML-5.6. All-year balanced panel on *actually observed* interregional E01 edges.
# This is an independent coverage-stability study, not synthetic completion of the OD matrix.
_adr_od=OD_INTER[['observation_year','origin_region','destination_region','observed_value']].dropna().copy()
_adr_od['observation_year']=pd.to_numeric(_adr_od['observation_year'],errors='coerce')
_adr_od=_adr_od.loc[_adr_od.observation_year.notna() & _adr_od.observed_value.ge(0)].copy()
_adr_od['observation_year']=_adr_od.observation_year.astype(int)
_adr_keys=['origin_region','destination_region']
_adr_years=sorted(_adr_od.observation_year.unique())
if _adr_od.duplicated(['observation_year']+_adr_keys).any():
    raise RuntimeError('ADR E01: duplicate observed OD edge-year')
if len(_adr_years)>=2:
    _adr_counts=_adr_od.groupby(_adr_keys).observation_year.nunique()
    _adr_shared=_adr_counts[_adr_counts==len(_adr_years)].index
    ADR_OD_BALANCED=_adr_od.set_index(_adr_keys).loc[_adr_shared].reset_index() if len(_adr_shared) else _adr_od.head(0).copy()
    _adr_bal=ADR_OD_BALANCED.groupby('observation_year',as_index=False).agg(
        n_common_edges=('observed_value','size'),total_on_common_edges=('observed_value','sum'),
        median_on_common_edges=('observed_value','median')) if len(ADR_OD_BALANCED) else pd.DataFrame()
    if len(_adr_bal):
        _adr_bal['all_years_shared']=len(_adr_years)
        _adr_bal['scope']='OBSERVED_BALANCED_EDGE_INTERSECTION_NOT_NATIONAL_TOTAL'
    qa_table('adr_od_balanced_edges_by_year',_adr_bal,allow_empty=True)
    if len(ADR_OD_BALANCED):
        qa_table('adr_od_balanced_edges',ADR_OD_BALANCED,['observation_year']+_adr_keys)
    _adr_pairs=[]
    for _left,_right in zip(_adr_years[:-1],_adr_years[1:]):
        if int(_right)!=int(_left)+1:continue
        _x=_adr_od.loc[_adr_od.observation_year.eq(_left),_adr_keys+['observed_value']].rename(columns={'observed_value':'prior'})
        _y=_adr_od.loc[_adr_od.observation_year.eq(_right),_adr_keys+['observed_value']].rename(columns={'observed_value':'current'})
        _both=_x.merge(_y,on=_adr_keys,validate='one_to_one')
        _adr_pairs.append({'prior_year':int(_left),'current_year':int(_right),
           'prior_observed_edges':len(_x),'current_observed_edges':len(_y),'common_edges':len(_both),
           'coverage_of_prior':len(_both)/len(_x) if len(_x) else np.nan,
           'coverage_of_current':len(_both)/len(_y) if len(_y) else np.nan,
           'MAE_persistence_common':float(np.abs(_both.current-_both.prior).mean()) if len(_both) else np.nan,
           'sum_prior_common':float(_both.prior.sum()),'sum_current_common':float(_both.current.sum()),
           'coverage_break_at_2023':int(_right)==2023,
           'claim_scope':'COMMON_REPORTED_EDGES_ONLY_WITHOUT_MISSING_ZERO_IMPUTATION'})
    ADR_OD_PAIRWISE=qa_table('adr_od_pairwise_comparable',pd.DataFrame(_adr_pairs),allow_empty=True)
    adr_log('E01_STABLE_OD',E01['asset_id'],'COMPLETE' if len(ADR_OD_BALANCED) else 'NO_BALANCED_EDGES',
        len(ADR_OD_BALANCED),'intersection of observed OD keys across all available years',
        'No permanent-residence movement or year-of-admission claim')
else:adr_log('E01_STABLE_OD',E01['asset_id'],'SKIPPED_NO_TIME_SUPPORT',len(_adr_od),'at least two years')

# ML-5.2: exhaustively inspect admission-labeled E02 rows at original organization+indicator+year grain.
# Single row may be admitted only when explicit source terminology, count unit and uniqueness all hold.
_adr_e02_out=ROOT/'derived'/'adr_e02_admission_candidates.parquet'
_adr_e02_tmp=ROOT/'derived'/'adr_e02_admission_candidates.incomplete.parquet'
_adr_e02_tmp.unlink(missing_ok=True)
# Stage I E02 read starts only AFTER SQL preflight succeeds.
_adr_e02_path=fetch_asset(E02)
progress_event('PANELS','E02: запись всех подходящих исходных строк на диск',announce=True)
with duck_connect() as _adr_con:
    _adr_con.execute(_adr_e02_copy_statement(_adr_e02_path,_adr_e02_tmp))
    _adr_total=int(_adr_con.execute('SELECT COUNT(*) FROM read_parquet(?)',[str(_adr_e02_tmp)]).fetchone()[0])
    ADR_ADMISSION_CANDIDATES=_adr_con.execute(_ADR_E02_SUMMARY_SQL,[str(_adr_e02_tmp)]).df()
    _adr_e02_tmp.replace(_adr_e02_out)
progress_event('PANELS',f'E02 проверено {_adr_total:,} кандидатных строк; индикаторов {len(ADR_ADMISSION_CANDIDATES)}',announce=True)
if len(ADR_ADMISSION_CANDIDATES):
    _adr_label=adr_text(ADR_ADMISSION_CANDIDATES.indicator_name)
    _adr_unit=adr_text(ADR_ADMISSION_CANDIDATES.indicator_unit)
    ADR_ADMISSION_CANDIDATES['explicit_count_noun']=_adr_label.str.contains(r'численност|количеств|число',regex=True)
    ADR_ADMISSION_CANDIDATES['explicit_admitted_people']=_adr_label.str.contains(r'принят|зачислен',regex=True)
    ADR_ADMISSION_CANDIDATES['ambiguous_meaning']=_adr_label.str.contains(r'заявлен|конкурс|контрольн|план|квот|приемн|доля|процент|на работу',regex=True)
    ADR_ADMISSION_CANDIDATES['unit_is_people']=_adr_unit.str.contains(r'чел|человек',regex=True)
    ADR_ADMISSION_CANDIDATES['semantic_candidate']=ADR_ADMISSION_CANDIDATES.explicit_count_noun & ADR_ADMISSION_CANDIDATES.explicit_admitted_people & ~ADR_ADMISSION_CANDIDATES.ambiguous_meaning & ADR_ADMISSION_CANDIDATES.unit_is_people
    ADR_ADMISSION_CANDIDATES['user_or_method_approval']=ADR_ADMISSION_CANDIDATES.indicator_name.isin(CFG.APPROVED_VO_ADMISSION_NAMES)
    ADR_ADMISSION_CANDIDATES['strict_source_label_supported']=ADR_ADMISSION_CANDIDATES.semantic_candidate & ADR_ADMISSION_CANDIDATES.invalid_numeric_rows.eq(0) & ADR_ADMISSION_CANDIDATES.invalid_count_rows.eq(0)
    ADR_ADMISSION_CANDIDATES['automatic_official_approval']=False  # Source label != externally verified cohort methodology.
qa_table('adr_organizational_admission_dictionary',ADR_ADMISSION_CANDIDATES,allow_empty=True)
adr_log('E02_ADMISSIONS',E02['asset_id'],'SCANNED_FULL',_adr_total,
    'count-unit, label, original reporting grain, single unambiguous organization-year observation',
    'Candidates without methodology approval are described, not asserted as admissions')

# Persist full candidates, but only an explicitly approved indicator with a unique reporting total can enter a new target.
_adr_approved=ADR_ADMISSION_CANDIDATES.loc[
    (ADR_ADMISSION_CANDIDATES['strict_source_label_supported'] & ADR_ADMISSION_CANDIDATES['user_or_method_approval']),
    'indicator_name'].dropna().astype(str).unique().tolist() if len(ADR_ADMISSION_CANDIDATES) else []
if _adr_approved:
    with duck_connect() as _adr_con:
        _adr_con.execute("SET memory_limit='1536MB'")
        _adr_literal_names=','.join("'"+x.replace("'","''")+"'" for x in _adr_approved)
        _adr_sel=_adr_con.execute(_ADR_E02_TOTAL_SQL.replace('/*VALID_NAMES*/',_adr_literal_names),
            [str(_adr_e02_out)]).df()
    _adr_sel=_adr_sel.loc[_adr_sel.organization_id.notna()&_adr_sel.observation_year.notna()&
        _adr_sel.n_reporting_cells.eq(1)&_adr_sel.area_total.fillna(False)&_adr_sel.specialty_total.fillna(False)&
        _adr_sel.min_count.notna()&_adr_sel.min_count.ge(0)&
        _adr_sel.min_count.eq(_adr_sel.max_count)&_adr_sel.min_count.mod(1).eq(0)].copy()
    _adr_sel=_adr_sel.drop_duplicates(['organization_id','observation_year'],keep=False)
    _adr_sel['entity_id']='VO|'+norm_code(_adr_sel.organization_id)
    _adr_sel['actual']=_adr_sel.min_count
    _adr_sel['target_definition']='SOURCE_LABEL_REPORTED_ADMISSION_COUNT_NOT_OFFICIAL_COHORT'
    _adr_sel['methodology_approval']=_adr_sel.indicator_name.isin(CFG.APPROVED_VO_ADMISSION_NAMES)
    _adr_sel['scientific_scope']='EXACT_E02_SOURCE_LABEL_AND_UNIQUE_ORG_YEAR_TOTAL'
    ADR_ADMISSION_YEAR=qa_table('adr_admission_organization_year',_adr_sel,
          ['entity_id','observation_year'],allow_empty=True)
else:
    adr_log('E02_ORG_ADMISSIONS',E02['asset_id'],'SKIPPED_NO_SEMANTICALLY_UNAMBIGUOUS_TARGET',0,
      'explicit source reported admitted count, unit people, unique organization-year TOTAL row',
      'No organization-year admission target invented from section/area counts')

# ML-5.7: relate *reported SPO coordinates* to D04 settlement identifiers, with an ambiguity flag.
# This never silently converts a spatial proxy into an officially verified campus location.
try:
    from sklearn.neighbors import BallTree
    _adr_sp=sporeg[['entity_id','observation_year','region_name','city','lat','lon']].copy()
    _adr_sp['lat_num']=adr_numbers(_adr_sp['lat']); _adr_sp['lon_num']=adr_numbers(_adr_sp['lon'])
    _adr_sp=_adr_sp.loc[_adr_sp.lat_num.between(40,85)&_adr_sp.lon_num.between(19,190)].copy()
    _adr_sp['region_key']=adr_text(_adr_sp.region_name)
    _adr_sp['city_key']=adr_text(_adr_sp.city)
    _adr_sp=_adr_sp.loc[_adr_sp.region_key.ne('')&_adr_sp.city_key.ne('')].copy()
    _adr_grid=fetch_asset(SPATIAL)
    with duck_connect() as _adr_con:
        _adr_d=_adr_con.execute('''SELECT CAST(source_oktmo AS VARCHAR) source_oktmo,
             CAST(region_name AS VARCHAR) region_name, CAST(municipality_name AS VARCHAR) municipality_name,
             CAST(settlement_name AS VARCHAR) settlement_name,
             TRY_CAST(latitude AS DOUBLE) latitude,TRY_CAST(longitude AS DOUBLE) longitude,
             CAST(coordinates_valid AS VARCHAR) coordinates_valid
             FROM read_parquet(?) WHERE TRY_CAST(latitude AS DOUBLE) BETWEEN 40 AND 85
               AND TRY_CAST(longitude AS DOUBLE) BETWEEN 19 AND 190''',[str(_adr_grid)]).df()
    _adr_d['region_key']=adr_text(_adr_d.region_name)
    _adr_d['city_key']=adr_text(_adr_d.settlement_name)
    _adr_d['mun_key']=adr_text(_adr_d.municipality_name)
    _adr_d['source_oktmo']=norm_code(_adr_d.source_oktmo)
    _adr_d=_adr_d.loc[nonmissing_code(_adr_d.source_oktmo)&_adr_d.region_key.ne('')].reset_index(drop=True)
    _adr_match=[]
    for _reg,_rows in _adr_sp.groupby('region_key',sort=False):
        _places=_adr_d.loc[_adr_d.region_key.eq(_reg)].copy()
        if len(_places)<2:continue
        _tree=BallTree(np.radians(_places[['latitude','longitude']].to_numpy(float)),metric='haversine',leaf_size=40)
        _dist,_ix=_tree.query(np.radians(_rows[['lat_num','lon_num']].to_numpy(float)),k=2)
        _places=_places.reset_index(drop=True)
        for _j,_rr in enumerate(_rows.itertuples(index=False)):
            _nearest=_places.iloc[int(_ix[_j,0])]
            _d1=float(_dist[_j,0]*6371.0088); _d2=float(_dist[_j,1]*6371.0088)
            _city=str(_rr.city_key)
            _city_match=(_city==str(_nearest.city_key) or _city==str(_nearest.mun_key))
            _high=bool(_city_match and _d1<=2.0 and _d2-_d1>=0.50)
            _adr_match.append({'entity_id':_rr.entity_id,'observation_year':_rr.observation_year,
                'reported_city':_rr.city,'reported_region':_rr.region_name,
                'candidate_municipal_oktmo':_nearest.source_oktmo,
                'nearest_centroid_km':_d1,'second_nearest_centroid_km':_d2,
                'city_name_consistent':bool(_city_match),'spatially_unambiguous':bool(_high),
                'source_id_spo':SPO_ORG['asset_id'],'source_id_geography':SPATIAL['asset_id'],
                'identity_status':'PROXIMITY_MATCH_NOT_OFFICIAL_CAMPUS_LOCATION' if _high else 'UNRESOLVED'})
    ADR_GEO_SPO=pd.DataFrame(_adr_match)
    if len(ADR_GEO_SPO):
        # Inconsistent historical assignments for the same institution-year are quarantined.
        _dup=ADR_GEO_SPO.duplicated(['entity_id','observation_year'],keep=False)
        ADR_GEO_SPO.loc[_dup,'spatially_unambiguous']=False
        ADR_GEO_SPO.loc[_dup,'identity_status']='CONFLICTING_HISTORICAL_LOCATION'
        ADR_GEO_VERIFIED=ADR_GEO_SPO.loc[ADR_GEO_SPO.spatially_unambiguous].drop_duplicates(['entity_id','observation_year']).copy()
    qa_table('adr_spo_geolocation_candidates',ADR_GEO_SPO,allow_empty=True)
    qa_table('adr_spo_geolocation_high_confidence',ADR_GEO_VERIFIED,allow_empty=True)
    adr_log('SPO_GEOCODE',SPATIAL['asset_id'],'PROXIMITY_RECONSTRUCTION',len(ADR_GEO_VERIFIED),
        'same region, city/settlement match, <2km, nearest separation >=0.5km',
        'Geocoded municipality is a reconstructed spatial proxy, not officially confirmed campus placement')
except (KeyError,ValueError,TypeError) as _geo_error:
    adr_log('SPO_GEOCODE',SPATIAL['asset_id'],'SKIPPED_INCOMPATIBLE_SPATIAL_FIELDS',0,'valid coordinates/municipal code',str(_geo_error))

# V7.4 · ещё до муниципального эксперимента: многолетний конфликт прокси-ОКТМО.
if len(ADR_GEO_VERIFIED):
    _v72_geo_conflict=(ADR_GEO_VERIFIED.groupby('entity_id').candidate_municipal_oktmo.nunique(dropna=True))
    _v72_geo_conflict=set(_v72_geo_conflict[_v72_geo_conflict>1].index)
    if _v72_geo_conflict:
        _v72_unstable=ADR_GEO_VERIFIED.loc[ADR_GEO_VERIFIED.entity_id.isin(_v72_geo_conflict)].copy()
        QUARANTINE.extend(_v72_unstable.head(100).assign(source='V72_UNVERIFIED_HISTORICAL_OKTMO_CHANGE').to_dict('records'))
        ADR_GEO_VERIFIED=ADR_GEO_VERIFIED.loc[~ADR_GEO_VERIFIED.entity_id.isin(_v72_geo_conflict)].copy()
        adr_log('SPO_GEOCODE_STABILITY',SPATIAL['asset_id'],'EXCLUDED_UNVERIFIED_HISTORICAL_CHANGE',
            len(_v72_unstable),'same candidate municipality for all reported years',
            'Historical move or inconsistent geocoding cannot be verified from proximity alone')

qa_table('adr_preprocessing_provenance',pd.DataFrame(ADR_PREPROCESS_AUDIT),allow_empty=True)
progress_event('PANELS',f'V7.4: common OD={len(ADR_OD_BALANCED):,}; approved admissions={len(ADR_ADMISSION_YEAR):,}; SPO geo proxies={len(ADR_GEO_VERIFIED):,}',announce=True)

mark_stage('PANELS')


## 6. Временные признаки, экономика и транспорт
Все признаки имеют исторический период и осторожный лаг доступности. Дистанции между городами остаются прокси, не истинным расстоянием регионов.

In [ ]:
memory_point('before_features')
progress_event('FEATURES','География, экономический контекст и исторические признаки')
# Экономический и пространственный контекст: только фактические данные, с пометкой уровня агрегации.
NAME_MAP=pd.concat([
    OD_INTER[['origin_region','origin_region_name']].rename(columns={'origin_region':'region_code','origin_region_name':'region_name'}),
    OD_INTER[['destination_region','destination_region_name']].rename(columns={'destination_region':'region_code','destination_region_name':'region_name'})
],ignore_index=True).drop_duplicates()
NAME_MAP['region_name_norm']=NAME_MAP.region_name.astype('string').str.lower().str.replace('ё','е').str.strip()
conflicts=NAME_MAP.groupby('region_code',dropna=False).region_name_norm.nunique(dropna=True)
if (conflicts>1).any():
    QUARANTINE.extend(NAME_MAP[NAME_MAP.region_code.isin(conflicts[conflicts>1].index)].head(150)
                      .assign(source='REGION_CODE_NAME_CONFLICT').to_dict('records'))
    NAME_MAP=NAME_MAP[~NAME_MAP.region_code.isin(conflicts[conflicts>1].index)].copy()
NAME_MAP=NAME_MAP.drop_duplicates(subset=['region_code'])
if NAME_MAP.region_code.duplicated().any():fail('GEOGRAPHY','Конфликтная карта кодов субъектов')
# Название региона должно задавать единственный код. Иначе исключаем сомнительные соединения.
_name_counts=NAME_MAP.groupby('region_name_norm',dropna=False).region_code.nunique()
_ambiguous=set(_name_counts[_name_counts>1].index)
if _ambiguous:
    QUARANTINE.extend(NAME_MAP[NAME_MAP.region_name_norm.isin(_ambiguous)].head(100).assign(source='AMBIGUOUS_REGION_NAME').to_dict('records'))
    ISSUES.append({'module':'GEOGRAPHY','status':'AMBIGUOUS_REGION_NAME','reason':f'{len(_ambiguous)} названий соответствуют нескольким кодам','required':'territorial crosswalk'})
NAME_LOOKUP=NAME_MAP[NAME_MAP.region_name_norm.notna() & ~NAME_MAP.region_name_norm.isin(_ambiguous)].copy()
REGION_CONTEXT=[];DISTANCE=pd.DataFrame()

try:
    raw=read_table(TRANSPORT,['region_from','region_to','distance_km','transport_type'],
                   limit=CFG.MAX_READ_ROWS_SMOKE if CFG.MODE=='SMOKE' else None)
    raw['origin_region_name_norm']=raw.region_from.astype('string').str.lower().str.replace('ё','е').str.strip()
    raw['destination_region_name_norm']=raw.region_to.astype('string').str.lower().str.replace('ё','е').str.strip()
    raw['distance_proxy_km']=number(raw.distance_km)
    raw=raw[(raw.distance_proxy_km>0)&raw.distance_proxy_km.notna()]
    d=raw.groupby(['origin_region_name_norm','destination_region_name_norm'],as_index=False).agg(
        distance_proxy_km=('distance_proxy_km','median'),distance_measurements=('distance_proxy_km','size'))
    o=NAME_LOOKUP[['region_code','region_name_norm']].rename(columns={'region_code':'origin_region',
        'region_name_norm':'origin_region_name_norm'})
    t=NAME_LOOKUP[['region_code','region_name_norm']].rename(columns={'region_code':'destination_region',
        'region_name_norm':'destination_region_name_norm'})
    DISTANCE=d.merge(o,on='origin_region_name_norm',validate='many_to_one').merge(
        t,on='destination_region_name_norm',validate='many_to_one')[['origin_region','destination_region',
        'distance_proxy_km','distance_measurements']]
    DISTANCE=DISTANCE.drop_duplicates(['origin_region','destination_region'])
    note('GEOGRAPHY','Расстояние — медиана транспортных пар городов; НЕ истинное расстояние субъект-субъект')
except Exception as ex:
    note('GEOGRAPHY','Транспортный контекст недоступен: '+type(ex).__name__,'WARN')
qa_table('region_distance_proxy',DISTANCE,allow_empty=True)
# Карантин дополняется при географическом сопоставлении, исходная панель обновляется после этапа.
PANELS['entity_match_quarantine']=pd.DataFrame(QUARANTINE)

# Региональный контекст собирается только из источников с явно указанным субъектным grain.
D05_LABOR=source_asset(source_id='D05')
for a in D05_LABOR:
    if 'object_level' not in a.get('columns_manifest',[]) or 'indicator_name' not in a.get('columns_manifest',[]):continue
    try:
        path=fetch_asset(a)
        with duck_connect() as con:
            sql='''SELECT object_name, object_level, year, indicator_name, indicator_unit,
                         TRY_CAST(indicator_value AS DOUBLE) AS indicator_value,
                         COUNT(*) as duplicates
                    FROM read_parquet(?)
                    WHERE regexp_matches(lower(indicator_name), 'безработ|заработ|доход|зарплат|численность рабочей силы')
                    GROUP BY ALL'''
            v=con.execute(sql,[str(path)]).df()
        v['source_asset_id']=a['asset_id']
        v['observation_year']=safe_year(v.year)
        # Нельзя считать объект регионом только по совпадению названия; требуется документированный уровень.
        v=v[v.object_level.astype(str).str.contains(r'регион|субъект|region',case=False,regex=True,na=False)].copy()
        if len(v):REGION_CONTEXT.append(v)
    except Exception as ex:
        ISSUES.append({'module':'ML-5.1','status':'ECONOMIC_SOURCE_GATED',
            'reason':a['asset_id']+': '+type(ex).__name__,'required':'correct unit and regional grain'})
ECON_RAW=pd.concat(REGION_CONTEXT,ignore_index=True) if REGION_CONTEXT else pd.DataFrame()
qa_table('economic_indicator_candidates',ECON_RAW,allow_empty=True)

# Точное сопоставление названий; отсутствие точной пары = NA, не произвольное присвоение.
ECON_PANEL=pd.DataFrame()
if len(ECON_RAW):
    e=ECON_RAW.copy()
    e['region_name_norm']=e.object_name.astype('string').str.lower().str.replace('ё','е').str.strip()
    e=e.merge(NAME_LOOKUP[['region_code','region_name_norm']],on='region_name_norm',how='inner',
              validate='many_to_one')
    e=e.dropna(subset=['indicator_value','observation_year'])
    e['indicator_key']=e.indicator_name.astype(str)+'|'+e.indicator_unit.astype(str)
    # Только единичное значение: похожие категории и единицы не складываются.
    natural=['region_code','observation_year','indicator_key']
    multiplicity=e.groupby(natural,dropna=False)['indicator_value'].transform('size')
    if 'duplicates' in e:
        multiplicity=multiplicity.where(e['duplicates'].fillna(1).le(1),other=2)
    rejected=e.loc[multiplicity.ne(1)].copy()
    if len(rejected):
        ISSUES.append({'module':'ML-5.1','status':'ECONOMIC_GRAIN_CONFLICT',
           'reason':f'Исключено {len(rejected)} неоднозначных строк регион×год×индикатор',
           'required':'verified regional stratum and indicator definition'})
        QUARANTINE.extend(rejected.head(200).assign(source='ECON_CONFLICTING_REGION_YEAR_INDICATOR').to_dict('records'))
    e=e.loc[multiplicity.eq(1)].copy()
    if len(e) and e.duplicated(natural).any():fail('ECONOMY','Конфликт натурального ключа регион×год×индикатор')
    if not e.empty:
        top=e.groupby('indicator_key').region_code.nunique().sort_values(ascending=False).head(3).index
        e=e[e.indicator_key.isin(top)]
        ECON_PANEL=e[['region_code','observation_year','indicator_key','indicator_value','source_asset_id']]
qa_table('regional_economy',ECON_PANEL,allow_empty=True)

# Подготовка лага/скользящих сумм без доступа к целевому году.
def historical_pair_panel(observed,entity_fields,target_col,feature_lag=None,horizon=None):
    fl=CFG.FEATURE_PUBLICATION_LAG_YEARS if feature_lag is None else feature_lag
    h=CFG.HORIZON if horizon is None else horizon
    if observed.empty:return pd.DataFrame()
    o=observed.copy()
    if 'observation_year' not in o:fail('TIME','Нет observation_year')
    o['observation_year']=safe_year(o.observation_year)
    if o.duplicated(entity_fields+['observation_year']).any():fail('TIME','Дубли натурального временного ключа')
    out=o.copy()
    out['target_year']=out.observation_year.astype('Int64')
    out['origin_year']=out.target_year-h
    out['information_cutoff_year']=out.origin_year
    out['historical_year']=out.origin_year-fl
    out['historical_year_2']=out.historical_year-1
    out=out.rename(columns={target_col:'actual'})
    hist=o[entity_fields+['observation_year',target_col]].rename(columns={
        'observation_year':'historical_year',target_col:'prior_value'})
    hist2=o[entity_fields+['observation_year',target_col]].rename(columns={
        'observation_year':'historical_year_2',target_col:'prior2_value'})
    out=out.merge(hist,on=entity_fields+['historical_year'],how='left',validate='many_to_one')
    out=out.merge(hist2,on=entity_fields+['historical_year_2'],how='left',validate='many_to_one')
    out=out[out.prior_value.notna()&out.actual.notna()].copy()
    out['feature_available_year']=out.historical_year+fl
    out['target_available_year']=out.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS
    out['horizon']=h
    out['source_vintage']=INDEX['created_at']
    out['vintage_warning']='RETROSPECTIVE_ONLY_HISTORICAL_VINTAGES_NOT_ARCHIVED'
    out['split_id']='UNASSIGNED'
    if not (out.feature_available_year<=out.information_cutoff_year).all():
        fail('TIME','Будущие признаки попали в historical panel')
    return out

OD_MODEL=historical_pair_panel(
    OD_INTER[['observation_year','origin_region','destination_region','observed_value']].dropna(subset=['observed_value']).copy(),
    ['origin_region','destination_region'],'observed_value')
if not OD_MODEL.empty:
    OD_MODEL['entity_id']=OD_MODEL.origin_region.astype(str)+'|'+OD_MODEL.destination_region.astype(str)
    OD_MODEL['sector']='SPO'
    # Маргинали/число партнёров строго за доступный прошлый год.
    counts=OD_INTER.groupby(['observation_year','origin_region']).agg(
        origin_mass=('observed_value','sum'),origin_destinations=('destination_region','nunique')).reset_index()
    dests=OD_INTER.groupby(['observation_year','destination_region']).agg(
        destination_mass=('observed_value','sum'),destination_origins=('origin_region','nunique')).reset_index()
    outm=counts.rename(columns={'observation_year':'historical_year'})
    inm=dests.rename(columns={'observation_year':'historical_year'})
    OD_MODEL=OD_MODEL.merge(outm,on=['historical_year','origin_region'],validate='many_to_one',how='left')
    OD_MODEL=OD_MODEL.merge(inm,on=['historical_year','destination_region'],validate='many_to_one',how='left')
    if len(DISTANCE):OD_MODEL=OD_MODEL.merge(DISTANCE[['origin_region','destination_region','distance_proxy_km']],
                                          on=['origin_region','destination_region'],validate='many_to_one',how='left')
    # Лагирование экономических факторов проверяется датой доступности. Никаких future margins.
    if len(ECON_PANEL):
        e=ECON_PANEL.pivot(index=['region_code','observation_year'],columns='indicator_key',
                           values='indicator_value').reset_index()
        keys=[k for k in e.columns if k not in ('region_code','observation_year')]
        for direction in ('origin','destination'):
            f=e.rename(columns={'region_code':f'{direction}_region','observation_year':'historical_year',
                                **{k:f'{direction}_econ_{i}' for i,k in enumerate(keys)}})
            OD_MODEL=OD_MODEL.merge(f,on=[f'{direction}_region','historical_year'],
                                    how='left',validate='many_to_one')
    OD_MODEL['target_id']=OD_SEMANTIC['label']
    flagged=coverage_df.loc[coverage_df.potential_break.eq(True),'year'] if 'potential_break' in coverage_df else pd.Series(dtype=int)
    OD_MODEL['coverage_break_target_year']=OD_MODEL.target_year.isin(flagged).astype(bool)
    OD_MODEL['coverage_break_historical_year']=OD_MODEL.historical_year.isin(flagged).astype(bool)
qa_table('forecast_od_candidates',OD_MODEL,allow_empty=True)

# Отток: СУММА известных межрегиональных образовательных маршрутов, не migration_residence.
OUTFLOW=OD_INTER.dropna(subset=['observed_value']).groupby(['observation_year','origin_region'],as_index=False).agg(
    observed_interregional_routes=('observed_value','sum'),observed_destinations=('destination_region','nunique'))
OUTFLOW['sector']='SPO';OUTFLOW['target_id']='E01_observed_interregional_routes_from_school_region'
qa_table('outflow_observed',OUTFLOW,['observation_year','origin_region'],allow_empty=True)
OUTFLOW_MODEL=historical_pair_panel(OUTFLOW[['observation_year','origin_region','observed_interregional_routes']],
                                  ['origin_region'],'observed_interregional_routes')
if len(OUTFLOW_MODEL):
    OUTFLOW_MODEL['entity_id']=OUTFLOW_MODEL.origin_region.astype(str)
    OUTFLOW_MODEL['sector']='SPO';OUTFLOW_MODEL['target_id']='E01_observed_interregional_routes_from_school_region'
qa_table('forecast_outflow_candidates',OUTFLOW_MODEL,allow_empty=True)

ORG_MODEL=pd.DataFrame()
if len(ORG_YEAR):
    for sector,g in ORG_YEAR.groupby('sector'):
        m=historical_pair_panel(g[['entity_id','observation_year','actual']],['entity_id'],'actual')
        if len(m):
            m['sector']=sector
            m['target_id']='verified_organizational_admission_count'
            if ORG_MODEL.empty:ORG_MODEL=m
            else:ORG_MODEL=pd.concat([ORG_MODEL,m],ignore_index=True)
    if len(ORG_MODEL):
        rg=REGISTRY[['sector','entity_id','observation_year','parent_group_id','region']].rename(
            columns={'observation_year':'target_year'})
        ORG_MODEL=ORG_MODEL.merge(rg,on=['sector','entity_id','target_year'],how='left',validate='many_to_one')
qa_table('forecast_org_candidates',ORG_MODEL,allow_empty=True)

# Профиль программ по одному источнику; несопоставимые классификаторы не агрегируем.
PROGRAM_PROFILE=pd.DataFrame()
if len(PROGRAM) and {'specialty_code','value'}.issubset(PROGRAM):
    pp=PROGRAM[(PROGRAM.sector=='VO')&PROGRAM.specialty_code.notna()].copy()
    pp=pp.dropna(subset=['value','observation_year'])
    pp['specialty_code']=norm_code(pp.specialty_code)
    pp=pp[pp.value.ge(0)]
    # Не складывать разные индикаторы в одном направлении/году: оставляем строгую согласованную совокупность.
    natural=['entity_id','observation_year','specialty_code','indicator_name']
    multiplicity=pp.groupby(natural,dropna=False).size().rename('records').reset_index()
    ok=multiplicity.loc[multiplicity.records.eq(1),natural]
    excluded=int(len(pp)-len(ok))
    if excluded:ISSUES.append({'module':'ML-5.4','status':'AMBIGUOUS_PROGRAM_GRAIN',
      'reason':f'{excluded} строк нельзя однозначно агрегировать по программе; карантин без произвольного сложения',
      'required':'programme/level/form/classifier disaggregation'})
    pp=pp.merge(ok,on=natural,how='inner',validate='one_to_one')
    # Суммы на одном и том же идентификаторе индикатора допустимы лишь как суммы строк с непересекающимися организациями.
    grp=pp.groupby(['observation_year','specialty_code','indicator_name'],as_index=False).agg(
        observed_value=('value','sum'),organizations=('entity_id','nunique'))
    if not grp.empty:
        grp['sector']='VO';grp['source_grain_gate']='UNIQUE_ORG_YEAR_SPECIALTY_INDICATOR_NOT_POPULATION_TOTAL';PROGRAM_PROFILE=grp
qa_table('program_profile',PROGRAM_PROFILE,allow_empty=True)
# ВПО-1 может содержать реальную специализацию при отсутствии утверждённой организации-приёма: не отключать описание.
VPO_PROGRAM_DESCRIPTION=pd.DataFrame()
if len(vpo_admission):
    z=vpo_admission.dropna(subset=['observation_year','specialty_code','value']).copy()
    z['specialty_code']=norm_code(z.specialty_code)
    if len(z):
        VPO_PROGRAM_DESCRIPTION=z.groupby(['observation_year','specialty_code','indicator_name','object_level',
          'education_form','education_level','admission_type'],dropna=False,as_index=False).agg(
          observed_records=('value','size'),median_reported_value=('value','median'),
          distinct_geographic_names=('object_name','nunique'))
        VPO_PROGRAM_DESCRIPTION['totals_additive_confirmed']=False
if VPO_FULL_PROGRAM_PATH is None:
    qa_table('vpo_program_description',VPO_PROGRAM_DESCRIPTION,allow_empty=True)
else:
    VPO_PROGRAM_DESCRIPTION=VPO_FULL_PROGRAM_DESCRIPTION
    note('VPO','ВПО-1: полная агрегированная панель находится в evidence-Parquet; Pandas содержит только preview')

progress_event('FEATURES',f'Готовы исторические панели: OD {len(OD_MODEL):,}, отток {len(OUTFLOW_MODEL):,}',announce=True)
print('Feature coverage:',len(OD_MODEL),'OD historical rows,',len(OUTFLOW_MODEL),'outflow,',
      len(ORG_MODEL),'organization; economic indicators',len(ECON_PANEL),
      'distance pairs',len(DISTANCE))

memory_point('after_features')
mark_stage('FEATURES')


## 6А. Исследовательское использование всех групп источников

Этот блок выводит дополнительные контекстные показатели без смешения пространственных уровней: экономические и образовательные индикаторы БДМО, муниципальную историю ОКТМО, географические справочники, все E04/E05 и все разделы ВПО. Крупные группировки записываются непосредственно в Parquet. Математически корректное сопоставление сохраняет исходный `indicator_code` и ограничения на сложение пересекающихся разрезов.

In [ ]:
# FULL52 second pass: thematic *derived* panels at the original statistical grain.
# Each of 52 datasets has already passed a physical SHA + row-coverage audit above.
memory_point('before_cross_source');progress_event('CROSS_SOURCE','БДМО, ВПО, E04/E05, СПО и история ОКТМО',announce=True)
CONTEXT_PARQUET_FILES=[];D02_PROFILE=[];VPO_ALL_PROFILE=[];SPO_YEAR_CHECK=[];FAMILY_E04E05=[];SPATIAL_CROSS=[]
BDMO_EDU_PROXY=pd.DataFrame();BDMO_EDU_PROXY_CANDIDATES=pd.DataFrame()
BDMO_SPATIAL=pd.DataFrame();D04_SPATIAL=pd.DataFrame();HISTORY_MUNICIPAL=pd.DataFrame();BDMO_PROXY_FULL_VALID=False
GRAD_ALL_ASSETS=[]

def _sql_literal(v):return "'"+str(v).replace("'","''")+"'"

def thematic_sql_disk(asset,query,out_stem,preview=1500,mem='2GB'):
    # SELECT query may contain __INPUT__ placeholder; never use SELECT * to pandas.
    p=fetch_asset(asset);out=ROOT/'derived'/'full52_context'/(out_stem+'.parquet')
    out.parent.mkdir(parents=True,exist_ok=True)
    sql=query.replace('__INPUT__',_sql_literal(p))
    tmp=out.with_suffix('.partial.parquet');tmp.unlink(missing_ok=True)
    with duckdb.connect(database=':memory:') as con:
        con.execute(f"SET memory_limit='{mem}'")
        con.execute('SET threads=1');con.execute('SET preserve_insertion_order=false')
        con.execute('SET temp_directory='+_sql_literal(DB_TMP))
        con.execute('COPY ('+sql+') TO '+_sql_literal(tmp)+' (FORMAT PARQUET, COMPRESSION ZSTD)')
        data=con.execute('SELECT * FROM read_parquet(?) LIMIT '+str(int(preview)),[str(tmp)]).df()
    tmp.replace(out);pf=pq.ParquetFile(out)
    CONTEXT_PARQUET_FILES.append(out)
    return out,data,int(pf.metadata.num_rows)

# A. Every economic/BDMO source: content-level year/indicator/region coverage (but no false additive sums).
# 5-NDFL/7-NDFL cut across multiple tax forms; count/coverage ≠ observed number of residents.
for i,a in enumerate(source_asset(segment='02') if CFG.MODE=='FULL' else source_asset(prefix='02::labor'),1):
    aid=a['asset_id'];cols=set(a['columns_manifest']);progress_event('CROSS_SOURCE',f'D02 [{i}/11]: {aid}',i,11,announce=True)
    ind=next((c for c in ('indicator_code','indicator_name') if c in cols),None)
    yr='year' if 'year' in cols else None
    geo=next((c for c in ('region_name','report_region_name','object_name') if c in cols),None)
    if not ind or not yr:continue
    select_geo='CAST('+_sql_literal('UNSPECIFIED_LEVEL')+' AS VARCHAR)' if not geo else 'CAST("'+geo+'" AS VARCHAR)'
    level_where="AND regexp_matches(lower(CAST(object_level AS VARCHAR)),'регион|субъект|region')" if geo=='object_name' and 'object_level' in cols else ''
    query=f'''SELECT CAST("{yr}" AS VARCHAR) observed_year,
      {select_geo} AS reported_geography,CAST("{ind}" AS VARCHAR) indicator_key,
      COUNT(*)::BIGINT AS reported_records,
      COUNT(DISTINCT CAST("{geo}" AS VARCHAR)) AS n_locations
      FROM read_parquet(__INPUT__) WHERE "{ind}" IS NOT NULL {level_where}
      GROUP BY 1,2,3'''
    if geo is None:query=query.replace('COUNT(DISTINCT CAST("None" AS VARCHAR))','COUNT(*)::BIGINT')
    try:
        p,preview,rows=thematic_sql_disk(a,query,'d02_'+str(i).zfill(2),preview=800,mem='2GB')
        D02_PROFILE.append({'asset_id':aid,'rows_in_source':a['rows_actual'],'year_indicator_geography_groups':rows,
            'profile_parquet':p.name,'status':'GROUPED_ALL_SOURCE_ROWS_NO_DOUBLE_COUNT'})
    except Exception as ex:
        ISSUES.append({'module':'D02','status':'SOURCE_PROFILE_FAILURE','reason':aid+' '+type(ex).__name__+': '+str(ex)[:130],
                       'required':'repair dedicated source query; full SHA inventory still valid'})
        D02_PROFILE.append({'asset_id':aid,'rows_in_source':a['rows_actual'],'year_indicator_geography_groups':None,
                            'status':'GROUP_PROFILE_FAILED'})
qa_table('d02_indicator_coverage',pd.DataFrame(D02_PROFILE),['asset_id'])

# B. All 11 E02/VPO tables: count by publication year and exact indicator name, each input processed independently.
for i,a in enumerate(source_asset(segment='04') if CFG.MODE=='FULL' else source_asset(prefix='04::E02'),1):
    cols=set(a.get('columns_manifest') or [])
    if 'indicator_name' not in cols:continue
    query='''SELECT TRY_CAST(year AS INTEGER) observation_year,CAST(indicator_name AS VARCHAR) indicator_name,
      COUNT(*)::BIGINT observed_rows
      FROM read_parquet(__INPUT__) WHERE indicator_name IS NOT NULL
      GROUP BY 1,2'''
    try:
        p,preview,rows=thematic_sql_disk(a,query,'vpo_'+str(i).zfill(2),preview=500,mem='2GB')
        VPO_ALL_PROFILE.append({'asset_id':a['asset_id'],'input_rows':a['rows_actual'],
                                'year_indicator_groups':rows,'parquet':p.name,'status':'PASS'})
    except Exception as ex:
        VPO_ALL_PROFILE.append({'asset_id':a['asset_id'],'input_rows':a['rows_actual'],
                                'status':'PROFILE_FAILED_'+type(ex).__name__})
        ISSUES.append({'module':'VPO-1','status':'SOURCE_PROFILE_FAILURE','reason':a['asset_id']+' '+type(ex).__name__,
                       'required':'retry annual indicator group scan'})
qa_table('vpo_all_sections_audit',pd.DataFrame(VPO_ALL_PROFILE),['asset_id'])

# C. SPO all-years and independent yearly shards; diagnose overlaps without summing them.
spo_assets=source_asset(segment='03',role='organization_indicators')
annual=pd.DataFrame(FULL52_YEAR_STATS)
master_id=SPO_MASTER['asset_id'] if SPO_MASTER else None
if len(annual) and master_id:
    ref=annual.loc[annual.asset_id.eq(master_id)].set_index('year_label')['rows']
    for a in spo_assets:
        cur=annual[annual.asset_id.eq(a['asset_id'])]
        for year,sub in cur.groupby('year_label'):
            n=int(sub['rows'].sum());v=ref.get(year,pd.NA)
            SPO_YEAR_CHECK.append({'asset_id':a['asset_id'],'year':year,'rows_in_representation':n,
               'rows_in_master':v,'same_count':bool(n==v) if pd.notna(v) else None,
               'treatment':'NOT_ADDITIVE_WITH_MASTER'})
qa_table('spo_annual_vs_master',pd.DataFrame(SPO_YEAR_CHECK),allow_empty=True)

# D. All four E04/E05 datasets: by actual year, educational level, gender or area (never sum intersecting cross-sections).
for i,a in enumerate(source_asset(segment='05') if CFG.MODE=='FULL' else source_asset(segment='05')[:1],1):
    cols=set(a.get('columns_manifest') or [])
    group=['year','education_level']+[q for q in ('gender','study_area') if q in cols]
    keys=','.join('"'+k+'"' for k in group)
    q=f'''SELECT {keys},COUNT(*)::BIGINT source_records,
          MEDIAN(TRY_CAST(count_graduate AS DOUBLE)) median_reported_graduates,
          MEDIAN(TRY_CAST(percent_employed AS DOUBLE)) median_employed_percent,
          MEDIAN(TRY_CAST(average_salary_fact_med AS DOUBLE)) median_wage
          FROM read_parquet(__INPUT__) GROUP BY {keys}'''
    try:
        p,d,n=thematic_sql_disk(a,q,'graduates_source_'+str(i),preview=2000,mem='2GB')
        FAMILY_E04E05.append({'asset_id':a['asset_id'],'input_rows':a['rows_actual'],
           'group_rows':n,'group_columns':','.join(group),'status':'PASS_NOT_ADDITIVE',
           'employment_asof':'2024-12-31','geography_of_work_known':False})
        qa_table('graduate_family_'+str(i),d,allow_empty=True)
        PANEL_FILES['graduate_family_'+str(i)]=p
    except Exception as ex:
        FAMILY_E04E05.append({'asset_id':a['asset_id'],'status':'FAILED_'+type(ex).__name__})
        ISSUES.append({'module':'ML-5.5','status':'FAMILY_PROFILE_FAILED','reason':a['asset_id']+' '+type(ex).__name__,
           'required':'recheck exact statistical strata'})
qa_table('e04_e05_full_family_audit',pd.DataFrame(FAMILY_E04E05),['asset_id'])

# E. Historical municipal reference: full SQL join, no 431k+ rows in Pandas.
for key,asset_prefix in [('D04','06::D04::D04_member00::spatial_reference'),
                         ('BDMO_REG','06::BDMO6::BDMO6_member00::municipality_registry_observed'),
                         ('BDMO_HIST','06::BDMO6::BDMO6_member00::municipality_history_observed')]:
    a=required_one(source_asset(prefix=asset_prefix),key)
    cols=a.get('columns_manifest') or []
    chosen=[x for x in ('source_oktmo','region_name','municipality_name','settlement_name','observation_year',
      'coordinates_valid','latitude','longitude','census_year','observed_name_change','observed_type_change',
      'boundary_change_not_verified') if x in cols]
    if chosen:
        frame=read_table(a,chosen,limit=2000)  # explicitly labelled preview; full source profiled above
        if key=='D04':D04_SPATIAL=frame
        elif key=='BDMO_REG':BDMO_SPATIAL=frame
        else:HISTORY_MUNICIPAL=frame
        qa_table('spatial_'+key+'_preview',frame,allow_empty=True)

registry_asset=required_one(source_asset(prefix='06::BDMO6::BDMO6_member00::municipality_registry_observed'),'BDMO REG')
municipal_path=fetch_asset(MUN); registry_path=fetch_asset(registry_asset)
query=f'''SELECT CAST(d.oktmo AS VARCHAR) oktmo, TRY_CAST(d.year AS INTEGER) observation_year,
    MIN(CAST(d.region AS VARCHAR)) region, MIN(CAST(d.municipality AS VARCHAR)) municipality,
    MIN(CAST(b.municipality_name AS VARCHAR)) municipality_name,
    COUNT(DISTINCT CAST(b.municipality_name AS VARCHAR)) n_registry_names
    FROM read_parquet({_sql_literal(municipal_path)}) d
    INNER JOIN read_parquet({_sql_literal(registry_path)}) b
      ON CAST(d.oktmo AS VARCHAR)=CAST(b.source_oktmo AS VARCHAR)
     AND TRY_CAST(d.year AS INTEGER)=TRY_CAST(b.observation_year AS INTEGER)
    GROUP BY 1,2'''
try:
    with duckdb.connect(':memory:') as con:
        con.execute("SET memory_limit='2GB'");con.execute('SET threads=1')
        joined=con.execute(query).df()  # upper bound: 31,445 D01 rows
    joined['municipality_names_equal']=joined['municipality'].astype(str).str.lower().str.replace('ё','е').str.strip().eq(
       joined['municipality_name'].astype(str).str.lower().str.replace('ё','е').str.strip())
    joined['spatial_gate']='EXACT_OKTMO_YEAR_NAME_COMPARE_ONLY_NOT_CAMPUS'
    SPATIAL_CROSS=joined
    qa_table('spatial_exact_crosswalk',joined,['oktmo','observation_year'],allow_empty=True)
except Exception as ex:
    ISSUES.append({'module':'GEOGRAPHY','status':'CROSSWALK_FAILED','reason':type(ex).__name__+': '+str(ex)[:140],
       'required':'inspect municipality-year and historical OKTMO definitions'})
    qa_table('spatial_exact_crosswalk',pd.DataFrame(),allow_empty=True)

# F. Educational infrastructure D02::education at municipality×year×indicator_code, conservative uniqueness.
edu_source=required_one(source_asset(prefix='02::education'),'BDMO educational indicators')
edu_cols=edu_source['columns_manifest']
# Selection is a *hypothesis candidate*, not an automatic campus register.
pattern='(профессиональн|высш[еиг]|колледж|техникум|университет|образовательн.*организац)'
q=f'''SELECT CAST(oktmo AS VARCHAR) oktmo,TRY_CAST(year AS INTEGER) observation_year,
  CAST(indicator_code AS VARCHAR) indicator_code,CAST(indicator_name AS VARCHAR) indicator_name,
  CAST(indicator_unit AS VARCHAR) indicator_unit,
  COUNT(*)::BIGINT n_raw_rows,
  MIN(TRY_CAST(indicator_value AS DOUBLE)) AS min_reported,
  MAX(TRY_CAST(indicator_value AS DOUBLE)) AS max_reported,
  MEDIAN(TRY_CAST(indicator_value AS DOUBLE)) AS median_reported
  FROM read_parquet(__INPUT__)
  WHERE regexp_matches(lower(CAST(indicator_name AS VARCHAR)), {_sql_literal(pattern)})
  AND oktmo IS NOT NULL AND year IS NOT NULL
  GROUP BY 1,2,3,4,5'''
try:
    if CFG.MODE!='FULL':raise ValueError('MUNICIPAL_ECONOMIC_FULL_SCAN_ONLY')
    fp,preview,n=thematic_sql_disk(edu_source,q,'bdmo_education_municipal',preview=1000,mem='2GB')
    # Bounded filtered result only. If high-cardinality, use SQL to filter exact one-row groups first.
    with duckdb.connect(':memory:') as con:
        con.execute("SET memory_limit='2GB'");con.execute('SET threads=1')
        nuniq=int(con.execute('SELECT COUNT(*) FROM read_parquet(?) WHERE n_raw_rows=1 AND min_reported=max_reported',[str(fp)]).fetchone()[0])
        if nuniq<=CFG.MAX_PANDAS_ROWS:
            BDMO_PROXY_FULL_VALID=True
            BDMO_EDU_PROXY_CANDIDATES=con.execute('''SELECT * FROM read_parquet(?)
              WHERE n_raw_rows=1 AND min_reported=max_reported AND median_reported>=0''',[str(fp)]).df()
        else:
            BDMO_EDU_PROXY_CANDIDATES=preview.copy()
            ISSUES.append({'module':'ML-5.7','status':'PROXY_PREVIEW_ONLY',
               'reason':f'{nuniq} unique municipal education rows exceed RAM cap','required':'SQL-only E0/E1'})
    BDMO_EDU_PROXY_CANDIDATES['source_definition']='BDMO_MUNICIPAL_EDUCATION_INDICATOR_NOT_CAMPUS'
    qa_table('bdmo_education_candidates',BDMO_EDU_PROXY_CANDIDATES,allow_empty=True)
except Exception as ex:
    ISSUES.append({'module':'ML-5.7','status':'BDMO_EDUCATION_REVIEW_SKIPPED_OR_FAILED',
       'reason':type(ex).__name__+': '+str(ex)[:150],'required':'review BDMO indicator units/levels'})
    qa_table('bdmo_education_candidates',pd.DataFrame(),allow_empty=True)

# Human-readable quality conditions, all sourced from actually processed files.
qa_table('municipal_infrastructure_candidates',BDMO_EDU_PROXY_CANDIDATES,allow_empty=True)
print('CROSS SOURCE:',len(D02_PROFILE),'D02 members;',len(VPO_ALL_PROFILE),'E02/VPO;',
      len(FAMILY_E04E05),'E04/E05;',len(SPATIAL_CROSS),'exact municipal code-year matches;',
      len(BDMO_EDU_PROXY_CANDIDATES),'municipal infrastructure candidates')
memory_point('after_cross_source')
# Финальная отметка этапа перенесена после всех его подмодулей


## 6Б. Содержательная проверка 52/52 источников
Пофайловое содержание без подмены единиц измерения; годовые СПО и raw-матрицы не суммируются повторно.

In [ ]:
# ML-05 V4 · independent full-content utilization register (52/52).
# These are measurements of the SOURCE itself, not automatically legal predictive features.
progress_event('CROSS_SOURCE','V4: физическое обследование значений всех наборов Stage I',announce=True)
V4_SOURCE_PROFILES=[];V4_SOURCE_ERRORS=[]
for _i,_a in enumerate(ASSETS,1):
    _id=_a['asset_id'];_p=fetch_asset(_a);_cols=list(pq.ParquetFile(_p).schema_arrow.names)
    _time=next((x for x in ('year','observation_year','graduation_year') if x in _cols),None)
    _measure=next((x for x in ('indicator_value','observed_value','count_graduate','population','value','migration') if x in _cols),None)
    _y=f'TRY_CAST("{_time}" AS INTEGER)' if _time else 'NULL::INTEGER'
    _v=f'TRY_CAST("{_measure}" AS DOUBLE)' if _measure else 'NULL::DOUBLE'
    _q=f'''SELECT {_y} year_label,COUNT(*)::BIGINT source_rows,
      COUNT({_v})::BIGINT numeric_values,
      SUM(CASE WHEN {_v}<0 THEN 1 ELSE 0 END)::BIGINT negative_values,
      SUM(CASE WHEN {_v}=0 THEN 1 ELSE 0 END)::BIGINT zero_values,
      SUM(CASE WHEN {_v} IS NULL THEN 1 ELSE 0 END)::BIGINT null_or_non_numeric,
      MIN({_v}) AS minimum_observed_value,
      MAX({_v}) AS maximum_observed_value,
      AVG({_v}) AS unweighted_mean_of_source_records,
      STDDEV_POP({_v}) AS dispersion_of_source_records
      FROM read_parquet({_sql_literal(_p)}) GROUP BY 1'''
    _out=ROOT/'derived'/'v4_source_values'/f'source_{_i:02d}.parquet';_out.parent.mkdir(parents=True,exist_ok=True)
    _tmp=_out.with_suffix('.partial.parquet');_tmp.unlink(missing_ok=True)
    progress_event('CROSS_SOURCE',f'V4 [{_i}/52] число, пропуски, нули: {_id}',_i,len(ASSETS),announce=(_i==1 or _i%5==0 or _i==len(ASSETS)))
    try:
        with duckdb.connect(':memory:') as _db:
            _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
            _db.execute('SET preserve_insertion_order=false')
            _db.execute('SET temp_directory='+_sql_literal(DB_TMP))
            _db.execute('COPY ('+_q+') TO '+_sql_literal(_tmp)+' (FORMAT PARQUET, COMPRESSION ZSTD)')
            _r=_db.execute('SELECT COALESCE(SUM(source_rows),0),COUNT(*) FROM read_parquet(?)',[str(_tmp)]).fetchone()
        if int(_r[0])!=int(_a['rows_actual']):raise AssertionError(f'Content count {_r[0]} != {_a["rows_actual"]}')
        _tmp.replace(_out);CONTEXT_PARQUET_FILES.append(_out)
        V4_SOURCE_PROFILES.append({'asset_id':_id,'segment':_a['segment'],'source_id':_a.get('source_id'),
           'rows_actual':int(_r[0]),'time_field':_time,'numeric_field':_measure,
           'groups_by_year':int(_r[1]),'sha_verified':_id in INPUT_SHA,'content_status':'FULL_VALUE_COVERAGE_PASS',
           'model_eligibility':'REQUIRES_TARGET_SEMANTICS_AND_GRAIN','derived_parquet':_out.name})
    except Exception as _ex:
        _tmp.unlink(missing_ok=True)
        V4_SOURCE_ERRORS.append({'asset_id':_id,'error':type(_ex).__name__,'details':str(_ex)[:280]})
        V4_SOURCE_PROFILES.append({'asset_id':_id,'segment':_a['segment'],'source_id':_a.get('source_id'),
           'rows_actual':None,'time_field':_time,'numeric_field':_measure,'groups_by_year':None,
           'sha_verified':_id in INPUT_SHA,'content_status':'FAIL_CONTENT_SCAN','derived_parquet':''})
_v4_used=pd.DataFrame(V4_SOURCE_PROFILES)
qa_table('v4_all52_source_value_audit',_v4_used,['asset_id'])
qa_table('v4_source_scan_failures',pd.DataFrame(V4_SOURCE_ERRORS),allow_empty=True)
if CFG.MODE=='FULL' and (len(_v4_used)!=52 or len(V4_SOURCE_ERRORS)):
    fail('ALL52_CONTENT','One or more Stage I members failed full numeric content scan: '+str(V4_SOURCE_ERRORS[:2]))

# Source -> defensible research use. Each stage is recorded independently, no claim of 52 model predictors.
_v4_roles=[]
for _a in ASSETS:
    _seg=_a['segment'];_r=_a.get('role','') or ''
    _topics={'01':'Municipal demography and ML-5.7','02':'Economic/education context and ML-5.1/ML-5.7',
      '03':'SPO OD, organizational panels and network ML-5.1/2/3/6',
      '04':'VO monitoring, admissions and specialties ML-5.2/5.4',
      '05':'Graduate employability and outcomes ML-5.5',
      '06':'Historic OKTMO, places and geographic accessibility ML-5.1/5.7'}
    _direct=_a['asset_id'] in {E01['asset_id'],E02['asset_id'],VPO_211['asset_id'],
              GRAD_ORG['asset_id'],GRAD_FIELD['asset_id'],MUN['asset_id'],TRANSPORT['asset_id']}
    _v4_roles.append({'asset_id':_a['asset_id'],'segment':_seg,'role_in_program':_topics.get(_seg),
       'content_scanned':True,'scientific_content_used':'YES_SOURCE_SPECIFIC_PROFILE',
       'main_prediction_input':bool(_direct),
       'not_main_input_reason':('Derived/alternate reporting representation or incompatible key/grain;'
                                'not pooled into target by assumption' if not _direct else ''),
       'prevent_double_counting':_seg=='03' and (_r=='organization_indicators' or _r=='education_od_matrix_raw_cells'),
       'source_sha':INPUT_SHA.get(_a['asset_id'],'')})
qa_table('v4_all52_research_use',pd.DataFrame(_v4_roles),['asset_id'])
print('V4 / source use:',len(V4_SOURCE_PROFILES),'complete value-audits;',len(V4_SOURCE_ERRORS),'failures')


In [ ]:
# V7.4.0 · источники: обследование содержимого и проверяемые аналитические роли.
# Ни один из 52 наборов не объявляется модельным признаком только потому, что SHA сошлась.
progress_event('CROSS_SOURCE','V7: контроль научного обследования всех 52 источников',announce=True)
V7_SOURCE_AUDIT=[]
_v7_profile={r['asset_id']:r for r in V4_SOURCE_PROFILES}
_v7_inventory={r['asset_id']:r for r in FULL52_INVENTORY}
_v7_roles={
 '01':'Муниципальная демография: временные величины и состав E0/E1',
 '02':'Доходы, занятость, бюджет и образование: раздельные источниковые распределения',
 '03':'СПО, OD, реестры, сопоставление матриц и годовых вариантов',
 '04':'Мониторинг ВО, приём и структура специальностей',
 '05':'Выпуск, занятость и зарплаты: отдельные непересекающиеся страты',
 '06':'История ОКТМО, география, поселения и транспорт'
}
for _a in ASSETS:
    _id=_a['asset_id'];_p=_v7_profile.get(_id,{});_i=_v7_inventory.get(_id,{})
    _full=_p.get('content_status')=='FULL_VALUE_COVERAGE_PASS'
    _asset_name=_p.get('derived_parquet','')
    _stats_path=ROOT/'derived'/'v4_source_values'/_asset_name
    V7_SOURCE_AUDIT.append({
      'asset_id':_id,'source_group':_a['segment'],'rows_in_manifest':int(_a['rows_actual']),
      'sha256':_a['sha256'],'verified_sha256':INPUT_SHA.get(_id)==_a['sha256'],
      'footer_rows_verified':bool(_i.get('footer_rows_ok')),
      'all_values_profiled':bool(_full),
      'numeric_profile_file':_asset_name,
      'numeric_profile_exists':bool(_stats_path.is_file()),
      'time_field':_p.get('time_field'),'numeric_field':_p.get('numeric_field'),
      'source_specific_statistical_use':_v7_roles.get(_a['segment'],'UNREGISTERED'),
      'predictor_role':'NOT_DETERMINED_UNTIL_MODEL_FIT',
      'scientific_constraint':'NO_CROSS_SOURCE_ADDITIVITY_WITHOUT_GRAIN_AND_UNIT_VALIDATION'
    })
V7_SOURCE_AUDIT=pd.DataFrame(V7_SOURCE_AUDIT)
qa_table('v7_all52_scientific_source_audit',V7_SOURCE_AUDIT,['asset_id'])
if CFG.MODE=='FULL' and (len(V7_SOURCE_AUDIT)!=52 or not
        (V7_SOURCE_AUDIT.verified_sha256 & V7_SOURCE_AUDIT.footer_rows_verified &
         V7_SOURCE_AUDIT.all_values_profiled & V7_SOURCE_AUDIT.numeric_profile_exists).all()):
    raise RuntimeError('V7: хотя бы один источник не прошёл полный числовой/временной/хеш-контроль')
print('V7: исходники 52/52 проверены, содержательные профили раздельны; модельная трассировка будет сформирована после обучения')

mark_stage('CROSS_SOURCE')


## 7. Предрегистрация, временные разбиения, модели
Сопоставляются persistence, среднее двух наблюдений, затухающий тренд, Ridge, CatBoost; PPML только для доказанных неотрицательных счётных величин. TEST не выбирает победителя.

In [ ]:
progress_event('MODEL_DESIGN','Разбиения, метрики, базовые методы, Ridge/CatBoost')
# ML-05 · план эксперимента и функции, одинаковые для всех доступных модулей.
EXPERIMENT_PLAN={
 'version':CFG.VERSION,'mode':CFG.MODE,'horizon':CFG.HORIZON,
 'source_commit':SOURCE_COMMIT,'source_index_blob':index_info['sha'],
 'feature_lag':CFG.FEATURE_PUBLICATION_LAG_YEARS,
 'label_lag':CFG.LABEL_PUBLICATION_LAG_YEARS,
 'candidate_models':['persistence','rolling_mean_2','damped_trend','ridge','elasticnet','catboost','lightgbm','ppml_if_observed_integer_counts','negbin_fixed_dispersion_if_integer_counts'],
 'admission_approval_required':True,'unknown_edges_never_zero':True,
 'E01_is_not_certified_admissions':True,
 'validation':'DEV only if its label is available by TEST origin; otherwise fixed preregistered comparison, no selection',
 'seeds_dev':CFG.DEV_SEEDS,'seeds_final':CFG.FINAL_SEEDS,
 'test_keys_policy':'strict common support; no synthetic missing OD edges',
 'source_vintage':INDEX['created_at'],
 'publication_policy':'only FULL + local checks + explicit opt-in + science gate',
}
PLAN_HASH=canonical_hash(EXPERIMENT_PLAN)
PRED_KEYS=['module','target_id','sector','entity_id','origin_year','horizon','target_year',
           'split_id','model','seed','information_cutoff_year','source_vintage',
           'geography_grain','observation_definition','actual','predicted']

# Предварительно заданные признаки; семантика источника не попадает в X как oracle.
CAND_NUM=['prior_value','prior2_value','origin_mass','destination_mass',
          'origin_destinations','destination_origins','distance_proxy_km',
          'confirmed_vo_sites','prev_population','prev_migration_rate',
          'origin_year']
CAND_CAT=['origin_region','destination_region','region','sector','parent_group_id']

def feature_spec(frame):
    nums=[c for c in CAND_NUM if c in frame.columns and number(frame[c]).notna().any()]
    cats=[c for c in CAND_CAT if c in frame.columns and frame[c].notna().any()]
    for c in frame.columns:
        if c.startswith(('origin_econ_','destination_econ_')) and number(frame[c]).notna().any():nums.append(c)
    if 'prior_value' not in nums:fail('MODEL','Нет prior_value: нельзя оценить историческую инерцию')
    return nums,cats

def x_prepare(frame,nums,cats):
    x=frame[nums+cats].copy()
    for c in nums:x[c]=number(x[c]).astype(float)
    for c in cats:x[c]=x[c].fillna('__MISSING__').astype(str)
    return x

def effective_mae(y,yp):
    y=np.asarray(y,dtype=float);p=np.asarray(yp,dtype=float)
    mask=np.isfinite(y)&np.isfinite(p)
    if not mask.any():return {'MAE':np.nan,'RMSE':np.nan,'WAPE':np.nan,'n':0}
    y,p=y[mask],p[mask]
    return {'MAE':float(np.mean(np.abs(y-p))),'RMSE':float(np.sqrt(np.mean((y-p)**2))),
            'WAPE':float(np.abs(y-p).sum()/np.abs(y).sum()) if np.abs(y).sum()>0 else np.nan,'n':int(len(y))}

# Проверка, что один и тот же target не оказался одновременно в train/dev/test.
def temporal_partition(frame,min_train=None,min_test=None):
    if frame.empty:return None,'EMPTY_FRAME'
    f=frame.sort_values('target_year').copy()
    years=sorted(int(x) for x in f.target_year.dropna().unique())
    if len(years)<2:return None,'INSUFFICIENT_TARGET_YEARS'
    test_year=years[-1];test_origin=test_year-CFG.HORIZON
    tr=f[(f.target_year<test_year)&
         (f.target_available_year<=test_origin)].copy()
    test=f[f.target_year==test_year].copy()
    min_train=int(CFG.MIN_TRAIN_ROWS if min_train is None else min_train)
    min_test=int(CFG.MIN_TEST_ROWS if min_test is None else min_test)
    if len(tr)<min_train or len(test)<min_test:
        return None,f'LOW_SUPPORT: train={len(tr)},test={len(test)}'
    # DEV может существовать только если доступен к моменту финального решения.
    available_dev=sorted(int(y) for y in tr.target_year.unique())
    dev=None
    if len(available_dev)>=2:
        dv=available_dev[-1];dev_origin=dv-CFG.HORIZON
        early=tr[(tr.target_year<dv)&(tr.target_available_year<=dev_origin)].copy()
        potential=tr[tr.target_year==dv].copy()
        if len(early)>=min_train and len(potential)>=CFG.MIN_DEV_ROWS:
            dev=potential;tr=early
    if tr.duplicated(['entity_id','target_year']).any():fail('SPLIT','Неверная кратность train')
    if test.duplicated(['entity_id','target_year']).any():fail('SPLIT','Неверная кратность test')
    if (tr.target_available_year>test_origin).any():fail('SPLIT','train label published after test origin')
    if (test.feature_available_year>test.information_cutoff_year).any():fail('SPLIT','future feature in test')
    if dev is not None and (dev.target_available_year>test_origin).any():fail('SPLIT','dev label leaked to test')
    out={'train':tr,'dev':dev,'test':test,'test_year':test_year,'test_origin':test_origin,
         'protocol':'DEV_SELECTED' if dev is not None else 'FIXED_NO_DEV',
         'retrospective_vintage':True}
    SPLIT_AUDIT.append({'module':f.get('module',pd.Series(['UNSET'])).iloc[0],
       'protocol':out['protocol'],'train_rows':len(tr),'dev_rows':0 if dev is None else len(dev),
       'test_rows':len(test),'test_year':test_year,'test_origin':test_origin,
       'labels_known_asof_test':True,'source_vintage_at_test':'NOT_ARCHIVED'})
    return out,None


# V4: fixed-dispersion NB GLM, not mislabeled as an estimated NB2 dispersion model.
from sklearn.base import BaseEstimator, RegressorMixin
class NegativeBinomialFixedDispersion(RegressorMixin, BaseEstimator):
    """NB2 (alpha fixed), penalized MLE with stable log-link and declared prediction-support bound.

    Count targets must be observed nonnegative integers. Fits a low-dimensional, numeric-only
    design with L2 penalty; never densifies the high-cardinality categorical one-hot matrix.
    The predeclared upper bound is 10×max(train count): clipping is reported as diagnostic,
    so predictions cannot silently explode into 1e68 while still receiving PASS.
    """
    def __init__(self, alpha=1.0, penalty=0.5, support_multiplier=10.0):
        self.alpha=float(alpha);self.penalty=float(penalty);self.support_multiplier=float(support_multiplier)
    def fit(self,X,y):
        from scipy.optimize import minimize
        from scipy.special import expit
        a=np.asarray(X,dtype=np.float64); y=np.asarray(y,dtype=np.float64).reshape(-1)
        if a.ndim==1:a=a.reshape(-1,1)
        if a.shape[0]!=len(y) or not len(y) or a.shape[1]>64:raise ValueError('NB2 incompatible design')
        if not np.isfinite(y).all() or (y<0).any() or not np.allclose(y,np.round(y),atol=1e-8,rtol=0):
            raise ValueError('NB2 requires observed integer nonnegative counts')
        a=np.clip(np.nan_to_num(a,nan=0.0,posinf=0.0,neginf=0.0),-8.,8.)
        a=np.column_stack([np.ones(len(y)),a]); r=1.0/self.alpha
        # Stable NB log-likelihood: (y+r)*softplus(eta-log(r))-y*(eta-log(r)), up to constants.
        scale=max(1.,float(np.mean(y+r)));x0=np.zeros(a.shape[1]);x0[0]=np.log(max(.001,float(np.mean(y))))
        def fun(beta):
            e=a@beta-np.log(r)
            loss=np.mean(((y+r)*np.logaddexp(0.,e)-y*e))/scale+0.5*self.penalty*np.dot(beta[1:],beta[1:])
            g=(a.T@((y+r)*expit(e)-y))/(len(y)*scale)
            g[1:]+=self.penalty*beta[1:]
            return float(loss),g
        bounds=[(-22.,22.)]+[(-6.,6.)]*(a.shape[1]-1)
        opt=minimize(fun,x0,method='L-BFGS-B',jac=True,bounds=bounds,options={'maxiter':350,'ftol':1e-10,'gtol':1e-7})
        if not opt.success or not np.isfinite(opt.fun):
            raise RuntimeError('NB2 regularized likelihood did not converge: '+str(opt.message))
        self.coef_=opt.x;self.fit_converged_=True;self.fit_objective_=float(opt.fun)
        self.train_max_=float(np.max(y));self.train_mean_=float(np.mean(y))
        self.n_fit_=len(y);self.count_truncated_=0;self.max_raw_prediction_=np.nan
        return self
    def predict(self,X):
        a=np.asarray(X,dtype=np.float64)
        if a.ndim==1:a=a.reshape(-1,1)
        a=np.clip(np.nan_to_num(a,nan=0.,posinf=0.,neginf=0.),-8.,8.)
        eta=np.column_stack([np.ones(len(a)),a])@self.coef_
        # The model's response space is constrained by a *predeclared* historical support bound.
        # This is not replacement of missing observations. Truncation frequency is an output QC.
        max_mean=max(1.,self.train_max_)*self.support_multiplier
        clipped=np.clip(eta,-30.,np.log(max_mean))
        self.count_truncated_=int(np.count_nonzero(clipped!=eta))
        self.truncated_fraction_=float(self.count_truncated_/max(1,len(a)))
        self.max_raw_logmu_=float(np.max(eta)) if len(a) else np.nan
        pred=np.exp(clipped)
        if not np.isfinite(pred).all():raise FloatingPointError('NB2 unstable response')
        return pred

def pipeline_fit_predict(model,train,test,seed=11,allow_ppml=False,nonnegative=True):
    nums,cats=feature_spec(train)
    if set(nums+cats)&set(['actual','target_year','target_available_year','observed_value']):
        fail('LEAKAGE','В модель попала фактическая целевая переменная или период метки')
    Xtr=x_prepare(train,nums,cats);Xte=x_prepare(test,nums,cats)
    y=number(train.actual).to_numpy(float)
    if not np.isfinite(y).all():fail('MODEL','Невалидные training labels')
    if model=='ridge':
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                                        ('scale',StandardScaler())]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.5)),
                      ('model',Ridge(alpha=30.,solver='lsqr'))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='elasticnet':
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                                        ('scale',StandardScaler())]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.7)),
             ('model',ElasticNet(alpha=1.0,l1_ratio=.35,max_iter=4000,random_state=(None if int(seed)<0 else int(seed)),selection='cyclic'))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='lightgbm':
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median'))]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.6)),
             ('model',LGBMRegressor(n_estimators=220,num_leaves=15,max_depth=5,learning_rate=.05,
                       reg_lambda=8.,min_child_samples=15,verbosity=-1,n_jobs=1,random_state=(None if int(seed)<0 else int(seed))))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='ppml':
        if not allow_ppml or np.any(y<0):raise ValueError('PPML requires verified nonnegative counts')
        transformers=[('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                                        ('scale',StandardScaler())]),nums)]
        if cats:transformers.append(('cats',OneHotEncoder(handle_unknown='ignore'),cats))
        obj=Pipeline([('prep',ColumnTransformer(transformers,sparse_threshold=.5)),
                      ('model',PoissonRegressor(alpha=3.,max_iter=200))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats}
    elif model=='negbin':
        if not allow_ppml or np.any(y<0) or not np.isfinite(y).all():
            raise ValueError('Negative binomial GLM requires observed nonnegative finite counts')
        # NB2: numeric-only design; avoids high-cardinality one-hot IRLS explosion.
        # All numeric inputs are observed past information, not current target.
        obj=Pipeline([('prep',ColumnTransformer([
                 ('nums',Pipeline([('fill',SimpleImputer(strategy='median')),
                          ('scale',StandardScaler())]),nums)],sparse_threshold=0.)),
            ('model',NegativeBinomialFixedDispersion(alpha=1.,penalty=0.5,
                          support_multiplier=CFG.MAX_COUNT_PREDICTION_MULTIPLIER))])
        obj.fit(Xtr,y);yp=obj.predict(Xte)
        cats=[];artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats,
                  'dispersion':'FIXED_ALPHA_1_PENALIZED_NB2',
                  'extrapolation_rule':'PREDICT_MEAN_AT_MOST_10X_MAX_OBSERVED_TRAIN_COUNT',
                  'prediction_guard_events':int(obj.named_steps['model'].count_truncated_),
                  'prediction_guard_fraction':float(obj.named_steps['model'].truncated_fraction_)}
    elif model=='catboost':
        med={c:float(Xtr[c].median()) if Xtr[c].notna().any() else 0. for c in nums}
        for c in nums:
            Xtr[c]=Xtr[c].fillna(med[c]);Xte[c]=Xte[c].fillna(med[c])
        obj=CatBoostRegressor(iterations=60 if CFG.MODE=='SMOKE' else 260,
                  depth=5,learning_rate=.05,l2_leaf_reg=8.,loss_function='RMSE',
                  random_seed=int(seed),verbose=False,thread_count=1,used_ram_limit='2gb',allow_writing_files=False)
        obj.fit(Xtr,y,cat_features=cats);yp=obj.predict(Xte)
        artifact={'kind':model,'pipeline':obj,'nums':nums,'cats':cats,'medians':med}
    else:fail('MODEL','Неизвестная модель '+model)
    yp=np.asarray(yp,dtype=np.float64)
    if len(yp)!=len(test) or not np.isfinite(yp).all():raise FloatingPointError(model+' invalid prediction array')
    if nonnegative and (yp<0).any():yp=np.maximum(yp,0)
    return yp,artifact

def reload_predict(artifact,test,nonnegative=True):
    X=x_prepare(test,artifact['nums'],artifact['cats'])
    if artifact['kind']=='catboost':
        for c,v in artifact['medians'].items():X[c]=X[c].fillna(v)
    p=artifact['pipeline'].predict(X)
    return np.maximum(p,0) if nonnegative else p

def baseline_prediction(test,kind,nonnegative=True):
    a=number(test.prior_value).to_numpy(float)
    if kind=='persistence':b=a
    elif kind in ('rolling_mean_2','damped_trend'):
        other=number(test.prior2_value).to_numpy(float)
        b=np.where(np.isfinite(other),(a+other)/2 if kind=='rolling_mean_2' else a+.35*(a-other),a)
    else:fail('BASELINE','Неизвестный baseline '+kind)
    return np.maximum(b,0) if nonnegative else b

def predictions_frame(frame,module,model,yp,seed=-1,protocol='ROLLING_TIME'):
    out=pd.DataFrame({'module':module,'target_id':frame.get('target_id',pd.Series(['UNDEFINED']*len(frame))).to_numpy(),
     'sector':frame.get('sector',pd.Series(['UNKNOWN']*len(frame))).to_numpy(),
     'entity_id':frame.entity_id.astype(str).to_numpy(),
     'origin_year':frame.origin_year.to_numpy(),'horizon':CFG.HORIZON,'target_year':frame.target_year.to_numpy(),
     'split_id':protocol,'model':model,'seed':int(seed),
     'information_cutoff_year':frame.information_cutoff_year.to_numpy(),
     'feature_available_year':frame.feature_available_year.to_numpy(),
     'target_available_year':frame.target_available_year.to_numpy(),
     'source_vintage':INDEX['created_at'],
     'geography_grain':'ORIGIN_DESTINATION' if module=='ML-5.1' else ('REGION' if module=='ML-5.3' else 'ORGANIZATION'),
     'observation_definition':frame.get('target_id',pd.Series(['observed_value']*len(frame))).to_numpy(),
     'actual':number(frame.actual).to_numpy(float),'predicted':np.asarray(yp,dtype=float)})
    for c in ('origin_region','destination_region','parent_group_id','region'):
        if c in frame:out[c]=frame[c].astype('string').to_numpy()
    return out

def record_prediction(frame):
    # Прогнозы всех сидов сохраняются немедленно; массивы не остаются в RAM.
    global PRED_TOTAL
    out=ROOT/'derived'/'prediction_parts';out.mkdir(parents=True,exist_ok=True)
    path=out/f'part_{len(PRED_PARTS):05d}.parquet'
    write_df_chunks(frame,path)
    PRED_PARTS.append(str(path));PRED_TOTAL+=int(len(frame))
    del frame
    if len(PRED_PARTS)%10==0:
        gc.collect();memory_point(f'predictions_saved_{len(PRED_PARTS)}')

def fit_and_record(module,model,train,test,seed=-1,nonnegative=True,allow_ppml=False,protocol='ROLLING_TIME'):
    sector=str(test.sector.iloc[0]) if 'sector' in test and test.sector.nunique()==1 else 'MIXED'
    if model in ('persistence','rolling_mean_2','damped_trend'):
        pred=baseline_prediction(test,model,nonnegative);artifact=None
    else:
        pred,artifact=pipeline_fit_predict(model,train,test,seed=seed,allow_ppml=allow_ppml,
                                         nonnegative=nonnegative)
        ident={'module':module,'sector':sector,'protocol':protocol,
               'target_id':sorted(test.target_id.astype(str).unique().tolist()) if 'target_id' in test else [],
               'model':model,'seed':int(seed)}
        short=hashlib.sha256(json.dumps(ident,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:16]
        name=f'{module.replace("-","").replace(".","_")}_{re.sub(r"[^A-Za-z0-9_]+","_",sector)[:28]}_{re.sub(r"[^A-Za-z0-9_]+","_",protocol)[:32]}_{model}_{seed}_{short}.joblib'
        path=ROOT/'models'/name
        if name in MODEL_FILES or path.exists():
            raise RuntimeError('DUPLICATE_MODEL_ARTIFACT: '+name)
        joblib.dump(artifact,path,compress=3)
        loaded=joblib.load(path)
        rp=reload_predict(loaded,test,nonnegative=nonnegative)
        if not np.allclose(rp,pred,rtol=1e-7,atol=1e-7):fail('MODEL_RELOAD',name+' discrepancy')
        MODEL_FILES[name]=str(path)
        MODEL_RECORDS.append({'module':module,'sector':sector,'protocol':protocol,'target_id':ident['target_id'],'model':model,'seed':seed,'path':name,
            'file_sha256':hash_file(path),'feature_num':artifact['nums'],'feature_cat':artifact['cats'],
            'training_rows':len(train),'test_rows':len(test),'reloaded':True,
            'prediction_guard_events':artifact.get('prediction_guard_events',0),
            'prediction_guard_fraction':artifact.get('prediction_guard_fraction',0.)})
    if len(pred)!=len(test):fail('SUPPORT','Модель изменила число тестовых ключей')
    record_prediction(predictions_frame(test,module,model,pred,seed,protocol))
    met=effective_mae(test.actual.to_numpy(float),pred)
    METRICS.append({'module':module,'sector':sector,'model':model,'protocol':protocol,'seed':seed,
       'target_year':int(test.target_year.max()),'source_vintage':INDEX['created_at'],
       **met})
    # Процент неизвестен (ветвление зависит от данных); выводим точный счётчик.
    progress_event(None,f'{module}: {model}, рассчитано {len(METRICS):,} метрик, сохранено {len(MODEL_FILES):,} моделей',
                   current=len(METRICS),total=None,announce=False)
    return pred

mark_stage('MODEL_DESIGN')


## 8. Встроенные интеграционные проверки без GitHub-публикации
11 автоматических тестов на малом искусственном наборе проверяют чистоту времени, восстановление моделей и карантин. Их численные значения не используются как научные результаты.

In [ ]:
progress_event('UNIT_TESTS','Проверка моделей, утечек и повторного чтения')
# Локальные встроенные тесты математической и организационной логики; не являются научными наблюдениями.
SELF_TESTS=[]
def unit_result(name,condition):
    SELF_TESTS.append({'test':name,'result':'PASS' if bool(condition) else 'FAIL'})
    if not bool(condition):fail('UNIT',name)

# Временной split — метки всегда доступны к соответствующему моменту принятия решения.
_fixture=pd.DataFrame([{'entity_id':f'TEST_{e:02d}','observation_year':y,
                        'response':float(e+2*(y-2010)), 'sector':'UNIT'}
                       for e in range(35) for y in range(2012,2026)])
_panel=historical_pair_panel(_fixture[['entity_id','observation_year','response']],['entity_id'],'response')
_panel['sector']='UNIT';_panel['module']='UNIT';_panel['region']='UNIT_REGION'
_panel['target_id']='ARTIFICIAL_TEST_ONLY'
_split,_reason=temporal_partition(_panel)
unit_result('TIME_PANEL_EXISTS',len(_panel)>100)
unit_result('NO_FUTURE_FEATURES',bool((_panel.feature_available_year<=_panel.information_cutoff_year).all()))
unit_result('SPLIT_AVAILABLE',_split is not None)
unit_result('TRAIN_LABELS_KNOWN_AS_OF_TEST',bool((_split['train'].target_available_year<=_split['test_origin']).all()))
unit_result('TEST_KEYS_UNIQUE',not _split['test'].duplicated(['entity_id','target_year']).any())
if _split['dev'] is not None:
    unit_result('DEV_LABELS_KNOWN_AS_OF_TEST',bool((_split['dev'].target_available_year<=_split['test_origin']).all()))

# Модельный прогноз должен повторяться без дообучения после восстановления состояния.
_train=_split['train'];_test=_split['test'];_y_ridge,_ridge_art=pipeline_fit_predict('ridge',_train,_test)
unit_result('RIDGE_RELOAD_EQUAL',np.allclose(_y_ridge,reload_predict(_ridge_art,_test),atol=1e-8,rtol=1e-8))
_cb_y,_cb_art=pipeline_fit_predict('catboost',_train,_test,seed=CFG.DEV_SEEDS[0])
unit_result('CATBOOST_RELOAD_EQUAL',np.allclose(_cb_y,reload_predict(_cb_art,_test),atol=1e-7,rtol=1e-7))
unit_result('MODEL_OUTPUT_FINITE',np.isfinite(_y_ridge).all() and np.isfinite(_cb_y).all())
_en_y,_en_art=pipeline_fit_predict('elasticnet',_train,_test,seed=-1)
unit_result('ELASTICNET_SENTINEL_NEGATIVE_SEED_SAFE',np.isfinite(_en_y).all() and
             np.allclose(_en_y,reload_predict(_en_art,_test)))
_nb_fixture=_train.copy();_nb_test=_test.copy()
for _q in (_nb_fixture,_nb_test):_q['actual']=np.maximum(np.rint(_q.actual),0)
_po_y,_po_art=pipeline_fit_predict('ppml',_nb_fixture,_nb_test,seed=-1,allow_ppml=True)
unit_result('PPML_OBSERVED_INTEGER_FIT_AND_RELOAD',np.isfinite(_po_y).all() and
             np.allclose(_po_y,reload_predict(_po_art,_nb_test)))
_nb_y,_nb_art=pipeline_fit_predict('negbin',_nb_fixture,_nb_test,seed=-1,allow_ppml=True)
unit_result('NEGBIN_FIXED_DISPERSION_RELOAD',np.isfinite(_nb_y).all() and
             np.allclose(_nb_y,reload_predict(_nb_art,_nb_test)))
unit_result('BASELINE_NOT_FUTURE',np.allclose(baseline_prediction(_test,'persistence'),_test.prior_value))

# Спорный id×год отбрасывается с содержательным карантином, не объединяется по имени.
_quarantine_n=len(QUARANTINE)
_bad_registry=pd.DataFrame([{'entity_id':'X','observation_year':2024,'name':'Вуз 1'},
                            {'entity_id':'X','observation_year':2024,'name':'Вуз 2'},
                            {'entity_id':'Y','observation_year':2024,'name':'Колледж'}])
_clean=quarantine_conflicting_registry(_bad_registry,'TEST_ONLY')
unit_result('AMBIGUOUS_REGISTRY_QUARANTINED',len(_clean)==1 and _clean.entity_id.iloc[0]=='Y')
del QUARANTINE[_quarantine_n:] # тестовые записи не являются реальными спорными организациями

qa_table('embedded_unit_tests',pd.DataFrame(SELF_TESTS),['test'])
assert all(r['result']=='PASS' for r in SELF_TESTS)
print('ML-05 embedded tests:',len(SELF_TESTS),'PASS (контрольные данные; не научные результаты)')
del _fixture,_panel,_split,_reason,_train,_test,_ridge_art,_cb_art,_en_art,_en_y,_po_art,_po_y,_nb_art,_nb_y,_nb_fixture,_nb_test,_y_ridge,_cb_y,_quarantine_n,_bad_registry,_clean

mark_stage('UNIT_TESTS')


## 9. Семь предметных блоков ML-5.1–ML-5.7
Научные блоки выполняются на фактически наблюдаемых источниках. При отсутствии target или подтверждённого grain предусмотрен описательный результат или мотивированный отказ от прогноза.

In [ ]:
memory_point('before_model_training')
progress_event('SEVEN_MODULES','Расчёт семи модулей; расширенные статусы',announce=True)
# Семь предметных экспериментов. Наличие модели не является условием выполнения описательной части.
MODULE_DIAGNOSTICS=[];CONFIRMATORY=[];HYPOTHESES=[];MODULE_SUBTASKS=[]

def blocked(module,why,required=''):
    limitation(module,why,required)
    MODULE_DIAGNOSTICS.append({'module':module,'prediction_feasible':False,
        'n_observed':0,'reason':why})

def two_way_mae_bootstrap(base,model,groups_a,groups_b,reps=399,seed=20261104):
    # Two-way pigeonhole bootstrap — строки получают произведение кратностей независимых блоков.
    a=pd.Series(groups_a).fillna('__UNKNOWN__').astype(str).to_numpy()
    b=pd.Series(groups_b).fillna('__UNKNOWN__').astype(str).to_numpy()
    unique_a=np.unique(a);unique_b=np.unique(b)
    if len(unique_a)<8 or len(unique_b)<8:return {'delta_MAE':np.nan,'ci_low':np.nan,'ci_high':np.nan,'p_value':np.nan,'clusters_a':len(unique_a),'clusters_b':len(unique_b)}
    v=np.asarray(model,dtype=float);u=np.asarray(base,dtype=float)
    d=v-u;delta=float(d.mean());rng=np.random.default_rng(seed)
    boot=[]
    for _ in range(reps):
        ca=Counter(rng.choice(unique_a,len(unique_a),replace=True))
        cb=Counter(rng.choice(unique_b,len(unique_b),replace=True))
        w=np.array([ca[x]*cb[y] for x,y in zip(a,b)],dtype=float)
        if w.sum():boot.append(float(np.average(d,weights=w)))
    if len(boot)<100:return {'delta_MAE':delta,'ci_low':np.nan,'ci_high':np.nan,'p_value':np.nan,'clusters_a':len(unique_a),'clusters_b':len(unique_b)}
    arr=np.asarray(boot)
    p=min(1.,2*min((np.sum(arr<=0)+1)/(len(arr)+1),(np.sum(arr>=0)+1)/(len(arr)+1)))
    return {'delta_MAE':delta,'ci_low':float(np.quantile(arr,.025)),
        'ci_high':float(np.quantile(arr,.975)),'p_value':float(p),
        'clusters_a':len(unique_a),'clusters_b':len(unique_b)}

def test_predictive(module,raw,nonnegative=True,count_confirmed=False):
    if raw.empty:return blocked(module,'Нет валидной целевой панели после семантического контроля',
       'entity_id, observed target years, known historical features')
    progress_event('MODELS',f'{module}: подготовка временного теста ({len(raw):,} строк)',announce=True)
    frame=raw.copy();frame['module']=module
    split,reason=temporal_partition(frame,min_train=CFG.REGION_MIN_TRAIN_ROWS if module=='ML-5.3' else None, min_test=CFG.REGION_MIN_TEST_ROWS if module=='ML-5.3' else None)
    if split is None:return blocked(module,reason,'train + independent test + as-of labels')
    train,dev,test=split['train'],split['dev'],split['test']
    cand=['persistence','rolling_mean_2','damped_trend','ridge','elasticnet','catboost','lightgbm']
    if count_confirmed:cand+=['ppml','negbin']
    if CFG.MODE=='SMOKE':cand=['persistence','rolling_mean_2','ridge']
    if CFG.MODE=='PILOT10' and len(train)>10000:
        train=train.sample(n=10000,random_state=CFG.RANDOM_SEED)
    primary='persistence';dev_scores={};dev_selection=False
    if dev is not None:
        # DEV labels действительно доступны к времени TEST-предсказания.
        for model in cand:
            try:
                if model in ('persistence','rolling_mean_2','damped_trend'):
                    p=baseline_prediction(dev,model,nonnegative)
                else:
                    p,_=pipeline_fit_predict(model,train,dev,seed=CFG.DEV_SEEDS[0],
                                             allow_ppml=count_confirmed,nonnegative=nonnegative)
                dev_scores[model]=effective_mae(dev.actual,p)['MAE']
            except Exception as ex:
                note('DEV',module+' '+model+': '+type(ex).__name__,'WARN')
        if dev_scores:
            priority={m:i for i,m in enumerate(cand)}
            primary=min(dev_scores,key=lambda m:(round(dev_scores[m],8),priority[m]))
            dev_selection=True
            # dev labels допустимо добавить в обучение только при as-of совпадении.
            train=pd.concat([train,dev],ignore_index=True)
    else:
        note('TEMPORAL',module+': DEV не опубликован к дате прогноза; модели предзаданы, без выбора по TEST','WARN')
    pred_by_model={}
    for model in cand:
        progress_event('MODELS',f'{module}: расчёт {model}',announce=True)
        seeds=[-1]
        if model=='catboost':
            if CFG.MODE=='SMOKE':seeds=[CFG.DEV_SEEDS[0]]
            elif CFG.MODE=='PILOT10' and dev_selection and primary=='catboost':seeds=list(CFG.DEV_SEEDS)
            elif CFG.MODE=='FULL':seeds=list(CFG.FINAL_SEEDS)[:CFG.ML54_FINAL_SEEDS] if module=='ML-5.4' else list(CFG.FINAL_SEEDS)  # 50 seeds for all FULL temporal tasks
            else:seeds=[CFG.FINAL_SEEDS[0]]
        preds=[]
        for i_seed,seed in enumerate(seeds,1):
            progress_event('MODELS',f'{module}: {model}, сид {i_seed}/{len(seeds)}',i_seed,len(seeds),announce=(i_seed==1 or i_seed%5==0 or i_seed==len(seeds)))
            try:
                p=fit_and_record(module,model,train,test,seed=seed,
                    nonnegative=nonnegative,allow_ppml=count_confirmed)
                preds.append(p)
            except Exception as ex:
                ISSUES.append({'module':module,'status':'MODEL_FAILED',
                     'reason':model+': '+type(ex).__name__+' — '+str(ex)[:180],
                     'required':'Inspect modeling frame; no fabricated substitute'})
                if CFG.MODE=='FULL' and model in ('negbin','ppml','ridge','elasticnet','lightgbm'):
                    note('MODEL_QA',module+' '+model+' failed, recorded as scientific FAIL','FAIL')
                if model==primary:raise
        if not preds:continue
        pred_by_model[model]=np.mean(preds,axis=0)
        if model=='catboost' and len(preds)>1:
            record_prediction(predictions_frame(test,module,'catboost_ensemble',pred_by_model[model],-1))
            METRICS.append({'module':module,'sector':str(test.sector.iloc[0]) if 'sector' in test else 'UNKNOWN','model':'catboost_ensemble','protocol':'ROLLING_TIME',
                 'seed':-1,'target_year':split['test_year'],
                 **effective_mae(test.actual,pred_by_model[model])})
    if 'persistence' not in pred_by_model:fail('MODEL','Нет сопоставимого исторического baseline')
    y=test.actual.to_numpy(float)
    base_err=np.abs(y-pred_by_model['persistence'])
    primary_err=np.abs(y-pred_by_model.get(primary,pred_by_model['persistence']))
    if module=='ML-5.1':
        ga=test.origin_region;gb=test.destination_region
    elif module=='ML-5.3':
        ga=test.origin_region;gb=test.target_year.astype(str)
    else:
        ga=test.parent_group_id.fillna(test.entity_id) if 'parent_group_id' in test else test.entity_id
        gb=test.target_year.astype(str)
    if dev_selection and primary!='persistence':
        ci=two_way_mae_bootstrap(base_err,primary_err,ga,gb,
               reps=99 if CFG.MODE=='SMOKE' else CFG.BOOTSTRAP_REPS,seed=CFG.RANDOM_SEED)
    else:
        ci={'delta_MAE':np.nan,'ci_low':np.nan,'ci_high':np.nan,'p_value':np.nan,
            'clusters_a':len(pd.Series(ga).dropna().unique()),
            'clusters_b':len(pd.Series(gb).dropna().unique())}
    chosen=effective_mae(y,pred_by_model.get(primary,pred_by_model['persistence']))
    original=effective_mae(y,pred_by_model['persistence'])
    CONFIRMATORY.append({'module':module,'sector':str(test.sector.iloc[0]) if 'sector' in test else 'UNKNOWN','test_year':split['test_year'],'test_n':len(test),
        'choice_basis':'DEV_ASOF' if dev_selection else 'NO_CONFIRMATORY_COMPARISON_NO_DEV',
        'selected_model':primary,'baseline_mae':original['MAE'],'selected_mae':chosen['MAE'],
        'delta_mae':chosen['MAE']-original['MAE'],**ci,'source_vintage_warning':True})
    progress_event('MODELS',f'{module}: тест завершён, инерция MAE={original["MAE"]:.3g}, выбранная={chosen["MAE"]:.3g}',announce=True)
    MODULES[module].update(status='TESTED_FULL' if CFG.MODE=='FULL' else 'TESTED_PILOT',
        reason=('Ретроспективный тест с неподтверждёнными историческими версиями источника; '
                'DEV-проверка доступности соблюдена; улучшение не гарантировано'),
        n=len(test),test_year=split['test_year'],selected=primary)
    MODULE_DIAGNOSTICS.append({'module':module,'prediction_feasible':True,
        'n_observed':len(raw),'n_test':len(test),'test_year':split['test_year'],
        'dev_available_asof':dev_selection,'selected_model':primary,
        'mae_baseline':original['MAE'],'mae_selected':chosen['MAE']})
    if not dev_selection:ISSUES.append({'module':module,'status':'SEVERE_TEMPORAL_LIMIT',
       'reason':'Нет DEV-меток, опубликованных до cutoff TEST; сравнения альтернатив только разведочные',
       'required':'older archived vintages or longer observations'})
    return split

# ML-5.1: истинная единица E01 пока не подтверждена как приём; регрессируем именно E01_observed_value.
if len(OD_INTER):
    MODULE_DIAGNOSTICS.append({'module':'ML-5.1','prediction_feasible':None,
        'n_observed':len(OD_INTER),'target_definition':OD_SEMANTIC['label'],
        'unknown_codes':len(OD_UNKNOWN),'diagonal_excluded':len(OD_INTRA),
        'coverage_breaks':int(coverage_df.potential_break.fillna(False).sum())})
    test_predictive('ML-5.1',OD_MODEL,nonnegative=True,count_confirmed=bool(OD_MODEL.actual.ge(0).all() and np.isclose(OD_MODEL.actual,np.rint(OD_MODEL.actual),atol=1e-8,rtol=0).all()))
    if coverage_df.potential_break.fillna(False).any():
        ISSUES.append({'module':'ML-5.1','status':'UNRESOLVED_YEAR_COVERAGE_BREAK',
          'reason':'Смена числа OD-пар в 2023 г. не доказана как изменение реальной мобильности; модельные результаты — условно разведочные',
          'required':'semantics/coverage harmonization across 2022–2023'})
else:blocked('ML-5.1','Нет истинно наблюдаемых межрегиональных E01-рёбер')

# ML-5.2: отдельно ВО/СПО; не моделировать учреждения, если не удалось установить уникальную величину.
if len(ORG_YEAR):
    for sector,group in ORG_MODEL.groupby('sector') if len(ORG_MODEL) else []:
        if len(group):
            result={'status':'NOT_STARTED','reason':'source not evaluated','n':0}
            try:
                test_predictive('ML-5.2',group,nonnegative=True,count_confirmed=True)
                result=MODULES['ML-5.2'].copy()
            except Exception as ex:
                result={'status':'FAILED','reason':f'{sector}: {type(ex).__name__}: {str(ex)[:100]}','n':0}
                raise
            finally:
                MODULE_SUBTASKS.append({'module':'ML-5.2','sector':sector,**result})
            MODULE_DIAGNOSTICS.append({'module':'ML-5.2','sector':sector,'n_organizations':group.entity_id.nunique(),
                    'n_observed':len(group)})
    if MODULES['ML-5.2']['status']=='NOT_STARTED':
        limitation('ML-5.2','Установлены потенциальные индикаторы, но недостаточно исторических пар для модели',
                   'admissions×organization×year with unambiguous units')
else:
    limitation('ML-5.2','Предметный каталог E02/VPO/SPO прочитан, но однозначное зачисление организация×год не подтверждено',
               'source-admission indicator and organization with unique key')

if MODULE_SUBTASKS:
    qa_table('organization_sector_subtasks',pd.DataFrame(MODULE_SUBTASKS),allow_empty=True)
    statuses=[r['status'] for r in MODULE_SUBTASKS]
    if any(x.startswith('TESTED') for x in statuses):
        MODULES['ML-5.2']['status']='TESTED_FULL' if CFG.MODE=='FULL' else 'TESTED_PILOT'
        MODULES['ML-5.2']['reason']='Отдельные сектора: '+', '.join(f"{r['sector']}={r['status']}" for r in MODULE_SUBTASKS)

# ML-5.3: только межрегиональные маршруты из региона получения общего образования.
if len(OUTFLOW_MODEL):test_predictive('ML-5.3',OUTFLOW_MODEL,nonnegative=True,count_confirmed=bool(OUTFLOW_MODEL.actual.ge(0).all() and np.isclose(OUTFLOW_MODEL.actual,np.rint(OUTFLOW_MODEL.actual),atol=1e-8,rtol=0).all()))
else:blocked('ML-5.3','Недостаточно наблюдений агрегированного межрегионального E01',
             'origin schooling region, destination study region, comparable years')

# ML-5.4: реальная программа, не фиктивное происхождение по специальности.
if len(PROGRAM_PROFILE):
    pp=PROGRAM_PROFILE.copy()
    pp['denominator']=pp.groupby(['observation_year','indicator_name']).observed_value.transform('sum')
    pp['relative_share_qualifier']='CONDITIONAL_ON_EXACT_INDICATOR_AND_OBSERVED_PROGRAM_RECORDS'
    pp['share_observed']=np.where(pp.denominator>0,pp.observed_value/pp.denominator,np.nan)
    pp['herfindahl_component']=pp.share_observed**2
    concentration=pp.groupby(['observation_year','indicator_name'],as_index=False).agg(
        reported_programs=('specialty_code','nunique'),reported_value=('observed_value','sum'),
        HHI=('herfindahl_component','sum'))
    qa_table('program_concentration',concentration,allow_empty=True)
    qa_table('program_distribution',pp,allow_empty=True)
    MODULES['ML-5.4'].update(status='DESCRIPTIVE_VALID',reason='Программы — условный срез конкретных индикаторов и кодов; OD×programme отсутствует',n=len(pp))
    MODULE_DIAGNOSTICS.append({'module':'ML-5.4','prediction_feasible':False,
       'n_observed':len(pp),'reason':'Нельзя прогнозировать выбор программы по OD без OD×programme'})
elif len(VPO_PROGRAM_DESCRIPTION):
    MODULES['ML-5.4'].update(status='DESCRIPTIVE_VALID',
        reason='Реальные направления ВПО-1 по категориям и статистическим уровням, без двойного суммирования и ненаблюдаемого OD×programme',
        n=len(VPO_PROGRAM_DESCRIPTION))
    MODULE_DIAGNOSTICS.append({'module':'ML-5.4','prediction_feasible':False,
      'n_observed':len(VPO_PROGRAM_DESCRIPTION),'reason':'Представлены медианы исходных разрезов; точные когорты программы не установлены'})
else:limitation('ML-5.4','Доступные словари программ не дали согласованного временного показателя',
                'stable specialty codes, verified target, level and year')

# ML-5.5: полная E04/E05 обрабатывается потоково в DuckDB; не сравнивать preview со всей базой.
if CFG.MODE=='FULL':
    if GRAD_E04_VALID_ROWS>0 and not GRADUATE_GROUP_SUMMARY.empty:
        MODULES['ML-5.5'].update(status='DESCRIPTIVE_VALID',
            reason='Полные E04/E05: статистические группировки всех наблюдений в DuckDB; employment_asof 2024-12-31; RETENTION_NOT_OBSERVABLE',
            n=GRAD_E04_VALID_ROWS)
        MODULE_DIAGNOSTICS.append({'module':'ML-5.5','prediction_feasible':False,
            'n_observed_e04':GRAD_E04_VALID_ROWS,'n_observed_e05':GRAD_E05_VALID_ROWS,
            'employment_asof':'2024-12-31','reason':'Срез на одной референсной дате; место работы не подтверждено'})
    else:
        limitation('ML-5.5','Агрегат E04 по всем строкам пуст после валидации показателя')
elif len(GRADUATES):
    gr=GRADUATES.dropna(subset=['count_graduate']).copy()
    gr=gr[gr.count_graduate.ge(0)]
    gr_summary=gr.groupby(['graduation_year','object_level','gender','education_level'],dropna=False).agg(
       n_source_records=('university','size'),median_reported_graduates=('count_graduate','median'),
       median_observed_salary=('average_salary_fact_med','median'),
       median_employment_percentage=('percent_employed','median')).reset_index()
    gr_summary['aggregate_totals_identified']=False
    qa_table('graduate_descriptive_statistics',gr_summary,allow_empty=True)
    if 'graduate_specialty_outcomes' in PANELS:
        f=PANELS['graduate_specialty_outcomes'].dropna(subset=['graduation_year','specialty_code'])
        f=f[f.count_graduate.notna() & f.count_graduate.ge(0)]
        source_groups=f.groupby(['graduation_year','object_level','education_level','specialty_code'],dropna=False).agg(
          observed_records=('specialty','size'),median_reported_graduates=('count_graduate','median'),
          median_wage=('average_salary_fact_med','median')).reset_index()
        source_groups['aggregate_totals_identified']=False
        qa_table('graduate_specialty_description',source_groups,allow_empty=True)
    MODULES['ML-5.5'].update(status='DESCRIPTIVE_VALID',
      reason='SMOKE/PILOT: описательная статистика по ограниченным данным; retention неизвестен',n=len(gr))
    MODULE_DIAGNOSTICS.append({'module':'ML-5.5','prediction_feasible':False,'n_observed':len(gr),
      'employment_asof':'2024-12-31','reason':'Одна дата занятости; нет географии работы'})
else:limitation('ML-5.5','E04/E05: нет валидной описательной статистики')

# ML-5.6: динамическая сеть, петли исключены; сетевые изменения с учетом неодинакового покрытия.
NETWORK=[];NETWORK_EDGES=[];COMMUNITIES=[];NET_SETS={}
for year,sub in OD_INTER.dropna(subset=['observed_value']).groupby('observation_year'):
    progress_event('NETWORK',f'Год {year}: расчёт наблюдаемой сети',announce=True)
    g=nx.DiGraph()
    for r in sub.itertuples(index=False):
        if pd.notna(r.observed_value) and r.observed_value>0:
            g.add_edge(str(r.origin_region),str(r.destination_region),weight=float(r.observed_value))
    if nx.number_of_selfloops(g):fail('NETWORK','Обнаружена диагональ в межрегиональном графе')
    if not g.number_of_edges():continue
    pager=nx.pagerank(g,weight='weight')
    undirected=g.to_undirected()
    try:comms=nx.community.louvain_communities(undirected,weight='weight',seed=CFG.RANDOM_SEED)
    except Exception:comms=list(nx.community.greedy_modularity_communities(undirected,weight='weight'))
    membership={node:i for i,community in enumerate(comms) for node in community}
    NET_SETS[int(year)]=set((a,b) for a,b in g.edges())
    for node in g.nodes():
        incoming=g.in_degree(node,weight='weight');outgoing=g.out_degree(node,weight='weight')
        vals=np.asarray([float(d['weight']) for _,_,d in g.out_edges(node,data=True)],float)
        sh=vals/vals.sum() if vals.sum()>0 else np.asarray([],float)
        out_hhi=float((sh*sh).sum()) if len(sh) else np.nan
        out_entropy=float(-(sh*np.log(sh)).sum()) if len(sh) else np.nan
        NETWORK.append({'year':int(year),'region_code':node,'in_strength':incoming,'out_strength':outgoing,
          'outbound_HHI_observed':out_hhi,'outbound_entropy_observed':out_entropy,
          'pagerank':pager.get(node,np.nan),'community_id':membership.get(node,-1),
          'n_in':g.in_degree(node),'n_out':g.out_degree(node),'observed_edges':g.number_of_edges()})
    for a,b,d in g.edges(data=True):NETWORK_EDGES.append({'year':int(year),'origin':a,'destination':b,
        'value':d['weight']})
    COMMUNITIES.append({'year':int(year),'communities':len(comms),
        'n_regions':len(g.nodes()),'n_edges':g.number_of_edges(),
        'coverage_break_warning':bool(coverage_df.loc[coverage_df.year==int(year),'potential_break'].fillna(False).any())})
qa_table('education_network_year',pd.DataFrame(NETWORK),allow_empty=True)
qa_table('education_network_edges',pd.DataFrame(NETWORK_EDGES),allow_empty=True)
qa_table('education_network_communities',pd.DataFrame(COMMUNITIES),allow_empty=True)
if len(NET_SETS)>=2:
    stability=[]
    for a,b in zip(sorted(NET_SETS)[:-1],sorted(NET_SETS)[1:]):
        x,y=NET_SETS[a],NET_SETS[b]
        stability.append({'year_from':a,'year_to':b,'edge_jaccard':len(x&y)/len(x|y) if x|y else np.nan,
                          'n_edges_from':len(x),'n_edges_to':len(y),
                          'coverage_break_warning':bool(coverage_df.loc[coverage_df.year==b,'potential_break'].fillna(False).any())})
    qa_table('network_stability',pd.DataFrame(stability),allow_empty=True)
if len(NETWORK):
    MODULES['ML-5.6'].update(status='DESCRIPTIVE_VALID',reason='Сетевые показатели наблюдаемой E01 без самопетель; покрытие лет различается',n=len(NETWORK))
else:limitation('ML-5.6','Нет валидных взвешенных межрегиональных ребер E01')

# ML-5.7: одинаковые муниципалитеты для E0/E1; пропущенный факт нахождения кампуса нельзя считать нулём.
context=MUNI_EDU[MUNI_EDU.education_coverage_known].copy()
if (context.oktmo.nunique()>=CFG.MIN_MUNICIPALITIES_E0E1 and
        context.observation_year.nunique()>=CFG.MIN_YEARS_FOR_TEMPORAL and
        context.confirmed_vo_sites.nunique()>1):
    x=context[['oktmo','observation_year','migration_rate','population','confirmed_vo_sites']].rename(
      columns={'oktmo':'entity_id','migration_rate':'actual','population':'prev_population'})
    # Не подтягивать число университетов из целевого года: lag, as-of.
    m0=historical_pair_panel(x[['entity_id','observation_year','actual']],['entity_id'],'actual')
    lag=context.rename(columns={'oktmo':'entity_id','observation_year':'historical_year',
                                'confirmed_vo_sites':'confirmed_vo_sites',
                                'population':'prev_population'})
    m0=m0.merge(lag[['entity_id','historical_year','confirmed_vo_sites','prev_population']],
        on=['entity_id','historical_year'],validate='many_to_one',how='left')
    m0['target_id']='municipal_migration_rate';m0['sector']='MUNICIPAL_CONTEXT'
    qa_table('municipal_e0e1_model_frame',m0,allow_empty=True)
    # Прогнозная процедура E0/E1 — одна и та же модель и поддержка; отключаем инфра в E0.
    split,reason=temporal_partition(m0)
    if split:
        tr=split['train'];te=split['test'];features0=['prior_value','prev_population','origin_year']
        features1=features0+['confirmed_vo_sites']
        for version,features in [('E0',features0),('E1',features1)]:
            model=Pipeline([('imputer',SimpleImputer(strategy='median')),
                            ('scaler',StandardScaler()),('reg',Ridge(alpha=20.))])
            model.fit(tr[features],tr.actual)
            predictions=model.predict(te[features])
            record_prediction(predictions_frame(te,'ML-5.7',version,predictions))
            METRICS.append({'module':'ML-5.7','sector':'MUNICIPAL_CONTEXT','model':version,'protocol':'COMMON_SUPPORT_MUNICIPAL',
              'seed':-1,'target_year':split['test_year'],**effective_mae(te.actual,predictions)})
            fp=ROOT/'models'/f'ML_5_7_{version}.joblib';joblib.dump({'model':model,'features':features},fp)
            reload=joblib.load(fp)
            assert np.allclose(reload['model'].predict(te[features]),predictions)
            MODEL_FILES[fp.name]=str(fp)
        MODULES['ML-5.7'].update(status='TESTED_FULL' if CFG.MODE=='FULL' else 'TESTED_PILOT',
            reason='E0/E1 на одинаковом подтверждённом составе ОКТМО',n=len(te))
    else:limitation('ML-5.7',reason,'more confirmed municipality-years')
else:
    limitation('ML-5.7',f'Недостаточно подтвержденного кампусного покрытия: {context.oktmo.nunique()} ОКТМО; неизвестно != 0',
               'verified municipality campus registry with historical validity')

# Все семь модулей получают явный предметный статус; ни один не теряется.
for key,result in MODULES.items():
    if result['status']=='NOT_STARTED':
        limitation(key,'Данные просканированы, но применимый протокол не смог сформировать результат')
qa_table('seven_module_status',pd.DataFrame([{'module':k,**v} for k,v in MODULES.items()]),['module'])
qa_table('module_diagnostics',pd.DataFrame(MODULE_DIAGNOSTICS),allow_empty=True)
print('MODULES:',{k:v['status'] for k,v in MODULES.items()})

memory_point('after_seven_modules')
mark_stage('SEVEN_MODULES')


## 9А. Дополнительные исследования ML-5.2–ML-5.7

Строго отделяются проверенный приём и неутверждённые кандидаты, описательная структура программ и индивидуальный выбор, занятость выпускников и территориальное закрепление. Региональные и муниципальные экспериментальные протоколы запускаются только на проверенной общей области наблюдения. Сетевые показатели проверяются на устойчивость. Отрицательные результаты фиксируются без замены тестовой модели задним числом.

In [ ]:
# ML-05: substantive completion of all seven modules with explicit evidence gates.
progress_event('EXTENDED_MODULES','Дополнительные исследования ML-5.2–ML-5.7 и сетевой стресс-тест',announce=True)
EXT_MODULES=[]

# ML-5.2: distinguish candidates, verified admission, and non-additive section splits.
if 'CANDIDATE_CATALOG' in globals() and len(CANDIDATE_CATALOG):
    cand=CANDIDATE_CATALOG.loc[CANDIDATE_CATALOG.semantic_role.eq('ADMISSION_CANDIDATE')].copy()
    sector_rows=cand.groupby('source',dropna=False).agg(indicator_names=('indicator_name','nunique'),
                         rows_in_source=('rows','sum')).reset_index() if len(cand) else pd.DataFrame()
    if len(sector_rows):
        sector_rows['approved_entity_year_targets']=len(ORG_YEAR)
        sector_rows['interpretation']='SOURCE_CANDIDATES_ONLY_NOT_VERIFIED_ADMISSIONS'
    qa_table('ml52_admission_candidates',sector_rows,allow_empty=True)
    if len(sector_rows) and not len(ORG_YEAR):
        MODULES['ML-5.2'].update(status='DESCRIPTIVE_ONLY_TARGET_UNAPPROVED',n=int(sector_rows.rows_in_source.sum()),
          reason='Семантические кандидаты действительно найдены, но утверждённой численности приёма организация×год нет')
        EXT_MODULES.append({'module':'ML-5.2','evidence':'SEMANTIC_CANDIDATES_NOT_COUNTS',
          'n_candidates':int(sector_rows.indicator_names.sum()),'prediction_status':'NOT_FEASIBLE'})

# ML-5.4: whole-dataset specialty histories, not a preview of the first 2,000 aggregate rows.
ML54_HISTORY=pd.DataFrame()
if VPO_FULL_PROGRAM_PATH and Path(VPO_FULL_PROGRAM_PATH).exists():
    p=Path(VPO_FULL_PROGRAM_PATH)
    q='''SELECT observation_year,CAST(specialty_code AS VARCHAR) specialty_code,
      CAST(indicator_name AS VARCHAR) indicator_name,
      SUM(observed_records)::BIGINT rows_in_source,
      COUNT(*)::BIGINT independently_reported_strata,
      MEDIAN(median_reported_value) median_of_source_stratum_medians
      FROM read_parquet(?)
      WHERE specialty_code IS NOT NULL AND TRY_CAST(observation_year AS INT) IS NOT NULL
      GROUP BY 1,2,3'''
    with duck_connect() as con:
        n=int(con.execute('SELECT COUNT(*) FROM ('+q+')',[str(p)]).fetchone()[0])
        if n<=CFG.MAX_PANDAS_ROWS:
            ML54_HISTORY=con.execute(q,[str(p)]).df()
        else:
            out=ROOT/'derived'/'ml54_full_history.parquet'
            con.execute('COPY ('+q.replace('read_parquet(?)','read_parquet('+_sql_literal(p)+')')+
              ') TO '+_sql_literal(out)+' (FORMAT PARQUET,COMPRESSION ZSTD)')
            PANEL_FILES['ml54_specialty_history']=out
            ML54_HISTORY=con.execute('SELECT * FROM read_parquet(?) LIMIT 2000',[str(out)]).df()
    qa_table('ml54_specialty_history',ML54_HISTORY,allow_empty=True)
    if n:
        MODULES['ML-5.4'].update(status='DESCRIPTIVE_FULL_SOURCE_NOT_OD_PROGRAMME',n=n,
          reason='Всё содержимое VPO-1 сгруппировано по годам/кодам/показателям; составное медианное значение не равно численности зачисленных')
    EXT_MODULES.append({'module':'ML-5.4','full_specialty_year_indicator_groups':n,
          'forecast_valid':'NO_VERIFIED_ADMISSIONS_TARGET','unit':'source_records_and_medians_not_additive'})

# Exploratory specialty continuity forecast for stable time series (not a verified admission forecast).
# Predictions are directly row counts of source records in a published statistical strata grid.
ML54_TEST=pd.DataFrame()
if len(ML54_HISTORY) and 'rows_in_source' in ML54_HISTORY and 'observation_year' in ML54_HISTORY:
    z=ML54_HISTORY.copy();z=z.dropna(subset=['observation_year','specialty_code','indicator_name'])
    z['observation_year']=pd.to_numeric(z.observation_year,errors='coerce')
    z=z.sort_values(['specialty_code','indicator_name','observation_year'])
    g=z.groupby(['specialty_code','indicator_name'],sort=False)
    z['lag1_rows']=g.rows_in_source.shift(1)
    z['lag1_year']=g.observation_year.shift(1)
    last=int(z.observation_year.max()) if len(z) else 0
    ML54_TEST=z[(z.observation_year.eq(last))&(z.lag1_year.eq(last-1))&z.lag1_rows.notna()].copy()
    if len(ML54_TEST):
        ML54_TEST['abs_error_persistence']=(ML54_TEST.rows_in_source-ML54_TEST.lag1_rows).abs()
        ML54_TEST['measure_definition']='VPO_REPORTING_STRATA_COVERAGE_ONLY'
        qa_table('ml54_history_persistence_diagnostic',ML54_TEST,allow_empty=True)
        EXT_MODULES.append({'module':'ML-5.4','coverage_persistence_year':last,
          'test_cells':len(ML54_TEST),'MAE_source_row_coverage':float(ML54_TEST.abs_error_persistence.mean()),
          'scientific_gate':'NOT_ADMISSIONS_OR_STUDENT_FLOW'})

# ML-5.5: all 4 E04/E05 sources, male/female strata kept separate, no artificial employment location.
GRAD_DISPARITIES=pd.DataFrame()
family_frames=[]
for i in range(1,5):
    name='graduate_family_'+str(i)
    path=PANEL_FILES.get(name)
    if path and Path(path).exists():
        with duck_connect() as con:
            f=con.execute('SELECT * FROM read_parquet(?) LIMIT 30000',[str(path)]).df()
        f['asset_id']=source_asset(segment='05')[i-1]['asset_id']
        family_frames.append(f)
if family_frames:
    GRAD_DISPARITIES=pd.concat(family_frames,ignore_index=True,sort=False)
    GRAD_DISPARITIES['employment_reference']='2024-12-31'
    GRAD_DISPARITIES['outcome_grain']='MEDIAN_OF_PUBLISHED_AGGREGATE_RECORDS'
    qa_table('ml55_outcome_disparities',GRAD_DISPARITIES,allow_empty=True)
    EXT_MODULES.append({'module':'ML-5.5','source_families_analysed':len(family_frames),
        'output_strata':len(GRAD_DISPARITIES),'time_reference':'2024-12-31',
        'retention':'NOT_OBSERVABLE_WITHOUT_WORK_LOCATION'})
    MODULES['ML-5.5'].update(status='DESCRIPTIVE_ALL_FOUR_FAMILIES',
       reason='E04/E05 всех четырёх типов изучены по исходным разрезам, без двойного суммирования; закрепление неизвестно',n=len(GRAD_DISPARITIES))

# ML-5.6: reciprocity, removal robustness, SCC/WCC and stable network coverage.
ML56_STRESS=[]
for yr,sub in pd.DataFrame(NETWORK_EDGES).groupby('year') if len(NETWORK_EDGES) else []:
    g=nx.DiGraph()
    for r in sub.itertuples(index=False):
        if r.value>0 and r.origin!=r.destination:g.add_edge(str(r.origin),str(r.destination),weight=float(r.value))
    if not g.number_of_nodes():continue
    pg=nx.pagerank(g,weight='weight');leading=max(pg,key=pg.get)
    reciprocal=sum(1 for u,v in g.edges() if g.has_edge(v,u))
    w0=len(max(nx.weakly_connected_components(g),key=len))
    s0=len(max(nx.strongly_connected_components(g),key=len))
    h=g.copy();h.remove_node(leading)
    w1=len(max(nx.weakly_connected_components(h),key=len)) if h.number_of_nodes() else 0
    edges_before=g.number_of_edges();edges_after=h.number_of_edges()
    ML56_STRESS.append({'year':int(yr),'n_nodes':g.number_of_nodes(),'n_edges':edges_before,
       'reciprocity_directed_edge_share':reciprocal/edges_before if edges_before else np.nan,
       'largest_weak_component':w0,'largest_strong_component':s0,'removed_top_pagerank_region':leading,
       'weak_component_after_removal':w1,'edges_after_removal':edges_after,
       'lost_edges_percent':100*(edges_before-edges_after)/edges_before if edges_before else np.nan,
       'interpretation':'STRUCTURAL_STRESS_NOT_EMPIRICAL_MIGRATION_DISRUPTION'})
qa_table('ml56_network_stress',pd.DataFrame(ML56_STRESS),allow_empty=True)
if len(ML56_STRESS):
    MODULES['ML-5.6'].update(status='DESCRIPTIVE_FULL_NETWORK_STRESS',n=len(NETWORK),
      reason='Динамические сети, PageRank, Louvain, концентрация, reciprocity и устойчивость удаления центрального узла; сопоставимость лет ограничена')
    EXT_MODULES.append({'module':'ML-5.6','n_years':len(ML56_STRESS),'validation':'OBSERVED_EDGES_ONLY'})

# ML-5.7: BDMO education-infrastructure proxy E0/E1, same OKTMO-year rows, no fabricated university locations.
ML57_SUPPORT=pd.DataFrame();ML57_RESULT={}
if BDMO_PROXY_FULL_VALID and len(BDMO_EDU_PROXY_CANDIDATES) and len(MUNI):
    e=BDMO_EDU_PROXY_CANDIDATES.dropna(subset=['oktmo','observation_year','indicator_code','median_reported']).copy()
    e['oktmo']=norm_code(e.oktmo);e['observation_year']=safe_year(e.observation_year)
    e=e[e.n_raw_rows.eq(1)&e.median_reported.ge(0)].copy()
    if len(e):
        mig=MUNI[['oktmo','observation_year','migration_rate','population']].drop_duplicates(['oktmo','observation_year']).copy()
        mig['observation_year']=safe_year(mig.observation_year)
        # only same historically observed municipality, lookback=one year at origin.
        mig=mig.sort_values(['oktmo','observation_year'])
        mig['prev_year']=mig.observation_year-1
        past=mig[['oktmo','observation_year','migration_rate','population']].rename(
             columns={'observation_year':'prev_year','migration_rate':'lag_migration_rate','population':'lag_population'})
        mig=mig.merge(past,on=['oktmo','prev_year'],how='inner',validate='many_to_one')
        for c in ('migration_rate','lag_migration_rate','lag_population'):mig[c]=number(mig[c])
        last=int(mig.observation_year.max()) if len(mig) else None
        # Candidate selection entirely on PRE-TEST historical years; never on test labels.
        prior=e[e.observation_year.lt(last)] if last is not None else e.iloc[:0]
        stats=prior.groupby('indicator_code',dropna=False).agg(n_muni=('oktmo','nunique'),
           n_years=('observation_year','nunique'),n_rows=('median_reported','size'),
           n_distinct=('median_reported','nunique')).reset_index()
        stats=stats[(stats.n_muni>=CFG.MIN_MUNICIPALITIES_E0E1)&
                    (stats.n_years>=CFG.MIN_YEARS_FOR_TEMPORAL)&(stats.n_distinct>1)]
        if len(stats):
            stats=stats.sort_values(['n_muni','n_years','n_rows','indicator_code'],ascending=[False,False,False,True])
            selected=str(stats.iloc[0].indicator_code)
            x=e[e.indicator_code.astype(str).eq(selected)][['oktmo','observation_year','median_reported']].copy()
            x=x.rename(columns={'observation_year':'prev_year','median_reported':'lag_education_proxy'})
            # municipality's published education indicator is used at t-1 only, not t.
            ML57_SUPPORT=mig.merge(x,on=['oktmo','prev_year'],how='inner',validate='many_to_one')
            ML57_SUPPORT=ML57_SUPPORT.dropna(subset=['migration_rate','lag_migration_rate',
                               'lag_population','lag_education_proxy'])
            ML57_SUPPORT['indicator_code']=selected
            ML57_SUPPORT['source_definition']='BDMO_PUBLISHED_EDUCATION_PROXY_NOT_VERIFIED_CAMPUS'
            qa_table('ml57_common_support',ML57_SUPPORT,allow_empty=True)
            tr=ML57_SUPPORT[ML57_SUPPORT.observation_year.lt(last)].copy()
            te=ML57_SUPPORT[ML57_SUPPORT.observation_year.eq(last)].copy()
            if (len(tr)>=max(50,CFG.REGION_MIN_TRAIN_ROWS) and
                len(te)>=CFG.MIN_MUNICIPALITIES_E0E1 and tr.oktmo.nunique()>=CFG.MIN_MUNICIPALITIES_E0E1):
                # Identical E0/E1 keys and model, change ONLY historic education covariate.
                estimates={};baselines=te.lag_migration_rate.to_numpy(float)
                baseline_score=effective_mae(te.migration_rate,baselines)
                for vers,features in [('E0',['lag_migration_rate','lag_population']),
                                      ('E1',['lag_migration_rate','lag_population','lag_education_proxy'])]:
                    model=Pipeline([('imputer',SimpleImputer(strategy='median')),
                                   ('scale',StandardScaler()),('ridge',Ridge(alpha=20.0))])
                    model.fit(tr[features].astype(float),tr.migration_rate.to_numpy(float))
                    pred=model.predict(te[features].astype(float));estimates[vers]=effective_mae(te.migration_rate,pred)
                    fp=ROOT/'models'/('ML_5_7_proxy_'+vers+'.joblib')
                    artifact={'pipeline':model,'features':features,'proxy_indicator_code':selected}
                    joblib.dump(artifact,fp,compress=3);rp=joblib.load(fp)['pipeline'].predict(te[features].astype(float))
                    if not np.allclose(rp,pred):fail('ML-5.7','Model serialization discrepancy')
                    MODEL_FILES[fp.name]=str(fp)
                    MODEL_RECORDS.append({'module':'ML-5.7','sector':'MUNICIPAL_BDMO_PROXY','model':vers,
                      'seed':-1,'path':fp.name,'reloaded':True,'training_rows':len(tr),'test_rows':len(te),
                      'feature_num':features,'feature_cat':[],'file_sha256':hash_file(fp)})
                    out=te[['oktmo','observation_year','migration_rate','prev_year']].copy()
                    out=out.rename(columns={'oktmo':'entity_id','observation_year':'target_year',
                         'migration_rate':'actual','prev_year':'origin_year'})
                    out['information_cutoff_year']=out.origin_year
                    out['feature_available_year']=out.origin_year
                    out['target_available_year']=out.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS
                    out['target_id']='D01_MUNICIPAL_MIGRATION_RATE'
                    out['sector']='MUNICIPAL_BDMO_PROXY'
                    pf=predictions_frame(out,'ML-5.7',vers,pred,seed=-1,protocol='RETROSPECTIVE_COMMON_SUPPORT')
                    pf['geography_grain']='MUNICIPALITY';record_prediction(pf)
                    METRICS.append({'module':'ML-5.7','sector':'MUNICIPAL_BDMO_PROXY','model':vers,
                       'protocol':'RETROSPECTIVE_COMMON_SUPPORT','seed':-1,'target_year':last,
                       'source_vintage':INDEX['created_at'],**estimates[vers]})
                ML57_RESULT={'status':'RETROSPECTIVE_PROXY_E0E1','year':last,'n_train':len(tr),
                  'n_test':len(te),'n_muni_test':te.oktmo.nunique(),'indicator_code':selected,
                  'inertia_MAE':baseline_score['MAE'],'e0_MAE':estimates['E0']['MAE'],
                  'e1_MAE':estimates['E1']['MAE'],'delta_MAE_E1_minus_E0':estimates['E1']['MAE']-estimates['E0']['MAE']}
                MODULES['ML-5.7'].update(status='RETROSPECTIVE_MUNICIPAL_PROXY_TEST',n=len(te),
                    reason='E0/E1 на общих ОКТМО с опубликованным BDMO-показателем: не доказана фактическая образовательная площадка; причинные выводы запрещены')
if not ML57_RESULT:
    ML57_RESULT={'status':'NOT_FEASIBLE_WITH_VERIFIED_GEOGRAPHY',
      'reason':'Недостаточная муниципальная область наблюдения или отсутствующий сопоставимый показатель инфраструктуры'}
EXT_MODULES.append({'module':'ML-5.7',**ML57_RESULT})
# ML-5.1 generalization checks: both time and territory blocked, no test-based hyperparameter selection.
SPATIAL_MODEL_RESULTS=[]
if len(OD_MODEL):
    frame=OD_MODEL.copy()
    frame['module']='ML-5.1'
    years=sorted(int(y) for y in frame.target_year.dropna().unique())
    if len(years)>=2:
        ty=years[-1];origin=ty-CFG.HORIZON
        origins=sorted(frame.origin_region.dropna().astype(str).unique())
        destinations=sorted(frame.destination_region.dropna().astype(str).unique())
        def stable_holdout(code,salt):
            return int(hashlib.sha256((salt+'|'+str(code)).encode()).hexdigest()[:8],16)%5==0
        oo={r for r in origins if stable_holdout(r,'ML5_ORIGIN_HOLDOUT')}
        dd={r for r in destinations if stable_holdout(r,'ML5_DEST_HOLDOUT')}
        protocols=[('TERRITORY_ONLY',frame.origin_region.astype(str).isin(oo),
                    ~frame.origin_region.astype(str).isin(oo)),
                   ('CROSSED',frame.origin_region.astype(str).isin(oo)&frame.destination_region.astype(str).isin(dd),
                    ~frame.origin_region.astype(str).isin(oo)&~frame.destination_region.astype(str).isin(dd))]
        for protocol,is_test,is_train in protocols:
            tr=frame[frame.target_year.lt(ty)&frame.target_available_year.le(origin)&is_train].copy()
            te=frame[frame.target_year.eq(ty)&is_test].copy()
            meta={'module':'ML-5.1','protocol':protocol,'train_rows':len(tr),'test_rows':len(te),
                  'n_origins':te.origin_region.nunique(),'n_destinations':te.destination_region.nunique(),
                  'date_origin':origin,'test_year':ty,'selection_basis':'PREDECLARED_GEOGRAPHIC_HASH'}
            if len(tr)<CFG.MIN_TRAIN_ROWS or len(te)<CFG.MIN_TEST_ROWS:
                meta['status']='NOT_FEASIBLE_SUPPORT';SPATIAL_MODEL_RESULTS.append(meta);continue
            # Persist the baseline on precisely the same test keys as each spatial model.
            pbase=fit_and_record('ML-5.1','persistence',tr,te,seed=-1,
                                 nonnegative=True,protocol=protocol)
            reference=effective_mae(te.actual,pbase)
            meta['baseline_mae']=reference['MAE'];meta['status']='TESTED_RETROSPECTIVE_EXPLORATORY'
            # Single fixed seed; no TEST choice; same held-out keys for all models.
            for model in ('ridge','elasticnet','lightgbm'):
                try:
                    yp=fit_and_record('ML-5.1',model,tr,te,seed=CFG.RANDOM_SEED,
                                     nonnegative=True,allow_ppml=False,protocol=protocol)
                    mt=effective_mae(te.actual,yp)
                    meta[model+'_MAE']=mt['MAE'];meta[model+'_minus_baseline']=mt['MAE']-reference['MAE']
                except (ValueError,RuntimeError) as ex:
                    ISSUES.append({'module':'ML-5.1','status':'SPATIAL_MODEL_FAILURE',
                        'reason':protocol+' '+model+': '+type(ex).__name__,'required':'inspect spatial support'})
            SPATIAL_MODEL_RESULTS.append(meta)
qa_table('ml51_spatial_protocols',pd.DataFrame(SPATIAL_MODEL_RESULTS),allow_empty=True)
if len(SPATIAL_MODEL_RESULTS):
    EXT_MODULES.append({'module':'ML-5.1','protocols':len(SPATIAL_MODEL_RESULTS),
      'tested':sum(x['status'].startswith('TESTED') for x in SPATIAL_MODEL_RESULTS),
      'interpretation':'SPATIAL_GENERALIZATION_RETROSPECTIVE_CAVEATS'})

qa_table('module_52_extension',pd.DataFrame(EXT_MODULES),allow_empty=True)
qa_table('seven_module_status_final',pd.DataFrame([{'module':k,**v} for k,v in MODULES.items()]),['module'])
print('Extended modules:',[(k,v['status']) for k,v in MODULES.items()])
# Финальная отметка этапа перенесена после всех его подмодулей


## 9Б. Региональный приём по специальностям и вузовский контингент
Подтверждённые количества по ВПО-1, отличные от отсутствующего приёма организации.

In [ ]:
# V4: measured organization outcomes and complete regional specialty admissions, with separate scientific targets.
progress_event('EXTENDED_MODULES','V4: реальный приём по специальностям ВПО-1; организация — отдельная величина',announce=True)
V4_SUBEXPERIMENTS=[]

def v4_block(name,reason,required='',rows=0):
    V4_SUBEXPERIMENTS.append({'subexperiment':name,'status':'NOT_FEASIBLE_FOR_TARGET',
       'reason':reason,'required':required,'n':int(rows)})
    note('V4_DATA_GATE',name+': '+reason,'WARN')

# Source VPO_2_1_1 has exact textual 'Принято, всего'. This is NOT a campus/branch observation.
_v4_vpo_p=fetch_asset(VPO_211)
V4_ADMISSION_PATH=ROOT/'derived'/'v4_regional_specialty_admissions.parquet'
_v4_s=f'''SELECT TRY_CAST(year AS INTEGER) observation_year, CAST(object_level AS VARCHAR) object_level,
    CAST(object_name AS VARCHAR) object_name, CAST(specialty_code AS VARCHAR) specialty_code,
    CAST(education_level AS VARCHAR) education_level,CAST(education_form AS VARCHAR) education_form,
    CAST(admission_type AS VARCHAR) admission_type,CAST(university_type AS VARCHAR) university_type,
    COUNT(*)::BIGINT records_in_cell,MIN(TRY_CAST(indicator_value AS DOUBLE)) minimum,
    MAX(TRY_CAST(indicator_value AS DOUBLE)) maximum
  FROM read_parquet({_sql_literal(_v4_vpo_p)})
  WHERE TRIM(CAST(indicator_name AS VARCHAR))='Принято, всего'
    AND regexp_matches(lower(CAST(object_level AS VARCHAR)),'регион|субъект')
    AND specialty_code IS NOT NULL AND year IS NOT NULL
  GROUP BY 1,2,3,4,5,6,7,8'''
_v4_tmp=V4_ADMISSION_PATH.with_suffix('.partial.parquet');_v4_tmp.unlink(missing_ok=True)
with duckdb.connect(':memory:') as _db:
    _db.execute("SET memory_limit='2048MB'");_db.execute('SET threads=1')
    _db.execute('SET temp_directory='+_sql_literal(DB_TMP));_db.execute('SET preserve_insertion_order=false')
    _db.execute('COPY ('+_v4_s+') TO '+_sql_literal(_v4_tmp)+' (FORMAT PARQUET,COMPRESSION ZSTD)')
    _v4_counts=_db.execute('''SELECT COUNT(*),COUNT(*) FILTER (WHERE records_in_cell=1 AND minimum=maximum
      AND minimum>=0 AND ABS(minimum-ROUND(minimum))<1e-8),COUNT(DISTINCT observation_year)
      FROM read_parquet(?)''',[str(_v4_tmp)]).fetchone()
_v4_tmp.replace(V4_ADMISSION_PATH);CONTEXT_PARQUET_FILES.append(V4_ADMISSION_PATH)
qa_table('v4_specialty_admission_gate',pd.DataFrame([{'source_id':VPO_211['asset_id'],
   'full_source_rows':int(VPO_211['rows_actual']),'exact_name':'Принято, всего',
   'regional_specialty_strata':int(_v4_counts[0]),'valid_singleton_nonnegative_integer_strata':int(_v4_counts[1]),
   'years':int(_v4_counts[2]),'source_grain':'REGION_X_SPECIALTY_X_EDUCATION_LEVEL_X_FORM_X_ADMISSION_TYPE_X_UNIVERSITY_TYPE',
   'warning':'NOT_ORGANIZATION_ADMISSIONS_AND_NOT_OD_BY_SPECIALTY'}]))
V4_PROGRAM_FRAME=pd.DataFrame()
if int(_v4_counts[1])>0:
    with duckdb.connect(':memory:') as _db:
        _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
        _v4_q='''SELECT * FROM read_parquet(?) WHERE records_in_cell=1 AND minimum=maximum
             AND minimum>=0 AND ABS(minimum-ROUND(minimum))<1e-8
             AND observation_year IS NOT NULL AND specialty_code!='' '''
        _v4_n=int(_db.execute('SELECT COUNT(*) FROM ('+_v4_q+')',[str(V4_ADMISSION_PATH)]).fetchone()[0])
        if CFG.MODE=='FULL' and CFG.ML54_FULL_NO_STRATA_CAP:
            # ALL verified strata (not 'longest 15000'); 237k observed in V4.
            # Pandas materialization is restricted to verified regional/specialty cells,
            # not the 7.5 million source rows. Never truncate study population.
            V4_PROGRAM_FRAME=_db.execute(_v4_q,[str(V4_ADMISSION_PATH)]).df()
            V4_PROGRAM_FULL_VALID_ROWS=int(len(V4_PROGRAM_FRAME))
        elif _v4_n<=CFG.MAX_PANDAS_ROWS:
            V4_PROGRAM_FRAME=_db.execute(_v4_q,[str(V4_ADMISSION_PATH)]).df()
            V4_PROGRAM_FULL_VALID_ROWS=int(len(V4_PROGRAM_FRAME))
        else:
            raise RuntimeError('ML54_FULL_STRATA_REQUIRED: no discretionary sample allowed')
    if int(_v4_n)!=int(len(V4_PROGRAM_FRAME)):
        raise RuntimeError(f'ML54_COVERAGE_LOSS: eligible={_v4_n}, materialized={len(V4_PROGRAM_FRAME)}')
    qa_table('v5_ml54_full_strata_coverage',pd.DataFrame([{
        'verified_rows':int(_v4_n),'materialized_rows':int(len(V4_PROGRAM_FRAME)),
        'fraction':1.0,'bounded_sample_used':False}]))
if len(V4_PROGRAM_FRAME):
    _f=V4_PROGRAM_FRAME.copy()
    for _c in ['object_name','specialty_code','education_level','education_form','admission_type','university_type']:
        _f[_c]=_f[_c].fillna('__MISSING__').astype(str)
    _fields=['object_name','specialty_code','education_level','education_form','admission_type','university_type']
    _f['entity_id']=_f[_fields].agg('|'.join,axis=1)
    _f['value']=_f['minimum'].astype(float)
    # A unique reporting stratum must not be interpreted as the sum of its components.
    _f=_f.drop_duplicates(['entity_id','observation_year'],keep=False)
    _f=_f[['entity_id','observation_year','value','object_name','specialty_code','education_level']]
    _rows=historical_pair_panel(_f[['entity_id','observation_year','value']],['entity_id'],'value')
    if len(_rows):
        _rows['sector']='VO_REGIONAL_SPECIALTY'
        _rows['target_id']='VPO_2_1_1_REPORTED_ADMITTED_EXACT_STRATUM'
        _rows['geography_grain']='REPORTED_REGION_SPECIALTY'
        qa_table('v4_specialty_model_frame',_rows,allow_empty=True)
        test_predictive('ML-5.4',_rows,nonnegative=True,count_confirmed=True)
        if MODULES['ML-5.4']['status'].startswith('TESTED'):
            MODULES['ML-5.4']['reason']='Exact VPO admitted value regional specialty/level/form/admission_type stratum; not OD and not institution-level'
            V4_SUBEXPERIMENTS.append({'subexperiment':'ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS',
               'status':'MODEL_TESTED','n':len(_rows),'source':VPO_211['asset_id']})
        else:v4_block('ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS','No historically available train/test for exact cells',rows=len(_rows))
    else:v4_block('ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS','Too few historical observations under publication lag',rows=len(_f))
else:v4_block('ML-5.4_REGIONAL_SPECIALTY_ADMISSIONS','No unique regional speciality admission strata',rows=int(_v4_counts[0]))
del V4_PROGRAM_FRAME;gc.collect()

# ML-5.2: do not auto-approve a number of admitted students from name-matched E02.
# A separate observed 'Приведенный контингент студентов' is a permissible alternative target,
# but it CANNOT be reported as organization admissions or interregional entrants.
_v4_e02_p=fetch_asset(E02)
_v4_org_target='Приведенный контингент студентов'
with duckdb.connect(':memory:') as _db:
    _db.execute("SET memory_limit='1024MB'");_db.execute('SET threads=1')
    _org=_db.execute('''SELECT CAST(id AS VARCHAR) id,TRY_CAST(year AS INTEGER) observation_year,
        COUNT(*)::BIGINT n_source_records,COUNT(DISTINCT CAST(area AS VARCHAR)) n_area,
        MIN(TRY_CAST(indicator_value AS DOUBLE)) low_value,MAX(TRY_CAST(indicator_value AS DOUBLE)) high_value
        FROM read_parquet(?) WHERE indicator_name=? AND id IS NOT NULL AND year IS NOT NULL
        GROUP BY 1,2''',[str(_v4_e02_p),_v4_org_target]).df()
qa_table('v4_organization_contingent_candidates',_org,allow_empty=True)
_org=_org[_org.n_source_records.eq(1)&_org.low_value.notna()&_org.low_value.eq(_org.high_value)&_org.low_value.ge(0)].copy()
if len(_org):
    _org['entity_id']='VO|'+norm_code(_org.id)
    _observed=_org[['entity_id','observation_year','low_value']].rename(columns={'low_value':'value'})
    _observed=_observed.drop_duplicates(['entity_id','observation_year'],keep=False)
    _panel=historical_pair_panel(_observed,['entity_id'],'value')
    if len(_panel):
        _panel['sector']='VO';_panel['target_id']='E02_REPORTED_STUDENT_CONTINGENT_NOT_ADMISSIONS'
        _panel['parent_group_id']=_panel['entity_id'];_panel['geography_grain']='ORGANIZATION'
        _split,_why=temporal_partition(_panel,min_train=CFG.MIN_TRAIN_ROWS,min_test=CFG.MIN_TEST_ROWS)
        if _split is not None:
            _tr=_split['train'];_te=_split['test']
            for _model in ('persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm'):
                fit_and_record('ML-5.2',_model,_tr,_te,seed=CFG.RANDOM_SEED,nonnegative=True,
                  protocol='ALT_CONTINGENT_RETROSPECTIVE')
            qa_table('v4_organization_contingent_model_frame',_panel,allow_empty=True)
            V4_SUBEXPERIMENTS.append({'subexperiment':'ML-5.2_ALTERNATIVE_CONTINGENT','status':'MODEL_TESTED_NOT_ADMISSIONS',
                                     'n':len(_te),'target':_v4_org_target})
            MODULES['ML-5.2'].update(status='ALTERNATIVE_TARGET_TESTED_ADMISSIONS_NOT_FEASIBLE',
                  reason='E02 student contingent model evaluated; original number admitted still unavailable for organization × year',n=len(_te))
        else:v4_block('ML-5.2_ALTERNATIVE_CONTINGENT',str(_why),rows=len(_panel))
    else:v4_block('ML-5.2_ALTERNATIVE_CONTINGENT','Insufficient stable organization/year contingent panel',rows=len(_org))
else:v4_block('ML-5.2_ALTERNATIVE_CONTINGENT','No unique organization/year source value; cannot sum area splits',rows=len(_org))
v4_block('ML-5.2_ORGANIZATION_ADMISSIONS','No certified admitted-count × organization × year in Stage I; student contingent is not admissions',
         'exact admission count, organization/branch identifiers, year and verified indicator/units') if len(ORG_YEAR)==0 else None
qa_table('v4_additional_subexperiments',pd.DataFrame(V4_SUBEXPERIMENTS),allow_empty=True)
progress_event('EXTENDED_MODULES','V4: проверка фактических величин ВПО/ВО завершена',announce=True)


## 9Г. ML-5.2 · углублённый временной эксперимент по организации и образовательной страте

Целевая величина — реально опубликованный контингент Е02. Это **не** число зачисленных; наблюдения разных страт не суммируются и не выдаются за организационный итог.

In [ ]:
# V5 · дополнительный строгий эксперимент ML-5.2: контингент организация × образовательная страта.
# Не равно приёму, не суммируется по перекрывающимся направлениям, не подменяет ML-5.2 admission target.
progress_event('EXTENDED_MODULES','V5 ML-5.2: проверка 106 тыс. исходных наблюдений по образовательным стратам',announce=True)
V5_ORG_AREA_STATUS={'status':'NOT_FEASIBLE_SOURCE_GRAIN','source':E02['asset_id']}
_v5_52_file=fetch_asset(E02)
_v5_52_sql='''SELECT CAST(id AS VARCHAR) id,TRY_CAST(year AS INTEGER) observation_year,
    COALESCE(CAST(area AS VARCHAR),'__NA__') area,
    COALESCE(CAST(specialty_code AS VARCHAR),'__NA__') specialty_code,
    COALESCE(CAST(specialty_section AS VARCHAR),'__NA__') specialty_section,
    COALESCE(CAST(indicator_unit AS VARCHAR),'__NA__') indicator_unit,
    COUNT(*)::BIGINT n_original_records,
    MIN(TRY_CAST(indicator_value AS DOUBLE)) min_value,
    MAX(TRY_CAST(indicator_value AS DOUBLE)) max_value
    FROM read_parquet(?) WHERE TRIM(CAST(indicator_name AS VARCHAR))='Приведенный контингент студентов'
      AND id IS NOT NULL AND year IS NOT NULL GROUP BY 1,2,3,4,5,6'''
with duckdb.connect(':memory:') as _con:
    _con.execute("SET memory_limit='1536MB'");_con.execute('SET threads=1')
    _v5_52_raw=_con.execute(_v5_52_sql,[str(_v5_52_file)]).df()
_v5_52_eligible=_v5_52_raw.loc[
    _v5_52_raw.n_original_records.eq(1)&_v5_52_raw.min_value.notna()&
    _v5_52_raw.min_value.eq(_v5_52_raw.max_value)&_v5_52_raw.min_value.ge(0)].copy()
V5_ORG_AREA_STATUS.update(raw_strata=len(_v5_52_raw),uniquely_reported_strata=len(_v5_52_eligible),
    duplicate_or_ambiguous_strata=int(len(_v5_52_raw)-len(_v5_52_eligible)))
if len(_v5_52_eligible):
    # The full combination, including area and specialty, is the statistical entity.
    _v5_52_eligible['entity_id']='VOAREA|'+_v5_52_eligible[
        ['id','area','specialty_code','specialty_section','indicator_unit']].astype(str).agg(
        lambda row:hashlib.sha256(json.dumps(row.tolist(),ensure_ascii=False).encode()).hexdigest()[:24],axis=1)
    _v5_52_eligible=_v5_52_eligible.drop_duplicates(['entity_id','observation_year'],keep=False)
    _v5_52_obs=_v5_52_eligible[['entity_id','observation_year','min_value']].rename(columns={'min_value':'value'})
    _v5_52_panel=historical_pair_panel(_v5_52_obs,['entity_id'],'value')
    V5_ORG_AREA_STATUS['complete_source_entity_year_rows']=len(_v5_52_obs)
    V5_ORG_AREA_STATUS['eligible_lagged_rows']=len(_v5_52_panel)
    qa_table('v5_ml52_organization_area_source_strata',_v5_52_eligible,allow_empty=True)
    if len(_v5_52_panel):
        _v5_52_panel['sector']='VO_CONTINGENT_AREA';_v5_52_panel['target_id']='E02_CONTINGENT_ORGANIZATION_AREA_NOT_ADMISSIONS'
        _v5_52_panel['parent_group_id']=_v5_52_panel.entity_id
        _v5_52_panel['geography_grain']='ORGANIZATION_AREA_SPECIALTY'
        _v5_52_split,_v5_52_reason=temporal_partition(_v5_52_panel)
        if _v5_52_split is not None:
            progress_event('EXTENDED_MODULES','ML-5.2: полный разведочный временной эксперимент по контингенту',announce=True)
            test_predictive('ML-5.2',_v5_52_panel,nonnegative=True,count_confirmed=False)
            if MODULES['ML-5.2']['status'].startswith('TESTED'):
                MODULES['ML-5.2'].update(status='ALTERNATIVE_ORG_AREA_CONTINGENT_TESTED_NOT_ADMISSIONS',
                    reason='Observed E02 student contingent by organization/area/specialty; NOT admitted cohorts, not institutional total')
                V5_ORG_AREA_STATUS.update(status='ALTERNATIVE_RETROSPECTIVE_TESTED_NOT_ADMISSIONS',
                    n_test=int(len(_v5_52_split['test'])),target_year=int(_v5_52_split['test_year']))
        else:
            V5_ORG_AREA_STATUS.update(status='NOT_FEASIBLE_HISTORICAL_SUPPORT',reason=str(_v5_52_reason))
else:
    V5_ORG_AREA_STATUS['reason']='No uniquely reported organization-area-year source cell, no admissible target'
qa_table('v5_ml52_area_model_execution',pd.DataFrame([V5_ORG_AREA_STATUS]),allow_empty=True)
print('ML-5.2 alternative:',V5_ORG_AREA_STATUS['status'],'| scientific admission target still gated')
del _v5_52_raw,_v5_52_eligible
import gc;gc.collect()


## 9В. E0/E1 по муниципальной образовательной инфраструктуре и выпускники
Полный дисковый SQL по БДМО; отдельный групповой тест занятости без выводов о закреплении.

In [ ]:
# V4. ML-5.7 disk-backed E0/E1; ML-5.5 independently held-out geography test.
progress_event('EXTENDED_MODULES','V4: E0/E1 полный муниципальный SQL и выпускники E04',announce=True)
V4_MUNICIPAL_STATUS={};V4_GRADUATE_STATUS={}
_v4_bdmo_file=ROOT/'derived'/'full52_context'/'bdmo_education_municipal.parquet'
if CFG.MODE=='FULL' and _v4_bdmo_file.is_file():
    _mi=MUNI[['oktmo','observation_year','migration_rate','population']].copy()
    _mi['oktmo']=norm_code(_mi.oktmo)
    for _c in ('migration_rate','population'):_mi[_c]=number(_mi[_c])
    _mi=_mi.dropna(subset=['oktmo','observation_year','migration_rate'])
    _mi=_mi.drop_duplicates(['oktmo','observation_year'],keep=False)
    _mi['target_year']=_mi.observation_year.astype(int)
    _mi['origin_year']=_mi.target_year-CFG.HORIZON
    _mi['feature_year']=_mi.origin_year-CFG.FEATURE_PUBLICATION_LAG_YEARS
    _hist=_mi[['oktmo','observation_year','migration_rate','population']].rename(columns={
        'observation_year':'feature_year','migration_rate':'lag_migration_rate','population':'lag_population'})
    _mi=_mi.merge(_hist,on=['oktmo','feature_year'],validate='many_to_one',how='inner')
    _mi=_mi.dropna(subset=['lag_migration_rate','lag_population'])
    _latest=int(_mi.target_year.max()) if len(_mi) else 0
    with duckdb.connect(':memory:') as _db:
        _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
        _db.execute('SET temp_directory='+_sql_literal(DB_TMP))
        _db.register('m',_mi)
        _q='''SELECT m.oktmo,m.target_year,m.origin_year,m.feature_year,m.migration_rate actual,
            m.lag_migration_rate,m.lag_population,CAST(e.indicator_code AS VARCHAR) indicator_code,
            CAST(e.indicator_name AS VARCHAR) indicator_name,CAST(e.indicator_unit AS VARCHAR) indicator_unit,
            e.median_reported education_historic_value
            FROM m JOIN read_parquet(?) e ON CAST(m.oktmo AS VARCHAR)=CAST(e.oktmo AS VARCHAR)
               AND m.feature_year=e.observation_year
            WHERE e.n_raw_rows=1 AND e.min_reported=e.max_reported AND e.median_reported>=0
              AND e.indicator_code IS NOT NULL'''
        _matched=ROOT/'derived'/'v4_municipality_education_exact_support.parquet'
        _db.execute('COPY ('+_q.replace('read_parquet(?)','read_parquet('+_sql_literal(_v4_bdmo_file)+')')+
                    ') TO '+_sql_literal(_matched)+' (FORMAT PARQUET,COMPRESSION ZSTD)')
        # Indicators are selected by training coverage, unit constancy and values, not TEST errors.
        _stats=_db.execute('''SELECT indicator_code,indicator_unit,MIN(indicator_name) AS example_indicator_name,
            COUNT(*) FILTER (WHERE target_year<=?) n_train_candidates,
            COUNT(DISTINCT oktmo) FILTER (WHERE target_year<=?) n_municipalities_train,
            COUNT(DISTINCT target_year) FILTER (WHERE target_year<=?) n_years_train,
            COUNT(DISTINCT education_historic_value) FILTER (WHERE target_year<=?) n_distinct_train,
            COUNT(*) FILTER (WHERE target_year=?) n_test_coverage,
            COUNT(DISTINCT oktmo) FILTER (WHERE target_year=?) n_municipalities_test,
            COUNT(DISTINCT indicator_name) n_indicator_names,
            COUNT(*) as rows_all
            FROM read_parquet(?) GROUP BY 1,2''',
          [_latest-2]*4+[_latest]*2+[str(_matched)]).df()
        _stats=_stats.sort_values(['n_municipalities_train','n_test_coverage','indicator_code'],ascending=[False,False,True])
        qa_table('v4_ml57_indicator_support',_stats,allow_empty=True)
        _eligible=_stats.loc[(_stats.n_train_candidates>=50)&(_stats.n_municipalities_train>=CFG.MIN_MUNICIPALITIES_E0E1)&
            (_stats.n_years_train>=2)&(_stats.n_distinct_train>1)&
            (_stats.n_municipalities_test>=CFG.MIN_MUNICIPALITIES_E0E1)&(_stats.n_indicator_names==1)]
        if len(_eligible):
            _eligible=_eligible.copy()
            _eligible['is_professional_education']=_eligible.example_indicator_name.astype(str).str.contains(
                r'профессиональн|высш|колледж|техникум|университет',case=False,regex=True,na=False)
            _eligible=_eligible.sort_values(['is_professional_education','n_municipalities_train','n_test_coverage','indicator_code'],
                ascending=[False,False,False,True])
            _one=_eligible.iloc[0];_selected=str(_one.indicator_code);_unit=str(_one.indicator_unit)
            V4_MUNICIPAL_INDICATOR_SEMANTIC='PROFESSIONAL_EDUCATION_CONTEXT' if bool(_one.is_professional_education) else 'GENERAL_EDUCATION_PROXY_ONLY'
            _rows=_db.execute('''SELECT * FROM read_parquet(?) WHERE indicator_code=?
                  AND COALESCE(indicator_unit,'')=COALESCE(?,'')''',[str(_matched),_selected,_unit]).df()
        else:_rows=pd.DataFrame();_selected=None
    CONTEXT_PARQUET_FILES.append(_matched)
    V4_MUNICIPAL_STATUS={'candidate_indicators':len(_stats),'matched_municipality_indicator_year_rows':int(pq.ParquetFile(_matched).metadata.num_rows),
       'target_year':_latest,'feasible_indicators':len(_eligible),'selected_code':_selected,'selected_semantic':locals().get('V4_MUNICIPAL_INDICATOR_SEMANTIC','UNAVAILABLE')}
    if len(_rows):
        _rows=_rows.sort_values(['oktmo','target_year','indicator_code'])
        _rows=_rows.drop_duplicates(['oktmo','target_year'],keep=False)
        _tr=_rows[_rows.target_year.le(_latest-2)].dropna(subset=['actual','lag_migration_rate','lag_population','education_historic_value']).copy()
        _te=_rows[_rows.target_year.eq(_latest)].dropna(subset=['actual','lag_migration_rate','lag_population','education_historic_value']).copy()
        if len(_tr)>=50 and len(_te)>=CFG.MIN_MUNICIPALITIES_E0E1:
            # Historical features are at target_year - 1 - publication lag; not current-year infrastructure.
            assert (_tr.feature_year<_tr.target_year).all() and (_te.feature_year<=_te.origin_year).all()
            assert (_tr.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS<=_te.origin_year.min()).all()
            qa_table('v4_ml57_exact_common_support',_rows,allow_empty=True)
            _p0=['lag_migration_rate','lag_population'];_p1=_p0+['education_historic_value']
            _results=[]
            for _label,_features in [('E0',_p0),('E1',_p1)]:
                _mod=Pipeline([('fill',SimpleImputer(strategy='median')),
                     ('scale',StandardScaler()),('ridge',Ridge(alpha=20.0))])
                _mod.fit(_tr[_features].to_numpy(float),_tr.actual.to_numpy(float))
                _yp=_mod.predict(_te[_features].to_numpy(float))
                _f=ROOT/'models'/f'v4_ml57_{_label}.joblib'
                joblib.dump({'model':_mod,'features':_features,'indicator_code':_selected},_f,compress=3)
                _loaded=joblib.load(_f)
                assert np.allclose(_loaded['model'].predict(_te[_features].to_numpy(float)),_yp)
                MODEL_FILES[_f.name]=str(_f)
                MODEL_RECORDS.append({'module':'ML-5.7','sector':'BDMO_EDUCATION_PROXY','model':_label,'seed':-1,
                  'path':_f.name,'file_sha256':hash_file(_f),'feature_num':_features,'feature_cat':[],
                  'training_rows':len(_tr),'test_rows':len(_te),'reloaded':True})
                _p=pd.DataFrame({'entity_id':_te.oktmo.astype(str).to_numpy(),
                  'origin_year':_te.origin_year.to_numpy(),'target_year':_te.target_year.to_numpy(),
                  'information_cutoff_year':_te.origin_year.to_numpy(),
                  'feature_available_year':_te.origin_year.to_numpy(),
                  'target_available_year':_te.target_year.to_numpy()+CFG.LABEL_PUBLICATION_LAG_YEARS,
                  'sector':'BDMO_EDUCATION_PROXY','target_id':'D01_MUNICIPAL_MIGRATION_RATE',
                  'actual':_te.actual.to_numpy()})
                _pred=predictions_frame(_p,'ML-5.7',_label,_yp,protocol='RETROSPECTIVE_PROXY_E0E1')
                _pred['geography_grain']='MUNICIPALITY';record_prediction(_pred)
                _mt=effective_mae(_te.actual,_yp)
                METRICS.append({'module':'ML-5.7','sector':'BDMO_EDUCATION_PROXY','model':_label,
                    'protocol':'RETROSPECTIVE_PROXY_E0E1','seed':-1,'target_year':_latest,
                    'source_vintage':INDEX['created_at'],**_mt})
                _results.append({'variant':_label,'n_test':len(_te),'n_train':len(_tr),'municipalities_test':_te.oktmo.nunique(),**_mt})
            qa_table('v4_ml57_results',pd.DataFrame(_results),allow_empty=True)
            V4_MUNICIPAL_STATUS.update(status='E0_E1_TESTED_PROXY_NOT_VERIFIED_CAMPUS',
                 n_train=len(_tr),n_test=len(_te),MAE_E0=_results[0]['MAE'],MAE_E1=_results[1]['MAE'])
            MODULES['ML-5.7'].update(status='RETROSPECTIVE_MUNICIPAL_PROXY_TEST',
                  reason='BDMO lagged '+V4_MUNICIPAL_INDICATOR_SEMANTIC+' E0/E1; same OKTMO-year support, DOES NOT verify campus or causal effect',n=len(_te))
        else:V4_MUNICIPAL_STATUS.update(status='NOT_FEASIBLE_TEMPORAL_SUPPORT',train=len(_tr),test=len(_te))
    else:V4_MUNICIPAL_STATUS['status']='NOT_FEASIBLE_EXACT_SHARED_SUPPORT'
else:V4_MUNICIPAL_STATUS={'status':'NOT_FEASIBLE_NO_BDMO_FULL_FILE'}
qa_table('v4_ml57_execution',pd.DataFrame([V4_MUNICIPAL_STATUS]),allow_empty=True)

# ML-5.5: out-of-region validation of employment rate on E04's original reporting rows.
# This is deliberately cross-sectional, not a forecast of retention or postgraduation residence.
progress_event('EXTENDED_MODULES','V4: трудоустройство E04, проверка вне регионов обучения',announce=True)
_gpath=fetch_asset(GRAD_ORG)
with duckdb.connect(':memory:') as _db:
    _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
    _gg=_db.execute('''SELECT CAST(object_name AS VARCHAR) region_name,
      CAST(object_level AS VARCHAR) object_level,CAST(university AS VARCHAR) university,
      CAST(gender AS VARCHAR) gender,CAST(education_level AS VARCHAR) education_level,
      TRY_CAST(year AS INTEGER) graduation_year,
      TRY_CAST(count_graduate AS DOUBLE) count_graduate,
      TRY_CAST(percent_employed AS DOUBLE) employed_pct
      FROM read_parquet(?) WHERE regexp_matches(lower(CAST(object_level AS VARCHAR)),'регион|субъект')
      AND TRY_CAST(percent_employed AS DOUBLE) BETWEEN 0 AND 100
      AND TRY_CAST(count_graduate AS DOUBLE)>=0''',[str(_gpath)]).df()
_gkeys=['region_name','university','gender','education_level','graduation_year']
_gduplicates=_gg.duplicated(_gkeys,keep=False).sum()
_gg=_gg.drop_duplicates(_gkeys,keep=False).copy()
_gcount=_gg.region_name.nunique()
V4_GRADUATE_STATUS={'valid_individual_reporting_rows':len(_gg),'ambiguous_reporting_rows_dropped':int(_gduplicates),
                    'n_geographies':int(_gcount),'employment_reference':'2024-12-31','is_longitudinal':False}
if len(_gg)>=100 and _gcount>=8 and _gg.employed_pct.nunique()>1:
    from sklearn.model_selection import GroupShuffleSplit
    _sp=GroupShuffleSplit(n_splits=1,test_size=.2,random_state=CFG.RANDOM_SEED)
    _a,_b=next(_sp.split(_gg,groups=_gg.region_name))
    _gtr=_gg.iloc[_a].copy();_gte=_gg.iloc[_b].copy()
    assert set(_gtr.region_name).isdisjoint(_gte.region_name)
    _cn=['count_graduate','graduation_year'];_cc=['education_level','gender']
    _Xtr=_gtr[_cn+_cc].copy();_Xte=_gte[_cn+_cc].copy()
    for _k in _cc:
        _Xtr[_k]=_Xtr[_k].fillna('UNKNOWN').astype(str);_Xte[_k]=_Xte[_k].fillna('UNKNOWN').astype(str)
    _gpred=[];_gy=_gte.employed_pct.to_numpy(float)
    _gbase=np.full(len(_gte),float(_gtr.employed_pct.mean()))
    _gpred.append(('region_blind_train_mean',_gbase))
    for _kind in ('ridge','lightgbm','catboost'):
        _num=Pipeline([('fill',SimpleImputer(strategy='median')),('scale',StandardScaler())])
        if _kind=='catboost':
            _med={k:float(_Xtr[k].median()) for k in _cn}
            _A=_Xtr.copy();_B=_Xte.copy()
            for _k in _cn:
                _A[_k]=_A[_k].fillna(_med[_k]);_B[_k]=_B[_k].fillna(_med[_k])
            _mod=CatBoostRegressor(iterations=180,depth=4,learning_rate=.06,verbose=False,
                random_seed=CFG.RANDOM_SEED,thread_count=1,allow_writing_files=False)
            _mod.fit(_A,_gtr.employed_pct.to_numpy(float),cat_features=_cc)
            _gp=_mod.predict(_B)
            _art={'model':_mod,'medians':_med,'categoricals':_cc,'features':_cn+_cc}
        else:
            _prep=ColumnTransformer([('n',_num,_cn),('c',OneHotEncoder(handle_unknown='ignore'),_cc)])
            if _kind=='ridge':_est=Ridge(alpha=30.,solver='lsqr')
            else:_est=LGBMRegressor(n_estimators=170,max_depth=4,num_leaves=12,
                   n_jobs=1,verbosity=-1,random_state=CFG.RANDOM_SEED)
            _mod=Pipeline([('prep',_prep),('reg',_est)])
            _mod.fit(_Xtr,_gtr.employed_pct.to_numpy(float));_gp=_mod.predict(_Xte)
            _art={'model':_mod,'features':_cn+_cc}
        _gp=np.clip(_gp,0,100)
        _gpred.append((_kind,_gp))
        _mp=ROOT/'models'/f'v4_ml55_region_heldout_{_kind}.joblib'
        joblib.dump(_art,_mp,compress=3)
        _rt=joblib.load(_mp)
        if _kind=='catboost':
            _verify=_Xte.copy()
            for _k in _cn:_verify[_k]=_verify[_k].fillna(_rt['medians'][_k])
        else:_verify=_Xte
        assert np.allclose(np.clip(_rt['model'].predict(_verify),0,100),_gp)
        MODEL_FILES[_mp.name]=str(_mp)
        MODEL_RECORDS.append({'module':'ML-5.5','sector':'GRAD_E04_CROSS_SECTION',
            'model':_kind,'seed':CFG.RANDOM_SEED,'path':_mp.name,'file_sha256':hash_file(_mp),
            'training_rows':len(_gtr),'test_rows':len(_gte),'reloaded':True,'feature_num':_cn,'feature_cat':_cc})
    _gm=[];_gpframe=_gte[['region_name','university','graduation_year','employed_pct']].reset_index(drop=True).copy()
    for _kind,_yp in _gpred:
        _gpframe['pred_'+_kind]=_yp
        _gm.append({'module':'ML-5.5','model':_kind,'protocol':'REGION_HELD_OUT_CROSS_SECTION_2024',
            'n_test':len(_gte),'n_regions_test':_gte.region_name.nunique(),**effective_mae(_gy,_yp)})
    qa_table('v4_ml55_crosssection_predictions',_gpframe,allow_empty=True)
    qa_table('v4_ml55_crosssection_metrics',pd.DataFrame(_gm),allow_empty=True)
    V4_GRADUATE_STATUS.update(status='REGIONAL_HOLDOUT_EMPLOYMENT_TEST_NOT_RETENTION',n_test=len(_gte),n_train=len(_gtr),
       baseline_mae=_gm[0]['MAE'],tested_methods=len(_gpred))
    MODULES['ML-5.5'].update(status='DESCRIPTIVE_AND_CROSS_SECTIONAL_TEST',
       reason='All four E04/E05 descriptive; E04 regional OOS employment test does not imply retention/temporal forecast',n=len(_gte))
else:V4_GRADUATE_STATUS['status']='NOT_FEASIBLE_REGION_HELDOUT_AFTER_EXACT_STRATA_GATE'
qa_table('v4_ml55_execution',pd.DataFrame([V4_GRADUATE_STATUS]),allow_empty=True)

# ML-5.6: historical node centrality is tested on common regions only; coverage break is an explicit caveat.
_v4_net=pd.DataFrame(NETWORK)
_v4_nresults=[]
if len(_v4_net) and _v4_net.year.nunique()>1:
    _v4_years=sorted(_v4_net.year.unique())
    for _prev,_now in zip(_v4_years[:-1],_v4_years[1:]):
        _p=_v4_net[_v4_net.year.eq(_prev)][['region_code','pagerank']].rename(columns={'pagerank':'prior_pagerank'})
        _n=_v4_net[_v4_net.year.eq(_now)][['region_code','pagerank']].rename(columns={'pagerank':'actual_pagerank'})
        _joined=_p.merge(_n,on='region_code',validate='one_to_one')
        _rho,_pv=(spearmanr(_joined.prior_pagerank,_joined.actual_pagerank) if len(_joined)>=3 else (np.nan,np.nan))
        _v4_nresults.append({'origin_year':int(_prev),'target_year':int(_now),'common_regions':len(_joined),
            'pagerank_persistence_mae':float(np.abs(_joined.prior_pagerank-_joined.actual_pagerank).mean()) if len(_joined) else np.nan,
            'pagerank_rank_spearman':float(_rho) if np.isfinite(_rho) else np.nan,
            'nominal_p_value_NOT_CONFIRMATORY':float(_pv) if np.isfinite(_pv) else np.nan,
            'coverage_break':'UNRESOLVED_2023_OR_OTHER_YEAR_COVERAGE'})
qa_table('v4_ml56_node_forecast',pd.DataFrame(_v4_nresults),allow_empty=True)


## 9Д. ML-5.7 · независимое географическое испытание образовательного муниципального контекста

Дополнительный тест E0/E1 по пространственно отложенным муниципалитетам. Образовательный признак берётся из БДМО в тот же наблюдаемый год и поэтому **не** является историческим прогнозным признаком. Это только диагностическая ассоциация, а не эффект университета или доказательство миграции.

In [ ]:
# V5 · ML-5.7: независимый *пространственный*, не временной E0/E1 diagnostic.
# General education proxy from BDMO; no academic-campus location, no residence effect inferred.
progress_event('EXTENDED_MODULES','V5 ML-5.7: пространственный E0/E1 на наблюдаемом образовательном контексте',announce=True)
V5_MUNICIPAL_CROSS_STATUS={'status':'NOT_IDENTIFIABLE_ON_AVAILABLE_CROSS_SECTION'}
if CFG.MODE=='FULL' and 'BDMO_EDU_PROXY_CANDIDATES' in globals() and len(BDMO_EDU_PROXY_CANDIDATES):
    _v5_e=BDMO_EDU_PROXY_CANDIDATES.copy()
    _v5_e['oktmo']=norm_code(_v5_e.oktmo);_v5_e['observation_year']=safe_year(_v5_e.observation_year)
    _v5_e=_v5_e.loc[_v5_e.n_raw_rows.eq(1)&_v5_e.min_reported.eq(_v5_e.max_reported)&
        _v5_e.median_reported.notna()&_v5_e.median_reported.ge(0)].copy()
    _v5_m=MUNI[['oktmo','observation_year','migration_rate','population']].copy()
    _v5_m['oktmo']=norm_code(_v5_m.oktmo);_v5_m['observation_year']=safe_year(_v5_m.observation_year)
    for _k in ('migration_rate','population'):_v5_m[_k]=number(_v5_m[_k])
    _v5_m=_v5_m.dropna(subset=['oktmo','observation_year','migration_rate','population'])
    _v5_m=_v5_m.drop_duplicates(['oktmo','observation_year'],keep=False)
    _v5_join=_v5_m.merge(_v5_e,on=['oktmo','observation_year'],how='inner',validate='one_to_many')
    _v5_join['is_heldout']=_v5_join.oktmo.astype(str).map(lambda x:int(
         hashlib.sha256(('V5_MUNICIPAL_HOLDOUT|'+x).encode()).hexdigest()[:8],16)%5==0)
    _v5_join['indicator_code']=_v5_join.indicator_code.astype(str)
    _v5_join['indicator_unit']=_v5_join.indicator_unit.astype(str)
    _v5_stats=_v5_join.loc[~_v5_join.is_heldout].groupby(['indicator_code','indicator_unit'],dropna=False).agg(
        n_train_muni=('oktmo','nunique'),n_train_rows=('migration_rate','size'),
        n_train_years=('observation_year','nunique'),value_variation=('median_reported','nunique')).reset_index()
    _v5_stats=_v5_stats.sort_values(['n_train_muni','n_train_years','indicator_code'],ascending=[False,False,True])
    qa_table('v5_ml57_spatial_proxy_eligibility',_v5_stats,allow_empty=True)
    _v5_best=_v5_stats.loc[(_v5_stats.n_train_muni>=CFG.MIN_MUNICIPALITIES_E0E1)&
        (_v5_stats.value_variation>1)]
    if len(_v5_best):
        _v5_code=str(_v5_best.iloc[0].indicator_code);_v5_unit=str(_v5_best.iloc[0].indicator_unit)
        _v5_k=_v5_join.loc[_v5_join.indicator_code.eq(_v5_code)&_v5_join.indicator_unit.eq(_v5_unit)].copy()
        _v5_k=_v5_k.drop_duplicates(['oktmo','observation_year'],keep=False)
        _v5_train=_v5_k.loc[~_v5_k.is_heldout].copy();_v5_test=_v5_k.loc[_v5_k.is_heldout].copy()
        if len(_v5_train)>=100 and _v5_test.oktmo.nunique()>=CFG.MIN_MUNICIPALITIES_E0E1:
            _v5_features={'E0':['population','observation_year'],
                          'E1':['population','observation_year','median_reported']}
            _v5_out=[]
            for _variant,_cols in _v5_features.items():
                _v5_mod=Pipeline([('fill',SimpleImputer(strategy='median')),
                     ('scale',StandardScaler()),('ridge',Ridge(alpha=30.0))])
                _v5_mod.fit(_v5_train[_cols].to_numpy(float),_v5_train.migration_rate.to_numpy(float))
                _v5_pred=_v5_mod.predict(_v5_test[_cols].to_numpy(float))
                _v5_f=ROOT/'models'/('v5_ml57_crosssection_'+_variant+'_'+hashlib.sha256(
                    (_v5_code+'|'+_v5_unit).encode()).hexdigest()[:10]+'.joblib')
                if _v5_f.exists() or _v5_f.name in MODEL_FILES:raise RuntimeError('V5 ML57 model artifact collision')
                joblib.dump({'model':_v5_mod,'features':_cols,'indicator_code':_v5_code,
                   'protocol':'CROSS_SECTION_NOT_TEMPORAL'},_v5_f,compress=3)
                _v5_loaded=joblib.load(_v5_f)
                if not np.allclose(_v5_pred,_v5_loaded['model'].predict(_v5_test[_cols].to_numpy(float))):
                    raise RuntimeError('V5 ML57 reloaded cross-sectional model mismatch')
                MODEL_FILES[_v5_f.name]=str(_v5_f)
                MODEL_RECORDS.append({'module':'ML-5.7','sector':'BDMO_GENERAL_EDUCATION_CONTEXT',
                  'protocol':'SPATIAL_CROSS_SECTION_NOT_FORECAST','model':_variant,'seed':CFG.RANDOM_SEED,
                  'path':_v5_f.name,'file_sha256':hash_file(_v5_f),'feature_num':_cols,'feature_cat':[],
                  'training_rows':len(_v5_train),'test_rows':len(_v5_test),'reloaded':True})
                _v5_pf=pd.DataFrame({'entity_id':_v5_test.oktmo.astype(str).to_numpy(),
                  'origin_year':_v5_test.observation_year.to_numpy(),'target_year':_v5_test.observation_year.to_numpy(),
                  'information_cutoff_year':_v5_test.observation_year.to_numpy(),
                  'feature_available_year':_v5_test.observation_year.to_numpy(),
                  'target_available_year':_v5_test.observation_year.to_numpy(),
                  'sector':'BDMO_GENERAL_EDUCATION_CONTEXT',
                  'target_id':'D01_MUNICIPAL_MIGRATION_RATE_CONTEMPORANEOUS_ASSOCIATION',
                  'actual':_v5_test.migration_rate.to_numpy(float)})
                _v5_pred_rows=predictions_frame(_v5_pf,'ML-5.7',_variant,_v5_pred,
                                   seed=CFG.RANDOM_SEED,protocol='SPATIAL_CROSS_SECTION_NOT_FORECAST')
                _v5_pred_rows['geography_grain']='MUNICIPALITY';record_prediction(_v5_pred_rows)
                _v5_met=effective_mae(_v5_test.migration_rate,_v5_pred)
                METRICS.append({'module':'ML-5.7','sector':'BDMO_GENERAL_EDUCATION_CONTEXT',
                  'model':_variant,'protocol':'SPATIAL_CROSS_SECTION_NOT_FORECAST',
                  'seed':CFG.RANDOM_SEED,'target_year':int(_v5_test.observation_year.max()),**_v5_met})
                _v5_out.append({'variant':_variant,'n_train':len(_v5_train),'n_test':len(_v5_test),
                       'heldout_municipalities':_v5_test.oktmo.nunique(),**_v5_met})
            qa_table('v5_ml57_spatial_E0_E1_results',pd.DataFrame(_v5_out),allow_empty=True)
            V5_MUNICIPAL_CROSS_STATUS={'status':'CROSS_SECTION_ASSOCIATION_TESTED_NOT_TEMPORAL',
                'selected_source_indicator':_v5_code,'unit':_v5_unit,
                'years':int(_v5_k.observation_year.nunique()),'n_train':len(_v5_train),
                'n_test':len(_v5_test),'MAE_E0':_v5_out[0]['MAE'],'MAE_E1':_v5_out[1]['MAE'],
                'target':'municipal_migration_rate','not_campus_metric':True}
            if MODULES['ML-5.7'].get('status','').startswith('NOT_FEASIBLE'):
                MODULES['ML-5.7'].update(status='SPATIAL_ASSOCIATION_ONLY_NOT_E0E1_TEMPORAL',
                    reason='Municipal held-out E0/E1 general education context, no temporal as-of claim and no verified VO/SPO campuses')
        else:V5_MUNICIPAL_CROSS_STATUS['reason']='Insufficient held-out municipality count for spatial test'
    else:V5_MUNICIPAL_CROSS_STATUS['reason']='No valid unique municipal indicator across enough training municipalities'
qa_table('v5_ml57_spatial_E0_E1_execution',pd.DataFrame([V5_MUNICIPAL_CROSS_STATUS]),allow_empty=True)
print('ML-5.7 supplemental:',V5_MUNICIPAL_CROSS_STATUS['status'],'| temporal E0/E1 remains a separate gate')


## V7 · дополнительные предметные проверки: OD, кандидаты приёма, муниципальные E0/E1

In [ ]:
# V7.4.0 · дополнительные проверки научной идентифицируемости и устойчивости.
progress_event('EXTENDED_MODULES','V7: сопоставимость OD, точные целевые показатели, муниципальная поддержка',announce=True)

# 5.1/5.3/5.6: диагностируем устойчивость только на ОБЩИХ фактически наблюдавшихся OD-парах.
# Отсутствующие рёбра никогда не превращаем в нули и не делаем вывод об общей динамике страны.
_v7_od=OD_INTER[['observation_year','origin_region','destination_region','observed_value']].dropna().copy()
_v7_od['observation_year']=pd.to_numeric(_v7_od.observation_year,errors='coerce')
_v7_od=_v7_od[_v7_od.observation_year.notna()].copy()
_v7_od['observation_year']=_v7_od.observation_year.astype(int)
if _v7_od.duplicated(['observation_year','origin_region','destination_region']).any():
    raise RuntimeError('V7 OD: неоднозначный ключ год×откуда×куда')
_v7_years=sorted(_v7_od.observation_year.unique())
V7_OD_COMPARABLE=[]
for _cy in _v7_years[1:]:
    _pr=_v7_od[_v7_od.observation_year.eq(_cy-1)]
    _cr=_v7_od[_v7_od.observation_year.eq(_cy)]
    _pair=_cr.merge(_pr[['origin_region','destination_region','observed_value']].rename(
        columns={'observed_value':'previous_value'}),on=['origin_region','destination_region'],how='inner',validate='one_to_one')
    _shared=int(len(_pair));_now=int(len(_cr));_old=int(len(_pr))
    V7_OD_COMPARABLE.append({'year':int(_cy),'observed_edges_previous':_old,'observed_edges_current':_now,
       'shared_edges':_shared,'previous_coverage':_shared/_old if _old else np.nan,
       'current_coverage':_shared/_now if _now else np.nan,
       'MAE_persistence_on_common_edges':float((_pair.observed_value-_pair.previous_value).abs().mean()) if _shared else np.nan,
       'sum_current_on_common_edges':float(_pair.observed_value.sum()),
       'sum_previous_on_common_edges':float(_pair.previous_value.sum()),
       'interpretation':'INTERSECTION_OF_OBSERVED_EDGES_ONLY_NOT_TOTAL_FLOW',
       'coverage_break_unresolved':bool(_old and _now and (_now/_old>1.5 or _now/_old<.67))})
qa_table('v7_od_common_support_sensitivity',pd.DataFrame(V7_OD_COMPARABLE),allow_empty=True)

# 5.2: полная поисковая проверка утверждаемых целевых индикаторов E02 (в отличие от контингента).
# Публикация названия, значения и числа наблюдений — доказательство кандидата, но НЕ его утверждение.
V7_ORG_ADMISSION_SEARCH=pd.DataFrame()
try:
    _v7_e02=fetch_asset(E02)
    with duck_connect() as _db:
        _v7_g=_db.execute('''SELECT CAST(indicator_name AS VARCHAR) indicator_name,
             CAST(indicator_unit AS VARCHAR) indicator_unit,
             COUNT(*)::BIGINT n_source_rows,COUNT(DISTINCT CAST(id AS VARCHAR)) n_organization_ids,
             COUNT(DISTINCT TRY_CAST(year AS INT)) n_years,
             COUNT(*) FILTER (WHERE TRY_CAST(indicator_value AS DOUBLE) IS NOT NULL) numeric_rows,
             COUNT(*) FILTER (WHERE TRY_CAST(indicator_value AS DOUBLE)<0) negative_rows,
             COUNT(DISTINCT COALESCE(CAST(area AS VARCHAR),'__NULL__')) n_area_values
          FROM read_parquet(?) WHERE regexp_matches(lower(CAST(indicator_name AS VARCHAR)),
            'принят|приём|прием|зачислен|поступив') GROUP BY 1,2''',[str(_v7_e02)]).df()
    _v7_g['indicator_approved']=_v7_g.indicator_name.isin(tuple(CFG.APPROVED_VO_ADMISSION_NAMES))
    _v7_g['has_organization_year_provenance']=_v7_g.n_organization_ids.gt(0)&_v7_g.n_years.ge(2)
    _v7_g['is_certified_admission_total']=False  # name matching and counts never certify statistical semantics
    _v7_g['gate_reason']='REQUIRES_PUBLISHED_METHOD_AND_UNIQUE_ORGANIZATION_YEAR_COHORT'
    V7_ORG_ADMISSION_SEARCH=_v7_g
except Exception as _ex:
    ISSUES.append({'module':'ML-5.2','status':'V7_ADMISSION_DISCOVERY_FAILURE',
                   'reason':type(_ex).__name__+': '+str(_ex)[:160],
                   'required':'repair E02 indicator inventory scan'})
    if CFG.MODE=='FULL': raise
qa_table('v7_organization_admission_discovery',V7_ORG_ADMISSION_SEARCH,allow_empty=True)

# 5.7: строго показываем, почему E0/E1 с учебными площадками выполним или нет.
_v7_muni_status=globals().get('V4_MUNICIPAL_STATUS',{})
_v7_spatial=globals().get('V5_MUNICIPAL_CROSS_STATUS',{})
_v7_campus_n=int(REGISTRY.campus_geo_status.eq('CONFIRMED_STUDY_CAMPUS').sum()) if len(REGISTRY) else 0
V7_MUNICIPAL_SUPPORT=pd.DataFrame([{
    'module':'ML-5.7','municipal_demography_rows':int(len(MUNI)),
    'bdmo_proxy_rows':int(len(BDMO_EDU_PROXY_CANDIDATES)),
    'verified_education_campus_records':_v7_campus_n,
    'temporal_e0e1_status':str(_v7_muni_status.get('status','NOT_VERIFIED')),
    'spatial_e0e1_status':str(_v7_spatial.get('status','NOT_VERIFIED')),
    'original_campus_e0e1_identifiable':bool(_v7_campus_n>0 and MODULES['ML-5.7']['status'].startswith('TESTED')),
    'interpretation':'MUNICIPAL_PROXIES_NOT_IDENTICAL_TO_HIGHER_EDUCATION_CAMPUS_CAPACITY',
    'requirement':'VERIFIED_CAMPUS_OKTMO_BY_YEAR_AND_COMPARABLE_MUNICIPAL_OUTCOME'
}])
qa_table('v7_municipal_identification_gate',V7_MUNICIPAL_SUPPORT,['module'])
print('V7: общая поддержка OD',len(V7_OD_COMPARABLE),'пар лет; кандидатов E02',len(V7_ORG_ADMISSION_SEARCH),
      '; подтверждённых учебных площадок',_v7_campus_n)


## 9Е. Дополнительные проверяемые эксперименты V7.1
При наличии подтверждённого целевого показателя проверяется новый временной прогноз; при отсутствии цель исключается без искусственных значений. Сохраняются все ранее выполненные модели V7.

In [ ]:
# ML-05 V7.4.0 · All defensible new tests, retaining the completed V7 seven-module models.
progress_event('EXTENDED_MODULES','V7.4: новые проверяемые подэксперименты ML-5.1–ML-5.7',announce=True)
ADR_EXPERIMENTS=[]

def adr_record_exp(module,code,status,n_train=0,n_test=0,reason='',source_ids='',scientific_claim=''):
    ADR_EXPERIMENTS.append({'module':module,'experiment':code,'status':status,
        'train_rows':int(n_train),'test_rows':int(n_test),'source_ids':source_ids,
        'scientific_claim':scientific_claim,'reason':str(reason)[:500]})

# ML-5.1/3/6: independent stability diagnostics use the same recorded source E01 only.
if len(ADR_OD_PAIRWISE):
    _adr_break=ADR_OD_PAIRWISE.loc[ADR_OD_PAIRWISE.coverage_break_at_2023.eq(True)]
    _adr_ok_years=ADR_OD_PAIRWISE.loc[(ADR_OD_PAIRWISE.coverage_of_prior>=.8)&
                                      (ADR_OD_PAIRWISE.coverage_of_current>=.8)].copy()
    qa_table('adr_od_high_overlap_year_comparisons',_adr_ok_years,allow_empty=True)
    adr_record_exp('ML-5.1','OD_HIGH_OVERLAP_STABILITY','DESCRIPTIVE_COMPLETED',
        len(ADR_OD_BALANCED),len(_adr_ok_years),
        '2023 coverage break recorded separately; no missing OD zero replacement',E01['asset_id'],
        'Only the intersection of actually reported regional schooling-study edges')
    adr_record_exp('ML-5.3','OUTFLOW_BALANCED_OD_SUPPORT','DESCRIPTIVE_COMPLETED',
        len(ADR_OD_BALANCED),len(_adr_ok_years),'OD annual marginal totals not comparable without common support',E01['asset_id'],
        'Observed outflow from prior schooling region, not permanent population outmigration')
    adr_record_exp('ML-5.6','NETWORK_BALANCED_GRAPH_SUPPORT','DESCRIPTIVE_COMPLETED',
        len(ADR_OD_BALANCED),len(_adr_ok_years),'Temporal graph robustness restricted to comparable reported edges',E01['asset_id'],
        'Observed OD network topology, not residential migration infrastructure')
else:
    adr_record_exp('ML-5.1','OD_HIGH_OVERLAP_STABILITY','SKIPPED_NO_MATCHED_YEARS',reason='No contiguous years of observed OD pairs',source_ids=E01['asset_id'])

# Additional testable V7.1 sensitivity: fixed-source common-edge (and origin marginal) time forecasts.
# Each protocol has a separate artifact ID and only uses observed rows; no TEST-based model choice.
def adr_balanced_time_experiment(module,observation,entity_fields,target,sector,target_id,protocol,min_train=100,min_test=20):
    if not len(observation):
        adr_record_exp(module,protocol,'SKIPPED_NO_COMMON_SOURCE_EDGES',source_ids=E01['asset_id'])
        return
    _f=historical_pair_panel(observation,entity_fields,target)
    if _f.empty:
        adr_record_exp(module,protocol,'SKIPPED_NO_HISTORICAL_LAG',source_ids=E01['asset_id'])
        return
    _f['entity_id']=_f[entity_fields].astype(str).agg('|'.join,axis=1)
    _f['sector']=sector;_f['target_id']=target_id
    _split,_reason=temporal_partition(_f,min_train=min_train,min_test=min_test)
    if _split is None:
        adr_record_exp(module,protocol,'SKIPPED_TIME_SUPPORT',len(_f),0,_reason,E01['asset_id'])
        return
    _tr=_split['train'];_te=_split['test']
    _models=['persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm']
    # PPML/NB are already evaluated exhaustively in the primary V7 OD protocol;
    # the balanced sensitivity adds independent, predeclared non-count-specific checks.
    _before=len(METRICS)
    for _model in _models:
        progress_event('EXTENDED_MODULES',f'{protocol}: {_model}',announce=True)
        fit_and_record(module,_model,_tr,_te,seed=-1,protocol=protocol,
                       nonnegative=True,allow_ppml=False)
    adr_record_exp(module,protocol,'MODEL_COMPLETED',len(_tr),len(_te),
        f'{len(METRICS)-_before} predeclared methods; no selection from TEST',E01['asset_id'],
        'Observed stable OD subpopulation, not a national movement count or source-year admission')

if len(ADR_OD_BALANCED):
    _adr_observed=ADR_OD_BALANCED[['observation_year','origin_region','destination_region','observed_value']].copy()
    adr_balanced_time_experiment('ML-5.1',_adr_observed,['origin_region','destination_region'],
        'observed_value','SPO_BALANCED_OD','E01_BALANCED_OBSERVED_EDGES','ADR_BALANCED_OD_TIME',100,20)
    _adr_balanced_outflow=_adr_observed.groupby(['observation_year','origin_region'],as_index=False).agg(
        balanced_observed_outflow=('observed_value','sum'))
    adr_balanced_time_experiment('ML-5.3',_adr_balanced_outflow,['origin_region'],
        'balanced_observed_outflow','SPO_BALANCED_ORIGIN','E01_BALANCED_ORIGIN_OUTFLOW','ADR_BALANCED_OUTFLOW_TIME',
        CFG.REGION_MIN_TRAIN_ROWS,CFG.REGION_MIN_TEST_ROWS)
else:
    for _m,_protocol in [('ML-5.1','ADR_BALANCED_OD_TIME'),('ML-5.3','ADR_BALANCED_OUTFLOW_TIME')]:
        adr_record_exp(_m,_protocol,'SKIPPED_NO_ALL_YEAR_OD_SUPPORT',0,0,
          'No OD edge appears in every source year; no imputation',E01['asset_id'])

# ML-5.2: actual admissions are modeled only if official indicator labels have been explicitly approved
# and the organizational-year target has no specialty/area duplication.
if len(ADR_ADMISSION_YEAR):
    _adr_approved_frame=ADR_ADMISSION_YEAR[['entity_id','observation_year','actual']].rename(columns={'actual':'observed_admitted'}).copy()
    _adr_org_frame=historical_pair_panel(_adr_approved_frame,['entity_id'],'observed_admitted')
    if len(_adr_org_frame):
        _adr_org_frame['sector']='VO_VERIFIED_ADMISSION'
        _adr_org_frame['target_id']='CERTIFIED_VO_UNIQUE_ORG_YEAR_ADMISSIONS'
        _adr_org_frame['parent_group_id']=_adr_org_frame.entity_id
        _adr_org_frame['historical_target_has_semantic_approval']=True
        _adr_split,_adr_split_reason=temporal_partition(_adr_org_frame,min_train=CFG.MIN_TRAIN_ROWS,min_test=CFG.MIN_TEST_ROWS)
        if _adr_split is not None:
            _adr_ex_before=len(METRICS)
            test_predictive('ML-5.2',_adr_org_frame,nonnegative=True,count_confirmed=True)
            _adr_new_models=len(METRICS)-_adr_ex_before
            adr_record_exp('ML-5.2','APPROVED_VO_ADMISSIONS_RETROSPECTIVE','MODEL_COMPLETED',
                len(_adr_split['train']),len(_adr_split['test']),f'{_adr_new_models} new metric rows',E02['asset_id'],
                'Source-approved admissions by organization and year; not student-stock or applications')
        else:
            adr_record_exp('ML-5.2','APPROVED_VO_ADMISSIONS_RETROSPECTIVE','SKIPPED_TIME_SUPPORT',
                len(_adr_org_frame),0,_adr_split_reason,E02['asset_id'])
    else:
        adr_record_exp('ML-5.2','APPROVED_VO_ADMISSIONS_RETROSPECTIVE','SKIPPED_NO_HISTORICAL_PAIRS',
            len(ADR_ADMISSION_YEAR),0,'No historically available count lag',E02['asset_id'])
else:
    adr_record_exp('ML-5.2','APPROVED_VO_ADMISSIONS_RETROSPECTIVE','SKIPPED_SEMANTICALLY_UNVERIFIED',
        0,0,'Published source data did not certify unique admissions by organization-year; V7 contingent experiment is retained',E02['asset_id'],
        'Candidate labels cannot be converted into real admissions without verified definition')

# ML-5.4/5.5 existing V7 tests already use all matching raw source rows; no invented additional target.
for _adr_mod,_adr_exp,_adr_asset in [('ML-5.4','FULL_SPECIALTY_STRATA_MODEL',VPO_211['asset_id']),
     ('ML-5.5','ALL_FOUR_GRADUATE_FAMILIES',GRAD_ORG['asset_id'])]:
    _adr_success=bool(any(r.get('module')==_adr_mod for r in MODEL_RECORDS) or
                      (_adr_mod=='ML-5.5' and len(PANELS.get('graduate_descriptive_statistics',pd.DataFrame()))))
    adr_record_exp(_adr_mod,_adr_exp,'V7_COMPLETED' if _adr_success else 'SKIPPED_UNAVAILABLE_TARGET',
        reason='Existing V7 observational tests retained; no graduate job-location inference',source_ids=_adr_asset)

# ML-5.7: optional independent E0/E1 *associational* forecasting on geocoded SPO proxies.
# Only available observed SPO institution-years are used. Neither absent organizations nor sites are filled with zero.
ADR_ML57_GEO_STATUS={'status':'SKIPPED_NO_SPATIAL_PROXY','n_spo_location_proxies':len(ADR_GEO_VERIFIED)}
if len(ADR_GEO_VERIFIED):
    _adr_sites=ADR_GEO_VERIFIED[['entity_id','observation_year','candidate_municipal_oktmo']].copy()
    _adr_sites['oktmo']=norm_code(_adr_sites.candidate_municipal_oktmo)
    _adr_sites['observation_year']=safe_year(_adr_sites.observation_year)
    _adr_sites=_adr_sites.dropna(subset=['oktmo','observation_year']).drop_duplicates()
    _adr_site_counts=_adr_sites.groupby(['oktmo','observation_year'],as_index=False).agg(
        observed_geocoded_spo_entities=('entity_id','nunique'))
    qa_table('adr_spo_site_year_observed_counts',_adr_site_counts,['oktmo','observation_year'],allow_empty=True)
    _adr_m=MUNI[['oktmo','observation_year','population','migration_rate']].copy()
    _adr_m['oktmo']=norm_code(_adr_m.oktmo)
    _adr_m['observation_year']=safe_year(_adr_m.observation_year)
    _adr_m['actual']=number(_adr_m.migration_rate)
    _adr_m['population']=number(_adr_m.population)
    _adr_m=_adr_m.dropna(subset=['oktmo','observation_year','actual']).drop_duplicates(['oktmo','observation_year'],keep=False)
    _adr_m['target_year']=_adr_m.observation_year.astype('Int64')
    _adr_m['origin_year']=_adr_m.target_year-CFG.HORIZON
    _adr_m['feature_year']=_adr_m.origin_year-CFG.FEATURE_PUBLICATION_LAG_YEARS
    _adr_lags=_adr_m[['oktmo','observation_year','actual','population']].rename(columns={
        'observation_year':'feature_year','actual':'lag_migration_rate','population':'lag_population'})
    _adr_m=_adr_m.merge(_adr_lags,on=['oktmo','feature_year'],how='inner',validate='many_to_one')
    _adr_sites_lag=_adr_site_counts.rename(columns={'observation_year':'feature_year'})
    _adr_m=_adr_m.merge(_adr_sites_lag,on=['oktmo','feature_year'],how='inner',validate='many_to_one')
    _adr_m=_adr_m.dropna(subset=['lag_migration_rate','lag_population','observed_geocoded_spo_entities'])
    _adr_m['entity_id']='MUNICIPAL|'+_adr_m.oktmo.astype(str)
    _adr_m['sector']='SPO_SPATIAL_PROXY'
    _adr_m['target_id']='D01_OBSERVED_MUNICIPAL_MIGRATION_RATE'
    _adr_m['information_cutoff_year']=_adr_m.origin_year
    _adr_m['target_available_year']=_adr_m.target_year+CFG.LABEL_PUBLICATION_LAG_YEARS
    _adr_m['feature_available_year']=_adr_m.feature_year+CFG.FEATURE_PUBLICATION_LAG_YEARS
    _adr_m['prior_value']=_adr_m.lag_migration_rate
    _adr_m['source_vintage']=INDEX['created_at']
    qa_table('adr_ml57_spo_proxy_common_support',_adr_m,allow_empty=True)
    _adr_years57=sorted(_adr_m.target_year.dropna().astype(int).unique()) if len(_adr_m) else []
    if len(_adr_years57)>=2:
        _adr_testyear=max(_adr_years57)
        _adr_origin=_adr_testyear-CFG.HORIZON
        _adr_tr=_adr_m.loc[(_adr_m.target_year<_adr_testyear)&
             (_adr_m.target_available_year<=_adr_origin)].copy()
        _adr_te=_adr_m.loc[_adr_m.target_year.eq(_adr_testyear)].copy()
        if len(_adr_tr)>=80 and len(_adr_te)>=CFG.MIN_MUNICIPALITIES_E0E1 and _adr_tr.observed_geocoded_spo_entities.nunique()>1:
            _adr_results=[]
            for _variant,_xcols in [('E0',['lag_migration_rate','lag_population']),
                                     ('E1',['lag_migration_rate','lag_population','observed_geocoded_spo_entities'])]:
                _fit=Pipeline([('imputer',SimpleImputer(strategy='median')),
                   ('scale',StandardScaler()),('ridge',Ridge(alpha=20.0))])
                _fit.fit(_adr_tr[_xcols].to_numpy(float),_adr_tr.actual.to_numpy(float))
                _pred=_fit.predict(_adr_te[_xcols].to_numpy(float))
                _saved=ROOT/'models'/f'adr_ml57_spo_geocode_{_variant}.joblib'
                if _saved.name in MODEL_FILES:raise RuntimeError('ADR ML57 artifact name collision')
                joblib.dump({'model':_fit,'features':_xcols,'scientific_scope':'SPO_GEO_PROXY_ASSOCIATION'},_saved,compress=3)
                if not np.allclose(joblib.load(_saved)['model'].predict(_adr_te[_xcols].to_numpy(float)),_pred):
                    raise RuntimeError('ADR ML57 model reload mismatch')
                MODEL_FILES[_saved.name]=str(_saved)
                MODEL_RECORDS.append({'module':'ML-5.7','sector':'SPO_SPATIAL_PROXY',
                    'model':'adr_'+_variant,'protocol':'SPATIAL_PROXY_TEMPORAL_ASOF_NOT_CAUSAL',
                    'seed':-1,'path':_saved.name,'file_sha256':hash_file(_saved),'feature_num':_xcols,
                    'feature_cat':[],'training_rows':len(_adr_tr),'test_rows':len(_adr_te),'reloaded':True})
                _pf=pd.DataFrame({'entity_id':_adr_te.entity_id.to_numpy(),
                     'origin_year':_adr_te.origin_year.to_numpy(),'target_year':_adr_te.target_year.to_numpy(),
                     'information_cutoff_year':_adr_te.information_cutoff_year.to_numpy(),
                     'feature_available_year':_adr_te.feature_available_year.to_numpy(),
                     'target_available_year':_adr_te.target_available_year.to_numpy(),
                     'sector':'SPO_SPATIAL_PROXY','target_id':'D01_OBSERVED_MUNICIPAL_MIGRATION_RATE',
                     'actual':_adr_te.actual.to_numpy()})
                _predrows=predictions_frame(_pf,'ML-5.7','adr_'+_variant,_pred,protocol='SPATIAL_PROXY_TEMPORAL_ASOF_NOT_CAUSAL')
                _predrows['geography_grain']='MUNICIPALITY';record_prediction(_predrows)
                _met=effective_mae(_adr_te.actual,_pred)
                METRICS.append({'module':'ML-5.7','sector':'SPO_SPATIAL_PROXY','model':'adr_'+_variant,
                    'protocol':'SPATIAL_PROXY_TEMPORAL_ASOF_NOT_CAUSAL','seed':-1,'target_year':_adr_testyear,**_met})
                _adr_results.append({'variant':_variant,'n_train':len(_adr_tr),'n_test':len(_adr_te),**_met})
            qa_table('adr_ml57_spatial_e0e1_metrics',pd.DataFrame(_adr_results),allow_empty=True)
            ADR_ML57_GEO_STATUS={'status':'TESTED_ASSOCIATION_NOT_CONFIRMED_CAMPUS',
                'n_train':len(_adr_tr),'n_test':len(_adr_te),'n_spo_location_proxies':len(ADR_GEO_VERIFIED),
                'MAE_E0':_adr_results[0]['MAE'],'MAE_E1':_adr_results[1]['MAE']}
            adr_record_exp('ML-5.7','GEO_RECONSTRUCTED_SPO_E0_E1','MODEL_COMPLETED',len(_adr_tr),len(_adr_te),
                 'Same-support as-of tests; SPO site count only a proximity proxy',
                 SPO_ORG['asset_id']+';'+SPATIAL['asset_id']+';'+MUN['asset_id'],
                 'Predictive association of reported proxy, not causal effect or confirmed campus exposure')
        else:
            ADR_ML57_GEO_STATUS={'status':'SKIPPED_INSUFFICIENT_TIME_OR_SPATIAL_SUPPORT',
              'n_train':len(_adr_tr),'n_test':len(_adr_te)}
    if ADR_ML57_GEO_STATUS.get('status','').startswith('SKIPPED'):
        adr_record_exp('ML-5.7','GEO_RECONSTRUCTED_SPO_E0_E1',ADR_ML57_GEO_STATUS['status'],
           reason='No as-of historical support for unique high-confidence geocoded SPO entities',
           source_ids=SPO_ORG['asset_id']+';'+SPATIAL['asset_id']+';'+MUN['asset_id'])
else:
    adr_record_exp('ML-5.7','GEO_RECONSTRUCTED_SPO_E0_E1','SKIPPED_NO_MATCHED_SPATIAL_SITES',
       reason='No high-confidence spatial links in the actual source; no fabricated campus locations',
       source_ids=SPO_ORG['asset_id']+';'+SPATIAL['asset_id'])

qa_table('adr_ml57_geographic_execution',pd.DataFrame([ADR_ML57_GEO_STATUS]),allow_empty=True)
qa_table('adr_all_feasible_experiments',pd.DataFrame(ADR_EXPERIMENTS),allow_empty=True)
print('V7.1 reconstruction experiments:',len(ADR_EXPERIMENTS),'| new model procedures:',
      sum(x['status']=='MODEL_COMPLETED' for x in ADR_EXPERIMENTS), '| retained all V7 modules')


## 9Ж. V7.4 · предобработка семи исследовательских направлений

Проверки выполняются по наблюдаемым значениям. Неоднозначные соответствия не включаются в целевые переменные.

In [ ]:
# ML-05 V7.4.0 · систематическая верификация предобработки: ML-5.1–ML-5.7.
# Работает ТОЛЬКО с уже прочитанными первичными показателями; нет новых/синтетических наблюдений.
progress_event('EXTENDED_MODULES','V7.4: 7 модулей · семантика, годы, классификаторы, география, сопоставимость',announce=True)
V72_RECONSTRUCTION=[]
def v72_audit(module,experiment,status,rows=0,source='',note='',evidence=''):
    V72_RECONSTRUCTION.append({'module':module,'experiment':experiment,'status':status,
        'observed_rows':int(rows),'primary_source':str(source),'rule':str(evidence),'interpretation':str(note)})

# ML-5.1: изменения покрытия интерпретируются отдельно от изменения численности.
V72_OD_COVERAGE=pd.DataFrame()
if len(ADR_OD_PAIRWISE):
    _p=ADR_OD_PAIRWISE.copy().sort_values(['prior_year','current_year'])
    _p['joint_coverage_min']=_p[['coverage_of_prior','coverage_of_current']].min(axis=1)
    _p['ratio_on_common_edges']=np.where(_p.sum_prior_common.gt(0),
        _p.sum_current_common/_p.sum_prior_common,np.nan)
    _p['coverage_change_factor']=np.where(_p.prior_observed_edges.gt(0),
        _p.current_observed_edges/_p.prior_observed_edges,np.nan)
    _p['coverage_shift_flag']=(_p.joint_coverage_min.lt(.8)|
        _p.coverage_change_factor.lt(.75)|_p.coverage_change_factor.gt(1.33))
    _p['historic_2023_incomplete']=_p.current_year.eq(2023)|_p.prior_year.eq(2023)
    _p['missing_edges_imputed_zero']=False
    _p['sum_claim']='COMMON_REPORTED_OD_EDGES_ONLY'
    V72_OD_COVERAGE=qa_table('v72_od_year_pair_coverage_decomposition',_p,allow_empty=True)
    v72_audit('ML-5.1','E01_COVERAGE_CHANGE_SEPARATION','PASS',len(_p),E01['asset_id'],
              'Оценка динамики только на пересечении зарегистрированных направлений',
              'no imputation; common-support sum; explicit coverage flags')
else:
    v72_audit('ML-5.1','E01_COVERAGE_CHANGE_SEPARATION','SKIPPED_NO_ADJACENT_YEARS',source=E01['asset_id'])

# ML-5.2: решение о пригодности индикатора по наблюдаемым типам и уникальности,
# а не по одному совпадению русского названия.
V72_ADMISSION_DECISION=pd.DataFrame()
if len(ADR_ADMISSION_CANDIDATES):
    _a=ADR_ADMISSION_CANDIDATES.copy()
    _approval=_a['user_or_method_approval'].fillna(False).astype(bool)
    _semantic=_a['strict_source_label_supported'].fillna(False).astype(bool)
    _a['source_semantics_verified']=_semantic
    _a['methodology_approved']=_approval
    _a['allow_model_observed_admissions']=_semantic&_approval
    _a['decision']=np.select([_a.allow_model_observed_admissions,_semantic],
                            ['SOURCE_AND_APPROVAL_PASS','SEMANTIC_CANDIDATE_NOT_METHOD_APPROVED'],
                            default='REJECT_AMBIGUOUS_UNIT_LABEL_OR_VALUE')
    _a['no_sum_of_programme_subtotals']=True
    V72_ADMISSION_DECISION=qa_table('v72_org_admission_indicator_decision',_a,allow_empty=True)
    v72_audit('ML-5.2','E02_ADMISSION_PROOF','PASS_WITH_EXCLUSIONS',len(_a),E02['asset_id'],
        'Единица измерения, значение, утверждение определения и уникальная организационно-годовая ячейка',
        'No approval means no admission target, even for semantically plausible labels')
else:
    v72_audit('ML-5.2','E02_ADMISSION_PROOF','SKIPPED_NO_MATCHING_LABELS',source=E02['asset_id'])

# ML-5.3: региональный отток не смешивается с населением или всей совокупностью направлений.
V72_OUTFLOW_BALANCED=pd.DataFrame()
if len(ADR_OD_BALANCED):
    _bo=ADR_OD_BALANCED.groupby(['observation_year','origin_region'],as_index=False).agg(
        observed_common_edge_outflow=('observed_value','sum'),observed_common_destinations=('destination_region','nunique'))
    _bo['definition']='SAME_REPORTED_EDGES_IN_EVERY_SOURCE_YEAR_NOT_POPULATION_OUTMIGRATION'
    V72_OUTFLOW_BALANCED=qa_table('v72_origin_common_support_outflow',_bo,
                                  ['observation_year','origin_region'],allow_empty=True)
    v72_audit('ML-5.3','REGIONAL_OUTFLOW_CONSTANT_EDGES','PASS',len(_bo),E01['asset_id'],
        'Суммируются лишь наблюдавшиеся ежегодно направления', 'Do not infer total migration from E01')
else:v72_audit('ML-5.3','REGIONAL_OUTFLOW_CONSTANT_EDGES','SKIPPED_NO_SHARED_EDGES',source=E01['asset_id'])

# ML-5.4: аудит смены классификатора по опубликованным кодам — без неутверждённых синонимов.
V72_SPECIALTY_BRIDGE=pd.DataFrame()
if len(ML54_HISTORY):
    _hist=ML54_HISTORY[['observation_year','specialty_code','indicator_name']].copy()
    _hist['year']=pd.to_numeric(_hist.observation_year,errors='coerce').astype('Int64')
    _hist['raw_code']=_hist.specialty_code.astype('string').str.strip().str.replace(' ','',regex=False)
    _hist['standardized_code']=(_hist.raw_code.str.replace(r'[^0-9A-Za-z]','',regex=True)
                                .str.upper().replace('',pd.NA))
    _hist['code_format_valid']=_hist.standardized_code.str.fullmatch(r'[0-9A-Z]{4,12}',na=False)
    _hist['source_indicator']=_hist.indicator_name.astype('string')
    _group=_hist.dropna(subset=['year','standardized_code']).groupby(
        ['standardized_code','year'],as_index=False).agg(
            n_source_indicator_names=('source_indicator','nunique'),n_source_rows=('raw_code','size'),
            n_raw_notations=('raw_code','nunique'),valid_format=('code_format_valid','all'))
    _group['classification_bridge_status']=np.where(_group.valid_format,
        'NORMALIZED_EXACT_SPELLING_ONLY_NOT_CROSS_STANDARD','UNVERIFIED_CODE_FORMAT')
    _group['source_table_scope']=('FULL_SOURCE_AGGREGATE' if 'ml54_specialty_history' not in PANEL_FILES
                                  else 'AVAILABLE_IN_MEMORY_PREVIEW_ONLY')
    V72_SPECIALTY_BRIDGE=qa_table('v72_specialty_code_observed_crosswalk',_group,allow_empty=True)
    v72_audit('ML-5.4','SPECIALTY_CLASSIFIER_SPELLING_AUDIT','PASS',len(_group),VPO_211['asset_id'],
        'Меняются только написания одного кода; разные коды не объявляются одной специальностью',
        'Separate historical code coverage from claimed student admissions')
else:v72_audit('ML-5.4','SPECIALTY_CLASSIFIER_SPELLING_AUDIT','SKIPPED_NO_CODE_HISTORY',source=VPO_211['asset_id'])

# ML-5.5: статистика только внутри исходных E04/E05 семей; несовпадающие срезы не складываются.
V72_GRADUATE_FAMILY=pd.DataFrame()
if len(GRAD_DISPARITIES):
    _g=GRAD_DISPARITIES.copy()
    _k='asset_id' if 'asset_id' in _g else next((c for c in ('source_id','family') if c in _g),None)
    if _k:
        _measures=[c for c in _g if pd.api.types.is_numeric_dtype(_g[c]) and
                   not any(x in c.lower() for x in ('year','rows','count','n_source'))]
        _out=[]
        for _fam,_part in _g.groupby(_k,dropna=False,sort=False):
            _item={'source_family':str(_fam),'observed_aggregate_strata':len(_part),
                'summing_families_forbidden':True,'workplace_geography_observed':False}
            for _col in _measures[:6]:
                _valid=pd.to_numeric(_part[_col],errors='coerce').dropna()
                _item[_col+'_median_of_observed_strata']=float(_valid.median()) if len(_valid) else np.nan
            _out.append(_item)
        V72_GRADUATE_FAMILY=qa_table('v72_graduate_family_independent_descriptives',
            pd.DataFrame(_out),['source_family'],allow_empty=True)
        v72_audit('ML-5.5','E04_E05_NONADDITIVE_GROUPS','PASS',len(V72_GRADUATE_FAMILY),
             ';'.join(a['asset_id'] for a in source_asset(segment='05')),
             'Медианы агрегированных срезов, отдельно для каждой первичной семьи',
             'No inference of workplace region or graduate retention')
    else:v72_audit('ML-5.5','E04_E05_NONADDITIVE_GROUPS','SKIPPED_NO_SOURCE_FAMILY_FIELD')
else:v72_audit('ML-5.5','E04_E05_NONADDITIVE_GROUPS','SKIPPED_NO_GRADUATE_SUMMARY')

# ML-5.6: оценка сопоставимой сети ограничена общими наблюдавшимися узлами и рёбрами.
V72_NETWORK_STABLE=pd.DataFrame()
if len(NETWORK_EDGES):
    _ne=pd.DataFrame(NETWORK_EDGES)
    _years=sorted(_ne.year.dropna().unique())
    _rows=[]
    for _a,_b in zip(_years[:-1],_years[1:]):
        if int(_b)!=int(_a)+1:continue
        _aedge=_ne.loc[_ne.year.eq(_a),['origin','destination','value']].copy()
        _bedge=_ne.loc[_ne.year.eq(_b),['origin','destination','value']].copy()
        if _aedge.duplicated(['origin','destination']).any() or _bedge.duplicated(['origin','destination']).any():
            raise RuntimeError('V72 NETWORK: повторное направление внутри наблюдаемого года')
        _anodes=set(_aedge.origin.astype(str))|set(_aedge.destination.astype(str))
        _bnodes=set(_bedge.origin.astype(str))|set(_bedge.destination.astype(str))
        _shared_nodes=_anodes&_bnodes
        _left=_aedge[_aedge.origin.astype(str).isin(_shared_nodes)&_aedge.destination.astype(str).isin(_shared_nodes)]
        _right=_bedge[_bedge.origin.astype(str).isin(_shared_nodes)&_bedge.destination.astype(str).isin(_shared_nodes)]
        _joined=_left.merge(_right,on=['origin','destination'],suffixes=('_prev','_curr'),validate='one_to_one')
        _rows.append({'prior_year':int(_a),'current_year':int(_b),'n_shared_nodes':len(_shared_nodes),
             'edges_on_shared_nodes_prior':len(_left),'edges_on_shared_nodes_current':len(_right),
             'same_observed_edges':len(_joined),
             'prior_strength_common':float(_joined.value_prev.sum()),
             'current_strength_common':float(_joined.value_curr.sum()),
             'interpretation':'SAME_NODES_AND_EDGES_ONLY_NO_UNOBSERVED_EDGE_ZERO_FILL'})
    V72_NETWORK_STABLE=qa_table('v72_network_same_node_edge_dynamics',pd.DataFrame(_rows),allow_empty=True)
    v72_audit('ML-5.6','NETWORK_SAME_NODE_EDGE_YEAR_PAIRS',
         'PASS' if len(_rows) else 'SKIPPED_NO_ADJACENT_YEARS',len(_rows),E01['asset_id'],
         'Изменение по неизменной наблюдаемой топологии; не общая динамика сети')
else:v72_audit('ML-5.6','NETWORK_SAME_NODE_EDGE_YEAR_PAIRS','SKIPPED_NO_EDGES',source=E01['asset_id'])

# ML-5.7: адрес/координаты рядом с поселением — ещё не подтверждённый адрес кампуса;
# проверяем согласованность исторических прокси и не выдаём их за первичные ОКТМО.
V72_SPATIAL_CONFIDENCE=pd.DataFrame()
if len(ADR_GEO_SPO):
    _geo=ADR_GEO_SPO.copy()
    _geo['municipality_proxy_only']=True
    _geo['confirmed_campus_location']=False
    _geo['spatial_confidence_band']=np.select([
        _geo.spatially_unambiguous.fillna(False),
        _geo.city_name_consistent.fillna(False)],['PROXIMITY_HIGH_NOT_CERTIFIED','CITY_ONLY_UNVERIFIED'],
        default='UNRESOLVED')
    _valid=_geo[_geo.spatially_unambiguous.fillna(False)].copy()
    _c=_valid.groupby('entity_id').candidate_municipal_oktmo.nunique() if len(_valid) else pd.Series(dtype=int)
    _conflict=set(_c[_c>1].index)
    _geo['multi_year_oktmo_conflict']=_geo.entity_id.isin(_conflict)
    _geo.loc[_geo.multi_year_oktmo_conflict,'spatial_confidence_band']='INCONSISTENT_OVER_TIME'
    _geo['may_enter_official_campus_target']=False
    V72_SPATIAL_CONFIDENCE=qa_table('v72_spo_municipal_proxy_confidence',_geo,allow_empty=True)
    if _conflict:
        QUARANTINE.extend(_geo[_geo.multi_year_oktmo_conflict].head(100).assign(
             source='ML57_SPATIAL_MULTIPERIOD_CONFLICT').to_dict('records'))
    v72_audit('ML-5.7','D04_SPO_SPATIAL_PROXY_STABILITY','PASS_WITH_QUARANTINE',len(_geo),
        SPO_ORG['asset_id']+';'+SPATIAL['asset_id'],
        'Проверено постоянство прокси-ОКТМО; подтверждённых кампусов эта процедура не создаёт',
        'NEVER treat nearest settlement as verified study campus')
else:v72_audit('ML-5.7','D04_SPO_SPATIAL_PROXY_STABILITY','SKIPPED_NO_GEO_MATCHES',source=SPATIAL['asset_id'])

V72_RECON_DF=qa_table('v72_all_seven_reconstruction_evidence',pd.DataFrame(V72_RECONSTRUCTION),
                      ['module','experiment'])
print('V7.4 · семь модулей: проверено',len(V72_RECON_DF),'подисследований,',
      sum(str(x['status']).startswith('PASS') for x in V72_RECONSTRUCTION),'с подтверждёнными результатами')

mark_stage('EXTENDED_MODULES')


## 10. Независимые метрики, canary, статистика
Метрики пересчитываются по прогнозным строкам. При отсутствии законного DEV сравнения не становятся подтверждающими. Отрицательные результаты сохраняются.

In [ ]:
progress_event('STATISTICS','Независимые метрики, канарейки, проверки гипотез')
# Независимые сквозные проверки: источники/будущее/статистика/содержательные выводы.
CANARIES=[]

def test_future_mutation(raw,keys,value_col,module):
    if raw.empty or 'observation_year' not in raw or value_col not in raw:
        return {'module':module,'status':'NOT_APPLICABLE','reason':'Нет валидной временной панели'}
    r=raw[keys+['observation_year',value_col]].dropna(subset=['observation_year']).copy()
    if r.empty or r.observation_year.nunique()<2:return {'module':module,'status':'NOT_APPLICABLE','reason':'Мало лет'}
    r=r.drop_duplicates(keys+['observation_year'])
    if r.empty:return {'module':module,'status':'NOT_APPLICABLE','reason':'Нет уникальных ключей'}
    p1=historical_pair_panel(r,keys,value_col)
    mx=int(r.observation_year.max());r2=r.copy()
    r2.loc[r2.observation_year.eq(mx),value_col]=number(r2.loc[r2.observation_year.eq(mx),value_col]).fillna(0)+10**8
    p2=historical_pair_panel(r2,keys,value_col)
    # Проверяется строгое отсутствие будущих лагов; отдельный тест экономических и сетевых join — ниже.
    if len(p1) and (p1.feature_available_year>p1.information_cutoff_year).any():
        fail('CANARY','Исходная панель содержит будущие признаки: '+module)
    # До мутации будущего итоговые X прежних origins должны быть идентичны.
    common=[c for c in keys+['target_year','prior_value','prior2_value','historical_year'] if c in p1 and c in p2]
    early1=p1[p1.target_year<mx][common].sort_values(keys+['target_year']).reset_index(drop=True)
    early2=p2[p2.target_year<mx][common].sort_values(keys+['target_year']).reset_index(drop=True)
    if not early1.equals(early2):fail('CANARY','Мутация будущего изменила прошлые признаки: '+module)
    return {'module':module,'status':'PASS','n_untouched':len(early1),'mutated_year':mx}

CANARIES.append(test_future_mutation(OD_INTER,['origin_region','destination_region'],'observed_value','ML-5.1'))
CANARIES.append(test_future_mutation(OUTFLOW,['origin_region'],'observed_interregional_routes','ML-5.3'))
if len(ORG_YEAR):CANARIES.append(test_future_mutation(ORG_YEAR,['sector','entity_id'],'actual','ML-5.2'))
if len(PROGRAM_PROFILE):
    prog_test=PROGRAM_PROFILE.drop_duplicates(['specialty_code','indicator_name','observation_year'])
    CANARIES.append(test_future_mutation(prog_test,['specialty_code','indicator_name'],'observed_value','ML-5.4'))
if len(GRADUATES):
    CANARIES.append({'module':'ML-5.5','status':'DESCRIPTIVE_ONLY',
      'reason':'Единственная дата измерения занятости; модель предсказаний не запускалась'})
if len(MUNI):CANARIES.append(test_future_mutation(MUNI,['oktmo'],'migration_rate','ML-5.7'))
CANARIES.append({'module':'ECONOMIC_CONTEXT','status':'STATIC_OR_ASOF_AUDIT',
 'reason':'Использована только экономическая панель observation_year=historical_year; future rows not joined'})
if len(OD_MODEL):
    # Отдельный тест исходных маргиналей: признаки прошлого года должны совпадать с исходными агрегатами.
    src_mass=OD_INTER.groupby(['observation_year','origin_region'],as_index=False).observed_value.sum()
    tmp=OD_MODEL[['origin_region','historical_year','origin_mass']].dropna().drop_duplicates()
    exp=src_mass.rename(columns={'observation_year':'historical_year','observed_value':'expected_mass'})
    comp=tmp.merge(exp,on=['historical_year','origin_region'],how='left',validate='many_to_one')
    if len(comp) and not np.allclose(comp.origin_mass,comp.expected_mass,equal_nan=True):
        fail('CANARY','Региональные маргинали не совпали с историческим исходным годом')
    CANARIES.append({'module':'OD_MARGINS','status':'PASS' if len(comp) else 'NOT_APPLICABLE',
       'verified_origin_year_margins':int(len(comp))})
qa_table('future_mutation_canaries',pd.DataFrame(CANARIES),allow_empty=True)

# Весь стек прогнозов, включая 50 сидов, объединяется НА ДИСКЕ.
PRED=pd.DataFrame();RECALCULATED=[];PRED_STORAGE_FILE=None
if PRED_PARTS:
    progress_event('STATISTICS',f'Независимая проверка {PRED_TOTAL:,} строк из {len(PRED_PARTS)} файлов',announce=True)
    PRED_STORAGE_FILE=ROOT/'derived'/'education_predictions.parquet'
    # read_parquet(list) обрабатывается потоком в DuckDB, не через pandas.concat.
    with duck_connect() as con:
        _files='['+','.join("'"+str(p).replace("'","''")+"'" for p in PRED_PARTS)+']'
        _q=f"SELECT * FROM read_parquet({_files},union_by_name=true)"
        _fp=str(PRED_STORAGE_FILE).replace("'","''")
        con.execute(f"COPY ({_q}) TO '{_fp}' (FORMAT PARQUET, COMPRESSION ZSTD)")
        _num=con.execute('SELECT COUNT(*) FROM read_parquet(?)',[str(PRED_STORAGE_FILE)]).fetchone()[0]
        if _num!=PRED_TOTAL:fail('REPRO',f'Прогнозов в Parquet {_num} вместо {PRED_TOTAL}')
        _invalid=con.execute('''SELECT COUNT(*) FROM read_parquet(?)
            WHERE predicted IS NULL OR actual IS NULL OR NOT isfinite(predicted) OR NOT isfinite(actual)
            OR feature_available_year>information_cutoff_year''',[str(PRED_STORAGE_FILE)]).fetchone()[0]
        if _invalid:fail('REPRO',f'Невалидных прогнозов: {_invalid}')
        _duplicates=con.execute('''SELECT COUNT(*) FROM (
            SELECT module,sector,entity_id,target_year,model,seed,split_id,COUNT(*) AS n
            FROM read_parquet(?) GROUP BY ALL HAVING COUNT(*)>1)''',[str(PRED_STORAGE_FILE)]).fetchone()[0]
        if _duplicates:fail('SUPPORT',f'Повторные натуральные ключи прогнозов: {_duplicates}')
        # Точные MAE/RMSE/WAPE по ВСЕМ строкам всех файлов.
        RECALCULATED=con.execute('''SELECT module,sector,model,seed,target_year,split_id,
            AVG(ABS(actual-predicted)) AS MAE,
            SQRT(AVG(POWER(actual-predicted,2))) AS RMSE,
            SUM(ABS(actual-predicted))/NULLIF(SUM(ABS(actual)),0) AS WAPE,
            COUNT(*)::BIGINT AS n
            FROM read_parquet(?) GROUP BY 1,2,3,4,5,6''',[str(PRED_STORAGE_FILE)]).df().to_dict('records')
        PRED=con.execute('SELECT * FROM read_parquet(?) LIMIT 2000',[str(PRED_STORAGE_FILE)]).df()
    PANEL_FILES['education_predictions']=PRED_STORAGE_FILE
    final=pd.DataFrame(RECALCULATED)
    for r in final.itertuples(index=False):
        original=[m for m in METRICS if m['module']==r.module and m.get('sector')==r.sector
                  and m['model']==r.model and m['seed']==r.seed and m['target_year']==r.target_year and m.get('protocol')==r.split_id]
        if not original:fail('REPRO','Нет строки метрики для '+r.module+' '+r.model)
        for m in original:
            if not np.isclose(m['MAE'],r.MAE,atol=1e-7,rtol=1e-7):fail('REPRO','MAE не совпал: '+r.module+' '+r.model)
qa_table('metrics_recomputed_from_predictions',pd.DataFrame(RECALCULATED),allow_empty=True)

# Предрегистрация Holm: только подтверждающие сравнения, для которых есть валидный p.
CONFIRM=pd.DataFrame(CONFIRMATORY)
if not CONFIRM.empty:
    valid=CONFIRM.p_value.notna()&np.isfinite(CONFIRM.p_value)
    CONFIRM['p_holm']=np.nan
    if valid.any():
        CONFIRM.loc[valid,'p_holm']=multipletests(CONFIRM.loc[valid,'p_value'].to_numpy(float),
                                                method='holm')[1]
    CONFIRM['interpretation']=np.where(CONFIRM.delta_mae.lt(0),'LOWER_MAE_ON_SELECTED_COMMON_SUPPORT',
                                      np.where(CONFIRM.delta_mae.gt(0),'HIGHER_MAE_ON_SELECTED_COMMON_SUPPORT','EQUAL_MAE'))
    CONFIRM['historical_vintage_validated']=False
    CONFIRM['claim_level']='EXPLORATORY_RETROSPECTIVE_SOURCE_2026'

qa_table('confirmatory_comparisons',CONFIRM,allow_empty=True)

# Доказательный реестр 7 гипотез: отдельно результат и предел идентификации.
for i in range(1,8):
    module=f'ML-5.{i}'
    status=MODULES[module]
    subset=CONFIRM[CONFIRM.module.eq(module)] if len(CONFIRM) else pd.DataFrame()
    if len(subset):
        r=subset.iloc[0]
        if r.choice_basis!='DEV_ASOF':
            conclusion='Подтверждающий отбор невозможен: DEV-метки недоступны на момент TEST; сравнения моделей разведочные'
            metric=f'baseline_MAE={r.baseline_mae:.5g}; confirmatory_delta=NOT_APPLICABLE'
        else:
            conclusion=('выбранная на DEV модель не превзошла инерцию на ретроспективном тесте' if r.delta_mae>=0
                        else 'в ретроспективном тесте MAE ниже инерции; исторические версии данных не подтверждены')
            metric=f'baseline_MAE={r.baseline_mae:.5g}; selected_MAE={r.selected_mae:.5g}; delta={r.delta_mae:+.5g}'
    else:conclusion=status['reason'];metric='Нет корректного независимого прогнозного теста'
    HYPOTHESES.append({'hypothesis':f'H5.{i}','module':module,'status':status['status'],
        'metric_or_basis':metric,'scientific_conclusion':conclusion,
        'source_observability':('E01_schooling_to_study_NOT_residence' if i in (1,3,6)
             else 'source_specific_unverified_retention' if i==5 else 'source_specific'),
        'causal_identification':'NOT_ESTABLISHED'})
qa_table('seven_hypotheses',pd.DataFrame(HYPOTHESES),['hypothesis'])
qa_table('split_audit',pd.DataFrame(SPLIT_AUDIT),allow_empty=True)
qa_table('model_metrics',pd.DataFrame(METRICS),allow_empty=True)
qa_table('model_registry',pd.DataFrame(MODEL_RECORDS),allow_empty=True)

# Статусы: техническая полнота != научное подтверждение семи гипотез.
executed_all=all(m['status']!='NOT_STARTED' for m in MODULES.values())
failed_modules=[k for k,m in MODULES.items() if m['status']=='FAILED']
research_full=all(m['status']=='TESTED_FULL' for m in MODULES.values()) and OD_SEMANTIC['yearly_count_unit_confirmed']
STATE['technical']='PASS_ASSEMBLED' if executed_all and not failed_modules else 'FAIL'
STATE['scientific']='FULL_RESEARCH_PASS' if research_full and CFG.MODE=='FULL' else (
    'FULL_EXECUTION_WITH_LIMITATIONS' if CFG.MODE=='FULL' and executed_all and not failed_modules else 'PILOT_OR_PARTIAL')
STATE['overall']=STATE['scientific']

print('Technical / scientific statuses:',STATE['technical'],STATE['scientific'])
print('Confirmed comparisons:',len(CONFIRM),'| exported predictions:',PRED_TOTAL)

# Финальная отметка этапа перенесена после всех его подмодулей


## 10А. Парные сравнения и проверка завершённости программы
Разделение вычислительной полноты и научной идентифицируемости; отсутствующие измерения не фабрикуются.

In [ ]:
# V4 · audit model completion independently of the notebook's technical PASS.
progress_event('STATISTICS','V4: численные сравнения на общих ключах, матрица научной полноты',announce=True)
V4_PAIRED_COMPARISONS=[]
if PRED_STORAGE_FILE and Path(PRED_STORAGE_FILE).is_file():
    with duckdb.connect(':memory:') as _db:
        _db.execute("SET memory_limit='1536MB'");_db.execute('SET threads=1')
        _unique=_db.execute('''SELECT DISTINCT module,sector,split_id,model FROM read_parquet(?) ORDER BY 1,2,3,4''',
                           [str(PRED_STORAGE_FILE)]).df()
        _combos=_unique[~_unique.model.isin(['persistence','rolling_mean_2','damped_trend','catboost'])].copy()
        for _rec in _combos.itertuples(index=False):
            _pred=_db.execute('''WITH b AS (SELECT entity_id,target_year,actual,predicted baseline,
                     origin_region,destination_region,parent_group_id FROM read_parquet(?)
                  WHERE module=? AND sector=? AND split_id=? AND model='persistence' AND seed=-1),
                 v AS (SELECT entity_id,target_year,actual,predicted as candidate FROM read_parquet(?)
                  WHERE module=? AND sector=? AND split_id=? AND model=?)
                  SELECT b.*,v.candidate FROM b JOIN v USING(entity_id,target_year)
                  WHERE ABS(b.actual-v.actual)<1e-7''',
               [str(PRED_STORAGE_FILE),_rec.module,_rec.sector,_rec.split_id,
                str(PRED_STORAGE_FILE),_rec.module,_rec.sector,_rec.split_id,_rec.model]).df()
            if not len(_pred):continue
            _a=np.abs(_pred.actual.to_numpy(float)-_pred.baseline.to_numpy(float))
            _v=np.abs(_pred.actual.to_numpy(float)-_pred.candidate.to_numpy(float))
            if _rec.module=='ML-5.1' and {'origin_region','destination_region'}.issubset(_pred):
                _g1=_pred.origin_region.fillna('__UNK__').astype(str)
                _g2=_pred.destination_region.fillna('__UNK__').astype(str)
                _ci=two_way_mae_bootstrap(_a,_v,_g1,_g2,reps=CFG.BOOTSTRAP_REPS,seed=CFG.RANDOM_SEED)
            else:
                _g1=(_pred.parent_group_id.fillna(_pred.entity_id).astype(str) if
                     'parent_group_id' in _pred else _pred.entity_id.astype(str))
                # Only one test target_year; two-way geographic-year bootstrap not identified.
                _unique_groups=_g1.nunique()
                if _unique_groups>=8:
                    _rng=np.random.default_rng(CFG.RANDOM_SEED)
                    _groups=_g1.to_numpy();_base=np.asarray(_v-_a,float)
                    _levels=np.unique(_groups);_boot=[]
                    for _ in range(CFG.BOOTSTRAP_REPS):
                        _mult=Counter(_rng.choice(_levels,len(_levels),replace=True))
                        _weights=np.fromiter((_mult[v] for v in _groups),dtype=float,count=len(_groups))
                        if _weights.sum()>0:_boot.append(float(np.average(_base,weights=_weights)))
                    _vals=np.asarray(_boot)
                    _ci={'delta_MAE':float(np.mean(_base)),'ci_low':float(np.quantile(_vals,.025)) if len(_vals)>20 else np.nan,
                         'ci_high':float(np.quantile(_vals,.975)) if len(_vals)>20 else np.nan,
                         'p_value':min(1.0,2*min((np.count_nonzero(_vals<=0)+1)/(len(_vals)+1),
                            (np.count_nonzero(_vals>=0)+1)/(len(_vals)+1))) if len(_vals)>20 else np.nan,
                         'clusters_a':_unique_groups,'clusters_b':None}
                else:_ci={'delta_MAE':float(np.mean(_v-_a)),'ci_low':np.nan,'ci_high':np.nan,
                           'p_value':np.nan,'clusters_a':_unique_groups,'clusters_b':None}
            V4_PAIRED_COMPARISONS.append({'module':_rec.module,'sector':_rec.sector,
                'split_id':_rec.split_id,'model':_rec.model,'n_common_keys':len(_pred),
                'persistence_MAE':float(_a.mean()),'candidate_MAE':float(_v.mean()),**_ci,
                'inference_type':'EXPLORATORY_PAIRED_RETROSPECTIVE_NOT_CONFIRMATORY',
                'historical_source_vintage_at_prediction':'NOT_ARCHIVED'})
_V4_BOOT=pd.DataFrame(V4_PAIRED_COMPARISONS)
if len(_V4_BOOT):
    _mask=_V4_BOOT['p_value'].notna()&np.isfinite(_V4_BOOT['p_value'])
    _V4_BOOT['p_holm_exploratory']=np.nan
    if _mask.any():_V4_BOOT.loc[_mask,'p_holm_exploratory']=multipletests(
        _V4_BOOT.loc[_mask,'p_value'].to_numpy(float),method='holm')[1]
qa_table('v4_exploratory_paired_comparisons',_V4_BOOT,allow_empty=True)

V4_METHOD_EXECUTIONS=[]
# Check every required model separately, not merely that a module produced one metric.
_v4_contract_base={'persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm','catboost_ensemble'}
_v4_expected={}
for _m in ('ML-5.1','ML-5.3','ML-5.4'):
    _actual={z.get('model') for z in METRICS if z.get('module')==_m and z.get('protocol')=='ROLLING_TIME'}
    if _actual:
        _req=_v4_contract_base.copy()
        # PPML/NB only if their observed-source integer semantics passed the count gate.
        _frame=OD_MODEL if _m=='ML-5.1' else OUTFLOW_MODEL if _m=='ML-5.3' else None
        _count_eligible=bool(_frame is not None and len(_frame) and
             number(_frame.actual).ge(0).all() and
             np.isclose(number(_frame.actual),np.rint(number(_frame.actual)),atol=1e-8,rtol=0).all())
        if _m=='ML-5.4':
            _count_eligible=MODULES[_m]['status'].startswith('TESTED')
        if _count_eligible:_req.update({'ppml','negbin'})
        _v4_expected[_m]=_req
if any(z.get('module')=='ML-5.2' and z.get('protocol')=='ALT_CONTINGENT_RETROSPECTIVE' for z in METRICS):
    _v4_expected['ML-5.2']={'persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm'}
if any(z.get('module')=='ML-5.7' and z.get('protocol')=='RETROSPECTIVE_PROXY_E0E1' for z in METRICS):
    _v4_expected['ML-5.7']={'E0','E1'}
for _module in ['ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7']:
    _r=[m for m in METRICS if m.get('module')==_module]
    _failed=[x for x in ISSUES if x.get('module')==_module and x.get('status') in
            ('MODEL_FAILED','FAMILY_PROFILE_FAILED','SOURCE_PROFILE_FAILURE','SPATIAL_MODEL_FAILURE')]
    _expected=_v4_expected.get(_module,set())
    _actually={z.get('model') for z in _r}
    _missing=sorted(_expected-_actually)
    if _missing:_failed.append({'reason':'Feasible models without metrics: '+','.join(_missing),
                               'status':'METHOD_NOT_EXECUTED'})
    if _module=='ML-5.5' and V4_GRADUATE_STATUS.get('status')=='REGIONAL_HOLDOUT_EMPLOYMENT_TEST_NOT_RETENTION':
        if len(PANELS.get('v4_ml55_crosssection_metrics',pd.DataFrame()))<4:
            _failed.append({'reason':'E04 cross-sectional baseline/Ridge/LightGBM/CatBoost incomplete'})
    if _module=='ML-5.6' and len(NETWORK) and not len(PANELS.get('v4_ml56_node_forecast',pd.DataFrame())):
        _failed.append({'reason':'Node centrality persistence test missing'})
    _feasible=bool(_r) or MODULES[_module].get('status','').startswith('DESCRIPTIVE')
    V4_METHOD_EXECUTIONS.append({'module':_module,'module_status':MODULES[_module].get('status'),
        'models_with_metrics':len(set(x.get('model') for x in _r)),
        'metric_rows':len(_r),'saved_model_artifacts':sum(m.get('module')==_module for m in MODEL_RECORDS),
        'prediction_rows_validated':bool(any(x.get('module')==_module for x in RECALCULATED)),
        'required_models':','.join(sorted(_expected)), 'missing_required_models':','.join(_missing),
        'computational_failures':len(_failed),
        'failure_reason':' | '.join(str(x.get('reason',''))[:120] for x in _failed[:3]),
        'target_is_scientifically_identified':bool(_module in ('ML-5.4','ML-5.6')),
        'historical_vintage_validated':False,
        'scientific_completion':'RETROSPECTIVE_OR_DESCRIPTIVE_WITH_GATES'})
qa_table('v4_methods_execution_ledger',pd.DataFrame(V4_METHOD_EXECUTIONS),['module'])
V4_COMPUTATIONAL_COMPLETE=bool(not V4_SOURCE_ERRORS and not any(x['computational_failures'] for x in V4_METHOD_EXECUTIONS))
V4_SCIENTIFIC_COMPLETE=bool(V4_COMPUTATIONAL_COMPLETE and
                           all(MODULES[m]['status']=='TESTED_FULL' for m in MODULES) and
                           OD_SEMANTIC.get('yearly_count_unit_confirmed',False) and
                           len(ORG_YEAR)>0)
STATE['scientific']='FULL_RESEARCH_PASS' if V4_SCIENTIFIC_COMPLETE else (
    'FULL_COMPUTATIONS_WITH_DOCUMENTED_DATA_LIMITS' if V4_COMPUTATIONAL_COMPLETE and CFG.MODE=='FULL'
     else 'FULL_PARTIAL_COMPUTATIONS_OR_MODEL_FAILURES')
STATE['overall']=STATE['scientific']
qa_table('v4_scientific_acceptance',pd.DataFrame([{'version':CFG.VERSION,'source_assets':len(ASSETS),
    'value_scans_passed':sum(x['content_status']=='FULL_VALUE_COVERAGE_PASS' for x in V4_SOURCE_PROFILES),
    'source_assets_not_in_model':sum(not x['main_prediction_input'] for x in _v4_roles),
    'seven_modules_reported':len(MODULES),'computational_failures':sum(x['computational_failures'] for x in V4_METHOD_EXECUTIONS),
    'full_computations_without_failures':V4_COMPUTATIONAL_COMPLETE,'all_seven_scientifically_identified':V4_SCIENTIFIC_COMPLETE,
    'scientific_status':STATE['scientific'],'human_approval_needed_for_admissions':len(ORG_YEAR)==0,
    'OD_2023_coverage_break_unresolved':bool(coverage_df.potential_break.fillna(False).any()),
    'historical_vintages_not_archived':True,'retention_unobservable_without_work_region':True}]))
print('V4 acceptability:',STATE['scientific'],'| calculations with actual errors:',not V4_COMPUTATIONAL_COMPLETE)

# Rolling-origin feasibility is measured year by year. Lack of archived source vintages forbids a confirmatory claim.
_v4_roll=[]
for _m,_f in [('ML-5.1',OD_MODEL),('ML-5.3',OUTFLOW_MODEL)]:
    if _f is None or not len(_f):continue
    _years=sorted(int(y) for y in _f.target_year.dropna().unique())
    for _y in _years[1:]:
        _cut=_y-CFG.HORIZON
        _train=_f[_f.target_year.lt(_y)&_f.target_available_year.le(_cut)]
        _test=_f[_f.target_year.eq(_y)]
        _gate=len(_train)>=(CFG.REGION_MIN_TRAIN_ROWS if _m=='ML-5.3' else CFG.MIN_TRAIN_ROWS) and \
              len(_test)>=(CFG.REGION_MIN_TEST_ROWS if _m=='ML-5.3' else CFG.MIN_TEST_ROWS)
        _v4_roll.append({'module':_m,'target_year':_y,'n_asof_train':len(_train),
             'n_test_observed':len(_test),'legally_feasible_given_publication_lag':bool(_gate),
             'source_vintage_asof_origin':'NOT_ARCHIVED',
             'scientific_status':'RETROSPECTIVE_ONLY' if _gate else 'NOT_FEASIBLE_SMALL_HISTORICAL_TRAIN'})
qa_table('v4_rolling_origin_feasibility',pd.DataFrame(_v4_roll),allow_empty=True)


# 10Б. Контракт V5: полнота моделей, численная состоятельность и доказательная база

Этот блок проверяет результаты независимо от общей отметки выполнения ноутбука. Модели со сбоями, отсутствующими прогнозами или нестабильными значениями не засчитываются. Нет исходных показателей — `NOT_IDENTIFIABLE`, а не вымышленный результат.

In [ ]:
# ML-05 V5 · fail-closed model registry, numerical diagnostics and evidence matrix.
progress_event('STATISTICS','V5: полный контракт моделей, протоколов и проверка числа сидов',announce=True)
V5_MODEL_QA=[];V5_MODULE_ACCEPTANCE=[]
_artifact_paths=[m['path'] for m in MODEL_RECORDS]
if len(_artifact_paths)!=len(set(_artifact_paths)):
    raise RuntimeError('V5: different model experiments have duplicate output paths')
if len(MODEL_FILES)!=len(set(_artifact_paths)):
    raise RuntimeError('V5: model registry and actual model files differ')
for _m in MODEL_RECORDS:
    _fp=ROOT/'models'/_m['path']
    _ok=_fp.is_file() and _m.get('file_sha256')==hash_file(_fp) and _m.get('reloaded')
    V5_MODEL_QA.append({'module':_m['module'],'model':_m['model'],'protocol':_m.get('protocol','CUSTOM'),
        'seed':_m['seed'],'path':_m['path'],'sha_verified':bool(_ok),
        'guarded_predictions':_m.get('prediction_guard_events',0),
        'guard_fraction':_m.get('prediction_guard_fraction',0.)})
if V5_MODEL_QA and not all(x['sha_verified'] for x in V5_MODEL_QA):
    raise RuntimeError('V5: saved models changed or reload failed')
qa_table('v5_model_files_independently_verified',pd.DataFrame(V5_MODEL_QA),allow_empty=True)

# Reconcile required model names and their independent saved metrics, not a mere code branch.
for _m in ['ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7']:
    _mm=[r for r in METRICS if r.get('module')==_m]
    _issues=[r for r in ISSUES if r.get('module')==_m and r.get('status') in
      ('MODEL_FAILED','SPATIAL_MODEL_FAILURE','SOURCE_PROFILE_FAILURE','FAMILY_PROFILE_FAILED')]
    _status=MODULES[_m].get('status','UNKNOWN')
    _preds=sum(int(x.get('n',0)) for x in _mm)
    _time={x['model'] for x in _mm if x.get('protocol')=='ROLLING_TIME'}
    _required={'persistence','rolling_mean_2','damped_trend','ridge','elasticnet','lightgbm','catboost_ensemble','ppml','negbin'}
    _targets_confirmed=bool(_status.startswith('TESTED'))
    if _m in ('ML-5.1','ML-5.3','ML-5.4') and _targets_confirmed:
        _missing=sorted(_required-_time)
        _seeds={int(x['seed']) for x in _mm if x.get('model')=='catboost' and x.get('protocol')=='ROLLING_TIME'}
        _missing_seeds=sorted(set(int(z) for z in CFG.FINAL_SEEDS)-_seeds) if CFG.MODE=='FULL' else []
    else:_missing=[];_missing_seeds=[]
    _numeric_invalid=[x['model'] for x in _mm if not np.isfinite(x.get('MAE',np.nan)) or
        not np.isfinite(x.get('RMSE',np.nan)) or float(x.get('MAE',0))<0]
    _failures=bool(_issues or _missing or _missing_seeds or _numeric_invalid)
    _scientific=('OBSERVED_TARGET_TESTED_RETROSPECTIVELY' if _targets_confirmed else
                 'DESCRIPTIVE_OR_PROXY_TEST_ONLY' if _mm or 'DESCRIPTIVE' in _status or 'PROXY' in _status else
                 'NOT_IDENTIFIABLE_WITH_AVAILABLE_STAGE_I')
    V5_MODULE_ACCEPTANCE.append({'module':_m,'status':_status,'metrics':len(_mm),
        'model_file_count':sum(x['module']==_m for x in MODEL_RECORDS),'failed_methods':len(_issues),
        'missing_methods':','.join(_missing),'missing_final_seeds':','.join(map(str,_missing_seeds)),
        'nonfinite_metrics':','.join(_numeric_invalid),
        'computational_pass':not _failures,'scientific_scope':_scientific,
        'original_target_fully_identified':bool(_m=='ML-5.4' and _targets_confirmed),
        'historical_source_vintage_validated':False})

V5_LEDGER=pd.DataFrame(V5_MODULE_ACCEPTANCE)
qa_table('v5_seven_module_acceptance',V5_LEDGER,['module'])

# Source evidence: 52 original datasets are all profiled but not uniformly identifiable as model features.
V5_SOURCE_EVIDENCE=pd.DataFrame(V4_SOURCE_PROFILES)
if len(V5_SOURCE_EVIDENCE)!=len(ASSETS) or not all(r.get('content_status')=='FULL_VALUE_COVERAGE_PASS' for r in V4_SOURCE_PROFILES):
    if CFG.MODE=='FULL':raise RuntimeError('V5 source coverage is not 52/52')
qa_table('v5_all52_source_evidence',V5_SOURCE_EVIDENCE,allow_empty=True)
V5_COMPUTATIONAL_PASS=bool(not V4_SOURCE_ERRORS and V5_LEDGER.computational_pass.all())
V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS=bool(V5_COMPUTATIONAL_PASS and all(V5_LEDGER.original_target_fully_identified))
V5_ACCEPTANCE_STATUS='FULL_COMPUTATIONS_WITH_SCIENTIFIC_LIMITS' if V5_COMPUTATIONAL_PASS else 'MODEL_OR_EVIDENCE_FAILURE'
STATE['scientific']=V5_ACCEPTANCE_STATUS
print('V5: 52-source integrity',len(V5_SOURCE_EVIDENCE),'/',len(ASSETS),
      '| all computational methods',V5_COMPUTATIONAL_PASS,'| scientifically identified 7/7',V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS)


## V7 · независимая трассировка источников и моделей: 52/52 + научные границы семи модулей

In [ ]:
# V7.4.0 · проверяемая цепочка Stage I → исследовательская панель → сохранённая модель.
# Не называем технически обследованные 52 источника 52 модельными признаками.
progress_event('STATISTICS','V7: проверяемая трассировка 52 источников и 7 модулей',announce=True)

_v7_source_panels={}
def _v7_map(asset_or_list,panels,methods,relationship='PRIMARY_SOURCE'):
    for _a in (asset_or_list if isinstance(asset_or_list,(tuple,list)) else [asset_or_list]):
        if _a is None:continue
        _id=_a['asset_id'] if isinstance(_a,dict) else str(_a)
        _v7_source_panels.setdefault(_id,[]).append({'panels':list(panels),'modules':list(methods),'relationship':relationship})
_v7_map(E01,['edu_od_year','forecast_od_candidates','outflow_observed','education_network_year'],['ML-5.1','ML-5.3','ML-5.6'])
_v7_map(E02,['edu_org_registry','v7_organization_admission_discovery','v4_organization_contingent_candidates'],['ML-5.2'])
_v7_map(SPO_ORG,['edu_org_registry'],['ML-5.2'])
_v7_map(SPO_MASTER,['spo_admission_raw_candidates'],['ML-5.2'])
_v7_map(VPO_211,['vpo_program_description','v4_specialty_model_frame','v7_organization_admission_discovery'],['ML-5.4','ML-5.2'])
_v7_map(GRAD_ORG,['graduate_descriptive_statistics','v4_ml55_crosssection_metrics'],['ML-5.5'])
_v7_map(GRAD_FIELD,['graduate_specialty_description'],['ML-5.5'])
_v7_map(source_asset(segment='05'),['e04_e05_full_family_audit'],['ML-5.5'],'FULL_FAMILY_DESCRIPTION')
_v7_map(MUN,['municipal_demography','municipality_education_context'],['ML-5.7'])
_v7_map(TRANSPORT,['region_distance_proxy'],['ML-5.1'],'CONTEXT_PROXIED_NOT_LITERAL_REGION_DISTANCES')
_v7_map(source_asset(segment='02'),['d02_indicator_coverage'],['ML-5.1','ML-5.7'],'SEPARATE_THEMATIC_CONTEXT')
_v7_map(source_asset(segment='04'),['vpo_all_sections_audit'],['ML-5.2','ML-5.4'],'SEPARATE_VPO_SECTIONS')
_v7_map(source_asset(segment='03',role='education_od_matrix_raw_cells'),['e01_raw_cell_sources'],['ML-5.1'],'SOURCE_CROSSCHECK_NOT_ADDITIVE')
_v7_map(source_asset(segment='06'),['spatial_exact_crosswalk','spatial_D04_preview','spatial_BDMO_HIST_preview'],['ML-5.1','ML-5.7'],'SPATIAL_CONTEXT_WITH_MATCHING_GATE')

# Confirm only *actually used* predictive sources, based on target panel and model feature registry.
_v7_executed={r['module'] for r in MODEL_RECORDS}
_v7_real={}
_v7_real[E01['asset_id']]={'ML-5.1','ML-5.3'} & _v7_executed
_v7_real[VPO_211['asset_id']]={'ML-5.4'} & _v7_executed
_v7_real[E02['asset_id']]={'ML-5.2'} & _v7_executed
_v7_real[GRAD_ORG['asset_id']]={'ML-5.5'} & _v7_executed
_v7_real[MUN['asset_id']]={'ML-5.7'} & _v7_executed
_v7_od_features=set()
for _r in MODEL_RECORDS:
    if _r.get('module')=='ML-5.1':
        _v7_od_features.update(str(f) for f in (_r.get('feature_num') or []))
if 'distance_proxy_km' in _v7_od_features and len(DISTANCE):
    _v7_real[TRANSPORT['asset_id']]={'ML-5.1'}
if any(x.startswith(('origin_econ_','destination_econ_')) for x in _v7_od_features) and len(ECON_PANEL):
    for _id in ECON_PANEL.source_asset_id.dropna().astype(str).unique():
        _v7_real[str(_id)]={'ML-5.1'}

# Model ancestry table: source inputs for every saved trained model, no guessing from source names.
V7_MODEL_LINEAGE=[]
for _r in MODEL_RECORDS:
    _mod=_r.get('module','')
    _src=sorted(_id for _id,_mods in _v7_real.items() if _mod in _mods)
    if not _src and _mod in ('ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.7'):
        ISSUES.append({'module':_mod,'status':'V7_LINEAGE_MISSING','reason':'No verified source linkage for recorded model '+str(_r.get('path')),'required':'exact lineage'})
    V7_MODEL_LINEAGE.append({'module':_mod,'model':_r.get('model'),'protocol':_r.get('protocol'),
      'seed':_r.get('seed'),'model_file':_r.get('path'),'model_sha256':_r.get('file_sha256'),
      'asset_ids':';'.join(_src),'n_sources':len(_src),
      'feature_columns':';'.join([str(x) for x in (_r.get('feature_num') or [])]),
      'feature_categories':';'.join([str(x) for x in (_r.get('feature_cat') or [])]),
      'claim_scope':'AS_STORED_MODEL_FIT_AND_PANEL_DERIVATION_NOT_CAUSAL_SOURCE_ATTRIBUTION'})
V7_MODEL_LINEAGE=pd.DataFrame(V7_MODEL_LINEAGE)
qa_table('v7_model_lineage',V7_MODEL_LINEAGE,allow_empty=True)

V7_SOURCE_LINEAGE=[]
for _a in ASSETS:
    _id=_a['asset_id'];_profiles=_v7_source_panels.get(_id,[])
    _num=V7_SOURCE_AUDIT.loc[V7_SOURCE_AUDIT.asset_id.eq(_id)].iloc[0]
    _panels=sorted({p for x in _profiles for p in x['panels']})
    _modules=sorted({m for x in _profiles for m in x['modules']})
    _direct=sorted(_v7_real.get(_id,set()))
    _derived=ROOT/'derived'/'v4_source_values'/_num.numeric_profile_file
    V7_SOURCE_LINEAGE.append({'asset_id':_id,'segment':_a['segment'],
      'sha256':_a['sha256'],'rows_actual':int(_a['rows_actual']),
      'content_profile_sha256':hash_file(_derived) if _derived.is_file() else '',
      'source_coverage_pass':bool(_num.all_values_profiled and _num.verified_sha256 and _num.footer_rows_verified),
      'source_specific_analysis':'FULL_VALUE_DISTRIBUTION_YEAR_IF_AVAILABLE',
      'analytical_panels':';'.join(_panels),
      'research_modules_possible':';'.join(_modules),
      'actual_saved_model_modules':';'.join(_direct),
      'direct_model_input':bool(_direct),'model_usage_semantics':('PROVEN_PANEL_CHAIN' if _direct else 'THEMATIC_OR_CONTEXT_ONLY'),
      'nonmodel_reason':('MISMATCHED_STATISTICAL_GRAIN_OR_OVERLAPPING_SOURCE_OR_NOT_IDENTIFIED_AS_PREDICTOR'
                         if not _direct else ''),
      'not_invented_source_join':False})
V7_SOURCE_LINEAGE=pd.DataFrame(V7_SOURCE_LINEAGE)
qa_table('v7_source_lineage_52',V7_SOURCE_LINEAGE,['asset_id'])
if CFG.MODE=='FULL' and (len(V7_SOURCE_LINEAGE)!=len(ASSETS) or
    not V7_SOURCE_LINEAGE.source_coverage_pass.all() or
    not V7_MODEL_LINEAGE.n_sources.ge(1).all()):
    raise RuntimeError('V7 LINEAGE: исходники, модели или реестр происхождения не согласованы')

V7_SCIENCE_GATES=[]
for _mod in ('ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7'):
    _row=V5_LEDGER.loc[V5_LEDGER.module.eq(_mod)].iloc[0]
    _raw={
      'ML-5.1':'E01_REGION_OF_PRIOR_SCHOOLING_VS_CURRENT_SPO_STUDY_NOT_RESIDENCE',
      'ML-5.2':'VERIFIED_ADMISSIONS_BY_ORGANIZATION_YEAR_NOT_STUDENT_CONTINGENT',
      'ML-5.3':'OBSERVED_E01_OUTFLOW_SUM_OF_REPORTED_EDGES_NOT_RESIDENTIAL_MIGRATION',
      'ML-5.4':'REGIONAL_SPECIALTY_ADMITTED_COUNT_NOT_ORG_BY_SPECIALTY_OD',
      'ML-5.5':'EMPLOYMENT_SNAPSHOT_NOT_RETENTION_OR_LOCATION_OF_WORK',
      'ML-5.6':'DYNAMIC_OBSERVED_OD_NETWORK_WITH_COVERAGE_BREAK',
      'ML-5.7':'VERIFIED_EDUCATIONAL_CAMPUSES_AND_PREPERIOD_CAPACITY_BY_OKTMO'}[_mod]
    _original_ident=bool(_row.get('original_target_fully_identified',False))
    _status=MODULES[_mod].get('status','UNREPORTED')
    V7_SCIENCE_GATES.append({'module':_mod,'target_definition':_raw,
      'computational_pass':bool(_row.computational_pass),
      'original_scientific_target_identified':_original_ident,
      'scientific_status':str(_row.scientific_scope),
      'module_execution_status':_status,
      'model_records':int(V7_MODEL_LINEAGE.module.eq(_mod).sum()),
      'historical_vintage_proven':False,
      'remaining_identification_dependency':('ORIGINAL_SOURCE_DEFINITION_OR_SPATIAL_CONFIRMATION_REQUIRED' if not _original_ident else ''),
      'no_synthetic_data':True})
V7_SCIENCE_GATES=pd.DataFrame(V7_SCIENCE_GATES)
qa_table('v7_seven_scientific_gates',V7_SCIENCE_GATES,['module'])
print('V7: 52/52 lineages; true direct model sources',int(V7_SOURCE_LINEAGE.direct_model_input.sum()),
      '/52; models with source lineage',len(V7_MODEL_LINEAGE),'/',len(MODEL_RECORDS))


## 10В. Независимая приёмка реконструкций
Состав и происхождение признаков, годовые разрезы, 52/52 источника, фактические и пропущенные эксперименты; доказательные статусы без подмены научного результата технической готовностью.

In [ ]:
# ML-05 V7.4.0 · Per-experiment acceptance without inventing any missing research target.
progress_event('STATISTICS','V7.4: проверка предобработки, 52 источников и всех выполнимых экспериментов',announce=True)
ADR_RECONSTRUCTED_LINEAGE=V7_MODEL_LINEAGE.copy()
if len(ADR_RECONSTRUCTED_LINEAGE):
    for _idx,_r in ADR_RECONSTRUCTED_LINEAGE.iterrows():
        _sources=set(str(_r.asset_ids).split(';')) if pd.notna(_r.asset_ids) else set()
        if _r.module=='ML-5.7' and str(_r.model) in ('adr_E0','adr_E1'):
            _sources.update([MUN['asset_id'],SPO_ORG['asset_id'],SPATIAL['asset_id']])
        if _r.module=='ML-5.2':
            _sources.add(E02['asset_id'])
        _sources={s for s in _sources if s in ASSET_BY_ID}
        ADR_RECONSTRUCTED_LINEAGE.at[_idx,'asset_ids']=';'.join(sorted(_sources))
        ADR_RECONSTRUCTED_LINEAGE.at[_idx,'n_sources']=len(_sources)
    qa_table('adr_reconstructed_model_lineage',ADR_RECONSTRUCTED_LINEAGE,allow_empty=True)

# A fixed research contract: existing V7 calculations and all added observable subexperiments.
_adrex=pd.DataFrame(ADR_EXPERIMENTS)
_adr_qa=[]
for _mod in ('ML-5.1','ML-5.2','ML-5.3','ML-5.4','ML-5.5','ML-5.6','ML-5.7'):
    _ev=_adrex.loc[_adrex.module.eq(_mod)] if len(_adrex) else pd.DataFrame()
    _bad=_ev.loc[_ev.status.str.startswith('FAILED',na=False)] if len(_ev) else pd.DataFrame()
    _v5=V5_LEDGER.loc[V5_LEDGER.module.eq(_mod)]
    _existing=bool(len(_v5) and _v5.iloc[0].computational_pass)
    _ran=bool(len(_ev) and _ev.status.isin(['MODEL_COMPLETED','V7_COMPLETED','DESCRIPTIVE_COMPLETED']).any())
    _scientific=bool(len(_v5) and _v5.iloc[0].original_target_fully_identified)
    _adr_qa.append({'module':_mod,'v7_computational_pass':_existing,'v71_added_experiments':len(_ev),
        'v71_observed_experiment_completed':_ran,'v71_new_failures':len(_bad),
        'original_scientific_target_fully_identified':_scientific,
        'scientific_scope':'ONLY_PUBLISHED_OBSERVED_TARGETS_AND_LEGALLY_MATCHED_PROXIES',
        'missing_data_not_fabricated':True,'missing_data_experiments_skipped':int(_ev.status.str.startswith('SKIPPED',na=False).sum()) if len(_ev) else 0})
ADR_SEVEN_MODULE_FINAL=qa_table('adr_v71_seven_module_acceptance',pd.DataFrame(_adr_qa),['module'])
ADR_COMPUTATIONAL_PASS=bool(V5_COMPUTATIONAL_PASS and ADR_SEVEN_MODULE_FINAL.v71_new_failures.eq(0).all())
ADR_SCIENTIFIC_7_ORIGINAL_IDENTIFIED=bool(ADR_COMPUTATIONAL_PASS and ADR_SEVEN_MODULE_FINAL.original_scientific_target_fully_identified.all())
ADR_SOURCE_AUDIT=pd.DataFrame([{'asset_id':a['asset_id'],'sha256_expected':a['sha256'],
        'sha256_checked':INPUT_SHA.get(a['asset_id']),
        'is_source_checked':INPUT_SHA.get(a['asset_id'])==a['sha256'],
        'source_analytical_role':V7_SOURCE_LINEAGE.loc[V7_SOURCE_LINEAGE.asset_id.eq(a['asset_id']),'model_usage_semantics'].iloc[0]
            if V7_SOURCE_LINEAGE.asset_id.eq(a['asset_id']).any() else 'UNMAPPED',
        'source_methodological_grain':a.get('grain_status','UNVERIFIED'),
        'direct_model_input':bool(V7_SOURCE_LINEAGE.loc[V7_SOURCE_LINEAGE.asset_id.eq(a['asset_id']),'direct_model_input'].iloc[0])
            if V7_SOURCE_LINEAGE.asset_id.eq(a['asset_id']).any() else False} for a in ASSETS])
qa_table('adr_source_52_completeness',ADR_SOURCE_AUDIT,['asset_id'])

# Exact original inputs are immutable: compare expected Stage I fingerprint and do not reinterpret source rows as new observations.
if CFG.MODE=='FULL' and (len(ADR_SOURCE_AUDIT)!=52 or not ADR_SOURCE_AUDIT.is_source_checked.all()):
    raise RuntimeError('ADR SOURCE_GATE: source coverage failure')
if ADR_COMPUTATIONAL_PASS and len(ADR_RECONSTRUCTED_LINEAGE):
    if not ADR_RECONSTRUCTED_LINEAGE.n_sources.ge(1).all():
        raise RuntimeError('ADR MODEL_GATE: saved model lacks verified source lineage')
if len(ADR_ADMISSION_YEAR) and not (ADR_ADMISSION_YEAR.area_total.fillna(False).all() and
        ADR_ADMISSION_YEAR.specialty_total.fillna(False).all() and ADR_ADMISSION_YEAR.n_reporting_cells.eq(1).all()):
    raise RuntimeError('ADR SEMANTIC_GATE: source-reported admission cell is not an unambiguous total')
if len(ADR_GEO_VERIFIED) and not ADR_GEO_VERIFIED.identity_status.eq('PROXIMITY_MATCH_NOT_OFFICIAL_CAMPUS_LOCATION').all():
    raise RuntimeError('ADR SPATIAL_GATE: proxy falsely asserted as confirmed campus')

ADR_FINAL_REPORT={
 'version':CFG.VERSION,'source_catalog_entries':len(ASSETS),'source_sha_passed':int(ADR_SOURCE_AUDIT.is_source_checked.sum()),
 'all_seven_modules_reported':len(ADR_SEVEN_MODULE_FINAL)==7,'v7_technical_methods_retained':V5_COMPUTATIONAL_PASS,
 'v71_additional_experiment_count':len(ADR_EXPERIMENTS),'v71_additional_model_experiment_count':sum(
    r['status']=='MODEL_COMPLETED' for r in ADR_EXPERIMENTS),
 'v71_computational_pass':ADR_COMPUTATIONAL_PASS,
 'seven_original_scientific_targets_identified':ADR_SCIENTIFIC_7_ORIGINAL_IDENTIFIED,
 'OD_all_year_balanced_edges':int(len(ADR_OD_BALANCED)),
 'VO_source_approved_org_year_admissions':int(len(ADR_ADMISSION_YEAR)),
 'SPO_spatially_reconstructed_site_year_proxies':int(len(ADR_GEO_VERIFIED)),
 'municipal_geocode_E0E1':ADR_ML57_GEO_STATUS,
 'methodological_rule':'MISSING_NOT_OBSERVED_IS_SKIPPED_NOT_ZERO_FILLED_AND_NOT_FABRICATED'
}
qa_table('adr_final_scientific_diagnostics',pd.DataFrame([{
    k:(json.dumps(v,ensure_ascii=False,default=str) if isinstance(v,(dict,list)) else v)
    for k,v in ADR_FINAL_REPORT.items()}]),allow_empty=True)
print('V7.4: source coverage',ADR_FINAL_REPORT['source_sha_passed'],'/52 | additional experiments',
    ADR_FINAL_REPORT['v71_additional_experiment_count'],'| computational gate',ADR_COMPUTATIONAL_PASS,
    '| scientific 7/7',ADR_SCIENTIFIC_7_ORIGINAL_IDENTIFIED)


## 10Г. V7.4 · независимая приёмка исследовательских выводов

Проверка временной доступности, сохранённых моделей и всей цепочки от первичного источника до результата. Выбор по TEST не допускается.

In [ ]:
# ML-05 V7.4.0 · независимая научная/временная/публикационная диагностика.
progress_event('STATISTICS','V7.4: временные границы, выбор моделей, источник→признак→результат',announce=True)
import importlib.metadata as _v72_metadata

V72_REPRODUCIBILITY={
 'version':CFG.VERSION,'python':platform.python_version(),'mode':CFG.MODE,
 'source_commit':SOURCE_COMMIT,'source_catalog_blob_sha':index_info['sha'],
 'catalog_n_assets':len(ASSETS),'sha_verified_n':len(INPUT_SHA),
 'seed_main':int(CFG.RANDOM_SEED),'seeds_final':list(CFG.FINAL_SEEDS),
 'training_model_count':len(MODEL_RECORDS),'metric_rows':len(METRICS),
 'prediction_rows':int(PRED_TOTAL),'source_vintage':str(INDEX.get('created_at')),
 'prequential_historical_source_vintage_archived':False,
 'dependencies':{pkg:(_v72_metadata.version(pkg) if _v72_metadata.packages_distributions() else 'unknown')
                 for pkg in ()}
}
for _package in ('numpy','pandas','duckdb','pyarrow','scikit-learn','scipy','catboost',
                 'lightgbm','statsmodels','networkx','nbformat','xlsxwriter','requests'):
    try:V72_REPRODUCIBILITY['dependencies'][_package]=_v72_metadata.version(_package)
    except _v72_metadata.PackageNotFoundError:V72_REPRODUCIBILITY['dependencies'][_package]='NOT_INSTALLED'

# Explicit as-of protocol for each prediction test; not a real source-archived forecast.
V72_ASOF_AUDIT=pd.DataFrame()
_v72_hist=[]
for _m in METRICS:
    _ty=_m.get('target_year');_tag=_m.get('source_vintage',INDEX.get('created_at'))
    _ty_num=pd.to_numeric(_ty,errors='coerce')
    _vint=pd.to_datetime(str(_tag),utc=True,errors='coerce')
    _real_asof=bool(pd.notna(_ty_num) and pd.notna(_vint) and _vint.year<=int(_ty_num)-CFG.HORIZON)
    _v72_hist.append({'module':str(_m.get('module')),'model':str(_m.get('model')),
        'protocol':str(_m.get('protocol')),'seed':int(_m.get('seed',-1)),
        'target_year':int(_ty_num) if pd.notna(_ty_num) else None,
        'source_vintage':str(_tag),'archived_source_before_forecast_origin':_real_asof,
        'interpretation':('POINT_IN_TIME_SOURCE_VINTAGE' if _real_asof else
                          'RETROSPECTIVE_ONLY_CURRENT_SOURCE_VINTAGE')})
V72_ASOF_AUDIT=qa_table('v72_each_metric_asof_vintage_gate',pd.DataFrame(_v72_hist),allow_empty=True)
V72_REPRODUCIBILITY['n_point_in_time_validated_metrics']=int(V72_ASOF_AUDIT.archived_source_before_forecast_origin.sum()) if len(V72_ASOF_AUDIT) else 0
V72_REPRODUCIBILITY['n_retrospective_only_metrics']=int((~V72_ASOF_AUDIT.archived_source_before_forecast_origin).sum()) if len(V72_ASOF_AUDIT) else 0

# Paired evidence is already calculated from the same prediction entity-year keys, with Holm adjustment.
# Champion selection MUST NOT maximize performance on TEST when DEV labels were unavailable.
V72_MODEL_SELECTION=[]
_v72_m=pd.DataFrame(METRICS)
if len(_v72_m):
    _v72_m['MAE']=pd.to_numeric(_v72_m.MAE,errors='coerce')
    _v72_m['target_year']=pd.to_numeric(_v72_m.target_year,errors='coerce')
    _v72_m=_v72_m.loc[_v72_m.MAE.notna()&np.isfinite(_v72_m.MAE)].copy()
    for (_mod,_sector,_protocol,_year),_all in _v72_m.groupby(['module','sector','protocol','target_year'],dropna=False):
        _values=_all.groupby('model',dropna=False).MAE.agg(['mean','count']).reset_index()
        if not len(_values):continue
        _rank=_values.sort_values(['mean','model'],kind='stable')
        _best=_rank.iloc[0]
        _base=_values.loc[_values.model.eq('persistence')]
        _base=float(_base.iloc[0]['mean']) if len(_base) else np.nan
        _selected='persistence' if np.isfinite(_base) else 'NO_PREDECLARED_BASELINE'
        _choice='PREDECLARED_PERSISTENCE_NOT_TEST_OPTIMIZED'
        if len(CONFIRM):
            _confirm=CONFIRM.loc[(CONFIRM.module.eq(_mod)) &
                                 (CONFIRM.sector.eq(_sector)) &
                                 (CONFIRM.choice_basis.eq('DEV_ASOF'))]
            if len(_confirm):
                _candidate=str(_confirm.iloc[0].selected_model)
                if _candidate in set(_values.model.astype(str)):
                    _selected=_candidate;_choice='DEV_ASOF_PRESELECTED'
        _cur=_values.loc[_values.model.eq(_selected)]
        _selected_mae=float(_cur.iloc[0]['mean']) if len(_cur) else np.nan
        V72_MODEL_SELECTION.append({'module':_mod,'sector':_sector,'protocol':_protocol,
            'test_year':int(_year) if pd.notna(_year) else None,'selected_model_without_test_optimization':_selected,
            'selection_basis':_choice,'selected_test_MAE_retrospective':_selected_mae,
            'persistence_MAE_retrospective':_base,'best_TEST_MAE_for_diagnostics_only':float(_best['mean']),
            'best_TEST_model_not_selected':str(_best['model']),
            'n_test_models_reported':int(len(_values)),
            'historical_source_vintage_validated':False,
            'conclusion':'EXPLORATORY_RETROSPECTIVE_NO_TEST_BASED_DEPLOYMENT'})
V72_MODEL_SELECTION=qa_table('v72_ex_ante_model_selection_audit',
    pd.DataFrame(V72_MODEL_SELECTION),allow_empty=True)

# Independent natural-key model lineage, now with an explicit source/transform/panel link per model.
V72_MODEL_PROVENANCE=[]
_v72_sources={str(x.asset_id):x for x in V7_SOURCE_LINEAGE.itertuples(index=False)}
for _r in V7_MODEL_LINEAGE.itertuples(index=False):
    _ids=sorted({x for x in str(_r.asset_ids or '').split(';') if x in ASSET_BY_ID})
    for _asset in _ids:
        _s=_v72_sources[_asset]
        V72_MODEL_PROVENANCE.append({'module':_r.module,'model_file':_r.model_file,
          'model_sha256':_r.model_sha256,'asset_id':_asset,'source_sha256':ASSET_BY_ID[_asset]['sha256'],
          'transform_or_panel_ref':str(_s.analytical_panels),
          'numerical_features':str(_r.feature_columns),'categorical_features':str(_r.feature_categories),
          'source_relation':'PANEL_DERIVATION_NOT_CAUSAL_FEATURE_ATTRIBUTION',
          'reconstructed_target':False,
          'source_verified_sha256':INPUT_SHA.get(_asset)==ASSET_BY_ID[_asset]['sha256']})
V72_MODEL_PROVENANCE=qa_table('v72_verified_source_panel_model_links',pd.DataFrame(V72_MODEL_PROVENANCE),
                              ['model_file','asset_id'],allow_empty=True)

# Seven scientific gates, with observed tests and explicit skipped original targets side by side.
V72_MODULE_ACCEPTANCE=[]
for _row in V7_SCIENCE_GATES.itertuples(index=False):
    _matches=V72_RECON_DF[V72_RECON_DF.module.eq(_row.module)]
    _results=ADR_SEVEN_MODULE_FINAL.loc[ADR_SEVEN_MODULE_FINAL.module.eq(_row.module)]
    _base_ok=bool(_row.computational_pass)
    _new_errors=int(_matches.status.str.startswith('FAILED').sum()) if len(_matches) else 0
    V72_MODULE_ACCEPTANCE.append({'module':_row.module,
       'baseline_v711_computational_pass':_base_ok,
       'new_v72_validated_substudies':int(_matches.status.str.startswith('PASS').sum()) if len(_matches) else 0,
       'new_v72_skipped_no_source':int(_matches.status.str.startswith('SKIPPED').sum()) if len(_matches) else 0,
       'new_computational_failures':_new_errors,
       'original_target_identified':bool(_row.original_scientific_target_identified),
       'historical_vintage_validated':bool(_row.historical_vintage_proven),
       'honest_scientific_scope':str(_row.scientific_status),
       'acceptance':'COMPUTATIONAL_PASS_SCIENTIFIC_GATED' if _base_ok and not _new_errors else 'COMPUTATIONAL_FAIL',
       'no_missing_source_value_inferred':True})
V72_MODULE_ACCEPTANCE=qa_table('v72_seven_modules_scientific_acceptance',
                                pd.DataFrame(V72_MODULE_ACCEPTANCE),['module'])

V72_ENGINEERING_GATES={
 'all52_sources_sha':len(ASSETS)==52 and len(INPUT_SHA)==52 and
        all(INPUT_SHA.get(a['asset_id'])==a['sha256'] for a in ASSETS),
 'all_7_scientific_decisions_documented':len(V72_MODULE_ACCEPTANCE)==7 and
        V72_MODULE_ACCEPTANCE.module.nunique()==7,
 'preserved_v711_completed_models':len(MODEL_RECORDS)>=233,
 'preserved_v711_metrics':len(METRICS)>=254,
 'same_model_files_and_lineage':len(MODEL_RECORDS)==len(V7_MODEL_LINEAGE) and
        len(V72_MODEL_PROVENANCE)>=len(V7_MODEL_LINEAGE),
 'source_to_model_sha_all_valid':bool(len(V72_MODEL_PROVENANCE) and
        V72_MODEL_PROVENANCE.source_verified_sha256.all()),
 'reconstruction_has_no_failure':bool(len(V72_RECON_DF) and
        not V72_RECON_DF.status.str.startswith('FAILED').any()),
 'no_automatic_org_admission_from_unapproved_label':bool(not len(ADR_ADMISSION_YEAR) or
        bool(CFG.APPROVED_VO_ADMISSION_NAMES)),
 'no_geographic_proxy_becomes_official_campus':bool(not len(V72_SPATIAL_CONFIDENCE) or
        not V72_SPATIAL_CONFIDENCE.confirmed_campus_location.any()),
 'no_test_optimal_model_automatically_selected':bool(not len(V72_MODEL_SELECTION) or
        V72_MODEL_SELECTION.selection_basis.isin([
            'PREDECLARED_PERSISTENCE_NOT_TEST_OPTIMIZED','DEV_ASOF_PRESELECTED']).all()),
 'all_model_testing_vintage_scopes_explicit':len(V72_ASOF_AUDIT)==len(METRICS),
}
qa_table('v72_engineering_decision_gates',pd.DataFrame([
    {'gate':k,'passed':bool(v)} for k,v in V72_ENGINEERING_GATES.items()]),['gate'])
if CFG.MODE=='FULL' and not all(V72_ENGINEERING_GATES.values()):
    bad=[k for k,v in V72_ENGINEERING_GATES.items() if not v]
    raise RuntimeError('V7.2 independent QA failure: '+', '.join(bad))
V72_REPRODUCIBILITY['v72_gates']=V72_ENGINEERING_GATES
V72_REPRODUCIBILITY['new_observed_reconstruction_checks']=len(V72_RECON_DF)
V72_REPRODUCIBILITY['new_original_targets_verified']=int(V72_MODULE_ACCEPTANCE.original_target_identified.sum())
print('V7.4: 52 source records,',len(V72_MODEL_PROVENANCE),'model-source links,',
      len(V72_ASOF_AUDIT),'time-vintage controls,',len(V72_MODULE_ACCEPTANCE),'module decisions')


### 11б. V7.4.0 — новые количественные исследования: парные ошибки прогнозов, кластерный bootstrap, коррекция Holm и декомпозиция ошибок без повторного обучения 233 моделей

In [ ]:
# ML-05 V7.4.0 · STATISTICS · RAM-SAFE FULL EMPIRICAL EXPERIMENTS
# Полностью заменить упавшую ячейку. Без выборочного сокращения данных.
# Модели, исходные метрики, 50 сидов и все наблюдения сохраняются.

import gc, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as _v74_pq
from statsmodels.stats.multitest import multipletests

progress_event('STATISTICS', 'V7.4: потоковая группировка по моделям, парные тесты, bootstrap, Holm', announce=True)

V74_PAIRED_TESTS, V74_ERROR_STRATA = [], []
V74_PAIRED_AUDIT = {
    'version': CFG.VERSION,
    'prediction_rows_input': int(PRED_TOTAL),
    'existing_metric_rows_before': int(len(METRICS)),
    'existing_models_before': int(len(MODEL_RECORDS)),
    'inference': 'cluster-randomization sign-flips; exploratory retrospective',
    'selection_policy': 'predeclared persistence; no model selected by TEST',
    'resampling_repetitions': int(CFG.BOOTSTRAP_REPS),
    'sign_flip_repetitions': int(CFG.PERMUTATION_REPS),
    'holm_family_level': 'all eligible paired models vs persistence',
    'alpha': float(CFG.INFERENCE_ALPHA),
    'raw_test_observations_not_independent_across_models': True,
    'multiple_testing': 'Holm-Bonferroni across all qualified paired comparisons',
    'processing': 'ALL rows; task/model grouping; Arrow streaming; no sampling'
}

if PRED_STORAGE_FILE is None or not Path(PRED_STORAGE_FILE).is_file():
    raise RuntimeError('V7.4: отсутствует полный Parquet прогнозов')

_v74_source = Path(PRED_STORAGE_FILE)
_v74_collapsed = ROOT / 'derived' / 'v74_predictions_ensemble_by_observation.parquet'
_v74_collapsed.parent.mkdir(parents=True, exist_ok=True)
_v74_temp = _v74_collapsed.with_suffix('.tmp.parquet')
_v74_temp.unlink(missing_ok=True)

# Отдельный лимит DuckDB только для этой ячейки.
# Учитывается фактически доступная RAM, включая ограничения cgroup.

def _v74_free_ram():
    import psutil
    available = int(psutil.virtual_memory().available)
    try:
        cg = Path('/sys/fs/cgroup')
        if (cg / 'memory.max').exists():
            cap = (cg / 'memory.max').read_text().strip()
            if cap != 'max':
                current = int((cg / 'memory.current').read_text())
                available = min(available, max(0, int(cap) - current))
        elif (cg / 'memory/memory.limit_in_bytes').exists():
            cap = int((cg / 'memory/memory.limit_in_bytes').read_text())
            if cap < 2**60:
                current = int((cg / 'memory/memory.usage_in_bytes').read_text())
                available = min(available, max(0, cap - current))
    except (OSError, ValueError):
        pass
    return available

_v74_available = _v74_free_ram()

if _v74_available < 900 * 2**20:
    raise MemoryError(f'V7.4: доступно лишь {_v74_available/2**20:.0f} MiB RAM')

_v74_limit_mb = max(512, min(2048, int((_v74_available / 2**20) * 0.33)))

if shutil.disk_usage(str(DB_TMP)).free < 2 * 2**30:
    raise RuntimeError('V7.4: требуется не менее 2 GiB свободного диска')

def _v74_connect():
    con = duck_connect()
    con.execute(f"SET memory_limit='{_v74_limit_mb}MB'")
    con.execute('SET threads=1')
    con.execute('SET preserve_insertion_order=false')
    return con

def _v74_native(value):
    if value is None or pd.isna(value):
        return None
    return value.item() if isinstance(value, np.generic) else value

# Читаем только список уникальных сочетаний задача × модель.
# Все прогнозные записи остаются на диске.

with _v74_connect() as db:
    _v74_jobs = db.execute("""
        SELECT DISTINCT module, sector, split_id, target_id, target_year, model
        FROM read_parquet(?)
        ORDER BY module, sector, split_id, target_id, target_year, model
    """, [str(_v74_source)]).df()

if _v74_jobs.empty:
    raise RuntimeError('V7.4: отсутствуют модельные группы')

_v74_job_count = len(_v74_jobs)
V74_PAIRED_AUDIT['model_groups_to_collapse'] = int(_v74_job_count)

print(f'V7.4: групп задач/моделей: {_v74_job_count}; RAM DuckDB: {_v74_limit_mb} MiB', flush=True)

# Вместо одного огромного GROUP BY — отдельная агрегация каждой модели.
# Сведение 50 сидов: среднее предсказание для каждого entity × year.
# COUNT(DISTINCT seed) сохраняет контроль отсутствия дублирования сидов.

_v74_group_sql = """
    SELECT module, sector, split_id, target_id, target_year, entity_id, model,
           MIN(actual)::DOUBLE AS actual,
           MAX(actual)::DOUBLE AS actual_max,
           AVG(predicted)::DOUBLE AS prediction,
           COUNT(*)::BIGINT AS seed_runs,
           COUNT(DISTINCT seed)::BIGINT AS distinct_seeds,
           MIN(COALESCE(CAST(origin_region AS VARCHAR),
                        CAST(parent_group_id AS VARCHAR),
                        CAST(entity_id AS VARCHAR))) AS cluster_min,
           MAX(COALESCE(CAST(origin_region AS VARCHAR),
                        CAST(parent_group_id AS VARCHAR),
                        CAST(entity_id AS VARCHAR))) AS cluster_id
    FROM read_parquet(?)
    WHERE module IS NOT DISTINCT FROM ?
      AND sector IS NOT DISTINCT FROM ?
      AND split_id IS NOT DISTINCT FROM ?
      AND target_id IS NOT DISTINCT FROM ?
      AND target_year IS NOT DISTINCT FROM ?
      AND model IS NOT DISTINCT FROM ?
    GROUP BY module, sector, split_id, target_id, target_year, entity_id, model
"""

_v74_writer, _v74_written = None, 0

try:
    with _v74_connect() as db:
        for j, job in enumerate(_v74_jobs.itertuples(index=False, name=None), 1):
            args = [str(_v74_source)] + [_v74_native(x) for x in job]
            reader = db.execute(_v74_group_sql, args).fetch_record_batch(rows_per_batch=16384)

            if _v74_writer is None:
                _v74_writer = _v74_pq.ParquetWriter(str(_v74_temp), reader.schema, compression='zstd')
            elif not reader.schema.equals(_v74_writer.schema, check_metadata=False):
                raise RuntimeError('V7.4: несовместимая схема сгруппированных моделей')

            for batch in reader:
                _v74_writer.write_batch(batch)
                _v74_written += int(batch.num_rows)

            del reader

            if j == 1 or j % 10 == 0 or j == _v74_job_count:
                progress_event(
                    'STATISTICS',
                    f'Группировка {j}/{_v74_job_count}; наблюдений {_v74_written:,}',
                    j, _v74_job_count, announce=True
                )

            if j % 20 == 0:
                gc.collect()

    if _v74_writer is not None:
        _v74_writer.close()
        _v74_writer = None

    if _v74_written == 0:
        raise RuntimeError('V7.4: не получены агрегированные прогнозы')

    # Независимая проверка всех агрегированных наблюдений.
    with _v74_connect() as db:
        _v74_pre = db.execute("""
            SELECT COUNT(*)::BIGINT,
                   COUNT(*) FILTER (WHERE actual IS DISTINCT FROM actual_max)::BIGINT,
                   COUNT(*) FILTER (WHERE seed_runs <> distinct_seeds)::BIGINT,
                   COUNT(*) FILTER (WHERE cluster_min IS DISTINCT FROM cluster_id)::BIGINT
            FROM read_parquet(?)
        """, [str(_v74_temp)]).fetchone()

        _v74_dup = db.execute("""
            SELECT COUNT(*) FROM (
                SELECT module, sector, split_id, target_id, target_year, entity_id, model
                FROM read_parquet(?)
                GROUP BY ALL
                HAVING COUNT(*) > 1
            )
        """, [str(_v74_temp)]).fetchone()[0]

        _v74_groups = db.execute("""
            SELECT DISTINCT module, sector, split_id, target_id, target_year
            FROM read_parquet(?)
            WHERE model = 'persistence'
            ORDER BY 1, 2, 3, 4, 5
        """, [str(_v74_temp)]).df()

    if _v74_pre[0] != _v74_written or any(_v74_pre[1:]) or _v74_dup:
        raise RuntimeError(f'V7.4: нарушена целостность: {_v74_pre}; дубли={_v74_dup}')

    # Атомарная замена файла только после успешных проверок.
    _v74_temp.replace(_v74_collapsed)

except BaseException:
    if _v74_writer is not None:
        _v74_writer.close()
    _v74_temp.unlink(missing_ok=True)
    raise

V74_PAIRED_AUDIT['unique_model_observations'] = int(_v74_written)
V74_PAIRED_AUDIT['predeclared_baseline_groups'] = int(len(_v74_groups))

if _v74_groups.empty:
    raise RuntimeError('V7.4: отсутствует инерционный baseline persistence')

# Полная исходная статистическая методика: кластерная рандомизация,
# знаковые перестановки, кластерный bootstrap, доверительные интервалы.

def v74_cluster_randomization(delta, cluster, reps=None, bootstrap_reps=None, seed=1703):
    """Exploratory paired cluster wild sign flips and cluster bootstrap.

    No sampling of observations; resampling *independent clusters* only.
    Exact two-sided sign enumeration when 8 <= G <= 13; Monte Carlo otherwise.
    All resampling is batched, never allocating a reps × rows matrix.
    """
    reps = int(CFG.PERMUTATION_REPS if reps is None else reps)
    bootstrap_reps = int(CFG.BOOTSTRAP_REPS if bootstrap_reps is None else bootstrap_reps)
    if reps < 999 or bootstrap_reps < 99:
        raise ValueError('Insufficient inferential repetitions')
    vals = pd.DataFrame({'d': np.asarray(delta, dtype=np.float64),
                         'g': pd.Series(cluster).astype('string')})
    # Missing cluster IDs must have been resolved by the upstream audited fallback.
    if vals.g.isna().any():
        raise RuntimeError('INFERENCE: missing cluster IDs')
    if not np.isfinite(vals.d.to_numpy()).all():
        raise RuntimeError('INFERENCE: nonfinite error difference')
    grouped = vals.groupby('g',sort=True,observed=True).d.agg(['sum','count'])
    g,n = int(len(grouped)),int(len(vals))
    out={'n_clusters':g,'n_rows':n,'permutations_requested':reps,
         'bootstrap_repetitions':bootstrap_reps}
    if g < 8 or n < 25:
        return {**out,'p_value':np.nan,'ci_low':np.nan,'ci_high':np.nan,
                'min_attainable_p':np.nan,'permutations_effective':0,
                'inference_status':'INSUFFICIENT_INDEPENDENT_CLUSTERS'}
    sums = grouped['sum'].to_numpy(np.float64)
    counts = grouped['count'].to_numpy(np.float64)
    observed = float(sums.sum()/counts.sum())
    rng = np.random.default_rng(int(seed))
    extreme=0
    if g<=13:
        # Enumerate all 2**g sign vectors, including the observed all-positive signs.
        k = 1 << g
        for begin in range(0,k,256):
            codes=np.arange(begin,min(k,begin+256),dtype=np.uint32)[:,None]
            signs=(2*((codes >> np.arange(g,dtype=np.uint32)) & 1).astype(np.int8)-1).astype(np.float64)
            permuted=signs@sums/counts.sum()
            extreme+=int(np.count_nonzero(np.abs(permuted)>=abs(observed)-1e-12))
        p=float(extreme/k)
        pmin=float(2/k) # two-sided sign-invariant statistic has at least ±all-positive
        method='EXACT_CLUSTER_SIGN_ENUMERATION'
        effective=k
    else:
        # Monte-Carlo finite-sample correction (1 + hits)/(1 + reps).
        for begin in range(0,reps,int(CFG.RESAMPLING_BATCH_ROWS)):
            b=min(int(CFG.RESAMPLING_BATCH_ROWS),reps-begin)
            signs=rng.integers(0,2,size=(b,g),dtype=np.int8).astype(np.float64)*2.-1.
            permuted=signs@sums/counts.sum()
            extreme+=int(np.count_nonzero(np.abs(permuted)>=abs(observed)-1e-12))
        p=float((1+extreme)/(1+reps))
        pmin=float(1/(1+reps))
        method='MONTE_CARLO_CLUSTER_SIGN_FLIP'
        effective=reps
    # Cluster bootstrap percentile CI, also batched with bounded memory.
    boots=np.empty(bootstrap_reps,dtype=np.float64)
    for begin in range(0,bootstrap_reps,int(CFG.RESAMPLING_BATCH_ROWS)):
        b=min(int(CFG.RESAMPLING_BATCH_ROWS),bootstrap_reps-begin)
        selected=rng.integers(0,g,size=(b,g),dtype=np.int32)
        boots[begin:begin+b]=sums[selected].sum(axis=1)/counts[selected].sum(axis=1)
    lo,hi=np.quantile(boots,[.025,.975]).tolist()
    if not (0<=p<=1 and np.isfinite([lo,hi,observed]).all()):
        raise RuntimeError('INFERENCE: invalid resampling result')
    return {**out,'p_value':p,'ci_low':float(lo),'ci_high':float(hi),
            'min_attainable_p':pmin,'permutations_effective':effective,
            'observed_mean_MAE_delta':observed,
            'inference_status':method}

# Проверка работы статистического метода.
_probe = v74_cluster_randomization([2., -1.] * 15, [f'g{i}' for i in range(30)], reps=999, bootstrap_reps=99)
assert _probe['n_rows'] == 30 and _probe['n_clusters'] == 30 and 0 < _probe['p_value'] <= 1

# JOIN выполняется по ОДНОЙ модели.
# В память загружается только её собственная парная выборка.
# Никакого урезания тестовых наблюдений.

_v74_pair_sql = """
    SELECT a.model, a.entity_id, a.cluster_id,
           a.actual, b.actual AS baseline_actual,
           a.prediction AS candidate, b.prediction AS baseline,
           a.seed_runs, a.distinct_seeds
    FROM read_parquet(?) a
    JOIN read_parquet(?) b
      ON a.module = b.module
     AND a.sector = b.sector
     AND a.split_id = b.split_id
     AND a.target_id = b.target_id
     AND a.target_year = b.target_year
     AND a.entity_id = b.entity_id
    WHERE b.model = 'persistence'
      AND a.model = ?
      AND a.module IS NOT DISTINCT FROM ?
      AND a.sector IS NOT DISTINCT FROM ?
      AND a.split_id IS NOT DISTINCT FROM ?
      AND a.target_id IS NOT DISTINCT FROM ?
      AND a.target_year IS NOT DISTINCT FROM ?
"""

_v74_pair_count, _v74_skipped = 0, 0

with _v74_connect() as db:
    for i, group in enumerate(_v74_groups.itertuples(index=False, name=None), 1):
        keys = [_v74_native(x) for x in group]

        models = db.execute("""
            SELECT DISTINCT model
            FROM read_parquet(?)
            WHERE module IS NOT DISTINCT FROM ?
              AND sector IS NOT DISTINCT FROM ?
              AND split_id IS NOT DISTINCT FROM ?
              AND target_id IS NOT DISTINCT FROM ?
              AND target_year IS NOT DISTINCT FROM ?
              AND model <> 'persistence'
            ORDER BY model
        """, [str(_v74_collapsed), *keys]).fetchall()

        for (model,) in models:
            paired = db.execute(
                _v74_pair_sql,
                [str(_v74_collapsed), str(_v74_collapsed), model, *keys]
            ).df()

            if paired.empty:
                _v74_skipped += 1
                continue

            actual = paired.actual.to_numpy(float)
            baseline_actual = paired.baseline_actual.to_numpy(float)
            candidate = paired.candidate.to_numpy(float)
            baseline = paired.baseline.to_numpy(float)

            if (not np.isfinite(actual).all()
                or not np.isfinite(candidate).all()
                or not np.isfinite(baseline).all()
                or not np.array_equal(actual, baseline_actual)):
                raise RuntimeError(f'V7.4: несовместимые факты или прогнозы: {keys}, {model}')

            diff = np.abs(actual - candidate) - np.abs(actual - baseline)
            clusters = paired.cluster_id.fillna(paired.entity_id).astype(str).to_numpy()

            stats = v74_cluster_randomization(
                diff, clusters, reps=CFG.PERMUTATION_REPS,
                bootstrap_reps=CFG.BOOTSTRAP_REPS,
                seed=int(CFG.RANDOM_SEED) + len(V74_PAIRED_TESTS)
            )

            mae = float(np.abs(actual - candidate).mean())
            baseline_mae = float(np.abs(actual - baseline).mean())

            V74_PAIRED_TESTS.append({
                'module': keys[0], 'sector': keys[1],
                'protocol': keys[2], 'target_id': keys[3],
                'test_year': keys[4], 'model': str(model),
                'comparator': 'persistence',
                'n_paired_observations': int(len(diff)),
                'mean_candidate_MAE': mae,
                'mean_baseline_MAE': baseline_mae,
                'MAE_delta_candidate_minus_baseline': float(diff.mean()),
                'candidate_better_on_test_descriptive_only': bool(mae < baseline_mae),
                'n_model_seed_runs': int(paired.seed_runs.max()),
                'source_vintage_verified_for_origin': False,
                **stats
            })

            strata = pd.cut(
                actual, [-np.inf, .5, 5.5, 25.5, 100.5, np.inf],
                labels=['0', '1-5', '6-25', '26-100', '>100']
            )

            part = pd.DataFrame({
                'stratum': strata,
                'candidate_error': np.abs(actual - candidate),
                'baseline_error': np.abs(actual - baseline)
            })

            for s, subset in part.groupby('stratum', observed=True):
                V74_ERROR_STRATA.append({
                    'module': keys[0], 'sector': keys[1],
                    'protocol': keys[2], 'target_id': keys[3],
                    'test_year': keys[4], 'model': str(model),
                    'stratum': str(s), 'n': int(len(subset)),
                    'candidate_MAE': float(subset.candidate_error.mean()),
                    'persistence_MAE': float(subset.baseline_error.mean()),
                    'delta_MAE': float((subset.candidate_error - subset.baseline_error).mean()),
                    'all_observations_retained': True
                })

            _v74_pair_count += 1
            del paired, part

        progress_event(
            'STATISTICS',
            f'Парные тесты {i}/{len(_v74_groups)}; сравнений {_v74_pair_count}',
            i, len(_v74_groups), announce=True
        )
        gc.collect()

# Поправка Холма сразу по всем квалифицированным сравнениям.

V74_PAIRED_TESTS = pd.DataFrame(V74_PAIRED_TESTS)
V74_ERROR_STRATA = pd.DataFrame(V74_ERROR_STRATA)

if V74_PAIRED_TESTS.empty:
    raise RuntimeError('V7.4: нет парных сравнений с persistence')

pvalues = pd.to_numeric(V74_PAIRED_TESTS.p_value, errors='coerce')
valid = pvalues.notna() & np.isfinite(pvalues) & pvalues.between(0, 1)

V74_PAIRED_TESTS['p_holm'] = np.nan

if valid.any():
    V74_PAIRED_TESTS.loc[valid, 'p_holm'] = multipletests(
        pvalues[valid].to_numpy(float), method='holm'
    )[1]

V74_PAIRED_TESTS['interpretation'] = np.where(
    V74_PAIRED_TESTS.p_holm.lt(.05) &
    V74_PAIRED_TESTS.MAE_delta_candidate_minus_baseline.lt(0),
    'EXPLORATORY_ADVANTAGE_CANDIDATE',
    np.where(
        V74_PAIRED_TESTS.p_holm.lt(.05) &
        V74_PAIRED_TESTS.MAE_delta_candidate_minus_baseline.gt(0),
        'EXPLORATORY_ADVANTAGE_PERSISTENCE',
        'NO_DETECTED_ADVANTAGE_OR_INSUFFICIENT_INFERENCE;_NOT_EQUIVALENCE'
    )
)

V74_PAIRED_AUDIT.update({
    'paired_comparisons': int(len(V74_PAIRED_TESTS)),
    'eligible_p_values': int(valid.sum()),
    'error_strata': int(len(V74_ERROR_STRATA)),
    'unpaired_candidate_groups': int(_v74_skipped),
    'models_after': int(len(MODEL_RECORDS)),
    'metrics_after': int(len(METRICS)),
    'training_artifacts_unchanged': bool(
        len(MODEL_RECORDS) == V74_PAIRED_AUDIT['existing_models_before']
        and len(METRICS) == V74_PAIRED_AUDIT['existing_metric_rows_before']
    )
})

if not V74_PAIRED_AUDIT['training_artifacts_unchanged']:
    raise RuntimeError('V7.4: новые тесты не должны изменять модели и метрики')


# Inference resolution and independent-family audit (mandatory for scientific QA).
V74_PAIRED_AUDIT['eligible_family_size']=int(valid.sum())
V74_PAIRED_AUDIT['theoretical_mc_holm_floor']=float(min(1, int(valid.sum())/(CFG.PERMUTATION_REPS+1)))
V74_PAIRED_AUDIT['mc_resolution_adequate_for_holm']=bool(int(valid.sum())/(CFG.PERMUTATION_REPS+1)<CFG.INFERENCE_ALPHA)
V74_PAIRED_AUDIT['exact_low_cluster_comparisons']=int((V74_PAIRED_TESTS.inference_status=='EXACT_CLUSTER_SIGN_ENUMERATION').sum())
V74_PAIRED_AUDIT['mc_comparisons']=int((V74_PAIRED_TESTS.inference_status=='MONTE_CARLO_CLUSTER_SIGN_FLIP').sum())
V74_PAIRED_AUDIT['insufficient_cluster_comparisons']=int((V74_PAIRED_TESTS.inference_status=='INSUFFICIENT_INDEPENDENT_CLUSTERS').sum())
V74_PAIRED_AUDIT['statistics_rerun_without_retraining']=True
if not V74_PAIRED_AUDIT['mc_resolution_adequate_for_holm']:
    raise RuntimeError('INFERENCE: Holm significance impossible at configured Monte Carlo resolution')
# Strong manufactured signal regression: must yield p_Holm < 0.05 for 50 comparisons.
_v74_regression=v74_cluster_randomization(-np.ones(40), [f'cluster_{i}' for i in range(40)],
                                           reps=CFG.PERMUTATION_REPS,bootstrap_reps=99,seed=123)
assert _v74_regression['p_value']*50 < CFG.INFERENCE_ALPHA, 'INFERENCE: low res p-value; Holm impossible'
V74_PAIRED_AUDIT['strong_effect_holm_resolution_regression_passed']=True

qa_table('v74_paired_model_error_tests', V74_PAIRED_TESTS, allow_empty=False)
qa_table('v74_error_stratum_decomposition', V74_ERROR_STRATA, allow_empty=False)
qa_table('v74_prediction_pair_audit', pd.DataFrame([V74_PAIRED_AUDIT]))

print(
    f'V7.4 STATISTICS PASS: {len(V74_PAIRED_TESTS)} парных тестов; '
    f'{len(V74_ERROR_STRATA)} страт; Holm p-values: {int(valid.sum())}; '
    f'моделей: {len(MODEL_RECORDS)}; метрик: {len(METRICS)}',
    flush=True
)

mark_stage('STATISTICS')


## 11. Научные изображения
Диаграммы привязаны к исходному показателю и его периоду, без фиктивной геометрии и трактовки отсутствующей связи как нулевой.

In [ ]:
progress_event('VISUALS','Визуализация фактических E01, сетей и тестовых ошибок')
# Фигуры только на реально наблюдаемых исходных величинах; без псевдогеографических карт.
plt.rcParams.update({'font.family':'DejaVu Sans','figure.figsize':(10,5),'axes.grid':True})

def figure_save(stem):
    path=ROOT/'figures'/(stem+'.png')
    plt.tight_layout();plt.savefig(path,dpi=160,bbox_inches='tight');plt.close()
    FIGURES.append(str(path))
    return path

# Композиция берётся из неизменяемого PANELS, а не из локального counts (он изменяется в FEATURES).
od_comp=PANELS.get('od_composition',pd.DataFrame()).copy()
required={'observation_year','code_status','total_value'}
if not required.issubset(od_comp.columns):
    if {'observation_year','code_status','observed_value'}.issubset(OD.columns):
        od_comp=OD.groupby(['observation_year','code_status'],dropna=False,as_index=False).agg(total_value=('observed_value','sum'))
    else:fail('VISUALS','Потеряна первичная схема E01 / code_status')
if not od_comp.empty:
    od_comp['total_value']=pd.to_numeric(od_comp.total_value,errors='coerce')
    od_comp=od_comp.dropna(subset=['observation_year','code_status','total_value'])
    tab=od_comp.groupby(['observation_year','code_status']).total_value.sum().unstack(fill_value=0)
    tab=tab.reindex(columns=['INTER_REGION','INTRA_REGION','UNKNOWN_CODE'],fill_value=0).sort_index()
    if len(tab):
        fig,ax=plt.subplots(figsize=(10,5));tab.plot(kind='bar',stacked=True,ax=ax)
        ax.set(title='E01: состав зарегистрированного показателя (не постоянная миграция)',xlabel='Отчётный год',ylabel='Наблюдаемое значение E01')
        ax.legend(['Межрегиональные','Внутрирегиональные','Неопределённые коды'],fontsize=8)
        ax.tick_params(axis='x',rotation=0)
        figure_save('01_od_value_composition')
if len(coverage_df):
    cv=coverage_df.copy();cv['year']=pd.to_numeric(cv.year,errors='coerce');cv['observed_edges']=pd.to_numeric(cv.observed_edges,errors='coerce')
    cv=cv.dropna(subset=['year','observed_edges']).sort_values('year')
    if len(cv):
        fig,ax=plt.subplots(figsize=(10,4));ax.plot(cv.year,cv.observed_edges,'o-')
        flagged=cv.get('potential_break',pd.Series(False,index=cv.index)).fillna(False).astype(bool)
        for r in cv.loc[flagged].itertuples():
            ax.annotate('Изменение покрытия',(r.year,r.observed_edges),xytext=(0,12),textcoords='offset points',ha='center',fontsize=8)
        ax.set(title='E01: число зарегистрированных межрегиональных направлений',xlabel='Год',ylabel='Наблюдаемые пары регионов')
        ax.set_xticks(sorted(cv.year.astype(int).unique()))
        figure_save('02_edge_coverage')
if len(NETWORK):
    net=pd.DataFrame(NETWORK);year=int(net.year.max())
    x=net[net.year==year].nlargest(15,'in_strength').sort_values('in_strength')
    if len(x):
        fig,ax=plt.subplots(figsize=(11,6));ax.barh(x.region_code.astype(str),x.in_strength)
        ax.set(title=f'E01: крупнейшие регионы обучения, {year}',xlabel='Сумма исходного E01',ylabel='Код региона')
        figure_save('03_destination_centres')
if len(CONFIRM):
    q=CONFIRM[['module','baseline_mae','selected_mae']].copy()
    q['baseline_mae']=pd.to_numeric(q.baseline_mae,errors='coerce');q['selected_mae']=pd.to_numeric(q.selected_mae,errors='coerce')
    q=q.dropna();q=q[q.baseline_mae.gt(0)&q.selected_mae.ge(0)].copy()
    q['ratio']=q.selected_mae/q.baseline_mae;q=q[np.isfinite(q.ratio)]
    if len(q):
        fig,ax=plt.subplots(figsize=(10,5));at=np.arange(len(q));w=.34
        ax.bar(at-w/2,np.ones(len(q)),width=w,label='Инерция = 1,0')
        ax.bar(at+w/2,q.ratio,width=w,label='Выбранная до теста модель')
        ax.axhline(1,color='gray',linewidth=1,alpha=.7)
        ax.set_xticks(at);ax.set_xticklabels(q.module.astype(str).tolist())
        ax.set(ylabel='MAE / MAE инерции',title='Относительная ошибка на общих тестовых ключах')
        ax.legend();figure_save('04_mae_preselected_models')
print('Generated figures:',len(FIGURES))

mark_stage('VISUALS')


## 12. Экспорт: полный evidence в GitHub, аналитический Excel и воспроизводимый встроенный ноутбук

In [ ]:
memory_point('before_export')
progress_event('EXPORT','Excel, evidence-Parquet, модели, ноутбук и манифест')
import zipfile
from itertools import chain
EXCEL=ROOT/'exports'/'ML5_EDUCATION_MIGRATION_RESULTS.xlsx'
EVIDENCE=ROOT/'exports'/'ML5_EVIDENCE.zip'
NOTEBOOK=ROOT/'exports'/'ML5_EDUCATION_MIGRATION.ipynb'
MANIFEST=ROOT/'exports'/'run_manifest.json'

# Экспорт строго типизирован: смешанный int/string test_year не допускается.
STRING_HINTS={'entity_id','parent_group_id','origin_region','destination_region','region_code',
              'source_asset_id','oktmo','organization_id','indicator_name','test_period_label',
              'object_oktmo','specialty_code','area','campus_id','region_oktmo'}
INT_HINTS={'target_year','origin_year','observation_year','historical_year','historical_year_2',
           'information_cutoff_year','feature_available_year','target_available_year',
           'test_year','year','horizon','seed','graduation_year','salary_reference_year'}

def prepare_for_parquet(name,frame):
    f=frame.copy()
    f.columns=[str(c) for c in f.columns]
    if len(set(f.columns))!=len(f.columns):fail('EXPORT','Дубли названий столбцов '+name)
    if 'test_year' in f and f['test_year'].dtype in ('object','string'):
        original=f['test_year'].copy();numeric_year=number(original)
        descriptive=original.where(original.notna() & numeric_year.isna()).astype('string')
        if 'test_period_label' not in f:f['test_period_label']=descriptive.fillna('')
        else:f['test_period_label']=f['test_period_label'].astype('string').fillna(descriptive).fillna('')
        f['test_year']=numeric_year.astype('Int64')
    for c in f.columns:
        if c in INT_HINTS:
            z=number(f[c]);invalid=f[c].notna()&z.isna()
            if invalid.any():fail('PARQUET_SCHEMA',f'{name}.{c}: нечисловые годы/сиды')
            f[c]=z.astype('Int64')
        elif c in STRING_HINTS:f[c]=f[c].astype('string')
        elif pd.api.types.is_object_dtype(f[c]):
            sample=f[c].dropna()
            kinds={type(t) for t in sample.head(2000)}
            if not kinds or kinds<={str}:
                f[c]=f[c].astype('string')
            elif kinds<={int,float,np.int64,np.float64}:
                f[c]=number(f[c])
            elif any(isinstance(t,(dict,list,tuple,set)) for t in sample.head(2000)):
                f[c]=f[c].map(lambda v:json.dumps(v,ensure_ascii=False,default=str,sort_keys=True)
                                    if isinstance(v,(dict,list,tuple,set)) else None if v is None else str(v)).astype('string')
            elif all(isinstance(t,(str,bytes,int,float,complex,np.number)) for t in sample.head(2000)):
                # Смешанный административный код переводится в строку только как metadata,
                # строгие числовые признаки/метрики не трогаем.
                if re.search(r'(value|actual|predicted|MAE|RMSE|salary|rate|population)',c,re.I):
                    fail('PARQUET_SCHEMA',f'{name}.{c}: смешанный тип числового поля')
                f[c]=f[c].astype('string')
            else:
                f[c]=f[c].map(lambda v:str(v) if pd.notna(v) else None).astype('string')
    return f

# Регрессионный тест известной ArrowInvalid: числа и текстовый исторический период.
_schema_probe=prepare_for_parquet('group_protocols',pd.DataFrame({
    'test_year':[2025,'historical',pd.NA],
    'test_period_label':['',pd.NA,''],
    'entity_id':['001','002','003']}))
if str(_schema_probe.test_year.dtype)!='Int64' or _schema_probe.test_period_label.iloc[1]!='historical':
    fail('PARQUET_SCHEMA','Смешанные test_year снова повреждают метку исторического периода')
del _schema_probe

EVIDENCE_PARQUET={}
def parquet_pack(name,frame):
    progress_event('EXPORT',f'Parquet {name}: {len(frame):,} строк, порциями',announce=True)
    # Физически пишем по 10 тыс. строк; никакой pa.Table из полного DataFrame.
    clean=prepare_for_parquet(name,frame)
    path=ROOT/'derived'/(name+'.parquet')
    write_df_chunks(clean,path)
    pf=pq.ParquetFile(path)
    if pf.metadata.num_rows!=len(clean):fail('EXPORT','row count changed in '+name)
    if pf.schema_arrow.names!=list(clean.columns):fail('EXPORT','schema changed in '+name)
    sample=pf.read_row_group(0).slice(0,min(100,len(clean))).to_pandas()
    if len(sample)!=min(len(clean),100):fail('EXPORT','roundtrip failed '+name)
    EVIDENCE_PARQUET[name]={'rows':len(clean),'columns':len(clean.columns),
        'schema':str(pf.schema_arrow),'sha256':hash_file(path),'path':str(path)}
    del sample,clean,pf;gc.collect()
    return path

def excel_table(writer,source,sheet,cap=None):
    if not isinstance(source,pd.DataFrame) or source.empty:return False
    # XlsxWriter constant_memory + последовательная запись строк (pandas.to_excel с ним несовместим).
    cap=int(cap or CFG.EXCEL_PREVIEW_ROWS)
    x=source.head(cap)
    ws=writer.book.add_worksheet(sheet[:31]);ws.freeze_panes(1,0)
    header=writer.book.add_format({'bold':True,'bg_color':'#294862','font_color':'#FFFFFF','text_wrap':True,'valign':'vcenter'})
    ws.write_row(0,0,[str(c) for c in x.columns],header)
    ws.set_row(0,30)
    for j,c in enumerate(x.columns):ws.set_column(j,j,min(43,max(11,len(str(c))+2)))
    for idx,row in enumerate(x.itertuples(index=False,name=None),1):
        values=[]
        for v in row:
            if v is None or v is pd.NA:values.append(None)
            elif isinstance(v,(dict,list,tuple,set)):values.append(json.dumps(v,ensure_ascii=False,default=str))
            elif isinstance(v,(datetime,pd.Timestamp)):values.append(str(v))
            elif isinstance(v,(np.integer,)):values.append(int(v))
            elif isinstance(v,(np.floating,)):values.append(float(v) if np.isfinite(v) else None)
            elif isinstance(v,(int,float,bool,str)):values.append(v if not isinstance(v,float) or np.isfinite(v) else None)
            else:values.append(str(v))
        ws.write_row(idx,0,values)
    ws.autofilter(0,0,len(x),len(x.columns)-1)
    return True

V7_EXCEL_EXTRA=[
 ('v7_all52_scientific_source_audit','V7 научный аудит 52'),
 ('v7_od_common_support_sensitivity','V7 OD общие пары'),
 ('v7_organization_admission_discovery','V7 кандидаты приема'),
 ('v7_municipal_identification_gate','V7 муниципалитеты'),
 ('v7_source_lineage_52','V7 происхождение 52'),
 ('v7_model_lineage','V7 модели источники'),
 ('v7_seven_scientific_gates','V7 семь целей')]
EXCEL_TABLES=[
    ('v74_paired_model_error_tests','V74 парные модели'),
    ('v74_error_stratum_decomposition','V74 страты ошибок'),
    ('v74_prediction_pair_audit','V74 приёмка прогнозов'),
    ('adr_preprocessing_provenance','V71 происхождение реконструкций'),
    ('adr_od_pairwise_comparable','V71 OD сопоставимые пары'),
    ('adr_od_balanced_edges_by_year','V71 OD общие рёбра'),
    ('adr_organizational_admission_dictionary','V71 поиск приёма E02'),
    ('adr_admission_organization_year','V71 приём подтвержден'),
    ('adr_spo_geolocation_candidates','V71 география СПО'),
    ('adr_spo_geolocation_high_confidence','V71 география высокая'),
    ('adr_all_feasible_experiments','V71 все эксперименты'),
    ('adr_ml57_geographic_execution','V71 E0 E1 география'),
    ('adr_ml57_spatial_e0e1_metrics','V71 E0 E1 результаты'),
    ('adr_v71_seven_module_acceptance','V71 7 модулей'),
    ('adr_source_52_completeness','V71 источники 52'),
    ('adr_reconstructed_model_lineage','V71 происхождение моделей'),
    ('adr_final_scientific_diagnostics','V71 научные границы'),

 ('v4_all52_source_value_audit','V4 данные 52 из 52'),
 ('v4_all52_research_use','V4 использование 52'),
 ('v4_source_scan_failures','V4 ошибки источников'),
 ('v4_specialty_admission_gate','V4 прием спец. данные'),
 ('v4_specialty_model_frame','V4 прогноз спец. база'),
 ('v4_organization_contingent_candidates','V4 контингент ВО'),
 ('v4_organization_contingent_model_frame','V4 прогноз контингента'),
 ('v4_additional_subexperiments','V4 дополнительные задачи'),
 ('v4_ml57_indicator_support','V4 охват БДМО'),
 ('v4_ml57_exact_common_support','V4 E0 E1 состав'),
 ('v4_ml57_results','V4 E0 E1 метрики'),
 ('v4_ml57_execution','V4 E0 E1 статус'),
 ('v4_ml55_crosssection_metrics','V4 занятость тест'),
 ('v4_ml55_execution','V4 выпуск тест'),
 ('v4_ml56_node_forecast','V4 сеть тест'),
 ('v4_exploratory_paired_comparisons','V4 парные сравнения'),
 ('v4_methods_execution_ledger','V4 выполнение методов'),
 ('v4_rolling_origin_feasibility','V4 скользящие годы'),
 ('v4_scientific_acceptance','V4 научная приемка'),
 ('full52_source_inventory', 'Все 52 источника'),
 ('seven_module_status_final', 'Семь модулей итог'),
 ('ml51_spatial_protocols', 'Пространственный тест OD'),
 ('ml54_history_persistence_diagnostic', 'Тренды спец. диагностика'),
 ('ml52_admission_candidates', 'Кандидаты приема'),
 ('full52_year_coverage', 'Все годы источников'),
 ('full52_theme_coverage', 'Охват тематик'),
 ('full52_raw_originals', 'Оригинальные XLSX'),
 ('bdmo_education_candidates', 'БДМО образование'),
 ('municipal_infrastructure_candidates', 'Инфраструктура ОКТМО'),
 ('spatial_exact_crosswalk', 'География ОКТМО'),
 ('e04_e05_full_family_audit', 'Все массивы выпускников'),
 ('vpo_all_sections_audit', 'Все разделы ВПО'),
 ('spo_annual_vs_master', 'СПО годовые сверки'),
 ('d02_indicator_coverage', 'Экономика 11 источников'),
 ('module_52_extension', 'Доп модули'),
 ('ml54_specialty_history', 'История специальностей'),
 ('ml55_outcome_disparities', 'Выпускники различия'),
 ('ml56_network_stress', 'Устойчивость сети'),
 ('ml57_common_support','БДМО E0 E1 исходные'),

 ('stage1_sources','Источники Stage I'),('semantic_indicator_catalog','Семантика индикаторов'),
 ('admission_approval_register','Утверждение индикаторов'),('registry_coverage_audit','Охват реестров'),
 ('organization_sector_subtasks','ВО СПО итоги модулей'),
 ('e01_original_headers','E01 первичные заголовки'),('e01_raw_cell_sources','E01 raw-проверка'),
 ('od_composition','Состав OD'),('od_edge_coverage','Покрытие OD'),
 ('od_edge_transitions','Новые и исчезнувшие OD'),('edu_org_registry','Реестр организаций'),
 ('edu_org_year','Приём ВО СПО'),('vpo_admission_audit','Приём ВПО-1'),
 ('program_profile','Программы'),('program_concentration','Концентрация программ'),
 ('vpo_program_description','ВПО программы отдельно'),
 ('graduate_full_scan_audit','E04 E05 полное сканирование'),('graduate_descriptive_statistics','Занятость выпускников'),('graduate_specialty_description','Выпуск по специальностям'),('municipality_education_context','Муниципалитеты E0 E1'),
 ('economic_indicator_candidates','Экономика источники'),('regional_economy','Региональные факторы'),
 ('region_distance_proxy','Транспорт и расстояния'),('education_network_year','Сетевые показатели'),
 ('education_network_communities','Сообщества'),('network_stability','Динамика сети'),
 ('seven_module_status','Семь модулей'),('module_diagnostics','Статусы модулей'),
 ('seven_hypotheses','Семь гипотез'),('model_metrics','Метрики'),
 ('confirmatory_comparisons','Подтверждающие тесты'),('split_audit','Временные разбиения'),
 ('future_mutation_canaries','Защита от утечек'),('metrics_recomputed_from_predictions','Независимый пересчёт'),
 ('entity_match_quarantine','Спорные совпадения'),('graduate_outcomes','Выпускники исходные'),
 ('edu_od_interregional','Рёбра межрегиональные')]

EXCEL_TABLES += V7_EXCEL_EXTRA
EXCEL_TABLES += [
 ('v72_od_year_pair_coverage_decomposition','V72 OD годовые пары'),
 ('v72_org_admission_indicator_decision','V72 E02 приём семантика'),
 ('v72_origin_common_support_outflow','V72 отток общие рёбра'),
 ('v72_specialty_code_observed_crosswalk','V72 коды специальностей'),
 ('v72_graduate_family_independent_descriptives','V72 E04 E05 отдельно'),
 ('v72_network_same_node_edge_dynamics','V72 сеть общие связи'),
 ('v72_spo_municipal_proxy_confidence','V72 география карантин'),
 ('v72_all_seven_reconstruction_evidence','V72 предобработка'),
 ('v72_each_metric_asof_vintage_gate','V72 временные винтажи'),
 ('v72_ex_ante_model_selection_audit','V72 выбор моделей'),
 ('v72_verified_source_panel_model_links','V72 источник модель'),
 ('v72_seven_modules_scientific_acceptance','V72 семь модулей'),
 ('v72_engineering_decision_gates','V72 контроль программы')
]

passport=pd.DataFrame([
  {'Параметр':'Проект','Значение':'Демографический советник · ML-05 Образовательная миграция'},
  {'Параметр':'Версия','Значение':CFG.VERSION},
  {'Параметр':'Режим','Значение':CFG.MODE},
  {'Параметр':'Снапшот Stage I','Значение':SOURCE_COMMIT},
  {'Параметр':'Каталог Stage I','Значение':index_info['sha']},
  {'Параметр':'Наборы в каталоге','Значение':len(ASSETS)},
  {'Параметр':'Наблюдаемая OD-связка','Значение':'Регион общего образования → регион СПО; не является фактом переезда'},
  {'Параметр':'E01 зачисления','Значение':'Не подтверждены первичным определением'},
  {'Параметр':'Статус научного исполнения','Значение':STATE['scientific']},
  {'Параметр':'Ограничение временного теста','Значение':'Исторические source vintages не доступны; ретроспективная проверка'},
  {'Параметр':'Публикация','Значение':'Только FULL при локальной приёмке и явно включённом разрешении'},
])

# При повторном запуске после ошибки NOTEBOOK_CAPTURE не пересоздаём огромный evidence.
def _completed_evidence_available():
    if not all(p.is_file() and p.stat().st_size>100 for p in (EXCEL,EVIDENCE)):return False
    try:
        with zipfile.ZipFile(EVIDENCE) as z:
            cfg=json.loads(z.read('metadata/config.json'))
            names=z.namelist()
            return (cfg.get('VERSION')==CFG.VERSION and 'metadata/v5_module_acceptance.json' in names
                    and 'predictions/education_predictions.parquet' in names
                    and any(p.startswith('models/') for p in names)
                    and any(p.startswith('panels/') for p in names))
    except Exception:return False

_REUSE_EXPORT=_completed_evidence_available()
if _REUSE_EXPORT:
    progress_event('EXPORT','Готовые Excel/evidence V7.4.0 обнаружены: повторная упаковка не требуется',announce=True)
    # Восстанавливаем реестр Parquet, сброшенный в начале этой ячейки.
    with zipfile.ZipFile(EVIDENCE) as z:
        for member in z.namelist():
            if not member.endswith('.parquet') or not member.startswith(('panels/','predictions/')):continue
            key=Path(member).stem
            p=Path(PRED_STORAGE_FILE) if key=='education_predictions' and PRED_STORAGE_FILE else Path(PANEL_FILES.get(key,ROOT/'derived'/(key+'.parquet')))
            if not p.is_file():
                p=ROOT/'derived'/(key+'.parquet');p.parent.mkdir(parents=True,exist_ok=True)
                with z.open(member) as source,p.open('wb') as target:shutil.copyfileobj(source,target,4*1024**2)
            pf=pq.ParquetFile(p)
            EVIDENCE_PARQUET[key]={'rows':pf.metadata.num_rows,'columns':len(pf.schema_arrow.names),
              'schema':str(pf.schema_arrow),'sha256':hash_file(p),'path':str(p)}
    progress_event('EXPORT',f'Восстановлен реестр {len(EVIDENCE_PARQUET)} Parquet без пересчёта',announce=True)

if not _REUSE_EXPORT:
    progress_event('EXPORT','Сборка многолистового Excel',announce=True)
    with pd.ExcelWriter(EXCEL,engine='xlsxwriter',engine_kwargs={'options':{'constant_memory':True,'strings_to_formulas':False,'strings_to_urls':False}}) as writer:
        excel_table(writer,passport,'Паспорт')
        for _i,(key,title) in enumerate(EXCEL_TABLES,1):
            progress_event('EXPORT',f'Excel {_i}/{len(EXCEL_TABLES)}: {key}',_i,len(EXCEL_TABLES),announce=False)
            excel_table(writer,PANELS.get(key),title)
        excel_table(writer,pd.DataFrame(TABLE_QA),'Контроль таблиц')
        excel_table(writer,pd.DataFrame(ISSUES),'Ограничения')
        excel_table(writer,pd.DataFrame(LOG),'Журнал диагностики')

METHODOLOGY='''# ML-05 V7.4.0. Научный протокол

V7.4: дополнительные сравнения на одинаковых тестовых наблюдениях с усреднением 50 сидов.
Статистические выводы по кластерным перестановкам и Holm только разведочные, не исторический прогноз.

Восемь различных событий мобильности не отождествляются.
E01 region_of_previous_general_education × region_of_current_study НЕ является доказанным годовым приёмом.
Ретроспективные модели оценивают только измеренную E01 величину; нулевые отсутствующие рёбра не создаются.
Разбиения train/DEV/TEST подчинены доступности меток по консервативному лагу. Если DEV не был известен к test origin,
победителя из теста не выбирают; persistence остаётся предзаданной основной моделью.
Историческая версия источника 2026 не доказывает фактическую доступность значений в 2021–2025.
Сети описывают только наблюдаемые связи регионов предыдущего образования и обучения.
Выпускники E04/E05 — отдельный employment_reference 2024-12-31: место работы не известно, retention нельзя оценить.
Пересекающиеся географические/гендерные агрегаты E04/E05 нельзя суммировать, поэтому описаны медианы исходных строк по разрезам.
В пространственном E0/E1 отсутствие подтверждённой образовательной площадки не равно отсутствию площадки.
Экономические и транспортные признаки включаются только при доказанном grain и лаге.
Отрицательные результаты сохраняются, причинные эффекты по прогнозной важности не идентифицируются.
'''

if not _REUSE_EXPORT:
    progress_event('EXPORT','Сборка evidence-ZIP и контроль отдельных Parquet',announce=True)
    with zipfile.ZipFile(EVIDENCE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6,allowZip64=True) as z:
        z.writestr('research_methods.md',METHODOLOGY)
        z.writestr('metadata/config.json',json.dumps(asdict(CFG),ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/plan.json',json.dumps(EXPERIMENT_PLAN,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/semantics_e01.json',json.dumps(OD_SEMANTIC,ensure_ascii=False,indent=2))
        z.writestr('metadata/qa.json',json.dumps(TABLE_QA,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/source_hashes.json',json.dumps(INPUT_SHA,ensure_ascii=False,indent=2))
        z.writestr('metadata/issues.json',json.dumps(ISSUES,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/module_status.json',json.dumps(MODULES,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/splits.json',json.dumps(SPLIT_AUDIT,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/catalog_52_sha.json',json.dumps(INPUT_SHA,ensure_ascii=False,indent=2))
        z.writestr('metadata/v5_module_acceptance.json',V5_LEDGER.to_json(orient='records',force_ascii=False,indent=2))
        z.writestr('metadata/v5_model_registry.json',json.dumps(MODEL_RECORDS,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/catalog_52_protocol.md', 'Каждый из 52 Parquet: SHA-256, число строк, schema, годовой охват. '+'Годовые shards СПО не суммируются с общей панелью. Другие уровни данных не приписываются образовательному OD.')
        _export_panels=list(PANELS.items())
        for _i,(key,df) in enumerate(_export_panels,1):
            progress_event('EXPORT',f'Evidence {_i}/{len(_export_panels)}: {key}',_i,len(_export_panels),announce=False)
            if not isinstance(df,pd.DataFrame) or df.empty:continue
            if key in PANEL_FILES:
                p=Path(PANEL_FILES[key]);pf=pq.ParquetFile(p)
                EVIDENCE_PARQUET[key]={'rows':pf.metadata.num_rows,'columns':len(pf.schema_arrow.names),
                    'schema':str(pf.schema_arrow),'sha256':hash_file(p),'path':str(p)}
            else:p=parquet_pack(key,df)
            z.write(p,'panels/'+key+'.parquet')
        if PRED_STORAGE_FILE is not None:
            p=Path(PRED_STORAGE_FILE);pf=pq.ParquetFile(p)
            EVIDENCE_PARQUET['education_predictions']={'rows':pf.metadata.num_rows,'columns':len(pf.schema_arrow.names),
                'schema':str(pf.schema_arrow),'sha256':hash_file(p),'path':str(p)}
            z.write(p,'predictions/education_predictions.parquet')
        # Large source-grouped context: include every successfully built Parquet, not only Excel preview.
        for p in CONTEXT_PARQUET_FILES:
            if not Path(p).exists():fail('EXPORT','Отсутствует исходный контекстный Parquet '+str(p))
            z.write(p,'context/'+Path(p).name)
        for path in FIGURES:z.write(path,'figures/'+Path(path).name)
        for name,path in MODEL_FILES.items():z.write(path,'models/'+name)
        z.writestr('metadata/v74_paired_inference.json',json.dumps(V74_PAIRED_AUDIT,ensure_ascii=False,default=str,indent=2))
        z.writestr('metadata/v72_reproducibility.json',json.dumps(V72_REPRODUCIBILITY,ensure_ascii=False,indent=2,default=str))
        z.writestr('metadata/v72_seven_module_acceptance.json',V72_MODULE_ACCEPTANCE.to_json(orient='records',force_ascii=False,indent=2))
        z.writestr('logs/run_events.json',json.dumps(LOG,ensure_ascii=False,indent=2))
        z.write(ROOT/'logs'/'stages_v74.jsonl','logs/stages_v74.jsonl')

# V6 SOURCE CAPTURE: восстанавливаем полный код из самого ноутбука; Colab get_ipynb НЕ используется.
progress_event('EXPORT','Сохранение встроенного неизменяемого V7.4.0 notebook',announce=True)
snap=notebook_from_embedded_source()
for _cell in snap.cells:
    if _cell.cell_type=='code':
        _cell.outputs=[]
        _cell.execution_count=None
nbformat.validate(snap)
clean_text=nbformat.writes(snap)
_token=globals().get('_TOKEN')
if (isinstance(_token,str) and len(_token)>15 and _token in clean_text) or re.search(r'(?:ghp_|github_pat_)[A-Za-z0-9_]{20,}',clean_text):
    raise RuntimeError('V7.4.0 SECURITY: notebook contains GitHub token; export blocked')
_tmp_notebook=NOTEBOOK.with_suffix('.tmp.ipynb')
nbformat.write(snap,_tmp_notebook)
_tmp_notebook.replace(NOTEBOOK)
_saved_nb=nbformat.read(NOTEBOOK,as_version=4)
if [c.source for c in _saved_nb.cells]!=[c.source for c in snap.cells]:
    raise RuntimeError('V7.4.0: notebook roundtrip changed sources')
progress_event('EXPORT',f'V7.4.0 notebook saved, {len(snap.cells)} cells',announce=True)

# Manifest создаётся после локальных артефактов и содержит их реальные SHA; его собственный SHA нельзя включить в него же.
output_hashes={p.name:hash_file(p) for p in [EXCEL,EVIDENCE,NOTEBOOK] if p.exists()}
manifest={
 'version':CFG.VERSION,'mode':CFG.MODE,'created_utc':stamp(),
 'source_repo':CFG.REPO,'source_commit':SOURCE_COMMIT,'source_index_git_blob_sha':index_info['sha'],
 'source_index_version':INDEX['version'],'source_hashes':INPUT_SHA,
 'context_derived_files':{Path(p).name:{'rows':pq.ParquetFile(p).metadata.num_rows,'sha256':hash_file(p)} for p in CONTEXT_PARQUET_FILES},
 'catalog_52_coverage':{'registered':len(ASSETS),'sha_verified':len(INPUT_SHA),
 'verified_every_asset':len(INPUT_SHA)==len(ASSETS),
 'year_profiled':len(FULL52_YEAR_STATS)},
 'experiment_plan_sha256':PLAN_HASH,'v5_science_gates':{'all52_source_content_verified':not V4_SOURCE_ERRORS,'all_feasible_computations_succeeded':V5_COMPUTATIONAL_PASS,'all_seven_scientifically_identified':V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS,'model_failure_count':int((~V5_LEDGER.computational_pass).sum())},'status':STATE.copy(),'module_status':MODULES,
 'observed_od_counts':{'all':len(OD),'inter':len(OD_INTER),'intra':len(OD_INTRA),'unknown':len(OD_UNKNOWN)},
 'real_prediction_rows':PRED_TOTAL,'v5_model_file_count':len(MODEL_FILES),
 'v5_module_acceptance':V5_MODULE_ACCEPTANCE,
 'v5_original_seven_target_identifiable':V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS,'metric_rows':len(METRICS),'panels':{
      k:{'rows':v['rows'],'columns':v['columns'],'sha256':v['sha256']} for k,v in EVIDENCE_PARQUET.items()},
 'adr_advanced_data_reconstruction':ADR_FINAL_REPORT,
 'v72_reproducibility':V72_REPRODUCIBILITY,
 'v74_paired_inference_audit':V74_PAIRED_AUDIT,
 'v74_paired_comparisons':len(V74_PAIRED_TESTS),
 'v74_error_strata':len(V74_ERROR_STRATA),
 'v72_module_decisions':V72_MODULE_ACCEPTANCE.to_dict('records'),
 'v72_new_preprocessing':V72_RECON_DF.to_dict('records'),
 'adr_experiment_ledger':ADR_EXPERIMENTS,
 'output_sha256':output_hashes,'runtime_seconds':round(time.monotonic()-START_TIME,1),
 'limitations':ISSUES,'publication_status':'NOT_PUBLISHED',
 'commit_self_reference':'Commit SHA can be reported after Git commit; cannot be embedded in the same hashed blob'
}
progress_event('EXPORT','Манифест с SHA-256 локальных результатов',announce=True)
MANIFEST.write_text(json.dumps(manifest,ensure_ascii=False,default=str,indent=2),encoding='utf-8')
print('Local package:',EXCEL.name,round(EXCEL.stat().st_size/1e6,2),'MB;',
      EVIDENCE.name,round(EVIDENCE.stat().st_size/1e6,2),'MB; notebook:',NOTEBOOK.exists())

memory_point('after_export')
mark_stage('EXPORT')


## 13. Строгая локальная приёмка семи модулей, численных результатов и 52 источников

In [ ]:
progress_event('QA_AND_PUBLICATION','Локальная приёмка, воспроизводимость и безопасная публикация')
# Финальный QA запускается до первого GitHub-запроса на запись.
LOCAL_CHECKS=[]
def check(name,condition,why=''):
    ok=bool(condition)
    LOCAL_CHECKS.append({'check':name,'status':'PASS' if ok else 'FAIL','reason':why})
    if not ok:note('QA',name+': '+why,'FAIL')
    return ok

check('DUCKDB_SQL_PREFLIGHT_PRODUCTION_QUERIES',bool(globals().get('SQL_PREFLIGHT_PASSED')),
      'Production COPY, candidate SUMMARY and org-year TOTAL must parse and execute before Stage I download')
check('INPUT_SOURCE_PIN',bool(SOURCE_COMMIT and index_info['sha']==CFG.EXPECTED_INDEX_BLOB_SHA))
check('SEVEN_MODULES_REPORTED',len(MODULES)==7 and all(v['status']!='NOT_STARTED' for v in MODULES.values()))
check('EXPECTED_CATALOG',len(ASSETS)==52)
check('FULL52_SHA_COVERAGE',CFG.MODE!='FULL' or (len(INPUT_SHA)==len(ASSETS) and all(a['asset_id'] in INPUT_SHA for a in ASSETS)))
check('FULL52_ROW_COVERAGE',CFG.MODE!='FULL' or (len(FULL52_INVENTORY)==len(ASSETS) and all(r.get('footer_rows_ok') for r in FULL52_INVENTORY)))
check('FULL52_YEAR_DISTRIBUTION',CFG.MODE!='FULL' or all(r['year_scan_status'] in ('SCANNED_FULL','NO_TIME_KEY') for r in FULL52_INVENTORY))
check('FULL52_D02_PROFILE',CFG.MODE!='FULL' or (len(D02_PROFILE)==11 and all(r['status']=='GROUPED_ALL_SOURCE_ROWS_NO_DOUBLE_COUNT' for r in D02_PROFILE)))
check('FULL52_VPO_PROFILE',CFG.MODE!='FULL' or (len(VPO_ALL_PROFILE)==11 and all(r['status']=='PASS' for r in VPO_ALL_PROFILE)))
check('FULL52_E04_E05_PROFILE',CFG.MODE!='FULL' or (len(FAMILY_E04E05)==4 and all(r['status']=='PASS_NOT_ADDITIVE' for r in FAMILY_E04E05)))
check('FULL52_CONTEXT_EXPORT',CFG.MODE!='FULL' or len(CONTEXT_PARQUET_FILES)>=26)
check('V4_SOURCE_VALUES_52',CFG.MODE!='FULL' or (len(V4_SOURCE_PROFILES)==52 and not V4_SOURCE_ERRORS))
check('V4_MODEL_FAILURES_ZERO',V4_COMPUTATIONAL_COMPLETE,'Failed model or source value analysis is a real computational failure')
check('V5_COMPLETE_COMPUTATIONAL_CONTRACT',V5_COMPUTATIONAL_PASS,'All feasible models and checks must complete; no hidden failures')
check('V5_UNIQUE_MODEL_ARTIFACTS',len(MODEL_FILES)==len(MODEL_RECORDS),'No overwritten spatial model files')
check('V5_ALL_MODELS_VERIFIED',bool(V5_MODEL_QA) and all(x['sha_verified'] for x in V5_MODEL_QA))
check('V5_SEVEN_MODULE_LEDGER',len(V5_LEDGER)==7 and V5_LEDGER.module.nunique()==7)
check('V5_ALL52_SOURCE_EVIDENCE',CFG.MODE!='FULL' or len(V5_SOURCE_EVIDENCE)==52)
check('V5_ML54_NO_SAMPLE',CFG.MODE!='FULL' or (len(PANELS.get('v5_ml54_full_strata_coverage',pd.DataFrame()))==1))
check('V4_FINAL_LEDGER',len(V4_METHOD_EXECUTIONS)==7)
check('V4_SCIENTIFIC_TRUTHFUL',not V4_SCIENTIFIC_COMPLETE or (len(ORG_YEAR)>0 and OD_SEMANTIC.get('yearly_count_unit_confirmed',False)))
check('E01_PRIMARY_OPENED',CFG.MODE!='FULL' or bool(OD_SEMANTIC.get('original_opened')))
check('SEMANTIC_APPROVAL_GATED',not len(ORG_YEAR) or all(
    name in set(CFG.APPROVED_VO_ADMISSION_NAMES)|set(CFG.APPROVED_SPO_ADMISSION_NAMES)
    for name in ORG_YEAR.indicator_name.dropna().astype(str).unique()))
check('EDGES_PARTITION',len(OD)==len(OD_INTER)+len(OD_INTRA)+len(OD_UNKNOWN))
check('INTRA_EXCLUDED',OD_INTER.origin_region.ne(OD_INTER.destination_region).all())
check('NO_FUTURE_FEATURES',all(v.get('status')!='FAIL' for v in CANARIES))
check('CANARIES_EXECUTED',any(v.get('module')=='ML-5.1' and v.get('status')=='PASS' for v in CANARIES) and
      any(v.get('module')=='ML-5.3' and v.get('status')=='PASS' for v in CANARIES))
check('NO_TEST_SELECTED_MODEL',not len(CONFIRM) or
      (CONFIRM.choice_basis.isin(['DEV_ASOF','NO_CONFIRMATORY_COMPARISON_NO_DEV']).all() and
       CONFIRM.loc[CONFIRM.choice_basis.eq('NO_CONFIRMATORY_COMPARISON_NO_DEV'),
                   'selected_model'].eq('persistence').all()))
check('NEGATIVE_FINDINGS_PRESERVED',len(HYPOTHESES)==7)
check('ZIP_CRC',zipfile.ZipFile(EVIDENCE).testzip() is None)
with zipfile.ZipFile(EXCEL) as z:
    check('EXCEL_OPENS',z.testzip() is None and 'xl/workbook.xml' in z.namelist())
check('MODEL_RELOAD',all(m.get('reloaded') for m in MODEL_RECORDS))
check('NOTEBOOK_CAPTURED',NOTEBOOK.exists(),'Необходимо получить очищенный самостоятельный Colab notebook')
if NOTEBOOK.exists():
    try:
        nb=nbformat.read(NOTEBOOK,as_version=4)
        nbformat.validate(nb)
        content=NOTEBOOK.read_text(encoding='utf-8')
        suspicious=re.findall(r'\b(?:ghp_|github_pat_)[A-Za-z0-9_]{20,}\b',content)
        check('NOTEBOOK_NO_TOKEN',not suspicious and _TOKEN not in content)
        check('NOTEBOOK_NO_OUTPUTS',sum(len(c.get('outputs',[])) for c in nb.cells if c.cell_type=='code')==0)
        check('NOTEBOOK_SELF_CONTAINED',not re.search(r'(?m)^\s*(?:%run|%%writefile)\b',content))
        check('NOTEBOOK_REQUIRED_SECTION_COUNT',sum(c.cell_type=='code' for c in nb.cells)>=12)
        check('NOTEBOOK_REASONABLE_SIZE',NOTEBOOK.stat().st_size<2_000_000)
    except Exception as ex:check('NOTEBOOK_VALIDATE',False,type(ex).__name__)

# Проверяем независимые Parquet исходя из физического ZIP, не доверяя только собственным логам.
with zipfile.ZipFile(EVIDENCE) as z:
    for name,info in EVIDENCE_PARQUET.items():
        target=('predictions/' if name=='education_predictions' else 'panels/')+name+'.parquet'
        if target not in z.namelist():
            check('PARQUET_'+name,False,'Нет файла в evidence');continue
        try:
            # Не z.read(): весь Parquet в bytes может мгновенно исчерпать RAM.
            p=Path(info['path']);pf=pq.ParquetFile(p)
            member=z.getinfo(target)
            check('PARQUET_'+name,int(pf.metadata.num_rows)==info['rows'] and
                  len(pf.schema_arrow.names)==info['columns'] and
                  int(member.file_size)==p.stat().st_size and hash_file(p)==info['sha256'])
        except Exception as ex:check('PARQUET_'+name,False,type(ex).__name__)

# V7: repeat independent source/model lineage checks on physical exported tables.
check('V7_SOURCE_AUDIT_ALL52',CFG.MODE!='FULL' or
      (len(V7_SOURCE_AUDIT)==52 and V7_SOURCE_AUDIT.verified_sha256.all() and
       V7_SOURCE_AUDIT.all_values_profiled.all() and V7_SOURCE_AUDIT.numeric_profile_exists.all()))
check('V7_LINEAGE_52_UNIQUE',len(V7_SOURCE_LINEAGE)==len(ASSETS) and
      V7_SOURCE_LINEAGE.asset_id.nunique()==len(ASSETS) and
      V7_SOURCE_LINEAGE.source_coverage_pass.all())
check('V7_ALL_MODELS_LINKED',len(V7_MODEL_LINEAGE)==len(MODEL_RECORDS) and
      (V7_MODEL_LINEAGE.n_sources.ge(1).all() if len(V7_MODEL_LINEAGE) else True),
      'Every recorded model needs a traceable source input')
check('V7_ORIGINAL_7_GATES_REPORTED',len(V7_SCIENCE_GATES)==7 and
      V7_SCIENCE_GATES.module.nunique()==7 and
      V7_SCIENCE_GATES.computational_pass.all())
check('V7_OD_COMPARABLE_COMMON_SUPPORT',bool(len(V7_OD_COMPARABLE)) and
      all(x['shared_edges']<=min(x['observed_edges_previous'],x['observed_edges_current'])
          for x in V7_OD_COMPARABLE))
check('V71_PUBLISH_CHECK_SEPARATED_FROM_LOCAL_QA',
      bool(MANIFEST.is_file() and EXCEL.is_file() and EVIDENCE.is_file() and
           manifest.get('version')==CFG.VERSION),
      'Remote publication is checked only after uploading and reading the GitHub Release')
check('V71_ADVANCED_SOURCE_COVERAGE',len(ADR_SOURCE_AUDIT)==52 and ADR_SOURCE_AUDIT.is_source_checked.all())
check('V71_SEVEN_MODULE_RECONSTRUCTION_LEDGER',len(ADR_SEVEN_MODULE_FINAL)==7 and ADR_SEVEN_MODULE_FINAL.module.nunique()==7)
check('V71_RECONSTRUCTION_MODEL_AND_SEMANTICS',ADR_COMPUTATIONAL_PASS,
      'All actually executed reconstruction models passed; missing source targets are explicitly skipped')
check('V71_NO_INVENTED_ADMISSIONS',not len(ADR_ADMISSION_YEAR) or bool(CFG.APPROVED_VO_ADMISSION_NAMES))
check('V71_NO_INVENTED_CAMPUSES',not len(ADR_GEO_VERIFIED) or
      ADR_GEO_VERIFIED.identity_status.eq('PROXIMITY_MATCH_NOT_OFFICIAL_CAMPUS_LOCATION').all())

# V7.4 · new independent gates (scientific limits are not technical failures).
for _name,_passed in V72_ENGINEERING_GATES.items():
    check('V72_'+_name.upper(),_passed,'Independently computed source, model and scientific contract')
check('V72_RESEARCH_MODULES_SEVEN',len(V72_MODULE_ACCEPTANCE)==7)
check('V72_RECONSTRUCTIONS_RECORDED',len(V72_RECON_DF)>=7)
check('V72_SOURCE_VINTAGE_NOT_MISLABELED',
      len(V72_ASOF_AUDIT)==len(METRICS) and
      not V72_MODEL_SELECTION.historical_source_vintage_validated.any(),
      'Retrospective results must not be relabeled as real-time historical forecasts')
check('V74_RELEASE_PLAN_PREFLIGHT',bool(globals().get('V74_PUBLICATION_PLAN_PREFLIGHT_PASSED')), '8 assets path validation must pass before input download')
check('V74_REAL_PAIRED_TESTS',len(V74_PAIRED_TESTS)>0 and V74_PAIRED_TESTS.n_paired_observations.ge(1).all())
check('V74_MODEL_RESULT_PRESERVATION',bool(V74_PAIRED_AUDIT.get('training_artifacts_unchanged')) and len(MODEL_RECORDS)>=233 and len(METRICS)>=254)
check('V74_PAIRED_SOURCE_COVERAGE',V74_PAIRED_AUDIT.get('prediction_rows_input')==PRED_TOTAL and V74_PAIRED_AUDIT.get('predeclared_baseline_groups',0)>0)
check('V74_INFERENCE_MULTIPLICITY',bool(V74_PAIRED_TESTS.p_holm.dropna().between(0,1).all()))
check('V74_PERMUTATION_RESOLUTION_HOLM',
      V74_PAIRED_AUDIT.get('mc_resolution_adequate_for_holm') and
      V74_PAIRED_AUDIT.get('strong_effect_holm_resolution_regression_passed') and
      CFG.PERMUTATION_REPS >= 4999 and
      V74_PAIRED_AUDIT.get('eligible_family_size') == int(V74_PAIRED_TESTS.p_holm.notna().sum()),
      'All qualified comparisons share Holm family; Monte Carlo p-resolution cannot preclude 5% significance')
check('V74_CLUSTER_RESAMPLING_ACTUALLY_EXECUTED',
      bool(V74_PAIRED_TESTS.inference_status.isin([
          'EXACT_CLUSTER_SIGN_ENUMERATION','MONTE_CARLO_CLUSTER_SIGN_FLIP',
          'INSUFFICIENT_INDEPENDENT_CLUSTERS']).all()) and
      V74_PAIRED_AUDIT.get('sign_flip_repetitions') == CFG.PERMUTATION_REPS and
      V74_PAIRED_AUDIT.get('statistics_rerun_without_retraining'),
      'Exact signs for small independent clusters; 9,999 Monte Carlo permutations otherwise')
_v74_stage_file=ROOT/'logs'/'stages_v74.jsonl'
_v74_stage_entries=[json.loads(x) for x in _v74_stage_file.read_text(encoding='utf-8').splitlines()] if _v74_stage_file.is_file() else []
_v74_session_stages=[x['stage'] for x in _v74_stage_entries if x.get('run_session_id')==RUN_SESSION_ID]
check('V74_DURABLE_STAGE_JOURNAL',_v74_session_stages==list(PIPELINE_STAGES)[:13],
      'Expected 13 completed research/export stages in this run session before QA and GitHub')
LOCAL_QA=pd.DataFrame(LOCAL_CHECKS)
progress_event('QA_AND_PUBLICATION',f'Проверено {len(LOCAL_QA):,} контролей; ошибок: {int(LOCAL_QA.status.eq("FAIL").sum())}',announce=True)
# Перезаписываем локальный manifest после QA, но Excel/ZIP уже фиксированы по SHA.
manifest['local_qa']=LOCAL_CHECKS
manifest['status']['technical']='PASS' if LOCAL_QA.status.eq('PASS').all() else 'FAIL'
manifest['status']['publication']='NOT_ATTEMPTED'
manifest['status']['overall']=(STATE['scientific'] if manifest['status']['technical']=='PASS'
                              else 'PARTIAL_LOCAL_QA')
MANIFEST.write_text(json.dumps(manifest,ensure_ascii=False,default=str,indent=2),encoding='utf-8')
print('Independent QA:',int(LOCAL_QA.status.eq('PASS').sum()),'PASS /',len(LOCAL_QA),'checks')

# Публикация отсюда не выполняется, даже при FAIL локальной QA.
# Новый финальный этап публикует артефакты с честным статусом QA; сами данные не теряются.
PUBLICATION_RECEIPT={'status':'PENDING_FINAL_STAGE'}
STATE['publication']='PENDING_FINAL_STAGE'

print('ML-05 V7.4.0:',STATE['scientific'],'| technical:',manifest['status']['technical'],
      '| GitHub:',STATE['publication'])
print('Output paths:',EXCEL,EVIDENCE,NOTEBOOK,MANIFEST)

mark_stage('QA_AND_PUBLICATION')
# Heartbeat работает и во время сборки/загрузки итогового ZIP; закрываем после последней ячейки.


## Приёмочный контракт V7.4.0

- **TECHNICAL_PASS**: все 52 исходных набора проверены по SHA, 7 модулей имеют журналы исполнения, обязательные по данным расчёты завершены, модельные файлы сверены по SHA и локальная QA без ошибок.
- **SCIENTIFIC_LIMITED**: измеряемая цель не идентифицируется в первичных данных. Этот статус честно сохраняется в отчёте и не подменяется иной величиной.
- **REPLACE**: если локальные вычисления и QA успешны, публикуется новый Git-коммит, заменяющий только целевой ML-05 сегмент. При техническом FAIL полные результаты могут быть сохранены как черновой Release, но старая рабочая версия не удаляется.


- **SQL_PREFLIGHT**: рабочие DuckDB-запросы E02 сначала исполняются на маленьком наборе, затем на полном источнике; ошибка синтаксиса приводит к остановке до многогигабайтных операций.

## 14. Итог: компактная автозагрузка отчёта → проверенный GitHub Release → атомарная замена старого ML-05

In [ ]:
# ML-05 V7.4.0 · COMPLETE PIPELINE / VERIFIED GITHUB REPLACEMENT / COMPACT DOWNLOAD
# FULL -> 52 SHA verified -> seven modules -> QA -> small report -> draft Release -> atomic segment replace.
# Does not delete Stage I, unrelated projects or historical git commits. Every destructive change is prefix-scoped.
import json, os, math, hashlib, base64, zipfile, shutil, re
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

progress_event('DOWNLOAD_RESULTS','V7.4: итоговый QA, компактная аналитика и безопасная замена ML-05',announce=True)
if CFG.MODE != 'FULL': raise RuntimeError('V7.4: для публикации необходим FULL')
if not CFG.PUBLISH_RESULTS_TO_PRIVATE_GITHUB: raise RuntimeError('V7.4: публикация в приватный GitHub выключена')
if not CFG.STORE_DERIVED_RESTRICTED_DATA: raise RuntimeError('V7.4: не разрешено публиковать производные результаты')
if not api_json('GET',BASE).get('private'): raise RuntimeError('V7.4: репозиторий не приватный')
if len(ASSETS)!=52 or len(INPUT_SHA)!=52 or len(V5_SOURCE_EVIDENCE)!=52:
    raise RuntimeError('V7.4: 52 источника не подтверждены SHA и профилями')
if len(V5_LEDGER)!=7 or V5_LEDGER.module.nunique()!=7:
    raise RuntimeError('V7.4: отсутствует одна или несколько приёмок семи модулей')

out=ROOT/'exports'; out.mkdir(parents=True,exist_ok=True)
needed=[Path(p) for p in (EXCEL,EVIDENCE,NOTEBOOK,MANIFEST)]
for p in needed:
    if not p.is_file() or p.stat().st_size==0: raise RuntimeError('V7.4: отсутствует файл '+str(p))
_saved=json.loads(Path(MANIFEST).read_text(encoding='utf-8'))
if _saved.get('version')!=CFG.VERSION or _saved.get('source_commit')!=SOURCE_COMMIT:
    raise RuntimeError('V7.4: манифест не соответствует активному запуску')
for p in (EXCEL,EVIDENCE,NOTEBOOK):
    if _saved.get('output_sha256',{}).get(Path(p).name)!=hash_file(p):
        raise RuntimeError('V7.4: нарушена контрольная сумма '+str(p))

# Source capture is a deterministic embedded copy of this notebook, not a stale Colab get_ipynb.
recovered=notebook_from_embedded_source()
if hash_file(NOTEBOOK)!=hashlib.sha256(nbformat.writes(recovered).encode('utf-8')).hexdigest():
    # nbformat serialization may differ in insignificant line breaks. Revalidate actual source instead.
    _check=nbformat.read(NOTEBOOK,as_version=4)
    if sum(c.cell_type=='code' for c in _check.cells)!=sum(c.cell_type=='code' for c in recovered.cells):
        raise RuntimeError('V7.4: сохранённый ноутбук не совпал с встроенным источником')
    _check_src=[c.source for c in _check.cells]
    if _check_src != [c.source for c in recovered.cells]:
        raise RuntimeError('V7.4: исходный код ноутбука отличается от эталона')

sha_evidence=hash_file(EVIDENCE)
run_key=hashlib.sha256((CFG.VERSION+'|'+SOURCE_COMMIT+'|'+sha_evidence).encode()).hexdigest()[:20]
release_tag='ml05-v7.4.0-full-'+run_key
prefix=CFG.OUTPUT_PREFIX.rstrip('/')+'/'
if prefix!='stage2/ml05_education/': raise RuntimeError('V7.4: удаление разрешено ТОЛЬКО в stage2/ml05_education/')
qa_ok=bool(len(LOCAL_QA) and LOCAL_QA.status.eq('PASS').all())
if not V74_PAIRED_AUDIT.get('mc_resolution_adequate_for_holm', False):
    raise RuntimeError('V7.4: incomplete inference resolution; refusing GitHub publication')
compute_ok=bool(V5_COMPUTATIONAL_PASS and ADR_COMPUTATIONAL_PASS)
science_ok=bool(V5_ORIGINAL_SEVEN_SCIENTIFIC_PASS)
can_replace=bool(qa_ok and compute_ok)

# V7 source/model lineage is derived from exact input panels and stored feature lists.
# No false claim that all 52 sources enter predictive X; all 52 have full numerical research profiles.
source_ledger=V7_SOURCE_LINEAGE.to_dict('records')
adr_ledger_csv=out/'ML5_V740_RECONSTRUCTED_EXPERIMENTS.csv'
pd.DataFrame(ADR_EXPERIMENTS).to_csv(adr_ledger_csv,index=False,encoding='utf-8-sig')
adr_summary_file=out/'ML5_V740_RECONSTRUCTION_ACCEPTANCE.json'
adr_summary_file.write_text(json.dumps(ADR_FINAL_REPORT,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
adr_lineage_csv=out/'ML5_V740_RECONSTRUCTED_MODEL_LINEAGE.csv'
ADR_RECONSTRUCTED_LINEAGE.to_csv(adr_lineage_csv,index=False,encoding='utf-8-sig')
source_csv=out/'ML5_V740_SOURCE_COVERAGE_52.csv'
V7_SOURCE_LINEAGE.to_csv(source_csv,index=False,encoding='utf-8-sig')
lineage_csv=out/'ML5_V740_MODEL_LINEAGE.csv'
V7_MODEL_LINEAGE.to_csv(lineage_csv,index=False,encoding='utf-8-sig')
gates_csv=out/'ML5_V740_SEVEN_SCIENTIFIC_GATES.csv'
V7_SCIENCE_GATES.to_csv(gates_csv,index=False,encoding='utf-8-sig')
source_stats_csv=out/'ML5_V740_SOURCE_RESEARCH_52.csv'
V7_SOURCE_AUDIT.to_csv(source_stats_csv,index=False,encoding='utf-8-sig')
module_csv=out/'ML5_V740_SEVEN_MODULES.csv';V5_LEDGER.to_csv(module_csv,index=False,encoding='utf-8-sig')
metrics_csv=out/'ML5_V740_METRICS.csv';pd.DataFrame(METRICS).to_csv(metrics_csv,index=False,encoding='utf-8-sig')
models_csv=out/'ML5_V740_MODELS.csv';pd.DataFrame(V5_MODEL_QA).to_csv(models_csv,index=False,encoding='utf-8-sig')
qa_csv=out/'ML5_V740_TECHNICAL_QA.csv';LOCAL_QA.to_csv(qa_csv,index=False,encoding='utf-8-sig')
issues_csv=out/'ML5_V740_SCIENTIFIC_LIMITATIONS.csv';pd.DataFrame(ISSUES).to_csv(issues_csv,index=False,encoding='utf-8-sig')
# V7.4 · reproducibility and independent research acceptance documents.
v72_files={}
for _name,_df in {
    'V72_RECONSTRUCTION_EVIDENCE':V72_RECON_DF,
    'V72_SOURCE_PANEL_MODEL':V72_MODEL_PROVENANCE,
    'V72_SEVEN_MODULE_DECISIONS':V72_MODULE_ACCEPTANCE,
    'V72_ASOF_VINTAGE':V72_ASOF_AUDIT,
    'V72_MODEL_SELECTION':V72_MODEL_SELECTION,
    'V72_OD_COVERAGE':V72_OD_COVERAGE,
    'V72_ADMISSION_DECISIONS':V72_ADMISSION_DECISION,
    'V72_GEO_CONFIDENCE':V72_SPATIAL_CONFIDENCE,
    'V72_ENGINEERING_GATES':pd.DataFrame([{'gate':k,'passed':v} for k,v in V72_ENGINEERING_GATES.items()])
}.items():
    _path=out/('ML5_'+_name+'.csv')
    _df.to_csv(_path,index=False,encoding='utf-8-sig')
    v72_files[_name]=_path
_v72_repro_file=out/'ML5_V72_REPRODUCIBILITY.json'
_v72_repro_file.write_text(json.dumps(V72_REPRODUCIBILITY,ensure_ascii=False,default=str,indent=2),encoding='utf-8')
v72_files['V72_REPRODUCIBILITY']=_v72_repro_file


v74_paired_csv=out/'ML5_V740_PAIRED_TESTS.csv'
V74_PAIRED_TESTS.to_csv(v74_paired_csv,index=False,encoding='utf-8-sig')
v74_strata_csv=out/'ML5_V740_ERROR_STRATA.csv'
V74_ERROR_STRATA.to_csv(v74_strata_csv,index=False,encoding='utf-8-sig')
v74_audit_json=out/'ML5_V740_PAIRED_AUDIT.json'
v74_audit_json.write_text(json.dumps(V74_PAIRED_AUDIT,ensure_ascii=False,default=str,indent=2),encoding='utf-8')

summary={
    'version':CFG.VERSION,'run_key':run_key,'git_source_commit':SOURCE_COMMIT,
    'catalog_assets':len(ASSETS),'sha_verified':len(INPUT_SHA),
    'content_profiled':sum(x.get('content_status')=='FULL_VALUE_COVERAGE_PASS' for x in V4_SOURCE_PROFILES),
    'direct_model_sources':int(V7_SOURCE_LINEAGE.direct_model_input.sum()),
    'model_lineage_records':len(V7_MODEL_LINEAGE),'source_profiles_researched':int(V7_SOURCE_AUDIT.all_values_profiled.sum()),
    'seven_modules':V5_MODULE_ACCEPTANCE,'v7_scientific_gates':V7_SCIENCE_GATES.to_dict('records'),'qa_pass':int(LOCAL_QA.status.eq('PASS').sum()),
    'qa_fail':int(LOCAL_QA.status.eq('FAIL').sum()),'computational_pass':can_replace,
    'original_seven_targets_scientifically_identified':science_ok,
    'prediction_rows':PRED_TOTAL,'metric_rows':len(METRICS),'model_count':len(MODEL_FILES),
    'evidence_sha256':sha_evidence,'evidence_bytes':EVIDENCE.stat().st_size,
    'scientific_caveat':'Невозможный по имеющимся данным прогноз не подменяется иным целевым показателем.',
    'adr_reconstruction':ADR_FINAL_REPORT,
    'v72_engineering_gates':V72_ENGINEERING_GATES,
    'v74_publication_preflight':V74_PUBLICATION_PLAN_PREFLIGHT_PASSED,
    'v72_scientific_module_acceptance':V72_MODULE_ACCEPTANCE.to_dict('records'),
    'v72_reproduction_fingerprint':V72_REPRODUCIBILITY,
    'v74_paired_tests':V74_PAIRED_AUDIT
}
summary_file=out/'ML5_V740_FINAL_ACCEPTANCE.json'
summary_file.write_text(json.dumps(summary,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

# Reports are downloaded before remote upload; complete evidence and heavy models NEVER enter this ZIP.
plots=sorted({Path(p) for p in FIGURES if Path(p).is_file()},key=lambda p:p.name)
reports=[Path(EXCEL),Path(NOTEBOOK),Path(MANIFEST),source_csv,lineage_csv,gates_csv,source_stats_csv,module_csv,metrics_csv,models_csv,qa_csv,issues_csv,summary_file,adr_ledger_csv,adr_summary_file,adr_lineage_csv]+list(v72_files.values())+[v74_paired_csv,v74_strata_csv,v74_audit_json]+plots
small_zip=out/('ML5_V740_ANALYTICS_'+run_key+'.zip')
with zipfile.ZipFile(small_zip,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
    for p in reports:
        z.write(p,('figures/' if p in plots else 'reports/')+p.name)
    z.writestr('reports/checksums.json',json.dumps({p.name:hash_file(p) for p in reports},indent=2))
with zipfile.ZipFile(small_zip) as z:
    if z.testzip() is not None: raise RuntimeError('V7.4: повреждён отчётный ZIP')
if small_zip.stat().st_size>100*2**20:raise RuntimeError('V7.4: отчётный архив >100 MiB, проверьте содержимое')
progress_event('DOWNLOAD_RESULTS',f'V7.4: готов отчётный архив {small_zip.stat().st_size/2**20:.1f} МиБ',announce=True)
print('АНАЛИТИКА:',small_zip,'|','EVIDENCE только в GitHub:',EVIDENCE.stat().st_size/2**20,'МиБ',flush=True)
if CFG.DOWNLOAD_COMPACT_REPORTS:
    try:
        from google.colab import files as colab_files
        colab_files.download(str(small_zip))
    except ImportError:
        from IPython.display import FileLink,display
        display(FileLink(str(small_zip)))

# Precompute resumable chunks on disk; verify recombined SHA before GitHub upload.
size_part=int(CFG.GH_RELEASE_PART_MIB)*2**20
if not 16<=int(CFG.GH_RELEASE_PART_MIB)<=256:raise RuntimeError('V7.4: part size вне 16..256 MiB')
chunk_dir=out/('release_chunks_'+run_key);chunk_dir.mkdir(parents=True,exist_ok=True)
chunk_index={'version':CFG.VERSION,'source_evidence':Path(EVIDENCE).name,
    'evidence_sha256':sha_evidence,'evidence_bytes':EVIDENCE.stat().st_size,'parts':[]}
with Path(EVIDENCE).open('rb') as src:
    total=math.ceil(EVIDENCE.stat().st_size/size_part)
    for i in range(total):
        name=f'ML5_EVIDENCE.zip.part{i+1:03d}'
        p=chunk_dir/name
        n=min(size_part,EVIDENCE.stat().st_size-i*size_part)
        if p.exists() and p.stat().st_size==n:
            src.seek(n,1)
        else:
            tmp=p.with_suffix('.part.tmp')
            with tmp.open('wb') as dest:
                remaining=n
                while remaining:
                    buf=src.read(min(4*2**20,remaining))
                    if not buf:raise IOError('V7.4: преждевременный конец evidence')
                    dest.write(buf);remaining-=len(buf)
            tmp.replace(p)
        chunk_index['parts'].append({'name':name,'bytes':n,'sha256':hash_file(p)})
        progress_event('DOWNLOAD_RESULTS',f'Локальная часть evidence {i+1}/{total}',announce=True)
verify_hash=hashlib.sha256()
for part in chunk_index['parts']:
    with (chunk_dir/part['name']).open('rb') as f:
        for buf in iter(lambda:f.read(4*2**20),b''):verify_hash.update(buf)
if verify_hash.hexdigest()!=sha_evidence:raise RuntimeError('V7.4: разбитый evidence SHA не совпадает с исходным')
index_file=out/'ML5_V740_RELEASE_PARTS.json'
index_file.write_text(json.dumps(chunk_index,ensure_ascii=False,indent=2),encoding='utf-8')

# Helpers for GitHub. An existing asset is reused ONLY if GitHub's SHA-256 and size match.
def remote_release_assets(release_id):
    return {a['name']:a for a in api_json('GET',BASE+f'/releases/{release_id}/assets',params={'per_page':100})}

def upload_release_asset(release_id,p,mime):
    p=Path(p);sha=hash_file(p)
    already=remote_release_assets(release_id).get(p.name)
    if already and already.get('state')=='uploaded' and already.get('digest')=='sha256:'+sha and int(already.get('size',-1))==p.stat().st_size:
        progress_event('DOWNLOAD_RESULTS','GitHub SHA уже совпадает: '+p.name,announce=True)
        return already
    if already:
        resp=SESS.delete(BASE+'/releases/assets/'+str(already['id']),timeout=(30,120))
        status=resp.status_code;resp.close()
        if status!=204:raise RuntimeError('V7.4: не удалён повреждённый Release-asset '+p.name)
    upload_url=f'https://uploads.github.com/repos/{CFG.REPO}/releases/{release_id}/assets'
    progress_event('DOWNLOAD_RESULTS',f'GitHub: {p.name} · {p.stat().st_size/2**20:.1f} МиБ',announce=True)
    try:
        with p.open('rb') as f:
            resp=SESS.post(upload_url,params={'name':p.name},headers={'Content-Type':mime},data=f,timeout=(30,3600))
        status=resp.status_code
        if status==201:obj=resp.json()
        else:obj=None
        resp.close()
    except Exception:
        obj=None
    if obj is None:
        obj=remote_release_assets(release_id).get(p.name)
    if not obj or obj.get('state')!='uploaded' or obj.get('digest')!='sha256:'+sha or int(obj.get('size',-1))!=p.stat().st_size:
        raise RuntimeError('V7.4: ошибка GitHub asset '+p.name+'; можно повторить только финальную ячейку')
    return obj

branch=CFG.TARGET_BRANCH
remote_head=repo_head(branch)
if remote_head!=TARGET_HEAD_AT_PREFLIGHT:
    # A rerun after the successful commit is allowed, but unrelated changes must not be overwritten.
    try:
        existing=json.loads(read_git_blob(prefix+'publication_receipt.json',ref=remote_head))
        already_done=(existing.get('run_key')==run_key and existing.get('evidence_sha256')==sha_evidence)
    except Exception:
        already_done=False
    if not already_done:raise RuntimeError('V7.4: ветка изменилась после preflight. Не перезаписываю чужие изменения.')
else:already_done=False

# Atomic replacement is permitted ONLY if all computational checks and local QA passed.
# Otherwise publish a diagnostic draft Release; leave old active segment unchanged.
if not can_replace:
    print('V7.4: QA/вычисления не полностью пройдены. Отчёты скачаны. Старый рабочий сегмент НЕ удаляю.',flush=True)

publication_plan=out/'ML5_V740_PUBLICATION_PLAN.json'
planned=None
if publication_plan.is_file():
    try:
        j=json.loads(publication_plan.read_text('utf-8'))
        if j.get('run_key')==run_key and j.get('base_sha')==TARGET_HEAD_AT_PREFLIGHT:planned=j
    except Exception:pass

publication_receipt={
    'version':CFG.VERSION,'run_key':run_key,'release_tag':release_tag,
    'release_url':f'https://github.com/{CFG.REPO}/releases/tag/{release_tag}',
    'evidence_sha256':sha_evidence,'parts_manifest':index_file.name,
    'source_commit':SOURCE_COMMIT,'source_count':52,'computational_pass':can_replace,
    'scientific_targets_all_identified':science_ok,
    'reconstruction_computational_pass':ADR_COMPUTATIONAL_PASS,'target_prefix':prefix,
    'replacement_policy':'atomic current ML-05 tree only; preserve git history and Stage I',
    'remote_verification_state':'PREPARED_NOT_YET_VERIFIED'
}
receipt_path=out/'publication_receipt.json'
receipt_path.write_text(json.dumps(publication_receipt,ensure_ascii=False,indent=2),encoding='utf-8')
pubmanifest=out/'run_manifest_publication.json'
pubm=json.loads(Path(MANIFEST).read_text('utf-8'))
pubm['publication_status']='PENDING_GITHUB_VERIFICATION' if can_replace else 'DRAFT_PARTIAL_FAIL'
pubm['status']['publication']=pubm['publication_status']
pubm['publication_reference']={'release_tag':release_tag,'evidence_sha256':sha_evidence}
pubmanifest.write_text(json.dumps(pubm,ensure_ascii=False,indent=2,default=str),encoding='utf-8')

if already_done:
    _remote_manifest=read_git_blob(prefix+'run_manifest.json',ref=remote_head)
    _remote_receipt=read_git_blob(prefix+'publication_receipt.json',ref=remote_head)
    if json.loads(_remote_manifest).get('status',{}).get('publication')!='PUBLISHED_GITHUB_VERIFIED':
        raise RuntimeError('V7.4: final verified status missing on a re-run')
    pubmanifest.write_bytes(_remote_manifest)
    receipt_path.write_bytes(_remote_receipt)

# What remains in active stage2/ml05_education AFTER successful update.
target_files={
    prefix+'ML5_EDUCATION_MIGRATION.ipynb':Path(NOTEBOOK),
    prefix+'ML5_EDUCATION_MIGRATION_RESULTS.xlsx':Path(EXCEL),
    prefix+'run_manifest.json':pubmanifest,
    prefix+'publication_receipt.json':receipt_path,
    prefix+index_file.name:index_file,
    prefix+summary_file.name:summary_file,
    prefix+source_csv.name:source_csv,
    prefix+lineage_csv.name:lineage_csv,
    prefix+gates_csv.name:gates_csv,
    prefix+source_stats_csv.name:source_stats_csv,
    prefix+module_csv.name:module_csv,
    prefix+metrics_csv.name:metrics_csv,
    prefix+qa_csv.name:qa_csv,
    prefix+adr_ledger_csv.name:adr_ledger_csv,
    prefix+adr_summary_file.name:adr_summary_file,
    prefix+adr_lineage_csv.name:adr_lineage_csv,
    **{prefix+p.name:p for p in v72_files.values()},
    prefix+'ML5_V740_PAIRED_TESTS.csv':v74_paired_csv,
    prefix+'ML5_V740_ERROR_STRATA.csv':v74_strata_csv,
    prefix+'ML5_V740_PAIRED_AUDIT.json':v74_audit_json,
}
if not all(k.startswith(prefix) for k in target_files): raise RuntimeError('V7.4: нарушена изоляция сегмента')

# Prepare a candidate Git commit *before* creating a release; tag points to V7.4 source.
commit_sha=remote_head if already_done else None
if can_replace and not already_done:
    if planned:
        candidate=api_json('GET',BASE+'/git/commits/'+planned['candidate_sha'])
        commit_sha=planned['candidate_sha']
        if candidate['parents'][0]['sha']!=remote_head:raise RuntimeError('V7.4: кандидат не является потомком HEAD')
    else:
        old_tree=api_json('GET',BASE+'/git/trees/'+remote_head,params={'recursive':1})
        if old_tree.get('truncated'):raise RuntimeError('V7.4: неполное дерево Git, удаление запрещено')
        old_paths={x['path'] for x in old_tree.get('tree',[]) if x.get('type')=='blob' and x.get('path','').startswith(prefix)}
        obsolete=sorted(old_paths-set(target_files))
        changes=[]
        for filename,path in sorted(target_files.items()):
            binary=Path(path).read_bytes() # Small files only; evidence is Release-only.
            if len(binary)>45*2**20:raise RuntimeError('V7.4: слишком большой Git blob '+filename)
            blob=api_json('POST',BASE+'/git/blobs',json={'content':base64.b64encode(binary).decode(),'encoding':'base64'})
            changes.append({'path':filename,'mode':'100644','type':'blob','sha':blob['sha']})
        for filename in obsolete:
            changes.append({'path':filename,'mode':'100644','type':'blob','sha':None})
        old_commit=api_json('GET',BASE+'/git/commits/'+remote_head)
        tree=api_json('POST',BASE+'/git/trees',json={'base_tree':old_commit['tree']['sha'],'tree':changes})
        candidate=api_json('POST',BASE+'/git/commits',json={
            'message':f'ML-05 V7.4.0 FULL verified replacement: {run_key} (old segment files retired)',
            'tree':tree['sha'],'parents':[remote_head]})
        commit_sha=candidate['sha']
        plan={'run_key':run_key,'base_sha':remote_head,'candidate_sha':commit_sha,
              'obsolete_paths':obsolete,'new_files':sorted(target_files)}
        publication_plan.write_text(json.dumps(plan,ensure_ascii=False,indent=2),encoding='utf-8')
        progress_event('DOWNLOAD_RESULTS',f'Кандидат Git commit готов, старых файлов к замене {len(obsolete)}',announce=True)

# Create a draft Release. The tag MUST point to the prepared V7.4 commit.
all_releases=api_json('GET',BASE+'/releases',params={'per_page':100})
release=next((x for x in all_releases if x.get('tag_name')==release_tag),None)
if release is None:
    release=api_json('POST',BASE+'/releases',json={
        'tag_name':release_tag,'target_commitish':commit_sha or remote_head,
        'name':'ML-05 V7.4.0 FULL '+run_key,'draft':True,'prerelease':True,
        'body':f'Stage I 52/52. Computational PASS: {can_replace}. Scientific 7/7 identified: {science_ok}. '
               f'Evidence SHA-256 {sha_evidence}. Full results belong to this private repository.'})
release_id=release['id']
_v74_uploads=v74_release_asset_plan(index_file,chunk_dir,chunk_index,small_zip)
for _v72_i,(p,mime) in enumerate(_v74_uploads,1):
    progress_event('DOWNLOAD_RESULTS',f'GitHub asset {_v72_i}/{len(_v74_uploads)}: {Path(p).name}',_v72_i-1,len(_v74_uploads),announce=True)
    upload_release_asset(release_id,p,mime)
    progress_event('DOWNLOAD_RESULTS',f'Проверено {_v72_i}/{len(_v74_uploads)} assets',_v72_i,len(_v74_uploads),announce=True)
assets=remote_release_assets(release_id)
needed_assets=[p for p,_mime in _v74_uploads]
for p in needed_assets:
    a=assets.get(p.name)
    if not a or a.get('state')!='uploaded' or a.get('digest')!='sha256:'+hash_file(p) or int(a.get('size',-1))!=p.stat().st_size:
        raise RuntimeError('V7.4: удалённый Release-asset не прошёл приёмку '+p.name)

if can_replace:
    if not already_done:
        if repo_head(branch)!=remote_head:raise RuntimeError('V7.4: конфликт обновления Git ветки, старый сегмент не изменён')
        api_json('PATCH',BASE+'/git/refs/heads/'+quote(branch,safe='/'),json={'sha':commit_sha,'force':False})
        if repo_head(branch)!=commit_sha:raise RuntimeError('V7.4: не подтверждён Git HEAD после коммита')
    # Check EVERY committed file by round-trip SHA256 from GitHub, then ensure old files are absent.
    for filename,path in target_files.items():
        remote=read_git_blob(filename,ref=commit_sha)
        if hashlib.sha256(remote).hexdigest()!=hash_file(path):
            raise RuntimeError('V7.4: GitHub roundtrip SHA mismatch '+filename)
    finaltree=api_json('GET',BASE+'/git/trees/'+commit_sha,params={'recursive':1})
    if finaltree.get('truncated'):raise RuntimeError('V7.4: Git tree truncated in final check')
    now={x['path'] for x in finaltree['tree'] if x['type']=='blob' and x['path'].startswith(prefix)}
    if now!=set(target_files):raise RuntimeError('V7.4: остались устаревшие файлы ML-05 или пропали новые')
    if release.get('draft'):
        release=api_json('PATCH',BASE+'/releases/'+str(release_id),json={'draft':False,'prerelease':True})
    published=api_json('GET',BASE+'/releases/'+str(release_id))
    if published.get('draft') or published.get('tag_name')!=release_tag:
        raise RuntimeError('V7.4: новый Release не подтвердился')
    # Confirm the tag points to the new commit, not the retired V2 commit.
    tagref=api_json('GET',BASE+'/git/ref/tags/'+quote(release_tag,safe=''))
    tagsha=tagref.get('object',{}).get('sha')
    if tagsha!=(commit_sha if not already_done else existing.get('prepared_candidate_sha')):
        raise RuntimeError('V7.4: release tag does not point to the checked code commit')
    # V7.4: second metadata-only commit records independently verified publication.
    # The first verified commit atomically replaces only the ML-05 tree; this second one updates
    # the publication state after (not before) the GitHub/Release round-trip succeeds.
    if not already_done:
        pubm['status']['publication']='PUBLISHED_GITHUB_VERIFIED'
        pubm['publication_status']='PUBLISHED_GITHUB_VERIFIED'
        pubm['publication_reference']['candidate_commit_sha']=commit_sha
        pubm['publication_reference']['release_id']=release_id
        pubm['publication_reference']['verification']='GIT_TREE_AND_RELEASE_ASSET_DIGESTS_PASS'
        pubmanifest.write_text(json.dumps(pubm,ensure_ascii=False,default=str,indent=2),encoding='utf-8')
        publication_receipt.update({'remote_verification_state':'PASS_GITHUB',
            'prepared_candidate_sha':commit_sha,'release_id':release_id,
            'release_digests_confirmed':True,'stage1_unchanged':True})
        receipt_path.write_text(json.dumps(publication_receipt,ensure_ascii=False,indent=2),encoding='utf-8')
        _base_commit=api_json('GET',BASE+'/git/commits/'+commit_sha)
        _changes=[]
        for _path,_file in ((prefix+'run_manifest.json',pubmanifest),
                            (prefix+'publication_receipt.json',receipt_path)):
            _data=Path(_file).read_bytes()
            _blob=api_json('POST',BASE+'/git/blobs',json={
                'content':base64.b64encode(_data).decode(),'encoding':'base64'})
            _changes.append({'path':_path,'mode':'100644','type':'blob','sha':_blob['sha']})
        _tree=api_json('POST',BASE+'/git/trees',json={
            'base_tree':_base_commit['tree']['sha'],'tree':_changes})
        _receipt_commit=api_json('POST',BASE+'/git/commits',json={
            'message':'ML-05 V7.4.0: verified release and GitHub publication receipt '+run_key,
            'tree':_tree['sha'],'parents':[commit_sha]})['sha']
        if repo_head(branch)!=commit_sha:
            raise RuntimeError('V7.4: concurrent branch modification before final receipt')
        api_json('PATCH',BASE+'/git/refs/heads/'+quote(branch,safe='/'),json={
            'sha':_receipt_commit,'force':False})
        if repo_head(branch)!=_receipt_commit:
            raise RuntimeError('V7.4: final publication receipt commit was not installed')
        for _path,_file in ((prefix+'run_manifest.json',pubmanifest),
                            (prefix+'publication_receipt.json',receipt_path)):
            if hashlib.sha256(read_git_blob(_path,ref=_receipt_commit)).hexdigest()!=hash_file(_file):
                raise RuntimeError('V7.4: final publication receipt failed round-trip '+_path)
        commit_sha=_receipt_commit
        print('V7.4: final GitHub publication receipt independently verified',commit_sha,flush=True)
    else:
        existing=json.loads(read_git_blob(prefix+'publication_receipt.json',ref=remote_head))
        if existing.get('remote_verification_state')!='PASS_GITHUB':
            raise RuntimeError('V7.4: rerun found incomplete metadata receipt; manual recovery required')
        commit_sha=remote_head
    STATE['publication']='PASS_GITHUB_V74_VERIFIED'
    print('ЗАМЕНА ML-05 ПОДТВЕРЖДЕНА:',commit_sha,flush=True)
    print('Только новый набор файлов в',prefix,'; старые файлы удалены из текущего дерева',flush=True)
else:
    STATE['publication']='DRAFT_REVIEW_ONLY_OLD_SEGMENT_PRESERVED'
    print('Новая доказательная база в черновом Release. Старый сегмент сохранён из-за FAIL.',flush=True)

receipt={**publication_receipt,'commit_sha':commit_sha,'release_id':release_id,
    'final_publication_status':STATE['publication'],
    'time_utc':datetime.now(timezone.utc).isoformat()}
(out/'ML5_V740_PUBLICATION_CONFIRMATION.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2),encoding='utf-8')
print('GitHub Release:',publication_receipt['release_url'],flush=True)
print('Небольшой отчётный архив:',small_zip,flush=True)
print('Научная приёмка:', 'PASS' if science_ok else 'LIMITED_BY_OBSERVABILITY',flush=True)
if can_replace:mark_stage('DOWNLOAD_RESULTS')
else:progress_event('DOWNLOAD_RESULTS','Публикация частичного исследования; 100% исполнения не подтверждены',announce=True)
bar=globals().get('MAIN_PROGRESS')
if bar is not None: bar.close()
